# SEC filings → price impact — end-to-end study (SPEC v5 + pre-coding addendum)

**What this notebook does.** It builds a point-in-time event panel of **10-K, 10-Q and 8-K filings** for up to **500 US firms**
that are common to our OHLCV files and the Hugging Face 10-K corpus (`JanosAudran/financial-reports-sec`), from **Jan 2000 to
Jul 2026**. On that panel it runs three separate tracks:

| Track | Question | Main outputs |
|---|---|---|
| **A — event study** | What happens to prices, volume and volatility around disclosures? (association only) | CAR paths, pre-trends, SUE-conditioned drift, verdicts |
| **B — prediction** | What could have been predicted at decision time? Primary target **T_trade h=1** (Up / Sideways / Down) | ablation ladder L0–L8, tuned heads, fine-tuned encoders, stacking, dev holdout, locked test, backtest |
| **C — content validity** | Are sections, item mappings and extracted events correct? (human gold labels only) | gold templates, extraction F1 / faithfulness |

**How to run.** Set the owner settings in the first code cell, then *Runtime → Run all*. Every stage is resumable: re-running skips
stages whose `_SUCCESS` marker matches; the EDGAR download (S05) checkpoints per firm.

1. **smoke** (default, hard rule 4): 5 firms (4 S&P 500 + 1 non-member), 2021–2023, ~20–30 min on CPU. Verifies the chain on real data.
2. **full**: 500 firms (S&P 500 members first, then non-members), 2000–2026, reported as **S&P 500 / non-S&P 500 / combined**.
   Gate **G2** = your opt-in + the preflight budget check, which runs before the download (the owner waived the pilot, D-058).
   Expect **~10–14 h** for the EDGAR download/parse on a CPU runtime (fair-access limit 6 requests/s; S05 stops cleanly after
   `s05_max_wall_hours` and resumes on re-run), then **~3–6 h of A100** for embeddings, tuning and fine-tuning. Run **Part 1 on a
   CPU (High-RAM) runtime**, switch to **A100** for Part 2, back to CPU for Part 3 (`PART` setting). Restart when switching.
3. **pilot** (optional): ~40 sector-balanced firms from 2012.

**Rules this notebook enforces** (SPEC.md §0.2): SEC filings only as text; no news; four timestamps per event; features available no
later than decision time; the report contains computed values only; association ≠ causation; prediction ≠ profitability;
results are *validation-selected*, never a global optimum; the locked test is scored **once**, and only when you unseal it.

Repository build: `5c19aff` · Artifacts: `MyDrive/sec-price-study/<profile>/…` · Report: `…/<profile>/reports/results_report.html`

## 0. Owner settings

In [ ]:
RUN_PROFILE = "smoke"      # "smoke" -> "full" ("pilot" optional)   (hard rule 4: smoke is the default)
PART = "all"               # "cpu1" (S00-S12, S20-S21) | "gpu" (S13-S17) | "cpu2" (S18-S24) | "all"
DRIVE_FOLDER = "sec-price-study"

# Study window: configs/default.yaml sets Jan 2000 - Jul 2026 for pilot/full (owner); smoke uses 2021-2023.
OWNER_OVERRIDES = {
    "extensions": {"ext_stacking": True},                 # owner: ensemble of the best models (H5), gated by G3c
    "gates": {"owner_override_g3_finetune": False},       # True = fine-tune even if G3 fails (results labelled EXPLORATORY)
    "rag": {"run_extraction_without_gold": False},        # True = teacher/student extraction without gold-val (EXPLORATORY)
    "lockbox": {"unlock_locked_test": False,              # True only after reviewing the development-holdout results
                "acknowledge_reuse": False},
    "full": {"owner_opt_in_full": True,                   # owner opted in to the full study (2026-10-23)
             "require_pilot_success": False},             # owner waived the pilot gate (D-058); the preflight still applies
    # Colab sessions can disconnect. S05 (the long EDGAR download, ~9 h for 500 firms) stops cleanly after this many hours;
    # run the same cell again and finished firms are skipped. 0 = no limit.
    "sec": {"s05_max_wall_hours": 8},
}

## 1. Environment
Colab: mounts Google Drive and reads the SEC User-Agent from **Colab secrets** (🔑 icon → add `SEC_USER_AGENT` with value
`Your Name your.email@example.com`; optional `HF_TOKEN`). The User-Agent is never hard-coded (hard rule 13).

In [ ]:
import os, sys, json, time, subprocess, importlib, shutil
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    DATA_ROOT = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    HOT_ROOT = Path("/content/hot")
    REPO = Path("/content/sec-filings-price-impact")
    for k in ("SEC_USER_AGENT", "HF_TOKEN"):
        try:
            v = userdata.get(k)
            if v: os.environ[k] = v
        except Exception:
            pass
else:
    DATA_ROOT = Path(os.environ.get("DATA_ROOT", Path.cwd() / "sec-price-study"))
    HOT_ROOT = Path(os.environ.get("HOT_ROOT", DATA_ROOT / "_hot"))
    REPO = Path(os.environ.get("SECPRICE_NB_REPO", Path.cwd() / "sec-filings-price-impact"))
for p in (DATA_ROOT, HOT_ROOT, REPO):
    p.mkdir(parents=True, exist_ok=True)
os.environ.update({"DATA_ROOT": str(DATA_ROOT), "HOT_ROOT": str(HOT_ROOT), "SECPRICE_REPO": str(REPO),
                    "SECPRICE_BUILD_COMMIT": "5c19aff", "TOKENIZERS_PARALLELISM": "false"})
print("DATA_ROOT", DATA_ROOT, "| HOT_ROOT", HOT_ROOT, "| REPO", REPO)
print("SEC_USER_AGENT set:", bool(os.environ.get("SEC_USER_AGENT")))
try:
    import torch; print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
except Exception as e:
    print("torch not available:", e)

Install only what is missing, pinned against the runtime's own core libraries (torch / pandas / numpy are never reinstalled).

In [ ]:
# Add-on pins verified by the test suite and the pilot (requirements.lock). Runtime core libraries are never reinstalled.
PINS = {"catboost": "1.2.10", "optuna": "5.0.0", "arch": "8.0.0", "pandas_market_calendars": "5.5.0", "reportlab": "5.1.0",
        "lm-format-enforcer": "0.11.3", "yfinance": "1.7.0", "lightgbm": "4.7.0", "statsmodels": "0.15.0",
        "transformers": "5.20.2", "markdown": "3.11.2", "jsonschema": "4.26.0", "openpyxl": "3.1.5", "pytest": "9.1.1",
        "nbformat": "5.11.1", "lxml": "6.1.3"}
from importlib import metadata as _md
def _ver(pkg):
    try: return _md.version(pkg)
    except Exception: return None
missing = [f"{k}=={v}" for k, v in PINS.items() if _ver(k) != v]
pins = []
for mod in ("torch", "pandas", "numpy", "scipy", "sklearn", "pyarrow"):
    try:
        m = importlib.import_module(mod)
        pins.append(f"{'scikit-learn' if mod == 'sklearn' else mod}=={m.__version__.split('+')[0]}")
    except Exception:
        pass
Path("/tmp").mkdir(exist_ok=True)
Path("/tmp/secprice_constraints.txt").write_text("\n".join(pins))
if missing:
    print("installing pinned:", missing)
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-c", "/tmp/secprice_constraints.txt", *missing])
    if r.returncode != 0:
        # a pin conflicts with this runtime: install each package alone, keep the pin where possible, report the rest
        for spec in missing:
            r1 = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-c", "/tmp/secprice_constraints.txt", spec])
            if r1.returncode != 0:
                print("WARNING: could not install", spec, "- using the runtime's version", _ver(spec.split("==")[0]))
import transformers
print("ready; transformers", transformers.__version__, "| pinned installs attempted:", missing)

## 2. Package source (embedded from the repository — do not edit here)
The cells below write the `secprice` package, its configs, tests and documents to `REPO`. They are generated from the
repository by `tools/build_notebook.py`, so the notebook and the tested code are identical.

In [ ]:
for d in ['.', 'configs', 'configs/profiles', 'docs', 'src/secprice', 'src/secprice/audit', 'src/secprice/backtest', 'src/secprice/evaluation', 'src/secprice/events', 'src/secprice/extensions', 'src/secprice/features', 'src/secprice/gold', 'src/secprice/market', 'src/secprice/models', 'src/secprice/parse', 'src/secprice/rag', 'src/secprice/report', 'src/secprice/sec', 'src/secprice/splits', 'src/secprice/targets', 'src/secprice/testing', 'src/secprice/track_a', 'src/secprice/universe', 'src/secprice/viz', 'tests/fixtures', 'tests/integration', 'tests/unit']:
    (REPO / d).mkdir(parents=True, exist_ok=True)
print(len(['.', 'configs', 'configs/profiles', 'docs', 'src/secprice', 'src/secprice/audit', 'src/secprice/backtest', 'src/secprice/evaluation', 'src/secprice/events', 'src/secprice/extensions', 'src/secprice/features', 'src/secprice/gold', 'src/secprice/market', 'src/secprice/models', 'src/secprice/parse', 'src/secprice/rag', 'src/secprice/report', 'src/secprice/sec', 'src/secprice/splits', 'src/secprice/targets', 'src/secprice/testing', 'src/secprice/track_a', 'src/secprice/universe', 'src/secprice/viz', 'tests/fixtures', 'tests/integration', 'tests/unit']), 'folders ready')

In [ ]:
%%writefile $REPO/CLAUDE.md
# CLAUDE.md - sec-filings-price-impact (digest of SPEC.md v5 + SPEC_ADDENDUM.md)

## Hard rules (SPEC.md §0.2, condensed)
1. SEC filings only (10-K, 10-Q, 8-K) as text. Never load news / FNSPID / FinMultiTime news / StockNet tweets.
2. Tracks stay separate: A event study (association), B prospective prediction, C content validity.
3. Four timestamps per event; features available no later than decision_time; prior documents strictly < decision_time.
4. RUN_PROFILE = smoke by default; pilot/full by explicit change; full needs G2 (pilot done + preflight + owner opt-in).
5. Never fabricate data, dates, results or metrics; the report contains computed values only.
6. Association is not causation; prediction is not profitability.
7. EAR never a feature for its own event/cluster; SUE only from the event's own filing/exhibit or earlier public accessions.
8. 8-Ks are event-driven (no expectation). 9. Never choose documents/events using later price moves.
10. High scores trigger an audit, not termination. 11. Provenance decides claims (strict vs retrospective leaderboards).
12. Verify before use; pin revisions; log substitutions. 13. SEC fair access: UA from env, <= 6 rps, backoff, pause on block.
14. No paid/hosted model APIs. 15. Seeds, config hash, commit, environment manifest, pinned deps. 16. Tests are part of the deliverable.

## Addendum (binding)
- Primary Track B target = **T_trade h=1** (first executable price after decision_time); T_info = Track A response.
- Stock and benchmark returns on identical intervals; never a close-to-close factor against an open-to-close return.
- Public availability = acceptance + lag is an **assumption**: wording "point-in-time evaluation under an assumed
  availability/processing-latency policy".
- accounting.predecessor_lookback >= 4 years; CPU smoke uses fixtures + mock LLM; gold-test single use enforced.

## Environment (§13)
- Local: `.venv` (uv, Python 3.12), `pip install -e .[dev]`, `pytest`, `secprice run --profile smoke --stages S00-S12`.
- Colab: `notebooks/SEC_Filings_Price_Impact_v5_EndToEnd.ipynb` (embeds this repo). DATA_ROOT = Drive folder
  `MyDrive/sec-price-study`; HOT_ROOT = /content/hot. Secrets: `SEC_USER_AGENT` ("Name email"), optional `HF_TOKEN`.
- Parts: CPU S00-S12, S20-S21 -> GPU S13-S17 -> CPU S18-S19, S22-S24. Restart the runtime when switching CPU/GPU.

## Layout (§14)
`configs/` (default + profiles + prereg + models ledger + taxonomy + question bank) | `src/secprice/` (stages_data,
stages_model, stages_report, pipeline/cli, sec, universe, market, audit, parse, events, targets, features, splits,
models, rag, gold, evaluation, track_a, backtest, viz, report, extensions, testing) | `tests/` (unit, integration,
fixtures) | `docs/decision_log.md` | `tools/` (input preparation, notebook builder).

## Current milestone
M0-M13 implemented; 99 tests (unit, offline integration, opt-in live online smoke); notebook runs end to end on smoke on a
clean data root. **Full profile Part 1 (S00-S12, S20, S21, S24) ran on all 500 firms, 2000-2026, real data**; artifacts on
Drive (`MyDrive/sec-price-study/full`), seen as done by the notebook's own settings. Owner decisions: no pilot gate (D-058),
S&P / non-S&P / combined reporting (D-056), survivor-only wording (D-057). Next: owner runs Part 2 (A100, `PART="gpu"`) and
Part 3 on Colab, annotates gold (Track C), then unlocks the lockbox once. Decisions D-001..D-073; audit with closure table
in docs/audit_2026-10-23.md.


In [ ]:
%%writefile $REPO/configs/default.yaml
# All parameters (SPEC.md §3, §15). Profiles in configs/profiles/ overlay this file.
# Owner overrides (notebook "Owner settings" cell) overlay the profile. Every override is logged.
run_profile: smoke            # hard rule 4: smoke is the default; pilot/full need an explicit change
seed: 20261023

paths:
  data_root_env: DATA_ROOT    # artifacts root: Drive folder on Colab, a project data folder locally
  hot_root_env: HOT_ROOT      # fast scratch (Colab /content); raw HTTP cache lives here
  inputs_dirname: inputs      # DATA_ROOT/inputs: owner-supplied files (OHLCV parquet, lexicon, caches)

sec:
  max_rps: 6                  # SEC ceiling is 10; default 6
  user_agent_env: SEC_USER_AGENT
  cache_days: 30
  cache_max_bytes: 0          # 0 = cache every response (zlib-compressed); full profile caps it to fit the disk budget
  timeout_s: 60
  max_retries: 6
  backoff_base_s: 2.0
  workers: 8                  # download threads sharing one rate limiter
  parse_workers: 0            # 0 = os.cpu_count() - 1
  retry_failed_filings: true  # a re-run of S05 re-fetches only filings that ended in a transient error
  max_parse_workers: 8        # cap on S05 processes; each gets max_rps / processes (12 works on a 15 GB laptop)
  max_8k_bytes: 15000000      # full submission .txt read cap for 8-Ks
  max_doc_bytes: 40000000
  blocked_403_limit: 20       # consecutive 403s -> pause and report (hard rule 13)
  s05_max_wall_hours: 0       # 0 = unlimited; >0 = S05 stops cleanly after this many hours (re-run resumes per firm)
  header_check_sample: 150    # periodic filings whose SGML header is fetched to re-verify the JSON timestamp rule
  batch_size: 400             # parse checkpoint batch

hf:
  repo: JanosAudran/financial-reports-sec
  revision: 5da7e3c8b920a586b8c36eecba4aaa0152a59a52
  qa_sample_filings: 200      # 10-K section-parser QA against HF sections
  qa_enabled: true
  qa_max_shards: 2

universe:
  n_stocks: 500
  source: sp500_fja05680
  sp500_repo: fja05680/sp500
  sp500_commit: a2430f2af0c79ddf0748e91de11bdeb1616ab5a7
  require_hf_overlap: true    # owner: firms common to the HF 10-K corpus and our OHLCV data
  require_ohlcv: true
  exclude_foreign_filers: true
  selection: sp500_first_then_seeded_sector_stratified
  fixed_tickers: []           # profile may pin tickers (smoke)
  ff12_url: https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/Siccodes12.zip

study:
  start: "2000-01-01"
  end: "2026-07-31"           # owner: Jan 2000 - Jul 2026 (spec default 2026-10-31 is overridable)

market:
  warmup_start: "1998-01-01"
  benchmark: SPY
  calendar: NYSE
  price_provider: local_triframe_plus_yfinance
  local_ohlcv_subdir: ohlcv_1day          # DATA_ROOT/inputs/ohlcv_1day/<TICKER>.parquet (split-adjusted)
  local_actions_file: yahoo_actions.parquet  # DATA_ROOT/inputs: dividends + splits cache
  yfinance: {auto_adjust: false, actions: true, back_adjust: false, repair: false, interval: 1d}
  seam_check_days: 40
  factors:
    ff5_url: https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_5_Factors_2x3_daily_CSV.zip
    mom_url: https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Momentum_Factor_daily_CSV.zip

text:
  predecessor_lookback_years: 2
  sections:
    "10-K": [10K_1, 10K_1A, 10K_7, 10K_7A]
    "10-Q": [10Q_MDA, 10Q_MR, 10Q_RF]
    "8-K": [8K_ITEMS, 8K_EX99]
  max_chars_per_section: 300000
  chunk_words: 350
  chunk_overlap_words: 50
  max_chunks_per_section: 24
  drop_numeric_lines: true
  model_text_chars: 12000     # characters per event fed to TF-IDF / fine-tuning text builders

accounting:
  predecessor_lookback_years: 4   # addendum P1.4: separate from text lookback, >= 4 years
  sue_min_diffs: 4
  sue_max_diffs: 8
  eps_tags: [EarningsPerShareDiluted, EarningsPerShareBasic, EarningsPerShareBasicAndDiluted]
  fact_tags: [Assets, Liabilities, StockholdersEquity, NetIncomeLoss, Revenues,
              RevenueFromContractWithCustomerExcludingAssessedTax, SalesRevenueNet, OperatingIncomeLoss,
              CashAndCashEquivalentsAtCarryingValue, LongTermDebt]
  shares_tag: EntityCommonStockSharesOutstanding

timing:
  public_lag_minutes: 5
  public_lag_sensitivity: [0, 15, 60]
  decision_latency_minutes: 30
  decision_latency_sensitivity: [10, 120]
  date_only_assumed_time: "17:30"
  json_rule_max_mismatch: 0.01   # share of header checks allowed to disagree before JSON times drop to quality C

taxonomy_version: taxonomy_8k_v1

targets:
  horizons: [1, 2, 5, 20, 60]
  primary: {target_def: T_trade, horizon: 1, kind: label3}       # addendum P0.1
  complementary: {target_def: T_info, horizon: 1, kind: label3}
  regression: [ar, vol_ratio, abs_std]
  est_window: [-250, -30]
  est_min_obs: 120
  vol_post: [1, 20]
  vol_pre: [-60, -5]
  sensitivity_horizons: [1, 5]

labels:
  k: 0.5
  k_sensitivity: [0.25, 0.75]

contam_window_sessions: 60

splits:
  min_train_years: 8
  val_block_years: 2
  dev_years: 3
  dev_start: "2020-01-01"
  locked_test_start: "2023-01-01"
  embargo_days: 90
  stress_folds: 5
  purge_horizon_sessions: 60

models:
  roster_file: configs/models.yaml
  encoders: [sec_bert, finbert_tone, chrono_bert, bge_base]
  max_seq_len: 512
  embed_batch: 128
  pca_dims: 64

rag:
  top_k: 8
  bm25_k1: 1.5
  bm25_b: 0.75
  rrf_k: 60
  rerank_top_n: 30
  max_prompt_passages: 8
  max_new_tokens: 1800        # 12 fields x (object + <= 2 short quotes); truncation would void the JSON
  run_extraction_without_gold: false   # G3b cannot pass without gold-val; owner may set true (labelled exploratory)
  prefill_dev_val: true                # LLM pre-fill for gold-dev/val templates only (never gold-test)

distill:
  teacher_filings: 3000
  student_model: modernbert_base
  epochs: 3
  lr: 3.0e-5
  batch: 16

tuning:
  sampler: tpe
  pruner: median
  heads: [lightgbm, catboost, enet]
  trials: {lightgbm: 15, catboost: 6, enet: 8}
  tuned_rungs: [L1, L2, L3, L4, L6]   # L4_levels / L4_change / L6_nochange inherit the parent rung's parameters
  tune_max_train_rows: 30000  # Optuna screen: time-stratified subsample per fold (final runs use all rows)
  tune_last_folds: 3          # Optuna screen on the most recent walk-forward folds (final runs use all folds)
  timeout_s_per_study: 5400
  inner_val_frac: 0.15
  min_train_rows: 50          # a fold with fewer training rows is skipped and reported
  min_test_rows: 10
  permutations: 10
  finetune_grid:
    lr: [1.0e-5, 2.0e-5, 3.0e-5]
    batch: [8, 16]
    epochs: 3
    weight_decay: [0.0, 0.01]
    max_trials: 12
  finetune_max_train_events: 60000
  finetune_eval_folds: last      # grid selection on the last walk-forward fold, chosen config refit per fold

gates:
  text_lift: 0.01
  detector_min_precision: 0.80
  extraction_min_f1: 0.70
  faithfulness_min: 0.90
  owner_override_g3_finetune: false   # true -> fine-tune even if G3 fails; results labelled exploratory

audit:
  auc_trigger: 0.65
  f1_lift_trigger: 0.10
  ece_max: 0.10
  overfit_gap: 0.10
  underfit_margin: 0.02
  unstable_std: 0.03

bootstrap:
  n: 1000
  block: month

fdr:
  q: 0.05

trivial_effect_bps: {h_le_2: 10, h_5: 25, h_ge_20: 50}

coverage_levels: [1.0, 0.5, 0.25, 0.1]

backtest:
  horizon: 5
  long_pct: 0.80
  short_pct: 0.20
  max_weight: 0.05
  min_names_per_side: 10
  costs_bps: [10, 20]
  borrow_bps_per_year: 50
  lag_sessions_sensitivity: 1

budget:
  max_storage_gb: 60
  max_gpu_hours_b: 10
  max_gpu_hours_c: 8
  max_error_rate: 0.05
  llm_teacher_filings: 3000
  llm_gold_filings: 250
  # CPU-side preflight before the EDGAR download (G2; refuses `full` when a budget would be exceeded). Constants were
  # measured on the pilot (11,370 filings): sections -> chunks per filing, parsed-text bytes per filing.
  preflight_chunks_per_filing: {"10-K": 62, "10-Q": 26, "8-K": 10}
  preflight_bytes_per_filing: 20000
  preflight_assumed_chunks_per_s: 400       # per encoder on an A100, conservative; S13 re-measures with a micro-benchmark
  preflight_llm_s_per_prompt: 20

lockbox:
  unlock_locked_test: false     # S19 runs only when the owner sets this true (single use, logged)
  acknowledge_reuse: false

extensions:
  ext_encoders: false
  ext_feature_models: false
  ext_llms: false
  ext_rag: false
  ext_analogue: false
  ext_timeseries: false
  ext_stacking: false
  ext_heavy: false
  ext_generation_metrics: false
  ext_data_modules: false
  ext_app: false

runtime:                      # resource settings that never change results (no stage hash reads them)
  ladder_cache_max_gb: 2.0    # transformed TF-IDF / PCA blocks kept in memory (bytes-bounded, oldest evicted first)


In [ ]:
%%writefile $REPO/configs/models.yaml
# Core roster + provenance ledger (SPEC.md §6.4, §7.1). Revisions verified on the Hugging Face Hub 2026-10-23.
# stated_cutoff: a date only when the card documents one; otherwise "unknown" -> retrospective leaderboard only.
encoders:
  sec_bert:
    model_id: nlpaueb/sec-bert-base
    revision: d511591f5e74052afdab08f1f14c4ff2a1e55749
    card_url: https://huggingface.co/nlpaueb/sec-bert-base
    license: cc-by-sa-4.0
    training_corpus: "260,773 10-K filings 1993-2019 (model card)"
    stated_cutoff: "2019-12-31"
    fine_tune_data: none
    task: masked-LM encoder
    labels: none
    pooling: mean
    max_len: 512
  finbert_tone:
    model_id: yiyanghkust/finbert-tone
    revision: 4921590d3c0c3832c0efea24c8381ce0bda7844b
    card_url: https://huggingface.co/yiyanghkust/finbert-tone
    license: "not stated on card"
    training_corpus: "10-K/10-Q (2.5B tokens), earnings calls (1.3B), analyst reports (1.1B); tone fine-tune on analyst-report sentences"
    stated_cutoff: unknown
    fine_tune_data: "analyst-report sentences (tone)"
    task: tone classifier (Neutral/Positive/Negative)
    labels: [Neutral, Positive, Negative]
    pooling: mean
    max_len: 512
    native_outputs: true
  chrono_bert:
    model_id_template: "manelalab/chrono-bert-v1-{year}1231"
    vintages: [1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012,
               2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
    revisions:
      "1999": 972f0d46401d074be2adda28c3e2baafa3419e14
      "2000": 014b770948fd1bb26dad4e804ca9fafd50622a7d
      "2001": 0dd38ced4726fd3f26cd94de1605085681c43e9f
      "2002": a19895a4803e37f8e757fc4736c028fb5019b999
      "2003": 0ac1fffdd0612571529bf6f652bdb0248f25c164
      "2004": 8638b41321741066645ff2b691bd7d3229ac242a
      "2005": 1801859ac7e537cefe8a321238b9c05afda4b3d9
      "2006": ec84e6c74d57a50ff98f916c49535d0b856ddd91
      "2007": 8ec5d5c79e8f685a0411a07d6546a149623c9707
      "2008": 6a6b51e56f913729f21977c40e01421cc5204102
      "2009": 8f1408aebbe7b6a1787450ac48ceffa38c96a9c0
      "2010": 1520cf239e7922584ce7762af9e5040e8edd9374
      "2011": c525b313b750d52d650a0df6adc28df85570daea
      "2012": a21a301631c64230f82020a53c095269c82cb751
      "2013": 071ed3022975063514f5426d8c42815246241412
      "2014": 9e61adb8d6674d363f14d0d4a3153e4fea3ebefd
      "2015": 01122aa3fd7a07879a524b6897c41e55c1db091a
      "2016": 89dff1bb2043cc929c00756002469dfa9482f280
      "2017": 02bde7f9d3d8c55c1e44d7e3ee071f374a1b9bb4
      "2018": e716596f5c90945781e74e78ab90c591b9b46080
      "2019": 33a5fc6e87910b614cf4319eb995301195d9a99d
      "2020": 8402477b6b77b799ee220665fcb258939eaf7455
      "2021": 2d187f3f29b1071392b546fa1195d6b55036ae9b
      "2022": 58bdf9ce273a6744bf8c8acda2df56655e7f1f51
      "2023": d1cbfa139a61b067ce0089c12d93f196798ad6b1
      "2024": 2478c3bd4a676e2e7bdc72f8100262b39bde4b0d
    card_url: https://huggingface.co/collections/manelalab/chronobert
    license: mit
    training_corpus: "timestamped open-source text; each vintage trained only on text up to its date"
    stated_cutoff: vintage   # vintage YYYY1231
    fine_tune_data: none
    task: masked-LM encoder (ModernBERT architecture)
    labels: none
    pooling: mean
    max_len: 512             # 1,024-token context; chunks are shared across encoders (same rows, same inputs)
    vintage_rule: "event year Y uses vintage Y-1; years after the last vintage use the last vintage (still precedes the event)"
  bge_base:
    model_id: BAAI/bge-base-en-v1.5
    revision: a5beb1e3e68b9ab74eb54cfd186867f64f240e1a
    card_url: https://huggingface.co/BAAI/bge-base-en-v1.5
    license: mit
    training_corpus: "general web-scale text (card); no date cutoff stated"
    stated_cutoff: unknown
    fine_tune_data: contrastive retrieval pairs
    task: sentence embedding (CLS, normalized); dense retriever
    labels: none
    pooling: cls
    max_len: 512
reranker:
  bge_reranker:
    model_id: BAAI/bge-reranker-base
    revision: 2cfc18c9415c912f9d8155881c133215df768a70
    card_url: https://huggingface.co/BAAI/bge-reranker-base
    license: mit
    stated_cutoff: unknown
llm:
  qwen:
    model_id: Qwen/Qwen2.5-7B-Instruct
    revision: a09a35458c702b33eeacc393d103063234e8bc28
    card_url: https://huggingface.co/Qwen/Qwen2.5-7B-Instruct
    license: apache-2.0
    stated_cutoff: unknown   # not precisely documented -> retrospective only (SPEC.md §7.1)
    gated: false
judge:
  nli_judge:
    model_id: cross-encoder/nli-deberta-v3-base
    revision: 6c749ce3425cd33b46d187e45b92bbf96ee12ec7
    card_url: https://huggingface.co/cross-encoder/nli-deberta-v3-base
    license: apache-2.0
    stated_cutoff: unknown
    note: "faithfulness judge; deliberately not the generator (SPEC.md §9.2)"
student:
  modernbert_base:
    model_id: answerdotai/ModernBERT-base
    revision: 8949b909ec900327062f0ebf497f51aef5e6f0c8
    card_url: https://huggingface.co/answerdotai/ModernBERT-base
    license: apache-2.0
    stated_cutoff: unknown
ner:
  bert_ner:
    model_id: dslim/bert-base-NER
    revision: d1a3e8f13f8c3566299d95fcfc9a8d2382a9affc
    card_url: https://huggingface.co/dslim/bert-base-NER
    license: mit
    stated_cutoff: unknown
    note: "PER/ORG only, for anonymization; never money (SPEC.md §7.1)"
tabular:
  lightgbm: {stated_cutoff: trained_on_folds}
  catboost: {stated_cutoff: trained_on_folds}
  enet: {stated_cutoff: trained_on_folds}
lexical:
  loughran_mcdonald:
    source: "https://sraf.nd.edu/loughranmcdonald-master-dictionary/ (Google Sheet export of Loughran-McDonald_MasterDictionary_1993-2025)"
    sheet_export_url: "https://docs.google.com/spreadsheets/d/1P3KtWuDyQT2Eu_VwTU3D2sEeQxR9dWCS/export?format=csv"
    sha256_verified_2026_10_23: 147cb24a1687ccde60363a0fa29c62be4097debcc698c01a83e51bb8a176dbb3
    license: "academic use free; commercial use needs a licence (loughranmcdonald@gmail.com)"
    stated_cutoff: "dictionary words by year added; static word lists"


In [ ]:
%%writefile $REPO/configs/preregistration.yaml
# Pre-registration (SPEC.md §2, §15; addendum P0.1). Hashed by `secprice prereg-freeze` before any
# development-holdout scoring. Do not edit after freezing: a changed hash is logged as a protocol deviation.
version: 2
frozen_by: owner
primary:
  track: B
  target_def: T_trade          # addendum P0.1: primary Track B target is genuinely prospective
  horizon: 1
  kind: label3                 # Up / Sideways / Down of the abnormal return
  metric: macro_f1_walk_forward_mean
  guardrails:
    logloss_not_worse_than_prior: true
    ece_max: 0.10
complementary:
  - {track: A, target_def: T_info, horizons: [1, 2, 5, 20, 60], note: "information response / event study; never presented as a prospective forecast"}
  - {track: B, target_def: T_info, horizon: 1, kind: label3, note: "separate model; never the signal for the T_trade backtest"}
labels:
  k: 0.5
  k_sensitivity: [0.25, 0.75]
timing_policy: "point-in-time evaluation under an assumed availability/processing-latency policy: public = acceptance + 5 min (assumed), decision = public + 30 min"
hypotheses:
  core:
    H1: "SEC text adds out-of-time predictive value beyond price, metadata and accounting baselines (L7/L8 vs L6, common cohort)"
    H2: "Changes in filing language add more value than absolute sentiment alone (change+novelty vs LM levels inside L4/L7)"
    H3: "Validated structured event extraction adds value beyond frozen embeddings and lexical features (L8 vs L7)"
    H4: "Strict point-in-time model vintages perform differently from retrospective static checkpoints (ChronoBERT vintage vs static encoders, identical rows and heads)"
  extension:
    H5: "A stacked multi-tower ensemble beats the best single model"
    H6: "Richer RAG designs beat hybrid + rerank"
    H7: "Analogue retrieval or time-series models add value"
    H8: "Gains shrink under strict timestamps, costs, clean-event filtering and the locked test"
coverage_levels: [1.0, 0.5, 0.25, 0.1]
fdr_families:
  ladder_lifts: "L1..L8 vs L6 lifts on the primary target (common cohort)"
  hypotheses_core: "H1-H4"
  track_a_car: "CAR windows x forms x categories (all-event)"
  drift: "drift tests in SPEC.md §10.2"
  extensions: "H5-H8"
trivial_effect_bps: {h_le_2: 10, h_5: 25, h_ge_20: 50}
backtest:
  signal: "s = P(Up) - P(Down) from the frozen T_trade model"
  horizon: 5
  thresholds: "long if s >= training-period 80th pct; short if s <= 20th pct"
gates:
  G3_text_lift_macro_f1: 0.01
selection_rule: "validation-selected (walk-forward mean macro-F1; tie-break lower std, then simpler model within one bootstrap SE)"


In [ ]:
%%writefile $REPO/configs/profiles/full.yaml
# full: the defaults in configs/default.yaml (SPEC.md §3). G2 must pass first: owner opt-in (`owner_opt_in_full: true`) and
# the preflight budget check, which runs before the EDGAR download. The owner waived the pilot requirement on 2026-10-23
# (`require_pilot_success: false` in the notebook owner settings; decision log D-058).
full:
  require_pilot_success: true
sec:
  # ~200k filings: caching every raw body would need well over 100 GB; keep small responses (JSON, headers, prefixes)
  # and let large bodies stream (the parsed sections are what the later stages read)
  cache_max_bytes: 400000


In [ ]:
%%writefile $REPO/configs/profiles/pilot.yaml
# pilot: ~40 sector-balanced firms, 2012 onward, full core pipeline with small samples (SPEC.md §3).
universe:
  n_stocks: 40
study:
  start: "2012-01-01"
market:
  warmup_start: "2010-01-01"
splits:
  min_train_years: 4
  val_block_years: 2
  dev_years: 3
  dev_start: "2020-01-01"
  locked_test_start: "2023-01-01"
tuning:
  trials: {lightgbm: 8, catboost: 4, enet: 6}
  tune_max_train_rows: 0
  tune_last_folds: 0
  timeout_s_per_study: 1200
  finetune_grid:
    lr: [2.0e-5, 3.0e-5]
    batch: [16]
    epochs: 2
    weight_decay: [0.01]
    max_trials: 2
  finetune_max_train_events: 8000
distill:
  teacher_filings: 300
budget:
  llm_teacher_filings: 300
  llm_gold_filings: 100
bootstrap:
  n: 300
backtest:
  min_names_per_side: 3


In [ ]:
%%writefile $REPO/configs/profiles/smoke.yaml
# smoke: 5 firms, 2021-2023, ~10 filings per firm-form, one encoder, LLM on <= 20 filings.
# Must finish on a laptop CPU (except the optional GPU smoke). SPEC.md §3, addendum P1.5.
universe:
  n_stocks: 5
  fixed_tickers: [AAPL, MSFT, JNJ, XOM, GPI]   # 4 S&P 500 + 1 non-member (GPI) so both index buckets run (D-061)
study:
  start: "2021-01-01"
  end: "2023-12-31"
market:
  warmup_start: "2019-06-01"
text:
  predecessor_lookback_years: 1
  max_chunks_per_section: 6
accounting:
  predecessor_lookback_years: 4
smoke:
  max_filings_per_firm_form: 16
splits:
  min_train_years: 1
  val_block_years: 1
  dev_years: 0.5
  dev_start: "2023-01-01"
  locked_test_start: "2023-07-01"
  embargo_days: 30
  stress_folds: 2
  purge_horizon_sessions: 20
targets:
  horizons: [1, 2, 5, 20]
contam_window_sessions: 20
models:
  encoders: [bge_base]
  embed_batch: 16
  pca_dims: 8
distill:
  teacher_filings: 20
  epochs: 1
tuning:
  trials: {lightgbm: 3, catboost: 2, enet: 3}
  min_train_rows: 12
  min_test_rows: 3
  permutations: 3
  timeout_s_per_study: 300
  finetune_grid:
    lr: [2.0e-5]
    batch: [8]
    epochs: 1
    weight_decay: [0.0]
    max_trials: 1
  finetune_max_train_events: 200
bootstrap:
  n: 100
budget:
  llm_teacher_filings: 20
  llm_gold_filings: 20
backtest:
  min_names_per_side: 1


In [ ]:
%%writefile $REPO/configs/question_bank_v1.yaml
# Question bank v1 (SPEC.md §8), pre-registered. One JSON object per field:
# {present: bool, direction: enum|null, magnitude: number|null, unit: str|null, certainty: low|medium|high,
#  evidence: [{chunk_id, start, end, quote}]}
version: question_bank_v1
fields:
  guidance:
    question: "Does the filing initiate, raise, lower, withdraw or reaffirm forward-looking guidance?"
    directions: [initiated, raised, lowered, withdrawn, reaffirmed]
    path: current
  results_tone:
    question: "Does the filing describe results as better (beat) or worse (miss) than the prior year or expectations?"
    directions: [beat, miss, mixed]
    path: current
  impairment:
    question: "Does the filing report an impairment or write-down? Give the amount if stated."
    directions: [recorded, reversed]
    path: current
  restructuring:
    question: "Does the filing announce restructuring, layoffs or exit/disposal activities?"
    directions: [announced, expanded, completed]
    path: current
  liquidity_stress:
    question: "Does the filing disclose liquidity stress or going-concern doubt?"
    directions: [going_concern, liquidity_stress]
    path: current
  financing_covenant:
    question: "Does the filing disclose new financing, a covenant issue, waiver or default?"
    directions: [new_financing, covenant_breach, waiver, default]
    path: current
  mna:
    question: "Does the filing announce a merger, acquisition or disposition?"
    directions: [acquisition, disposition, merger]
    path: current
  dividend:
    question: "Does the filing announce a dividend change? Give the type and amount if stated."
    directions: [initiated, increased, decreased, suspended, special, maintained]
    path: current
  buyback:
    question: "Does the filing announce a share repurchase authorization or change?"
    directions: [authorized, increased, suspended]
    path: current
  litigation_regulatory:
    question: "Does the filing disclose new litigation or regulatory action against the company?"
    directions: [new, settled, adverse_ruling]
    path: current
  management_change:
    question: "Does the filing announce a change of CEO, CFO or board chair?"
    directions: [departure, appointment]
    path: current
  risk_factor_change:
    question: "Compared with the firm's prior filings, are risk factors new or removed?"
    directions: [new, removed]
    path: historical      # uses the historical evidence path (prior filings with public < decision)
certainty_levels: [low, medium, high]


In [ ]:
%%writefile $REPO/configs/taxonomy_8k_v1.yaml
# 8-K item taxonomy v1 (SPEC.md §5.2, §18.1 #11). Raw/legacy item -> category, with effective dates.
# Modern items took effect 2004-08-23 (SEC Release 33-8400). Legacy Item 12 (Results of Operations) took
# effect 2003-03-28 (Regulation G). Anything not listed maps to OTHER_OR_UNMAPPED (catch-all).
version: taxonomy_8k_v1
modern_effective: "2004-08-23"
legacy_item12_effective: "2003-03-28"
catch_all: OTHER_OR_UNMAPPED
modern:
  "1.01": material_agreement
  "1.02": material_agreement
  "1.03": bankruptcy
  "1.04": mine_safety
  "1.05": cybersecurity
  "2.01": acquisition_disposition
  "2.02": earnings
  "2.03": financial_obligation
  "2.04": financial_obligation
  "2.05": restructuring_exit
  "2.06": impairment
  "3.01": delisting
  "3.02": unregistered_sales
  "3.03": security_rights
  "4.01": auditor_change
  "4.02": non_reliance_restatement
  "5.01": control_change
  "5.02": management_change
  "5.03": bylaws_fiscal_year
  "5.04": benefit_plan_blackout
  "5.05": ethics_code
  "5.06": shell_status
  "5.07": shareholder_vote
  "5.08": director_nomination
  "6.01": abs
  "6.02": abs
  "6.03": abs
  "6.04": abs
  "6.05": abs
  "6.06": abs
  "6.10": abs
  "7.01": reg_fd
  "8.01": other_events
  "9.01": exhibits
legacy:
  "1": control_change
  "2": acquisition_disposition
  "3": bankruptcy
  "4": auditor_change
  "5": other_events
  "6": management_change
  "7": exhibits
  "8": bylaws_fiscal_year
  "9": reg_fd
  "10": ethics_code
  "11": benefit_plan_blackout
  "12": earnings
# Legacy items that can carry an earnings release before Item 12 existed. They count as earnings only when
# the exhibit content is confirmed as a results release (SPEC.md §5.2 "legacy equivalent").
legacy_earnings_equivalents: ["5", "7", "9", "12"]
# Text titles used when items metadata is missing (pre-2004 or absent): regex on ITEM INFORMATION / headings.
title_patterns:
  earnings: "results of operations"
  material_agreement: "material definitive agreement"
  acquisition_disposition: "acquisition or disposition"
  management_change: "departure of directors|election of directors|appointment of certain officers|changes in registrant'?s certifying|resignations of registrant'?s directors"
  reg_fd: "regulation fd"
  other_events: "other events"
  exhibits: "financial statements and exhibits"
  restructuring_exit: "exit or disposal"
  impairment: "material impairments"
  financial_obligation: "direct financial obligation|triggering events"
  auditor_change: "certifying accountant"
  non_reliance_restatement: "non-reliance"
  shareholder_vote: "submission of matters to a vote"
  bylaws_fiscal_year: "amendments to articles|change in fiscal year"
  delisting: "delisting|transfer of listing"
  unregistered_sales: "unregistered sales"
  bankruptcy: "bankruptcy or receivership"
  control_change: "changes in control"
  cybersecurity: "cybersecurity incidents"


In [ ]:
%%writefile $REPO/docs/decision_log.md
# Decision log

Every ambiguity, deviation and addendum resolution (SPEC.md §0.1.3; SPEC_ADDENDUM.md "Instruction to Claude Code").
Conservative choices are preferred for leakage and point-in-time issues.

| ID | Decision | Why / evidence | Spec reference |
|---|---|---|---|
| D-001 | Deliverable runs end to end from **one** Colab notebook that embeds this repository (package, configs, tests); the repository also runs locally (`secprice` CLI, pytest). The notebook doubles as the walkthrough (inline plots + "how to read"). | Owner asked for a single `.ipynb`; SPEC.md allows the end-to-end option "from the same runner". | §0 Deliverable, §14 |
| D-002 | `study.end = 2026-07-31` (owner: Jan 2000 - Jul 2026). Usable cutoff = min(last price date, last document date, study end); the last month is flagged partial when applicable. | Owner instruction; the 2026-10-31 default is overridable. | §3 |
| D-003 | Acceptance time: the submissions-JSON `acceptanceDateTime` (suffix `Z`) is the Eastern wall time shifted by the ET->UTC offset **twice** (+8 h EDT, +10 h EST). Two UTC->ET conversions reproduce the SGML `ACCEPTANCE-DATETIME` to the second (verified 2026-10-23 on 2002, 2004, 2012, 2016, 2020, 2023 and 2026 filings). The SGML header is used whenever a download carries it (every 8-K full submission); JSON times are quality A only while the header checks agree (mismatch <= 1%), else quality C. Pre-2002 filings carry midnight placeholders -> quality B, assumed 17:30 ET. Unit test against a recorded header fixture. | SPEC.md asks to verify timezone semantics on a sample. | §5.1, Appendix C |
| D-004 | "The Hugging Face repository" = `JanosAudran/financial-reports-sec` (named in every owner spec). Census of the raw JSONL shards (revision 5da7e3c8b920): **10-K only, 4,677 firms, 55,349 filings, 1994-01 to 2021-09**; no 10-Q or 8-K. It defines the firm overlap requested by the owner ("stocks common to HF and our OHLCV"), the audit and a 10-K parser QA sample. Study text (10-K, 10-Q, 8-K) comes from our own point-in-time EDGAR pipeline. Its return labels are never read (assertion). | Owner instruction + SPEC.md §13.5 ("audit / cross-check only; never use its labels"; never a substitute for our pipeline). | §13.5 |
| D-005 | Universe: our 979 OHLCV tickers mapped to CIK (SEC `company_tickers_exchange.json`, HF tickers fallback) -> in the HF 10-K corpus (736 firms) -> domestic 10-K/10-Q filers with prices in the window -> one row per CIK (longest price history among share classes). Selection of 500: S&P-500 members during the study first (`fja05680/sp500` @ a2430f2a, MIT), then a seeded, sector-stratified (Fama-French 12) fill. No price returns are read (hard rule 9). | §3 universe.source; owner overlap rule. | §3, §16 |
| D-006 | Prices: local TriFrame `final/1day` bars (split-adjusted, verified vs Yahoo) are primary; yfinance (auto_adjust=False, actions=True, back_adjust=False, repair=False) supplies the 1998-2000 warm-up and all dividends/splits, frozen on 2026-10-22 into `inputs/yf_frozen.parquet` (one ticker, LNT, failed and falls back to live yfinance). SPY from yfinance. Seam agreement reported per firm. Raw close, split-adjusted close and a total-return index kept separately. | §13.5 PriceProvider is pluggable; owner asked to use our OHLCV. | §5.5, §13.5 |
| D-007 | **Primary Track B target = T_trade, h = 1, 3-class** (addendum P0.1). T_info stays the Track A information response and a separate, complementary Track B model; it is never the backtest signal. Pre-registration v2 records this. | Addendum P0.1. | §2, §5.4 |
| D-008 | T_trade abnormal return = R_stock - beta x R_SPY on the identical open->close interval (alpha omitted on partial-day intervals). Carhart / FF5 are null for T_trade (factor returns are close-to-close only) and used for Track A, T_info and the backtest's close-to-close holding component. | Addendum P0.2. | §5.5 |
| D-009 | "Last regular-session close before acceptance" uses close <= acceptance (a close printed at the acceptance instant counts as before). | Only matters for acceptances at exactly 16:00:00. | §5.4 |
| D-010 | Artifact folders are keyed by a **stage-scoped** hash (the config keys the stage reads + upstream stage hashes); the global config hash is stamped on every artifact manifest. | Changing a model setting must not re-trigger a 10-hour EDGAR download. | §17.3 |
| D-011 | Download strategy: 10-K/10-Q primary document (full submission `.txt` when the primary is missing or 404); for 10-Ks whose MD&A is missing, by reference or < 15k characters, the full submission is read and Exhibit 13's MD&A used when longer. 8-Ks: full submission `.txt` (header, items, all exhibits; 15 MB cap). One process per firm; each process's limiter runs at max_rps / processes. | Verified on Apple, J&J and ExxonMobil filings 2001-2025 (MD&A by reference to EX-13; "Reference is made to..." within the 10-K). | §13.5 hard rule 13 |
| D-012 | Sections: 10-K Items 1, 1A, 7, 7A; 10-Q Part I Items 2-3 and Part II Item 1A; 8-K item text and EX-99 exhibits. Table rows are flattened to `a | b` text for EPS parsing; narrative consumers drop numeric-heavy lines. | "Selected sections only" (weakness). | §16 |
| D-013 | Anonymization: firm names (current + former) and tickers masked by regex in S05; NER PER/ORG masking (dslim/bert-base-NER) is available but off by default (`models.ner_mask`), because it doubles encoder GPU time. | Cost; regex covers the firm-memorization control. | §6.3 #4, §7.1 |
| D-014 | `accounting.predecessor_lookback = 4` years, separate from `text.predecessor_lookback = 2` (addendum P1.4). Pre-study 8-Ks are downloaded only when their items can carry an earnings release. SUE needs >= 4 seasonal differences (records the count; unavailable otherwise). | Addendum P1.4. | §5.6 |
| D-015 | Release EPS: diluted GAAP EPS from EX-99 tables (preferred) or prose ("$.26 per diluted share", "(assuming dilution) were $2.09"); rows mentioning non-GAAP / adjusted / excluding are skipped; confidence high/medium usable for SUE, "low" kept for QA only. The release's quarter = the latest "quarter ended <date>" phrase, overridden by the fiscal-calendar quarter end when they differ by > 45 days. QA = match rate vs the first-reported XBRL value. | Tested on Apple, J&J, ExxonMobil, JPMorgan, Abbott releases 2004-2025 (10 of 15 parsed; the rest -> no SUE_release). | §5.6 |
| D-016 | Verified earnings release = 8-K item 2.02 (or legacy item 12 after 2003-03-28; legacy items 5/7/9 only with confirmed results content) + results-release exhibit + timestamp quality A. Otherwise `earnings_date_ambiguous`. | Conservative: "usable timing". | §5.2 |
| D-017 | Cluster = a verified release + the firm's next 10-Q/10-K filed within 120 days whose report date is within 20 days of the release quarter end. | | §5.2 |
| D-018 | Overlap flags anchor on Track A day 0 (dates only). | | §5.7 |
| D-019 | Embargo is implemented as a gap: training events with decision dates in the 90 days before each test block are excluded, in addition to purging any training event whose T_info/T_trade window (purge horizon) reaches the block. | Conservative. | §6.1 |
| D-020 | Common cohort = events eligible for every compared rung: price features present, accounting available (any point-in-time XBRL fact or SUE), text sections present (and embeddings for L5/L7). Because XBRL starts in 2009, the common cohort is mostly the XBRL era; the maximum-available cohort covers 2000+. Both are reported. | Spec: "including SUE availability where accounting features are in the comparison". | §6.2 |
| D-021 | Tuning (Optuna TPE + median pruner, SQLite) runs on the common cohort for the primary target with the same budget per rung/encoder; the maximum-available cohort and secondary targets reuse those parameters (labelled "transferred"); regression heads use defaults. Default and tuned results are both reported. | Compute budget on a credit-limited Colab plan. | §9.4 |
| D-022 | Decision rule: heads train with class-balanced weights; their probabilities are prior-corrected to true frequencies, temperature-calibrated on an inner training slice, and the cost-balanced decision argmax p / prior (Laplace-smoothed; a class absent from training is never predicted) is applied once. | Fixes a double correction found in the offline smoke; keeps log-loss/ECE honest. | §9.2 |
| D-023 | Early stopping, calibration and clip bounds use an inner, time-ordered slice of the training fold; hyperparameters are selected on walk-forward validation blocks. | Leakage item 5. | §6.3 #5 |
| D-024 | Embedding pooling: mean and max over section-aware chunks (learned attention pooling is not implemented in the core). | Logged deviation. | §7.2 |
| D-025 | The fine-tuned encoder is its own candidate (rung L8_ft), scored directly; its walk-forward OOF probabilities feed the stacking extension. ChronoBERT fine-tuning uses the vintage preceding each block. Fine-tuning runs only if G3 passes, unless the owner sets `gates.owner_override_g3_finetune: true` (results labelled exploratory). | Using fine-tuned embeddings of training rows as features would be in-sample. | §7.2, §9.4, G3 |
| D-026 | Faithfulness judge = `cross-encoder/nli-deberta-v3-base` (Apache-2.0), added to the ledger. | Spec requires a local judge that is not the generator. | §9.2 |
| D-027 | LLM inference: transformers + `lm-format-enforcer` JSON-schema-constrained decoding; vLLM is not used (installing it reinstalls torch on Colab). Fallback: schema validation + one retry. The CPU smoke uses a deterministic mock generator (addendum P1.5). | §13.4 "do not reinstall torch". | §8, §13.3 |
| D-028 | `ext_stacking` (H5) is enabled by owner request ("group the best models together"); gate G3c recorded (core complete; budget; pre-stated question H5). Meta-learners see only walk-forward OOF predictions of strictly earlier folds. | Owner request. | §7.3 |
| D-029 | The lockbox is sealed by default (`lockbox.unlock_locked_test: false`). The owner sets it true after reviewing development results; the second scoring is refused unless `acknowledge_reuse`, which relabels the block as development. | Lockbox rule; the previous project opened its holdout twice. | §6.1, §17.2 |
| D-030 | "Global minima": convex heads (elastic-net logistic) get a certificate (equal penalized objective from several random starts); tree and transformer optima are validation-selected and are never called global. | Owner request vs SPEC.md §9.3 wording. | §9.3 |
| D-031 | Loughran-McDonald 1993-2025 dictionary from the official SRAF Google-Sheet export (sha256 147cb24a...); the CSV link hit a Drive quota on 2026-10-23. Academic-use licence noted. | Verify-before-use. | §7.1 |
| D-032 | Backtest signal = the frozen architecture refitted on T_trade h=5 labels (horizon matches the 5-session hold); when the primary model is fine-tuned or stacked, the best tabular finalist is used and the substitution is logged. | Addendum P0.1 (signal must be a T_trade model). | §11 |
| D-033 | Drift lag robustness uses CAR[+3,+61] (one-session lag); spread costs = 4 x bps (two legs, in and out). | | §10.2 |
| D-034 | Python 3.12 (Colab's version at build time) instead of 3.11. | §13.4 "match Colab's version". | §13.4 |
| D-035 | Smoke profile: five fixed tickers (AAPL, MSFT, JNJ, XOM, JPM), 2021-2023; offline integration smoke uses a deterministic synthetic world (tests/). | Addendum P1.5. | §3, §18.2 |
| D-036 | Filing-change similarity uses a fixed hashing space (no fitted vocabulary, so nothing is learned from the future); TF-IDF + SVD are fitted inside each fold. | Leakage item 5. | §6.3 #5 |
| D-037 | Without gold labels, Track C metrics are "not evaluated", G3b cannot pass and S17 (teacher/student, L8) does not run unless `rag.run_extraction_without_gold: true` (labelled exploratory). LLM pre-fill is written for gold-dev/val templates only; gold-test stays blind. | §12. | §8, §12 |
| D-038 | A firm keeps every CIK it filed under: the SEC's current ticker map and the HF corpus can disagree after a reorganisation (verified: XOM -> new holding-company CIK 2115436, history under 34088). Firm id = the CIK with the most filings in the window; each filing keeps its own `filer_cik` for archive URLs and XBRL company facts. Predecessors not linked by either map (e.g. Google Inc. before Alphabet) remain a limitation. | Found in the live smoke run. | §16 universe |
| D-039 | S14 builds a chunk store once (one read per firm file, partitioned by event year on local disk); encoders stream it year by year (ChronoBERT loads each vintage once). | The batch-by-time design would re-read every firm file per batch at full scale. | §7.2, §13.3 |
| D-040 | Stage hashes use only the config keys a stage reads (S05 no longer depends on `text.model_text_chars`), so model-side settings never invalidate the EDGAR download. `text.model_text_chars` = 12,000 (RAM headroom at ~175k events). | Found when a model-text change cascaded to S05. | §17.3 |
| D-041 | Tuning budget for `full`: LightGBM 15, CatBoost 6, elastic-net 8 trials per rung on L1, L2, L3, L4, L6 (median pruning); L4_levels / L4_change / L6_nochange reuse the parent rung's tuned parameters; permutation null 10 draws. Equal budget per encoder candidate in S15. | Projected S11 time ~3 h instead of 6-10 h on Colab. | §9.4 |
| D-042 | Gate G2 "pilot completed" = the pilot's S18 (development holdout) succeeded; `full.owner_opt_in_full: true` is always required. | §22 G2. | §22 |
| D-043 | Optuna SQLite studies live on local disk (HOT_ROOT) and are copied to the stage folder at the end (SQLite is unsafe on the Drive FUSE mount). Worker pools use the `spawn` context (CUDA may already be initialised on Colab). | Colab robustness. | §13.3 |
| D-044 | Encoder / tokenizer loading falls back to the BERT classes when a checkpoint's config lacks `model_type` (yiyanghkust/finbert-tone under transformers 5). lm-format-enforcer 0.11 imports `PreTrainedTokenizerBase` from a module path transformers 5 removed: a narrow alias is installed only when the attribute is missing. Constrained decoding verified on CPU with Qwen2.5-0.5B-Instruct (schema-valid output, faithfulness gate and citation offsets working). | Found by tools/check_gpu_paths_on_cpu.py. | §8, §13.5 |
| D-045 | LLM extraction is batched (16 prompts, left padding, OOM halving, per-item JSON retry) and checkpointed through the response cache every 64 filings. | One-at-a-time generation would take about a day for 3,000 teacher filings. | §8, §13.3 |
| D-046 | Release-EPS parser fixes from real releases: values are read after the label cell (keeps "(0.15)" negatives), a lone "Diluted" label line is joined with the numeric line below it (Apple 2020-21 layout), block elements inside table cells stay inline, a sentence reporting "$2.71 GAAP and $2.27 non-GAAP" yields the GAAP value, and 52/53-week fiscal years ending in the first week of a month map to the previous month. Live smoke: 25 releases, 23 parsed, 21 exact XBRL matches (the 2 sign errors were fixed after that run). | §5.6 QA. | §5.6 |
| D-047 | **Correction to D-003.** At pilot scale (8,789 header comparisons) the submissions-JSON acceptance time follows two conventions: ~70% double-shifted, ~30% true UTC (differences of exactly -4 h / -5 h), mixed within filing agents, hours, forms and years. The SGML header is now read for every filing (8-Ks: full submission; 10-K/10-Q: first 20 KB of the full submission, cached under a separate key). JSON-only times fall to quality C via the existing verification gate (mismatch 30% > 1%). | Measured on the pilot; the 2002-2026 spot sample in D-003 happened to contain only double-shifted filings. | §5.1 |
| D-048 | Release-EPS parser, second round from 2,086 pilot comparisons with XBRL (65% match before): a "(" marks a negative even when ")" is in the next cell; share-count rows are skipped; rows under annual / outlook / guidance headings and full-year prose are skipped unless the context names the quarter; "continuing operations" EPS is kept for QA but is not usable for SUE (XBRL EarningsPerShareDiluted is total EPS); table candidates are scored (income-statement heading +2, quarter context +1) instead of taking the first match; guidance ranges, discontinued-operations rows and "impact of an item" sentences are skipped. Pilot (40 firms, 2012-2026): match with the first-reported XBRL value 65% -> 77% overall, 78.5% at usable confidence (1,929 releases); remaining gaps include XBRL restatements and unit errors. | §5.6 QA. | §5.6 |
| D-049 | Tuning screen for `full`: Optuna trials score the 3 most recent walk-forward folds with a time-stratified subsample of at most 30,000 training rows per fold (a budget-driven deviation from SPEC.md v5 §9.4, which does not prescribe a screen; v5 is the single source of truth, so the earlier v3 wording is not a justification); every reported default and tuned result is then refit on all training rows of every fold. Pilot keeps the unscreened search. | Pilot S11 took 19.5 min for 10k events / 2 folds; a linear projection put full S11 at ~10-15 h. | §9.4 |
| D-050 | Adversarial validation is reported twice: with all features and with firm-level features only (market-wide `mkt_*` and calendar fields removed). Date-level features fingerprint each date, so the all-feature AUC is trivially ~1.0 (pilot: 1.0); the firm-level AUC is the interpretable covariate-shift diagnostic. | Pilot finding. | §6.3 #23 |
| D-051 | Adversarial validation also records the five features that separate train from validation (gain share), and the report reads a high AUC as covariate shift, not leakage. Pilot: the firm-level AUC stays ~1.0. In the L6 matrix the drivers are TF-IDF SVD components (filing vocabulary drifts over time; tokens are letters-only, so year numbers are not features); on the structured features alone the drivers are level fundamentals (log assets, log revenue, beta, ROA) that trend within each of the 40 firms (AUC 0.999). Both are point-in-time information, not leakage (the leakage audit and its four planted leaks pass). | Pilot finding after D-050. | §6.3 #23, §15 |
| D-052 | **Early-era real-data run (2000-2004; AAPL, XOM, AXP, IBM, PFE, INTC; 361 filings, 317 events) found two defects that the pilot (2012+) and fixtures could not.** (1) For AXP, IBM and PFE every 10-K's MD&A is by reference to Exhibit 13, and the EX-13 finder knew only "Management's Discussion and Analysis of ..." (annual reports use "FINANCIAL REVIEW", "Management Discussion"); 18 of 36 early 10-Ks had ~250 characters of MD&A. EX-13 now also accepts those line-anchored titles (EX-13 only, so main-document parsing is unchanged); after the fix 36 of 36 have 20k-190k characters. The same gap affected 18 of 668 pilot 10-Ks (2.7%); those 6 firms were re-parsed. (2) Fiscal Q4 releases quote full-year EPS and the parser can pick it (AXP $5.42, XOM $3.23 as "quarter"). New point-in-time check `flag_suspect_annual`: a Q4 value above 2x the firm's largest earlier quarterly EPS (>= 2 earlier values) becomes `suspect_annual` (QA only, never a SUE input). On the pilot it withholds 45 of 2,045 releases (35 disagreed with XBRL, 10 agreed: conservative) and raises the XBRL match rate from 78.5% to 79.8%. Residual: it needs two earlier quarters, so a firm's first releases are not covered (XOM 2003 here). | Hard rule 12 (verify before use); audit finding H3. | §5.6, §13.5 |
| D-053 | Early-era data limits, expected and handled per spec rather than defects: filings before ~2003 have no SGML acceptance time in a usable form (quality B date-only or C), so 2000-2002 events are largely B/C (T_info excludes C); SUE is unavailable before about 2004-05 (release-EPS history of 8 quarters needs several years of parsed releases and XBRL only starts in 2009); legacy 8-K item codes map with no OTHER_OR_UNMAPPED. The common cohort is therefore mostly 2009+ (D-020); the 2000-2008 events enter only the maximum-available cohort. | Early-era run. | §5.1, §6.2 |
| D-054 | Reproducibility: all 26 ChronoBERT vintages are pinned to Hub commit SHAs and S14 refuses an unpinned revision; `requirements.lock` records the tested versions; the notebook installs exact pins for the add-on libraries (core runtime libraries are never reinstalled) and falls back package by package with a printed warning. Tests assert every model has a 40-hex revision and that the notebook pins equal the lock file. | Audit findings M1, M2 (hard rules 12, 15). | §13.4 |
| D-055 | `sec.s05_max_wall_hours` (notebook default 8, config default 0 = unlimited): S05 stops cleanly after that many hours with `StageIncomplete` (recorded as "incomplete", not "failed"); running the same cell again skips finished firms. Colab sessions can disconnect during the ~9 h download; every finished firm is already on Drive, so nothing is lost either way. The key is outside the S05 stage hash. | Colab session limits. | §13.3, §17.3 |
| D-056 | **Owner decision (2026-10-23): keep 500 firms and analyse the index buckets.** The universe stays HF 10-K corpus ∩ our OHLCV, S&P 500 members first (about 353 eligible over 2000-2026), then a seeded, sector-stratified fill of non-members (about 147). Every firm carries `sp_bucket` = SP500 (ever a member in the study window per the pinned fja05680 list) or NonSP500 (not found in that list: the list is incomplete in early years, so this is not "verified never a member"); point-in-time membership on the decision date is reported alongside and is never a feature. Results are reported for SP500, NonSP500 and Combined: bucket composition; the combined models' walk-forward, development-holdout and locked-test scores per bucket; bucket-only models (same rung, head, tuned parameters and folds) against the pooled model on identical rows (paired month-block bootstrap); Track A CARs and CAR paths; SUE drift (own FDR family `drift_sp_bucket`; Track A buckets in `track_a_car_sp_bucket`, so the pre-registered families are unchanged); the backtest with the same frozen model and thresholds restricted to each bucket's names. Exploratory decomposition, not a pre-registered hypothesis. | Owner request; audit H2. | §3, §6.2, §9.2, §10, §11, §21 |
| D-057 | **Survivor-only universe, stated as such** (audit H1): the OHLCV set holds current listings only, so firms that delisted, merged or were acquired before 2026 are absent. The report, leakage item 14, F04 caption, universe text and the smallest defensible claim say "survivor-only (survivorship-biased)", not "survivorship-aware". S02 writes `sp500_coverage.json`: the share of fja05680 window members our eligible data covers (ticker-level, approximate). | Spec §4 weakness understated by the earlier wording. | §3, §4, §6.3 #14, §23 |
| D-058 | **Owner decision (2026-10-23): no pilot gate.** G2 = owner opt-in + preflight within budget; `full.require_pilot_success: false` is the owner's explicit override (logged by S00). The CPU preflight (`preflight_projection`, constants measured on the pilot) runs at the start of S05 before the download and refuses `full` when storage or GPU budgets would be exceeded; S13 re-checks with a measured micro-benchmark and now enforces `max_gpu_hours_c` as well as `_b`. The pilot artifacts remain as engineering evidence only. | Owner instruction; audit M7. | §13.3, §22 G2, hard rule 4 |
| D-059 | Provenance (audit M3): `git_commit` appends `-dirty` when src/configs/tests have uncommitted changes; `code_hash` normalises line endings; every artifact manifest lists the upstream stage hashes and the sha256 of each upstream parquet artifact; `run_manifest.json` records git commit, code hash, model revisions (all 26 ChronoBERT SHAs), data-source versions (HF revision, fja05680 commit, price provider and yfinance parameters, factor URLs) and the timing policy (public lag, decision latency and their sensitivities). The notebook stamp names the commit of the embedded code (last commit touching src/configs/tests, `-dirty` if uncommitted). Artifacts produced before this change keep their older, weaker stamps. | Hard rule 15; §16; §19; addendum P0.3. | §16, §19 |
| D-060 | Process deviation, recorded (audit M4): milestones M0-M13 were committed as one commit (`c1d5b3a`) instead of one commit per milestone (§0.1.2, §22). It cannot be undone; later work is committed per change set. | Honest record. | §0.1, §22 |
| D-061 | Smoke profile: AAPL, MSFT, JNJ, XOM + GPI (Group 1 Automotive, not in the fja05680 list). JPM is not in the HF 10-K corpus, so the owner overlap rule dropped it (4 of 5 firms); GPI restores 5 firms and exercises the non-S&P bucket. The smoke cap is 16 filings per firm and form (spec "about 10"): 10 left the development block too small for the walk-forward folds. | Audit M5, L5. | §3 profiles |
| D-062 | The pilot profile's fine-tuning grid has 2 trials (§9.4 asks 6-12 for the study); pilot = small samples. The full profile uses the spec grid (lr {1e-5, 2e-5, 3e-5} x batch {8, 16} x epochs <= 3 x weight decay {0, 0.01}, at most 12). | Audit L6. | §9.4 |
| D-063 | `filing_manifest_timed.parquet` (S07) completes the §16 filing-manifest contract: acceptance time, timestamp quality, period of report, items source, URL, checksum and parse status are known only after the SGML headers are read (S05, D-047), so S04's `filing_manifest.parquet` stays the index and S07 writes the complete per-filing table. | Audit L3. | §16 |
| D-064 | The pre-registered ECE guardrail (`ece_max` 0.10) is evaluated next to the log-loss guardrail (`guardrail_ece_ok`, `guardrail_logloss_ok` in results_long and a report table). | Audit L2. | §2, §9.2 |
| D-065 | Layout: `tools/write_docs.py` writes `docs/stage_cards/S00..S24.md` and `docs/annotation_guide.md` (run-specific cards stay next to the artifacts); `configs/lexicons/README.md` points to the Loughran-McDonald file (sha256 pinned in models.yaml) kept in `DATA_ROOT/inputs/lexicons` because its licence restricts redistribution; `secprice gold export|import|agreement` now run distinct actions. | Audit L4, L7. | §14, §17.2 |
| D-066 | Release-EPS parser, third round, from the 2005-2011 real-data run (12 firms, 6 S&P + 6 non-S&P) and a replay over all 3,360 pilot and early-era releases against XBRL: (1) losses: when "loss" (not income / earnings / profit) is the nearest results word before the number, the value is negative ("net loss of $25.3 million, or $0.80 per diluted share" was +0.80: 43 of 246 'medium' disagreements were sign errors); (2) sentence starts ignore decimal points (a "$25.3 million" no longer cuts the sentence); (3) plain-text table rows without cell separators ("Diluted $ 0.08 $ (0.05)", common before ~2008) enter the table path with all its filters; (4) EPS stated without "diluted" ("earnings per share were $0.67") forms its own class `medium_undiff`, ranked after the older patterns so no previously parsed value changes because of it, and admitted to USABLE_CONFIDENCE because its XBRL agreement (80.9%, n=335) exceeds the existing 'medium' class; (5) multi-year summary tables (three or more year headers, no quarter wording) are skipped. Replay result, before the point-in-time full-year check: usable-class agreement with XBRL 78.2% -> 83.9% (pilot, n=2,192), 66.1% -> 83.8% (2005-2011 run); parsed releases +282 (pilot) and +46 (early era). Remaining disagreements include XBRL-side derivation noise (fiscal-year mapping, YTD values), so 100% is not the target. | Hard rule 12; audit H3 follow-up. | §5.6 |
| D-067 | Preflight LLM projection follows the G3b rule: the teacher extraction (`llm_teacher_filings`, 3,000) is counted only when gold labels exist in `inputs/gold` or `rag.run_extraction_without_gold` is set (D-037); otherwise only the gold-sample prompts (250) are. First full-profile preflight (202,607 filings, 500 firms): encoders 9.3 GPU-h of 10, LLM 1.4 GPU-h of 8 (without gold), download about 15 h at 6 requests/s. **With gold labels the teacher sample projects to about 18 GPU-h > 8, so the preflight will refuse until the owner raises `max_gpu_hours_c` or lowers `llm_teacher_filings` (an explicit, logged decision).** | Found by the first full preflight. | §8, §13.3, §22 G2 |
| D-068 | **Defects found by the full 500-firm run (2026-10-23/24), all fixed with regression tests:** (1) S05 workers built their SEC client without `sec.cache_max_bytes`, so the full profile's cache cap was ignored (cache +1.8 GB per 20 firms; would have exceeded the disk); (2) catastrophic regex backtracking in HTML normalisation (`(\s*\|\s*)+$` on table rows with hundreds of empty cells, 2000s insurance statistical supplements) hung three workers at 100% CPU; every parser pattern is now timed on adversarial inputs (34 cases) and the slow ones were rewritten in unambiguous or bounded forms that match the same normalised lines (2005-2011 replay unchanged; pilot XBRL agreement 83.9% -> 84.2%); (3) the HF parser QA failed when the HF corpus holds two filings for one firm and date; (4) the full-year EPS check failed on a February 29 fiscal year end; (5) 63 filings (0.03%) failed on repeated HTTP 503s: a re-run of S05 now re-fetches only failed filings (`sec.retry_failed_filings`), recovering all 63. | Full-run verification. | §13.3, §17.3, hard rule 12 |
| D-069 | Full-run S05 throughput: downloads are latency-bound, so `sec.max_parse_workers` (default 8) caps the process count and each process gets `max_rps / processes`; the local run used 12 processes at 0.5 requests/s (6/s total, ~2.4 s per filing per process). On Windows a launcher script must import the package inside `if __name__ == "__main__"`, otherwise each spawned worker re-imports the whole pipeline (~1 GB each instead of ~100 MB). | Full-run verification. | §13.3, hard rule 13 |
| D-070 | **Full profile, Part 1 S05 result:** 500 firms, 202,607 filings (149,992 8-K, 39,373 10-Q, 13,242 10-K), 201,728 parsed with sections, 879 with no target section, 0 errors after the retry; 385,387 sections; SGML acceptance header read for 189,072 filings (the rest are pre-2003 filings without a usable header time: quality B/C, D-053); HF 10-K QA (200 filings) median token Jaccard: Item 1 0.95, 1A 1.00, 7 0.83, 7A 0.96. | Artifacts in `DATA_ROOT/full/S05/8029bf8ab662474f`. | §17.1 |
| D-071 | **Point-in-time defect found by the full run's availability assertion (S09):** a same-day group of 8-Ks is one event (§5.2) whose features use every filing in it (text, item categories, release EPS / SUE), but its clock started at the group's FIRST acceptance, so a later same-day 8-K's content leaked into a decision time before it was public (5,189 of 141,300 8-K events, 3.7%; the pilot had ~270). The assertion caught it through SUE; the text family's availability was defined at event level, so text could not trip it. Fix (conservative): the group's clock is its LAST acceptance (public, decision, session bucket, T_trade entry, Track A day 0 all follow it); `first_acceptance_time` is kept for diagnostics; event ids are unchanged. S07 now asserts that no filing in an event was accepted after the event's acceptance time. Unit test with a real 11:00 / 16:50 pair. Pilot results (engineering evidence only) predate this fix. | Hard rule 3; §5.1, §5.2, §6.3 #8. | §5.1, §5.2 |
| D-072 | **Hard-rule-7 defect found by the full run's SUE-rule assertion (S09):** S06 computes SUE per accession before clusters exist (S07), so when a firm filed its 10-Q minutes after the earnings 8-K on the same day, the 10-Q was public within the release's 35-minute decision window and fed the release's SUE history: a same-cluster source. S06 now excludes, for every release, all periodic filings of the firm filed on or after the release within the 120-day cluster window (`release_cluster_exclusions`, a conservative superset of the D-017 cluster), whatever their acceptance time; the S09 assertion stays as the independent check. | Hard rule 7; §5.6. | §5.6, §6.3 #11 |
| D-073 | **Drift-verdict audit of the full run (hard rule 10: a surprising result opens an audit).** SUE-conditioned drift came out at -3.0% (Q5-Q1, CAR4[+2,+60], n=11,885), opposite to classic PEAD. Checks: the SUE sign is right (CAR[-1,+1] rises monotonically from -0.69% in Q1 to +1.14% in Q5) and the reversal holds in every era (2000-08 -1.3%, 2009-16 -3.3%, 2017-26 -3.2%), so it is not a coding error; a survivor-only universe (D-057) biases low-SUE drift upwards and cannot be separated from a genuine large-cap reversal with these data. Two defects found and fixed: (1) `(p or 1) < 0.10` treated p = 0.0 as missing, so the one-session-lag check (-3.1%, t=-9.8) and the clean check were scored as failures (`robust_same_sign`, tested; unknown is None, never 'failed'); (2) overlap flags counted a release's own cluster 10-Q as contamination, leaving 130 of 46,447 releases 'clean'; cluster members are one information event (§5.2) and no longer flag each other (any other same-firm event still does). Never presented as a PEAD replication. | Full-run audit. | §5.7, §10.2, hard rule 10 |


In [ ]:
%%writefile $REPO/pyproject.toml
[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.build_meta"

[project]
name = "secprice"
version = "0.5.0"
description = "SEC filings -> price impact study (SPEC.md v5 + pre-coding addendum)"
requires-python = ">=3.10"
dependencies = [
  "numpy>=1.26", "pandas>=2.1", "pyarrow>=15", "pyyaml>=6", "requests>=2.31", "lxml>=5",
  "scikit-learn>=1.4", "scipy>=1.11", "statsmodels>=0.14", "lightgbm>=4.3", "catboost>=1.2", "optuna>=3.6",
  "pandas_market_calendars>=4.4", "yfinance>=0.2.40", "matplotlib>=3.8", "jsonschema>=4", "markdown>=3.5",
  "reportlab>=4", "arch>=6", "openpyxl>=3.1", "huggingface_hub>=0.23",
]

[project.optional-dependencies]
gpu = ["torch", "transformers>=4.48", "sentence-transformers>=3", "accelerate"]
llm = ["lm-format-enforcer>=0.10", "bitsandbytes"]
dev = ["pytest>=8", "nbformat", "nbclient", "ipykernel"]

[project.scripts]
secprice = "secprice.cli:main"

[tool.setuptools.packages.find]
where = ["src"]

[tool.pytest.ini_options]
testpaths = ["tests"]
markers = ["network: needs SEC / price network access (skipped offline)"]


In [ ]:
%%writefile $REPO/README.md
# sec-filings-price-impact

SEC filing text (10-K, 10-Q, 8-K) → price impact, built to `SPEC.md` (v5) and `SPEC_ADDENDUM.md`.
Three tracks: **A** event study (association), **B** prospective prediction (primary target T_trade h=1, 3-class),
**C** content validity (human gold labels).

## Quick start

**Colab (recommended for the full study).** Open `notebooks/SEC_Filings_Price_Impact_v5_EndToEnd.ipynb` in Colab, add the
secret `SEC_USER_AGENT` = `Your Name your.email@example.com`, check that `MyDrive/sec-price-study/inputs` holds the inputs,
and *Run all*. Profiles: `smoke` (default) → `pilot` → `full` (G2: pilot done + `owner_opt_in_full`).

**Local (CPU stages and tests, no credits).**
```bash
uv venv .venv --python 3.12 && uv pip install -e ".[dev,gpu]"
pytest                                   # unit + offline integration tests (gate G1); network tests are opt-in
export SEC_USER_AGENT="Your Name you@example.com" DATA_ROOT=/path/to/sec-price-study
secprice run --profile smoke --stages S00-S12 S20 S21
secprice prereg-freeze --profile smoke && secprice run --profile smoke --stages S13-S18 S22 S24
```

## Profiles
| Profile | Firms | Years | Purpose |
|---|---|---|---|
| smoke | 5 fixed (AAPL, MSFT, JNJ, XOM, JPM) | 2021-2023 | the chain on real data in under an hour on CPU |
| pilot | ~40 sector-balanced | 2012-2026 | full core pipeline, small samples; required before full (G2) |
| full | 500 | 2000-01 to 2026-07 | the study |

## Run order on Colab (Pro+)
Inputs already sit in `MyDrive/sec-price-study/inputs` (979 OHLCV files, frozen yfinance gap-fill, LM dictionary, HF census,
factor / S&P 500 / SIC caches). The owner waived the pilot gate (decision log D-058): G2 = opt-in (already set in the notebook)
+ the preflight budget check, which the notebook prints after S04 and S05 enforces before the download. Stage hashes do not
depend on the machine, so any stage already on Drive is skipped.

| Step | Runtime | Settings in the first code cell | Stages |
|---|---|---|---|
| 1. full Part 1 | CPU High-RAM | `RUN_PROFILE="full"`, `PART="cpu1"` | S00-S12, S20-S21: **already run** (artifacts on Drive), so this step only verifies and skips; from scratch ~15 h EDGAR at 6 req/s (resumable) + ~4 h S06-S11 |
| 2. full Part 2 | A100 | `PART="gpu"` | S13-S17: GPU preflight, embeddings, ChronoBERT vintages, L5/L7, G3, fine-tuning, RAG, distillation (~3-6 h) |
| 3. full Part 3 | CPU | `PART="cpu2"` | S18 dev holdout + MCS/PBO + G4 freeze, S22 backtest, S24 report (S19 stays sealed) |
| 4. lockbox | CPU/GPU | `OWNER_OVERRIDES["lockbox"]["unlock_locked_test"]=True` after reviewing S18 | S19, scored once; then S22 and S24 again |

Restart the runtime when switching between CPU and GPU. Fine-tuning runs only if G3 passes unless
`gates.owner_override_g3_finetune=True` (results then labelled EXPLORATORY). Track C (S23) needs the gold annotations
(`secprice gold export|import|agreement`).

## S&P 500 vs non-S&P 500 vs combined
The 500 firms are S&P 500 members (in the study window, pinned fja05680 list) first, then a seeded, sector-stratified fill of
non-members. Every result is reported for each bucket and combined (decision log D-056): composition, the combined models scored
per bucket (walk-forward, development holdout, locked test), bucket-only models against the pooled model on identical rows,
Track A CARs and CAR paths, SUE drift, and the backtest restricted to each bucket's names. The universe is **survivor-only**
(current listings): results are biased towards survivors (D-057).

## Pilot Part 1 results (superseded by the full run; predates D-066 to D-073)
40 sector-balanced S&P 500 firms, 2012-01 to 2026-07: 11,370 filings parsed (SGML acceptance header for every one), 10,105
events, all timestamp quality A; release-EPS parser agrees with XBRL on 79.8% of usable-confidence releases (45 probable full-year values withheld, D-052); leakage audit
passes and catches all 4 planted leaks.

| Track B, T_trade h=1, 3-class, walk-forward (2 folds, n = 2,871) | macro-F1 [95% block-bootstrap CI] |
|---|---|
| L1 price only (CatBoost, tuned) | 0.362 [0.342, 0.383] |
| L2 metadata (elastic net) | 0.396 [0.380, 0.412] |
| L4 text: lexical + change + novelty + TF-IDF (LightGBM, tuned) | 0.391 [0.373, 0.412] |
| L6 all CPU features (LightGBM, tuned, validation-selected) | 0.396 [0.378, 0.415] |
| Permutation null (10 within-month label shuffles) | mean 0.322; none reached 0.396 (p ≤ 0.09) |

Validation-selected, not a global optimum (the elastic-net certificate shows only that its convex loss reached the same
optimum from 3 starts). L6 does not beat L2 metadata alone (0.396 vs 0.396); training F1 0.85 against validation 0.40 flags
overfitting; adversarial AUC ≈ 1 reflects drift in filing vocabulary (covariate shift), not leakage (D-051). Track A: SUE
post-earnings drift Q5-Q1 = -2.5% [-4.7%, -0.6%], p = 0.10 (weak/mixed; opposite sign to classic PEAD in this small sample); text
beyond momentum not supported. Association is not causation; prediction is not profitability.

## Full profile, Part 1 (local CPU, 23-24 Oct 2026)
Run on the final code, real EDGAR data, 500 firms (S&P 500 members first, then non-members), Jan 2000 - Jul 2026; artifacts on
Drive at `MyDrive/sec-price-study/full` (Colab skips these stages and starts at Part 2).

| Stage | Result |
|---|---|
| S05 download + parse | 202,607 filings (149,992 8-K, 39,373 10-Q, 13,242 10-K); 201,728 parsed, 879 without a target section, 0 errors; 385,387 sections; HF 10-K QA median token Jaccard: Item 1 0.95, 1A 1.00, 7 0.83, 7A 0.96 |
| S06 accounting | 975,196 XBRL facts; 47,967 earnings releases, 34,661 with usable EPS, 82.7% agreement with XBRL (n=26,075); SUE for 59,490 filings |
| S07 events | 190,947 events (141,300 8-K, 37,135 10-Q, 12,512 10-K); 96% timestamp quality A (every year from 2003: 100%); usable cutoff 2026-07-31 |
| S09 / S10 | 130 features; 0 availability and 0 SUE-rule violations; leakage audit passes; 4/4 planted leaks caught; 6 walk-forward blocks 2008-2019, development holdout 2020-2022, locked test 2023 - Jul 2026 (sealed) |
| S11 baselines (walk-forward, T_trade h=1, n=89,376) | L6 (LightGBM, tuned) macro-F1 0.401 [0.396, 0.405] vs majority 0.189 and a permutation null of 0.265 (max of 10: 0.276); L2 metadata 0.377 [0.371, 0.384]; L1 price 0.368; L4 text 0.376; L3 accounting 0.356; L6 without filing-change features 0.401; elastic-net convexity certificate spread 1e-9 |
| S&P 500 vs non-S&P 500 | 353 / 147 firms, 137,541 / 53,406 events; non-members are smaller (median log market cap 21.4 vs 23.5) and more volatile; combined L6 scores 0.402 [0.397, 0.407] on S&P firms and 0.396 [0.389, 0.403] on the others (no detectable difference); pooling helps the non-S&P firms (pooled minus non-S&P-only model +0.008 [+0.000, +0.016], p=0.05) |
| S20 / S21 Track A | CAR[-1,+1] +0.10% (t 4.6) and abnormal log volume +0.31 (t 45.8) around disclosures; SUE-conditioned drift Q5-Q1 -3.0% [-3.7%, -2.3%] (n=11,885), verdict 'supported' but a REVERSAL, likely inflated by the survivor-only universe (non-S&P -4.7% vs S&P -2.4%); text beyond momentum weak/mixed |
| Report | `full/reports/results_report.html` (+ .pdf, .md): 35 figures; GPU-dependent sections (L5/L7/L8, H1, dev holdout, locked test, backtest, Track C) say 'not produced' until Part 2/3 run |

The full run found and fixed defects that smaller runs could not show (decision log D-068 to D-073), including two point-in-time
leaks caught by the pipeline's own assertions (same-day 8-K groups, same-day 10-Q in a release's SUE) and a drift-verdict bug.
The universe covers about 32% of the tickers that were S&P 500 members in 2000-2026 (survivor-only).

## Early-era check (2000-2004, real filings)
Six firms, 361 filings, 317 events, run on real EDGAR data before any full run (decision log D-052/D-053). It found and fixed two
defects (Exhibit 13 MD&A under "FINANCIAL REVIEW" titles; full-year EPS read as a quarter). Expect for 2000-2008: quality B/C
timestamps (no usable SGML acceptance time before ~2003), no SUE before ~2004-05, and a common cohort that is mostly 2009+.

## Long runs on Colab
`sec.s05_max_wall_hours` (notebook default 8) makes the EDGAR download stop cleanly and say so; run the same cell again and finished
firms are skipped. Dependency pins are in `requirements.lock`; all 26 ChronoBERT vintages are pinned to commit SHAs.

## Labelling (Track C)
S12 writes `gold_*_filings_template.csv` and `gold_passages_template.csv` + `ANNOTATION_GUIDE.md` into the S12 folder.
Complete them (gold-test blind; a second rater on ≥ 30% of gold-test), copy them to `DATA_ROOT/inputs/gold/`, and re-run
S12 → S16 → S17 → S23 with force.

## Where things are
`configs/` parameters, pre-registration, model ledger, 8-K taxonomy, question bank · `src/secprice/` stages and modules ·
`tests/` unit + integration + fixtures · `docs/decision_log.md` every decision and deviation · `tools/` input preparation
and the notebook builder.


In [ ]:
%%writefile $REPO/SPEC.md
# SEC Filings → Price Impact: Build-Ready Spec, v5

**For: Claude Code in VS Code.** This file is the single source of truth. Put it in the repository root as `SPEC.md`. It contains requirements only, **no code**.

**Deliverable:** a Python repository (package + configs + tests + two notebooks).
- **Local (VS Code):** run every CPU stage and every test on your own machine. That costs no Colab credits.
- **Colab Pro+ (GPU):** run only the GPU stages, through a runner notebook.
- **End-to-end option:** the whole pipeline can also run on Colab from the same runner.

**What v5 is:** v4's research design, corrected by a second independent review (the "professor review"). It is narrowed into a **core study** that must be completed and an **extension pack** that runs only when gates pass, and given the engineering detail Claude Code needs: repository layout, configuration, data contracts, stage interfaces, tests and a milestone plan.

---

## PART I: RESEARCH DESIGN

## 0. Read this first (Claude Code)

### 0.1 How to use this document
1. **Session start:**
   - Read the whole spec.
   - Create `CLAUDE.md` in the repo root: a one-page digest of Section 0.2 (hard rules), Section 13 (environment), Section 14 (repo layout) and the current milestone.
   - Keep `CLAUDE.md` updated at the end of every milestone.
2. **Milestones:** build in the order of Section 22. Each milestone has a "done when" list. Do not start the next milestone until it is satisfied and committed to git.
3. **Ambiguity:**
   - Prefer the more conservative choice for leakage and point-in-time issues.
   - Record the decision in `docs/decision_log.md`.
   - Never silently deviate from this spec.
4. **Spec changes:** if a rule is impossible to implement (an endpoint changed, a model is gone), log it, use the documented fallback, and tell the owner in the milestone summary.

### 0.2 Hard rules
1. **SEC filings only (10-K, 10-Q, 8-K) as text.** No news, social-media, transcript or analyst text. Never load the Hugging Face news repository, FNSPID, FinMultiTime news or StockNet tweets. Market prices are used only for outcomes and price features. Pretrained models whose authors used news-style text are allowed, with their provenance recorded.
2. **Three tracks stay separate** (Section 2): **A** event study (association), **B** prospective prediction, **C** content validity. They share cleaned data but not timestamps, targets or availability assumptions.
3. **Four timestamps per event** (Section 5.1). Features must be available **no later than** `decision_time`. Retrieval of *prior* documents uses strict `public_available_time < decision_time` of the current event, and always excludes the current filing and any later filing.
4. **`RUN_PROFILE = smoke` is the default.** `pilot` and `full` need an explicit change. `secprice preflight` must pass before any `full` run.
5. **Never fabricate** data, dates, results or metrics. The report contains only computed values.
6. **Association is not causation; prediction is not profitability.**
7. **The earnings announcement return (EAR) is never a feature for its own event or its earnings cluster** (Section 5.6). **SUE** uses only the event's own filing/exhibit or *earlier* public accessions. A later 10-Q/10-K of the same cluster can never feed an earnings-8-K event.
8. **8-Ks are event-driven:** a company-year without an 8-K is not missing data.
9. **Never choose documents or events using later price moves.**
10. **High scores trigger an audit, not termination.** A pre-event window is a diagnostic, not a required zero.
11. **Model provenance decides claims.** Only models with documented cutoffs before the evaluated events may appear on the **strict point-in-time leaderboard**. Unknown-cutoff models are **retrospective only** and can never support a "clean test" claim (Section 6.4).
12. **Verify before use.** Every model, dataset, endpoint and library is checked at build time; exact revisions are pinned; substitutions are logged.
13. **SEC fair access:**
    - declared User-Agent (name + email) read from an environment variable or Colab secret, never hard-coded
    - ≤ `sec.max_rps` (default 6; SEC's ceiling is 10)
    - exponential backoff on 429 / 5xx
    - no proxy rotation or evasion
    - pause and report if blocked
14. **No paid or hosted model APIs.** All models run locally or on Colab.
15. **Reproducibility:**
    - seeds fixed
    - config hash and git commit stamped on every artifact
    - an environment manifest
    - pinned dependencies (do not reinstall Colab's own torch without reason)
16. **Tests are part of the deliverable** (Section 18). Network-dependent tests are marked and skipped offline; unit tests run on fixtures.

---

## 1. What changed from v4

| Change | Why |
|---|---|
| **Default parameters set:** `N_STOCKS = 500` unique firms, `STUDY_START = 2000-01-01`, `STUDY_END = 2026-10-31` (usable cutoff = last observed price/document date at runtime), market warm-up from 1998 | The original intent; still overridable |
| **Four timestamps:** acceptance, public availability, feature-ready, decision | SEC states filings typically appear 1–3 minutes after the EDGAR timestamp, sometimes longer; acceptance ≠ readable time |
| **Intraday tradable target fixed:** every T_trade entry is at the **open of the first session starting after `decision_time`** | A same-day close fill is not guaranteed with daily data |
| **SUE timing made cluster-aware:** earnings-8-K events use EPS parsed from the release exhibit; XBRL from a later 10-Q is forbidden for that event and used only for quality checks or labelled ex-post sensitivity | Closes a subtle leak the review identified |
| **Gold set split** into gold-dev / gold-val / gold-test | Selection and final reporting must not share labels |
| **Strict point-in-time vs retrospective leaderboards**; no clean-test claim for unknown cutoffs | Honest temporal claims |
| **Common-cohort vs maximum-available comparisons** | Lift must not come from a different event set |
| **Narrowed core** (4 encoders, 3 retrieval setups, 1 LLM, lexical and structured baselines); everything else becomes a **gated extension pack** | A solo, credit-limited project must finish its core |
| **Generation metrics moved to an optional sub-study**; core content metrics are field P/R/F1, numeric fidelity, citation accuracy, faithfulness, retrieval metrics | Matches what the core study claims |
| **Concrete backtest:** signal score, ranking rule, minimum names, overlapping-position policy, execution per bucket, borrow assumptions | Removes ambiguity |
| **"Economically trivial" pre-defined in basis points** | Inference rule fixed before results |
| **Engineering spec:** repository, config, data contracts, CLI, artifacts, tests, milestones; CPU locally in VS Code, GPU on Colab | Makes the build executable end to end and saves credits |

Appendix A has the item-by-item reconciliation of the professor review. Appendix B summarises the earlier review's reconciliation (still in force).

---

## 2. Research question, tracks and hypotheses

**Main contribution (core):** *Do SEC filing text and evidence-grounded event features add reliable out-of-time information beyond point-in-time market, filing-metadata, accounting and filing-change baselines?*

**Tracks**
- **Track A (event study):** what happens around disclosures: abnormal returns, volume and volatility by form, section and 8-K category, with clean-event and pre-trend sensitivity, plus the drift tests. Association only.
- **Track B (prediction):** what could have been predicted at `decision_time`: the common-cohort ablation ladder, walk-forward tuning, a development holdout, a locked out-of-time test, strict versus retrospective leaderboards and one concrete backtest.
- **Track C (content validity):** are sections, item mappings, extracted events and numbers correct? Measured against human labels on held-out gold-test data, never against price outcomes.

**Core hypotheses** (pre-registered in `configs/preregistration.yaml`; its hash is recorded before any development-holdout scoring):
- **H1:** SEC text adds out-of-time predictive value beyond price, metadata and accounting baselines.
- **H2:** Changes in filing language add more value than absolute sentiment alone.
- **H3:** Validated structured event extraction adds value beyond frozen embeddings and lexical features.
- **H4:** Strict point-in-time model vintages perform differently from retrospective static checkpoints; the size of the gap estimates look-ahead sensitivity.

**Extension hypotheses** (tested only if extensions run):
- **H5:** A stacked multi-tower ensemble beats the best single model.
- **H6:** Richer RAG designs beat hybrid + rerank.
- **H7:** Analogue retrieval or time-series models add value.
- **H8:** Gains shrink under strict timestamps, costs, clean-event filtering and the locked test. This is evaluated for the core too.

**Primary task:** 3-class direction (Up / Sideways / Down) of **T_info** abnormal return at h = 1. **Primary metric:** mean walk-forward macro-F1, with guardrails (log-loss no worse than the class prior; calibration error below threshold). Secondary metrics are listed in Section 9.

**Expectations, stated in the report:**
- Low signal-to-noise is normal; small lifts are realistic, and a credible null result is a valid result.
- Volatility and absolute-move targets usually show more skill than direction.

---

## 3. Parameters and defaults

| Parameter | Default | Notes |
|---|---|---|
| `run_profile` | `smoke` | `pilot` and `full` by explicit edit |
| `universe.n_stocks` | 500 | Unique firms across the study. Report firms per year separately; never claim 500 per year unless verified |
| `universe.source` | historical S&P 500 membership | Candidate public source: the `fja05680/sp500` repo (MIT; history from 1996; built from Wikipedia changes, which its author calls incomplete; early years may miss symbols). Verify it; a licensed point-in-time list can replace it |
| `study.start` / `study.end` | 2000-01-01 / 2026-10-31 | Usable cutoff = min(last observed price date, last document date). If the run is before 31 Oct 2026, mark the month partial |
| `market.warmup_start` | 1998-01-01 | ≥ 250 sessions before the first event, for beta and volatility |
| `text.predecessor_lookback` | 2 years before `study.start` | Predecessor filings loaded only to compute change features for early events; they are never evaluated |
| `timing.public_lag_minutes` | 5 (sensitivity 0, 15, 60) | public_available = acceptance + lag |
| `timing.decision_latency_minutes` | 30 (sensitivity 10, 120) | decision = public_available + latency. One value for all models keeps the cohort and targets identical |
| `targets.horizons` | [1, 2, 5, 20, 60] | Sessions |
| `targets.primary` | T_info, h = 1, 3-class | Pre-registered |
| `labels.k` | 0.5 (sensitivity 0.25, 0.75) | Sideways band = ±k·σ·√h |
| `contam_window_sessions` | max horizon in the analysis | Clean-event filter |
| `splits.min_train_years` / `val_block_years` / `dev_years` | 8 / 2 / 3 | Walk-forward |
| `splits.locked_test_start` | 2023-01-01 | After SEC-BERT's 2019 cutoff and the 2020–22 regime; overridable |
| `splits.embargo_days` | 90 | Calendar days |
| `trivial_effect_bps` | 10 for h ≤ 2; 25 for h = 5; 50 for h ≥ 20 | Pre-registered "economically trivial" thresholds |
| `fdr.q` | 0.05 | Benjamini-Hochberg within pre-declared families |
| `bootstrap.n` | 1000 (smoke 100) | Block bootstrap by month |
| `gates.text_lift` | 0.01 macro-F1, CI excludes 0 | Gate G3 |
| `gates.detector_min_precision` | 0.80 | Below this, a detector is provisional |
| `audit.auc_trigger` / `audit.f1_lift_trigger` | 0.65 / 0.10 | Review triggers |
| `budget.*` | per-part GPU hours, storage GB, maximum error rate, LLM sample sizes | Section 13.3 |
| `extensions.*` | all `false` | Section 7.3 |

**Profiles**
- `smoke`: 5 firms, 2021–2023, ~10 filings per firm-form, one encoder, LLM on ≤ 20 filings, fixtures where possible; must finish on a laptop CPU, except the optional GPU smoke.
- `pilot`: ~40 sector-balanced firms, 2012 onward, the full core pipeline with small samples.
- `full`: the defaults above.

---

## 4. SWOT (v5, condensed)

**Strengths**
- Auditable primary-source event panel.
- Separate research tracks.
- Point-in-time accounting.
- A strong common-cohort baseline ladder.
- Vintage models for temporal provenance.
- Reproducible stage artifacts and tests.

**Weaknesses**
- Daily bars only.
- The S&P-based universe is survivorship-aware, not survivorship-free.
- Free prices are weak on delistings.
- Selected sections only.
- Single annotator unless a second is found.
- SUE coverage is limited before the XBRL era.

**Opportunities**
- Filing-change and novelty signals.
- Validated structured extraction.
- Measuring temporal provenance (look-ahead) effects.
- A reusable dataset.
- An honest null or positive result suitable for a capstone or workshop paper.
- University data (CRSP / Compustat / IBES) could later remove the survivorship and consensus limits.

**Threats**
- Timestamp mis-specification.
- Pretrained-model memory.
- LLM extraction errors.
- Multiple testing.
- Price-data gaps.
- Colab variability.
- SEC throttling.
- Over-building before the core works. Mitigated by milestones and gates.

---

## 5. Event clock, event types and targets

### 5.1 Four timestamps (all stored in US Eastern with timezone; trading calendar = NYSE, including early-close days)
| Field | Definition |
|---|---|
| `acceptance_time` | EDGAR acceptance date-time. **Verify timezone semantics** by comparing the submissions API value with the `ACCEPTANCE-DATETIME` in the filing's SGML header on a sample (unit test); store the raw string too |
| `public_available_time` | Exact if observed; otherwise `acceptance_time + public_lag_minutes`. Quality flag records which |
| `feature_ready_time` | `public_available_time + decision_latency_minutes` (assumed processing time) |
| `decision_time` | `feature_ready_time`. The prediction is made here. The current filing's own text is a legitimate input at this time |

- **Timestamp quality:** A = exact acceptance time; B = date only (assume after the close: acceptance 17:30 ET); C = derived or uncertain.
- **Same-day ordering:** `acceptance_time`, then accession number.
- **Session bucket** (from `acceptance_time` against the actual session open/close, including early closes): PRE / INTRA / POST / NONTRADING / UNKNOWN.

### 5.2 Event construction and types
- **Periodic events:** 10-K (+405 / KT), 10-Q. Amendments are kept for the audit, not used as events.
- **8-K groups:** same-firm, same-day 8-Ks merge into one multi-label event, keeping every item code and category.
- **Earnings types:**
  - `earnings_announcement_verified`: 8-K Item 2.02 (or a legacy equivalent) whose exhibit content is confirmed as a results release, with usable timing
  - `periodic_filing_only`: a 10-Q/10-K with no earlier verified release
  - `earnings_date_ambiguous`: conflicting or weak sources

  PEAD analysis uses the verified type only.
- **`cluster_id`:** links a verified release with its later 10-Q/10-K. A cluster is never treated as independent events in inference.
- **`event_id`:** a stable hash of (CIK, earliest accession in the group).

### 5.3 Track A clock
- **Day 0:** the session in which `public_available_time` falls if that is before the close; otherwise the next session.
- **Windows:** CAR[−1,+1], [0,+1], [0,+5], [+2,+20], [+2,+60]; pre-event [−20,−2] and [−5,−1].
- **INTRA:** day-0 returns are labelled a *mixed pre/post interval*.
- **Reporting:** results are reported by bucket.

### 5.4 Track B targets
| Target | Base | Exit | Eligibility |
|---|---|---|---|
| **T_info** (information response; primary) | Last regular-session close **before** `acceptance_time` | Close of the h-th session whose close is after `public_available_time` | PRE, POST, NONTRADING only. **INTRA excluded.** The pre-publication window (base → acceptance) contains no regular session; its length is logged |
| **T_trade** (tradable) | **Open of the first session whose open is at or after `decision_time`** (all buckets) | Close of the h-th session counting the entry session as 1 (h = 1 = entry-day open → close) | All buckets. Same economic holding interval for every bucket |

Sensitivity: entry one session later; `decision_latency` at 10 and 120 minutes; `public_lag` at 0, 15 and 60 minutes.

### 5.5 Returns, abnormal returns, labels
- **Series:** raw close, split-adjusted and total-return series kept separately. Outcomes use total return; dividend analyses also report raw price.
- **Abnormal returns:**
  - market model against SPY, beta on sessions −250 to −30 before the base (≥ 120 observations)
  - robustness: **Carhart 4-factor** (market, SMB, HML, momentum) primary; **FF5 + momentum** secondary (Kenneth French daily factors)
- **3-class label:**
  - Up if AR > +k·σ·√h; Down if AR < −k·σ·√h; otherwise Sideways
  - σ = point-in-time idiosyncratic daily volatility from the estimation window
- **Regression targets:**
  - AR
  - volatility ratio: log of realized vol over sessions 1–20 divided by sessions −60 to −5
  - standardized |AR|
- **Clipping:** ML clip bounds are estimated on training folds only; un-clipped test metrics are also reported. Track A CARs are never silently winsorized (a winsorized version is a labelled robustness check).
- **Censoring:** per horizon, drop events whose exit date is after the usable cutoff.
- **Weights:** label-uniqueness weights for same-firm overlapping windows; unweighted results also reported.

### 5.6 Earnings surprise (SUE) and EAR
- **SUE formula:** (EPS_q − EPS_{q−4}) / σ(ΔEPS), with σ from the prior 8 seasonal differences (minimum 4), each value as **originally reported** (the earliest accession that reported that period).
- **EPS source by event type:**
  - **Earnings-8-K events:** EPS_q parsed from the release exhibit (EX-99.x tables; diluted GAAP EPS preferred, basic as fallback, flagged). History EPS_{q−k} comes from XBRL company facts or prior releases with `public_available_time < decision_time`.
  - **10-Q/10-K events:** EPS_q from the event's own XBRL (quarterly derived from year-to-date by within-year differencing; units checked).
  - **Never** a later accession from the same cluster.
- **QA:** the parsed release EPS is compared with the later XBRL value (QA metric only). Coverage of `SUE_release` and `SUE_xbrl` is reported by year and size.
- **EAR:** AR[−1,+1] around a verified release.
  - It is a **Track A outcome** and a Track A drift sort variable (drift measured from +2).
  - As a Track B feature, it is allowed only for a **later, different cluster** whose `decision_time` is after the EAR window closes.
  - The feature registry enforces this, and a planted-leak test checks it.
- **Naming:** "SUE-conditioned post-earnings drift", never a consensus-based PEAD replication.

### 5.7 Overlaps and clean events
- **Flags:** an event is flagged if another qualifying same-firm event falls within `contam_window_sessions` after its base (defined from dates only).
- **Analyses:** two versions, **all-event** (clustered errors) and **clean-event**. Report events removed and whether conclusions change.

---

## 6. Splits, leakage control and provenance

### 6.1 Layout
- **Blocks:** walk-forward expanding folds (validation blocks of 2 years) → development holdout (2020–2022 by default) → locked test (2023-01-01 → usable cutoff).
- **Purge:** any training event whose T_info or T_trade window reaches the next block.
- **Embargo:** 90 days.
- **Stress tests:** firm-held-out and sector-held-out folds before the holdout.
- **Lockbox:**
  - the locked test is scored **once**, after the G4 manifest freeze (hash), by `secprice unlock-test`
  - retraining on all pre-test data happens before scoring
  - if the lockbox is opened early, it is relabelled a development holdout

### 6.2 Cohorts (every comparison declares one)
- **Common cohort (primary):** events eligible for *every* compared model (including SUE availability where accounting features are in the comparison).
- **Maximum-available (secondary):** each model on all events it can score. Clearly labelled as not like-for-like; sample differences reported.

### 6.3 Leakage checklist (each has an assertion in `splits/leakage` and a line in `leakage_audit.csv`)
Same 24 items as v4, updated:
1. Overlapping windows across blocks → purge + embargo.
2. Same-date shocks → date-disjoint splits.
3. Duplicate or near-duplicate filings → split by accession; hashes.
4. Firm memorization → anonymization; firm-held-out test.
5. Preprocessing fitted on future data (scalers, vocabularies, TF-IDF / BM25 statistics, topics, imputers, calibration, thresholds, clip bounds) → fitted on training indices only.
6. Test peeking → pre-registration hash; G4 freeze; lockbox command.
7. Intraday / pre-publication contamination → T_info eligibility; logged windows.
8. Timestamp errors → four-timestamp model; quality flags; sensitivity.
9. EAR misuse → registry rule + planted leak.
10. Earnings-date fabrication → three types.
11. **SUE from a later cluster accession** → registry rule + planted leak.
12. XBRL ordering → accession / `public_available_time` ordering.
13. Same-firm overlaps → flags + clean cohort.
14. Survivorship → universe table + wording.
15. Right-censoring → per horizon.
16. **Retrieval leakage** → the historical index returns only documents with `public_available_time < decision_time` and never the current filing; current-document evidence travels on a separate path.
17. Analogue outcomes (extension) → neighbour windows closed before `decision_time`.
18. Pretrained-model look-ahead → provenance rules (6.4).
19. Distillation / continued pretraining → training-period data only.
20. Stacking → walk-forward out-of-fold predictions only.
21. Outcome-based selection → assertion that event construction reads no price fields.
22. Data snooping → configuration counter; FDR; MCS / PBO as diagnostics.
23. Covariate shift → adversarial validation (an AUC near 0.5 does **not** prove a split is unbiased).
24. Broken audit → **planted-leak self-tests**: future-return feature, same-event EAR, later-cluster SUE, future-document retrieval. All must be caught.

### 6.4 Provenance leaderboards
- **Ledger** (`configs/models.yaml` + `reports/model_provenance_ledger.csv`): model ID, revision, card URL, license, training corpus, stated cutoff (or `unknown`), fine-tune data, task and labels.
- **Strict point-in-time leaderboard:** includes a model only if its documented cutoff precedes **every** event in the evaluated block:
  - vintage-matched ChronoBERT (an event in year Y uses vintage Y−1)
  - models trained from scratch on our folds (TF-IDF, GBDT, heads)
  - lexical / structured baselines
  - SEC-BERT only for blocks after 2019-12-31
- **Retrospective leaderboard:** all models, including unknown or later cutoffs.
- **Clean-test subset:** for a known-cutoff model, the locked-test events after its cutoff. **Unknown-cutoff models get no clean-test claim.**
- **Per-fold label:** every fold is marked strict or retrospective for each model.
- **H4:** vintage versus static comparison on identical rows and heads.

### 6.5 Review triggers and sanity experiments
- **Review triggers:** locked AUC > `audit.auc_trigger`, F1 lift > `audit.f1_lift_trigger`, or R² unusually high against the expanding-mean baseline.
  - **Effect:** open the audit checklist (chronology, target overlap, source fields, timestamps, duplicates, class balance, baselines, retrieval filters, provenance); outcomes go to the decision log.
  - **Not a stop:** they never terminate the run, and a positive R² alone is not evidence of leakage.
- **Sanity experiments:**
  - **Random-split demo:** pedagogical only.
  - **Permutation test:** within date blocks, class-stratified.
  - **Pre-event windows:** pre-trend diagnostics, interpreted rather than forced to zero.

---

## 7. Models

### 7.1 Core roster (default on)
| Role | Model (verify and pin) | Why |
|---|---|---|
| SEC-domain encoder | `nlpaueb/sec-bert-base` | 10-Ks 1993–2019; CC-BY-SA-4.0; strict after 2019 |
| Financial-tone encoder | `yiyanghkust/finbert-tone` | Pretrained on 10-K/10-Q, earnings calls and analyst reports |
| Point-in-time encoder | `manelalab/chrono-bert-v1-YYYY1231` (1999–2024 vintages; 1,024-token context) | Strict comparator for H4; verify each vintage used |
| Modern embedder | `BAAI/bge-base-en-v1.5` (512 tokens) | General baseline; also the dense retriever |
| Reranker | `BAAI/bge-reranker-base` (or v2-m3, verify) | Hybrid + rerank |
| Extraction LLM | `Qwen/Qwen2.5-7B-Instruct` (Apache-2.0, not gated) | One local generator; cutoff not precisely documented, so retrospective only |
| Distillation student | `answerdotai/ModernBERT-base` or the best core encoder | Applies extraction to the full corpus cheaply |
| NER (limited) | `dslim/bert-base-NER` | PER / ORG only, for anonymization; **never money** |
| Tabular learners | LightGBM, CatBoost, ElasticNet-logistic | All ladder rungs |
| Lexical | Loughran-McDonald dictionary (Notre Dame SRAF; academic use; verify terms), TF-IDF, BM25 | Baselines |

### 7.2 Encoder protocol
- **One model in memory at a time:** load → embed → persist float16 + index → unload.
- **Chunking:** section-aware chunks within the context limit; mean / max / attention pooling to section and document vectors; change vectors versus the prior same section.
- **Comparison:** all candidates use the **same event rows, labels, splits, head families, tuning budget and metric**.
- **Native outputs:** sentiment and tone scores are features, scored natively only against human labels.
- **Fine-tuning:** top 2 encoders by validation, and only if G3 passes (fixed grid, Section 9.4).

### 7.3 Extension pack (default off; each needs its flag + gate G3c + budget)
| Flag | Contents |
|---|---|
| `ext_encoders` | FinBERT (Prosus), FinBERT-pretrain, FLANG-BERT / RoBERTa, Financial-RoBERTa-large, DeBERTa-v3 finance, DistilRoBERTa finance, ModernBERT-large, FinModernBERT-embed, Qwen3-Embedding-0.6B, MPNet / MiniLM, `sec-bert-num` / `shape`, Longformer (only if chunking underperforms) |
| `ext_feature_models` | FinBERT-FLS, FinBERT-ESG, zero-shot DeBERTa-v3-large (only on ambiguous / `OTHER_OR_UNMAPPED` cases) |
| `ext_llms` | One challenger if core extraction misses G3b: `meta-llama/Llama-3.1-8B-Instruct` or `instruction-pretrain/finance-Llama3-8B`; further: Sujet-Finance-8B, Fin-R1, FinGPT adapters, FinMA (low). InvestLM only with an 80 GB GPU |
| `ext_rag` | HyDE, firm-memory RAG, CRAG (no web fallback), multi-hop, adaptive router, table-aware "multimodal" RAG; small-slice only: Self-RAG, GraphRAG, Agentic |
| `ext_analogue` | Point-in-time analogue retrieval (closed-window neighbours) |
| `ext_timeseries` | FinText Chronos vintages (strict); Chronos-2 / Bolt, TimesFM (retrospective); PatchTST, LSTM, N-BEATS, TFT hybrid |
| `ext_stacking` | Multi-tower stacking, Caruana ensemble, successive-halving architecture search |
| `ext_heavy` | QLoRA classifier, continued masked-language pretraining on training-period filings |
| `ext_generation_metrics` | Summary sub-study with ROUGE-N/L, BLEU, NIST, METEOR, chrF/chrF++, BERTScore, BLEURT, MoverScore, perplexity (needs human reference summaries) |
| `ext_data_modules` | Form 4, 13F, user-supplied licensed files (analyst, CRSP delistings) |
| `ext_app` | Gradio filing explorer |

**Excluded (unchanged from v3/v4):** BloombergGPT, BB-BERT, FinLlama (no public weights found), SecBERT / SecRoBERTa (cybersecurity), Qwen-Coder-Finance, Llama-2-Banking, Command R+, FinLLaVA, PRAGMA, FinMultiTime / FNSPID (datasets with news), StockNet (social media), FinDeBERTa (headline model).

---

## 8. Core RAG and extraction (Tracks C → B)

**Question bank** (`configs/question_bank_v1.yaml`, pre-registered; ~12 core fields):
- guidance direction (initiated / raised / lowered / withdrawn / reaffirmed)
- results versus prior-year tone (beat / miss language)
- impairment or write-down
- restructuring or layoffs
- liquidity or going-concern stress
- new financing or covenant issue
- M&A or disposition
- dividend change and type
- buyback
- litigation or regulatory action
- management change
- new or removed risk factors

Each field has a schema:
- `present`
- `direction`
- `magnitude` + unit (or null)
- `certainty`
- `evidence` (chunk IDs + character offsets)

**Retrieval setups (core tournament):**
1. BM25
2. Hybrid BM25 + bge-base dense, reciprocal-rank fusion, metadata filters
3. Hybrid + cross-encoder rerank

**Two evidence paths:**
- **Current-document path:** chunks of the filing being processed, allowed at `decision_time`.
- **Historical path:** the firm's prior filings with `public_available_time < decision_time`, used only for change questions (for example, "new risk factor").

**Pipeline:**
1. retrieve (top-k)
2. Qwen2.5-7B-Instruct with constrained JSON decoding
3. guardrails:
   - schema validation and retry
   - numeric cross-check against XBRL / source
   - faithfulness gate: an unsupported field becomes null
   - citation offsets verified
4. response cache keyed by model revision, prompt hash and passage hash

**Selection protocol:**
1. Retrieval-only metrics on **gold-val**.
2. End-to-end extraction F1 and faithfulness on **gold-val**.
3. Winner frozen (G3b).
4. Final content-validity numbers reported on **gold-test** only.

**Distillation:**
- **Teacher:** the winner extracts for a stratified teacher sample (`budget.llm_teacher_filings`, e.g. 3,000) from **training-period** filings.
- **Student:** a multi-task student is trained on those outputs and applied to all events.
- **Report:** student versus teacher (agreement) and student versus human (gold-test).

---

## 9. Evaluation (Track B)

### 9.1 Ablation ladder (same folds; common cohort primary)
| Rung | Features |
|---|---|
| L0 | Majority / class prior |
| L1 | Price only (prior returns, volatility, volume / liquidity, market regime) |
| L2 | Metadata only (form, 8-K categories, section availability, industry, year, bucket, timestamp quality) |
| L3 | Accounting only (point-in-time SUE + XBRL facts) |
| L4 | Lexical: Loughran-McDonald + TF-IDF + filing-change + novelty |
| L5 | Frozen embeddings only (per core encoder) |
| L6 | L1 + L2 + L3 + L4 (best valid structured set) |
| L7 | L6 + embeddings |
| L8 | L7 + distilled RAG-extraction features; + fine-tuned encoder if G3 passes |

The core evidence is the **lift of L7 / L8 over L6** with CIs, on strict and retrospective leaderboards. H2 compares change features against absolute sentiment inside L4/L7; H3 compares L8 against L7.

### 9.2 Metrics (core)
**Classification**
- Accuracy, balanced accuracy.
- Per-class and **macro** P / R / F1; weighted F1.
- ROC-AUC (one-vs-rest macro and per class), PR-AUC.
- Cohen's kappa, Matthews correlation.
- Log-loss, Brier, expected calibration error and reliability curves.
- Confusion matrices.
- Accuracy and F1 at pre-registered coverage (100 / 50 / 25 / 10%), and the coverage curve.
- **Null with a reason** when a metric is inapplicable.

**Regression**
- R² and **out-of-sample R² versus the expanding historical mean**, MAE, RMSE.
- Spearman, monthly rank IC (t-statistic, ICIR), directional accuracy.
- QLIKE for volatility.

**Content (Track C, gold-test)**
- Field-level P / R / F1.
- Numeric exact match and tolerance match (source and XBRL).
- Citation-offset accuracy.
- Faithfulness: local judge (not the generator), calibrated on ~100 human ratings, kappa reported.
- Retrieval Recall@k, nDCG@k, MRR, MAP on held-out qrels.
- Section-parse accuracy; 8-K mapping accuracy; detector P / R / F1.
- Cost per filing and latency.

**Uncertainty**
- Block bootstrap by month (clustered by firm where relevant).
- Paired tests for model comparisons.
- BH-FDR within families.
- Counts (events, firms, dates, overlaps) beside every result.
- Power language:
  - n < 30 → **warning**
  - 30–99 → **exploratory**
  - a wide CI → **inconclusive**

**Selection diagnostics**
- Model Confidence Set and PBO (combinatorially symmetric cross-validation) as **diagnostics only**. They never replace or rescue walk-forward and locked-test evidence.
- Each diagnostic documents which loss streams and configurations enter it.

### 9.3 Fit diagnosis
- **Outputs:** loss curves (train / validation per step and epoch), validation curves, learning curves.
- **Verdicts:**
  - **Overfit:** gap > 0.10, or validation loss rising 2 epochs while train loss falls.
  - **Underfit:** within 0.02 of majority (this may mean weak signal).
  - **Unstable:** fold std > 0.03.
- **Wording:** results are "validation-selected", never a global optimum.

### 9.4 Tuning
- **Heads:** Optuna (TPE sampler, median pruner, SQLite study). Search spaces as in v4 §14.1, fixed before the holdout is unlocked.
- **Fine-tuning grid (6–12 trials):** learning rate {1e-5, 2e-5, 3e-5} × batch {8, 16} × epochs ≤ 3 with early stopping; weight decay {0, 0.01}.
- **Losses:**
  - class-weighted cross-entropy (focal loss only if it wins on validation)
  - Huber for regression
  - weighted sum for multi-task
- **Before/after:** default and tuned results are reported for every model.

---

## 10. Track A and drift tests

### 10.1 Event-study analyses
- **Analyses:**
  - CAR, abnormal volume and volatility response by form, section, 8-K category (taxonomy versioned, with availability by year) and extracted field
  - all-event and clean-event versions
  - pre-trend windows
  - by bucket
  - correlations (filing ↔ filing tone persistence, release versus later 10-Q, filing ↔ price Spearman / partial)
  - topics fitted on training years only
- **Errors:** clustered by firm and date.
- **Factor alphas:** Carhart primary.

### 10.2 Drift verdicts
**Labels:**
- `supported`
- `weak/mixed`
- `not supported`
- `inconclusive/low power`
- `not testable in scope`

**Supported** requires:
- consistent definitions
- adequate coverage
- an FDR-adjusted p < 0.05 **and** a CI excluding the pre-registered trivial threshold
- the same sign in ≥ 2 of 3 subperiods
- clean-event robustness
- survival under 10 / 20 bps costs and a one-session lag

**Failing one test never proves an anomaly is absent.**

| Effect | Status | Design |
|---|---|---|
| SUE-conditioned post-earnings drift | Testable (qualified, exploratory) | Verified releases only. Primary sort `SUE_release`; sensitivity `SUE_xbrl` (labelled ex-post, not tradable) and EAR (observed before +2). Drift CAR[+2,+60] |
| Short-term momentum | Testable (benchmark) | 1-month reversal, 2–12 and 12–1. Does text add value beyond it? Never attributed to filings |
| Post-dividend announcement drift | Partial, only if validated counts give adequate power | Announcement ≠ ex-date; raw versus total return |
| Spin-off / restructuring drift | Partial, same condition | 8-K 2.05 / 2.06 / 2.01 + validated spin-off detection; announcement / completion / distribution dates distinct; parent-only unless new tickers are mapped |
| Analyst revisions drift | **Not testable** | A management-guidance proxy is a different construct, named as such |
| Insider trading drift | **Not testable (core)** | Form 4 extension |
| Institutional ownership drift | **Not testable (core)** | 13F extension |

---

## 11. Backtest (Track B economic check)

1. **Signal:** s = P(Up) − P(Down) from the frozen model (regression variant: predicted AR). Computed at `decision_time`.
2. **Thresholds:** long if s ≥ the training-period 80th percentile of s; short if ≤ the 20th percentile. Percentiles come from training folds only.
3. **Execution:** enter at the **open of the first session at or after `decision_time`** (same rule for every bucket); exit at the close of session h (pre-registered h, default 5); sensitivity: entry one session later.
4. **Portfolio:**
   - calendar-time daily portfolio of open positions
   - equal weight within each leg
   - maximum 5% per name
   - one position per firm (a newer event replaces the older one)
   - a leg with fewer than `min_names_per_side` (default 10) positions holds cash for that leg
5. **Costs:**
   - 10 and 20 bps per side
   - hypothetical borrow cost 50 bps per year on shorts (configurable)
   - results reported **long-only versus SPY** and **long-short (hypothetical)**
   - no claim of short availability or capacity
6. **Corporate events:**
   - halts → hold to the next price
   - delistings / mergers → delisting proceeds if the source has them, otherwise the last price, flagged
   - spin-offs → parent total-return series, flagged
7. **Metrics:** return, volatility, Sharpe, **Deflated Sharpe**, Carhart alpha, turnover, maximum drawdown, exposure counts.
8. **Wording:** an economic-relevance check, not an investable strategy, given unknowns about borrow, capacity and delisting data.

---

## 12. Gold set (Track C)

Sampled from **training-period** filings, stratified by form, era (2000–08 / 2009–19 / 2020+, using training years only), category and sector; rare categories deliberately oversampled.

| Split | Size (recommended) | Use |
|---|---|---|
| gold-dev | ~50 filings | Schema and prompt iteration, debugging |
| gold-val | ~100 filings + ~200 passages | RAG / LLM / detector selection and thresholds |
| gold-test | ~100 filings + ~200 passages | **Untouched** final content-validity numbers |

**Annotation tasks**
- **Filings:** question-bank fields with evidence spans (extraction gold + retrieval qrels); section boundaries; 8-K category check.
- **Passages:** does this passage support the assigned label? (detectors, categories, zero-shot extension).
- **Faithfulness:** ~100 judge-calibration ratings drawn from gold-val outputs.

**Second rater**
- A second annotator independently labels a stratified ≥ 30% of gold-test, with positives and negatives for every important category; Cohen's kappa + adjudication reported.
- Without a second rater, all content-validity findings are labelled **single-rater, provisional**.

**Pre-fill and blind labelling**
- LLM pre-fill is allowed for gold-dev and gold-val only.
- **gold-test is labelled blind** (no pre-fill), and anchoring risk is noted.

**Tooling**
- The notebook / CLI exports CSV/Excel templates + an annotation guide and imports completed files with validation.
- The pipeline runs without labels; dependent results are then marked *exploratory / unvalidated*.

**Rule:** price outcomes never validate semantic labels.

---

## PART II: ENGINEERING SPEC

## 13. Execution environment

### 13.1 Where things run
| Work | Where | Why |
|---|---|---|
| Development, unit tests, smoke run | Local VS Code (CPU) | Free; fast iteration |
| EDGAR download and parse, prices, events, features (no GPU), baselines, Track A, report | **Local** (recommended) or Colab CPU | No credits used; SEC throttling makes this wall-clock bound, not compute bound |
| Encoder embeddings and fine-tuning, LLM extraction, student training | **Colab Pro+ GPU** via `notebooks/colab_gpu_runner.ipynb` | Only GPU work spends credits |

### 13.2 Data movement
- **Two roots:** artifacts live under `DATA_ROOT` (environment variable). Local: a project data folder. Colab: a Drive folder (`/content/drive/MyDrive/sec-price-study`).
- **Sync:** local → Drive with Google Drive for desktop, or a provided `secprice sync` command that copies only the artifacts the GPU stages need (sections, events, splits, configs). GPU outputs come back the same way.
- **Hot temporary data:** on Colab, heavy temporary data goes to `/content` local disk and is copied to Drive at checkpoints.

### 13.3 Budgets and preflight
- **`secprice preflight`** prints, for the chosen profile:
  - firms, filings and sections
  - estimated chunks and tokens
  - embedding storage (documents × chunks × hidden size × bytes, plus about 10% index overhead)
  - model download sizes
  - LLM prompt count and tokens
  - disk free (local and Drive)
  - projected GPU time from a timed micro-benchmark
- **Refusal:** it refuses `full` if `budget.max_storage_gb` or `budget.max_gpu_hours_{b,c}` would be exceeded.
- **Monitoring:**
  - `budget.max_error_rate`: a stage stops cleanly if exceeded
  - wall time and GPU-active time are logged per stage; neither is a credit measurement
- **Colab rules:**
  - restart the runtime when switching CPU and GPU
  - every stage reloads from `DATA_ROOT`
  - never rely on in-memory state or background execution
- **GPU efficiency:**
  - mixed precision
  - length bucketing
  - 4-bit or AWQ for the 7B LLM, if needed
  - vLLM batch inference where compatible, otherwise Transformers
  - OOM retry that halves the batch

### 13.4 Python environment
- Python 3.11 (match Colab's version at build time).
- `pyproject.toml` with pinned core dependencies and extras `[gpu]`, `[llm]`, `[dev]`.
- On Colab, install only missing extras; do not reinstall torch.
- Lock file committed.
- `secprice env` writes `manifests/environment_manifest.json`.

### 13.5 Candidate libraries (verify; choose and document)
| Need | Candidate | Notes |
|---|---|---|
| EDGAR access | Own thin client on the official endpoints (submissions JSON, archives, companyfacts), **or** `edgartools` (MIT; identity required; rate-limit aware) | If a library is used, wrap it so our rate limiter, User-Agent and cache stay in control |
| 10-K item cross-check | `eloukas/edgar-corpus` (HF; 10-K items, 1993–2020; ~40 GB) | Optional QA of our 10-K section parser on a sample; never a substitute for our own point-in-time pipeline |
| Hugging Face SEC audit | `JanosAudran/financial-reports-sec` | Sentence-level 10-K data with return labels; audit / cross-check only; never use its labels |
| Trading calendar | `pandas_market_calendars` (NYSE, early closes) | Session boundaries |
| Prices | Pluggable `PriceProvider`: yfinance default (explicit `auto_adjust` / `actions`; record version); CSV import for licensed data (e.g. CRSP via university WRDS) | yfinance lacks most delisted history; the coverage table must show it |
| Factors | Kenneth French data library | Daily FF5 + momentum |
| ML | scikit-learn, LightGBM, CatBoost, Optuna | |
| NLP | transformers, sentence-transformers, rank-bm25 or bm25s, FAISS (CPU / GPU) | |
| LLM | vLLM (if compatible) or transformers + bitsandbytes; a JSON-constrained decoding library | |
| Stats | statsmodels (Newey-West, clustered errors), arch (MCS, if used) | |
| Storage | pyarrow / Parquet (zstd), DuckDB for queries | |

---

## 14. Repository layout

```text
sec-filings-price-impact/
├── SPEC.md                     # this file
├── CLAUDE.md                   # digest maintained by Claude Code
├── README.md                   # quick start: local smoke, Colab GPU, profiles, labelling
├── pyproject.toml / lock file
├── configs/
│   ├── default.yaml            # all parameters (Section 3)
│   ├── profiles/{smoke,pilot,full}.yaml
│   ├── preregistration.yaml    # hypotheses, primary metric/target, k, coverage levels, FDR families, trivial bps
│   ├── models.yaml             # roster + provenance ledger fields + pinned revisions
│   ├── taxonomy_8k_v1.yaml     # raw/legacy item → category, effective dates
│   ├── question_bank_v1.yaml
│   └── lexicons/               # Loughran-McDonald (downloaded, versioned)
├── src/secprice/
│   ├── cli.py                  # commands (Section 17.2)
│   ├── config.py               # load, merge profile, validate, hash
│   ├── storage.py              # DATA_ROOT, artifact paths, manifests, _SUCCESS markers, hashing
│   ├── runner.py               # stage registry, dependency resolution, skip-if-done
│   ├── logging_utils.py
│   ├── sec/                    # client (rate limit, UA, cache), submissions, archives, companyfacts, headers
│   ├── universe/               # membership, ticker↔CIK mapping, eligibility, universe table
│   ├── market/                 # calendar, price providers, corporate actions, factors
│   ├── audit/                  # coverage + reason codes, HF dataset audit
│   ├── parse/                  # html/text, sections, exhibits, tables, normalize, anonymize, taxonomy, eps_release
│   ├── events/                 # timestamps, grouping, earnings types, clusters, overlaps
│   ├── targets/                # returns, abnormal returns, labels, censoring
│   ├── features/               # registry, price, metadata, accounting (SUE), lexical, change, novelty, embeddings
│   ├── splits/                 # walk-forward, purge/embargo, cohorts, leakage assertions, planted leaks, provenance
│   ├── models/                 # ladder, heads, fine-tune, tuning, calibration, selective prediction
│   ├── rag/                    # chunking, bm25, dense, hybrid, rerank, extract, guardrails, distill
│   ├── gold/                   # sampling, templates, import, agreement
│   ├── evaluation/             # metrics (cls/reg/ir/content), bootstrap, fdr, mcs, pbo, results table
│   ├── track_a/                # event study, drift, correlations, topics
│   ├── backtest/
│   ├── viz/                    # one function per figure ID
│   ├── report/                 # results_summary.md builder
│   └── extensions/             # ext_* modules (Section 7.3), each behind its flag
├── tests/
│   ├── unit/                   # Section 18.1
│   ├── integration/            # offline smoke on fixtures
│   └── fixtures/               # tiny synthetic + recorded SEC/price responses
├── notebooks/
│   ├── colab_gpu_runner.ipynb  # mount Drive, clone/pull repo, install extras, run GPU stages by name
│   └── study_walkthrough.ipynb # reads artifacts; explains each stage with plots (learning notebook)
└── docs/
    ├── decision_log.md
    ├── annotation_guide.md
    └── stage_cards/            # one markdown per stage: purpose, inputs, outputs, assertions, how to read
```

---

## 15. Configuration

- **Files:** one YAML (`default.yaml`) plus profile overlays. Validated at load (types, ranges, required secrets present only when needed).
- **Config hash:** a hash of the resolved config, excluding paths and secrets, stamps every artifact.
- **Required sections:**
  - `run_profile`, `seed`, `paths`
  - `sec` (`max_rps`, `user_agent_env`, cache)
  - `universe`, `study`, `market`, `timing`, `text` (sections, char budgets, chunk sizes)
  - `taxonomy_version`, `targets`, `labels`, `splits`
  - `models` (core list + revisions), `rag`, `distill`
  - `tuning`, `gates`, `audit`, `bootstrap`, `fdr`, `trivial_effect_bps`
  - `backtest`, `budget`, `extensions`
- **Pre-registration:** `preregistration.yaml` is hashed by `secprice prereg-freeze` (writes `manifests/prereg_hash.txt`). Stages that score the development holdout refuse to run without it.

---

## 16. Data contracts (Parquet; one row definition each)

Every artifact also writes a sidecar manifest (`*.manifest.json`):
- stage
- config hash
- git commit
- input artifact hashes
- row count
- start and end times
- warnings

| Artifact | One row = | Key columns |
|---|---|---|
| `universe.parquet` | firm | cik, tickers_history, name, sic, ff12_industry, index_start, index_end, price_start, price_end, delisted_flag, eligible, exclusion_reason |
| `filing_manifest.parquet` | observed filing | accession, cik, form, is_amendment, filing_date, acceptance_raw, acceptance_time, ts_quality, period_of_report, primary_doc, items_raw, items_source, url, audit_reason_code, parse_status, checksum |
| `sections.parquet` | filing × section | accession, section_id, text_raw_ref, text_masked_ref, n_chars, n_tokens_est, parse_method, boilerplate_share, mask_counts |
| `exhibits.parquet` | filing × exhibit | accession, exhibit_type, text_ref, is_results_release |
| `xbrl_facts.parquet` | fact × accession | cik, accession, taxonomy, tag, unit, value, period_start, period_end, fy, fp, form, filed, public_available_time |
| `eps_release.parquet` | release | accession, eps_diluted, eps_basic, period_end, parse_confidence, xbrl_match_flag (QA) |
| `prices.parquet` | firm × session | cik, ticker, date, open, high, low, close_raw, close_split_adj, total_return_index, volume, source, source_version |
| `corporate_actions.parquet` | action | cik, date, type, value |
| `factors.parquet` | session | date, mkt_rf, smb, hml, rmw, cma, mom, rf |
| `events.parquet` | event | event_id, cik, accessions, forms, categories, event_type, earnings_type, cluster_id, acceptance_time, public_available_time, decision_time, ts_quality, session_bucket, trackA_day0, tinfo_base_date, tinfo_eligible, ttrade_entry_date, contam_minutes, overlap_flag_h{1..60} |
| `targets.parquet` | event × horizon × target_def | event_id, horizon, target_def, raw_ret, ar_mm, ar_carhart, sigma, label3, vol_ratio, abs_std, censored |
| `features/<family>.parquet` + `feature_registry.csv` | event | event_id + features; the registry gives name, family, source, availability rule, max availability time, strict/retrospective |
| `splits.parquet` | event × scheme | event_id, scheme, fold, role (train / val / purged / dev / locked), cohort flags |
| `embeddings/<model>/<rev>/` | chunk / doc | float16 matrix + index parquet (event_id, accession, section, chunk_id, offsets) |
| `extractions.parquet` | event × field | event_id, field, present, direction, magnitude, unit, certainty, evidence, source (teacher / student), confidence |
| `predictions.parquet` | run × event | run_id, model_id, rung, fold / block, p_up, p_side, p_down, yhat_ar, yhat_vol |
| `results_long.parquet` | metric value | run_id, track, rung, model_id, revision, leaderboard (strict / retro), cohort (common / max), target_def, horizon, block, event_set (all / clean), coverage, metric, value, ci_low, ci_high, n_events, n_firms, n_dates, n_overlap |
| `gold/*.parquet` | item | gold_split, item_id, annotator, labels, spans, timestamps |

**Rules**
- Text is stored zstd-compressed, by reference.
- Raw HTML is not retained beyond the cache window, but source URL and checksum are.
- Nulls mean "not applicable or unavailable", never zero.

---

## 17. Stage pipeline

### 17.1 Stages
Each stage declares its inputs, outputs, assertions and part. Each writes a stage card in `docs/stage_cards/`.

| ID | Stage | Part | Outputs | Key assertions |
|---|---|---|---|---|
| S00 | env / config / prereg check | CPU | environment manifest, config hash | config valid; prereg present |
| S01 | HF dataset audit | CPU | `hf_dataset_audit.md` | no labels used |
| S02 | Universe | CPU | `universe.parquet`, universe table | ≥ n_stocks unique firms (full) or failure report |
| S03 | Calendar, prices, actions, factors | CPU | prices, actions, factors, coverage CSV | split/dividend unit tests pass; explicit downloader parameters recorded |
| S04 | EDGAR index + coverage audit | CPU | `filing_manifest`, coverage CSVs, reason codes, `data_coverage_summary.md` | no 8-K "expected"; timestamp-quality table |
| S05 | Download + parse + taxonomy + anonymize | CPU | sections, exhibits, tables, parse QA | taxonomy version recorded; `OTHER_OR_UNMAPPED` share reported |
| S06 | XBRL + release EPS + SUE | CPU | xbrl_facts, eps_release, SUE features | no later-cluster source; ordering by public time |
| S07 | Events + clocks | CPU | `events.parquet` | four timestamps; buckets; earnings types; overlaps |
| S08 | Targets + labels | CPU | `targets.parquet` | INTRA not T_info-eligible; per-horizon censoring |
| S09 | Structured + lexical features | CPU | features (price, metadata, accounting, lexical, change, novelty), registry | availability ≤ decision_time for all |
| S10 | Splits + cohorts + leakage audit | CPU | `splits.parquet`, `leakage_audit.csv` | all assertions; 4 planted leaks caught |
| S11 | Baselines L0–L4, L6 | CPU | predictions, results | common cohort recorded |
| S12 | Gold export / import | CPU | templates; gold splits | gold-test blind; split integrity |
| S13 | Preflight (GPU) | GPU | workload report | budgets OK |
| S14 | Core embeddings (one model at a time) + vintage ChronoBERT | GPU | embeddings | pooling + change vectors; revisions pinned |
| S15 | L5 / L7 heads + tuning; fine-tune top 2 if G3 | GPU / CPU | predictions, curves | finalist caps; before/after |
| S16 | Core RAG: indexes, retrieval eval, extraction (gold-val), freeze | GPU | indexes, rag_tournament.csv | historical path time-filtered; current path separate |
| S17 | Teacher sample + student distillation → L8 | GPU | extractions, student model | training-period only |
| S18 | Development holdout + diagnostics (MCS / PBO) + G4 freeze | CPU | frozen_architecture.json + hash | no test access |
| S19 | `unlock-test`: retrain + locked test + clean test | CPU / GPU | locked results | single use; logged |
| S20 | Track A event study + correlations + topics | CPU | track_a results | clean / all; pre-trends |
| S21 | Drift tests | CPU | hypothesis_tests.csv | FDR families; verdict rubric |
| S22 | Backtest | CPU | backtest.csv | entry-rule assertions |
| S23 | Track C final (gold-test) | CPU | content_validity.csv | gold-test touched once |
| S24 | Figures + report | CPU | figures, `results_summary.md` | computed values only |
| X* | Extensions | per flag | per module | gate G3c |

### 17.2 CLI commands (implemented in `cli.py`)
- **`secprice run`:** `--profile`, `--stages` (single, list or range), `--force` (rerun even if done).
- **`secprice status`:** done / pending / stale per stage.
- **`secprice preflight`**
- **`secprice prereg-freeze`**
- **`secprice freeze`** (G4)
- **`secprice unlock-test`:** requires the G4 hash; refuses a second run unless `--acknowledge-reuse`, which relabels the block as development.
- **`secprice gold export|import|agreement`**
- **`secprice sync push|pull`**
- **`secprice report`**
- **`secprice env`**

### 17.3 Artifact rules
- **Layout:** `DATA_ROOT/<profile>/<stage>/<config_hash>/...` with a `_SUCCESS` marker.
- **Skipping:** a stage is skipped if `_SUCCESS` exists and its input hashes match; otherwise it is stale.
- **Atomic writes:** write to temp, then rename.
- **Checkpointing:** downloads and embeddings checkpoint in batches, and resume without duplicates.

---

## 18. Testing

### 18.1 Unit tests (must pass before G1)
1. **Session buckets:** across DST changes, holidays, early closes, and the 09:30 / 16:00 boundaries.
2. **Timestamps:**
   - acceptance-time timezone parsing against an SGML header fixture
   - public / decision computation
   - quality-B rule
3. **T_info:** base and exit per bucket; INTRA excluded.
4. **T_trade:** entry = first open at or after `decision_time` for every bucket (including INTRA and late PRE); h = 1 = open → close.
5. **Returns:** around known split and dividend fixtures; raw versus total return.
6. **Abnormal returns:** market model and Carhart on synthetic data with a known beta.
7. **Labels:** k-band labelling; per-horizon censoring.
8. **SUE:**
   - YTD → quarterly conversion
   - as-first-reported selection
   - **release-EPS path for 8-K events**
   - **rejection of later-cluster XBRL**
9. **EAR:** a same-cluster feature is rejected; a later-cluster feature is accepted only after the window closes.
10. **Overlaps:** flags and clean cohort.
11. **8-K taxonomy:** legacy → modern mapping by date; catch-all.
12. **Coverage:** reason codes; no 8-K expectations.
13. **Retrieval:** the historical index never returns current or future documents.
14. **Splits:** purge / embargo boundaries; date-disjointness; accession grouping.
15. **Leakage self-tests:** all four planted leaks are caught.
16. **Cohorts:** the common cohort is identical across compared models.
17. **Metrics:** each metric matches a hand-computed fixture; nulls for inapplicable cases.
18. **Bootstrap:** block structure respected.
19. **Resilience:** a resume does not duplicate rows; checkpoint reload reproduces predictions.
20. **Config:** validation and hash stability.

### 18.2 Integration
- **Offline smoke** on fixtures, S00 → S12 and S20 → S24 (CPU), with a GPU-free fallback (TF-IDF in place of embeddings) so the full chain runs locally in minutes.
- **Online smoke:** `smoke` profile against the live SEC and price source (marked `network`).

### 18.3 Optional CI
A GitHub Actions workflow runs unit and offline integration tests on push (CPU only).

---

## 19. Logging, manifests, reproducibility

- **Logs:** structured logs per stage (`logs/<stage>.log`), plus `errors.log` with error-rate tracking.
- **`run_manifest.json`:** profile, config hash, git commit, stage timings, wall time and GPU-active time, model revisions, data source versions.
- **`frozen_architecture.json`:** the G4 freeze, with chosen models, hyperparameters, thresholds and their hash.
- **Seeds:** set for Python, NumPy, torch and Optuna. Note where GPU nondeterminism may remain.

---

## 20. Figures

### 20.1 Core figures (`viz/`; each with a "how to read" caption in the stage card and walkthrough notebook)

**Data and coverage**
- F01 coverage heatmap (observed versus parsed) by year × form
- F02 audit reason codes over time
- F03 8-K category availability by year (taxonomy version)
- F04 universe: unique firms versus firms per year, delisted coverage
- F05 timestamp quality and session buckets by year
- F06 pre-publication contamination-window distribution and public-lag sensitivity

**Events and targets**
- F07 event-clock diagram (Track A, T_info, T_trade)
- F08 earnings types over time
- F09 overlap counts and clean share
- F10 target distributions and class balance

**Track A**
- F11 CAR paths by form and category (all versus clean)
- F12 by bucket
- F13 pre-trend diagnostics
- F14 abnormal volume and volatility response
- F15 correlation heatmaps
- F16 SUE-drift quintile CAR[+2,+60]

**Track B**
- F17 split timeline with purge / embargo, holdout, lockbox and per-fold provenance
- F18 ladder L0–L8 lift with CIs (common cohort; strict and retrospective)
- F19 vintage versus static (H4)
- F20 loss curves
- F21 validation curves
- F22 learning curves
- F23 before / after tuning
- F24 confusion matrices
- F25 ROC / PR curves
- F26 calibration
- F27 coverage–accuracy curve
- F28 rolling IC
- F29 T_info versus T_trade
- F30 backtest (cumulative, costs, lag)
- F31 performance by form, category, bucket, timestamp quality, size, sector, year

**Track C**
- F32 retrieval metrics by setup
- F33 extraction F1 / faithfulness (gold-test)
- F34 numeric fidelity
- F35 judge versus human
- F36 student versus teacher versus human

**Sanity**
- F37 permutation null
- F38 random-split demo
- F39 adversarial-validation ROC (with interpretation note)
- F40 MCS / PBO diagnostics

### 20.2 Extension figures
Produced by each extension module: encoder tournament, RAG variants frontier, time-series models, stacking weights, generation-metric panel.

---

## 21. Final report (`reports/results_summary.md`, computed values only)

1. **Data integrity:** universe naming and table; coverage and reason codes; 8-K category availability; timestamp quality; Hugging Face audit; limitations (survivorship-aware, large-cap tilt, selected sections).
2. **Design:** four timestamps, targets, labels, overlaps, censoring; pre-registration hash.
3. **Leakage and provenance audit:** assertions, planted leaks, sanity experiments, review-trigger outcomes, ledger, per-fold labels.
4. **Track A:** results (all / clean), pre-trends, correlations, SUE drift and other verdicts with power.
5. **Track B:**
   - ladder (common and maximum cohorts)
   - strict and retrospective leaderboards
   - H1–H4 results with CIs
   - tuning before / after
   - holdout and locked-test results (clean subsets for known-cutoff models)
   - selective prediction
   - T_info versus T_trade
   - backtest
6. **Track C:** gold-test content validity, single- or double-rater status.
7. **Fit diagnostics.**
8. **Extensions run** (if any) and their gates.
9. **Limitations, resource use, decision log.**
10. **Executive summary** ending with **the smallest defensible claim**.

---

## 22. Milestone plan for Claude Code

Each milestone ends with: tests green, stage cards written, `CLAUDE.md` updated, a git commit and a short summary to the owner (what works, what's stubbed, decisions logged).

| # | Milestone | Done when |
|---|---|---|
| M0 | Scaffold: pyproject, package, config loader + validation + hash, storage / manifests, runner + CLI skeleton, logging, test harness, `CLAUDE.md`, README | `secprice status` and `secprice run --profile smoke --stages S00` work; config tests pass |
| M1 | SEC client (rate limiter, UA from env, cache, backoff) + submissions + headers; universe (S02); EDGAR index + coverage audit (S04); HF audit (S01) | Smoke universe and manifest built; coverage summary readable; acceptance-time timezone test passes; recorded fixtures for offline tests |
| M2 | Calendar, prices (provider interface), corporate actions, factors (S03) | Split / dividend / early-close tests pass; coverage CSV shows delisted gaps |
| M3 | Download, parse, sections, exhibits, tables, normalize, anonymize, taxonomy (S05) | Parse QA table; section-extraction spot check on fixtures; taxonomy tests pass |
| M4 | XBRL facts, release-EPS parser, SUE (S06) | SUE tests pass, including the later-cluster rejection; coverage by year reported |
| M5 | Events, four timestamps, earnings types, clusters, overlaps (S07); targets / labels / censoring (S08) | All timing and target unit tests pass; F05–F10 render |
| M6 | Features + registry (S09); splits, cohorts, leakage audit, planted leaks (S10) | All four planted leaks caught; availability assertion passes |
| M7 | Metrics module, bootstrap, FDR, results table; baselines L0–L4, L6 (S11); core figures for coverage and Track B baselines | **G1:** offline integration smoke S00 → S11 passes end to end on CPU |
| M8 | Track A event study, correlations, topics, SUE drift, momentum, other drift tests with verdict rubric (S20–S21) | F11–F16; hypothesis_tests.csv with verdict labels |
| M9 | Gold-set tooling (S12) + annotation guide | Export / import round-trip; gold splits enforced; blind gold-test |
| M10 | Colab GPU runner; preflight (S13); core embeddings + vintage ChronoBERT (S14); heads, tuning, G3, fine-tune (S15) | GPU smoke runs on Colab; L5 / L7 results; curves F20–F23 |
| M11 | Core RAG + extraction + guardrails (S16); teacher / student distillation → L8 (S17) | Retrieval and extraction metrics on gold-val; G3b decision logged |
| M12 | Holdout, diagnostics, G4 freeze, unlock-test, locked and clean tests (S18–S19); backtest (S22); Track C final (S23) | Lockbox single-use test passes; strict / retrospective leaderboards produced |
| M13 | Figures + report builder (S24); walkthrough notebook | `results_summary.md` generated from artifacts; walkthrough runs on smoke artifacts |
| M14 | **G2:** pilot run end to end; review with owner | Pilot report; budget projection for full |
| M15+ | Full run; then extensions one by one (each needs flag + G3c + budget) | Per extension |

**Gates**
| Gate | Before | Passes when |
|---|---|---|
| G1 | anything on GPU | unit tests + offline smoke integration pass |
| G2 | full run | pilot completed; preflight within budget; owner opts in |
| G3 | fine-tuning | text lift over L6 ≥ 0.01 macro-F1 on validation, CI excludes 0 |
| G3b | teacher extraction at scale | gold-val extraction F1 and faithfulness thresholds met |
| G3c | any extension | core complete; budget remains; a pre-stated question the extension answers |
| G4 | locked test | frozen manifest hashed; prereg hash present |

---

## 23. Acceptance checklist

**Data**
- [ ] Unique-firm and per-year counts; universe named precisely; survivorship-aware wording.
- [ ] Coverage observed versus parsed, with reason codes; no 8-K expectation; 8-K taxonomy versioned with a catch-all and availability by year.
- [ ] Timestamp-quality table; acceptance timezone verified; four timestamps per event.
- [ ] Explicit price-source parameters; raw / split / total-return series separate; corporate-action tests.
- [ ] Usable cutoff printed; per-horizon censoring; the partial final month flagged.

**Timing and leakage**
- [ ] INTRA excluded from T_info; T_trade entry at the first open at or after decision time for all buckets.
- [ ] No same-cluster EAR; no later-cluster SUE; release-EPS path implemented.
- [ ] Historical retrieval time-filtered; current document on a separate path.
- [ ] All leakage assertions pass; four planted leaks caught.

**Modelling**
- [ ] Common-cohort and maximum-available comparisons both reported.
- [ ] Strict versus retrospective leaderboards; no clean-test claims for unknown cutoffs.
- [ ] Core roster only unless extensions were gated in.
- [ ] Default and tuned results; curves and verdicts.
- [ ] The lockbox scored once.

**Content**
- [ ] Gold-dev / val / test separation; gold-test blind; second-rater status stated.

**Inference**
- [ ] Verdict rubric with trivial thresholds; FDR families; power language.
- [ ] MCS / PBO used as diagnostics only.

**Backtest**
- [ ] Section 11 rules; long-only and hypothetical long-short; borrow caveat.

**Engineering**
- [ ] Smoke is the default; preflight enforces budgets; stages resumable.
- [ ] Tests green; offline smoke runs locally.
- [ ] GPU runner works on Colab.
- [ ] No news data anywhere; the report contains computed values only.

---

## Appendix A: Professor review reconciliation (v4 → v5)

| Review item | Decision | Where |
|---|---|---|
| P0.1 Set parameters to 500 firms, Jan 2000 – Oct 2026, warm-up, predecessor filings | **Accepted** (overridable defaults) | §3 |
| P0.2 Intraday T_trade fill at the same close is unsafe | **Accepted:** all buckets enter at the first open at or after decision time; h defined identically | §5.4 |
| P0.3 Acceptance vs public vs feature-ready vs decision time | **Accepted:** four timestamps; separate current / historical evidence paths; "≤ decision_time" for features, strict "<" for prior documents | §5.1, §8, §6.3 #16 |
| P0.4 Gold-set reuse | **Accepted:** gold-dev / val / test; final numbers on gold-test only; blind gold-test | §12 |
| P0.5 Unknown cutoffs | **Accepted:** strict versus retrospective leaderboards; no clean-test claims for unknown cutoffs | §6.4 |
| P0.6 SUE cluster-aware | **Accepted and extended:** v5 also notes that XBRL is generally not available at an earnings-8-K's decision time, so 8-K events need release-exhibit EPS parsing; XBRL SUE for those events is ex-post QA or sensitivity only | §5.6 |
| P0.7 Common cohort | **Accepted:** common (primary) and maximum-available (secondary) | §6.2, §9.1 |
| P1.8 PBO / MCS as diagnostics | Accepted | §9.2 |
| P1.9 Annotation reliability | Accepted: second rater on ≥ 30% of gold-test; single-rater label otherwise; no pre-fill on gold-test | §12 |
| P1.10 Narrow the default architecture | **Accepted with modification:** core roster as the review proposed (SEC-BERT, FinBERT-tone, ChronoBERT, BGE; BM25 / hybrid / hybrid + rerank; one LLM). The owner's broader roster stays implemented-on-demand as a gated extension pack rather than deleted | §7 |
| P1.11 Simplify generation metrics | Accepted: core content metrics redefined; the full generation panel moves to `ext_generation_metrics` (the owner's requested metrics remain available) | §9.2, §7.3 |
| P1.12 Concrete backtest | Accepted: signal score, thresholds from training folds, minimum names, overlap policy, execution, borrow, corporate events | §11 |
| P1.13 Drift inference rule | Accepted: trivial thresholds in bps pre-registered; failing a test never proves absence | §3, §10.2 |
| Keep / Add / Remove lists | Applied throughout | — |
| Implementation order | Adopted as the milestone plan | §22 |
| External checks (SEC lag, HF dataset scope, ChronoBERT, FinText) | Verified and recorded | Appendix C |

## Appendix B: Earlier review reconciliation (v3 → v4), still in force
- EAR is never a same-event feature.
- Verified versus periodic earnings types.
- 8-K is event-driven; reason codes.
- Smoke default.
- Survivorship wording.
- NER is never used for money.
- Review triggers instead of stops; pre-trends interpreted.
- Provenance ledger.
- Overlap controls.
- Versioned 8-K taxonomy.
- Explicit price handling.
- Ablation hierarchy.
- Native tasks scored separately.
- Selective zero-shot use.
- Carhart / FF5 naming.
- Correct reading of adversarial validation.
- Power language.

## Appendix C: Verification notes (re-check at build time)
- **SEC timing and access:** the SEC webmaster FAQ states a 10 requests/second maximum and that filings typically appear on sec.gov 1–3 minutes after the EDGAR timestamp, possibly longer under load; the lag is not guaranteed. This underpins `timing.public_lag_minutes`.
- **`JanosAudran/financial-reports-sec`:** sentence-level 10-K records with 1d / 5d / 30d return labels; configs `large_full` / `large_lite` / `small_full` / `small_lite`; Apache-2.0. Audit only.
- **`eloukas/edgar-corpus`:** annual-report items 1–15 (incl. 1A, 7, 7A), 1993–2020, ~40.7 GB; optional QA source for 10-K section parsing.
- **`edgartools`:** MIT; requires an identity; rate-limit aware; section extraction (risk factors, MD&A), 8-K items, XBRL / company facts. Wrap it if used.
- **`fja05680/sp500`:** historical S&P 500 components since 1996 (MIT). The author notes that Wikipedia-based changes are incomplete, early years may be missing symbols, and free sources lack delisted price history.
- **Model facts** (verified in the v3 pass):
  - SEC-BERT: 1993–2019 10-K corpus
  - ChronoBERT: vintages 1999–2024 (`manelalab/chrono-bert-v1-YYYY1231`)
  - FinText Chronos: dated vintages; confirm each year exists
  - SecBERT / SecRoBERTa: cybersecurity models
  - no public weights found for BloombergGPT / FinLlama / BB-BERT
- **To verify on a sample at build time (not assumed):** the timezone semantics of `acceptanceDateTime` in the submissions JSON versus the SGML header; availability of 8-K `items` metadata for pre-2004 filings (fall back to parsing headers or text); yfinance defaults in the installed version.


In [ ]:
%%writefile $REPO/SPEC_ADDENDUM.md
# V5 Pre-Coding Review Addendum
## SEC Filings → Price Impact

**Purpose:** Small corrective addendum to `SEC_Filings_Price_Impact_Build_Spec_v5.md`. This is not a replacement for v5. It clarifies issues to resolve before the relevant implementation milestones are accepted.

## Decision

**Approve milestone M0 and begin coding the repository scaffold.** Do not begin full data/model runs until the target definitions and return calculations below are resolved and covered by tests.

The v5 design is substantially improved: it separates event study, prediction and content validation; defines a repository and data contracts; narrows the default model roster; adds milestone gates; and makes smoke mode the default.

## P0 — Resolve before implementing prediction targets

### 1. Make the primary Track B target truly prospective

In v5, `T_info` starts from the last regular-session close before filing acceptance and measures the response after public availability. However, the model decision occurs later, at `decision_time = feature_ready_time`. The T_info target can therefore include price movement that occurred before the model could actually make its prediction.

Implement the following separation:

- **Track A / T_info:** retain as an information-response/event-study outcome. It answers how prices moved around a filing. Do not present a model scored on this target as an uncontaminated prospective forecast from `decision_time`.
- **Track B / T_trade:** make this the primary prospective prediction target and primary leaderboard for the trading-relevant task. Its outcome must begin at the first executable price after `decision_time`.
- Train a separate model per target if both targets are modeled. Do not use a model trained to predict T_info as the sole signal model for a T_trade backtest.
- Keep the same filing event eligible for Track A and Track B where possible, but report the different clocks, eligible cohorts and sample counts explicitly.
- Revisit the preregistration, primary metric and hypotheses so the primary Track B metric is defined on T_trade. T_info remains the complementary event-response analysis.

**Acceptance test:** demonstrate with synthetic PRE, POST, NONTRADING and INTRA events that no return component used as a Track B target occurs before its decision time.

### 2. Match stock returns, benchmark returns and factor intervals

The backtest enters at an open and exits at a close, while the project also uses daily close-to-close market and factor returns. These intervals must not be mixed.

- Keep the close-to-close total-return series for event-study outcomes where appropriate.
- Calculate executable T_trade returns from consistently adjusted entry and exit prices, with dividends and splits handled explicitly.
- Align benchmark returns to the exact same holding interval. For open-to-close entries, use a matching benchmark open-to-close component; for subsequent sessions, calculate the remaining holding-period return consistently.
- Do not subtract a close-to-close market/factor return from an open-to-close stock return.
- Preserve the separately specified factor-adjusted event-study method and backtest alpha method.
- Test a known split, a dividend during the holding period, overnight gaps and entry/exit across multiple sessions.

**Acceptance test:** unit tests establish the exact return for simple synthetic and known corporate-action examples, and verify that stock and benchmark intervals match.

### 3. Label estimated public availability honestly

The SEC documents that filing availability on sec.gov often follows the EDGAR acceptance timestamp by one to three minutes, with potentially longer delays and no guarantee. Therefore, historical `acceptance_time + 5 minutes` is an estimate, not a verified actual public-availability timestamp.

- Preserve acceptance time as observed metadata.
- Store actual public availability only when independently observed and supported; otherwise mark it as estimated.
- Preserve the configured public lag and feature-processing latency in the run manifest.
- Retain the 0 / 15 / 60-minute sensitivity (and the configured decision-latency sensitivity).
- Use wording such as **“point-in-time evaluation under an assumed availability/processing-latency policy”** when exact historical public availability is unavailable.
- Reserve “strict point-in-time” for model/data configurations whose chronology is actually documented and satisfied; do not imply the SEC publication delay was observed when it was inferred.

**Acceptance test:** the event table and report distinguish observed timestamps from assumed timestamps and label each relevant result with its timing policy.

## P1 — Resolve before accepting the corresponding milestones

### 4. Provide enough historical data for SUE

The config currently specifies a two-year predecessor lookback for filing-change features, while SUE's denominator uses up to eight prior seasonal EPS changes. Computing eight year-over-year seasonal changes requires more historical EPS observations than two years can normally supply.

- Separate `text.predecessor_lookback` from `accounting.predecessor_lookback`.
- For SUE, fetch enough earlier point-in-time quarterly/release EPS history to calculate the configured seasonal-difference history where available (a starting policy of at least four years before `STUDY_START` is safer than two).
- If the source cannot support the history, mark SUE unavailable for that event; do not fabricate values or silently shorten history without recording the actual number of observations.
- Report SUE coverage by year, filing type and company size.

**Acceptance test:** synthetic histories with 3, 4 and 8 valid seasonal differences produce the documented behavior and record the number of prior observations.

### 5. Keep the CPU smoke test realistically lightweight

The v5 smoke profile mentions a local LLM and RAG, but the LLM in the core roster is a 7B model and cannot be assumed to run quickly or fit in memory on every laptop CPU.

- The mandatory offline CPU smoke should use fixtures and a deterministic mock/stub generator for the LLM boundary; it must validate the same stage interfaces, schemas, caches and tests.
- Keep actual Qwen inference in an explicitly opt-in GPU smoke test on Colab.
- Use a tiny real model on CPU only if the machine passes a RAM/runtime preflight.
- Make the report unambiguous about which smoke stages use fixtures, real SEC/network responses, GPU or a local model.

**Acceptance test:** the mandatory offline CPU integration test completes without downloading a 7B model or requiring network/GPU access.

### 6. Guard the gold-test against repeated use

The gold-dev / gold-val / gold-test split is a major improvement. Make the code enforce it rather than relying on notebook discipline.

- Only gold-dev may be used for schema/prompt iteration.
- Gold-val selects retrieval, generator, detector and confidence thresholds.
- Gold-test remains blind until the winning pipeline is frozen; it is evaluated once for final content-validity reporting.
- Ensure the same filing or passage cannot appear across those partitions.
- Because rare categories are oversampled, do not treat the gold sample's class prevalence as population prevalence unless sampling weights are applied.

## Build order

Continue with v5's milestones:

1. **M0:** scaffold, config validation/hash, artifacts/manifests, CLI, logging, tests, `CLAUDE.md`, README.
2. **M1–M2:** SEC metadata, universe, prices/calendar/corporate actions and coverage audit.
3. **M3–M6:** parsing, taxonomy, accounting/SUE, event clocks, targets and leakage-safe features/splits.
4. **M7:** baselines and offline CPU integration smoke.
5. **M8–M9:** event studies and gold annotation tooling.
6. **M10 onward:** only then begin Colab GPU embeddings, tuning and gated RAG work.

Do not let M10 start expensive training if the target-timing tests or return-interval tests are still failing.

## Instruction to Claude Code

Apply this addendum alongside `SPEC.md`. Do not silently change v5: record each resolution in `docs/decision_log.md`, update the preregistration where primary-target definitions change, add automated tests for every item, and summarize the implementation choices at the end of each milestone.

**Bottom line:** v5 is ready for staged coding. The main remaining priority is to make the primary prediction target genuinely prospective, match tradable and benchmark return intervals, and be transparent that historical public-availability times are estimated when they are not observed.


In [ ]:
%%writefile $REPO/src/secprice/__init__.py
"""secprice: SEC filings -> price impact study (SPEC.md v5 + pre-coding addendum)."""
__version__ = "0.5.0"


In [ ]:
%%writefile $REPO/src/secprice/audit/__init__.py
"""secprice.audit"""


In [ ]:
%%writefile $REPO/src/secprice/audit/hf_audit.py
"""S01 Hugging Face SEC dataset audit (SPEC.md §13.5, Appendix C). Audit/cross-check only: the dataset's return
labels and returns are never read into any artifact (assertion)."""
from __future__ import annotations

import json
import time
from pathlib import Path

import pandas as pd
import requests

CENSUS_COLS = ["cik", "name", "tickers", "exchanges", "sic", "entity_type", "form", "filing_date", "acceptance_raw",
               "report_date", "doc_id", "n_sections_nonempty", "n_sentences", "split", "shard"]
FORBIDDEN_KEYS = {"labels", "returns"}
QA_SECTIONS = {"section_1": "10K_1", "section_1A": "10K_1A", "section_7": "10K_7", "section_7A": "10K_7A"}


def shard_files() -> list[str]:
    return [f"data/large/{s}/shard_{i}.jsonl" for s in ("train", "validate", "test") for i in range(10)]


def _url(repo: str, rev: str, fn: str) -> str:
    return f"https://huggingface.co/datasets/{repo}/resolve/{rev}/{fn}"


def _iter_lines(url: str, retries: int = 5):
    for attempt in range(retries):
        try:
            with requests.get(url, stream=True, timeout=180) as r:
                r.raise_for_status()
                for line in r.iter_lines(chunk_size=1 << 22):
                    if line:
                        yield line
            return
        except (requests.ConnectionError, requests.Timeout, requests.exceptions.ChunkedEncodingError):
            if attempt == retries - 1:
                raise
            time.sleep(10 * (attempt + 1))


def census_shard(repo: str, rev: str, fn: str) -> pd.DataFrame:
    rows = []
    for line in _iter_lines(_url(repo, rev, fn)):
        d = json.loads(line)
        for f in d.get("filings", []):
            rep = f.get("report") or {}
            rows.append((d.get("cik"), d.get("name"), "|".join(d.get("tickers") or []), "|".join(d.get("exchanges") or []),
                         d.get("sic"), d.get("entityType"), f.get("form"), f.get("filingDate"), f.get("acceptanceDateTime"),
                         f.get("reportDate"), f"{d.get('cik')}_{f.get('form')}_{str(f.get('reportDate') or '')[:4]}",
                         sum(1 for v in rep.values() if v), sum(len(v) for v in rep.values() if v),
                         fn.split("/")[2], fn))
    return pd.DataFrame(rows, columns=CENSUS_COLS)


def build_census(repo: str, rev: str, work_dir: Path, log=None) -> pd.DataFrame:
    """Stream every raw JSONL shard (~13.4 GB) and keep filing metadata only. Checkpointed per shard."""
    work_dir.mkdir(parents=True, exist_ok=True)
    parts = []
    for fn in shard_files():
        part = work_dir / (fn.replace("/", "__") + ".parquet")
        if not part.exists():
            t0 = time.time()
            df = census_shard(repo, rev, fn)
            df.to_parquet(part, index=False)
            if log:
                log.info(f"HF census {fn}: {len(df)} filings in {time.time() - t0:.0f}s")
        parts.append(pd.read_parquet(part))
    out = pd.concat(parts, ignore_index=True)
    assert not (set(out.columns) & FORBIDDEN_KEYS), "HF labels/returns must never be read (SPEC.md §13.5)"
    return out


def qa_sections_for(repo: str, rev: str, census: pd.DataFrame, wanted: pd.DataFrame) -> pd.DataFrame:
    """Section text (joined sentences) for a QA sample of (cik, filing_date) pairs; streams only needed shards."""
    want = set(zip(wanted["cik"].astype(str).str.zfill(10), wanted["filing_date"].astype(str)))
    shards = census[census.apply(lambda r: (str(r.cik).zfill(10), str(r.filing_date)) in want, axis=1)].shard.unique()
    rows = []
    for fn in shards:
        for line in _iter_lines(_url(repo, rev, fn)):
            d = json.loads(line)
            cik = str(d.get("cik")).zfill(10)
            for f in d.get("filings", []):
                if (cik, str(f.get("filingDate"))) not in want:
                    continue
                rep = f.get("report") or {}
                for k, sid in QA_SECTIONS.items():
                    sents = rep.get(k) or []
                    rows.append((int(cik), f.get("filingDate"), sid, " ".join(sents), len(sents)))
    return pd.DataFrame(rows, columns=["cik", "filing_date", "section_id", "hf_text", "hf_n_sentences"])


def audit_markdown(census: pd.DataFrame, repo: str, rev: str, card_note: str = "") -> str:
    c = census.copy()
    c["year"] = c.filing_date.astype(str).str[:4]
    by_year = c.groupby("year").size()
    forms = c.form.value_counts().to_dict()
    dup = int(c.duplicated(["cik", "filing_date", "form"]).sum())
    acc_null = int(c.acceptance_raw.isna().sum())
    empty = int((c.n_sentences == 0).sum())
    lines = [
        "# Hugging Face SEC dataset audit (S01)", "",
        f"- Repository: `{repo}` at revision `{rev}` (raw JSONL shards, `large` config)",
        "- Use: **audit, firm-overlap definition and 10-K section-parser QA only**. Its return labels and `returns` "
        "fields are never read (assertion in code). Filing text for the study comes from our own point-in-time "
        "EDGAR pipeline (SPEC.md §13.5).",
        f"- Firms (CIK): **{c.cik.nunique():,}**; filings: **{len(c):,}**; forms present: {forms}",
        f"- Filing dates: {c.filing_date.min()} to {c.filing_date.max()}",
        "- Forms **not** present: 10-Q and 8-K (they come from EDGAR).",
        f"- Duplicate (cik, filing date, form) rows: {dup}; missing acceptance timestamps: {acc_null}; "
        f"filings with zero sentences: {empty}",
        f"- Median sentences per filing: {int(c.n_sentences.median())}; median non-empty sections: "
        f"{int(c.n_sections_nonempty.median())} of 20", "",
        "## Filings by year", "", "| Year | Filings |", "|---|---|",
        *[f"| {y} | {n:,} |" for y, n in by_year.items()], "", card_note,
    ]
    return "\n".join(lines)


In [ ]:
%%writefile $REPO/src/secprice/backtest/__init__.py
"""secprice.backtest"""


In [ ]:
%%writefile $REPO/src/secprice/backtest/engine.py
"""Backtest (SPEC.md §11; addendum P0.2). An economic-relevance check, not an investable strategy.

Signal s = P(Up) - P(Down) from the frozen T_trade model. Long if s >= training-period 80th pct, short if
<= 20th pct (percentiles from training folds only). Entry at the open of the first session at or after
decision_time; exit at the close of session h (entry = session 1). Calendar-time daily portfolio, equal weight per
leg, max 5% per name, one position per firm (a newer event replaces the older one at the new entry open), legs with
fewer than `min_names_per_side` names hold cash. Benchmark SPY on the identical intervals per position. Carhart
alpha is estimated only on the close-to-close holding component; the open-to-close entry-day component is
market-adjusted with SPY open-to-close (never a close-to-close factor against an open-to-close return).
"""
from __future__ import annotations

import numpy as np
import pandas as pd


def thresholds(train_scores: np.ndarray, lo_pct: float, hi_pct: float) -> tuple[float, float]:
    s = np.asarray(train_scores, float)
    s = s[np.isfinite(s)]
    return float(np.quantile(s, lo_pct)), float(np.quantile(s, hi_pct))


def positions(sig: pd.DataFrame, lo: float, hi: float, h: int, lag: int = 0) -> pd.DataFrame:
    """sig: event_id, ticker, entry_idx, score. Returns one row per position with entry/exit indexes (after
    one-per-firm replacement)."""
    s = sig.copy()
    s["side"] = np.where(s.score >= hi, 1, np.where(s.score <= lo, -1, 0))
    s = s[s.side != 0].copy()
    s["entry"] = s.entry_idx + lag
    s["exit"] = s.entry + h - 1
    s["exit_at_open"] = False
    s = s.sort_values(["ticker", "entry"]).reset_index(drop=True)
    for tk, g in s.groupby("ticker"):
        idx = g.index.to_list()
        for a, b in zip(idx[:-1], idx[1:]):
            if s.at[b, "entry"] <= s.at[a, "exit"]:
                s.at[a, "exit"] = s.at[b, "entry"]      # replaced: old position exits at the new entry open
                s.at[a, "exit_at_open"] = True
    return s[s.exit >= s.entry]


def daily_book(pos: pd.DataFrame, panel, spy: dict, n_sessions: int, max_w: float, min_names: int,
               cost_bps: float, borrow_bps: float) -> pd.DataFrame:
    """Per-day leg returns with interval-matched SPY legs and entry/hold decomposition."""
    T = n_sessions
    rec = {k: np.zeros(T) for k in ("long_entry", "long_hold", "short_entry", "short_hold", "spy_long_entry",
                                     "spy_long_hold", "n_long", "n_short", "turnover_long", "turnover_short")}
    contrib = {1: [[] for _ in range(T)], -1: [[] for _ in range(T)]}
    s_open, s_close, s_tri = spy["open"], spy["close_split_adj"], spy["total_return_index"]
    for p in pos.itertuples(index=False):
        if not panel.has(p.ticker):
            continue
        P = panel[p.ticker]
        o, c, tri, dv = P["open"], P["close_split_adj"], P["total_return_index"], P["dividend"]
        e, x = int(p.entry), int(min(p.exit, T - 1))
        for t in range(e, x + 1):
            if t == e:
                r = c[t] / o[t] - 1
                rs = s_close[t] / s_open[t] - 1
                part = "entry"
            elif p.exit_at_open and t == x:
                r = (o[t] + dv[t]) / c[t - 1] - 1
                rs = (s_open[t] + spy["dividend"][t]) / s_close[t - 1] - 1
                part = "hold"
            else:
                r = tri[t] / tri[t - 1] - 1
                rs = s_tri[t] / s_tri[t - 1] - 1
                part = "hold"
            if np.isfinite(r):
                contrib[p.side][t].append((r, rs if np.isfinite(rs) else 0.0, part, t == e, t == x))
    for side, leg in ((1, "long"), (-1, "short")):
        for t in range(T):
            cs = contrib[side][t]
            n = len(cs)
            rec[f"n_{leg}"][t] = n
            if n < min_names or n == 0:
                continue
            w = min(1.0 / n, max_w)
            for r, rs, part, is_entry, is_exit in cs:
                rec[f"{leg}_{part}"][t] += w * r
                if side == 1:
                    rec[f"spy_long_{part}"][t] += w * rs
                rec[f"turnover_{leg}"][t] += w * (int(is_entry) + int(is_exit))
    d = pd.DataFrame(rec)
    c = cost_bps / 1e4
    d["long_ret"] = d.long_entry + d.long_hold - c * d.turnover_long
    d["short_ret"] = d.short_entry + d.short_hold
    borrow = borrow_bps / 1e4 / 252
    d["ls_ret"] = d.long_ret - d.short_ret - c * d.turnover_short - borrow * (d.n_short >= min_names)
    d["spy_matched"] = d.spy_long_entry + d.spy_long_hold
    d["long_excess_spy"] = d.long_ret - d.spy_matched
    return d


def exposure_stats(d: pd.DataFrame, min_names: int) -> dict:
    """Turnover and exposure of the daily book (SPEC.md §11.7): average names held, share of days each leg is invested,
    share of days a leg holds cash because it has fewer than min_names positions, mean daily one-way turnover."""
    nl, ns = d.n_long.to_numpy(float), d.n_short.to_numpy(float)
    return {"avg_names_long": float(nl[nl > 0].mean()) if (nl > 0).any() else 0.0,
            "avg_names_short": float(ns[ns > 0].mean()) if (ns > 0).any() else 0.0,
            "share_days_long_invested": float((nl >= min_names).mean()), "share_days_short_invested": float((ns >= min_names).mean()),
            "share_days_long_in_cash": float(((nl > 0) & (nl < min_names)).mean()),
            "share_days_short_in_cash": float(((ns > 0) & (ns < min_names)).mean()),
            "turnover_long_daily": float(d.turnover_long.mean()), "turnover_short_daily": float(d.turnover_short.mean())}


def perf(r: np.ndarray, n_trials: int, rf: np.ndarray | None = None) -> dict:
    from ..evaluation.mcs_pbo import deflated_sharpe
    from scipy.stats import kurtosis, skew
    r = np.asarray(r, float)
    r = r[np.isfinite(r)]
    if len(r) < 20 or np.std(r) == 0:
        return {"n_days": int(len(r)), "note": "too few active days"}
    sr_d = np.mean(r) / np.std(r, ddof=1)
    eq = np.cumprod(1 + r)
    dd = float(np.min(eq / np.maximum.accumulate(eq) - 1))
    return {"n_days": int(len(r)), "ann_return": float(np.mean(r) * 252), "ann_vol": float(np.std(r, ddof=1) * np.sqrt(252)),
            "sharpe": float(sr_d * np.sqrt(252)), "max_drawdown": dd,
            "deflated_sharpe_prob": deflated_sharpe(sr_d, len(r), max(1, n_trials), float(skew(r)), float(kurtosis(r, fisher=False))),
            "total_return": float(eq[-1] - 1)}


def carhart_alpha(r_hold: np.ndarray, dates: pd.DatetimeIndex, factors: pd.DataFrame) -> dict:
    """Carhart alpha of the close-to-close holding component (interval-matched to daily factors)."""
    import statsmodels.api as sm
    f = factors.set_index(pd.to_datetime(factors["date"]))
    d = pd.DataFrame({"r": r_hold}, index=dates).join(f, how="inner")
    d = d[d.r != 0]
    if len(d) < 60:
        return {"alpha_ann": None, "note": "too few days"}
    X = sm.add_constant(d[["mkt_rf", "smb", "hml", "mom"]])
    m = sm.OLS(d.r - d.rf, X).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    return {"alpha_ann": float(m.params["const"] * 252), "alpha_t": float(m.tvalues["const"]), "n_days": int(len(d)),
            "betas": {k: float(m.params[k]) for k in ("mkt_rf", "smb", "hml", "mom")}}


In [ ]:
%%writefile $REPO/src/secprice/cli.py
"""secprice CLI (SPEC.md §17.2)."""
from __future__ import annotations

import argparse
import json
import sys


def main(argv=None) -> int:
    from . import pipeline as P
    ap = argparse.ArgumentParser(prog="secprice")
    sub = ap.add_subparsers(dest="cmd", required=True)
    r = sub.add_parser("run")
    r.add_argument("--profile", default=None)
    r.add_argument("--stages", nargs="+", required=True, help="S05, 'S00-S11' or a list")
    r.add_argument("--force", action="store_true")
    r.add_argument("--with-deps", action="store_true")
    r.add_argument("--set", nargs="*", default=[], help="dotted overrides key=value (JSON values)")
    for name in ("status", "preflight", "prereg-freeze", "freeze", "report", "env"):
        p = sub.add_parser(name)
        p.add_argument("--profile", default=None)
        if name == "prereg-freeze":
            p.add_argument("--force", action="store_true")
    u = sub.add_parser("unlock-test")
    u.add_argument("--profile", default=None)
    u.add_argument("--acknowledge-reuse", action="store_true")
    g = sub.add_parser("gold")
    g.add_argument("action", choices=["export", "import", "agreement"])
    g.add_argument("--profile", default=None)
    s = sub.add_parser("sync")
    s.add_argument("direction", choices=["push", "pull"])
    s.add_argument("--profile", required=True)
    s.add_argument("--local", required=True)
    s.add_argument("--remote", required=True)
    a = ap.parse_args(argv)

    def ov(pairs):
        out = {}
        for kv in pairs:
            k, v = kv.split("=", 1)
            try:
                v = json.loads(v)
            except json.JSONDecodeError:
                pass
            cur = out
            parts = k.split(".")
            for p_ in parts[:-1]:
                cur = cur.setdefault(p_, {})
            cur[parts[-1]] = v
        return out

    if a.cmd == "run":
        print(P.run(a.stages, a.profile, ov(a.set), force=a.force, with_deps=a.with_deps))
    elif a.cmd == "status":
        print(P.status(a.profile).to_string(index=False))
    elif a.cmd == "preflight":
        print(json.dumps(P.preflight(a.profile), indent=1, default=str))      # CPU projection before the download (G2)
        print("GPU preflight with a measured benchmark: secprice run --stages S13 --force")
    elif a.cmd == "prereg-freeze":
        print(P.prereg_freeze(a.profile, force=a.force))
    elif a.cmd == "freeze":
        print(P.freeze(a.profile))
    elif a.cmd == "unlock-test":
        print(P.unlock_test(a.profile, acknowledge_reuse=a.acknowledge_reuse))
    elif a.cmd == "report":
        print(P.run(["S24"], a.profile, force=True))
    elif a.cmd == "env":
        print(P.run(["S00"], a.profile, force=True))
    elif a.cmd == "gold":
        print(json.dumps(P.gold(a.action, a.profile), indent=1, default=str))
    elif a.cmd == "sync":
        print(P.sync(a.direction, a.local, a.remote, a.profile))
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile $REPO/src/secprice/common.py
"""Shared helpers for stages: artifact loading, clients, calendar/panel construction."""
from __future__ import annotations

import json
import os
import platform
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from .config import load_side_config
from .events.build import events_from_parquet
from .market.calendar import TradingCalendar
from .market.prices import PricePanel


def sec_client(ctx):
    """Real SEC client (User-Agent from env) or the fixture client when services['fixture_root'] is set."""
    from .sec.client import FixtureSECClient, SECClient
    fr = ctx.services.get("fixture_root")
    if fr:
        return FixtureSECClient(fr)
    return SECClient.from_config(ctx.cfg, ctx.storage.hot_root / "sec_cache", ctx.log)


def roster(cfg) -> dict:
    return load_side_config(cfg, "models.yaml")


def taxonomy(cfg):
    from .parse.taxonomy import Taxonomy8K
    return Taxonomy8K(load_side_config(cfg, f"{cfg['taxonomy_version']}.yaml"))


def question_bank(cfg) -> dict:
    return load_side_config(cfg, "question_bank_v1.yaml")


def read(ctx, stage: str, name: str, columns=None) -> pd.DataFrame:
    return pd.read_parquet(ctx.dep(stage) / name, columns=columns)


def read_json(ctx, stage: str, name: str):
    return json.loads((ctx.dep(stage) / name).read_text(encoding="utf-8"))


def decode_lists(df: pd.DataFrame, cols) -> pd.DataFrame:
    """List columns are stored as JSON strings in Parquet (storage._to_parquet_safe); decode them on read."""
    for c in cols:
        if c in df.columns:
            df[c] = df[c].map(lambda v: json.loads(v) if isinstance(v, str) and v[:1] == "[" else
                              (list(v) if isinstance(v, (list, tuple, np.ndarray)) else []))
    return df


def manifest(ctx) -> pd.DataFrame:
    return decode_lists(read(ctx, "S04", "filing_manifest.parquet"), ["items_list"])


def calendar(ctx) -> TradingCalendar:
    return TradingCalendar.from_frame(read(ctx, "S03", "calendar.parquet"))


def panel_and_spy(ctx, cal: TradingCalendar):
    px = read(ctx, "S03", "prices.parquet")
    panel = PricePanel(px[px.ticker != "SPY"], cal)
    spy = PricePanel(px[px.ticker == "SPY"], cal)["SPY"]
    return panel, spy


def universe(ctx, selected_only: bool = True) -> pd.DataFrame:
    u = read(ctx, "S02", "universe.parquet")
    return u[u.selected] if selected_only else u


def events(ctx) -> pd.DataFrame:
    ev = events_from_parquet(read(ctx, "S07", "events.parquet"))
    ev["decision_date"] = pd.to_datetime(ev["decision_time"]).dt.tz_convert("America/New_York").dt.tz_localize(None).dt.normalize()
    return ev


def event_buckets(ctx) -> pd.DataFrame:
    """event_id -> sp_bucket (S&P 500 vs non-S&P 500, firm level) and point-in-time membership (D-056)."""
    from .evaluation.buckets import event_buckets as _eb
    return _eb(events(ctx), universe(ctx))


def factors(ctx) -> pd.DataFrame:
    return read(ctx, "S03", "factors.parquet")


def usable_cutoff(ctx) -> dict:
    return read_json(ctx, "S07", "usable_cutoff.json")


def env_manifest() -> dict:
    import importlib
    pk = {}
    for m in ["numpy", "pandas", "pyarrow", "sklearn", "scipy", "lightgbm", "catboost", "optuna", "statsmodels", "arch",
              "torch", "transformers", "sentence_transformers", "pandas_market_calendars", "yfinance", "lxml", "yaml",
              "matplotlib", "reportlab", "lmformatenforcer", "huggingface_hub", "jsonschema"]:
        try:
            x = importlib.import_module(m)
            pk[m] = getattr(x, "__version__", "?")
        except Exception:
            pk[m] = None
    gpu = None
    try:
        import torch
        if torch.cuda.is_available():
            gpu = {"name": torch.cuda.get_device_name(0), "count": torch.cuda.device_count(),
                   "mem_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)}
    except Exception:
        pass
    return {"python": sys.version, "platform": platform.platform(), "packages": pk, "gpu": gpu,
            "cpu_count": os.cpu_count(), "in_colab": "google.colab" in sys.modules}


def disk_free_gb(path: Path) -> float | None:
    import shutil
    try:
        return round(shutil.disk_usage(path).free / 1e9, 1)
    except Exception:
        return None


In [ ]:
%%writefile $REPO/src/secprice/config.py
"""Configuration: load default.yaml, overlay profile and owner overrides, validate, hash (SPEC.md §15)."""
from __future__ import annotations

import copy
import hashlib
import json
import os
from datetime import date
from pathlib import Path
from typing import Any

import yaml

PROFILES = ("smoke", "pilot", "full")
# Sections excluded from the config hash: paths and anything environment-specific (SPEC.md §15).
_HASH_EXCLUDE = {"paths", "_meta"}


def repo_root() -> Path:
    env = os.environ.get("SECPRICE_REPO")
    if env:
        return Path(env)
    return Path(__file__).resolve().parents[2]


def deep_merge(base: dict, over: dict) -> dict:
    out = copy.deepcopy(base)
    for k, v in (over or {}).items():
        if isinstance(v, dict) and isinstance(out.get(k), dict):
            out[k] = deep_merge(out[k], v)
        else:
            out[k] = copy.deepcopy(v)
    return out


def _flatten(d: dict, prefix: str = "") -> dict:
    flat = {}
    for k, v in d.items():
        key = f"{prefix}.{k}" if prefix else str(k)
        if isinstance(v, dict):
            flat.update(_flatten(v, key))
        else:
            flat[key] = v
    return flat


def get(cfg: dict, dotted: str, default: Any = None) -> Any:
    cur: Any = cfg
    for part in dotted.split("."):
        if not isinstance(cur, dict) or part not in cur:
            return default
        cur = cur[part]
    return cur


def load_yaml(path: Path) -> dict:
    with open(path, encoding="utf-8") as f:
        return yaml.safe_load(f) or {}


def load_config(profile: str | None = None, overrides: dict | None = None, root: Path | None = None) -> dict:
    root = Path(root) if root else repo_root()
    cfg = load_yaml(root / "configs" / "default.yaml")
    profile = profile or cfg.get("run_profile", "smoke")
    if profile not in PROFILES:
        raise ValueError(f"run_profile must be one of {PROFILES}, got {profile!r}")
    prof_path = root / "configs" / "profiles" / f"{profile}.yaml"
    prof = load_yaml(prof_path) if prof_path.exists() else {}
    cfg = deep_merge(cfg, prof)
    cfg["run_profile"] = profile
    applied = []
    if overrides:
        flat_before = _flatten(cfg)
        cfg = deep_merge(cfg, overrides)
        for k, v in _flatten(overrides).items():
            applied.append({"key": k, "before": flat_before.get(k), "after": v})
    cfg["_meta"] = {"repo_root": str(root), "profile_file": str(prof_path), "owner_overrides": applied}
    validate(cfg)
    return cfg


class ConfigError(ValueError):
    pass


def _req(cond: bool, msg: str) -> None:
    if not cond:
        raise ConfigError(msg)


def validate(cfg: dict) -> None:
    """Types, ranges and required sections (SPEC.md §15). Secrets are checked only when a stage needs them."""
    for sec in ["run_profile", "seed", "paths", "sec", "universe", "study", "market", "timing", "text",
                "taxonomy_version", "targets", "labels", "splits", "models", "rag", "distill", "tuning",
                "gates", "audit", "bootstrap", "fdr", "trivial_effect_bps", "backtest", "budget", "extensions"]:
        _req(sec in cfg, f"config section missing: {sec}")
    _req(cfg["run_profile"] in PROFILES, "run_profile invalid")
    _req(isinstance(cfg["seed"], int), "seed must be int")
    s = cfg["sec"]
    _req(0 < float(s["max_rps"]) <= 10, "sec.max_rps must be in (0, 10] (SEC ceiling 10)")
    _req(int(s["max_retries"]) >= 1, "sec.max_retries >= 1")
    st, en = date.fromisoformat(cfg["study"]["start"]), date.fromisoformat(cfg["study"]["end"])
    _req(st < en, "study.start must precede study.end")
    _req(date.fromisoformat(cfg["market"]["warmup_start"]) < st, "market.warmup_start must precede study.start")
    _req(int(cfg["universe"]["n_stocks"]) > 0, "universe.n_stocks > 0")
    hz = cfg["targets"]["horizons"]
    _req(all(isinstance(h, int) and h > 0 for h in hz), "targets.horizons must be positive ints")
    _req(cfg["targets"]["primary"]["horizon"] in hz, "primary horizon must be in targets.horizons")
    _req(cfg["targets"]["primary"]["target_def"] in ("T_trade", "T_info"), "primary target_def invalid")
    _req(float(cfg["labels"]["k"]) > 0, "labels.k > 0")
    sp = cfg["splits"]
    ds, lt = date.fromisoformat(sp["dev_start"]), date.fromisoformat(sp["locked_test_start"])
    _req(st < ds <= lt, "splits: study.start < dev_start <= locked_test_start")
    _req(float(sp["embargo_days"]) >= 0, "splits.embargo_days >= 0")
    _req(int(cfg["contam_window_sessions"]) >= 1, "contam_window_sessions >= 1")
    _req(int(cfg["bootstrap"]["n"]) >= 10, "bootstrap.n >= 10")
    _req(0 < float(cfg["fdr"]["q"]) < 1, "fdr.q in (0,1)")
    for k, v in cfg["extensions"].items():
        _req(isinstance(v, bool), f"extensions.{k} must be bool")
    for k in ("public_lag_minutes", "decision_latency_minutes"):
        _req(float(cfg["timing"][k]) >= 0, f"timing.{k} >= 0")


def hashable_view(cfg: dict) -> dict:
    return {k: v for k, v in cfg.items() if k not in _HASH_EXCLUDE}


def canonical_json(obj: Any) -> str:
    return json.dumps(obj, sort_keys=True, separators=(",", ":"), default=str)


def config_hash(cfg: dict) -> str:
    """Hash of the resolved config, excluding paths and secrets. Stable across key order."""
    return hashlib.sha256(canonical_json(hashable_view(cfg)).encode()).hexdigest()[:16]


def subset_hash(cfg: dict, keys: list[str], extra: Any = None) -> str:
    sub = {k: get(cfg, k) for k in sorted(keys)}
    payload = {"sub": sub, "extra": extra, "profile": cfg.get("run_profile")}
    return hashlib.sha256(canonical_json(payload).encode()).hexdigest()[:16]


def load_side_config(cfg: dict, name: str) -> dict:
    root = Path(cfg.get("_meta", {}).get("repo_root") or repo_root())
    return load_yaml(root / "configs" / name)


def text_sha256(path: Path) -> str:
    """Hash of a text file's normalized content (LF line endings, trailing whitespace trimmed), so the repository copy
    and the notebook-embedded copy of the pre-registration hash identically."""
    t = Path(path).read_text(encoding="utf-8").replace(chr(13) + chr(10), chr(10)).rstrip() + chr(10)
    return hashlib.sha256(t.encode("utf-8")).hexdigest()


def file_sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


In [ ]:
%%writefile $REPO/src/secprice/evaluation/__init__.py
"""secprice.evaluation"""


In [ ]:
%%writefile $REPO/src/secprice/evaluation/bootstrap.py
"""Block bootstrap by calendar month (SPEC.md §9.2 Uncertainty). Paired: both models resample the same months."""
from __future__ import annotations

import numpy as np
import pandas as pd

from .metrics import confusion, f1_from_cm


def month_ids(dates) -> np.ndarray:
    m = pd.to_datetime(pd.Series(dates)).dt.to_period("M").astype(str).to_numpy()
    _, inv = np.unique(m, return_inverse=True)
    return inv


def _per_month_cm(y, yhat, mid, n_months):
    cms = np.zeros((n_months, 3, 3), dtype=np.int64)
    np.add.at(cms, (mid, y.astype(int), yhat.astype(int)), 1)
    return cms


def macro_f1_ci(y, yhat, dates, n_boot=1000, seed=0, alpha=0.05) -> dict:
    mid = month_ids(dates)
    nm = mid.max() + 1 if len(mid) else 0
    cms = _per_month_cm(np.asarray(y), np.asarray(yhat), mid, nm)
    point = f1_from_cm(cms.sum(0))["macro_f1"]
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(n_boot):
        s = rng.integers(0, nm, nm)
        v = f1_from_cm(cms[s].sum(0))["macro_f1"]
        if v is not None:
            vals.append(v)
    lo, hi = (np.quantile(vals, [alpha / 2, 1 - alpha / 2]) if vals else (np.nan, np.nan))
    return {"value": point, "ci_low": float(lo), "ci_high": float(hi), "n_months": int(nm)}


def paired_macro_f1_diff(y, yhat_a, yhat_b, dates, n_boot=1000, seed=0, alpha=0.05) -> dict:
    """Lift of model A over model B on identical rows; CI by month-block bootstrap; two-sided p from the
    bootstrap distribution (share of resamples on the other side of zero, doubled)."""
    y = np.asarray(y)
    mid = month_ids(dates)
    nm = mid.max() + 1 if len(mid) else 0
    ca = _per_month_cm(y, np.asarray(yhat_a), mid, nm)
    cb = _per_month_cm(y, np.asarray(yhat_b), mid, nm)
    point = f1_from_cm(ca.sum(0))["macro_f1"] - f1_from_cm(cb.sum(0))["macro_f1"]
    rng = np.random.default_rng(seed)
    d = []
    for _ in range(n_boot):
        s = rng.integers(0, nm, nm)
        fa, fb = f1_from_cm(ca[s].sum(0))["macro_f1"], f1_from_cm(cb[s].sum(0))["macro_f1"]
        if fa is not None and fb is not None:
            d.append(fa - fb)
    d = np.array(d)
    lo, hi = (np.quantile(d, [alpha / 2, 1 - alpha / 2]) if len(d) else (np.nan, np.nan))
    p = float(min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))) if len(d) else np.nan
    return {"value": float(point), "ci_low": float(lo), "ci_high": float(hi), "p_boot": p, "n_months": int(nm)}


def mean_ci(values, dates, n_boot=1000, seed=0, alpha=0.05) -> dict:
    """CI of a mean (e.g. CAR) with month-block resampling."""
    v = np.asarray(values, float)
    ok = np.isfinite(v)
    v = v[ok]
    mid = month_ids(pd.Series(dates)[ok])
    if len(v) == 0:
        return {"value": None, "ci_low": None, "ci_high": None, "n_months": 0}
    nm = mid.max() + 1
    sums = np.bincount(mid, weights=v, minlength=nm)
    cnts = np.bincount(mid, minlength=nm)
    rng = np.random.default_rng(seed)
    bs = []
    for _ in range(n_boot):
        s = rng.integers(0, nm, nm)
        c = cnts[s].sum()
        if c:
            bs.append(sums[s].sum() / c)
    lo, hi = np.quantile(bs, [alpha / 2, 1 - alpha / 2])
    return {"value": float(v.mean()), "ci_low": float(lo), "ci_high": float(hi), "n_months": int(nm)}


In [ ]:
%%writefile $REPO/src/secprice/evaluation/buckets.py
"""Index-membership buckets: S&P 500 vs non-S&P 500 vs combined (owner request, decision log D-056).

Firm bucket (`sp_bucket`): "SP500" if the firm was an S&P 500 member at any time in the study window according to the
pinned fja05680/sp500 list, else "NonSP500". The list is built from Wikipedia changes and is incomplete in early years,
so "NonSP500" means "not found in that list", never "verified never a member".
Point-in-time membership on the event's decision date (`sp500_member_at_event`) is reported alongside; it is not a model
feature. Every bucket result is computed from the same frozen models and walk-forward predictions as the combined result.
"""
from __future__ import annotations

import json

import numpy as np
import pandas as pd

from .bootstrap import macro_f1_ci, paired_macro_f1_diff
from .metrics import classification_metrics, power_label

BUCKETS = ("Combined", "SP500", "NonSP500")
LABELS = {"Combined": "Combined (all firms)", "SP500": "S&P 500 members (in the study window)",
          "NonSP500": "Non-S&P 500 (not in the fja05680 list)"}
PCOLS = ["p_down", "p_side", "p_up"]


def firm_bucket(ever_sp500) -> np.ndarray:
    return np.where(pd.Series(ever_sp500).fillna(False).astype(bool).to_numpy(), "SP500", "NonSP500")


def _member_on(intervals_json, day) -> bool:
    try:
        ivs = json.loads(intervals_json) if isinstance(intervals_json, str) else (intervals_json or [])
    except (TypeError, ValueError):
        return False
    for a, b in ivs:
        if pd.Timestamp(a) <= day and (b is None or pd.isna(pd.Timestamp(b)) or pd.Timestamp(b) >= day):
            return True
    return False


def event_buckets(ev: pd.DataFrame, u: pd.DataFrame) -> pd.DataFrame:
    """event_id -> cik, sp_bucket (firm level), sp500_member_at_event (point in time, decision date)."""
    uu = u.drop_duplicates("cik").set_index(u.drop_duplicates("cik").cik.astype(int))
    if "sp_bucket" in uu:
        fb = uu["sp_bucket"]
    else:
        fb = pd.Series(firm_bucket(uu.get("ever_sp500", pd.Series(False, index=uu.index))), index=uu.index)
    cik = ev["cik"].astype(int)
    out = pd.DataFrame({"event_id": ev["event_id"].values, "cik": cik.values,
                        "sp_bucket": cik.map(fb).fillna("NonSP500").values})
    iv = cik.map(uu["sp500_intervals"]) if "sp500_intervals" in uu else pd.Series([None] * len(ev))
    day = pd.to_datetime(ev["decision_date"]) if "decision_date" in ev else \
        pd.to_datetime(ev["decision_time"]).dt.tz_convert("America/New_York").dt.tz_localize(None).dt.normalize()
    out["sp500_member_at_event"] = [_member_on(i, d) for i, d in zip(iv.values, day.values)]
    return out


def expand(df: pd.DataFrame, bk: pd.DataFrame, key: str = "event_id") -> pd.DataFrame:
    """Rows once per bucket they belong to: their own bucket and 'Combined'."""
    x = df.merge(bk[[key, "sp_bucket"]], on=key, how="left")
    x["sp_bucket"] = x["sp_bucket"].fillna("NonSP500")
    return pd.concat([x, x.assign(sp_bucket="Combined")], ignore_index=True)


def score_predictions(preds: pd.DataFrame, bk: pd.DataFrame, n_boot: int, seed: int, by=("run_id",)) -> pd.DataFrame:
    """Macro-F1 (month-block CI) and secondary metrics of existing predictions, per bucket. Uses the model's own decisions."""
    rows = []
    if preds is None or preds.empty:
        return pd.DataFrame()
    x = expand(preds, bk)
    for keys, g in x.groupby(list(by) + ["sp_bucket"], sort=False):
        g = g[g.y.notna()]
        if len(g) < 3:
            continue
        y = g.y.astype(int).values
        prior = np.bincount(y, minlength=3) / len(y)
        m = classification_metrics(y, g[PCOLS].values, prior, g.yhat.astype(int).values)
        ci = macro_f1_ci(y, g.yhat.astype(int).values, g.date.values, n_boot, seed)
        rk = dict(zip(list(by) + ["sp_bucket"], keys if isinstance(keys, tuple) else (keys,)))
        rows.append({**rk, "n_events": int(len(g)), "n_firms": int(g.event_id.map(bk.set_index("event_id").cik).nunique()),
                     "macro_f1": m.get("macro_f1"), "ci_low": ci["ci_low"], "ci_high": ci["ci_high"],
                     "accuracy": m.get("accuracy"), "balanced_accuracy": m.get("balanced_accuracy"), "auc_macro_ovr": m.get("auc_macro_ovr"),
                     "logloss": m.get("logloss"), "logloss_prior": m.get("logloss_prior"), "ece": m.get("ece"),
                     "share_up": float(prior[2]), "share_side": float(prior[1]), "share_down": float(prior[0]),
                     "power": power_label(len(g), ci["ci_high"] - ci["ci_low"] if np.isfinite(ci["ci_low"]) else None)})
    out = pd.DataFrame(rows)
    if len(out):
        order = {b: i for i, b in enumerate(BUCKETS)}
        out = out.assign(_o=out.sp_bucket.map(order)).sort_values(list(by) + ["_o"], kind="stable").drop(columns="_o")
    return out.reset_index(drop=True)


def pooled_vs_bucket(p_pooled: pd.DataFrame, p_bucket: pd.DataFrame, n_boot: int, seed: int) -> dict:
    """Paired comparison on identical rows: the combined (pooled) model vs a model trained on the bucket only."""
    m = p_pooled[["event_id", "y", "yhat", "date"]].merge(p_bucket[["event_id", "yhat"]], on="event_id",
                                                           suffixes=("_pooled", "_bucket"))
    if len(m) < 30:
        return {"n_events": int(len(m)), "note": "fewer than 30 shared rows"}
    d = paired_macro_f1_diff(m.y.astype(int).values, m.yhat_pooled.astype(int).values, m.yhat_bucket.astype(int).values,
                             m.date.values, n_boot, seed)
    return {"n_events": int(len(m)), "pooled_minus_bucket_macro_f1": d["value"], "ci_low": d["ci_low"], "ci_high": d["ci_high"],
            "p_boot": d["p_boot"]}


def composition(ev: pd.DataFrame, bk: pd.DataFrame, u: pd.DataFrame, labels: pd.DataFrame | None = None,
                features: pd.DataFrame | None = None) -> pd.DataFrame:
    """What differs between the buckets before any model: firms, events, event mix, timing, size, data availability."""
    x = expand(ev, bk)
    if "sp500_member_at_event" in bk and "sp500_member_at_event" not in x:
        x = x.merge(bk[["event_id", "sp500_member_at_event"]], on="event_id", how="left")
    if labels is not None and len(labels):
        x = x.merge(labels, on="event_id", how="left")
    if features is not None and len(features):
        keep = [c for c in ("acc_log_mcap", "sue_event", "has_text", "has_accounting", "px_vol_60") if c in features]
        x = x.merge(features[["event_id"] + keep], on="event_id", how="left")
    rows = []
    for b, g in x.groupby("sp_bucket", sort=False):
        r = {"sp_bucket": b, "firms": int(g.cik.nunique()), "events": int(len(g)),
             "events_per_firm": float(len(g) / max(1, g.cik.nunique())),
             "share_8k": float((g.event_type == "8K").mean()), "share_10q": float((g.event_type == "periodic_10Q").mean()),
             "share_10k": float((g.event_type == "periodic_10K").mean()),
             "share_verified_earnings": float((g.earnings_type == "earnings_announcement_verified").mean()),
             "share_ts_quality_A": float((g.ts_quality == "A").mean()),
             "member_at_event_share": float(g.sp500_member_at_event.mean()) if "sp500_member_at_event" in g else None,
             "first_year": int(pd.to_datetime(g.decision_date).dt.year.min()) if len(g) else None}
        if "label3" in g:
            y = g.label3.dropna()
            r.update({"labelled_T_trade_h1": int(len(y)), "share_up": float((y == 2).mean()) if len(y) else None,
                      "share_down": float((y == 0).mean()) if len(y) else None})
        for c, nm in (("acc_log_mcap", "median_log_mcap"), ("px_vol_60", "median_vol_60")):
            if c in g:
                r[nm] = float(g[c].median()) if g[c].notna().any() else None
        if "sue_event" in g:
            r["sue_available_share"] = float(g.sue_event.notna().mean())
        rows.append(r)
    out = pd.DataFrame(rows)
    out["label"] = out.sp_bucket.map(LABELS)
    order = {b: i for i, b in enumerate(BUCKETS)}
    return out.sort_values("sp_bucket", key=lambda s: s.map(order)).reset_index(drop=True)


In [ ]:
%%writefile $REPO/src/secprice/evaluation/fdr.py
"""Benjamini-Hochberg FDR within pre-declared families (SPEC.md §3 fdr.q)."""
from __future__ import annotations

import numpy as np
import pandas as pd


def bh(pvals, q: float = 0.05) -> tuple[np.ndarray, np.ndarray]:
    p = np.asarray(pvals, float)
    out_adj = np.full(len(p), np.nan)
    rej = np.zeros(len(p), bool)
    ok = np.isfinite(p)
    if not ok.any():
        return out_adj, rej
    pv = p[ok]
    m = len(pv)
    order = np.argsort(pv)
    ranked = pv[order] * m / (np.arange(m) + 1)
    adj = np.minimum.accumulate(ranked[::-1])[::-1]
    adj = np.minimum(adj, 1.0)
    res = np.empty(m)
    res[order] = adj
    out_adj[ok] = res
    rej[ok] = res <= q
    return out_adj, rej


def apply_families(df: pd.DataFrame, family_col: str = "fdr_family", p_col: str = "p_value", q: float = 0.05) -> pd.DataFrame:
    df = df.copy()
    df["p_adj_bh"] = np.nan
    df["reject_bh"] = False
    for fam, g in df.groupby(family_col):
        adj, rej = bh(g[p_col].values, q)
        df.loc[g.index, "p_adj_bh"] = adj
        df.loc[g.index, "reject_bh"] = rej
    return df


In [ ]:
%%writefile $REPO/src/secprice/evaluation/mcs_pbo.py
"""Model Confidence Set and PBO (CSCV) - diagnostics only; they never replace or rescue walk-forward and
locked-test evidence (SPEC.md §9.2). Each diagnostic records which loss streams and configurations entered it."""
from __future__ import annotations

import itertools

import numpy as np
import pandas as pd


def model_confidence_set(losses: pd.DataFrame, size: float = 0.10, reps: int = 1000, seed: int = 0) -> dict:
    """losses: rows = time blocks (months), columns = models; lower is better."""
    L = losses.dropna(axis=0, how="any")
    if L.shape[0] < 6 or L.shape[1] < 2:
        return {"included": list(L.columns), "pvalues": {}, "note": "too few blocks/models for MCS", "n_blocks": int(L.shape[0])}
    try:
        from arch.bootstrap import MCS
        m = MCS(L.values, size=size, reps=reps, block_size=max(1, int(np.sqrt(L.shape[0]))), seed=seed)
        m.compute()
        inc = [L.columns[i] for i in m.included]
        pv = {L.columns[int(i)]: float(p) for i, p in zip(m.pvalues.index, m.pvalues.iloc[:, 0])}
        return {"included": inc, "pvalues": pv, "n_blocks": int(L.shape[0]), "method": "arch.bootstrap.MCS (Hansen-Lunde-Nason)"}
    except Exception as e:
        return {"included": list(L.columns), "pvalues": {}, "note": f"MCS unavailable: {e}", "n_blocks": int(L.shape[0])}


def pbo_cscv(perf: pd.DataFrame, n_splits: int = 8) -> dict:
    """perf: rows = time blocks, columns = configurations; higher is better. Bailey et al. CSCV."""
    P = perf.dropna(axis=1, how="any")
    T, N = P.shape
    if N < 2 or T < n_splits:
        return {"pbo": None, "note": f"need >=2 configs and >= {n_splits} blocks (have {N}, {T})", "n_configs": int(N)}
    groups = np.array_split(np.arange(T), n_splits)
    logits = []
    for comb in itertools.combinations(range(n_splits), n_splits // 2):
        is_idx = np.concatenate([groups[i] for i in comb])
        oos_idx = np.concatenate([groups[i] for i in range(n_splits) if i not in comb])
        is_perf = P.iloc[is_idx].mean()
        oos_perf = P.iloc[oos_idx].mean()
        best = is_perf.idxmax()
        rank = (oos_perf.rank(pct=True)[best] * N) / (N + 1)
        rank = min(max(rank, 1e-6), 1 - 1e-6)
        logits.append(np.log(rank / (1 - rank)))
    logits = np.array(logits)
    return {"pbo": float((logits <= 0).mean()), "n_combinations": int(len(logits)), "n_configs": int(N),
            "n_blocks": int(T), "logit_median": float(np.median(logits))}


def deflated_sharpe(sr: float, n_obs: int, n_trials: int, skew: float = 0.0, kurt: float = 3.0,
                    sr_var_trials: float | None = None) -> float | None:
    """Deflated Sharpe ratio (Bailey & Lopez de Prado) of a per-period Sharpe `sr` given the number of trials."""
    from scipy.stats import norm
    if n_obs < 3 or n_trials < 1 or not np.isfinite(sr):
        return None
    v = sr_var_trials if sr_var_trials is not None else 1.0 / max(n_obs - 1, 1)
    e = 0.5772156649
    sr0 = np.sqrt(v) * ((1 - e) * norm.ppf(1 - 1.0 / n_trials) + e * norm.ppf(1 - 1.0 / (n_trials * np.e))) if n_trials > 1 else 0.0
    den = np.sqrt(max(1e-12, 1 - skew * sr + (kurt - 1) / 4 * sr ** 2))
    return float(norm.cdf((sr - sr0) * np.sqrt(n_obs - 1) / den))


In [ ]:
%%writefile $REPO/src/secprice/evaluation/metrics.py
"""Core metrics (SPEC.md §9.2). Every metric returns None with a reason when it is inapplicable (never 0)."""
from __future__ import annotations

import numpy as np
import pandas as pd

CLASSES = (0, 1, 2)          # Down, Sideways, Up
CLASS_NAMES = {0: "down", 1: "sideways", 2: "up"}


def _safe(fn, *a, **k):
    try:
        v = fn(*a, **k)
        return float(v) if v is not None and np.isfinite(v) else None
    except Exception:
        return None


def confusion(y: np.ndarray, yhat: np.ndarray, k: int = 3) -> np.ndarray:
    cm = np.zeros((k, k), dtype=np.int64)
    np.add.at(cm, (y.astype(int), yhat.astype(int)), 1)
    return cm


def f1_from_cm(cm: np.ndarray) -> dict:
    tp = np.diag(cm).astype(float)
    pred = cm.sum(0).astype(float)
    true = cm.sum(1).astype(float)
    with np.errstate(divide="ignore", invalid="ignore"):
        p = np.where(pred > 0, tp / pred, np.nan)
        r = np.where(true > 0, tp / true, np.nan)
        f = np.where((p + r) > 0, 2 * p * r / (p + r), 0.0)
    f = np.where(true > 0, np.nan_to_num(f), np.nan)
    present = true > 0
    macro = float(np.nanmean(f[present])) if present.any() else None
    weighted = float(np.nansum(f[present] * true[present]) / true[present].sum()) if present.any() else None
    return {"precision": p, "recall": r, "f1": f, "macro_f1": macro, "weighted_f1": weighted,
            "accuracy": float(tp.sum() / cm.sum()) if cm.sum() else None,
            "balanced_accuracy": float(np.nanmean(r[present])) if present.any() else None}


def ece_top(proba: np.ndarray, y: np.ndarray, bins: int = 15) -> float | None:
    if len(y) == 0:
        return None
    conf = proba.max(1)
    pred = proba.argmax(1)
    edges = np.linspace(0, 1, bins + 1)
    e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return float(e)


def reliability(proba: np.ndarray, y: np.ndarray, cls: int, bins: int = 10) -> pd.DataFrame:
    p = proba[:, cls]
    edges = np.linspace(0, 1, bins + 1)
    rows = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (p > lo) & (p <= hi)
        if m.any():
            rows.append({"bin_lo": lo, "bin_hi": hi, "mean_pred": float(p[m].mean()), "frac_pos": float((y[m] == cls).mean()),
                         "n": int(m.sum())})
    return pd.DataFrame(rows)


def decide(proba: np.ndarray, prior: np.ndarray, n_train: int | None = None) -> np.ndarray:
    """Cost-balanced decision rule: argmax p(c|x) / prior(c), prior from the training fold.

    The prior is Laplace-smoothed and a class absent from training is never predicted (otherwise a near-zero prior
    would make that class win every row)."""
    prior = np.asarray(prior, float)
    n = n_train if n_train else 1000
    pr = (prior * n + 1.0) / (n + len(prior))
    score = proba / pr
    score[:, prior <= 0] = -np.inf
    return np.argmax(score, axis=1)


def classification_metrics(y: np.ndarray, proba: np.ndarray, prior: np.ndarray, yhat: np.ndarray | None = None,
                           coverage_levels=(1.0, 0.5, 0.25, 0.1), ece_max: float = 0.10) -> dict:
    from sklearn.metrics import (average_precision_score, cohen_kappa_score, log_loss, matthews_corrcoef,
                                 roc_auc_score)
    y = np.asarray(y).astype(int)
    proba = np.clip(np.asarray(proba, float), 1e-9, 1)
    proba = proba / proba.sum(1, keepdims=True)
    yhat = decide(proba, prior) if yhat is None else yhat
    out: dict = {"n": int(len(y))}
    if len(y) == 0:
        out["null_reason"] = "no_rows"
        return out
    cm = confusion(y, yhat)
    f = f1_from_cm(cm)
    out.update({k: f[k] for k in ("accuracy", "balanced_accuracy", "macro_f1", "weighted_f1")})
    for c in CLASSES:
        nm = CLASS_NAMES[c]
        out[f"precision_{nm}"] = None if np.isnan(f["precision"][c]) else float(f["precision"][c])
        out[f"recall_{nm}"] = None if np.isnan(f["recall"][c]) else float(f["recall"][c])
        out[f"f1_{nm}"] = None if np.isnan(f["f1"][c]) else float(f["f1"][c])
        pos = (y == c)
        out[f"auc_{nm}"] = _safe(roc_auc_score, pos, proba[:, c]) if 0 < pos.sum() < len(y) else None
        out[f"prauc_{nm}"] = _safe(average_precision_score, pos, proba[:, c]) if pos.any() else None
    present = [c for c in CLASSES if 0 < (y == c).sum() < len(y)]
    out["auc_macro_ovr"] = (float(np.mean([out[f"auc_{CLASS_NAMES[c]}"] for c in present]))
                            if len(present) == 3 and all(out[f"auc_{CLASS_NAMES[c]}"] is not None for c in present) else None)
    if out["auc_macro_ovr"] is None:
        out["auc_macro_ovr_null_reason"] = "a class is absent"
    pr = [out[f"prauc_{CLASS_NAMES[c]}"] for c in CLASSES if out[f"prauc_{CLASS_NAMES[c]}"] is not None]
    out["prauc_macro"] = float(np.mean(pr)) if len(pr) == 3 else None
    out["kappa"] = _safe(cohen_kappa_score, y, yhat)
    out["mcc"] = _safe(matthews_corrcoef, y, yhat)
    out["logloss"] = _safe(log_loss, y, proba, labels=list(CLASSES))
    pri = np.tile(np.clip(prior, 1e-9, 1), (len(y), 1))
    out["logloss_prior"] = _safe(log_loss, y, pri, labels=list(CLASSES))
    onehot = np.eye(3)[y]
    out["brier"] = float(np.mean(np.sum((proba - onehot) ** 2, axis=1)))
    out["ece"] = ece_top(proba, y)
    out["guardrail_logloss_ok"] = (out["logloss"] is not None and out["logloss_prior"] is not None
                                   and out["logloss"] <= out["logloss_prior"])
    out["guardrail_ece_ok"] = out["ece"] is not None and out["ece"] <= ece_max     # pre-registered ece_max (0.10)
    out["cm"] = cm.tolist()
    conf = proba.max(1)
    order = np.argsort(-conf, kind="stable")
    for cov in coverage_levels:
        m = order[: max(1, int(round(cov * len(y))))]
        cmc = confusion(y[m], yhat[m])
        fc = f1_from_cm(cmc)
        out[f"acc_at_{int(cov * 100)}"] = fc["accuracy"]
        out[f"macro_f1_at_{int(cov * 100)}"] = fc["macro_f1"]
    return out


def regression_metrics(y: np.ndarray, yhat: np.ndarray, hist_mean: np.ndarray | None = None,
                       dates: pd.Series | None = None, kind: str = "return") -> dict:
    from scipy.stats import spearmanr
    y, yhat = np.asarray(y, float), np.asarray(yhat, float)
    ok = np.isfinite(y) & np.isfinite(yhat)
    y, yhat = y[ok], yhat[ok]
    out = {"n": int(ok.sum())}
    if len(y) < 3:
        out["null_reason"] = "fewer_than_3_rows"
        return out
    ss_res = np.sum((y - yhat) ** 2)
    out["r2"] = float(1 - ss_res / np.sum((y - y.mean()) ** 2)) if np.var(y) > 0 else None
    if hist_mean is not None:
        hm = np.asarray(hist_mean, float)[ok]
        den = np.sum((y - hm) ** 2)
        out["oos_r2_vs_hist_mean"] = float(1 - ss_res / den) if den > 0 else None
    out["mae"] = float(np.mean(np.abs(y - yhat)))
    out["rmse"] = float(np.sqrt(np.mean((y - yhat) ** 2)))
    out["spearman"] = _safe(lambda: spearmanr(y, yhat).statistic)
    out["directional_accuracy"] = float(np.mean(np.sign(y) == np.sign(yhat))) if kind == "return" else None
    if kind == "vol_ratio":
        rv, fv = np.exp(2 * y), np.exp(2 * yhat)
        q = rv / fv - np.log(rv / fv) - 1
        out["qlike"] = float(np.mean(q))
    if dates is not None:
        d = pd.to_datetime(pd.Series(dates).reset_index(drop=True)[ok]).dt.to_period("M")
        df = pd.DataFrame({"m": d.values, "y": y, "p": yhat})
        ics = df.groupby("m").apply(lambda g: spearmanr(g.y, g.p).statistic if len(g) >= 10 and g.p.nunique() > 1 else np.nan,
                                    include_groups=False).dropna()
        out["rank_ic_mean"] = float(ics.mean()) if len(ics) else None
        out["rank_ic_months"] = int(len(ics))
        if len(ics) >= 6 and ics.std() > 0:
            out["rank_ic_icir"] = float(ics.mean() / ics.std())
            out["rank_ic_t"] = float(ics.mean() / ics.std() * np.sqrt(len(ics)))
        else:
            out["rank_ic_icir"] = out["rank_ic_t"] = None
    return out


def power_label(n: int, ci_width: float | None = None, wide: float = 0.10) -> str:
    if n < 30:
        return "warning (n<30)"
    if n < 100:
        return "exploratory (30<=n<100)"
    if ci_width is not None and ci_width > wide:
        return "inconclusive (wide CI)"
    return "adequate"


In [ ]:
%%writefile $REPO/src/secprice/events/__init__.py
"""secprice.events"""


In [ ]:
%%writefile $REPO/src/secprice/events/build.py
"""S07 events and clocks (SPEC.md §5.1-5.3, §5.7, §16 events.parquet).

Periodic events: 10-K (incl. 405/KT) and 10-Q originals. 8-K events: same-firm, same-ET-day 8-Ks merged into one
multi-label event. Amendments are kept in the manifest for the audit only. Event construction never reads price
fields (hard rule 9 / leakage item 21).
"""
from __future__ import annotations

import hashlib
import json

import numpy as np
import pandas as pd

from .timestamps import derive_times, to_utc_ns

PERIODIC = {"10-K", "10-K405", "10-KT", "10-Q"}
EIGHTK = {"8-K"}


def event_id(cik: int, earliest_accession: str) -> str:
    return hashlib.sha1(f"{int(cik)}|{earliest_accession}".encode()).hexdigest()[:16]


def build_events(manifest: pd.DataFrame, cal, cfg: dict, tax, release_info: pd.DataFrame,
                 sections_avail: pd.DataFrame) -> pd.DataFrame:
    """manifest columns: accession, cik, form, filing_date, acceptance_time (ET), ts_quality, ts_source,
    report_date, items_list (list[str]), item_titles (list), in_study (bool), is_amendment.
    release_info: accession, is_results_release (bool), release_period_end.
    sections_avail: accession, section ids present (list)."""
    t = cfg["timing"]
    m = manifest[manifest["in_study"] & ~manifest["is_amendment"] & manifest["form"].isin(PERIODIC | EIGHTK)].copy()
    m["acceptance_time"] = pd.to_datetime(m["acceptance_time"])
    m = m[m["acceptance_time"].notna()]
    m["acc_date_et"] = m["acceptance_time"].dt.tz_convert("America/New_York").dt.tz_localize(None).dt.normalize()
    rel = release_info.set_index("accession") if len(release_info) else pd.DataFrame()
    sav = sections_avail.set_index("accession")["sections"].to_dict() if len(sections_avail) else {}

    rows = []
    per = m[m.form.isin(PERIODIC)]
    for r in per.itertuples(index=False):
        rows.append({"cik": r.cik, "accessions": [r.accession], "forms": [r.form], "acceptance_time": r.acceptance_time,
                     "first_acceptance_time": r.acceptance_time, "ts_quality": r.ts_quality, "ts_source": r.ts_source,
                     "event_type": "periodic_10K" if r.form.startswith("10-K") else "periodic_10Q",
                     "categories": [], "items": [], "report_date": r.report_date, "filing_date": r.filing_date})
    ek = m[m.form.isin(EIGHTK)].sort_values(["cik", "acc_date_et", "acceptance_time", "accession"])
    for (cik, d), g in ek.groupby(["cik", "acc_date_et"], sort=False):
        items, cats = [], set()
        for r in g.itertuples(index=False):
            its = list(r.items_list or [])
            items += its
            cats |= set(tax.categories(its, r.filing_date, list(r.item_titles or [])))
        first, last = g.iloc[0], g.iloc[-1]
        q = "A" if (g.ts_quality == "A").all() else ("B" if (g.ts_quality != "C").all() else "C")
        # a same-day group is one event whose features use every filing in it, so its clock starts at the LAST acceptance:
        # only then is all of its content public (decision log D-071; the first acceptance is kept for diagnostics)
        rows.append({"cik": cik, "accessions": g.accession.tolist(), "forms": g.form.tolist(),
                     "acceptance_time": last.acceptance_time, "first_acceptance_time": first.acceptance_time,
                     "ts_quality": q, "ts_source": last.ts_source,
                     "event_type": "8K", "categories": sorted(cats), "items": sorted(set(items)),
                     "report_date": first.report_date, "filing_date": first.filing_date})
    ev = pd.DataFrame(rows)
    if ev.empty:
        return ev
    ev["event_id"] = [event_id(c, sorted(a)[0]) for c, a in zip(ev.cik, ev.accessions)]
    tt = derive_times(ev["acceptance_time"], t["public_lag_minutes"], t["decision_latency_minutes"])
    ev = pd.concat([ev.reset_index(drop=True), tt.reset_index(drop=True)], axis=1)
    acc_ns = to_utc_ns(ev["acceptance_time"])
    pub_ns = to_utc_ns(ev["public_available_time"])
    dec_ns = to_utc_ns(ev["decision_time"])
    valid = np.ones(len(ev), bool)
    ev["session_bucket"] = cal.v_bucket(acc_ns, valid)
    base = cal.v_last_close_at_or_before(acc_ns)
    d0 = cal.v_first_close_after(pub_ns)
    entry = cal.v_first_open_at_or_after(dec_ns)
    n = len(cal)
    ev["tinfo_base_idx"] = base
    ev["trackA_day0_idx"] = d0
    ev["ttrade_entry_idx"] = entry
    ev["tinfo_base_date"] = [cal.date(i) if 0 <= i < n else pd.NaT for i in base]
    ev["trackA_day0"] = [cal.date(i) if 0 <= i < n else pd.NaT for i in d0]
    ev["ttrade_entry_date"] = [cal.date(i) if 0 <= i < n else pd.NaT for i in entry]
    ev["tinfo_eligible"] = ev["session_bucket"].isin(["PRE", "POST", "NONTRADING"]) & (ev.ts_quality != "C")
    base_close_ns = np.where((base >= 0) & (base < n), cal.close_utc[np.clip(base, 0, n - 1)], np.nan)
    ev["contam_minutes"] = (acc_ns - base_close_ns) / 6e10   # pre-publication window base close -> acceptance
    # addendum P0.3: public availability is an assumption unless independently observed
    ev["timing_policy"] = np.where(ev["public_time_source"] == "observed", "observed_public_time",
                                   "assumed_availability_latency_policy")
    ev = _earnings_types(ev, tax, rel)
    ev = _clusters(ev, rel)
    ev = _overlaps(ev, cfg)
    ev["section_ids"] = [sorted({s for a in accs for s in (sav.get(a) or [])}) for accs in ev.accessions]
    ev["year"] = pd.to_datetime(ev["decision_time"]).dt.year
    return ev.sort_values(["decision_time", "cik"]).reset_index(drop=True)


def _earnings_types(ev: pd.DataFrame, tax, rel: pd.DataFrame) -> pd.DataFrame:
    et = []
    rel_flag = rel["is_results_release"].to_dict() if len(rel) else {}
    for r in ev.itertuples(index=False):
        if r.event_type != "8K":
            et.append(None)
            continue
        modern, legacy_eq = tax.earnings_candidate(r.items, r.filing_date)
        confirmed = any(bool(rel_flag.get(a, False)) for a in r.accessions)
        if (modern or legacy_eq) and confirmed and r.ts_quality == "A":
            et.append("earnings_announcement_verified")
        elif modern or (legacy_eq and confirmed):
            et.append("earnings_date_ambiguous")
        else:
            et.append(None)
    ev["earnings_type"] = et
    if "earnings" not in ev.columns:
        pass
    return ev


def _clusters(ev: pd.DataFrame, rel: pd.DataFrame) -> pd.DataFrame:
    """Link each verified release with the firm's next periodic filing for the same fiscal period."""
    ev["cluster_id"] = None
    ev["cluster_role"] = None
    ev["release_period_end"] = pd.NaT
    per_end = rel["release_period_end"].to_dict() if len(rel) else {}
    for i, r in ev.iterrows():
        if r.earnings_type == "earnings_announcement_verified":
            pe = next((per_end.get(a) for a in r.accessions if per_end.get(a)), None)
            ev.at[i, "release_period_end"] = pd.Timestamp(pe) if pe else pd.NaT
    ev["report_date_ts"] = pd.to_datetime(ev["report_date"], errors="coerce")
    for cik, g in ev.groupby("cik"):
        rels = g[g.earnings_type == "earnings_announcement_verified"].sort_values("decision_time")
        pers = g[g.event_type.str.startswith("periodic")].sort_values("decision_time")
        used = set()
        for i, r in rels.iterrows():
            ev.at[i, "cluster_id"] = "C_" + r.event_id
            ev.at[i, "cluster_role"] = "release"
            if pd.isna(r.release_period_end):
                continue
            cand = pers[(pers.decision_time > r.decision_time)
                        & (pers.decision_time <= r.decision_time + pd.Timedelta(days=120))
                        & ((pers.report_date_ts - r.release_period_end).abs() <= pd.Timedelta(days=20))]
            cand = cand[~cand.index.isin(used)]
            if len(cand):
                j = cand.index[0]
                used.add(j)
                ev.at[j, "cluster_id"] = "C_" + r.event_id
                ev.at[j, "cluster_role"] = "periodic_after_release"
    per_mask = ev.event_type.str.startswith("periodic")
    ev.loc[per_mask & ev.cluster_id.isna(), "earnings_type"] = "periodic_filing_only"
    return ev.drop(columns=["report_date_ts"])


def _overlaps(ev: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    """overlap_flag_h: another same-firm event's day 0 falls within [day0, day0 + h] (dates only). Members of the same
    earnings cluster (a verified release and its own 10-Q/10-K) are one information event (§5.2) and do not flag each
    other (D-073); any other same-firm event does."""
    hs = sorted(set(cfg["targets"]["horizons"]) | {int(cfg["contam_window_sessions"])})
    for h in hs:
        ev[f"overlap_flag_h{h}"] = False
    has_cl = "cluster_id" in ev
    for cik, g in ev.groupby("cik"):
        d = g["trackA_day0_idx"].to_numpy()
        cl = g["cluster_id"].to_numpy() if has_cl else np.array([None] * len(g), dtype=object)
        order = np.argsort(d, kind="stable")
        ds, cs = d[order], cl[order]
        n = len(ds)
        for h in hs:
            lo = np.searchsorted(ds, ds, side="left")          # first event on the same day0
            hi = np.searchsorted(ds, ds + h, side="right")     # first event with day0 > day0 + h
            flag = np.zeros(n, bool)
            for i in range(n):
                for j in range(lo[i], hi[i]):
                    if j != i and (cs[i] is None or pd.isna(cs[i]) or cs[j] != cs[i]):
                        flag[i] = True
                        break
            out = np.zeros(len(d), bool)
            out[order] = flag
            ev.loc[g.index, f"overlap_flag_h{h}"] = out
    return ev


def events_to_parquet_safe(ev: pd.DataFrame) -> pd.DataFrame:
    out = ev.copy()
    for c in ("accessions", "forms", "categories", "items", "section_ids"):
        if c in out:
            out[c] = out[c].map(lambda v: json.dumps(list(v)) if isinstance(v, (list, tuple, np.ndarray)) else v)
    return out


def events_from_parquet(ev: pd.DataFrame) -> pd.DataFrame:
    out = ev.copy()
    for c in ("accessions", "forms", "categories", "items", "section_ids"):
        if c in out:
            out[c] = out[c].map(lambda v: json.loads(v) if isinstance(v, str) else (list(v) if v is not None else []))
    for c in ("acceptance_time", "first_acceptance_time", "public_available_time", "feature_ready_time", "decision_time"):
        if c in out:
            out[c] = pd.to_datetime(out[c], utc=True).dt.tz_convert("America/New_York")
    return out


In [ ]:
%%writefile $REPO/src/secprice/events/timestamps.py
"""The four timestamps per event (SPEC.md §5.1; addendum P0.3).

Verified 2026-10-23 against SGML headers (docs/decision_log.md D-003, D-047): the submissions-JSON
`acceptanceDateTime` (suffix 'Z') follows TWO conventions. About 70% of 8,789 pilot filings are the Eastern wall time
shifted by the ET->UTC offset twice (+8 h EDT / +10 h EST; two UTC->ET conversions reproduce the header), the other
~30% are true UTC; no field (agent, hour, form, year) predicts which. The SGML header ACCEPTANCE-DATETIME is therefore
read for every filing (8-K full submissions; a 20 KB prefix of the full submission for 10-K/10-Q). JSON-only times are
quality C. Pre-2002 filings carry midnight placeholders (no real time): quality B (date only).
"""
from __future__ import annotations

import re

import numpy as np
import pandas as pd

ET = "America/New_York"
_HDR_RE = re.compile(r"<ACCEPTANCE-DATETIME>\s*(\d{14})|ACCEPTANCE-DATETIME:\s*(\d{14})")


def parse_header_acceptance(header_text: str) -> str | None:
    """Raw 14-digit ACCEPTANCE-DATETIME (Eastern) from an SGML header, or None."""
    if not header_text:
        return None
    m = _HDR_RE.search(header_text[:20000])
    if not m:
        return None
    return m.group(1) or m.group(2)


def header_to_et(raw14: str) -> pd.Timestamp:
    t = pd.Timestamp(f"{raw14[:4]}-{raw14[4:6]}-{raw14[6:8]} {raw14[8:10]}:{raw14[10:12]}:{raw14[12:14]}")
    return t.tz_localize(ET, ambiguous=True, nonexistent="shift_forward")


def json_to_et(raw: str) -> pd.Timestamp:
    """Corrected ET time from the submissions-JSON acceptanceDateTime (double UTC->ET conversion)."""
    t = pd.Timestamp(str(raw)[:19])
    once = t.tz_localize("UTC").tz_convert(ET).tz_localize(None)
    return once.tz_localize("UTC").tz_convert(ET)


def acceptance_from_sources(json_raw: str | None, header_raw: str | None, filing_date: str | None,
                            assumed_time: str = "17:30", json_rule_ok: bool = True) -> tuple:
    """Return (acceptance_time_et, ts_quality, ts_source, date_only).

    A = exact acceptance time (SGML header, or JSON value under the verified correction rule)
    B = date only (assume after the close at `assumed_time` ET)
    C = derived or uncertain (JSON rule failed its sample verification, or only a filing date with conflicts)
    """
    if header_raw:
        t = header_to_et(header_raw)
        return t, "A", "sgml_header", False
    if json_raw and str(json_raw).strip():
        t = json_to_et(json_raw)
        if t.hour == 0 and t.minute == 0 and t.second == 0:
            d = t.tz_localize(None).normalize()
            return _assumed(d, assumed_time), "B", "json_midnight_placeholder", True
        return t, ("A" if json_rule_ok else "C"), "submissions_json_corrected", False
    if filing_date:
        d = pd.Timestamp(filing_date).normalize()
        return _assumed(d, assumed_time), "B", "filing_date_only", True
    return pd.NaT, "C", "missing", True


def _assumed(d: pd.Timestamp, hhmm: str) -> pd.Timestamp:
    hh, mm = hhmm.split(":")
    return (d + pd.Timedelta(hours=int(hh), minutes=int(mm))).tz_localize(ET)


def derive_times(acceptance_et: pd.Series, public_lag_min: float, decision_latency_min: float,
                 observed_public: pd.Series | None = None) -> pd.DataFrame:
    """public_available = observed if present, else acceptance + lag (flagged 'assumed'); feature_ready = public +
    latency; decision = feature_ready (SPEC.md §5.1)."""
    acc = pd.to_datetime(acceptance_et)
    pub = acc + pd.Timedelta(minutes=float(public_lag_min))
    src = pd.Series("assumed_lag", index=acc.index, dtype=object)
    if observed_public is not None:
        obs = pd.to_datetime(observed_public)
        has = obs.notna()
        pub = pub.where(~has, obs)
        src = src.where(~has, "observed")
    feat = pub + pd.Timedelta(minutes=float(decision_latency_min))
    return pd.DataFrame({"public_available_time": pub, "public_time_source": src,
                         "feature_ready_time": feat, "decision_time": feat})


def to_utc_ns(s: pd.Series) -> np.ndarray:
    return pd.DatetimeIndex(pd.to_datetime(s)).tz_convert("UTC").asi8.astype(np.int64)


def timing_policy_label(cfg: dict) -> str:
    t = cfg["timing"]
    return (f"point-in-time evaluation under an assumed availability/processing-latency policy "
            f"(public = acceptance + {t['public_lag_minutes']} min [assumed unless observed], "
            f"decision = public + {t['decision_latency_minutes']} min)")


In [ ]:
%%writefile $REPO/src/secprice/extensions/__init__.py
"""secprice.extensions"""


In [ ]:
%%writefile $REPO/src/secprice/extensions/stacking.py
"""ext_stacking (SPEC.md §7.3; H5): multi-tower stacking and a Caruana greedy ensemble, trained only on
walk-forward out-of-fold predictions in time order (each meta-model sees only folds strictly earlier than the fold
it predicts; SPEC.md §6.3 #20). Gate G3c: core complete + budget + the pre-stated question H5."""
from __future__ import annotations

import json

import numpy as np
import pandas as pd

from ..evaluation.bootstrap import paired_macro_f1_diff
from ..evaluation.metrics import classification_metrics, confusion, decide, f1_from_cm

PCOLS = ["p_down", "p_side", "p_up"]


def _base_runs(ctx, prim) -> dict:
    frames = []
    for s in ("S11", "S15"):
        p = ctx.dep(s) / "predictions.parquet"
        if p.exists():
            frames.append(pd.read_parquet(p))
    pr = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    if pr.empty:
        return {}
    pr = pr[(pr.target_def == prim["target_def"]) & (pr.horizon == prim["horizon"]) & (pr.tuned == "tuned")
            & ~pr.run_id.astype(str).str.startswith("dev|")]
    out = {}
    cohort = "common_emb" if (pr.cohort == "common_emb").any() else "common"
    sub = pr[pr.cohort == cohort]
    for rung in ("L6", "L7", "L8", "L8_ft"):
        g = sub[sub.rung == rung]
        if g.empty:
            continue
        best = None
        for rid, gg in g.groupby("run_id"):
            f = f1_from_cm(confusion(gg.y.values, gg.yhat.values))["macro_f1"]
            enc = gg.encoder.iloc[0]
            key = f"{rung}|{enc}" if rung in ("L7", "L8_ft") else rung
            if key not in out or f > out[key][1]:
                out[key] = (gg, f)
    return {k: v[0] for k, v in out.items()}


def _caruana(P: list[np.ndarray], y: np.ndarray, prior: np.ndarray, iters: int = 20) -> np.ndarray:
    w = np.zeros(len(P))
    cur = np.zeros_like(P[0])
    for _ in range(iters):
        best, bi = -1, 0
        for i, p in enumerate(P):
            cand = (cur * w.sum() + p) / (w.sum() + 1)
            f = f1_from_cm(confusion(y, decide(cand, prior)))["macro_f1"] or 0
            if f > best:
                best, bi = f, i
        cur = (cur * w.sum() + P[bi]) / (w.sum() + 1)
        w[bi] += 1
    return w / w.sum()


def run_stacking(ctx, lad, lab, common, res, dev_preds: pd.DataFrame) -> dict | None:
    from sklearn.linear_model import LogisticRegression
    cfg = ctx.cfg
    prim = cfg["targets"]["primary"]
    bases = _base_runs(ctx, prim)
    if len(bases) < 2:
        return {"ran": False, "reason": "fewer than two base models with OOF predictions"}
    keys = sorted(bases)
    ids = None
    for k in keys:
        s = set(bases[k].event_id)
        ids = s if ids is None else ids & s
    ids = sorted(ids)
    al = {k: bases[k].set_index("event_id").loc[ids] for k in keys}
    y = al[keys[0]].y.astype(int).values
    fold = al[keys[0]].fold.values
    dates = al[keys[0]].date.values
    Xlog = np.hstack([np.log(np.clip(al[k][PCOLS].values, 1e-9, 1)) for k in keys])
    folds = sorted(np.unique(fold))
    stack_p = np.full((len(ids), 3), np.nan)
    car_p = np.full((len(ids), 3), np.nan)
    for f in folds[1:]:
        tr, te = fold < f, fold == f
        if tr.sum() < 100 or len(np.unique(y[tr])) < 3:
            continue
        m = LogisticRegression(C=1.0, max_iter=2000).fit(Xlog[tr], y[tr])
        stack_p[te] = m.predict_proba(Xlog[te])
        prior = np.bincount(y[tr], minlength=3) / tr.sum()
        w = _caruana([al[k][PCOLS].values[tr] for k in keys], y[tr], prior)
        car_p[te] = sum(wi * al[k][PCOLS].values[te] for wi, k in zip(w, keys))
    ok = np.isfinite(stack_p).all(1)
    if ok.sum() < 50:
        return {"ran": False, "reason": "too few out-of-fold rows after the first fold"}
    prior_all = np.bincount(y[ok], minlength=3) / ok.sum()
    yh_stack = decide(stack_p[ok], prior_all)
    yh_car = decide(car_p[ok], prior_all)
    fold_scores = []
    for f in folds[1:]:
        m_ = ok & (fold == f)
        if m_.sum():
            fold_scores.append(f1_from_cm(confusion(y[m_], decide(stack_p[m_], prior_all)))["macro_f1"])
    single = {k: f1_from_cm(confusion(y[ok], al[k].yhat.values[ok].astype(int)))["macro_f1"] for k in keys}
    best_k = max(single, key=lambda k: single[k] or 0)
    nb, seed = int(cfg["bootstrap"]["n"]), int(cfg["seed"])
    h5 = paired_macro_f1_diff(y[ok], yh_stack, al[best_k].yhat.values[ok].astype(int), dates[ok], nb, seed)
    h5c = paired_macro_f1_diff(y[ok], yh_car, al[best_k].yhat.values[ok].astype(int), dates[ok], nb, seed)
    pm = classification_metrics(y[ok], stack_p[ok], prior_all, yh_stack)
    res.add("ext_stacking|STACK|walk_forward_folds>=1", "STACK", "stack_logistic", "+".join(keys), "retrospective", "common_emb",
            prim["target_def"], prim["horizon"], "walk_forward_pooled_folds_ge1", "all", pm, None, int(ok.sum()), None, None, None, "tuned")
    out = {"ran": True, "bases": keys, "wf_score": float(np.mean(fold_scores)) if fold_scores else None,
           "h5_stack_vs_best_single": {"best_single": best_k, **h5}, "h5_caruana_vs_best_single": h5c,
           "spec": {"meta": "multinomial logistic on base log-probabilities", "bases": keys},
           "g3c": "core complete (S15 done); pre-stated question H5; budget checked in preflight"}
    # development holdout: meta trained on all walk-forward OOF rows, applied to the finalists' dev predictions
    dv = {}
    for k in keys:
        nm = k if k.startswith("L8_ft") else k.split("|")[0]
        if nm and len(dev_preds) and (dev_preds.name == nm).any():
            dv[k] = dev_preds[dev_preds.name == nm].set_index("event_id")
    if len(dv) == len(keys):
        j = sorted(set.intersection(*[set(d.index) for d in dv.values()]))
        Xd = np.hstack([np.log(np.clip(dv[k].loc[j, PCOLS].values, 1e-9, 1)) for k in keys])
        m = LogisticRegression(C=1.0, max_iter=2000).fit(Xlog[ok | (fold == folds[0])], y[ok | (fold == folds[0])])
        P = m.predict_proba(Xd)
        yd = dv[keys[0]].loc[j, "y"].values.astype(int)
        yh = decide(P, np.bincount(y, minlength=3) / len(y))
        out["dev_preds"] = pd.DataFrame({"event_id": j, "y": yd, "date": dv[keys[0]].loc[j, "date"].values, "p_down": P[:, 0],
                                         "p_side": P[:, 1], "p_up": P[:, 2], "yhat": yh, "fold": 0, "run_id": "dev|STACK",
                                         "name": "STACK"})
    else:
        out["dev_preds"] = None
        out["dev_note"] = "dev predictions not available for every base (fine-tuned bases are walk-forward only)"
    return out


In [ ]:
%%writefile $REPO/src/secprice/features/__init__.py
"""secprice.features"""


In [ ]:
%%writefile $REPO/src/secprice/features/accounting.py
"""S06 accounting: XBRL facts with public-availability times, as-first-reported quarterly EPS, release EPS and
cluster-aware SUE (SPEC.md §5.6; addendum P1.4).

SUE = (EPS_q - EPS_{q-4}) / sd(seasonal differences over the prior quarters), using up to `sue_max_diffs`
(min `sue_min_diffs`) differences, each value as originally reported (earliest public accession for that
period). Every history value must satisfy public_available_time < decision_time of the event. Earnings-8-K
events take EPS_q from their own release exhibit; 10-Q/10-K events from their own XBRL. A later accession of the
same cluster can never feed an earnings-8-K event (assertion + planted-leak test).
"""
from __future__ import annotations

import numpy as np
import pandas as pd

EPS_KIND = {"EarningsPerShareDiluted": "diluted", "EarningsPerShareBasicAndDiluted": "diluted",
            "EarningsPerShareBasic": "basic"}


def _qkey(period_end: pd.Series) -> pd.Series:
    """Calendar-quarter key of a fiscal period end, shifted 15 days back so 52/53-week ends (Jan 1-3, Oct 1-2)
    land in the quarter they close."""
    d = pd.to_datetime(period_end) - pd.Timedelta(days=15)
    return (d.dt.year * 4 + (d.dt.quarter - 1)).astype("Int64")


def attach_public_time(facts: pd.DataFrame, filing_times: pd.DataFrame, assumed_lag_min: float,
                       assumed_time: str = "17:30") -> pd.DataFrame:
    """filing_times: accession, public_available_time (tz ET). Facts from accessions outside the manifest get
    filed-date 17:30 ET + lag (quality B)."""
    f = facts.merge(filing_times[["accession", "public_available_time"]], on="accession", how="left")
    miss = f["public_available_time"].isna()
    if miss.any():
        hh, mm = assumed_time.split(":")
        fd = pd.to_datetime(f.loc[miss, "filed"]) + pd.Timedelta(hours=int(hh), minutes=int(mm) + float(assumed_lag_min))
        f.loc[miss, "public_available_time"] = fd.dt.tz_localize("America/New_York")
        f["pub_time_source"] = np.where(miss, "filed_date_assumed", "manifest")
    else:
        f["pub_time_source"] = "manifest"
    f["public_available_time"] = pd.to_datetime(f["public_available_time"], utc=True).dt.tz_convert("America/New_York")
    return f


def quarterly_eps(facts: pd.DataFrame) -> pd.DataFrame:
    """Direct quarterly EPS facts plus Q4 = FY - 9M YTD (both first-reported; availability = the later source)."""
    e = facts[facts["tag"].isin(EPS_KIND) & (facts["unit"] == "USD/shares")].copy()
    if e.empty:
        return pd.DataFrame(columns=["cik", "period_end", "qkey", "kind", "value", "source_accession",
                                     "public_available_time", "derivation", "form"])
    e["kind"] = e["tag"].map(EPS_KIND)
    e["start"] = pd.to_datetime(e["period_start"])
    e["end"] = pd.to_datetime(e["period_end"])
    e["days"] = (e["end"] - e["start"]).dt.days
    e = e.sort_values("public_available_time")
    first = lambda df, keys: df.drop_duplicates(keys, keep="first")  # noqa: E731
    q = first(e[(e.days >= 70) & (e.days <= 110)], ["cik", "kind", "end"])
    out = [pd.DataFrame({"cik": q.cik, "period_end": q.end, "kind": q.kind, "value": q.value,
                         "source_accession": q.accession, "public_available_time": q.public_available_time,
                         "derivation": "direct", "form": q.form})]
    fy = first(e[(e.days >= 340) & (e.days <= 380)], ["cik", "kind", "end"])
    m9 = first(e[(e.days >= 250) & (e.days <= 290)], ["cik", "kind", "start", "end"])
    if len(fy) and len(m9):
        j = fy.merge(m9, on=["cik", "kind", "start"], suffixes=("_fy", "_9m"))
        j = j[(j.end_fy - j.end_9m).dt.days.between(70, 110)]
        pub = np.maximum(j.public_available_time_fy.values, j.public_available_time_9m.values)
        out.append(pd.DataFrame({"cik": j.cik, "period_end": j.end_fy, "kind": j.kind, "value": j.value_fy - j.value_9m,
                                 "source_accession": j.accession_fy, "public_available_time": pub,
                                 "derivation": "fy_minus_9m", "form": j.form_fy}))
    r = pd.concat(out, ignore_index=True)
    r["public_available_time"] = pd.to_datetime(r["public_available_time"], utc=True).dt.tz_convert("America/New_York")
    r = r.sort_values(["cik", "kind", "period_end", "public_available_time"])
    r = r.drop_duplicates(["cik", "kind", "period_end"], keep="first")      # as first reported
    r["qkey"] = _qkey(r["period_end"])
    return r.reset_index(drop=True)


def sue_for(eps_q: float, qk: int, hist: pd.DataFrame, min_diffs: int, max_diffs: int) -> dict:
    """hist: qkey, value (already filtered to public < decision, one value per qkey)."""
    s = dict(zip(hist["qkey"].astype(int), hist["value"].astype(float)))
    out = {"sue": np.nan, "eps_q": eps_q, "eps_q4": s.get(qk - 4, np.nan), "n_prior_diffs": 0, "sue_reason": None}
    if eps_q is None or not np.isfinite(eps_q):
        out["sue_reason"] = "no_eps_q"
        return out
    if not np.isfinite(out["eps_q4"]):
        out["sue_reason"] = "no_eps_q_minus_4"
        return out
    diffs = []
    for j in range(1, max_diffs + 1):
        a, b = s.get(qk - j), s.get(qk - j - 4)
        if a is not None and b is not None and np.isfinite(a) and np.isfinite(b):
            diffs.append(a - b)
    out["n_prior_diffs"] = len(diffs)
    if len(diffs) < min_diffs:
        out["sue_reason"] = f"only_{len(diffs)}_prior_diffs"
        return out
    sd = float(np.std(diffs, ddof=1))
    if not np.isfinite(sd) or sd <= 0:
        out["sue_reason"] = "zero_dispersion"
        return out
    out["sue"] = (eps_q - out["eps_q4"]) / sd
    out["sue_reason"] = "ok"
    return out


def history_before(eps_hist: pd.DataFrame, cik: int, decision_time, exclude_accessions: set | None = None) -> pd.DataFrame:
    h = eps_hist[(eps_hist.cik == cik) & (eps_hist.public_available_time < decision_time)]
    if exclude_accessions:
        h = h[~h.source_accession.isin(exclude_accessions)]
    h = h.sort_values("public_available_time").drop_duplicates("qkey", keep="first")
    return h


def compute_sue_table(events: pd.DataFrame, xbrl_eps: pd.DataFrame, rel_eps: pd.DataFrame, min_diffs: int,
                      max_diffs: int, cluster_later: dict | None = None) -> pd.DataFrame:
    """events: accession, cik, kind ('release' | 'periodic'), decision_time, period_end (release), qkey.
    xbrl_eps / rel_eps: cik, qkey, value, source_accession, public_available_time (diluted preferred).
    cluster_later: {accession: set(later same-cluster accessions)} -> excluded from history (rule 7)."""
    hist_all = pd.concat([xbrl_eps.assign(src="xbrl"), rel_eps.assign(src="release")], ignore_index=True)
    hist_all = hist_all.sort_values("public_available_time")
    rows, sources = [], []
    for ev in events.itertuples(index=False):
        excl = (cluster_later or {}).get(ev.accession, set())
        h = history_before(hist_all, ev.cik, ev.decision_time, excl)
        if ev.kind == "release":
            eps_q, qk, src = ev.eps_q, ev.qkey, "release_exhibit"
        else:
            own = xbrl_eps[(xbrl_eps.cik == ev.cik) & (xbrl_eps.source_accession == ev.accession)]
            if len(own):
                own = own.sort_values("period_end").iloc[-1]
                eps_q, qk, src = float(own.value), int(own.qkey), "own_xbrl"
            else:
                eps_q, qk, src = np.nan, ev.qkey, "none"
        h = h[h.qkey != qk] if qk is not None and not pd.isna(qk) else h
        qki = int(qk) if qk is not None and not pd.isna(qk) else -10**6
        r = sue_for(eps_q, qki, h, min_diffs, max_diffs)
        used = {qki - 4} | {q for j in range(1, max_diffs + 1) for q in (qki - j, qki - j - 4)}
        hu = h[h.qkey.isin(list(used))]
        for s in hu.itertuples(index=False):
            sources.append((ev.accession, s.source_accession, s.public_available_time, s.src))
        r.update({"accession": ev.accession, "cik": ev.cik, "eps_q_source": src, "qkey": qk,
                  "hist_max_public_time": hu.public_available_time.max() if len(hu) else pd.NaT,
                  "hist_sources": ",".join(sorted(set(hu.src))) if len(hu) else ""})
        rows.append(r)
    src_df = pd.DataFrame(sources, columns=["accession", "source_accession", "source_public_time", "source_kind"])
    return pd.DataFrame(rows), src_df


def pit_facts(events: pd.DataFrame, facts: pd.DataFrame, tags: list[str]) -> pd.DataFrame:
    """Latest value per tag with public time strictly before decision_time (merge_asof by cik)."""
    ev = events[["event_id", "cik", "decision_time"]].copy()
    ev["decision_time"] = pd.to_datetime(ev["decision_time"], utc=True)
    ev = ev.sort_values("decision_time")
    out = ev[["event_id"]].copy()
    pts = pd.DataFrame(index=ev.index)
    f = facts.copy()
    f["public_available_time"] = pd.to_datetime(f["public_available_time"], utc=True)
    f["days"] = (pd.to_datetime(f["period_end"]) - pd.to_datetime(f["period_start"])).dt.days
    for tag, kind in tags:
        sub = f[f.tag == tag]
        if kind == "fy":
            sub = sub[sub.days.between(340, 380)]
        elif kind == "q":
            sub = sub[sub.days.between(70, 110)]
        sub = sub.sort_values("public_available_time")[["cik", "public_available_time", "value", "period_end"]]
        if sub.empty:
            out[f"{tag}_{kind}"] = np.nan
            continue
        sub = sub.rename(columns={"public_available_time": "pt"})
        m = pd.merge_asof(ev, sub, left_on="decision_time", right_on="pt", by="cik", allow_exact_matches=False,
                          direction="backward")
        out[f"{tag}_{kind}"] = m["value"].values
        pts[f"{tag}_{kind}"] = pd.to_datetime(m["pt"].values, utc=True)
    out["_avail_accounting"] = pts.max(axis=1).values if pts.shape[1] else pd.NaT
    return out


In [ ]:
%%writefile $REPO/src/secprice/features/price.py
"""Price features at decision_time: only sessions whose close is at or before decision_time are used
(availability = that close). SPEC.md §9.1 L1."""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..targets.build import Estimator


def _ratio(tri, a, b):
    out = np.full(len(a), np.nan)
    ok = (b >= 0) & (a >= 0) & (a < len(tri)) & (b < len(tri))
    with np.errstate(divide="ignore", invalid="ignore"):
        out[ok] = tri[a[ok]] / tri[b[ok]] - 1
    return out


def _roll(fn, arr, ends, win, min_n):
    out = np.full(len(ends), np.nan)
    for i, c in enumerate(ends):
        if c - win + 1 < 0 or c >= len(arr):
            continue
        seg = arr[c - win + 1:c + 1]
        seg = seg[np.isfinite(seg)]
        if len(seg) >= min_n:
            out[i] = fn(seg)
    return out


def price_features(ev: pd.DataFrame, panel, spy: dict) -> pd.DataFrame:
    cal = panel.cal
    dec_ns = pd.DatetimeIndex(pd.to_datetime(ev["decision_time"])).tz_convert("UTC").asi8
    cidx = cal.v_last_close_at_or_before(dec_ns)
    ev = ev.assign(_cidx=cidx)
    rows = []
    s_tri, s_r = spy["total_return_index"], spy["ret"]
    for tk, g in ev.groupby("ticker", sort=False):
        c = g["_cidx"].to_numpy(int)
        f = pd.DataFrame({"event_id": g["event_id"].to_numpy()})
        if not panel.has(tk):
            rows.append(f)
            continue
        P = panel[tk]
        tri, r, close, vol = P["total_return_index"], P["ret"], P["close_split_adj"], P["volume"]
        for w in (1, 5, 20, 60, 250):
            f[f"px_ret_{w}"] = _ratio(tri, c, c - w)
        f["px_rev_1m"] = f["px_ret_20"]
        f["px_mom_12_2"] = _ratio(tri, c - 21, c - 252)
        f["px_vol_20"] = _roll(np.std, r, c, 20, 15)
        f["px_vol_60"] = _roll(np.std, r, c, 60, 40)
        dv = close * vol
        f["px_logdv_20"] = np.log(np.maximum(_roll(np.mean, dv, c, 20, 10), 1.0))
        f["px_volratio_5_60"] = np.log(np.maximum(_roll(np.mean, vol, c, 5, 3), 1) / np.maximum(_roll(np.mean, vol, c, 60, 30), 1))
        with np.errstate(divide="ignore", invalid="ignore"):
            am = np.abs(r) / dv * 1e6
        f["px_amihud_20"] = np.log1p(_roll(np.mean, am, c, 20, 10))
        hi = _roll(np.max, close, c, 252, 100)
        f["px_dist_52w_high"] = close[np.clip(c, 0, len(close) - 1)] / hi - 1
        est = Estimator(r, s_r)
        cnt, a, b, sig = est.window(c - 250, c - 30)
        f["px_beta"] = np.where(cnt >= 120, b, np.nan)
        f["px_idio_vol"] = np.where(cnt >= 120, sig, np.nan)
        f["mkt_ret_20"] = _ratio(s_tri, c, c - 20)
        f["mkt_ret_60"] = _ratio(s_tri, c, c - 60)
        f["mkt_vol_20"] = _roll(np.std, s_r, c, 20, 15)
        f["mkt_dd_252"] = s_tri[np.clip(c, 0, len(s_tri) - 1)] / _roll(np.max, s_tri, c, 252, 100) - 1
        f["_avail_price"] = [cal.close_time(i) if 0 <= i < len(cal) else pd.NaT for i in c]
        rows.append(f)
    out = pd.concat(rows, ignore_index=True)
    out["_avail_price"] = pd.to_datetime(out["_avail_price"], utc=True)
    return out


In [ ]:
%%writefile $REPO/src/secprice/features/registry.py
"""Feature registry + availability rules (SPEC.md §16 feature_registry.csv, §6.3 #9/#11/#21, hard rules 3 and 7).

Each family has an availability column `_avail_<family>` holding, per event, the latest timestamp of any input.
The rule is `_avail <= decision_time` (the current filing's own content is legitimate at decision time); prior
documents and history must be strictly earlier (enforced where they are built and re-checked here).
"""
from __future__ import annotations

import numpy as np
import pandas as pd

FAMILIES = {
    "price": {"prefixes": ["px_", "mkt_"], "rule": "closes at or before decision_time", "source": "prices (S03)"},
    "metadata": {"prefixes": ["meta_"], "rule": "own filing metadata (acceptance <= decision)", "source": "manifest/events"},
    "accounting": {"prefixes": ["acc_", "sue_"], "rule": "XBRL/release values with public_time < decision_time; "
                   "SUE never from a later same-cluster accession", "source": "S06"},
    "lexical": {"prefixes": ["lm_", "main_lm_"], "rule": "own filing text", "source": "S05 sections"},
    "change": {"prefixes": ["chg_", "rf_new", "rf_removed", "boilerplate_"], "rule": "prior same-firm documents with "
               "public_time < decision_time", "source": "S05 sections"},
    "novelty": {"prefixes": ["novelty_"], "rule": "prior same-firm documents with public_time < decision_time",
                "source": "S05 sections"},
    "ear": {"prefixes": ["ear_"], "rule": "EAR of a later, different cluster whose window closed before decision_time",
            "source": "S08/S20"},
}


def family_of(col: str) -> str | None:
    for fam, spec in FAMILIES.items():
        if any(col.startswith(p) for p in spec["prefixes"]):
            return fam
    return None


def registry_frame(feature_cols: list[str], strict_map: dict | None = None) -> pd.DataFrame:
    rows = []
    for c in feature_cols:
        fam = family_of(c)
        if fam is None:
            continue
        spec = FAMILIES[fam]
        rows.append({"name": c, "family": fam, "source": spec["source"], "availability_rule": spec["rule"],
                     "max_availability_col": f"_avail_{fam}", "leaderboard": (strict_map or {}).get(c, "strict")})
    return pd.DataFrame(rows)


def availability_violations(feat: pd.DataFrame, decision_time: pd.Series) -> pd.DataFrame:
    """Rows where any family's availability time is after decision_time."""
    dec = pd.to_datetime(decision_time, utc=True).reset_index(drop=True)
    out = []
    for c in [c for c in feat.columns if c.startswith("_avail_")]:
        av = pd.to_datetime(feat[c], utc=True).reset_index(drop=True)
        bad = av.notna() & (av > dec)
        if bad.any():
            out.append(pd.DataFrame({"event_id": feat["event_id"].reset_index(drop=True)[bad], "family": c[7:],
                                     "avail": av[bad], "decision": dec[bad]}))
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame(columns=["event_id", "family", "avail", "decision"])


def ear_rule_violations(feat: pd.DataFrame, events: pd.DataFrame) -> pd.DataFrame:
    """EAR features must come from a different cluster whose EAR window closed before decision_time."""
    if "ear_src_cluster" not in feat.columns:
        return pd.DataFrame(columns=["event_id", "reason"])
    m = feat[["event_id", "ear_src_cluster", "_avail_ear"]].merge(events[["event_id", "cluster_id", "decision_time"]],
                                                                   on="event_id")
    same = m["ear_src_cluster"].notna() & (m["ear_src_cluster"] == m["cluster_id"])
    late = pd.to_datetime(m["_avail_ear"], utc=True) > pd.to_datetime(m["decision_time"], utc=True)
    bad = m[same | late]
    return pd.DataFrame({"event_id": bad.event_id, "reason": np.where(same[same | late], "same_cluster", "window_not_closed")})


def sue_rule_violations(sue_src: pd.DataFrame, events: pd.DataFrame) -> pd.DataFrame:
    """SUE inputs must be public strictly before decision_time and never from a later same-cluster accession.
    sue_src: event_id, source_accession, source_public_time."""
    if sue_src is None or sue_src.empty:
        return pd.DataFrame(columns=["event_id", "reason"])
    acc2cl = {}
    for r in events[["accessions", "cluster_id", "decision_time"]].itertuples(index=False):
        for a in r.accessions:
            acc2cl[a] = (r.cluster_id, r.decision_time)
    m = sue_src.merge(events[["event_id", "cluster_id", "decision_time", "accessions", "earnings_type"]], on="event_id")
    rows = []
    for r in m.itertuples(index=False):
        if r.source_accession in (r.accessions or []):
            continue          # the event's own filing / release
        src_cl, _ = acc2cl.get(r.source_accession, (None, None))
        if pd.to_datetime(r.source_public_time, utc=True) >= pd.to_datetime(r.decision_time, utc=True):
            rows.append((r.event_id, "source_not_public_before_decision"))
        elif (r.earnings_type == "earnings_announcement_verified" and src_cl is not None
              and src_cl == r.cluster_id):
            rows.append((r.event_id, "later_same_cluster_source"))
    return pd.DataFrame(rows, columns=["event_id", "reason"])


In [ ]:
%%writefile $REPO/src/secprice/features/text.py
"""Lexical (Loughran-McDonald), filing-change and novelty features (SPEC.md §9.1 L4; H2).

No statistic is fitted here (no IDF, no vocabulary): similarity uses a fixed hashing space, so nothing is learned
from future data. Prior documents are those of the same firm whose decision time precedes the current event
(their public time is therefore strictly earlier); ties are processed as a batch so simultaneous events never see
each other.
"""
from __future__ import annotations

import re
import zlib

import numpy as np
import pandas as pd

from ..parse.html import drop_numeric_lines

LM_CATS = ["Negative", "Positive", "Uncertainty", "Litigious", "Strong_Modal", "Weak_Modal", "Constraining"]
_WORD = re.compile(r"[A-Za-z][A-Za-z'\-]+")
_SENT = re.compile(r"(?<=[.!?])\s+|\n+")
_DIG = re.compile(r"\d")
MAIN_SECTION = {"periodic_10K": ["10K_7", "10K_1A", "10K_1", "10K_7A"], "periodic_10Q": ["10Q_MDA", "10Q_RF", "10Q_MR"],
                "8K": ["8K_EX99", "8K_ITEMS"]}


def load_lm(path) -> dict:
    df = pd.read_csv(path)
    lm = {}
    for c in LM_CATS:
        if c in df.columns:
            for w in df.loc[df[c] > 0, "Word"].astype(str).str.upper():
                lm.setdefault(w, set()).add(c)
    return lm


def lm_props(text: str, lm: dict) -> dict:
    words = [w.upper() for w in _WORD.findall(text)]
    n = len(words)
    counts = dict.fromkeys(LM_CATS, 0)
    for w in words:
        cats = lm.get(w)
        if cats:
            for c in cats:
                counts[c] += 1
    out = {f"lm_{c.lower()}": (counts[c] / n if n else np.nan) for c in LM_CATS}
    out["lm_net_tone"] = ((counts["Positive"] - counts["Negative"]) / (counts["Positive"] + counts["Negative"])
                          if (counts["Positive"] + counts["Negative"]) else np.nan)
    out["lm_n_words"] = n
    return out


def sentence_hashes(text: str) -> set:
    out = set()
    for s in _SENT.split(text):
        s = " ".join(s.lower().split())
        if len(s.split()) < 6:
            continue
        s = _DIG.sub("0", s)
        out.add(zlib.crc32(s.encode()) ^ (len(s) << 32))
    return out


class _Vec:
    def __init__(self):
        from sklearn.feature_extraction.text import HashingVectorizer
        self.hv = HashingVectorizer(n_features=2 ** 18, alternate_sign=False, norm="l2", stop_words="english",
                                    token_pattern=r"(?u)\b[a-zA-Z]{2,}\b")

    def __call__(self, text: str):
        return self.hv.transform([text])


def event_doc(sec_texts: dict, event_type: str, narrative: bool = True) -> tuple[str, str]:
    """(full doc text, main-section text) from {section_id: text}."""
    order = MAIN_SECTION.get(event_type, [])
    parts = [sec_texts[s] for s in order if s in sec_texts] + [v for k, v in sec_texts.items() if k not in order]
    doc = "\n\n".join(parts)
    main = next((sec_texts[s] for s in order if s in sec_texts), "")
    if narrative:
        doc, main = drop_numeric_lines(doc), drop_numeric_lines(main)
    return doc, main


def firm_text_features(events: pd.DataFrame, texts: dict, lm: dict) -> pd.DataFrame:
    """events (one firm): event_id, event_type, decision_time, accessions, public_available_time.
    texts: accession -> {section_id: text_raw}."""
    vec = _Vec()
    ev = events.sort_values("decision_time").reset_index(drop=True)
    memory: set = set()
    last: dict = {}           # event_type -> (vector, lm props, sentence set, n_words, rf_sentences)
    rows = []
    for _, tie in ev.groupby("decision_time", sort=True):
        staged = []
        for r in tie.itertuples(index=False):
            secs = {}
            for a in r.accessions:
                for sid, t in (texts.get(a) or {}).items():
                    secs[sid] = (secs[sid] + "\n\n" + t) if sid in secs else t
            doc, main = event_doc(secs, r.event_type)
            f = {"event_id": r.event_id, "txt_has_text": bool(doc.strip()), "txt_doc_chars": len(doc)}
            if not doc.strip():
                rows.append(f)
                continue
            p_doc = lm_props(doc, lm)
            p_main = lm_props(main, lm) if main else {}
            f.update({k: v for k, v in p_doc.items()})
            f.update({f"main_{k}": v for k, v in p_main.items()})
            v = vec(doc)
            sh = sentence_hashes(doc)
            rf = sentence_hashes(drop_numeric_lines(secs.get("10K_1A", "") or secs.get("10Q_RF", "")))
            prev = last.get(r.event_type)
            if prev is not None:
                pv, pp, psh, pn, prf = prev
                f["chg_cosine"] = float(v.multiply(pv).sum())
                f["chg_jaccard_sent"] = len(sh & psh) / max(1, len(sh | psh))
                f["chg_log_len"] = np.log((p_doc["lm_n_words"] + 1) / (pn + 1))
                for c in ("lm_negative", "lm_positive", "lm_uncertainty", "lm_litigious", "lm_net_tone"):
                    a, b = p_doc.get(c), pp.get(c)
                    f[f"chg_{c}"] = (a - b) if (a is not None and b is not None and np.isfinite(a) and np.isfinite(b)) else np.nan
                f["boilerplate_share"] = len(sh & psh) / max(1, len(sh))
                if rf and prf:
                    f["rf_new_share"] = len(rf - prf) / max(1, len(rf))
                    f["rf_removed_share"] = len(prf - rf) / max(1, len(prf))
            f["novelty_share"] = len(sh - memory) / max(1, len(sh)) if memory else np.nan
            staged.append((r.event_type, v, p_doc, sh, p_doc["lm_n_words"], rf))
            rows.append(f)
        for et, v, p, sh, n, rf in staged:      # tie batch: update memory only after the whole batch
            memory |= sh
            prev = last.get(et)
            last[et] = (v, p, sh, n, rf if rf else (prev[4] if prev else set()))
    return pd.DataFrame(rows)


def model_text(sec_texts_masked: dict, event_type: str, max_chars: int) -> str:
    """Masked narrative text fed to TF-IDF and fine-tuning (main section first)."""
    doc, _ = event_doc(sec_texts_masked, event_type)
    return doc[:max_chars]


In [ ]:
%%writefile $REPO/src/secprice/gold/__init__.py
"""secprice.gold"""


In [ ]:
%%writefile $REPO/src/secprice/gold/tools.py
"""Gold set (Track C, SPEC.md §12; addendum P1.6): stratified sampling from training-period filings, disjoint
gold-dev / gold-val / gold-test partitions, CSV/Excel templates + annotation guide, validated import, Cohen's
kappa, and a code-enforced single use of gold-test."""
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

TEMPLATE_COLS = ["gold_split", "item_id", "event_id", "accession", "form", "filing_date", "section_id", "field",
                 "question", "present", "direction", "magnitude", "unit", "certainty", "evidence_quote",
                 "evidence_chunk_id", "annotator", "notes", "llm_prefill_json"]
PASSAGE_COLS = ["gold_split", "item_id", "event_id", "chunk_id", "assigned_label", "passage_text", "supports_label",
                "annotator", "notes"]


def era(year: int) -> str:
    return "2000-08" if year <= 2008 else ("2009-19" if year <= 2019 else "2020+")


def sample_gold(events: pd.DataFrame, train_end: pd.Timestamp, sizes: dict, seed: int) -> pd.DataFrame:
    """Stratify by form x era x category x sector over training-period events; oversample rare categories."""
    ev = events[pd.to_datetime(events["decision_date"]) < train_end].copy()
    ev["era"] = ev["year"].map(era)
    ev["cat0"] = ev["categories"].map(lambda c: sorted(c)[0] if len(c) else "periodic")
    ev["stratum"] = ev["event_type"] + "|" + ev["era"] + "|" + ev["cat0"].astype(str) + "|" + ev["ff12_industry"].fillna("Other")
    rng = np.random.default_rng(seed)
    total = sum(sizes.values())
    counts = ev["stratum"].value_counts()
    # inverse-sqrt frequency weights oversample rare strata; sampling weights kept for prevalence estimates
    w = ev["stratum"].map(lambda s: 1 / np.sqrt(counts[s]))
    p = (w / w.sum()).to_numpy()
    k = min(total, len(ev))
    idx = rng.choice(len(ev), size=k, replace=False, p=p)
    s = ev.iloc[idx].copy()
    s["sampling_weight"] = 1.0 / (p[idx] * len(ev))
    s = s.sample(frac=1.0, random_state=seed)
    split = []
    for name, n in sizes.items():
        split += [name] * n
    s["gold_split"] = (split + ["unused"] * len(s))[: len(s)]
    s = s[s.gold_split != "unused"]
    assert not s.duplicated("event_id").any(), "a filing may appear in only one gold partition"
    return s


def export_templates(sample: pd.DataFrame, qb: dict, chunks: pd.DataFrame | None, out_dir: Path,
                     prefill: dict | None = None, n_passages: int = 200, seed: int = 0) -> dict:
    """Filings x fields template; passage template (supports-label); gold-test never gets LLM pre-fill."""
    out_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for r in sample.itertuples(index=False):
        for f, spec in qb["fields"].items():
            pf = ""
            if prefill and r.gold_split in ("gold_dev", "gold_val"):
                pf = json.dumps((prefill.get(r.event_id) or {}).get(f, {}))
            rows.append({"gold_split": r.gold_split, "item_id": f"{r.event_id}:{f}", "event_id": r.event_id,
                         "accession": r.accessions[0], "form": r.forms[0], "filing_date": str(r.filing_date),
                         "section_id": "", "field": f, "question": spec["question"], "present": "", "direction": "",
                         "magnitude": "", "unit": "", "certainty": "", "evidence_quote": "", "evidence_chunk_id": "",
                         "annotator": "", "notes": "", "llm_prefill_json": pf})
    tmpl = pd.DataFrame(rows, columns=TEMPLATE_COLS)
    paths = {}
    for split, g in tmpl.groupby("gold_split"):
        p = out_dir / f"gold_{split}_filings_template.csv"
        g.to_csv(p, index=False)
        paths[f"{split}_filings"] = str(p)
    try:
        tmpl.to_excel(out_dir / "gold_filings_template.xlsx", index=False)
        paths["xlsx"] = str(out_dir / "gold_filings_template.xlsx")
    except Exception:
        pass
    if chunks is not None and len(chunks):
        rng = np.random.default_rng(seed)
        prow = []
        for split in ("gold_val", "gold_test"):
            ids = set(sample[sample.gold_split == split].event_id)
            c = chunks[chunks.event_id.isin(ids)]
            if c.empty:
                continue
            c = c.iloc[rng.choice(len(c), size=min(n_passages, len(c)), replace=False)]
            for x in c.itertuples(index=False):
                prow.append({"gold_split": split, "item_id": f"{x.event_id}:{x.section_id}:{x.chunk_no}",
                             "event_id": x.event_id, "chunk_id": f"{x.event_id}:{x.section_id}:{x.chunk_no}",
                             "assigned_label": "", "passage_text": x.text[:3000], "supports_label": "",
                             "annotator": "", "notes": ""})
        pt = pd.DataFrame(prow, columns=PASSAGE_COLS)
        p = out_dir / "gold_passages_template.csv"
        pt.to_csv(p, index=False)
        paths["passages"] = str(p)
    (out_dir / "ANNOTATION_GUIDE.md").write_text(annotation_guide(qb), encoding="utf-8")
    paths["guide"] = str(out_dir / "ANNOTATION_GUIDE.md")
    return paths


def annotation_guide(qb: dict) -> str:
    lines = ["# Annotation guide (gold set v1)", "",
             "Label each filing x field from the filing text only. Never look at prices or returns: price outcomes "
             "never validate semantic labels (SPEC.md §12).", "",
             "- `present`: TRUE / FALSE.", "- `direction`: one of the listed values, or blank.",
             "- `magnitude` + `unit`: the stated number and unit (e.g. 250, USD millions), or blank.",
             "- `certainty`: low / medium / high.", "- `evidence_quote`: a verbatim quote of <= 30 words supporting the label.", "",
             "**gold-test is labelled blind**: its template has no LLM pre-fill. gold-dev and gold-val may show a pre-fill "
             "column; treat it as a suggestion and correct it (anchoring risk noted in the report).", "",
             "A second annotator should label a stratified >= 30% of gold-test independently (columns `annotator`), so "
             "Cohen's kappa can be reported. Without one, all content-validity results are labelled single-rater, provisional.",
             "", "## Fields"]
    for k, f in qb["fields"].items():
        lines.append(f"- **{k}** - {f['question']} Directions: {', '.join(f['directions'])}.")
    lines += ["", "Return the completed CSV files to `DATA_ROOT/inputs/gold/` with the same file names."]
    return "\n".join(lines)


def import_gold(folder: Path, sample: pd.DataFrame, qb: dict) -> tuple[pd.DataFrame, list[str]]:
    errs = []
    frames = []
    for p in sorted(Path(folder).glob("gold_*_filings_template*.csv")):
        d = pd.read_csv(p, dtype=str).fillna("")
        missing = set(TEMPLATE_COLS) - set(d.columns)
        if missing:
            errs.append(f"{p.name}: missing columns {sorted(missing)}")
            continue
        frames.append(d)
    if not frames:
        return pd.DataFrame(columns=TEMPLATE_COLS), ["no completed gold files found"]
    g = pd.concat(frames, ignore_index=True)
    g = g[g["present"].astype(str).str.strip() != ""]
    valid_ids = set(sample.event_id)
    bad = ~g.event_id.isin(valid_ids)
    if bad.any():
        errs.append(f"{int(bad.sum())} rows reference events outside the frozen gold sample")
        g = g[~bad]
    split_of = dict(zip(sample.event_id, sample.gold_split))
    mism = g.gold_split != g.event_id.map(split_of)
    if mism.any():
        errs.append(f"{int(mism.sum())} rows changed gold partition (rejected)")
        g = g[~mism]
    for f, spec in qb["fields"].items():
        m = (g.field == f) & (g.direction != "") & ~g.direction.isin(spec["directions"])
        if m.any():
            errs.append(f"{int(m.sum())} invalid directions for {f}")
    g["present_b"] = g["present"].str.strip().str.lower().isin(["true", "1", "yes", "y"])
    return g, errs


def cohen_kappa(a, b) -> float | None:
    from sklearn.metrics import cohen_kappa_score
    if len(a) < 5:
        return None
    try:
        return float(cohen_kappa_score(a, b))
    except Exception:
        return None


class GoldTestLock:
    """gold-test may be evaluated once, after the pipeline freeze (addendum P1.6)."""

    def __init__(self, path: Path):
        self.path = Path(path)

    def used(self) -> dict | None:
        return json.loads(self.path.read_text()) if self.path.exists() else None

    def consume(self, frozen_hash: str) -> None:
        if self.path.exists():
            raise RuntimeError(f"gold-test already used: {self.path.read_text()}")
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.path.write_text(json.dumps({"used_at": pd.Timestamp.utcnow().isoformat(), "frozen_hash": frozen_hash}))


In [ ]:
%%writefile $REPO/src/secprice/logging_utils.py
"""Structured logs per stage, errors.log and error-rate tracking (SPEC.md §19, §13.3)."""
from __future__ import annotations

import json
import logging
import sys
from pathlib import Path

_FMT = "%(asctime)s %(levelname)s [%(name)s] %(message)s"


def get_logger(name: str, log_dir: Path | None = None) -> logging.Logger:
    log = logging.getLogger(f"secprice.{name}")
    if getattr(log, "_secprice_ready", False):
        return log
    log.setLevel(logging.INFO)
    log.propagate = False
    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(logging.Formatter("%(asctime)s [%(name)s] %(message)s", "%H:%M:%S"))
    log.addHandler(sh)
    if log_dir:
        log_dir = Path(log_dir)
        log_dir.mkdir(parents=True, exist_ok=True)
        fh = logging.FileHandler(log_dir / f"{name}.log", encoding="utf-8")
        fh.setFormatter(logging.Formatter(_FMT))
        log.addHandler(fh)
        eh = logging.FileHandler(log_dir / "errors.log", encoding="utf-8")
        eh.setLevel(logging.ERROR)
        eh.setFormatter(logging.Formatter(_FMT))
        log.addHandler(eh)
    log._secprice_ready = True  # type: ignore[attr-defined]
    return log


class ErrorRateExceeded(RuntimeError):
    pass


class StageIncomplete(RuntimeError):
    """A resumable stage stopped cleanly at its wall-clock budget; run it again to continue (nothing is lost)."""


class ErrorTracker:
    """Counts successes/failures; raises when the failure share exceeds budget.max_error_rate after min_n items."""

    def __init__(self, max_rate: float, min_n: int = 50, name: str = ""):
        self.max_rate, self.min_n, self.name = float(max_rate), int(min_n), name
        self.ok = 0
        self.fail = 0
        self.reasons: dict[str, int] = {}

    def success(self) -> None:
        self.ok += 1

    def failure(self, reason: str) -> None:
        self.fail += 1
        key = str(reason)[:80]
        self.reasons[key] = self.reasons.get(key, 0) + 1
        n = self.ok + self.fail
        if n >= self.min_n and self.fail / n > self.max_rate:
            raise ErrorRateExceeded(f"{self.name}: error rate {self.fail}/{n} > {self.max_rate}; "
                                    f"top reasons {json.dumps(self.top())}")

    @property
    def rate(self) -> float:
        n = self.ok + self.fail
        return self.fail / n if n else 0.0

    def top(self, k: int = 5) -> dict:
        return dict(sorted(self.reasons.items(), key=lambda kv: -kv[1])[:k])

    def summary(self) -> dict:
        return {"ok": self.ok, "fail": self.fail, "rate": round(self.rate, 4), "top_reasons": self.top()}


In [ ]:
%%writefile $REPO/src/secprice/market/__init__.py
"""secprice.market"""


In [ ]:
%%writefile $REPO/src/secprice/market/calendar.py
"""NYSE trading calendar with early closes; session lookups used by every clock (SPEC.md §5.1, §5.3, §5.4).

All lookups work on UTC nanoseconds so that DST, holidays and early closes are handled by the schedule itself.
"""
from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd

ET = "America/New_York"


def _to_utc_ns(ts) -> int:
    t = pd.Timestamp(ts)
    if t.tzinfo is None:
        raise ValueError("timestamp must be timezone-aware")
    return int(t.tz_convert("UTC").value)


@dataclass
class TradingCalendar:
    sessions: pd.DatetimeIndex          # naive session dates
    open_utc: np.ndarray                # int64 ns
    close_utc: np.ndarray               # int64 ns
    source: str = "pandas_market_calendars:NYSE"

    @classmethod
    def nyse(cls, start: str, end: str) -> "TradingCalendar":
        import pandas_market_calendars as mcal
        sch = mcal.get_calendar("NYSE").schedule(start_date=start, end_date=end)
        o = pd.DatetimeIndex(sch["market_open"]).tz_convert("UTC")
        c = pd.DatetimeIndex(sch["market_close"]).tz_convert("UTC")
        ver = getattr(mcal, "__version__", "?")
        return cls(pd.DatetimeIndex(sch.index).tz_localize(None).normalize(), o.asi8.astype(np.int64),
                   c.asi8.astype(np.int64), f"pandas_market_calendars {ver}:NYSE")

    @classmethod
    def from_sessions(cls, dates, open_time="09:30", close_time="16:00", early: dict | None = None) -> "TradingCalendar":
        """Synthetic calendar for fixtures/tests. early = {date_str: 'HH:MM'} early closes."""
        early = early or {}
        d = pd.DatetimeIndex(pd.to_datetime(list(dates))).normalize()
        opens, closes = [], []
        for x in d:
            ds = x.strftime("%Y-%m-%d")
            opens.append(pd.Timestamp(f"{ds} {open_time}").tz_localize(ET).tz_convert("UTC").value)
            closes.append(pd.Timestamp(f"{ds} {early.get(ds, close_time)}").tz_localize(ET).tz_convert("UTC").value)
        return cls(d, np.array(opens, dtype=np.int64), np.array(closes, dtype=np.int64), "synthetic")

    # ------------------------------------------------------------------ basic lookups
    def __len__(self) -> int:
        return len(self.sessions)

    def index_of_date(self, d) -> int:
        """Session index of a calendar date, or -1 if not a session."""
        d = pd.Timestamp(d).normalize()
        if d.tzinfo is not None:
            d = d.tz_localize(None)
        i = self.sessions.searchsorted(d)
        return int(i) if i < len(self.sessions) and self.sessions[i] == d else -1

    def bucket(self, ts) -> str:
        """PRE / INTRA / POST / NONTRADING from an ET-aware timestamp against the actual session open/close."""
        if ts is None or pd.isna(ts):
            return "UNKNOWN"
        t = pd.Timestamp(ts).tz_convert(ET)
        i = self.index_of_date(t.tz_localize(None).normalize())
        if i < 0:
            return "NONTRADING"
        v = _to_utc_ns(t)
        if v < self.open_utc[i]:
            return "PRE"
        if v < self.close_utc[i]:
            return "INTRA"
        return "POST"

    def last_close_at_or_before(self, ts) -> int:
        """Index of the last regular-session close <= ts (a close printed at the same instant counts as before)."""
        return int(np.searchsorted(self.close_utc, _to_utc_ns(ts), side="right") - 1)

    def first_close_after(self, ts) -> int:
        """Index of the first session whose close is strictly after ts."""
        return int(np.searchsorted(self.close_utc, _to_utc_ns(ts), side="right"))

    def first_open_at_or_after(self, ts) -> int:
        """Index of the first session whose open is at or after ts (T_trade entry rule)."""
        return int(np.searchsorted(self.open_utc, _to_utc_ns(ts), side="left"))

    def day0(self, public_ts) -> int:
        """Track A day 0: the session in which public time falls if before that session's close, else the next one."""
        return self.first_close_after(public_ts)

    def date(self, i: int):
        return self.sessions[i] if 0 <= i < len(self.sessions) else pd.NaT

    def close_time(self, i: int) -> pd.Timestamp:
        return pd.Timestamp(self.close_utc[i], tz="UTC").tz_convert(ET)

    def open_time(self, i: int) -> pd.Timestamp:
        return pd.Timestamp(self.open_utc[i], tz="UTC").tz_convert(ET)

    def is_early_close(self, i: int) -> bool:
        return self.close_time(i).hour < 16

    # ------------------------------------------------------------------ vectorised versions
    def v_bucket(self, ts_utc_ns: np.ndarray, valid: np.ndarray) -> np.ndarray:
        out = np.full(len(ts_utc_ns), "UNKNOWN", dtype=object)
        # session date in ET for each timestamp
        et_dates = pd.to_datetime(ts_utc_ns, utc=True).tz_convert(ET).tz_localize(None).normalize()
        idx = self.sessions.searchsorted(et_dates)
        idx_c = np.clip(idx, 0, len(self.sessions) - 1)
        is_sess = (idx < len(self.sessions)) & (self.sessions[idx_c] == et_dates)
        o, c = self.open_utc[idx_c], self.close_utc[idx_c]
        out[valid & ~is_sess] = "NONTRADING"
        m = valid & is_sess
        out[m & (ts_utc_ns < o)] = "PRE"
        out[m & (ts_utc_ns >= o) & (ts_utc_ns < c)] = "INTRA"
        out[m & (ts_utc_ns >= c)] = "POST"
        return out

    def v_last_close_at_or_before(self, ts_utc_ns: np.ndarray) -> np.ndarray:
        return np.searchsorted(self.close_utc, ts_utc_ns, side="right") - 1

    def v_first_close_after(self, ts_utc_ns: np.ndarray) -> np.ndarray:
        return np.searchsorted(self.close_utc, ts_utc_ns, side="right")

    def v_first_open_at_or_after(self, ts_utc_ns: np.ndarray) -> np.ndarray:
        return np.searchsorted(self.open_utc, ts_utc_ns, side="left")

    def to_frame(self) -> pd.DataFrame:
        return pd.DataFrame({
            "date": self.sessions,
            "open_et": pd.to_datetime(self.open_utc, utc=True).tz_convert(ET),
            "close_et": pd.to_datetime(self.close_utc, utc=True).tz_convert(ET),
        }).assign(early_close=lambda d: d.close_et.dt.hour < 16)

    @classmethod
    def from_frame(cls, df: pd.DataFrame, source: str = "artifact") -> "TradingCalendar":
        o = pd.DatetimeIndex(df["open_et"]).tz_convert("UTC").asi8.astype(np.int64)
        c = pd.DatetimeIndex(df["close_et"]).tz_convert("UTC").asi8.astype(np.int64)
        return cls(pd.DatetimeIndex(pd.to_datetime(df["date"])).normalize(), o, c, source)


In [ ]:
%%writefile $REPO/src/secprice/market/factors.py
"""Kenneth French daily factors: FF5 (2x3) + momentum -> factors.parquet (decimal returns)."""
from __future__ import annotations

import io
import zipfile

import pandas as pd
import requests


def _parse_french_zip(raw: bytes) -> pd.DataFrame:
    z = zipfile.ZipFile(io.BytesIO(raw))
    text = z.read(z.namelist()[0]).decode("latin1")
    lines = text.splitlines()
    start = next(i for i, l in enumerate(lines) if l.strip().startswith(",") and any(c.isalpha() for c in l))
    header = [h.strip() for h in lines[start].split(",")]
    rows = []
    for l in lines[start + 1:]:
        parts = [p.strip() for p in l.split(",")]
        if len(parts) < 2 or not parts[0].isdigit() or len(parts[0]) != 8:
            if rows:
                break
            continue
        rows.append(parts)
    df = pd.DataFrame(rows, columns=["date"] + header[1:])
    df["date"] = pd.to_datetime(df["date"], format="%Y%m%d")
    for c in df.columns[1:]:
        df[c] = pd.to_numeric(df[c], errors="coerce") / 100.0
    return df


def fetch_factors(ff5_url: str, mom_url: str, cache_dir=None) -> tuple[pd.DataFrame, dict]:
    import hashlib
    raws = {}
    for name, url in (("ff5", ff5_url), ("mom", mom_url)):
        cp = (cache_dir / f"{name}.zip") if cache_dir is not None else None
        if cp is not None and cp.exists():
            raws[name] = cp.read_bytes()
        else:
            r = requests.get(url, timeout=120)
            r.raise_for_status()
            raws[name] = r.content
            if cp is not None:
                cp.write_bytes(r.content)
    ff5 = _parse_french_zip(raws["ff5"]).rename(columns={"Mkt-RF": "mkt_rf", "SMB": "smb", "HML": "hml", "RMW": "rmw",
                                                         "CMA": "cma", "RF": "rf"})
    mom = _parse_french_zip(raws["mom"])
    mom = mom.rename(columns={mom.columns[1]: "mom"})[["date", "mom"]]
    out = ff5.merge(mom, on="date", how="left")[["date", "mkt_rf", "smb", "hml", "rmw", "cma", "mom", "rf"]]
    prov = {k: hashlib.sha256(v).hexdigest() for k, v in raws.items()}
    prov.update({"last_date": str(out.date.max().date()), "first_date": str(out.date.min().date())})
    return out, prov


In [ ]:
%%writefile $REPO/src/secprice/market/prices.py
"""S03 prices: pluggable providers and the firm x session panel (SPEC.md §13.5 'Prices', §16 prices.parquet).

Series kept separately (SPEC.md §5.5): close_raw, close_split_adj and a total-return index. Local TriFrame bars
are split-adjusted (Yahoo split history), not dividend-adjusted; dividends come from Yahoo actions (split-adjusted
amounts, the same basis). Before the local history starts, yfinance supplies the market warm-up (explicit
parameters recorded).
"""
from __future__ import annotations

import time
from pathlib import Path

import numpy as np
import pandas as pd

BAR_COLS = ["date", "open", "high", "low", "close", "volume", "source"]


class LocalTriframeProvider:
    """DATA_ROOT/inputs/ohlcv_1day/<TICKER>.parquet with columns date, open, high, low, close, volume, source."""

    def __init__(self, folder: Path, version: str = "triframe_final_1day"):
        self.folder = Path(folder)
        self.version = version

    def tickers(self) -> list[str]:
        return sorted(p.stem for p in self.folder.glob("*.parquet"))

    def coverage(self) -> pd.DataFrame:
        import pyarrow.compute as pc
        import pyarrow.parquet as pq
        rows = []
        for p in sorted(self.folder.glob("*.parquet")):
            t = pq.read_table(p, columns=["date"])
            rows.append({"ticker": p.stem, "price_start": pc.min(t["date"]).as_py(), "price_end": pc.max(t["date"]).as_py(),
                         "n_bars": t.num_rows})
        df = pd.DataFrame(rows)
        for c in ("price_start", "price_end"):
            df[c] = pd.to_datetime(df[c]).dt.normalize()
        return df

    def load(self, ticker: str) -> pd.DataFrame:
        p = self.folder / f"{ticker}.parquet"
        if not p.exists():
            return pd.DataFrame(columns=BAR_COLS)
        df = pd.read_parquet(p)
        df["date"] = pd.to_datetime(df["date"]).dt.normalize()
        if "source" not in df:
            df["source"] = "local"
        df["source"] = "local:" + df["source"].astype(str)
        return df[BAR_COLS].sort_values("date").reset_index(drop=True)


class YFinanceProvider:
    """yfinance with explicit parameters; reads a frozen cache file first when present (reproducible runs)."""

    def __init__(self, params: dict, cache_file: Path | None = None, log=None):
        self.params = dict(params)
        self.cache_file = Path(cache_file) if cache_file else None
        self.log = log
        self._cache = None
        try:
            import yfinance
            self.version = f"yfinance {yfinance.__version__}"
        except Exception:
            self.version = "yfinance unavailable"

    def _cached(self) -> pd.DataFrame | None:
        if self._cache is None and self.cache_file is not None and self.cache_file.exists():
            self._cache = pd.read_parquet(self.cache_file)
            self._cache["date"] = pd.to_datetime(self._cache["date"]).dt.normalize()
        return self._cache

    def history(self, ticker: str, start: str, end: str) -> tuple[pd.DataFrame, pd.DataFrame]:
        """(bars, actions) for [start, end). Actions: date, type in {dividend, split}, value."""
        c = self._cached()
        if c is not None and ticker in set(c["ticker"].unique()):
            sub = c[(c.ticker == ticker) & (c.date >= pd.Timestamp(start)) & (c.date < pd.Timestamp(end))]
            bars = sub[["date", "open", "high", "low", "close", "volume"]].assign(source="yfinance_frozen")
            acts = sub.melt(id_vars="date", value_vars=["dividends", "splits"], var_name="type", value_name="value")
            acts = acts[acts.value > 0].assign(type=lambda d: d.type.map({"dividends": "dividend", "splits": "split"}))
            return bars.reset_index(drop=True), acts.reset_index(drop=True)
        import yfinance as yf
        p = self.params
        for attempt in range(4):
            try:
                h = yf.Ticker(ticker.replace(".", "-")).history(start=start, end=end, interval=p.get("interval", "1d"),
                                                               auto_adjust=p.get("auto_adjust", False),
                                                               actions=p.get("actions", True),
                                                               back_adjust=p.get("back_adjust", False),
                                                               repair=p.get("repair", False))
                break
            except Exception as e:  # rate limits: back off, never evade
                if self.log:
                    self.log.warning(f"yfinance {ticker} attempt {attempt + 1}: {e}")
                time.sleep(5 * (attempt + 1))
        else:
            return pd.DataFrame(columns=BAR_COLS), pd.DataFrame(columns=["date", "type", "value"])
        if h is None or h.empty:
            return pd.DataFrame(columns=BAR_COLS), pd.DataFrame(columns=["date", "type", "value"])
        h = h.reset_index()
        h["date"] = pd.to_datetime(h["Date"]).dt.tz_localize(None).dt.normalize()
        bars = pd.DataFrame({"date": h.date, "open": h.Open, "high": h.High, "low": h.Low, "close": h.Close,
                             "volume": h.Volume, "source": "yfinance_live"})
        acts = []
        if "Dividends" in h:
            acts.append(pd.DataFrame({"date": h.date, "type": "dividend", "value": h.Dividends}))
        if "Stock Splits" in h:
            acts.append(pd.DataFrame({"date": h.date, "type": "split", "value": h["Stock Splits"]}))
        acts = pd.concat(acts) if acts else pd.DataFrame(columns=["date", "type", "value"])
        return bars, acts[acts.value > 0].reset_index(drop=True)


def load_actions_file(path: Path) -> pd.DataFrame:
    """inputs/yahoo_actions.parquet: ticker, date, type (dividend|split), value."""
    if not Path(path).exists():
        return pd.DataFrame(columns=["ticker", "date", "type", "value"])
    a = pd.read_parquet(path)
    a["date"] = pd.to_datetime(a["date"]).dt.normalize()
    return a


def build_series(bars: pd.DataFrame, actions: pd.DataFrame) -> pd.DataFrame:
    """close_raw, close_split_adj, total_return_index from split-adjusted bars + actions (split-adjusted dividends).

    close_raw(d) = close_split_adj(d) x product of split ratios with split date > d.
    TRI(t) = TRI(t-1) x (C(t) + D(t)) / C(t-1) where D(t) is the dividend with ex-date t.
    """
    b = bars.sort_values("date").drop_duplicates("date", keep="last").reset_index(drop=True).copy()
    b = b.rename(columns={"close": "close_split_adj"})
    splits = actions[actions.type == "split"][["date", "value"]] if len(actions) else pd.DataFrame(columns=["date", "value"])
    divs = actions[actions.type == "dividend"].groupby("date")["value"].sum() if len(actions) else pd.Series(dtype=float)
    factor = np.ones(len(b))
    for d, r in zip(pd.to_datetime(splits["date"]), splits["value"].astype(float)):
        if r > 0:
            factor[b["date"].values < np.datetime64(d)] *= r
    b["close_raw"] = b["close_split_adj"] * factor
    b["dividend"] = b["date"].map(divs).fillna(0.0).astype(float)
    c = b["close_split_adj"].to_numpy(float)
    dv = b["dividend"].to_numpy(float)
    gross = np.ones(len(b))
    if len(b) > 1:
        prev = c[:-1]
        with np.errstate(divide="ignore", invalid="ignore"):
            g = (c[1:] + dv[1:]) / prev
        g[~np.isfinite(g)] = np.nan
        gross[1:] = g
    tri = np.nancumprod(np.where(np.isnan(gross), 1.0, gross))
    tri[np.isnan(c)] = np.nan
    b["total_return_index"] = tri
    return b


def splice(local: pd.DataFrame, warm: pd.DataFrame, seam_days: int = 40) -> tuple[pd.DataFrame, dict]:
    """Use yfinance only before the first local date; report the seam agreement on overlapping days."""
    if local.empty:
        return warm.copy(), {"seam_overlap_days": 0, "seam_median_abs_pct": np.nan, "warmup_days": len(warm)}
    first = local["date"].min()
    ov = warm[(warm.date >= first)].merge(local[["date", "close"]], on="date", suffixes=("_yf", "_local")).head(seam_days)
    med = float((ov.close_local / ov.close_yf - 1).abs().median() * 100) if len(ov) else np.nan
    pre = warm[warm.date < first]
    out = pd.concat([pre, local], ignore_index=True).sort_values("date").reset_index(drop=True)
    return out, {"seam_overlap_days": int(len(ov)), "seam_median_abs_pct": med, "warmup_days": int(len(pre))}


class PricePanel:
    """Per-firm arrays aligned to the global session calendar (NaN where no bar)."""

    def __init__(self, prices: pd.DataFrame, cal):
        self.cal = cal
        self.n = len(cal)
        self.firms: dict[str, dict[str, np.ndarray]] = {}
        sess = cal.sessions
        for key, g in prices.groupby("ticker", sort=False):
            idx = sess.searchsorted(pd.to_datetime(g["date"]).values)
            ok = (idx < self.n) & (sess[np.clip(idx, 0, self.n - 1)] == pd.to_datetime(g["date"]).values)
            arr = {}
            for col in ("open", "high", "low", "close_split_adj", "close_raw", "total_return_index", "volume", "dividend"):
                a = np.full(self.n, np.nan)
                if col in g:
                    a[idx[ok]] = g[col].to_numpy(float)[ok]
                arr[col] = a
            tri = arr["total_return_index"]
            r = np.full(self.n, np.nan)
            r[1:] = tri[1:] / tri[:-1] - 1
            arr["ret"] = r
            self.firms[str(key)] = arr

    def has(self, ticker: str) -> bool:
        return ticker in self.firms

    def __getitem__(self, ticker: str) -> dict[str, np.ndarray]:
        return self.firms[ticker]


In [ ]:
%%writefile $REPO/src/secprice/market/returns.py
"""Interval returns with explicit dividend handling and interval-matched benchmark legs (addendum P0.2).

Rules
- close->close total return over sessions (b, x]: TRI[x] / TRI[b] - 1 (dividends with ex-date in (b, x]).
- open->close executable return from the entry open of session e to the close of session x >= e:
  (C[e] / O[e]) x TRI[x] / TRI[e] - 1. A dividend with ex-date e is excluded (an open buyer is not entitled);
  dividends with ex-date in (e, x] are included. All prices are on one split basis, so a split inside the
  holding interval has no effect.
- The benchmark (SPY) leg is computed on exactly the same interval with the same formula. A close-to-close
  benchmark is never subtracted from an open-to-close stock return.
"""
from __future__ import annotations

import numpy as np


def cc_return(tri: np.ndarray, b: int, x: int) -> float:
    if b < 0 or x >= len(tri) or x <= b:
        return np.nan
    return float(tri[x] / tri[b] - 1.0)


def oc_return(open_: np.ndarray, close: np.ndarray, tri: np.ndarray, e: int, x: int) -> float:
    if e < 0 or x >= len(tri) or x < e:
        return np.nan
    if not (np.isfinite(open_[e]) and open_[e] > 0 and np.isfinite(close[e]) and np.isfinite(tri[e]) and np.isfinite(tri[x])):
        return np.nan
    return float((close[e] / open_[e]) * (tri[x] / tri[e]) - 1.0)


def v_cc_return(tri: np.ndarray, b: np.ndarray, x: np.ndarray) -> np.ndarray:
    n = len(tri)
    ok = (b >= 0) & (x < n) & (x > b)
    out = np.full(len(b), np.nan)
    out[ok] = tri[x[ok]] / tri[b[ok]] - 1.0
    return out


def v_oc_return(open_: np.ndarray, close: np.ndarray, tri: np.ndarray, e: np.ndarray, x: np.ndarray) -> np.ndarray:
    n = len(tri)
    ok = (e >= 0) & (x < n) & (x >= e)
    out = np.full(len(e), np.nan)
    ee, xx = e[ok], x[ok]
    with np.errstate(divide="ignore", invalid="ignore"):
        out[ok] = (close[ee] / open_[ee]) * (tri[xx] / tri[ee]) - 1.0
    out[~np.isfinite(out)] = np.nan
    return out


def daily_returns(tri: np.ndarray) -> np.ndarray:
    r = np.full(len(tri), np.nan)
    r[1:] = tri[1:] / tri[:-1] - 1.0
    return r


In [ ]:
%%writefile $REPO/src/secprice/models/__init__.py
"""secprice.models"""


In [ ]:
%%writefile $REPO/src/secprice/models/embeddings.py
"""S14 frozen encoder embeddings (SPEC.md §7.2): one model in memory at a time -> embed -> persist float16 + index
-> unload. Section-aware chunks; mean/max pooling to section vectors; event vector = chunk-weighted mean of
section means; change = cosine to the firm's prior same-type event vector (prior decision time)."""
from __future__ import annotations

import gc
import time

import numpy as np
import pandas as pd

from ..parse.html import drop_numeric_lines


def chunk_words(text: str, size: int, overlap: int) -> list[str]:
    w = text.split()
    if not w:
        return []
    step = max(1, size - overlap)
    return [" ".join(w[i:i + size]) for i in range(0, max(1, len(w) - overlap), step)]


def select_chunks(chunks: list[str], max_n: int) -> list[str]:
    if len(chunks) <= max_n:
        return chunks
    idx = np.unique(np.linspace(0, len(chunks) - 1, max_n).round().astype(int))
    return [chunks[i] for i in idx]


def device_and_dtype():
    import torch
    if torch.cuda.is_available():
        return "cuda", (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16)
    return "cpu", torch.float32


def load_tokenizer(model_id: str, revision: str | None):
    """AutoTokenizer, falling back to BertTokenizerFast for BERT checkpoints whose config lacks `model_type`
    (e.g. yiyanghkust/finbert-tone under transformers 5)."""
    from transformers import AutoTokenizer, BertTokenizerFast
    try:
        return AutoTokenizer.from_pretrained(model_id, revision=revision)
    except Exception:
        return BertTokenizerFast.from_pretrained(model_id, revision=revision)


def load_model(model_id: str, revision: str | None, kind: str, dtype=None, **kw):
    """kind: 'base' | 'classifier'. Falls back to the BERT classes when the config has no model_type."""
    from transformers import AutoModel, AutoModelForSequenceClassification, BertForSequenceClassification, BertModel
    auto = AutoModelForSequenceClassification if kind == "classifier" else AutoModel
    bert = BertForSequenceClassification if kind == "classifier" else BertModel
    last = None
    for cls in (auto, bert):
        for dkw in (({"dtype": dtype} if dtype is not None else {}), ({"torch_dtype": dtype} if dtype is not None else {})):
            try:
                return cls.from_pretrained(model_id, revision=revision, **dkw, **kw)
            except TypeError as e:
                last = e
                continue
            except (ValueError, KeyError, OSError) as e:
                last = e
                break
    raise RuntimeError(f"could not load {model_id}@{revision}: {last}")


def load_encoder(model_id: str, revision: str | None, kind: str):
    dev, dt = device_and_dtype()
    tok = load_tokenizer(model_id, revision)
    m = load_model(model_id, revision, kind, dt)
    m.to(dev).eval()
    return tok, m, dev


def unload(*objs):
    import torch
    for o in objs:
        del o
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def encode(tok, model, dev, texts: list[str], max_len: int, batch: int, pooling: str, classifier: bool = False,
           log=None) -> tuple[np.ndarray, np.ndarray | None]:
    """Length-bucketed batched encoding with OOM retry that halves the batch."""
    import torch
    order = np.argsort([len(t) for t in texts])
    H = None
    probs = None
    out_vec, out_p = [None] * len(texts), [None] * len(texts)
    i, b = 0, batch
    while i < len(order):
        idx = order[i:i + b]
        enc = tok([texts[j] for j in idx], truncation=True, max_length=max_len, padding=True, return_tensors="pt")
        enc = {k: v.to(dev) for k, v in enc.items()}
        try:
            with torch.no_grad(), torch.autocast(device_type="cuda", enabled=(dev == "cuda")):
                o = model(**enc, output_hidden_states=classifier)
            hs = o.hidden_states[-1] if classifier else o.last_hidden_state
            mask = enc["attention_mask"].unsqueeze(-1).to(hs.dtype)
            if pooling == "cls":
                v = hs[:, 0]
                v = torch.nn.functional.normalize(v.float(), dim=-1)
            else:
                v = (hs * mask).sum(1) / mask.sum(1).clamp(min=1)
            v = v.float().cpu().numpy().astype(np.float16)
            p = torch.softmax(o.logits.float(), -1).cpu().numpy() if classifier else None
            for k, j in enumerate(idx):
                out_vec[j] = v[k]
                if p is not None:
                    out_p[j] = p[k]
            i += b
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if b == 1:
                raise
            b = max(1, b // 2)
            if log:
                log.warning(f"OOM: batch halved to {b}")
    V = np.vstack(out_vec) if out_vec else np.zeros((0, 768), np.float16)
    P = np.vstack(out_p) if classifier and out_p and out_p[0] is not None else None
    return V, P


def event_chunks(events: pd.DataFrame, sec_index: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    """Rows: event_id, section_id, chunk_no, text. sec_index: accession, section_id, text_masked."""
    t = cfg["text"]
    by_acc = {a: g for a, g in sec_index.groupby("accession")}
    rows = []
    for r in events.itertuples(index=False):
        for a in r.accessions:
            g = by_acc.get(a)
            if g is None:
                continue
            for s in g.itertuples(index=False):
                txt = drop_numeric_lines(s.text_masked) if t.get("drop_numeric_lines", True) else s.text_masked
                ch = select_chunks(chunk_words(txt, int(t["chunk_words"]), int(t["chunk_overlap_words"])),
                                   int(t["max_chunks_per_section"]))
                rows += [(r.event_id, s.section_id, i, c) for i, c in enumerate(ch)]
    return pd.DataFrame(rows, columns=["event_id", "section_id", "chunk_no", "text"])


def pool_events(chunks: pd.DataFrame, V: np.ndarray, P: np.ndarray | None = None) -> tuple[pd.Index, np.ndarray, pd.DataFrame]:
    """Event vector = chunk-count-weighted mean of section mean vectors (= mean over chunks); plus max-pooled
    section vectors averaged; native classifier probabilities averaged."""
    df = chunks[["event_id", "section_id"]].copy()
    df["row"] = np.arange(len(df))
    ev_ids, vecs, extra = [], [], []
    for eid, g in df.groupby("event_id", sort=False):
        rows = g.row.to_numpy()
        Vm = V[rows].astype(np.float32)
        mean_v = Vm.mean(0)
        max_v = np.mean([V[s.row.to_numpy()].astype(np.float32).max(0) for _, s in g.groupby("section_id")], axis=0)
        ev_ids.append(eid)
        vecs.append(np.concatenate([mean_v, max_v]).astype(np.float16))
        e = {"event_id": eid, "n_chunks": len(rows)}
        if P is not None:
            pm = P[rows].mean(0)
            e.update({"tone_neutral": pm[0], "tone_positive": pm[1], "tone_negative": pm[2]})
        extra.append(e)
    return pd.Index(ev_ids, name="event_id"), np.vstack(vecs) if vecs else np.zeros((0, 2 * V.shape[1]), np.float16), pd.DataFrame(extra)


def change_cosine(events: pd.DataFrame, ids: pd.Index, M: np.ndarray, dim: int) -> pd.Series:
    """Cosine between an event's mean vector and the firm's previous same-type event vector."""
    pos = {e: i for i, e in enumerate(ids)}
    out = {}
    ev = events[events.event_id.isin(pos)].sort_values("decision_time")
    for (cik, et), g in ev.groupby(["cik", "event_type"]):
        prev = None
        for eid in g.event_id:
            v = M[pos[eid], :dim].astype(np.float32)
            if prev is not None:
                den = np.linalg.norm(v) * np.linalg.norm(prev)
                out[eid] = float(v @ prev / den) if den > 0 else np.nan
            prev = v
    return pd.Series(out, name="chg_emb_cosine")


def chrono_vintage(year: int, vintages: list[int]) -> int:
    """Event year Y uses vintage Y-1; clipped to the available range (always precedes the event)."""
    v = min(year - 1, max(vintages))
    return max(v, min(vintages))


def benchmark(tok, model, dev, n: int = 256, max_len: int = 512) -> float:
    """Chunks per second on synthetic 350-word texts (preflight micro-benchmark)."""
    texts = [("financial results revenue margin outlook guidance risk " * 50)[:2400]] * n
    t0 = time.time()
    encode(tok, model, dev, texts, max_len, 64, "mean")
    return n / max(1e-6, time.time() - t0)


In [ ]:
%%writefile $REPO/src/secprice/models/finetune.py
"""S15 encoder fine-tuning (SPEC.md §7.2, §9.4): top-2 encoders by validation, only if G3 passes (or the owner
override, labelled exploratory). Fixed grid: lr {1e-5,2e-5,3e-5} x batch {8,16} x epochs <= 3 with early stopping
x weight decay {0, 0.01}. Class-weighted cross-entropy. Early stopping on an inner, time-ordered slice of the
training fold (never on the validation block); grid selection on the walk-forward validation block."""
from __future__ import annotations

import gc
import itertools
import math
import time

import numpy as np


def grid(cfg: dict) -> list[dict]:
    g = cfg["tuning"]["finetune_grid"]
    combos = [dict(lr=lr, batch=b, weight_decay=wd, epochs=int(g["epochs"]))
              for lr, b, wd in itertools.product(g["lr"], g["batch"], g["weight_decay"])]
    return combos[: int(g.get("max_trials", 12))]


def _class_weights(y: np.ndarray) -> np.ndarray:
    cnt = np.bincount(y.astype(int), minlength=3).astype(float)
    return np.where(cnt > 0, len(y) / (3 * np.maximum(cnt, 1)), 0.0)


def train_eval(model_id: str, revision: str | None, texts_tr, y_tr, w_tr, texts_iv, y_iv, texts_ev: list, hp: dict,
               max_len: int, seed: int, log=None) -> dict:
    """Fine-tune once; return per-step/epoch curves, inner-val best checkpoint predictions for each eval set."""
    import torch
    from torch.utils.data import DataLoader, TensorDataset
    from transformers import get_linear_schedule_with_warmup
    from .embeddings import load_model, load_tokenizer
    torch.manual_seed(seed)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    tok = load_tokenizer(model_id, revision)
    model = load_model(model_id, revision, "classifier", None, num_labels=3, ignore_mismatched_sizes=True).to(dev)
    use_bf16 = dev == "cuda" and torch.cuda.is_bf16_supported()

    def enc(texts):
        e = tok(list(texts), truncation=True, max_length=max_len, padding="max_length", return_tensors="pt")
        return e["input_ids"], e["attention_mask"]

    ids, am = enc(texts_tr)
    ds = TensorDataset(ids, am, torch.tensor(y_tr, dtype=torch.long), torch.tensor(w_tr, dtype=torch.float32))
    dl = DataLoader(ds, batch_size=int(hp["batch"]), shuffle=True, generator=torch.Generator().manual_seed(seed))
    iv_ids, iv_am = enc(texts_iv)
    cw = torch.tensor(_class_weights(np.asarray(y_tr)), dtype=torch.float32, device=dev)
    opt = torch.optim.AdamW(model.parameters(), lr=float(hp["lr"]), weight_decay=float(hp["weight_decay"]))
    total = len(dl) * int(hp["epochs"])
    sch = get_linear_schedule_with_warmup(opt, int(0.06 * total), max(1, total))
    evals_per_epoch = 4
    eval_every = max(1, len(dl) // evals_per_epoch)
    curve = {"step": [], "train_loss": [], "eval_step": [], "eval_epoch": [], "inner_val_loss": [], "lr": []}
    best = {"loss": math.inf, "state": None, "step": 0}
    bad, patience, step = 0, 2, 0

    def predict(i, a, bs=64):
        model.eval()
        out = []
        with torch.no_grad():
            for k in range(0, len(i), bs):
                with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_bf16):
                    lo = model(input_ids=i[k:k + bs].to(dev), attention_mask=a[k:k + bs].to(dev)).logits
                out.append(torch.softmax(lo.float(), -1).cpu().numpy())
        model.train()
        return np.vstack(out) if out else np.zeros((0, 3))

    model.train()
    stop = False
    t0 = time.time()
    for ep in range(int(hp["epochs"])):
        for bi, am_, yb, wb in dl:
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_bf16):
                logits = model(input_ids=bi.to(dev), attention_mask=am_.to(dev)).logits
            loss_vec = torch.nn.functional.cross_entropy(logits.float(), yb.to(dev), weight=cw, reduction="none")
            loss = (loss_vec * wb.to(dev)).sum() / wb.to(dev).sum().clamp(min=1e-6)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sch.step()
            opt.zero_grad(set_to_none=True)
            step += 1
            curve["step"].append(step)
            curve["train_loss"].append(float(loss.item()))
            curve["lr"].append(float(sch.get_last_lr()[0]))
            if step % eval_every == 0 or step == total:
                p = predict(iv_ids, iv_am)
                yv = np.asarray(y_iv).astype(int)
                vl = float(-np.mean(np.log(np.clip(p[np.arange(len(yv)), yv], 1e-9, 1)))) if len(yv) else math.nan
                curve["eval_step"].append(step)
                curve["eval_epoch"].append(step / max(1, len(dl)))
                curve["inner_val_loss"].append(vl)
                if vl < best["loss"] - 1e-4:
                    best = {"loss": vl, "state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, "step": step}
                    bad = 0
                else:
                    bad += 1
                    if bad >= patience * evals_per_epoch // 2 + 1:
                        stop = True
                        break
        if stop:
            break
    if best["state"] is not None:
        model.load_state_dict(best["state"])
    preds = [predict(*enc(t)) if len(t) else np.zeros((0, 3)) for t in texts_ev]
    p_tr = predict(ids[: min(len(ids), 4000)], am[: min(len(ids), 4000)])
    out = {"curve": curve, "best_step": best["step"], "best_inner_val_loss": best["loss"], "preds": preds,
           "train_proba_sample": p_tr, "train_y_sample": np.asarray(y_tr)[: len(p_tr)], "seconds": time.time() - t0,
           "early_stopped": stop}
    del model, opt, best
    gc.collect()
    if dev == "cuda":
        torch.cuda.empty_cache()
    return out


In [ ]:
%%writefile $REPO/src/secprice/models/heads.py
"""Tabular heads: LightGBM, CatBoost, ElasticNet-logistic (classification) and LightGBM/ElasticNet (regression).

Losses (SPEC.md §9.4): class-weighted cross-entropy (balanced weights x label-uniqueness weights); Huber for
regression. Early stopping and probability calibration use an inner, time-ordered slice of the training fold
only. Every fitted head keeps its train/validation loss curve.
"""
from __future__ import annotations

import os

import numpy as np

DEFAULTS = {
    "lightgbm": {"learning_rate": 0.05, "num_leaves": 31, "min_child_samples": 50, "subsample": 0.8,
                 "colsample_bytree": 0.8, "reg_alpha": 1e-3, "reg_lambda": 1.0, "max_depth": -1},
    "catboost": {"learning_rate": 0.05, "depth": 6, "l2_leaf_reg": 3.0, "random_strength": 1.0},
    "enet": {"C": 0.1, "l1_ratio": 0.5},
}


def space(trial, head: str) -> dict:
    """Search spaces fixed before the holdout is unlocked (SPEC.md §9.4; v3 §13.1)."""
    if head == "lightgbm":
        return {"learning_rate": trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
                "num_leaves": trial.suggest_int("num_leaves", 8, 128, log=True),
                "min_child_samples": trial.suggest_int("min_child_samples", 20, 400, log=True),
                "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
                "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
                "max_depth": trial.suggest_categorical("max_depth", [-1, 4, 6, 8])}
    if head == "catboost":
        return {"learning_rate": trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
                "depth": trial.suggest_int("depth", 4, 8),
                "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 20.0, log=True),
                "random_strength": trial.suggest_float("random_strength", 0.1, 5.0, log=True)}
    if head == "enet":
        return {"C": trial.suggest_float("C", 1e-3, 10.0, log=True),
                "l1_ratio": trial.suggest_float("l1_ratio", 0.0, 1.0)}
    raise ValueError(head)


def prior_correct(p_balanced: np.ndarray, prior: np.ndarray) -> np.ndarray:
    """A model trained with class-balanced weights estimates posteriors under uniform class priors. Multiplying by
    the training prior and renormalising returns true-frequency probabilities (then temperature-calibrated); the
    cost-balanced decision argmax p / prior is applied once, in evaluation.metrics.decide."""
    q = np.asarray(p_balanced, float) * np.maximum(np.asarray(prior, float), 1e-6)
    return q / q.sum(1, keepdims=True)


def _balanced_w(y: np.ndarray, k: int = 3) -> np.ndarray:
    cnt = np.bincount(y.astype(int), minlength=k).astype(float)
    cw = np.where(cnt > 0, len(y) / (k * np.maximum(cnt, 1)), 0.0)
    return cw[y.astype(int)]


def _gpu() -> bool:
    try:
        import torch
        return torch.cuda.is_available()
    except Exception:
        return False


class TemperatureScaler:
    def __init__(self):
        self.T = 1.0

    def fit(self, logp: np.ndarray, y: np.ndarray) -> "TemperatureScaler":
        from scipy.optimize import minimize_scalar
        y = y.astype(int)

        def nll(t):
            z = logp / t
            z = z - z.max(1, keepdims=True)
            lp = z - np.log(np.exp(z).sum(1, keepdims=True))
            return -lp[np.arange(len(y)), y].mean()

        if len(y) >= 30 and len(np.unique(y)) > 1:
            self.T = float(minimize_scalar(nll, bounds=(0.05, 20), method="bounded").x)
        return self

    def transform(self, proba: np.ndarray) -> np.ndarray:
        z = np.log(np.clip(proba, 1e-12, 1)) / self.T
        z = z - z.max(1, keepdims=True)
        e = np.exp(z)
        return e / e.sum(1, keepdims=True)


class Head:
    def __init__(self, kind: str, params: dict | None, task: str = "cls", seed: int = 0, n_jobs: int | None = None):
        self.kind, self.task, self.seed = kind, task, seed
        self.params = {**DEFAULTS.get(kind, {}), **(params or {})}
        self.n_jobs = n_jobs or max(1, (os.cpu_count() or 2) - 1)
        self.curve: dict = {}
        self.model = None
        self.cal = TemperatureScaler()
        self.prior = np.ones(3) / 3
        self.info: dict = {}

    # ------------------------------------------------------------------ fit
    def fit(self, X, y, w, Xv, yv, wv):
        y = np.asarray(y)
        yv = np.asarray(yv)
        if self.task == "cls":
            self.prior = np.bincount(y.astype(int), minlength=3) / max(1, len(y))
            w = np.asarray(w) * _balanced_w(y)
            wv = np.asarray(wv) * _balanced_w(yv) if len(yv) else wv
        getattr(self, f"_fit_{self.kind}")(X, y, w, Xv, yv, wv)
        if self.task == "cls" and len(yv):
            self.cal.fit(np.log(np.clip(self._raw_proba(Xv), 1e-12, 1)), yv)
        return self

    def _fit_lightgbm(self, X, y, w, Xv, yv, wv):
        import lightgbm as lgb
        p = dict(self.params)
        common = dict(n_estimators=3000, subsample_freq=1, random_state=self.seed, n_jobs=self.n_jobs, verbose=-1, max_bin=127, **p)
        if self.task == "cls":
            self.model = lgb.LGBMClassifier(objective="multiclass", num_class=3, **common)
            metric = "multi_logloss"
        else:
            self.model = lgb.LGBMRegressor(objective="huber", alpha=1.0, **common)
            metric = "huber"
        ev = {}
        cbs = [lgb.record_evaluation(ev)]
        es = len(yv) > 0
        if es:
            cbs.append(lgb.early_stopping(100, verbose=False))
        import warnings
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", message=".*eval_set.*deprecated.*")
            self.model.fit(X, y, sample_weight=w, eval_set=[(X, y), (Xv, yv)] if es else [(X, y)],
                           eval_sample_weight=[w, wv] if es else [w], eval_metric=metric, callbacks=cbs)
        keys = list(ev.keys())
        self.curve = {"train": list(ev[keys[0]][metric]) if keys else [],
                      "val": list(ev[keys[1]][metric]) if len(keys) > 1 else [], "metric": metric, "unit": "boosting round"}
        self.info["best_iteration"] = int(getattr(self.model, "best_iteration_", 0) or 0)

    def _fit_catboost(self, X, y, w, Xv, yv, wv):
        from catboost import CatBoostClassifier, CatBoostRegressor, Pool
        p = dict(self.params)
        gpu = _gpu()
        kw = dict(iterations=3000, random_seed=self.seed, verbose=False, allow_writing_files=False,
                  od_type="Iter", od_wait=100, task_type="GPU" if gpu else "CPU", **p)
        if not gpu:
            kw["thread_count"] = self.n_jobs
        if gpu:
            kw.pop("random_strength", None)
        if self.task == "cls":
            self.model = CatBoostClassifier(loss_function="MultiClass", **kw)
        else:
            self.model = CatBoostRegressor(loss_function="Huber:delta=1.0", **kw)
        tr = Pool(X, y, weight=w)
        if self.task == "cls" and len(yv):
            keep = np.isin(yv, np.unique(y))           # CatBoost rejects eval classes unseen in training
            Xv, yv, wv = Xv[keep], yv[keep], np.asarray(wv)[keep]
        va = Pool(Xv, yv, weight=wv) if len(yv) else None
        self.model.fit(tr, eval_set=va, use_best_model=va is not None)
        er = self.model.get_evals_result()
        lk = "MultiClass" if self.task == "cls" else list(er.get("learn", {}).keys())[0] if er.get("learn") else None
        self.curve = {"train": list(er.get("learn", {}).get(lk, [])), "val": list(er.get("validation", {}).get(lk, [])),
                      "metric": lk, "unit": "iteration"}
        self.info["best_iteration"] = int(self.model.get_best_iteration() or 0)

    def _fit_enet(self, X, y, w, Xv, yv, wv):
        from sklearn.impute import SimpleImputer
        from sklearn.linear_model import ElasticNet, LogisticRegression
        from sklearn.pipeline import make_pipeline
        from sklearn.preprocessing import StandardScaler
        p = self.params
        if self.task == "cls":
            est = LogisticRegression(solver="saga", penalty="elasticnet", C=p["C"], l1_ratio=p["l1_ratio"], max_iter=400,
                                     tol=1e-3, random_state=self.seed)
        else:
            est = ElasticNet(alpha=1.0 / max(p["C"], 1e-6) / 100, l1_ratio=p["l1_ratio"], max_iter=2000, random_state=self.seed)
        self.model = make_pipeline(SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True),
                                   StandardScaler(), est)
        self.model.fit(X, y, **{f"{type(est).__name__.lower()}__sample_weight": w})
        last = self.model[-1]
        self.info["n_iter"] = int(np.max(getattr(last, "n_iter_", [0])))
        self.info["converged"] = bool(self.info["n_iter"] < (400 if self.task == "cls" else 2000))
        self.curve = {}

    # ------------------------------------------------------------------ predict
    def _raw_proba(self, X) -> np.ndarray:
        """Always 3 columns (down, sideways, up) even when a training fold lacked a class."""
        P = np.asarray(self.model.predict_proba(X), float)
        cls = getattr(self.model, "classes_", None)
        if cls is None and hasattr(self.model, "steps"):
            cls = getattr(self.model[-1], "classes_", None)
        if not (P.shape[1] == 3 and (cls is None or list(np.asarray(cls).astype(int)) == [0, 1, 2])):
            full = np.full((P.shape[0], 3), 1e-6)
            for j, c in enumerate(np.asarray(cls).astype(int).ravel()):
                full[:, int(c)] = P[:, j]
            P = full / full.sum(1, keepdims=True)
        return prior_correct(P, self.prior) if self.task == "cls" else P

    def predict_proba(self, X) -> np.ndarray:
        return self.cal.transform(self._raw_proba(X))

    def predict(self, X) -> np.ndarray:
        return np.asarray(self.model.predict(X), float).ravel()


def enet_certificate(X, y, w, params: dict, seeds=(0, 1, 2)) -> dict:
    """Convex problem -> a unique global optimum. Refit from several random starts and compare the penalized
    objective; agreement within tolerance certifies the optimizer reached the global minimum."""
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    Xi = StandardScaler().fit_transform(SimpleImputer(strategy="median", keep_empty_features=True).fit_transform(X))
    y = np.asarray(y).astype(int)
    w = np.asarray(w) * _balanced_w(y)
    objs = []
    for s in seeds:
        m = LogisticRegression(solver="saga", penalty="elasticnet", C=params["C"], l1_ratio=params["l1_ratio"],
                               max_iter=3000, tol=1e-6, random_state=s, warm_start=False)
        rng = np.random.default_rng(s)
        perm = rng.permutation(len(y))
        m.fit(Xi[perm], y[perm], sample_weight=w[perm])
        p = np.clip(m.predict_proba(Xi), 1e-12, 1)
        nll = -np.sum(w * np.log(p[np.arange(len(y)), y])) / np.sum(w)
        coef = m.coef_.ravel()
        pen = (params["l1_ratio"] * np.abs(coef).sum() + 0.5 * (1 - params["l1_ratio"]) * (coef ** 2).sum()) / (params["C"] * np.sum(w))
        objs.append(float(nll + pen))
    spread = float(np.max(objs) - np.min(objs))
    return {"objective_by_start": objs, "spread": spread, "certified_global_min": bool(spread < 1e-4),
            "note": "elastic-net logistic loss is convex; equal objectives from different starts = global optimum"}


In [ ]:
%%writefile $REPO/src/secprice/models/ladder.py
"""Track B ablation ladder engine (SPEC.md §9.1-9.4).

Every comparison declares a cohort: common (events eligible for every compared rung) or maximum-available.
Fold-level preprocessing (TF-IDF + SVD, embedding PCA, imputers, scalers, calibration, clip bounds) is fitted on
the training rows of that fold only. Selection uses mean walk-forward macro-F1 ("validation-selected", never a
global optimum).
"""
from __future__ import annotations

import hashlib
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

from ..evaluation.bootstrap import macro_f1_ci, paired_macro_f1_diff
from ..evaluation.metrics import classification_metrics, decide, regression_metrics
from .heads import Head, enet_certificate, space

FAMILY_PREFIX = {
    "price": ("px_", "mkt_"), "metadata": ("meta_",), "accounting": ("acc_", "sue_"),
    "lexical": ("lm_", "main_lm_", "txt_doc"), "change": ("chg_", "rf_new", "rf_removed", "boilerplate_"),
    "novelty": ("novelty_",), "extraction": ("ext_",), "ft": ("ft_",),
}
RUNGS = {
    "L0": [], "L1": ["price"], "L2": ["metadata"], "L3": ["accounting"],
    "L4": ["lexical", "change", "novelty", "tfidf"], "L4_levels": ["lexical", "tfidf"], "L4_change": ["change", "novelty"],
    "L5": ["emb"], "L6": ["price", "metadata", "accounting", "lexical", "change", "novelty", "tfidf"],
    "L6_nochange": ["price", "metadata", "accounting", "lexical", "tfidf"],
    "L7": ["price", "metadata", "accounting", "lexical", "change", "novelty", "tfidf", "emb"],
    "L8": ["price", "metadata", "accounting", "lexical", "change", "novelty", "tfidf", "emb", "extraction", "ft"],
}
REQUIRES = {"price": "has_price", "accounting": "has_accounting", "lexical": "has_text", "change": "has_text",
            "novelty": "has_text", "tfidf": "has_text", "extraction": "has_extraction"}


def _hash_ids(ids) -> str:
    return hashlib.sha1(",".join(sorted(map(str, ids))).encode()).hexdigest()[:12]


class Ladder:
    def __init__(self, cfg: dict, events: pd.DataFrame, feats: pd.DataFrame, targets: pd.DataFrame,
                 splits: pd.DataFrame, texts: pd.Series | None = None, embs: dict | None = None,
                 out_dir: Path | None = None, log=None, provenance: pd.DataFrame | None = None):
        self.cfg, self.log = cfg, log
        self.ev = events.set_index("event_id")
        self.F = feats.set_index("event_id").reindex(self.ev.index)
        self.T = targets
        self.S = splits
        self.texts = texts
        self.embs = embs or {}
        self.out = Path(out_dir) if out_dir else None
        self.cache: dict = {}
        self.curves: list[dict] = []
        self.config_counter = 0
        self.prov = provenance
        self.inner_frac = float(cfg["tuning"]["inner_val_frac"])
        self.min_train = int(cfg["tuning"].get("min_train_rows", 50))
        self.min_test = int(cfg["tuning"].get("min_test_rows", 10))
        self.seed = int(cfg["seed"])
        self.clean_col = f"overlap_flag_h{int(cfg['contam_window_sessions'])}"
        self.avail = self._availability()

    # ------------------------------------------------------------------ data
    def _availability(self) -> pd.DataFrame:
        F = self.F
        a = pd.DataFrame(index=F.index)
        px = [c for c in F.columns if c.startswith("px_ret_20") or c.startswith("px_vol_20")]
        a["has_price"] = F[px].notna().all(axis=1) if px else False
        acc = [c for c in F.columns if c.startswith(FAMILY_PREFIX["accounting"])]
        a["has_accounting"] = F[acc].notna().any(axis=1) if acc else False
        a["has_text"] = F["txt_has_text"].fillna(False).astype(bool) if "txt_has_text" in F else False
        a["has_extraction"] = F[[c for c in F.columns if c.startswith("ext_")]].notna().any(axis=1) if any(
            c.startswith("ext_") for c in F.columns) else False
        for enc, E in self.embs.items():
            a[f"has_emb_{enc}"] = a.index.isin(E.index)
        return a

    def labelled(self, tdef: str, h: int, col: str = "label3") -> pd.DataFrame:
        t = self.T[(self.T.target_def == tdef) & (self.T.horizon == h)][["event_id", col, "weight_uniq"]]
        t = t[t[col].notna()].set_index("event_id")
        t = t[t.index.isin(self.ev.index)]
        out = pd.DataFrame({"y": t[col], "w": t["weight_uniq"].fillna(1.0)})
        out["date"] = pd.to_datetime(self.ev.loc[out.index, "decision_date"])
        return out.sort_values("date")

    def requirement_mask(self, rung: str, enc: str | None, ids) -> np.ndarray:
        m = np.ones(len(ids), bool)
        av = self.avail.loc[ids]
        for fam in RUNGS[rung]:
            if fam == "emb":
                m &= av[f"has_emb_{enc}"].to_numpy() if enc and f"has_emb_{enc}" in av else False
            elif REQUIRES.get(fam):
                m &= av[REQUIRES[fam]].to_numpy()
        return m

    def cohort(self, rungs: list[str], encs: list[str | None], ids) -> np.ndarray:
        m = np.ones(len(ids), bool)
        for r in rungs:
            for e in (encs if "emb" in RUNGS[r] else [None]):
                m &= self.requirement_mask(r, e, ids)
        return m

    def fold_ids(self, scheme: str, fold: int, ids: pd.Index) -> tuple[pd.Index, pd.Index]:
        s = self.S[(self.S.scheme == scheme) & (self.S.fold == fold)].set_index("event_id")["role"]
        s = s.reindex(ids)
        test_role = {"walk_forward": "val", "dev_holdout": "dev", "locked_test": "locked"}.get(scheme, "test")
        return ids[(s == "train").to_numpy()], ids[(s == test_role).to_numpy()]

    def folds(self, scheme: str) -> list[int]:
        return sorted(self.S[self.S.scheme == scheme].fold.unique().tolist())

    # ------------------------------------------------------------------ design matrices
    def _family_cols(self, fam: str) -> list[str]:
        pre = FAMILY_PREFIX.get(fam, ())
        return [c for c in self.F.columns if c.startswith(pre) and not c.startswith("_")]

    def _tfidf(self, tr_ids, key: str):
        ck = ("tfidf", key)
        if ck not in self.cache:
            from sklearn.decomposition import TruncatedSVD
            from sklearn.feature_extraction.text import TfidfVectorizer
            docs = self.texts.reindex(tr_ids).fillna("").tolist()
            nd = len(docs)
            vec = TfidfVectorizer(max_features=20000, min_df=min(5, max(1, nd // 50)), sublinear_tf=True,
                                  dtype=np.float32, token_pattern=r"(?u)\b[a-zA-Z]{3,}\b", stop_words="english")
            M = vec.fit_transform(docs)
            k = int(min(64, max(2, M.shape[1] - 1), max(2, nd - 1)))
            svd = TruncatedSVD(n_components=k, random_state=self.seed).fit(M)
            self.cache[ck] = (vec, svd)
        return self.cache[ck]

    def _pca(self, enc: str, tr_ids, key: str):
        ck = ("pca", enc, key)
        if ck not in self.cache:
            from sklearn.decomposition import PCA
            E = self.embs[enc]
            Xtr = E.reindex(tr_ids).dropna().to_numpy(np.float32)
            k = int(min(int(self.cfg["models"]["pca_dims"]), max(1, Xtr.shape[0] - 1), Xtr.shape[1]))
            self.cache[ck] = PCA(n_components=k, random_state=self.seed).fit(Xtr)
        return self.cache[ck]

    def _cached(self, key, fn):
        """Transformed blocks are reused across trials/heads; the cache is bounded in bytes (runtime.ladder_cache_max_gb,
        oldest blocks evicted first) so memory stays flat at full scale (~180k events)."""
        if key not in self.cache:
            x = fn()
            budget = float((self.cfg.get("runtime") or {}).get("ladder_cache_max_gb", 2.0)) * 1e9
            blocks = [k for k in self.cache if k[0] in ("tfidf_x", "pca_x")]
            used = sum(getattr(self.cache[k], "nbytes", 0) for k in blocks)
            while blocks and used + getattr(x, "nbytes", 0) > budget:
                k = blocks.pop(0)
                used -= getattr(self.cache[k], "nbytes", 0)
                del self.cache[k]
            self.cache[key] = x
        return self.cache[key]

    def matrices(self, rung: str, tr_ids, eval_ids: list, enc: str | None = None):
        key = _hash_ids(tr_ids)
        blocks_tr, blocks_ev, names = [], [[] for _ in eval_ids], []
        for fam in RUNGS[rung]:
            if fam == "tfidf":
                if self.texts is None:
                    continue
                vec, svd = self._tfidf(tr_ids, key)
                f = lambda ids, k=key: self._cached(("tfidf_x", k, _hash_ids(ids)), lambda: svd.transform(  # noqa: E731
                    vec.transform(self.texts.reindex(ids).fillna("").tolist())).astype(np.float32))
                blocks_tr.append(f(tr_ids))
                for j, ids in enumerate(eval_ids):
                    blocks_ev[j].append(f(ids))
                names += [f"tfidf_svd{i}" for i in range(blocks_tr[-1].shape[1])]
            elif fam == "emb":
                pca = self._pca(enc, tr_ids, key)
                E = self.embs[enc]
                f = lambda ids, k=key: self._cached(("pca_x", enc, k, _hash_ids(ids)), lambda: pca.transform(  # noqa: E731
                    E.reindex(ids).fillna(0).to_numpy(np.float32)).astype(np.float32))
                blocks_tr.append(f(tr_ids))
                for j, ids in enumerate(eval_ids):
                    blocks_ev[j].append(f(ids))
                names += [f"emb_{enc}_pc{i}" for i in range(blocks_tr[-1].shape[1])]
            else:
                cols = self._family_cols(fam)
                if not cols:
                    continue
                blocks_tr.append(self.F.loc[tr_ids, cols].to_numpy(np.float32))
                for j, ids in enumerate(eval_ids):
                    blocks_ev[j].append(self.F.loc[ids, cols].to_numpy(np.float32))
                names += cols
        if not blocks_tr:
            return np.zeros((len(tr_ids), 0), np.float32), [np.zeros((len(i), 0), np.float32) for i in eval_ids], names
        return np.hstack(blocks_tr), [np.hstack(b) for b in blocks_ev], names

    # ------------------------------------------------------------------ one configuration over folds
    def run(self, rung: str, head: str, params: dict | None, lab: pd.DataFrame, cohort_ids: pd.Index, scheme: str,
            task: str = "cls", enc: str | None = None, keep: bool = True, folds: list[int] | None = None,
            trial=None, curves: bool = False, max_train: int | None = None) -> dict:
        self.config_counter += 1
        out = {"fold_metrics": [], "preds": [], "fold_scores": [], "train_scores": []}
        ids_all = cohort_ids
        flist = folds if folds is not None else self.folds(scheme)
        for step, f in enumerate(flist):
            tr, te = self.fold_ids(scheme, f, ids_all)
            if len(tr) < self.min_train or len(te) < self.min_test or (task == "cls" and lab.loc[tr, "y"].nunique() < 2):
                out["fold_metrics"].append({"fold": f, "skipped": f"insufficient rows (train {len(tr)}, test {len(te)})"})
                continue
            trs = lab.loc[tr].sort_values("date")
            if max_train and len(trs) > max_train:
                # tuning screen only: time-stratified subsample (every k-th row in date order keeps all years)
                trs = trs.iloc[np.linspace(0, len(trs) - 1, int(max_train)).round().astype(int)]
            cut = int(len(trs) * (1 - self.inner_frac))
            itr, iva = trs.index[:cut], trs.index[cut:]
            ytr, yva, yte = lab.loc[itr, "y"].to_numpy(), lab.loc[iva, "y"].to_numpy(), lab.loc[te, "y"].to_numpy()
            wtr, wva = lab.loc[itr, "w"].to_numpy(), lab.loc[iva, "w"].to_numpy()
            if task == "reg":
                lo, hi = np.nanquantile(ytr, [0.005, 0.995])          # clip bounds from training rows only
                ytr_c, yva_c = np.clip(ytr, lo, hi), np.clip(yva, lo, hi)
            if rung == "L0":
                if task == "cls":
                    prior = np.bincount(ytr.astype(int), minlength=3) / len(ytr)
                    proba = np.tile(prior, (len(te), 1))
                    yhat = np.full(len(te), int(np.argmax(prior)))
                    m = classification_metrics(yte, proba, prior, yhat)
                    tr_m = m
                else:
                    yhat = np.full(len(te), float(np.mean(ytr)))
                    m = regression_metrics(yte, yhat, kind="return")
                    tr_m = m
            else:
                Xtr, (Xva, Xte), names = self.matrices(rung, itr, [iva, te], enc)
                if Xtr.shape[1] == 0:
                    out["fold_metrics"].append({"fold": f, "skipped": "no features"})
                    continue
                hd = Head(head, params, task=task, seed=self.seed)
                if task == "cls":
                    hd.fit(Xtr, ytr, wtr, Xva, yva, wva)
                    proba = hd.predict_proba(Xte)
                    yhat = decide(proba, hd.prior)
                    m = classification_metrics(yte, proba, hd.prior, yhat)
                    ptr = hd.predict_proba(Xtr)
                    tr_m = classification_metrics(ytr, ptr, hd.prior)
                else:
                    hd.fit(Xtr, ytr_c, wtr, Xva, yva_c, wva)
                    yhat = hd.predict(Xte)
                    m = regression_metrics(yte, yhat, kind="return")
                    tr_m = regression_metrics(ytr, hd.predict(Xtr), kind="return")
                if curves and hd.curve:
                    self.curves.append({"rung": rung, "head": head, "enc": enc, "fold": f, "scheme": scheme, **hd.curve})
                m["head_info"] = hd.info
            m["fold"] = f
            m["n_train"] = int(len(itr))
            out["fold_metrics"].append(m)
            key = "macro_f1" if task == "cls" else "r2"
            out["fold_scores"].append(m.get(key))
            out["train_scores"].append(tr_m.get(key))
            if keep:
                p = pd.DataFrame({"event_id": te, "fold": f, "y": yte, "date": lab.loc[te, "date"].values})
                if task == "cls":
                    p[["p_down", "p_side", "p_up"]] = proba
                    p["yhat"] = yhat
                else:
                    p["yhat_reg"] = yhat
                out["preds"].append(p)
            if trial is not None:
                sc = [s for s in out["fold_scores"] if s is not None]
                trial.report(float(np.mean(sc)) if sc else 0.0, step)
                if trial.should_prune():
                    import optuna
                    raise optuna.TrialPruned()
        sc = [s for s in out["fold_scores"] if s is not None]
        out["mean_score"] = float(np.mean(sc)) if sc else None
        out["std_score"] = float(np.std(sc)) if len(sc) > 1 else None
        trs = [s for s in out["train_scores"] if s is not None]
        out["mean_train_score"] = float(np.mean(trs)) if trs else None
        out["preds"] = pd.concat(out["preds"], ignore_index=True) if out["preds"] else pd.DataFrame()
        return out

    # ------------------------------------------------------------------ tuning
    def tune(self, rung, head, lab, cohort_ids, n_trials, timeout, storage_path: Path, study_name: str,
             task="cls", enc=None) -> tuple[dict, pd.DataFrame]:
        import optuna
        optuna.logging.set_verbosity(optuna.logging.WARNING)
        storage = f"sqlite:///{storage_path.as_posix()}"
        study = optuna.create_study(direction="maximize", study_name=study_name, storage=storage, load_if_exists=True,
                                    sampler=optuna.samplers.TPESampler(seed=self.seed),
                                    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=1))
        done = len([t for t in study.trials if t.state.is_finished()])
        if done == 0:
            from .heads import DEFAULTS
            study.enqueue_trial(DEFAULTS[head])

        tc = self.cfg["tuning"]
        last = int(tc.get("tune_last_folds") or 0)
        fl = self.folds("walk_forward")
        tune_folds = fl[-last:] if last else fl
        cap = int(tc.get("tune_max_train_rows") or 0) or None

        def obj(trial):
            params = space(trial, head)
            r = self.run(rung, head, params, lab, cohort_ids, "walk_forward", task, enc, keep=False, trial=trial,
                         folds=tune_folds, max_train=cap)
            return r["mean_score"] if r["mean_score"] is not None else 0.0

        remaining = max(0, int(n_trials) - done)
        if remaining:
            study.optimize(obj, n_trials=remaining, timeout=timeout, gc_after_trial=True)
        trials = study.trials_dataframe(attrs=("number", "value", "params", "state"))
        best = study.best_trial.params if study.best_trial is not None else {}
        return best, trials


def summarize(res: dict, lab: pd.DataFrame, n_boot: int, seed: int, task: str = "cls") -> dict:
    """Pooled-OOF metrics with month-block CI + mean fold score."""
    p = res["preds"]
    if p.empty:
        return {"mean_fold": res.get("mean_score"), "std_fold": res.get("std_score"), "mean_train": res.get("mean_train_score"),
                "pooled": {}, "ci": {}, "n": 0}
    if task == "cls":
        prior = np.bincount(p.y.astype(int), minlength=3) / len(p)
        pooled = classification_metrics(p.y.values, p[["p_down", "p_side", "p_up"]].values, prior, p.yhat.values)
        ci = macro_f1_ci(p.y.values, p.yhat.values, p.date.values, n_boot, seed)
    else:
        pooled = regression_metrics(p.y.values, p.yhat_reg.values, dates=p.date, kind="return")
        ci = {}
    return {"mean_fold": res.get("mean_score"), "std_fold": res.get("std_score"),
            "mean_train": res.get("mean_train_score"), "pooled": pooled, "ci": ci, "n": int(len(p))}


def fit_verdict(mean_val, mean_train, std, majority, audit: dict) -> str:
    if mean_val is None:
        return "not_evaluable"
    v = []
    if mean_train is not None and mean_train - mean_val > float(audit["overfit_gap"]):
        v.append("overfit")
    if majority is not None and mean_val - majority <= float(audit["underfit_margin"]):
        v.append("underfit (may mean weak signal)")
    if std is not None and std > float(audit["unstable_std"]):
        v.append("unstable")
    return ", ".join(v) or "good fit"


In [ ]:
%%writefile $REPO/src/secprice/parse/__init__.py
"""secprice.parse"""


In [ ]:
%%writefile $REPO/src/secprice/parse/anonymize.py
"""Firm anonymization: mask the firm's own names and tickers (firm-memorization control, SPEC.md §6.3 #4).
Regex masking runs on CPU for every section; NER-based PER/ORG masking (dslim/bert-base-NER, never money) is
applied on GPU before embedding when enabled."""
from __future__ import annotations

import re

_SUFFIX = re.compile(r"\b(incorporated|inc|corporation|corp|company|co|ltd|limited|plc|llc|lp|l\.p|holdings?|group|"
                     r"the|n\.?v|s\.?a|ag|trust|bancorp(oration)?|/[a-z]{2}/)\b\.?", re.I)
_TICKER_STOP = {"A", "I", "IT", "ON", "ALL", "ARE", "AN", "AM", "BE", "BY", "DO", "GO", "HAS", "HE", "IN", "IS", "ME",
                "NO", "OF", "OR", "SO", "TO", "UP", "US", "WE", "AT", "AS", "CEO", "CFO", "GAAP", "SEC", "USA", "EPS",
                "OUT", "NEW", "NOW", "ONE", "KEY", "WELL", "LOW", "HIGH", "BIG", "CAN", "FAST", "SEE", "TV", "PM", "AI",
                "RE", "EX", "IR", "PR", "Q", "K", "L", "R", "V", "X", "Y", "C", "D", "F", "T", "M", "O", "J", "S"}


def name_variants(names: list[str]) -> list[str]:
    out = set()
    for n in names:
        if not n:
            continue
        n = re.sub(r"\s+", " ", str(n)).strip()
        core = re.sub(r"\s+", " ", _SUFFIX.sub(" ", n.replace(",", " "))).strip(" .,&-")
        for v in (n, core):
            v = v.strip(" .,")
            if len(v) >= 4:
                out.add(v)
    return sorted(out, key=len, reverse=True)


def build_masker(names: list[str], tickers: list[str]):
    variants = name_variants(names)
    pats = []
    for v in variants:
        toks = v.split()
        esc = r"\s+".join(re.escape(t) for t in toks)
        if len(toks) >= 2:
            pats.append(re.compile(rf"(?<![\w]){esc}(?![\w])", re.I))
        else:
            # single-word names: exact Title/UPPER case only (avoid masking common words like 'target')
            pats.append(re.compile(rf"(?<![\w])(?:{re.escape(v.title())}|{re.escape(v.upper())})(?![\w])"))
    tick = [t.upper() for t in tickers if t and t.upper() not in _TICKER_STOP and len(t) >= 2]
    tpat = re.compile(r"(?<![\w$])(?:" + "|".join(re.escape(t) for t in sorted(tick, key=len, reverse=True)) + r")(?![\w])") if tick else None

    def mask(text: str) -> tuple[str, dict]:
        counts = {"firm": 0, "ticker": 0}
        for p in pats:
            text, n = p.subn("[FIRM]", text)
            counts["firm"] += n
        if tpat is not None:
            text, n = tpat.subn("[TICKER]", text)
            counts["ticker"] += n
        return text, counts

    return mask


In [ ]:
%%writefile $REPO/src/secprice/parse/eps_release.py
"""Earnings-release exhibit (EX-99.x) checks and release-EPS parsing (SPEC.md §5.2 verified releases, §5.6).

Diluted GAAP EPS is preferred; basic is the flagged fallback. Rows mentioning non-GAAP / adjusted measures are
skipped. The parsed value is compared with the later XBRL value only as a QA metric (never as an input).
"""
from __future__ import annotations

import re

import pandas as pd

_MONTHS = "january|february|march|april|may|june|july|august|september|october|november|december"
_DATE = re.compile(rf"\b({_MONTHS})\s+(\d{{1,2}}),?\s+(\d{{4}})", re.I)
_PERIOD = re.compile(rf"(three|3|thirteen|twelve|twelve-week|sixteen|quarter|fourth quarter|first quarter|second quarter|"
                     rf"third quarter|fiscal quarter)[\w\s\-]{{0,30}}(ended)\s+({_MONTHS})\s+(\d{{1,2}}),?\s+(\d{{4}})", re.I)
_NUM = re.compile(r"(\(\s{0,3}\$?\s{0,3}(?:\d{1,3}(?:,\d{3})*)?\.\d{1,3}\s{0,3}\)?|-?\$?\s{0,3}(?:\d{1,3}(?:,\d{3})*)?\.\d{1,3})")
_SHARES = re.compile(r"shares|weighted|average\s+number|outstanding", re.I)
_ANNUAL = re.compile(r"full[- ]year|for\s+the\s+(fiscal\s+)?year|twelve\s+months|year\s+ended|annual|outlook|guidance|"
                     r"expects?|forecast|range\s+of|fiscal\s+20\d\d\s+(earnings|eps)", re.I)
_CONT = re.compile(r"continuing\s+operations", re.I)
_DISC = re.compile(r"discontinued", re.I)
_RANGE = re.compile(r"\d\.\d{2}\s*(?:-|\u2013|\u2014|to)\s*\$?\s*\d*\.\d{2}|in\s+the\s+range|range\s+of", re.I)
_STATEMENT = re.compile(r"statements?\s+of\s+(consolidated\s+)?(income|operations|earnings)|income\s+statements?|"
                        r"consolidated\s+(results|statements?)|condensed\s+consolidated", re.I)
_IMPACT = re.compile(r"\b(impact\w*|charges?|benefit\w*|includ\w*|effects?|reduc\w*|related\s+to|costs?|gain\s+on|"
                     r"loss\s+on|items?|after[- ]tax|pre-?tax|dilution\s+from|accretive|dilutive)\b", re.I)
_NONGAAP = re.compile(r"non-?gaap|adjusted|excluding|excluded|pro\s*forma|core\b|operating\s+(eps|earnings per)|before\s+special|"
                      r"comparable|underlying|cash\s+eps|normalized|street|ongoing|specified\s+items", re.I)
_EPS_CTX = re.compile(r"per\s+(common\s+)?share|earnings\s+per|\beps\b|per\s+diluted", re.I)
_DIL = re.compile(r"dilut", re.I)
_V = r"(\(?-?\$?\s*\d*\.\d{2}\)?)"
# Ordered prose patterns: (regex, confidence)
_PROSE = [
    (re.compile(r"diluted\s+(?:earnings|eps)[^.$]{0,40}?(?:was|were|of)\s+\$\s*(\(?\d*\.\d{2}\)?)\s+GAAP", re.I), "medium_gaap_marked"),
    (re.compile(r"\$\s*(\(?\d*\.\d{2}\)?)\s+(?:per|a)\s+diluted\s+(?:common\s+)?share", re.I), "medium"),
    (re.compile(r"(?:diluted\s+(?:earnings|net\s+income|income|eps|net\s+loss|loss)[^.$]{0,50}?|\(assuming\s+dilution\)\s*|"
                r"assuming\s+dilution[^.$]{0,30}?)(?:of|was|were|totaled|to|:)\s*" + _V, re.I), "medium"),
    (re.compile(r"diluted\s+earnings\s+per\s+share[^.]{0,80}?were\s+\$[\d.,]+\s+(?:million|billion)\s+and\s+" + _V, re.I), "medium"),
    (re.compile(r"(?:net\s+income|net\s+earnings|earnings|net\s+profit)[^.]{0,80}?,?\s+or\s+\$\s*(\(?\d*\.\d{2}\)?)\s+per\s+(?:common\s+)?share", re.I),
     "low"),
    # undifferentiated EPS ("earnings per share were $0.67"; older releases rarely say "diluted"): own class, validated vs XBRL
    (re.compile(r"(?:net\s+)?(?:earnings|income|loss)\s+per\s+(?:common\s+)?(?:diluted\s+)?share[^.$]{0,40}?(?:of|was|were|totaled|to|:)\s*"
                + _V, re.I), "medium_undiff"),
    (re.compile(r"(?:net\s+income|net\s+earnings|earnings|net\s+profit|net\s+loss|loss)\s+(?:of|were|was)\s+\$\s*(\(?\d*\.\d{2}\)?)\s+"
                r"per\s+(?:common\s+)?share", re.I), "medium_undiff"),
]
# plain-text table rows (no cell separators, common before ~2008): "Diluted $ 0.08 $ (0.05)" -> "Diluted | 0.08 | (0.05)"
_TXT_ROW = re.compile(r"^\s*(?P<label>(?:[A-Za-z][A-Za-z ,'\u2019()/&\-]{0,70}?[\s\-\u2013\u2014:(])?(?:diluted|dilution|basic)\b"
                      r"[A-Za-z ,'\u2019()/&\-]{0,30}?)"
                      r"\s*[:\-\u2013\u2014]?\s*(?P<vals>\$?\s*\(?-?\d*\.\d{2}\)?(?:\s+(?:\$\s*)?\(?-?(?:[\d,]+(?:\.\d+)?|\.\d+)\)?)*)\s*$",
                      re.I)


def _sentence_start(text: str, pos: int) -> int:
    """Start of the sentence containing pos: the last sentence end (., ! or ? followed by whitespace, never a decimal point)
    or blank line within 600 characters."""
    lo = max(0, pos - 600)
    last = lo
    for m in re.finditer(r"(?<!\d)[.!?](?=\s)|\n\s*\n", text[lo:pos]):
        last = lo + m.end()
    return last


def _loss_governs(prefix: str) -> bool:
    """True when the nearest results word before the number is 'loss' (not income / earnings / profit)."""
    lo = [m.end() for m in re.finditer(r"\bloss(es)?\b", prefix, re.I)]
    gi = [m.end() for m in re.finditer(r"\b(income|earnings|profit)\b", prefix, re.I)]
    return bool(lo) and (not gi or lo[-1] > gi[-1])


def _text_table_row(line: str) -> str | None:
    if "|" in line:
        return None
    m = _TXT_ROW.match(line)
    if not m:
        return None
    label, vals_txt = m.group("label").rstrip(), m.group("vals")
    if label.endswith("("):                      # "Diluted (0.04)": the parenthesis belongs to the value (a negative)
        label, vals_txt = label[:-1].rstrip(), "(" + vals_txt.lstrip()
    vals = re.findall(r"\(?-?[\d,]*\.?\d+\)?", vals_txt)
    return label + " | " + " | ".join(vals) if vals else None
USABLE_CONFIDENCE = {"high", "medium", "medium_table", "medium_gaap_marked", "medium_undiff"}   # SUE_release uses only these; 'low' stays for QA


def results_release_score(text: str) -> int:
    t = (text or "")[:80000].lower()
    s = 0
    s += bool(re.search(r"net income|net earnings|net loss|earnings per (diluted )?share|per diluted share|diluted eps", t))
    s += bool(re.search(r"(first|second|third|fourth) quarter|three months ended|quarter ended|fiscal (year|quarter)|full[- ]year", t))
    s += bool(re.search(r"(report|reports|reported|announce|announces|announced)\b.{0,80}(results|earnings)|financial results|results of operations", t))
    s += bool(re.search(r"\brevenues?\b|net sales|total sales", t))
    return s


def is_results_release(text: str) -> bool:
    return results_release_score(text) >= 3


def _to_float(s: str) -> float | None:
    s = s.strip()
    neg = s.startswith("(") or s.startswith("-")
    v = re.sub(r"[^\d.]", "", s)
    try:
        x = float(v)
    except ValueError:
        return None
    return -x if neg else x


def _row_value(line: str) -> float | None:
    """First EPS-like value after the label cell (keeps '(0.15)' negatives intact)."""
    cut = line.find("|")
    if cut < 0:
        d = re.search(r"\d", line)
        cut = (d.start() - 1 if d and d.start() > 0 and line[d.start() - 1] in "($-" else (d.start() if d else 0))
    nums = [_to_float(m.group(0)) for m in _NUM.finditer(line[max(0, cut):])]
    nums = [x for x in nums if x is not None and abs(x) < 200]
    return nums[0] if nums else None


def parse_release_eps(text: str) -> dict:
    """{'eps_diluted', 'eps_basic', 'parse_confidence', 'source_line', 'period_end', 'period_source'}"""
    out = {"eps_diluted": None, "eps_basic": None, "parse_confidence": "none", "source_line": None,
           "period_end": None, "period_source": None}
    if not text:
        return out
    raw_lines = text.split("\n")
    lines = []
    i = 0
    while i < len(raw_lines):
        cur = raw_lines[i]
        nxt_i = i + 1
        while nxt_i < len(raw_lines) and not raw_lines[nxt_i].strip():
            nxt_i += 1
        nxt = raw_lines[nxt_i] if nxt_i < len(raw_lines) else ""
        if (cur.strip() and not re.search(r"\d", cur) and len(cur) < 80 and re.search(r"dilut|basic", cur, re.I)
                and re.match(r"^\s*[\$\(\-\d.|\s,)]+$", nxt) and re.search(r"\d", nxt)):
            lines.append(cur.strip() + " | " + nxt.strip())
            i = nxt_i + 1
            continue
        lines.append(_text_table_row(cur) or cur)
        i += 1
    best = None
    # 1) table rows: every candidate is scored; the best GAAP total-EPS row under an income-statement heading wins
    cands = []
    for i, line in enumerate(lines[:6000]):
        if "|" not in line:
            continue
        low = line.lower()
        is_dil, is_bas = bool(_DIL.search(low)), "basic" in low
        label = line.split("|")[0]
        if not (is_dil or is_bas) or _NONGAAP.search(line) or _SHARES.search(label) or _DISC.search(label):
            continue
        if _RANGE.search(line):
            continue                                   # guidance ranges ("$6.50 - $6.90", "in the range of ...")
        ctx = " ".join(lines[max(0, i - 4):i + 1])
        if not _EPS_CTX.search(ctx) or _NONGAAP.search(" ".join(lines[max(0, i - 2):i])):
            continue
        above = " ".join(lines[max(0, i - 8):i])
        if _ANNUAL.search(above) and not re.search(r"quarter|three\s+months|13\s+weeks", above, re.I):
            continue                                   # an annual / outlook table, not the quarter
        win = lines[max(0, i - 60):i]
        yrs = {x.strip().strip("|").strip() for x in win if re.fullmatch(r"\s*\|?\s*(?:19[89]\d|20[0-3]\d)\s*\|?\s*", x)}
        yrs |= set(re.findall(r"(?<!\d)(?:19[89]\d|20[0-3]\d)(?!\d)", " ".join(lines[max(0, i - 14):i])))
        if len(yrs) >= 3 and not re.search(r"quarter|three\s+months|13\s+weeks", " ".join(win[-30:]), re.I):
            continue                                   # a multi-year summary (selected financial data), not the quarter
        v = _row_value(line)
        if v is None:
            continue
        kind = "diluted" if is_dil else "basic"
        conf = "high" if _EPS_CTX.search(line) else "medium_table"
        score = 0
        if _CONT.search(label):
            conf, score = "continuing_ops", -2          # differs from XBRL EarningsPerShareDiluted (total): QA only
        if _STATEMENT.search(" ".join(lines[max(0, i - 60):i])):
            score += 2                                  # inside the income statement
        if re.search(r"quarter|three\s+months|13\s+weeks", " ".join(lines[max(0, i - 30):i]), re.I):
            score += 1
        cands.append((score, -i, kind, v, conf, line[:200]))
    dil = sorted([c for c in cands if c[2] == "diluted"], reverse=True)
    bas = sorted([c for c in cands if c[2] == "basic"], reverse=True)
    if dil:
        out["eps_diluted"] = dil[0][3]
        best = ("diluted", dil[0][3], dil[0][4], dil[0][5])
    if bas:
        out["eps_basic"] = bas[0][3]
        best = best or ("basic", bas[0][3], bas[0][4], bas[0][5])
    # 2) prose, in order of preference, skipping sentences about non-GAAP measures
    if out["eps_diluted"] is None:
        head = text[:60000]
        for rx, conf in _PROSE:
            hit = None
            for m in rx.finditer(head):
                s0 = _sentence_start(head, m.start())
                sentence = head[s0:m.end()]
                if _ANNUAL.search(sentence) and not re.search(r"quarter", sentence, re.I):
                    continue
                if _RANGE.search(sentence) or (_IMPACT.search(sentence) and conf != "medium_gaap_marked"):
                    continue                           # guidance ranges / impact-of-an-item sentences
                if conf == "medium_gaap_marked" or not _NONGAAP.search(sentence):
                    hit = m
                    break
            if hit:
                v = _to_float(hit.group(hit.lastindex))
                if v is not None and v > 0 and _loss_governs(head[_sentence_start(head, hit.start()):hit.start(hit.lastindex)]):
                    v = -v                             # "net loss of $25 million, or $0.80 per diluted share" -> -0.80
                out["eps_diluted"] = v
                best = ("diluted", v, conf, hit.group(0)[:200])
                break
    if best:
        out["parse_confidence"] = best[2] if best[0] == "diluted" else "basic_fallback"
        out["source_line"] = best[3]
    # current period = the latest 'quarter ended <date>' phrase near the top of the release
    dates = []
    for pm in _PERIOD.finditer(text[:25000]):
        try:
            dates.append(pd.Timestamp(f"{pm.group(3)} {pm.group(4)}, {pm.group(5)}"))
        except ValueError:
            pass
    if dates:
        out["period_end"] = max(dates).date().isoformat()
        out["period_source"] = "text_period_phrase_latest"
    return out


def infer_quarter_end(filing_date, fiscal_year_end: str | None) -> tuple[str | None, str]:
    """Most recent fiscal quarter end at least 7 days before the filing date (flagged as inferred)."""
    fd = pd.Timestamp(filing_date)
    try:
        mm = int(str(fiscal_year_end)[:2]) if fiscal_year_end else 12
        dd = int(str(fiscal_year_end)[2:4]) if fiscal_year_end and len(str(fiscal_year_end)) >= 4 else 28
        if not 1 <= mm <= 12:
            mm = 12
        if dd <= 7:                       # 52/53-week year ending early in a month (e.g. '0103') = previous month
            mm = 12 if mm == 1 else mm - 1
    except ValueError:
        mm = 12
    q_months = sorted({(mm - 1 - 3 * q) % 12 + 1 for q in range(4)})
    ends = [pd.Timestamp(year=y, month=m, day=1) + pd.offsets.MonthEnd(0)
            for y in (fd.year - 1, fd.year) for m in q_months]
    ends = sorted(e for e in ends if e <= fd - pd.Timedelta(days=7))
    return (ends[-1].date().isoformat(), "inferred_fiscal_calendar") if ends else (None, "none")


def flag_suspect_annual(rel: pd.DataFrame, fye_by_cik: dict | None = None, ratio: float = 2.0, min_prior: int = 2) -> pd.DataFrame:
    """Fiscal Q4 releases quote both quarter and full-year EPS; a parser can pick the full-year number. Point-in-time check:
    a parsed EPS whose magnitude exceeds `ratio` x the largest quarterly EPS the firm released earlier (>= `min_prior` earlier
    values, public time strictly before) is marked 'suspect_annual' (not in USABLE_CONFIDENCE: QA only, never a SUE input).
    Applied only to releases whose period ends within 45 days of the fiscal year end (when that is known).
    Conservative: a genuine jump above `ratio` x the prior maximum is also withheld (SUE unavailable, never wrong)."""
    out = rel.copy()
    if out.empty or "eps_diluted" not in out:
        return out
    out["suspect_annual"] = False
    order = out.sort_values("public_available_time", kind="stable").index
    seen: dict = {}
    for i in order:
        r = out.loc[i]
        v = r.eps_diluted
        if not (bool(r.is_results_release) and pd.notna(v) and r.parse_confidence in USABLE_CONFIDENCE):
            continue
        prior = seen.setdefault(r.cik, [])
        q4 = True
        fye = (fye_by_cik or {}).get(r.cik)
        if fye and pd.notna(r.period_end):
            fm = int(str(fye)[:2]) if str(fye)[:2].isdigit() else None
            if fm:
                pe = pd.Timestamp(r.period_end)
                # month-ends built per year (a February fiscal year end is the 28th or the 29th)
                fes = [pd.Timestamp(year=y, month=fm, day=1) + pd.offsets.MonthEnd(0) for y in (pe.year - 1, pe.year, pe.year + 1)]
                q4 = min(abs((pe - fe).days) for fe in fes) <= 45
        if q4 and len(prior) >= min_prior and abs(v) > ratio * max(abs(p) for p in prior[-4:]):
            out.loc[i, "suspect_annual"] = True
            out.loc[i, "parse_confidence"] = "suspect_annual"
            continue                                   # not added to the firm's quarterly history
        prior.append(float(v))
    return out


In [ ]:
%%writefile $REPO/src/secprice/parse/html.py
"""HTML / SGML-text -> normalized plain text. Tables are flattened to 'cell | cell' rows so numeric parsers
(release EPS) can read them; narrative consumers drop numeric-heavy lines later (text.drop_numeric_lines)."""
from __future__ import annotations

import html as _html
import re

_BLOCK = {"p", "div", "br", "tr", "li", "ul", "ol", "h1", "h2", "h3", "h4", "h5", "h6", "table", "section",
          "article", "center", "blockquote", "pre", "hr", "dt", "dd", "title", "page"}
_CELL = {"td", "th"}
_HTML_HINT = re.compile(r"<(html|body|div|p|table|font|br|span)\b", re.I)
_SGML_JUNK = re.compile(r"</?(PAGE|S|C|FN|F\d|R\d|CAPTION|TABLE)>", re.I)
_WS = re.compile(r"[ \t\f\v    ​]+")
_MULTI_NL = re.compile(r"\n{3,}")
_PAGE_LINE = re.compile(r"^\s{0,3}(page\s{0,3})?[-–]?\s{0,3}\d{1,3}\s{0,3}[-–]?\s{0,3}$|^\s{0,3}table of contents\s{0,3}$", re.I)


def decode(raw: bytes | str) -> str:
    if isinstance(raw, str):
        return raw
    for enc in ("utf-8", "cp1252", "latin-1"):
        try:
            return raw.decode(enc)
        except UnicodeDecodeError:
            continue
    return raw.decode("utf-8", errors="replace")


def is_html(text: str) -> bool:
    return bool(_HTML_HINT.search(text[:20000]))


def _in_cell(el) -> bool:
    for a in el.iterancestors():
        t = a.tag.split("}")[-1].lower() if isinstance(a.tag, str) else ""
        if t in ("td", "th"):
            return True
        if t in ("tr", "table", "body"):
            return False
    return False


def html_to_text(text: str) -> tuple[str, int]:
    """Returns (text, n_tables)."""
    import lxml.html
    from lxml import etree
    try:
        doc = lxml.html.document_fromstring(text.encode("utf-8", errors="replace"))
    except (etree.ParserError, ValueError):
        return plain_to_text(re.sub(r"<[^>]+>", " ", text)), 0
    for bad in doc.xpath("//script|//style|//head|//*[local-name()='header' and contains(name(), 'ix:')]"):
        bad.drop_tree()
    for hid in doc.xpath("//*[@style]"):
        st = (hid.get("style") or "").replace(" ", "").lower()
        if "display:none" in st:
            hid.drop_tree()
    n_tables = 0
    for el in doc.iter():
        tag = el.tag if isinstance(el.tag, str) else ""
        tag = tag.split("}")[-1].lower()
        if tag == "table":
            n_tables += 1
        if tag in _BLOCK and tag not in ("tr", "table") and _in_cell(el):
            el.tail = " " + (el.tail or "")          # a <p>/<div>/<br> inside a cell must not split the table row
            continue
        if tag in _BLOCK:
            el.tail = "\n" + (el.tail or "")
            if tag != "br":
                el.text = "\n" + (el.text or "")
        elif tag in _CELL:
            el.tail = " | " + (el.tail or "")
    out = doc.text_content()
    return normalize(out), n_tables


def plain_to_text(text: str) -> str:
    return normalize(_SGML_JUNK.sub(" ", text))


def _c1_map() -> dict:
    m = {}
    for c in range(0x80, 0xA0):
        try:
            m[c] = bytes([c]).decode("cp1252")
        except UnicodeDecodeError:
            m[c] = " "
    return m


_C1 = _c1_map()


_MULTI_PIPES = re.compile(r"\s*\|(?:\s*\|)+\s*")


def normalize(text: str) -> str:
    text = _html.unescape(text).translate(_C1)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    lines = []
    for line in text.split("\n"):
        line = _WS.sub(" ", line).strip()
        # unambiguous forms (each run of whitespace belongs to exactly one group): the older (\s*\|\s*)+ backtracked
        # exponentially on wide table rows with hundreds of empty cells (found on 2000s insurance statistical supplements)
        line = line.strip(" |")                 # leading / trailing empty cells (the line is already whitespace-collapsed)
        line = _MULTI_PIPES.sub(" | ", line)
        if not line or _PAGE_LINE.match(line):
            lines.append("")
            continue
        lines.append(line)
    out = "\n".join(lines)
    return _MULTI_NL.sub("\n\n", out).strip()


def to_text(raw: bytes | str) -> tuple[str, int, str]:
    """(text, n_tables, method)."""
    t = decode(raw)
    if is_html(t):
        txt, n = html_to_text(t)
        return txt, n, "html"
    return plain_to_text(t), t.upper().count("<TABLE>"), "plain"


_NUMTOK = re.compile(r"^[\(\$\-–—]*[\d,.]+%?\)?$")


def drop_numeric_lines(text: str, max_numeric_share: float = 0.5) -> str:
    """Narrative view: drop table rows / lines dominated by numbers."""
    keep = []
    for line in text.split("\n"):
        toks = line.replace("|", " ").split()
        if not toks:
            keep.append("")
            continue
        num = sum(1 for t in toks if _NUMTOK.match(t))
        if "|" in line and num / len(toks) > 0.3:
            continue
        if num / len(toks) > max_numeric_share:
            continue
        keep.append(line)
    return _MULTI_NL.sub("\n\n", "\n".join(keep)).strip()


In [ ]:
%%writefile $REPO/src/secprice/parse/sections.py
"""Section extraction for 10-K / 10-Q and 8-K item text (SPEC.md §5.2, §16 sections.parquet).

Headings are found by item number + title keywords; among candidate starts (table-of-contents lines repeat the
headings) the one with the longest span to the next different heading wins. Sections incorporated by reference
are flagged so the caller can fall back to Exhibit 13 (annual report) for MD&A.
"""
from __future__ import annotations

import re
from dataclasses import dataclass

_ITEM = re.compile(
    r"^\s*(?:part\s+(?P<part>i{1,3}|iv|1|2)\s*[,.\-–—:]*\s*)?item[\s\xa0]*(?P<num>\d{1,2})\s*(?P<let>\(?[a-c]\)?)?"
    r"(?![\d.]\d)\s*[.:\-–—]?\s*(?P<title>.{0,160})$", re.I)
_PART = re.compile(r"^\s*part\s+(i{1,3}|iv|1|2)\b\s*[.\-–—:]?\s*(.{0,80})$", re.I)
_BYREF = re.compile(r"incorporated\s+(herein\s+)?by\s+reference|is\s+included\s+in\s+(the\s+)?(registrant'?s\s+)?annual\s+report|"
                    r"set\s+forth\s+(under|in)\s+.{0,80}annual\s+report|appears?\s+(under|in)\s+.{0,80}annual\s+report|"
                    r"reference\s+is\s+made\s+to|see\s+.{0,80}(section|caption)\s+(entitled|captioned)|"
                    r"contained\s+in\s+.{0,80}(annual\s+report|financial\s+section)", re.I)

# (form family, section id) -> (item key, title regex, part requirement)
TARGETS = {
    "10-K": {
        "10K_1": ("1", r"business", None),
        "10K_1A": ("1A", r"risk\s+factors", None),
        "10K_7": ("7", r"management.{0,4}s?\s+discussion|discussion\s+and\s+analysis", None),
        "10K_7A": ("7A", r"quantitative\s+and\s+qualitative|market\s+risk", None),
    },
    "10-Q": {
        "10Q_MDA": ("2", r"management.{0,4}s?\s+discussion|discussion\s+and\s+analysis", "I"),
        "10Q_MR": ("3", r"quantitative\s+and\s+qualitative|market\s+risk", "I"),
        "10Q_RF": ("1A", r"risk\s+factors", "II"),
    },
}
TITLE_ONLY = {
    "10K_7": r"^\s*management.{0,4}s?\s+discussion\s+and\s+analysis\s+of\s+(financial\s+condition|results)",
    "10Q_MDA": r"^\s*management.{0,4}s?\s+discussion\s+and\s+analysis\s+of\s+(financial\s+condition|results)",
    "10K_1A": r"^\s*risk\s+factors\s*$",
    "10K_7A": r"^\s*quantitative\s+and\s+qualitative\s+disclosures?\s+about\s+market\s+risk",
}
# Annual-report exhibits (EX-13) name the MD&A in many ways ("FINANCIAL REVIEW", "Management Discussion", ...). Used for EX-13 only.
EX13_MDNA_TITLE = (r"^\s*(management.{0,4}s?\s+discussion(\s+and\s+analysis)?(\s+of\s+(financial\s+condition|results)[\w\s,&]{0,60})?"
                   r"|financial\s+review|financial\s+review\s+and\s+analysis|review\s+of\s+operations|"
                   r"financial\s+(condition\s+and\s+)?results\s+of\s+operations)\s*:?\s*$")
_ROMAN = {"1": "I", "2": "II", "i": "I", "ii": "II", "iii": "III", "iv": "IV"}


@dataclass
class Heading:
    line_no: int
    char_pos: int
    key: str           # e.g. "7A"
    title: str
    part: str | None


def form_family(form: str) -> str:
    f = (form or "").upper()
    if f.startswith("10-K"):
        return "10-K"
    if f.startswith("10-Q"):
        return "10-Q"
    if f.startswith("8-K"):
        return "8-K"
    return f


def find_headings(text: str) -> list[Heading]:
    heads: list[Heading] = []
    pos = 0
    lines = text.split("\n")
    cur_part = None
    for i, line in enumerate(lines):
        L = len(line)
        if L <= 220:
            pm = _PART.match(line)
            if pm and not re.search(r"item", line, re.I):
                cur_part = _ROMAN.get(pm.group(1).lower(), pm.group(1).upper())
            m = _ITEM.match(line)
            if m:
                let = (m.group("let") or "").strip("() ").upper()
                key = f"{int(m.group('num'))}{let}"
                title = (m.group("title") or "").strip(" .:-|")
                if len(title) < 4 and i + 1 < len(lines):
                    title = (title + " " + lines[i + 1][:160]).strip()
                part = m.group("part")
                part = _ROMAN.get(part.lower(), part.upper()) if part else cur_part
                heads.append(Heading(i, pos, key, title, part))
        pos += L + 1
    return heads


def _span(text: str, heads: list[Heading], j: int) -> tuple[int, int]:
    start = heads[j].char_pos
    end = len(text)
    for k in range(j + 1, len(heads)):
        if heads[k].key != heads[j].key:
            end = heads[k].char_pos
            break
    return start, end


def extract_sections(text: str, form: str, max_chars: int = 300000) -> dict[str, dict]:
    """{section_id: {text, start, end, method, by_reference}} for the form's target sections."""
    fam = form_family(form)
    targets = TARGETS.get(fam, {})
    heads = find_headings(text)
    out: dict[str, dict] = {}
    for sid, (key, title_rx, part_req) in targets.items():
        best = None
        for j, h in enumerate(heads):
            if h.key != key:
                continue
            if part_req and h.part and h.part != part_req:
                continue
            if title_rx and not re.search(title_rx, h.title, re.I):
                if not (fam == "10-K" and key in ("1", "7") and not h.title):
                    continue
            s, e = _span(text, heads, j)
            if best is None or (e - s) > (best[1] - best[0]):
                best = (s, e)
        method = "item_heading"
        if best is None or (best[1] - best[0]) < 200:
            alt = _title_only(text, sid)
            if alt:
                best, method = alt, "title_only"
        elif (best[1] - best[0]) < 2500 and sid in TITLE_ONLY:
            # Item points elsewhere in the same document (e.g. 'Reference is made to ... Financial Section')
            alt = _title_only(text, sid)
            if alt and (alt[1] - alt[0]) > 3 * (best[1] - best[0]) and alt[0] != best[0]:
                best, method = alt, "title_in_doc"
        if best is None:
            continue
        s, e = best
        body = text[s:e].strip()
        first_nl = body.find("\n")
        if 0 < first_nl < 240:
            body_wo_head = body[first_nl + 1:].strip()
        else:
            body_wo_head = body
        by_ref = len(body_wo_head) < 3000 and bool(_BYREF.search(body_wo_head[:3000]))
        out[sid] = {"text": body[:max_chars], "start": s, "end": e, "method": method, "by_reference": by_ref,
                    "truncated": len(body) > max_chars}
    return out


def _title_only(text: str, sid: str, rx_override: str | None = None):
    rx = rx_override or TITLE_ONLY.get(sid)
    if not rx:
        return None
    best = None
    # line-anchored override patterns need re.M (the slice below spans several lines); built-in patterns keep their behaviour
    pat = re.compile(rx, re.I | re.M) if rx_override else re.compile(rx, re.I)
    starts = [m.start() for m in re.finditer(r"(?m)^.{0,200}$", text) if pat.match(text[m.start():m.start() + 200])]
    stops = [m.start() for m in re.finditer(
        r"(?mi)^\s*(item\s*\d|quantitative\s+and\s+qualitative|financial\s+statements\s+and\s+supplementary|"
        r"report\s+of\s+independent|consolidated\s+statements?\s+of|controls\s+and\s+procedures)", text)]
    for s in starts:
        nxt = [p for p in stops if p > s + 50]
        e = nxt[0] if nxt else len(text)
        if best is None or e - s > best[1] - best[0]:
            best = (s, e)
    return best


def mdna_from_ex13(text: str, max_chars: int = 300000) -> dict | None:
    """MD&A located by title inside an annual-report exhibit (EX-13)."""
    span = _title_only(text, "10K_7")
    if not span or span[1] - span[0] < 1500:
        span = _title_only(text, "10K_7", EX13_MDNA_TITLE)
    if not span or span[1] - span[0] < 1500:
        return None
    s, e = span
    return {"text": text[s:e][:max_chars].strip(), "start": s, "end": e, "method": "ex13_title", "by_reference": False,
            "truncated": e - s > max_chars}


_8K_ITEM = re.compile(r"(?mi)^\s*item[\s\xa0]*(\d{1,2}(?:\.\d{2})?)\b[.:\s\-–—]*(.{0,120})$")
_SIG = re.compile(r"(?mi)^\s*signatures?\s*$|pursuant to the requirements of the securities exchange act of 1934, the registrant has duly caused")


def eightk_items_text(text: str, max_chars: int = 300000) -> dict:
    """Narrative of the 8-K body from the first item heading to the signature block; item codes seen in text."""
    m = list(_8K_ITEM.finditer(text))
    start = m[0].start() if m else 0
    sig = _SIG.search(text, start)
    end = sig.start() if sig else len(text)
    body = text[start:end].strip()
    codes = sorted({x.group(1) for x in m})
    return {"text": body[:max_chars], "start": start, "end": end, "method": "8k_body", "item_codes_text": codes,
            "by_reference": False, "truncated": len(body) > max_chars}


In [ ]:
%%writefile $REPO/src/secprice/parse/taxonomy.py
"""Versioned 8-K item taxonomy: raw / legacy item -> category by effective date, with a catch-all
(SPEC.md §5.2, §18.1 #11)."""
from __future__ import annotations

import re

import pandas as pd


class Taxonomy8K:
    def __init__(self, spec: dict):
        self.spec = spec
        self.version = spec["version"]
        self.modern = {str(k): v for k, v in spec["modern"].items()}
        self.legacy = {str(k): v for k, v in spec["legacy"].items()}
        self.modern_effective = pd.Timestamp(spec["modern_effective"])
        self.item12_effective = pd.Timestamp(spec["legacy_item12_effective"])
        self.catch_all = spec["catch_all"]
        self.legacy_earn = {str(x) for x in spec["legacy_earnings_equivalents"]}
        self.title_patterns = {k: re.compile(v, re.I) for k, v in spec.get("title_patterns", {}).items()}

    @staticmethod
    def split_items(items_raw) -> list[str]:
        if items_raw is None or (isinstance(items_raw, float) and pd.isna(items_raw)):
            return []
        return [x.strip() for x in str(items_raw).replace(";", ",").split(",") if x.strip()]

    def regime(self, filing_date) -> str:
        return "modern" if pd.Timestamp(filing_date) >= self.modern_effective else "legacy"

    def map_item(self, item: str, filing_date) -> str:
        item = str(item).strip()
        if "." in item:
            # modern codes; zero-pad the decimal part ("2.2" -> "2.02")
            a, b = item.split(".", 1)
            item = f"{int(a)}.{int(b):02d}" if a.isdigit() and b.isdigit() else item
            return self.modern.get(item, self.catch_all)
        if item.isdigit():
            if self.regime(filing_date) == "modern":
                return self.catch_all
            if item == "12" and pd.Timestamp(filing_date) < self.item12_effective:
                return self.catch_all
            return self.legacy.get(str(int(item)), self.catch_all)
        return self.catch_all

    def categories(self, items: list[str], filing_date, titles: list[str] | None = None) -> list[str]:
        cats = [self.map_item(i, filing_date) for i in items]
        if not items and titles:
            for t in titles:
                hit = [k for k, rx in self.title_patterns.items() if rx.search(t)]
                cats += hit or [self.catch_all]
        cats = sorted(set(cats)) or [self.catch_all]
        return cats

    def earnings_candidate(self, items: list[str], filing_date) -> tuple[bool, bool]:
        """(is_modern_or_item12_earnings, is_legacy_equivalent_needing_content_check)."""
        modern = any(self.map_item(i, filing_date) == "earnings" for i in items)
        legacy_eq = (not modern and self.regime(filing_date) == "legacy"
                     and any(str(i).strip() in self.legacy_earn for i in items))
        return modern, legacy_eq


In [ ]:
%%writefile $REPO/src/secprice/parse/worker.py
"""S05 per-firm download + parse worker. One process handles one firm at a time with its own SEC client whose
rate is max_rps / n_processes, so the global rate never exceeds sec.max_rps. Outputs are written per firm
atomically (resume never duplicates rows; SPEC.md §17.3)."""
from __future__ import annotations

import hashlib
import os
import time
import traceback
from pathlib import Path

import pandas as pd

from ..sec.archives import doc_url, full_txt_url, parse_sgml
from ..sec.client import FixtureSECClient, SECBlockedError, SECClient
from .anonymize import build_masker
from .html import to_text
from .sections import eightk_items_text, extract_sections, form_family, mdna_from_ex13

SECTION_COLS = ["accession", "cik", "form", "section_id", "text_raw", "text_masked", "n_chars", "n_tokens_est",
                "parse_method", "by_reference", "truncated", "mask_firm", "mask_ticker"]
EXHIBIT_COLS = ["accession", "cik", "exhibit_type", "sequence", "filename", "description", "text", "n_chars"]
LEDGER_COLS = ["accession", "cik", "form", "url", "status", "error", "bytes", "checksum", "n_tables", "text_method",
               "header_acceptance_raw", "header_items", "header_item_titles", "header_period", "truncated",
               "fallback", "n_sections", "elapsed_s"]
MDNA_FALLBACK_CHARS = 15000


def _client(job: dict):
    if job.get("fixture_root"):
        return FixtureSECClient(job["fixture_root"])
    return SECClient(job["user_agent"], max_rps=job["rps_per_proc"], cache_dir=job["cache_dir"],
                     cache_days=job["cache_days"], timeout_s=job["timeout_s"], max_retries=job["max_retries"],
                     backoff_base_s=job["backoff_base_s"], blocked_403_limit=job["blocked_403_limit"],
                     cache_max_bytes=job.get("cache_max_bytes", 0))


def _sec_rows(acc, cik, form, secs: dict, mask) -> list[dict]:
    rows = []
    for sid, s in secs.items():
        txt = s["text"]
        if not txt or len(txt) < 40:
            continue
        masked, mc = mask(txt)
        rows.append({"accession": acc, "cik": cik, "form": form, "section_id": sid, "text_raw": txt,
                     "text_masked": masked, "n_chars": len(txt), "n_tokens_est": int(len(txt.split()) * 1.3),
                     "parse_method": s["method"], "by_reference": bool(s.get("by_reference")),
                     "truncated": bool(s.get("truncated")), "mask_firm": mc["firm"], "mask_ticker": mc["ticker"]})
    return rows


def process_filing(client, f: dict, mask, max_chars: int, max_8k: int, max_doc: int) -> tuple[list, list, dict]:
    cik, acc, form = int(f["cik"]), f["accession"], f["form"]
    filer = int(f.get("filer_cik") or cik)               # URLs use the filer's CIK; outputs use the firm id
    fam = form_family(form)
    t0 = time.time()
    led = {"accession": acc, "cik": cik, "form": form, "status": "ok", "error": None, "bytes": 0, "checksum": None,
           "n_tables": 0, "text_method": None, "header_acceptance_raw": None, "header_items": None,
           "header_item_titles": None, "header_period": None, "truncated": False, "fallback": None, "n_sections": 0}
    sec_rows, ex_rows = [], []
    try:
        if fam == "8-K":
            url = full_txt_url(filer, acc)
            raw = client.get(url, max_bytes=max_8k)
            led["url"] = url
            if raw is None:
                led["status"] = "http_404"
                return sec_rows, ex_rows, led
            led["bytes"], led["checksum"] = len(raw), hashlib.sha256(raw).hexdigest()
            sub = parse_sgml(raw, truncated=len(raw) >= max_8k)
            led.update(header_acceptance_raw=sub.acceptance_raw, header_items=",".join(sub.items),
                       header_item_titles=" || ".join(sub.item_titles), header_period=sub.period,
                       truncated=sub.truncated)
            main = [d for d in sub.documents if d.type.upper().startswith("8-K")] or sub.documents[:1]
            secs = {}
            if main:
                txt, nt, meth = to_text(main[0].text)
                led["n_tables"], led["text_method"] = nt, meth
                secs["8K_ITEMS"] = eightk_items_text(txt, max_chars)
            ex_txt = []
            for d in sub.documents:
                if d.type.upper().startswith("EX-99"):
                    t, nt, _ = to_text(d.text)
                    led["n_tables"] += nt
                    ex_rows.append({"accession": acc, "cik": cik, "exhibit_type": d.type.upper(), "sequence": d.sequence,
                                    "filename": d.filename, "description": d.description, "text": t[:max_chars],
                                    "n_chars": len(t)})
                    ex_txt.append(t)
            if ex_txt:
                joined = "\n\n".join(ex_txt)
                secs["8K_EX99"] = {"text": joined[:max_chars], "method": "ex99", "by_reference": False,
                                   "truncated": len(joined) > max_chars}
            sec_rows = _sec_rows(acc, cik, form, secs, mask)
        else:
            sub = None
            if f.get("primary_doc"):
                url = doc_url(filer, acc, f["primary_doc"])
                raw = client.get(url, max_bytes=max_doc)
                body = raw
            else:
                url, raw, body = full_txt_url(filer, acc), None, None
            if raw is None:
                url = full_txt_url(filer, acc)
                raw = client.get(url, max_bytes=max_doc)
                led["fallback"] = "full_txt"
                if raw is None:
                    led["url"], led["status"] = url, "http_404"
                    return sec_rows, ex_rows, led
                sub = parse_sgml(raw)
                led["header_acceptance_raw"] = sub.acceptance_raw
                led["header_period"] = sub.period
                docs = [d for d in sub.documents if d.type.upper().startswith(fam[:4])] or sub.documents[:1]
                body = docs[0].text if docs else ""
            led["url"], led["bytes"], led["checksum"] = url, len(raw), hashlib.sha256(raw).hexdigest()
            txt, nt, meth = to_text(body)
            led["n_tables"], led["text_method"] = nt, meth
            secs = extract_sections(txt, form, max_chars)
            if fam == "10-K":
                m7 = secs.get("10K_7")
                if m7 is None or m7.get("by_reference") or len(m7["text"]) < MDNA_FALLBACK_CHARS:
                    if sub is None:
                        full = client.get(full_txt_url(filer, acc), max_bytes=max_doc)
                        sub = parse_sgml(full) if full else None
                        if sub is not None and not led["header_acceptance_raw"]:
                            led["header_acceptance_raw"] = sub.acceptance_raw
                    if sub is not None:
                        for d in sub.documents:
                            if d.type.upper().startswith("EX-13"):
                                t13, _, _ = to_text(d.text)
                                alt = mdna_from_ex13(t13, max_chars)
                                if alt and (m7 is None or len(alt["text"]) > 2 * len(m7["text"])):
                                    secs["10K_7"] = alt
                                    led["fallback"] = (led["fallback"] or "") + "+ex13_mdna"
                                    break
            sec_rows = _sec_rows(acc, cik, form, secs, mask)
            if not led["header_acceptance_raw"]:
                # the submissions-JSON time is unreliable (two conventions observed); read the SGML header from the first
                # 20 KB of the full submission (one small request) - decision log D-047
                pre = client.get(full_txt_url(filer, acc), max_bytes=20000)
                if pre:
                    hs = parse_sgml(pre, truncated=True)
                    led["header_acceptance_raw"] = hs.acceptance_raw
                    led["header_period"] = led["header_period"] or hs.period
        led["n_sections"] = len(sec_rows)
        if not sec_rows:
            led["status"] = "no_sections"
    except SECBlockedError:
        raise
    except Exception as e:
        led["status"], led["error"] = "error", f"{type(e).__name__}: {e}"[:300]
    led["elapsed_s"] = round(time.time() - t0, 2)
    return sec_rows, ex_rows, led


def run_firm(job: dict) -> dict:
    """Download + parse every task filing of one firm; write sections/exhibits/ledger parquet atomically."""
    out = Path(job["out_dir"])
    cik = int(job["cik"])
    done_flag = out / "ledger" / f"{cik:010d}.parquet"
    if done_flag.exists():
        if job.get("retry_errors"):
            return _retry_failed(job, out, cik, done_flag)
        return {"cik": cik, "skipped": True}
    client = _client(job)
    mask = build_masker(job["names"], job["tickers"])
    secs, exs, leds = [], [], []
    for f in job["filings"]:
        s, e, l = process_filing(client, f, mask, job["max_chars"], job["max_8k_bytes"], job["max_doc_bytes"])
        secs += s
        exs += e
        leds.append(l)
    for sub, rows, cols in (("sections", secs, SECTION_COLS), ("exhibits", exs, EXHIBIT_COLS)):
        d = out / sub
        d.mkdir(parents=True, exist_ok=True)
        df = pd.DataFrame(rows, columns=cols)
        tmp = d / f".tmp_{cik:010d}.parquet"
        df.to_parquet(tmp, index=False, compression="zstd")
        os.replace(tmp, d / f"{cik:010d}.parquet")
    (out / "ledger").mkdir(parents=True, exist_ok=True)
    led = pd.DataFrame(leds, columns=LEDGER_COLS)
    tmp = out / "ledger" / f".tmp_{cik:010d}.parquet"
    led.to_parquet(tmp, index=False, compression="zstd")
    os.replace(tmp, done_flag)
    return {"cik": cik, "skipped": False, "n": len(leds), "ok": int((led.status == "ok").sum()),
            "stats": getattr(client, "stats", {})}


def _retry_failed(job: dict, out: Path, cik: int, done_flag: Path) -> dict:
    """Re-process only the filings whose earlier attempt ended in a transient error (e.g. repeated HTTP 503) and replace
    their rows; every other filing of the firm is untouched (sec.retry_failed_filings)."""
    led = pd.read_parquet(done_flag)
    bad = set(led.loc[led.status == "error", "accession"])
    if not bad:
        return {"cik": cik, "skipped": True}
    client = _client(job)
    mask = build_masker(job["names"], job["tickers"])
    secs, exs, leds = [], [], []
    for f in job["filings"]:
        if f["accession"] in bad:
            s_, e_, l_ = process_filing(client, f, mask, job["max_chars"], job["max_8k_bytes"], job["max_doc_bytes"])
            secs += s_
            exs += e_
            leds.append(l_)
    for sub, rows, cols in (("sections", secs, SECTION_COLS), ("exhibits", exs, EXHIBIT_COLS)):
        p = out / sub / f"{cik:010d}.parquet"
        old = pd.read_parquet(p) if p.exists() else pd.DataFrame(columns=cols)
        new = pd.concat([old[~old.accession.isin(bad)], pd.DataFrame(rows, columns=cols)], ignore_index=True)
        tmp = out / sub / f".tmp_{cik:010d}.parquet"
        new.to_parquet(tmp, index=False, compression="zstd")
        os.replace(tmp, p)
    led2 = pd.concat([led[~led.accession.isin(bad)], pd.DataFrame(leds, columns=LEDGER_COLS)], ignore_index=True)
    tmp = out / "ledger" / f".tmp_{cik:010d}.parquet"
    led2.to_parquet(tmp, index=False, compression="zstd")
    os.replace(tmp, done_flag)
    return {"cik": cik, "skipped": False, "retried": len(bad), "ok": int((pd.DataFrame(leds).status == "ok").sum()) if leds else 0}


def run_firm_safe(job: dict) -> dict:
    try:
        return run_firm(job)
    except SECBlockedError as e:
        return {"cik": job["cik"], "blocked": str(e)}
    except Exception as e:
        return {"cik": job["cik"], "crash": f"{type(e).__name__}: {e}", "tb": traceback.format_exc()[-2000:]}


In [ ]:
%%writefile $REPO/src/secprice/pipeline.py
"""Programmatic API used by the CLI and the notebook (SPEC.md §17.2)."""
from __future__ import annotations

import hashlib
import json
import shutil
from pathlib import Path

import pandas as pd

from . import stages_data, stages_model, stages_report  # noqa: F401  (register stages)
from .config import load_config, text_sha256
from .runner import REGISTRY, Runner, _stage_key
from .storage import Storage, write_json

def quiet() -> None:
    """Silence known, understood library warnings so real messages stay visible (Colab output)."""
    import warnings
    for cat in (FutureWarning, DeprecationWarning, PendingDeprecationWarning):
        warnings.filterwarnings("ignore", category=cat)
    warnings.filterwarnings("ignore", message=".*ConstantInputWarning.*")
    warnings.filterwarnings("ignore", message=".*An input array is constant.*")
    warnings.filterwarnings("ignore", message=".*divide by zero.*")
    warnings.filterwarnings("ignore", message=".*invalid value encountered.*")
    warnings.filterwarnings("ignore", message=".*covariance of constraints does not have full rank.*")
    warnings.filterwarnings("ignore", message=".*X does not have valid feature names.*")
    warnings.filterwarnings("ignore", message=".*Mean of empty slice.*")
    warnings.filterwarnings("ignore", message=".*Degrees of freedom <= 0.*")
    try:
        from scipy.stats import ConstantInputWarning
        warnings.filterwarnings("ignore", category=ConstantInputWarning)
    except Exception:
        pass
    try:
        from statsmodels.tools.sm_exceptions import ValueWarning
        warnings.filterwarnings("ignore", category=ValueWarning)
    except Exception:
        pass


quiet()

CPU_PART_1 = ["S00", "S01", "S02", "S03", "S04", "S05", "S06", "S07", "S08", "S09", "S10", "S11", "S12", "S20", "S21"]
GPU_PART = ["S13", "S14", "S15", "S16", "S17"]
CPU_PART_2 = ["S18", "S19", "S22", "S23", "S24"]


def expand(stages) -> list[str]:
    """'S00', ['S01','S03'], 'S00-S11' -> ordered list."""
    if isinstance(stages, str):
        stages = [stages]
    out = []
    keys = sorted(REGISTRY, key=_stage_key)
    for s in stages:
        if "-" in s:
            a, b = s.split("-")
            out += [k for k in keys if _stage_key(a) <= _stage_key(k) <= _stage_key(b)]
        else:
            out.append(s)
    return out


def make_runner(profile: str | None = None, overrides: dict | None = None, data_root=None, hot_root=None,
                services: dict | None = None) -> Runner:
    cfg = load_config(profile, overrides)
    return Runner(cfg, Storage(cfg, data_root, hot_root), services or {})


def run(stages, profile=None, overrides=None, force=False, data_root=None, hot_root=None, services=None, with_deps=False) -> dict:
    r = make_runner(profile, overrides, data_root, hot_root, services)
    return r.run(expand(stages), force=force, with_deps=with_deps)


def status(profile=None, overrides=None, data_root=None) -> pd.DataFrame:
    return pd.DataFrame(make_runner(profile, overrides, data_root).status())


def prereg_freeze(profile=None, overrides=None, data_root=None, force=False) -> str:
    r = make_runner(profile, overrides, data_root)
    root = Path(r.cfg["_meta"]["repo_root"])
    h = text_sha256(root / "configs" / "preregistration.yaml")
    p = r.storage.manifests / "prereg_hash.txt"
    if p.exists() and p.read_text().strip() != h:
        if not force:
            raise RuntimeError(f"preregistration already frozen with a different hash ({p.read_text().strip()}); "
                               "a change is a protocol deviation: re-run with force=True to record it")
        dev = r.storage.manifests / "prereg_deviations.json"
        log = json.loads(dev.read_text()) if dev.exists() else []
        log.append({"old": p.read_text().strip(), "new": h, "at": pd.Timestamp.utcnow().isoformat()})
        write_json(dev, log)
    p.write_text(h)
    return h


def freeze(profile=None, overrides=None, data_root=None) -> str:
    """G4 freeze is performed by S18; this returns the current G4 hash."""
    r = make_runner(profile, overrides, data_root)
    p = r.storage.manifests / "g4_hash.txt"
    if not p.exists():
        raise RuntimeError("no G4 freeze yet: run S18")
    return p.read_text().strip()


def unlock_test(profile=None, overrides=None, data_root=None, acknowledge_reuse=False, services=None) -> dict:
    ov = dict(overrides or {})
    ov.setdefault("lockbox", {})
    ov["lockbox"] = {**ov["lockbox"], "unlock_locked_test": True, "acknowledge_reuse": bool(acknowledge_reuse)}
    return run(["S19"], profile, ov, force=True, data_root=data_root, services=services)


def preflight(profile=None, overrides=None, data_root=None, hot_root=None) -> dict:
    """CPU preflight (G2) from the S04 filing index: projected events, chunks, storage and GPU hours vs the budget."""
    from .stages_data import preflight_projection
    r = make_runner(profile, overrides, data_root, hot_root)
    p = r.storage.profile_root / "S04" / r.stage_hash("S04") / "filing_manifest.parquet"
    if not p.exists():
        raise RuntimeError("run S04 first: the preflight projects from the filing index")
    return preflight_projection(r.cfg, pd.read_parquet(p), r.storage.data_root, r.storage.hot_root)


def gold(action: str, profile=None, overrides=None, data_root=None, hot_root=None) -> dict:
    """`secprice gold export|import|agreement` (SPEC.md §17.2, §12).
    export: (re)write the stratified sample, CSV templates and annotation guide (S12).
    import: validate the completed files in DATA_ROOT/inputs/gold against the frozen sample, then import them (S12).
    agreement: Cohen's kappa per field on items labelled by two annotators."""
    from . import common as C
    from .gold.tools import cohen_kappa, import_gold
    r = make_runner(profile, overrides, data_root, hot_root)
    sdir = r.storage.profile_root / "S12" / r.stage_hash("S12")
    if action == "export":
        run(["S12"], profile, overrides, force=True, data_root=data_root, hot_root=hot_root)
        return {"templates": sorted(str(p) for p in (sdir / "templates").glob("*"))}
    if not (sdir / "gold_sample.parquet").exists():
        raise RuntimeError("run `secprice gold export` first: the frozen gold sample does not exist yet")
    if action == "import":
        g, errs = import_gold(r.storage.inputs / "gold", pd.read_parquet(sdir / "gold_sample.parquet"), C.question_bank(r.cfg))
        if len(g):
            run(["S12"], profile, overrides, force=True, data_root=data_root, hot_root=hot_root)
        st = json.loads((sdir / "gold_status.json").read_text()) if (sdir / "gold_status.json").exists() else {}
        return {"valid_rows": int(len(g)), "errors": errs, "status": st}
    if action == "agreement":
        p = sdir / "gold_labels.parquet"
        if not p.exists():
            return {"note": "no imported labels; run `secprice gold import` first"}
        g = pd.read_parquet(p)
        out = []
        for f, gf in g.groupby("field"):
            n_ann = gf.groupby("event_id").annotator.nunique()
            both = gf[gf.event_id.isin(n_ann[n_ann > 1].index)]
            if both.annotator.nunique() < 2:
                continue
            a, b = sorted(both.annotator.unique())[:2]
            x = both[both.annotator == a].drop_duplicates("event_id").set_index("event_id").present_b
            y = both[both.annotator == b].drop_duplicates("event_id").set_index("event_id").present_b
            j = x.index.intersection(y.index)
            out.append({"field": f, "n_items": int(len(j)), "kappa_present": cohen_kappa(x.loc[j], y.loc[j])})
        return {"agreement": out or "no items labelled by two annotators (single-rater, provisional)"}
    raise ValueError(f"unknown gold action {action!r}")


def sync(direction: str, src_root, dst_root, profile: str, stages: list[str] | None = None) -> list[str]:
    """Copy stage artifacts (and inputs) between DATA_ROOTs (local <-> Drive)."""
    src, dst = Path(src_root), Path(dst_root)
    if direction == "pull":
        src, dst = dst, src
    copied = []
    for s in stages or sorted(REGISTRY, key=_stage_key):
        a = src / profile / s
        if a.exists():
            shutil.copytree(a, dst / profile / s, dirs_exist_ok=True)
            copied.append(s)
    for sub in ("manifests", "reports"):
        a = src / profile / sub
        if a.exists():
            shutil.copytree(a, dst / profile / sub, dirs_exist_ok=True)
    return copied


In [ ]:
%%writefile $REPO/src/secprice/rag/__init__.py
"""secprice.rag"""


In [ ]:
%%writefile $REPO/src/secprice/rag/core.py
"""Core RAG (SPEC.md §8): BM25, hybrid BM25 + dense (RRF), hybrid + cross-encoder rerank; two evidence paths.

Current-document path: chunks of the filing being processed (allowed at decision_time).
Historical path: the firm's prior filings with public_available_time < decision_time, never the current filing
and never a later one (SPEC.md §6.3 #16). The filter is the only way to build a historical candidate set.
"""
from __future__ import annotations

import re

import numpy as np
import pandas as pd

_TOK = re.compile(r"[a-z0-9]+")


def tokenize(t: str) -> list[str]:
    return _TOK.findall(t.lower())


class BM25:
    def __init__(self, docs: list[str], k1: float = 1.5, b: float = 0.75):
        self.k1, self.b = k1, b
        self.toks = [tokenize(d) for d in docs]
        self.N = len(docs)
        self.avgdl = np.mean([len(t) for t in self.toks]) if self.toks else 0.0
        df: dict = {}
        for t in self.toks:
            for w in set(t):
                df[w] = df.get(w, 0) + 1
        self.idf = {w: np.log(1 + (self.N - n + 0.5) / (n + 0.5)) for w, n in df.items()}
        self.tf = [pd.Series(t).value_counts().to_dict() if t else {} for t in self.toks]

    def scores(self, query: str) -> np.ndarray:
        q = tokenize(query)
        s = np.zeros(self.N)
        for i, (tf, toks) in enumerate(zip(self.tf, self.toks)):
            dl = len(toks)
            for w in q:
                f = tf.get(w)
                if f:
                    s[i] += self.idf.get(w, 0) * f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * dl / max(self.avgdl, 1e-9)))
        return s


def rrf(rankings: list[np.ndarray], k: int = 60) -> np.ndarray:
    """Reciprocal-rank fusion of score arrays (higher = better)."""
    n = len(rankings[0])
    out = np.zeros(n)
    for sc in rankings:
        order = np.argsort(-sc, kind="stable")
        r = np.empty(n)
        r[order] = np.arange(1, n + 1)
        out += 1.0 / (k + r)
    return out


def historical_candidates(chunks: pd.DataFrame, cik: int, decision_time, current_accessions) -> pd.DataFrame:
    """The only constructor of the historical evidence set: same firm, public strictly before decision_time,
    current filing excluded."""
    dt = pd.Timestamp(decision_time)
    pub = pd.to_datetime(chunks["public_time"], utc=True)
    m = (chunks["cik"] == cik) & (pub < dt.tz_convert("UTC") if dt.tzinfo else pub < dt.tz_localize("UTC"))
    m &= ~chunks["accession"].isin(list(current_accessions))
    return chunks[m]


def retrieval_guard() -> bool:
    """Planted-leak self-test: a future document and the current filing must never enter the historical set."""
    now = pd.Timestamp("2015-06-01 12:00", tz="America/New_York")
    ch = pd.DataFrame({"chunk_id": ["past", "current", "future", "other_firm"], "cik": [1, 1, 1, 2],
                       "accession": ["a-past", "a-cur", "a-fut", "b-past"],
                       "public_time": [now - pd.Timedelta(days=90), now - pd.Timedelta(minutes=35),
                                       now + pd.Timedelta(days=1), now - pd.Timedelta(days=5)],
                       "text": ["x", "y", "z", "w"]})
    got = set(historical_candidates(ch, 1, now, ["a-cur"]).chunk_id)
    return got == {"past"}


class Retriever:
    """setup in {'bm25', 'hybrid', 'hybrid_rerank'}; dense / rerank models are injected (GPU) or None."""

    def __init__(self, setup: str, dense=None, reranker=None, k1=1.5, b=0.75, rrf_k=60, rerank_top_n=30):
        self.setup, self.dense, self.reranker = setup, dense, reranker
        self.k1, self.b, self.rrf_k, self.top_n = k1, b, rrf_k, rerank_top_n

    def rank(self, query: str, cands: pd.DataFrame, k: int) -> pd.DataFrame:
        if cands.empty:
            return cands.assign(score=[])
        texts = cands["text"].tolist()
        s_bm = BM25(texts, self.k1, self.b).scores(query)
        if self.setup == "bm25" or self.dense is None:
            score = s_bm
        else:
            s_d = self.dense(query, texts)
            score = rrf([s_bm, s_d], self.rrf_k)
        order = np.argsort(-score, kind="stable")
        out = cands.iloc[order].assign(score=score[order])
        if self.setup == "hybrid_rerank" and self.reranker is not None:
            top = out.head(self.top_n)
            rs = self.reranker(query, top["text"].tolist())
            top = top.assign(score=rs).sort_values("score", ascending=False)
            out = top
        return out.head(k)


def ir_metrics(ranked_ids: list[str], relevant: set, k: int) -> dict:
    """Recall@k, nDCG@k, MRR, AP for one query (binary relevance)."""
    r = ranked_ids[:k]
    hits = [1 if x in relevant else 0 for x in r]
    rec = sum(hits) / len(relevant) if relevant else None
    dcg = sum(h / np.log2(i + 2) for i, h in enumerate(hits))
    idcg = sum(1 / np.log2(i + 2) for i in range(min(k, len(relevant))))
    mrr = next((1 / (i + 1) for i, h in enumerate(hits) if h), 0.0)
    ap, n = 0.0, 0
    for i, h in enumerate(hits):
        if h:
            n += 1
            ap += n / (i + 1)
    return {"recall": rec, "ndcg": dcg / idcg if idcg else None, "mrr": mrr, "ap": ap / len(relevant) if relevant else None}


In [ ]:
%%writefile $REPO/src/secprice/rag/distill.py
"""S17 teacher sample -> student distillation (SPEC.md §8 Distillation). The teacher (frozen RAG + generator) labels a
stratified sample of TRAINING-PERIOD filings; a multi-task student (ModernBERT-base) is trained on those outputs and
applied to every event. Reported: student vs teacher agreement (and student vs human on gold-test in S23)."""
from __future__ import annotations

import json

import numpy as np
import pandas as pd

from .. import common as C
from ..storage import write_json


def teacher_sample(ev: pd.DataFrame, dev_start: pd.Timestamp, n: int, seed: int) -> pd.DataFrame:
    tr = ev[pd.to_datetime(ev.decision_date) < dev_start].copy()
    tr["stratum"] = tr.event_type + "|" + (tr.year // 5 * 5).astype(str)
    k = min(n, len(tr))
    w = tr.groupby("stratum").event_id.transform("size")
    p = (1 / np.sqrt(w)) / (1 / np.sqrt(w)).sum()
    idx = np.random.default_rng(seed).choice(len(tr), size=k, replace=False, p=p.to_numpy())
    s = tr.iloc[idx]
    assert (pd.to_datetime(s.decision_date) < dev_start).all(), "teacher sample must be training-period only"
    return s


def run_distillation(ctx, exploratory: bool) -> dict:
    import torch
    from ..models.embeddings import event_chunks
    from ..rag.core import Retriever
    from ..rag.extract import ResponseCache, build_prompt, cache_key, field_schema, guard
    from ..stages_model import _dense_and_reranker, _generator
    cfg = ctx.cfg
    qb = C.question_bank(cfg)
    fields = list(qb["fields"])
    ev = C.events(ctx)
    smp = teacher_sample(ev, pd.Timestamp(cfg["splits"]["dev_start"]), int(cfg["budget"]["llm_teacher_filings"]), int(cfg["seed"]))
    gen, gname = _generator(ctx, qb)
    dense, rerank = _dense_and_reranker(ctx) if gname != "mock" else (None, None)
    frozen = json.loads((ctx.dep("S16") / "rag_frozen.json").read_text())
    ret = Retriever(frozen["setup"] if dense else "bm25", dense, rerank, cfg["rag"]["bm25_k1"], cfg["rag"]["bm25_b"],
                    cfg["rag"]["rrf_k"], cfg["rag"]["rerank_top_n"])
    cache = ResponseCache(ctx.out / "teacher_cache.jsonl")
    secs = []
    for cik in smp.cik.unique():
        p = ctx.dep("S05") / "sections" / f"{int(cik):010d}.parquet"
        if p.exists():
            secs.append(pd.read_parquet(p, columns=["accession", "section_id", "text_masked"]))
    ch = event_chunks(smp, pd.concat(secs, ignore_index=True), cfg) if secs else pd.DataFrame()
    ch["chunk_id"] = ch.event_id + ":" + ch.section_id + ":" + ch.chunk_no.astype(str)
    q = " ".join(f["question"] for f in qb["fields"].values())
    rows = []
    pmap, todo = {}, []
    for eid in smp.event_id:
        c = ch[ch.event_id == eid]
        if c.empty:
            continue
        top = ret.rank(q, c, int(cfg["rag"]["top_k"]))
        passages = [{"chunk_id": r.chunk_id, "text": r.text} for r in top.itertuples(index=False)]
        k = cache_key(getattr(gen, "revision", gname), build_prompt(qb, passages), passages)
        pmap[eid] = (k, passages)
        if cache.get(k) is None:
            todo.append(eid)
    for b0 in range(0, len(todo), 64):                      # checkpointed through the response cache
        ids_b = todo[b0:b0 + 64]
        for e, raw in zip(ids_b, gen.batch(qb, [pmap[e][1] for e in ids_b])):
            cache.put(pmap[e][0], raw)
    for eid, (k, passages) in pmap.items():
        clean, _ = guard(cache.get(k) or {}, qb, passages, field_schema(qb))
        for f in fields:
            v = clean.get(f) or {}
            rows.append({"event_id": eid, "field": f, "present": v.get("present"), "direction": v.get("direction"),
                         "magnitude": v.get("magnitude"), "unit": v.get("unit"), "certainty": v.get("certainty"),
                         "evidence": json.dumps(v.get("evidence") or []), "source": "teacher", "confidence": None})
    teach = pd.DataFrame(rows)
    texts = C.read(ctx, "S09", "event_text.parquet").set_index("event_id")["text"]
    student = train_student(cfg, teach, texts, fields, qb, ctx.log)
    allp = student["predict"](texts.index.tolist(), texts.tolist())
    feats = pd.DataFrame({"event_id": texts.index})
    for j, f in enumerate(fields):
        feats[f"ext_{f}_present"] = allp[:, j]
    ctx.write(feats, "ext_features.parquet")
    ext = teach.copy()
    stud = feats.melt(id_vars="event_id", var_name="field", value_name="confidence")
    stud["field"] = stud.field.str.replace("ext_", "").str.replace("_present", "")
    stud["present"] = stud.confidence > 0.5
    stud["source"] = "student"
    ctx.write(pd.concat([ext, stud], ignore_index=True), "extractions.parquet")
    rep = {"ran": True, "exploratory": exploratory, "generator": gname, "teacher_filings": int(smp.shape[0]),
           "student_vs_teacher": student["agreement"], "label": "EXPLORATORY: G3b not passed" if exploratory else "G3b passed"}
    write_json(ctx.out / "student_report.json", rep)
    return rep


def train_student(cfg, teach: pd.DataFrame, texts: pd.Series, fields: list[str], qb: dict, log=None) -> dict:
    """Multi-label student: present probability per field (BCE). ModernBERT-base on GPU; logistic TF-IDF on CPU."""
    import torch
    piv = teach.pivot_table(index="event_id", columns="field", values="present", aggfunc="first").reindex(columns=fields)
    piv = piv.dropna(how="all").fillna(False).astype(float)
    X = texts.reindex(piv.index).fillna("").tolist()
    Y = piv.to_numpy(np.float32)
    n = len(X)
    cut = max(1, int(n * 0.85))
    if not torch.cuda.is_available() or n < 50:
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.linear_model import LogisticRegression
        vec = TfidfVectorizer(max_features=20000, min_df=1, sublinear_tf=True).fit(X[:cut])
        Xt = vec.transform(X)
        models = []
        for j in range(len(fields)):
            y = Y[:cut, j]
            models.append(LogisticRegression(max_iter=500).fit(Xt[:cut], y) if len(set(y)) > 1 else float(y.mean() if len(y) else 0))
        def predict(ids, txts):
            Z = vec.transform(txts)
            return np.column_stack([m.predict_proba(Z)[:, 1] if hasattr(m, "predict_proba") else np.full(len(txts), m) for m in models])
        kind = "tfidf_logistic_cpu_fallback"
    else:
        from transformers import AutoModel, AutoTokenizer
        st = C.roster(cfg)["student"][cfg["distill"]["student_model"]]
        tok = AutoTokenizer.from_pretrained(st["model_id"], revision=st["revision"])
        base = AutoModel.from_pretrained(st["model_id"], revision=st["revision"]).cuda()
        head = torch.nn.Linear(base.config.hidden_size, len(fields)).cuda()
        opt = torch.optim.AdamW(list(base.parameters()) + list(head.parameters()), lr=float(cfg["distill"]["lr"]))
        enc = tok(X, truncation=True, max_length=512, padding="max_length", return_tensors="pt")
        yt = torch.tensor(Y)
        bs = int(cfg["distill"]["batch"])
        base.train()
        for ep in range(int(cfg["distill"]["epochs"])):
            perm = torch.randperm(cut)
            for k in range(0, cut, bs):
                b = perm[k:k + bs]
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    h = base(input_ids=enc["input_ids"][b].cuda(), attention_mask=enc["attention_mask"][b].cuda()).last_hidden_state[:, 0]
                    lo = head(h.float())
                loss = torch.nn.functional.binary_cross_entropy_with_logits(lo.float(), yt[b].cuda())
                loss.backward()
                opt.step()
                opt.zero_grad()
            if log:
                log.info(f"student epoch {ep}: loss {loss.item():.4f}")
        base.eval()

        def predict(ids, txts):
            out = []
            with torch.no_grad():
                for k in range(0, len(txts), 64):
                    e = tok(txts[k:k + 64], truncation=True, max_length=512, padding=True, return_tensors="pt").to("cuda")
                    with torch.autocast("cuda", dtype=torch.bfloat16):
                        h = base(**e).last_hidden_state[:, 0]
                    out.append(torch.sigmoid(head(h.float())).float().cpu().numpy())
            return np.vstack(out) if out else np.zeros((0, len(fields)))
        kind = "modernbert_multitask"
    hold = predict(None, X[cut:]) if n > cut else np.zeros((0, len(fields)))
    from sklearn.metrics import f1_score
    agr = {}
    for j, f in enumerate(fields):
        yt_ = Y[cut:, j] > 0.5
        if len(yt_) and yt_.any():
            agr[f] = float(f1_score(yt_, hold[:, j] > 0.5))
    return {"predict": predict, "agreement": {"kind": kind, "heldout_n": int(n - cut), "f1_vs_teacher": agr}}


In [ ]:
%%writefile $REPO/src/secprice/rag/extract.py
"""Evidence-grounded extraction (SPEC.md §8): prompt -> JSON-constrained generation -> guardrails -> cache.

Guardrails: schema validation + one retry; numeric cross-check (a magnitude must appear in the cited evidence);
faithfulness gate (every evidence quote must be a substring of its cited chunk, otherwise the field becomes null);
citation offsets verified. The response cache is keyed by model revision, prompt hash and passage hash.
"""
from __future__ import annotations

import hashlib
import json
import re
from pathlib import Path

import numpy as np

SYSTEM = ("You extract facts from U.S. SEC filings. Use ONLY the numbered passages. If the passages do not support a "
          "field, set present=false. Quote evidence verbatim (<= 30 words) and cite the passage id. Output JSON only.")


def field_schema(qb: dict) -> dict:
    props = {}
    for name, f in qb["fields"].items():
        props[name] = {"type": "object", "properties": {
            "present": {"type": "boolean"},
            "direction": {"anyOf": [{"type": "string", "enum": f["directions"]}, {"type": "null"}]},
            "magnitude": {"anyOf": [{"type": "number"}, {"type": "null"}]},
            "unit": {"anyOf": [{"type": "string"}, {"type": "null"}]},
            "certainty": {"type": "string", "enum": qb.get("certainty_levels", ["low", "medium", "high"])},
            "evidence": {"type": "array", "maxItems": 2, "items": {"type": "object", "properties": {
                "chunk_id": {"type": "string"}, "quote": {"type": "string"}}, "required": ["chunk_id", "quote"]}}},
            "required": ["present", "direction", "magnitude", "unit", "certainty", "evidence"]}
    return {"type": "object", "properties": props, "required": list(props)}


def build_prompt(qb: dict, passages: list[dict]) -> str:
    qs = "\n".join(f"- {k}: {f['question']} directions={f['directions']}" for k, f in qb["fields"].items())
    ps = "\n\n".join(f"[{p['chunk_id']}] {p['text'][:1800]}" for p in passages)
    return f"Fields:\n{qs}\n\nPassages:\n{ps}\n\nReturn one JSON object with every field."


def cache_key(model_rev: str, prompt: str, passages: list[dict]) -> str:
    ph = hashlib.sha256(prompt.encode()).hexdigest()
    pa = hashlib.sha256("".join(p["chunk_id"] + p["text"] for p in passages).encode()).hexdigest()
    return hashlib.sha256(f"{model_rev}|{ph}|{pa}".encode()).hexdigest()


class ResponseCache:
    def __init__(self, path: Path):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.d = {}
        if self.path.exists():
            for line in self.path.read_text(encoding="utf-8").splitlines():
                try:
                    r = json.loads(line)
                    self.d[r["k"]] = r["v"]
                except Exception:
                    continue

    def get(self, k):
        return self.d.get(k)

    def put(self, k, v):
        self.d[k] = v
        with open(self.path, "a", encoding="utf-8") as f:
            f.write(json.dumps({"k": k, "v": v}) + "\n")


def _norm(s: str) -> str:
    return " ".join(re.sub(r"[^\w$%.,-]+", " ", s.lower()).split())


def guard(raw: dict, qb: dict, passages: list[dict], schema: dict) -> tuple[dict, dict]:
    """Apply guardrails. Returns (clean fields, flags)."""
    import jsonschema
    flags = {"schema_ok": True, "unsupported_nulled": 0, "numeric_nulled": 0, "citations_verified": 0, "citations_total": 0}
    try:
        jsonschema.validate(raw, schema)
    except Exception:
        flags["schema_ok"] = False
    by_id = {p["chunk_id"]: p["text"] for p in passages}
    out = {}
    for name, fdef in qb["fields"].items():
        f = raw.get(name) if isinstance(raw, dict) else None
        if not isinstance(f, dict):
            out[name] = {"present": None, "direction": None, "magnitude": None, "unit": None, "certainty": None,
                         "evidence": [], "status": "missing"}
            continue
        ev_ok = []
        for e in (f.get("evidence") or [])[:2]:
            flags["citations_total"] += 1
            txt = by_id.get(str(e.get("chunk_id")))
            q = str(e.get("quote") or "")
            if txt and q and _norm(q) and _norm(q) in _norm(txt):
                st = txt.lower().find(q.lower()[:40])
                ev_ok.append({"chunk_id": e["chunk_id"], "quote": q, "start": int(st) if st >= 0 else None,
                              "end": int(st + len(q)) if st >= 0 else None})
                flags["citations_verified"] += 1
        rec = {"present": bool(f.get("present")) if f.get("present") is not None else None,
               "direction": f.get("direction") if f.get("direction") in fdef["directions"] else None,
               "magnitude": f.get("magnitude") if isinstance(f.get("magnitude"), (int, float)) else None,
               "unit": f.get("unit"), "certainty": f.get("certainty"), "evidence": ev_ok, "status": "ok"}
        if rec["present"] and not ev_ok:          # faithfulness gate
            rec.update(present=None, direction=None, magnitude=None, status="unsupported_nulled")
            flags["unsupported_nulled"] += 1
        if rec["magnitude"] is not None:
            nums = " ".join(e["quote"] for e in ev_ok)
            mag = rec["magnitude"]
            cand = {f"{mag:g}", f"{mag:,.0f}", f"{mag:.1f}", f"{mag:.2f}"}
            if not any(c in nums.replace("$", "") for c in cand):
                rec["magnitude"] = None
                rec["status"] = "numeric_nulled"
                flags["numeric_nulled"] += 1
        out[name] = rec
    return out, flags


class MockGenerator:
    """Deterministic stub for the CPU smoke path (addendum P1.5): keyword rules, quotes copied from passages."""
    revision = "mock-v1"

    RULES = {"guidance": r"guidance|outlook", "results_tone": r"record|increase|decrease|declin", "impairment": r"impairment|write-?down",
             "restructuring": r"restructur|layoff|workforce reduction", "liquidity_stress": r"going concern|liquidity",
             "financing_covenant": r"credit agreement|notes due|covenant|indenture", "mna": r"acqui|merger|divest",
             "dividend": r"dividend", "buyback": r"repurchase|buyback", "litigation_regulatory": r"lawsuit|litigation|investigation",
             "management_change": r"chief executive|chief financial|resign|appoint", "risk_factor_change": r"risk factor"}

    def batch(self, qb: dict, passages_list: list, batch_size: int = 16) -> list:
        return [self(qb, p) for p in passages_list]

    def __call__(self, qb: dict, passages: list[dict]) -> dict:
        out = {}
        for name, fdef in qb["fields"].items():
            rx = re.compile(self.RULES.get(name, "$^"), re.I)
            hit = next(((p["chunk_id"], m) for p in passages for m in [rx.search(p["text"])] if m), None)
            if hit:
                cid, m = hit
                txt = next(p["text"] for p in passages if p["chunk_id"] == cid)
                s = max(0, m.start() - 60)
                quote = txt[s:m.end() + 60]
                out[name] = {"present": True, "direction": fdef["directions"][0], "magnitude": None, "unit": None,
                             "certainty": "low", "evidence": [{"chunk_id": cid, "quote": quote}]}
            else:
                out[name] = {"present": False, "direction": None, "magnitude": None, "unit": None, "certainty": "low",
                             "evidence": []}
        return out


def _shim_tokenization_utils() -> None:
    """transformers 5 removed `transformers.tokenization_utils`; lm-format-enforcer (<= 0.11) still imports
    PreTrainedTokenizerBase from it. Alias the module only when it is missing."""
    import importlib
    import sys
    base = importlib.import_module("transformers.tokenization_utils_base")
    try:
        mod = importlib.import_module("transformers.tokenization_utils")
    except ImportError:
        sys.modules["transformers.tokenization_utils"] = base
        return
    if not hasattr(mod, "PreTrainedTokenizerBase"):
        try:
            setattr(mod, "PreTrainedTokenizerBase", base.PreTrainedTokenizerBase)
        except Exception:
            sys.modules["transformers.tokenization_utils"] = base


class QwenGenerator:
    """Qwen2.5-7B-Instruct via transformers; JSON-schema-constrained decoding with lm-format-enforcer when
    available, otherwise schema validation + retry (logged)."""

    def __init__(self, model_id: str, revision: str, schema: dict, max_new_tokens: int = 900, log=None):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        self.revision = revision
        self.tok = AutoTokenizer.from_pretrained(model_id, revision=revision, padding_side="left")
        kw = dict(revision=revision, device_map="auto")
        try:
            self.model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.bfloat16, **kw)
        except TypeError:
            self.model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.bfloat16, **kw)
        self.model.eval()
        self.max_new = max_new_tokens
        self.schema = schema
        self.log = log
        self.constrained = False
        try:
            _shim_tokenization_utils()
            from lmformatenforcer import JsonSchemaParser
            from lmformatenforcer.integrations.transformers import build_transformers_prefix_allowed_tokens_fn
            self._parser = JsonSchemaParser(schema)
            self._fn = build_transformers_prefix_allowed_tokens_fn(self.tok, self._parser)
            self.constrained = True
        except Exception as e:
            if log:
                log.warning(f"lm-format-enforcer unavailable ({e}); using validation + retry")

    def generate(self, prompts: list[str]) -> list[str]:
        import torch
        msgs = [self.tok.apply_chat_template([{"role": "system", "content": SYSTEM}, {"role": "user", "content": p}],
                                             tokenize=False, add_generation_prompt=True) for p in prompts]
        enc = self.tok(msgs, return_tensors="pt", padding=True, truncation=True, max_length=7000).to(self.model.device)
        kw = dict(max_new_tokens=self.max_new, do_sample=False, pad_token_id=self.tok.eos_token_id)
        if self.constrained:
            kw["prefix_allowed_tokens_fn"] = self._fn
        with torch.no_grad():
            out = self.model.generate(**enc, **kw)
        return [self.tok.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True) for o in out]

    @staticmethod
    def _parse(txt: str):
        try:
            return json.loads(txt[txt.find("{"): txt.rfind("}") + 1])
        except Exception:
            return None

    def __call__(self, qb: dict, passages: list[dict]) -> dict:
        return self.batch(qb, [passages])[0]

    def batch(self, qb: dict, passages_list: list, batch_size: int = 16) -> list:
        """Batched generation (left padding); failures are retried one by one with an explicit JSON instruction."""
        import torch
        out: list = [None] * len(passages_list)
        prompts = [build_prompt(qb, p) for p in passages_list]
        order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
        bs, k = batch_size, 0
        while k < len(order):
            idx = order[k:k + bs]
            try:
                txts = self.generate([prompts[i] for i in idx])
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                if bs == 1:
                    raise
                bs = max(1, bs // 2)
                continue
            for i, t in zip(idx, txts):
                out[i] = self._parse(t)
            k += len(idx)
        for i, o in enumerate(out):
            if o is None:
                out[i] = self._parse(self.generate([prompts[i] + "\nReturn valid JSON only."])[0]) or {}
        return out


In [ ]:
%%writefile $REPO/src/secprice/report/__init__.py
"""secprice.report"""


In [ ]:
%%writefile $REPO/src/secprice/report/build.py
"""S24 report: reports/results_summary.md (SPEC.md §21, computed values only) + a self-contained HTML report with
figures embedded + a PDF. Every number in the text is read from an artifact; missing results are stated as such."""
from __future__ import annotations

import base64
import html
import json
from pathlib import Path

import numpy as np
import pandas as pd


def fmt(v, nd=3, col: str = "") -> str:
    if v is None or (isinstance(v, float) and not np.isfinite(v)):
        return "n/a"
    if isinstance(v, (int, np.integer)):
        return str(int(v)) if any(k in str(col).lower() for k in ("year", "fold", "item", "horizon", "lag")) else f"{int(v):,}"
    if isinstance(v, (float, np.floating)):
        c = str(col).lower()
        if float(v).is_integer() and (c == "n" or c.startswith("n_") or c.endswith("_n") or "count" in c or "days" in c
                                      or c in ("observed", "parsed", "firm_years", "lag", "cost_bps", "fold", "horizon")):
            return f"{int(v):,}"
        return f"{v:.{nd}f}"
    return str(v)


def md_table(df: pd.DataFrame | None, max_rows: int = 40, nd: int = 3) -> str:
    if df is None or len(df) == 0:
        return "_(no rows)_"
    d = df.head(max_rows)
    cols = list(d.columns)
    out = ["| " + " | ".join(map(str, cols)) + " |", "|" + "---|" * len(cols)]
    for _, r in d.iterrows():
        out.append("| " + " | ".join(fmt(r[c], nd, c) for c in cols) + " |")
    if len(df) > max_rows:
        out.append(f"\n_{len(df) - max_rows} more rows in the CSV._")
    return "\n".join(out)


def smallest_defensible_claim(f: dict) -> str:
    h1 = f.get("h1") or {}
    lift, lo, hi = h1.get("value"), h1.get("ci_low"), h1.get("ci_high")
    n = h1.get("n_events")
    block = h1.get("block", "walk-forward validation")
    if lift is None:
        return ("The core text-versus-baseline comparison (H1) could not be evaluated in this run; the study supports "
                "only the data-integrity and event-study statements above.")
    if lo is not None and lo > 0:
        return (f"On the common cohort ({fmt(n)} events, {block}), adding SEC-filing text to price, metadata, accounting "
                f"and lexical baselines raised macro-F1 by {fmt(lift)} (95% month-block CI {fmt(lo)} to {fmt(hi)}) under "
                f"the assumed availability/latency policy. This is an out-of-time association with modest predictive "
                f"value, not evidence of causation or of tradable profit.")
    if hi is not None and hi < 0:
        return (f"On the common cohort ({fmt(n)} events, {block}), adding SEC-filing text lowered macro-F1 by "
                f"{fmt(-lift)} (CI {fmt(lo)} to {fmt(hi)}): text did not add out-of-time value beyond the baselines.")
    return (f"On the common cohort ({fmt(n)} events, {block}), the lift from adding SEC-filing text was {fmt(lift)} "
            f"macro-F1 with a 95% CI of {fmt(lo)} to {fmt(hi)} that includes zero: we cannot claim that filing text adds "
            f"reliable out-of-time information beyond the structured baselines. A credible null is a valid result.")


def build_markdown(f: dict) -> str:
    L = []
    a = L.append
    a(f"# SEC filings -> price impact: results summary ({f.get('profile', '?')} profile)")
    a("")
    a(f"Generated {f.get('generated')} | config hash `{f.get('config_hash')}` | code `{f.get('code')}` | "
      f"pre-registration hash `{f.get('prereg_hash', 'not frozen')}`")
    a(f"\n**Timing policy:** {f.get('timing_policy')}")
    a("\n**Reading rules:** association is not causation; prediction is not profitability. Results are "
      "validation-selected, never a global optimum. Power labels: n<30 warning, 30-99 exploratory, wide CI inconclusive.")
    a("\n## Executive summary")
    for b in f.get("exec_bullets", []):
        a(f"- {b}")
    a(f"\n**Smallest defensible claim.** {smallest_defensible_claim(f)} All results are conditioned on a survivor-only "
      "universe (firms listed in 2026), which biases drift and backtest results towards survivors.")
    a("\n## 1. Data integrity")
    a(f.get("universe_text", ""))
    a("\n### Universe by year")
    a(md_table(f.get("firms_per_year")))
    a("\n### Filing coverage (observed vs parsed) and audit reason codes")
    a(md_table(f.get("coverage_by_form")))
    a(md_table(f.get("reason_codes")))
    a("\n### 8-K category availability (taxonomy v1)")
    a(md_table(f.get("category_by_year"), 30))
    a("\n### Timestamp quality")
    a(md_table(f.get("ts_quality")))
    a(f"\n{f.get('timestamp_text', '')}")
    a("\n### Hugging Face dataset audit")
    a(f.get("hf_text", ""))
    a("\n### Limitations of the data")
    for x in f.get("data_limitations", []):
        a(f"- {x}")
    a("\n## 2. Design")
    a(f.get("design_text", ""))
    a(md_table(f.get("event_counts")))
    a("\n## 3. Leakage and provenance audit")
    a(md_table(f.get("leakage_audit"), 30))
    a("\n### Model provenance ledger")
    a(md_table(f.get("ledger"), 30))
    a("\n### Sanity experiments and review triggers")
    a(md_table(f.get("sanity")))
    a("\n## 4. Track A: event study (association only)")
    a(md_table(f.get("track_a_main"), 40))
    a("\n### Drift tests and verdicts")
    a(md_table(f.get("hypothesis_tests"), 30))
    a(f.get("drift_text", ""))
    a("\n## 5. Track B: prediction")
    a(f.get("trackb_text", ""))
    a("\n### Ladder (walk-forward, primary target)")
    a(md_table(f.get("ladder"), 60))
    a("\n### Pre-registered guardrails (log-loss no worse than the class prior; ECE <= 0.10; 1 = met)")
    a(md_table(f.get("guardrails"), 40))
    a("\n### Hypotheses H1-H4 (with CIs)")
    a(md_table(f.get("hypotheses_core")))
    a("\n### Tuning: default vs tuned")
    a(md_table(f.get("tuning_before_after"), 40))
    a("\n### Development holdout")
    a(md_table(f.get("dev_results"), 40))
    a("\n### Locked test")
    a(f.get("locked_text", ""))
    a(md_table(f.get("locked_results"), 40))
    a("\n### Selective prediction (coverage)")
    a(md_table(f.get("selective"), 20))
    a("\n### T_info vs T_trade")
    a(md_table(f.get("tinfo_vs_ttrade"), 20))
    a("\n### Backtest (economic-relevance check; not an investable strategy)")
    a(f.get("backtest_text", ""))
    a(md_table(f.get("backtest"), 20))
    a("\n### Index membership: S&P 500 vs non-S&P 500 vs combined")
    a(f.get("sp_text", ""))
    a("\n**Composition**")
    a(md_table(f.get("sp_composition"), 10))
    a("\n**Prediction (walk-forward, primary target): the combined model scored per bucket**")
    a(md_table(f.get("sp_scores_wf"), 30))
    a("\n**Pooled vs bucket-only models (paired, identical rows)**")
    a(md_table(f.get("sp_pooled_vs_own"), 10))
    a("\n**Development holdout / locked test by bucket**")
    a(md_table(f.get("sp_scores_later"), 30))
    a("\n**Track A by bucket (association only)**")
    a(md_table(f.get("sp_track_a"), 30))
    a("\n**SUE-conditioned drift by bucket**")
    a(md_table(f.get("sp_drift"), 10))
    a("\n**Backtest by bucket (lowest cost, no lag)**")
    a(md_table(f.get("sp_backtest"), 20))
    a("\n## 6. Track C: content validity")
    a(f.get("trackc_text", ""))
    a(md_table(f.get("trackc"), 30))
    a("\n## 7. Fit diagnostics")
    a(md_table(f.get("fit_diag"), 40))
    a(f.get("optim_text", ""))
    a("\n## 8. Extensions run")
    a(md_table(f.get("extensions"), 20))
    a("\n## 9. Limitations, resource use, decision log")
    for x in f.get("limitations", []):
        a(f"- {x}")
    a(md_table(f.get("resources"), 40))
    a(f"\nDecision log: `{f.get('decision_log', 'docs/decision_log.md')}`")
    a("\n## Figures")
    for fig in f.get("figures", []):
        st = "" if fig["status"] == "ok" else f" - **{fig['status']}**"
        a(f"- **{fig['id']}** {fig['title']}{st}. {fig['caption']}")
    return "\n".join(L)


def build_html(md_text: str, figures: list[dict]) -> str:
    try:
        import markdown
        body = markdown.markdown(md_text, extensions=["tables"])
    except Exception:
        body = "<pre>" + html.escape(md_text) + "</pre>"
    imgs = []
    for fg in figures:
        if fg.get("path") and Path(fg["path"]).exists():
            b64 = base64.b64encode(Path(fg["path"]).read_bytes()).decode()
            imgs.append(f"<figure><img src='data:image/png;base64,{b64}' style='max-width:100%'/>"
                        f"<figcaption><b>{fg['id']}</b> {html.escape(fg['title'])}. {html.escape(fg['caption'])}</figcaption></figure>")
    css = ("body{font-family:Segoe UI,Arial,sans-serif;max-width:1100px;margin:auto;padding:16px;line-height:1.45;color:#222}"
           "table{border-collapse:collapse;font-size:12px;margin:8px 0}td,th{border:1px solid #ccc;padding:3px 6px}"
           "th{background:#f3f3f3}figure{margin:18px 0;border-top:1px solid #eee;padding-top:8px}"
           "figcaption{font-size:12px;color:#444}code{background:#f5f5f5;padding:1px 3px}")
    return (f"<!doctype html><html><head><meta charset='utf-8'><title>SEC filings price impact report</title>"
            f"<style>{css}</style></head><body>{body}<h2>Figures</h2>{''.join(imgs)}</body></html>")


def build_pdf(md_text: str, figures: list[dict], path: Path) -> str | None:
    try:
        from reportlab.lib.pagesizes import A4
        from reportlab.lib.styles import getSampleStyleSheet
        from reportlab.lib.units import cm
        from reportlab.platypus import Image, PageBreak, Paragraph, Preformatted, SimpleDocTemplate, Spacer
    except Exception as e:
        return f"reportlab unavailable: {e}"
    ss = getSampleStyleSheet()
    story = []
    table_buf: list[str] = []

    def flush():
        if table_buf:
            story.append(Preformatted("\n".join(x[:180] for x in table_buf[:60]), ss["Code"], maxLineLength=180))
            table_buf.clear()

    for line in md_text.splitlines():
        if line.startswith("|"):
            table_buf.append(line)
            continue
        flush()
        esc = html.escape(line).replace("**", "")
        if line.startswith("# "):
            story.append(Paragraph(esc[2:], ss["Title"]))
        elif line.startswith("## "):
            story.append(Paragraph(esc[3:], ss["Heading2"]))
        elif line.startswith("### "):
            story.append(Paragraph(esc[4:], ss["Heading3"]))
        elif line.strip():
            story.append(Paragraph(esc, ss["BodyText"]))
        else:
            story.append(Spacer(1, 4))
    flush()
    story.append(PageBreak())
    for fg in figures:
        if fg.get("path") and Path(fg["path"]).exists():
            story.append(Paragraph(f"{fg['id']} {html.escape(fg['title'])}", ss["Heading3"]))
            story.append(Image(fg["path"], width=17 * cm, height=9 * cm, kind="proportional"))
            story.append(Paragraph(html.escape(fg["caption"]), ss["BodyText"]))
            story.append(Spacer(1, 10))
    SimpleDocTemplate(str(path), pagesize=A4, leftMargin=1.5 * cm, rightMargin=1.5 * cm, topMargin=1.5 * cm,
                      bottomMargin=1.5 * cm).build(story)
    return None


In [ ]:
%%writefile $REPO/src/secprice/runner.py
"""Stage registry, dependency resolution, skip-if-done, status (SPEC.md §17)."""
from __future__ import annotations

import json
import random
import time
import traceback
from dataclasses import dataclass, field
from pathlib import Path
from typing import Callable

import numpy as np

from .config import config_hash, subset_hash
from .logging_utils import StageIncomplete, get_logger
from .storage import Storage, utcnow, write_json


@dataclass
class StageSpec:
    id: str
    name: str
    part: str                      # CPU / GPU / CPU-GPU
    deps: list[str]
    config_keys: list[str]
    func: Callable
    outputs: list[str] = field(default_factory=list)
    assertions: list[str] = field(default_factory=list)
    card: str = ""                 # how to read the outputs (stage card)


REGISTRY: dict[str, StageSpec] = {}


def stage(id: str, name: str, part: str, deps: list[str], config_keys: list[str], outputs=None,
          assertions=None, card: str = ""):
    def deco(fn):
        REGISTRY[id] = StageSpec(id, name, part, list(deps), list(config_keys), fn, outputs or [], assertions or [], card)
        return fn
    return deco


def set_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed % (2**32 - 1))
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except Exception:
        pass


class Context:
    """Passed to each stage function: config, storage, logger, resolved dependency dirs."""

    def __init__(self, runner: "Runner", spec: StageSpec, stage_hash: str):
        self.runner = runner
        self.cfg = runner.cfg
        self.storage: Storage = runner.storage
        self.spec = spec
        self.stage_id = spec.id
        self.stage_hash = stage_hash
        self.out = runner.storage.stage_dir(spec.id, stage_hash)
        self.log = get_logger(spec.id, runner.storage.profile_root / "logs")
        self.started = utcnow()
        self.warnings: list[str] = []
        self.services = runner.services

    def dep(self, stage_id: str) -> Path:
        h = self.runner.stage_hash(stage_id)
        p = self.storage.profile_root / stage_id / h
        if not (p / "_SUCCESS").exists():
            raise RuntimeError(f"dependency {stage_id} has no _SUCCESS for hash {h}; run it first")
        return p

    def dep_or_none(self, stage_id: str) -> Path | None:
        try:
            return self.dep(stage_id)
        except Exception:
            return None

    def input_hashes(self) -> dict:
        """Upstream stage hashes and the sha256 of every upstream parquet artifact (SPEC.md §16 'input artifact hashes')."""
        if getattr(self, "_inputs", None) is None:
            out = {}
            for d in self.spec.deps:
                p = self.dep_or_none(d)
                if p is None:
                    continue
                arts = {}
                for m in sorted(p.glob("*.manifest.json")):
                    try:
                        arts[m.name[: -len(".manifest.json")]] = json.loads(m.read_text(encoding="utf-8")).get("sha256")
                    except (OSError, ValueError):
                        arts[m.name[: -len(".manifest.json")]] = None
                out[d] = {"stage_hash": p.name, "artifacts": arts}
            self._inputs = out
        return self._inputs

    def write(self, df, name: str, inputs: dict | None = None) -> Path:
        return self.storage.write_parquet(df, self.out / name, self.stage_id, self.stage_hash, self.started,
                                          inputs if inputs is not None else self.input_hashes(), self.warnings)

    def warn(self, msg: str) -> None:
        self.warnings.append(msg)
        self.log.warning(msg)


class Runner:
    def __init__(self, cfg: dict, storage: Storage | None = None, services: dict | None = None):
        self.cfg = cfg
        self.storage = storage or Storage(cfg)
        self.services = services or {}
        self._hash_cache: dict[str, str] = {}
        self.log = get_logger("runner", self.storage.profile_root / "logs")
        self.timings_path = self.storage.manifests / "run_manifest.json"

    def stage_hash(self, stage_id: str) -> str:
        if stage_id in self._hash_cache:
            return self._hash_cache[stage_id]
        spec = REGISTRY[stage_id]
        dep_hashes = {d: self.stage_hash(d) for d in spec.deps}
        h = subset_hash(self.cfg, spec.config_keys, extra={"deps": dep_hashes, "stage": stage_id})
        self._hash_cache[stage_id] = h
        return h

    def order(self, stage_ids: list[str]) -> list[str]:
        seen: list[str] = []

        def visit(s: str):
            if s in seen:
                return
            for d in REGISTRY[s].deps:
                visit(d)
            seen.append(s)

        for s in stage_ids:
            visit(s)
        return seen

    def status(self) -> list[dict]:
        rows = []
        for sid in sorted(REGISTRY, key=_stage_key):
            h = self.stage_hash(sid)
            if self.storage.is_success(sid, h):
                st = "done"
            elif self.storage.other_successes(sid, h):
                st = "stale"
            else:
                st = "pending"
            rows.append({"stage": sid, "name": REGISTRY[sid].name, "part": REGISTRY[sid].part, "hash": h, "status": st})
        return rows

    def run(self, stage_ids: list[str], force: bool | list[str] = False, with_deps: bool = False) -> dict:
        todo = self.order(stage_ids) if with_deps else list(stage_ids)
        results = {}
        manifest = _load_json(self.timings_path) or {"stages": {}}
        manifest.update({"profile": self.cfg["run_profile"], "config_hash": config_hash(self.cfg), **_run_provenance(self.cfg)})
        for sid in todo:
            spec = REGISTRY[sid]
            h = self.stage_hash(sid)
            forced = force is True or (isinstance(force, list) and sid in force)
            if self.storage.is_success(sid, h) and not forced:
                self.log.info(f"{sid} {spec.name}: done (hash {h}), skipped")
                results[sid] = "skipped"
                continue
            for d in spec.deps:
                if not self.storage.is_success(d, self.stage_hash(d)):
                    raise RuntimeError(f"{sid} needs {d} first (no _SUCCESS); run {d} or pass with_deps=True")
            set_seeds(int(self.cfg["seed"]))
            ctx = Context(self, spec, h)
            self.log.info(f"=== {sid} {spec.name} [{spec.part}] hash {h} ===")
            t0 = time.time()
            gpu_t0 = _gpu_time()
            try:
                info = spec.func(ctx) or {}
            except StageIncomplete as e:
                self.log.warning(f"{sid} INCOMPLETE (resumable): {e}")
                manifest["stages"][sid] = {"status": "incomplete", "error": str(e)[:500], "at": utcnow()}
                write_json(self.timings_path, manifest)
                raise
            except Exception as e:
                self.log.error(f"{sid} failed: {e}\n{traceback.format_exc()}")
                manifest["stages"][sid] = {"status": "failed", "error": str(e)[:500], "at": utcnow()}
                write_json(self.timings_path, manifest)
                raise
            wall = time.time() - t0
            info = {"wall_s": round(wall, 1), "gpu_active_s": _gpu_time() - gpu_t0, "warnings": ctx.warnings, **info}
            self.storage.mark_success(sid, h, info)
            manifest["stages"][sid] = {"status": "done", "hash": h, "wall_s": round(wall, 1), "at": utcnow(),
                                       "summary": {k: v for k, v in info.items() if k != "warnings"}}
            write_json(self.timings_path, manifest)
            _write_stage_card(self.storage, spec, h, info)
            self.log.info(f"{sid} done in {wall:.1f}s")
            results[sid] = "done"
        return results


def _run_provenance(cfg: dict) -> dict:
    """run_manifest.json fields (SPEC.md §19; addendum P0.3): code version, model revisions, data-source versions and the
    timing policy (public lag and decision latency are assumptions and travel with every run)."""
    from .config import load_side_config
    from .storage import code_hash, git_commit
    out = {"git_commit": git_commit(), "code_hash": code_hash(), "timing_policy": dict(cfg.get("timing", {}))}
    try:
        rost = load_side_config(cfg, "models.yaml")
        revs = {}
        for sec in ("encoders", "reranker", "llm", "judge", "student", "ner"):
            for k, v in (rost.get(sec) or {}).items():
                if "model_id" in v:
                    revs[k] = f"{v['model_id']}@{v.get('revision')}"
                elif "revisions" in v:
                    revs[k] = {"template": v.get("model_id_template"), "revisions": v["revisions"]}
        out["model_revisions"] = revs
        out["lexicon"] = rost.get("lexical")
    except Exception as e:                         # never block a run on provenance bookkeeping
        out["model_revisions"] = f"unavailable ({type(e).__name__})"
    m, u, h = cfg.get("market", {}), cfg.get("universe", {}), cfg.get("hf", {})
    out["data_sources"] = {"hf_dataset": f"{h.get('repo')}@{h.get('revision')}",
                           "sp500_membership": f"{u.get('sp500_repo')}@{u.get('sp500_commit')}",
                           "prices": {"provider": m.get("price_provider"), "local_ohlcv_subdir": m.get("local_ohlcv_subdir"),
                                      "yfinance": m.get("yfinance"), "frozen_file": "inputs/yf_frozen.parquet"},
                           "factors": m.get("factors"), "sec": "EDGAR submissions JSON, archives (SGML), companyfacts"}
    return out


def _gpu_time() -> float:
    # GPU-active time is approximated by wall time while CUDA is in use; not a credit measurement (SPEC.md §13.3).
    try:
        import torch
        if torch.cuda.is_available():
            return round(time.time(), 1)
    except Exception:
        pass
    return 0.0


def _load_json(p: Path):
    try:
        return json.loads(Path(p).read_text(encoding="utf-8"))
    except Exception:
        return None


def _stage_key(s: str):
    return (0, int(s[1:])) if s[1:].isdigit() else (1, s)


def _write_stage_card(storage: Storage, spec: StageSpec, h: str, info: dict) -> None:
    """docs/stage_cards/<stage>.md equivalent written next to the artifacts (purpose, outputs, assertions, how to read)."""
    lines = [f"# {spec.id} {spec.name}", "", f"- Part: {spec.part}", f"- Depends on: {', '.join(spec.deps) or 'none'}",
             f"- Stage hash: `{h}`", f"- Outputs: {', '.join(spec.outputs) or 'see folder'}",
             f"- Assertions: {'; '.join(spec.assertions) or 'none'}", "", "## How to read", spec.card or "-", "",
             "## Run summary", "```json", json.dumps({k: v for k, v in info.items()}, indent=2, default=str)[:6000], "```"]
    d = storage.profile_root / "stage_cards"
    d.mkdir(parents=True, exist_ok=True)
    (d / f"{spec.id}.md").write_text("\n".join(lines), encoding="utf-8")


In [ ]:
%%writefile $REPO/src/secprice/sec/__init__.py
"""secprice.sec"""


In [ ]:
%%writefile $REPO/src/secprice/sec/archives.py
"""EDGAR archive URLs and the SGML full-submission parser (headers, documents, exhibits)."""
from __future__ import annotations

import re
from dataclasses import dataclass, field

ARCH = "https://www.sec.gov/Archives/edgar/data/{cik}/{nodash}/"


def nodash(acc: str) -> str:
    return acc.replace("-", "")


def doc_url(cik: int, acc: str, doc: str) -> str:
    return ARCH.format(cik=int(cik), nodash=nodash(acc)) + doc


def full_txt_url(cik: int, acc: str) -> str:
    return ARCH.format(cik=int(cik), nodash=nodash(acc)) + f"{acc}.txt"


def index_json_url(cik: int, acc: str) -> str:
    return ARCH.format(cik=int(cik), nodash=nodash(acc)) + "index.json"


def index_headers_url(cik: int, acc: str) -> str:
    return ARCH.format(cik=int(cik), nodash=nodash(acc)) + f"{acc}-index-headers.html"


@dataclass
class SgmlDocument:
    type: str
    sequence: str
    filename: str
    description: str
    text: str


@dataclass
class SgmlSubmission:
    header: str
    acceptance_raw: str | None
    form_type: str | None
    period: str | None
    filed: str | None
    items: list[str] = field(default_factory=list)
    item_titles: list[str] = field(default_factory=list)
    documents: list[SgmlDocument] = field(default_factory=list)
    truncated: bool = False


_DOC_RE = re.compile(r"<DOCUMENT>(.*?)(?:</DOCUMENT>|\Z)", re.S | re.I)
_TAG = lambda t: re.compile(rf"<{t}>\s*([^\n<]*)", re.I)  # noqa: E731
_TYPE, _SEQ, _FN, _DESC = _TAG("TYPE"), _TAG("SEQUENCE"), _TAG("FILENAME"), _TAG("DESCRIPTION")
_TEXT_RE = re.compile(r"<TEXT>(.*?)(?:</TEXT>|\Z)", re.S | re.I)
_ITEMS_RE = re.compile(r"<ITEMS>\s*([0-9.]+)", re.I)
_ITEM_INFO_RE = re.compile(r"ITEM INFORMATION:\s*(.+)", re.I)
_SKIP_TYPES = re.compile(r"^(GRAPHIC|ZIP|EXCEL|PDF|XML|JSON|EX-101|EX-27|EX-100|EX-104|EX-102|COVER)", re.I)


def _first(rx: re.Pattern, block: str) -> str:
    m = rx.search(block[:2000])
    return m.group(1).strip() if m else ""


def parse_sgml(raw: bytes | str, truncated: bool = False) -> SgmlSubmission:
    text = raw.decode("utf-8", errors="replace") if isinstance(raw, (bytes, bytearray)) else raw
    hend = text.find("<DOCUMENT>")
    header = text[: hend if hend > 0 else 20000]
    acc = re.search(r"<ACCEPTANCE-DATETIME>\s*(\d{14})|ACCEPTANCE-DATETIME:\s*(\d{14})", header)
    form = re.search(r"CONFORMED SUBMISSION TYPE:\s*(\S+)|<TYPE>\s*(\S+)", header)
    period = re.search(r"CONFORMED PERIOD OF REPORT:\s*(\d{8})|<PERIOD>\s*(\d{8})", header)
    filed = re.search(r"FILED AS OF DATE:\s*(\d{8})|<FILING-DATE>\s*(\d{8})", header)
    items = [m.group(1) for m in _ITEMS_RE.finditer(header)]
    titles = [m.group(1).strip() for m in _ITEM_INFO_RE.finditer(header)]
    docs = []
    for m in _DOC_RE.finditer(text):
        block = m.group(1)
        t = _TYPE.search(block)
        dtype = t.group(1).strip() if t else ""
        if _SKIP_TYPES.match(dtype):
            continue
        tm = _TEXT_RE.search(block)
        body = tm.group(1) if tm else ""
        docs.append(SgmlDocument(dtype, _first(_SEQ, block), _first(_FN, block), _first(_DESC, block), body))
    g = lambda m: (m.group(1) or m.group(2)) if m else None  # noqa: E731
    return SgmlSubmission(header=header, acceptance_raw=g(acc), form_type=g(form), period=g(period), filed=g(filed),
                          items=items, item_titles=titles, documents=docs, truncated=truncated)


In [ ]:
%%writefile $REPO/src/secprice/sec/client.py
"""Thin SEC EDGAR client: declared User-Agent from env, global rate limit, cache, backoff, block detection
(SPEC.md hard rule 13, §13.5). No proxy rotation or evasion; a block pauses the run and is reported."""
from __future__ import annotations

import hashlib
import json
import os
import threading
import zlib
import time
from pathlib import Path
from urllib.parse import urlparse

import requests


_ZMAGIC = b"ZC1\x00"          # compressed cache entry marker (older raw entries are still read)


class SECBlockedError(RuntimeError):
    """Raised after repeated 403s: the run pauses and the owner is told (never evaded)."""


class UserAgentMissing(RuntimeError):
    pass


class RateLimiter:
    """Token bucket shared by all download threads."""

    def __init__(self, rps: float):
        self.interval = 1.0 / float(rps)
        self.lock = threading.Lock()
        self.next_t = time.monotonic()
        self.pause_until = 0.0

    def wait(self) -> None:
        with self.lock:
            now = time.monotonic()
            start = max(now, self.next_t, self.pause_until)
            self.next_t = start + self.interval
        delay = start - time.monotonic()
        if delay > 0:
            time.sleep(delay)

    def pause(self, seconds: float) -> None:
        with self.lock:
            self.pause_until = max(self.pause_until, time.monotonic() + seconds)


def resolve_user_agent(env_name: str = "SEC_USER_AGENT") -> str:
    ua = os.environ.get(env_name, "").strip()
    if not ua or "@" not in ua:
        raise UserAgentMissing(
            f"Set {env_name} to 'Your Name your.email@example.com' (Colab secret or environment variable). "
            "SEC fair access requires a declared contact; it is never hard-coded.")
    return ua


class SECClient:
    def __init__(self, user_agent: str, max_rps: float = 6, cache_dir: str | os.PathLike | None = None,
                 cache_days: float = 30, timeout_s: float = 60, max_retries: int = 6, backoff_base_s: float = 2.0,
                 blocked_403_limit: int = 20, log=None, cache_max_bytes: int = 0):
        self.ua = user_agent
        self.cache_max_bytes = int(cache_max_bytes or 0)   # 0 = cache every response; >0 = skip larger bodies (disk budget)
        self.limiter = RateLimiter(max_rps)
        self.cache_dir = Path(cache_dir) if cache_dir else None
        if self.cache_dir:
            self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.cache_s = float(cache_days) * 86400
        self.timeout = timeout_s
        self.max_retries = int(max_retries)
        self.backoff = float(backoff_base_s)
        self.blocked_limit = int(blocked_403_limit)
        self._consec_403 = 0
        self._lock = threading.Lock()
        self.log = log
        self.stats = {"requests": 0, "cache_hits": 0, "429": 0, "5xx": 0, "403": 0, "404": 0, "bytes": 0, "truncated": 0}
        self._local = threading.local()

    @classmethod
    def from_config(cls, cfg: dict, cache_dir, log=None) -> "SECClient":
        s = cfg["sec"]
        return cls(resolve_user_agent(s["user_agent_env"]), s["max_rps"], cache_dir, s["cache_days"], s["timeout_s"],
                   s["max_retries"], s["backoff_base_s"], s.get("blocked_403_limit", 20), log, s.get("cache_max_bytes", 0))

    def _session(self) -> requests.Session:
        sess = getattr(self._local, "sess", None)
        if sess is None:
            sess = requests.Session()
            sess.headers.update({"User-Agent": self.ua, "Accept-Encoding": "gzip, deflate"})
            self._local.sess = sess
        return sess

    def _cache_path(self, url: str) -> Path | None:
        if not self.cache_dir:
            return None
        h = hashlib.sha1(url.encode()).hexdigest()
        return self.cache_dir / h[:2] / f"{h}.bin"

    def get(self, url: str, max_bytes: int | None = None, use_cache: bool = True) -> bytes | None:
        """Bytes of the URL, None on 404. Raises SECBlockedError on sustained 403."""
        key = url if not max_bytes or max_bytes >= 1_000_000 else f"{url}#prefix{max_bytes}"   # never mix a prefix with the full file
        cp = self._cache_path(key) if use_cache else None
        if cp and cp.exists() and (time.time() - cp.stat().st_mtime) < self.cache_s:
            with self._lock:
                self.stats["cache_hits"] += 1
            data = cp.read_bytes()
            if data.startswith(_ZMAGIC):
                data = zlib.decompress(data[len(_ZMAGIC):])
            return None if data == b"__404__" else data
        attempt = 0
        while True:
            attempt += 1
            self.limiter.wait()
            try:
                r = self._session().get(url, timeout=self.timeout, stream=True)
                status = r.status_code
                if status == 200:
                    buf = bytearray()
                    truncated = False
                    for chunk in r.iter_content(1 << 16):
                        buf.extend(chunk)
                        if max_bytes and len(buf) >= max_bytes:
                            truncated = True
                            break
                    r.close()
                    data = bytes(buf[:max_bytes] if max_bytes else buf)
                    with self._lock:
                        self.stats["requests"] += 1
                        self.stats["bytes"] += len(data)
                        self.stats["truncated"] += int(truncated)
                        self._consec_403 = 0
                    if cp and (not self.cache_max_bytes or len(data) <= self.cache_max_bytes):
                        cp.parent.mkdir(parents=True, exist_ok=True)
                        tmp = cp.with_suffix(f".{os.getpid()}.{threading.get_ident()}.tmp")
                        tmp.write_bytes(_ZMAGIC + zlib.compress(data, 6))      # filings are text: ~5-10x smaller on disk
                        os.replace(tmp, cp)
                    return data
                r.close()
            except (requests.ConnectionError, requests.Timeout, requests.exceptions.ChunkedEncodingError) as e:
                status = -1
                if attempt >= self.max_retries:
                    raise RuntimeError(f"network error after {attempt} attempts: {url}: {e}")
            with self._lock:
                self.stats["requests"] += 1
            if status == 404:
                with self._lock:
                    self.stats["404"] += 1
                if cp:
                    cp.parent.mkdir(parents=True, exist_ok=True)
                    cp.write_bytes(b"__404__")
                return None
            if status == 403:
                with self._lock:
                    self.stats["403"] += 1
                    self._consec_403 += 1
                    blocked = self._consec_403 >= self.blocked_limit
                if blocked:
                    raise SECBlockedError(f"{self._consec_403} consecutive 403 responses from SEC (last: {url}). "
                                          "Pausing per fair-access policy; check the User-Agent and request rate.")
                self.limiter.pause(30 * attempt)
            elif status == 429 or status >= 500 or status == -1:
                with self._lock:
                    self.stats["429" if status == 429 else "5xx"] += 1
                self.limiter.pause(self.backoff ** min(attempt, 6))
            else:
                if attempt >= 2:
                    return None
            if attempt >= self.max_retries:
                if status == 403:
                    return None
                raise RuntimeError(f"HTTP {status} after {attempt} attempts: {url}")

    def get_json(self, url: str, use_cache: bool = True):
        b = self.get(url, use_cache=use_cache)
        return None if b is None else json.loads(b)


class FixtureSECClient:
    """Offline client: serves recorded/synthetic responses from tests/fixtures/sec/<host>/<path>."""

    def __init__(self, root: str | os.PathLike):
        self.root = Path(root)
        self.stats = {"requests": 0, "cache_hits": 0, "429": 0, "5xx": 0, "403": 0, "404": 0, "bytes": 0, "truncated": 0}
        self.ua = "fixture"

    def _path(self, url: str) -> Path:
        u = urlparse(url)
        return self.root / u.netloc / u.path.lstrip("/")

    def get(self, url: str, max_bytes: int | None = None, use_cache: bool = True) -> bytes | None:
        p = self._path(url)
        self.stats["requests"] += 1
        if not p.exists():
            self.stats["404"] += 1
            return None
        data = p.read_bytes()
        self.stats["bytes"] += len(data)
        return data[:max_bytes] if max_bytes else data

    def get_json(self, url: str, use_cache: bool = True):
        b = self.get(url)
        return None if b is None else json.loads(b)


In [ ]:
%%writefile $REPO/src/secprice/sec/companyfacts.py
"""XBRL company facts (data.sec.gov/api/xbrl/companyfacts) -> long fact table for selected tags (S06)."""
from __future__ import annotations

import pandas as pd

CF_URL = "https://data.sec.gov/api/xbrl/companyfacts/CIK{cik:010d}.json"
FACT_COLS = ["cik", "accession", "taxonomy", "tag", "unit", "value", "period_start", "period_end", "fy", "fp",
             "form", "filed", "frame"]


def fetch_companyfacts(client, cik: int):
    return client.get_json(CF_URL.format(cik=int(cik)))


def facts_frame(js: dict, tags: list[str], dei_tags: list[str] | None = None) -> pd.DataFrame:
    if not js:
        return pd.DataFrame(columns=FACT_COLS)
    rows = []
    cik = int(js.get("cik", 0))
    want = {("us-gaap", t) for t in tags} | {("dei", t) for t in (dei_tags or [])}
    for tax, block in (js.get("facts") or {}).items():
        for tag, spec in block.items():
            if (tax, tag) not in want:
                continue
            for unit, vals in (spec.get("units") or {}).items():
                for v in vals:
                    rows.append((cik, v.get("accn"), tax, tag, unit, v.get("val"), v.get("start"), v.get("end"),
                                 v.get("fy"), v.get("fp"), v.get("form"), v.get("filed"), v.get("frame")))
    df = pd.DataFrame(rows, columns=FACT_COLS)
    if len(df):
        df["value"] = pd.to_numeric(df["value"], errors="coerce")
    return df


In [ ]:
%%writefile $REPO/src/secprice/sec/submissions.py
"""Submissions JSON (data.sec.gov/submissions): firm metadata and the full filing list, including the older
paged files (SPEC.md §13.5, S04)."""
from __future__ import annotations

import pandas as pd

SUB_URL = "https://data.sec.gov/submissions/CIK{cik:010d}.json"
SUB_PAGE_URL = "https://data.sec.gov/submissions/{name}"
TICKERS_URL = "https://www.sec.gov/files/company_tickers_exchange.json"

_COLS = {
    "accessionNumber": "accession", "filingDate": "filing_date", "reportDate": "report_date",
    "acceptanceDateTime": "acceptance_raw", "act": "act", "form": "form", "fileNumber": "file_number",
    "items": "items_raw", "size": "size", "isXBRL": "is_xbrl", "isInlineXBRL": "is_inline_xbrl",
    "primaryDocument": "primary_doc", "primaryDocDescription": "primary_doc_desc",
}


def _frame(block: dict) -> pd.DataFrame:
    if not block:
        return pd.DataFrame(columns=list(_COLS.values()))
    df = pd.DataFrame({k: block.get(k, [None] * len(block.get("accessionNumber", []))) for k in _COLS})
    return df.rename(columns=_COLS)


def fetch_submissions(client, cik: int) -> tuple[dict, pd.DataFrame]:
    js = client.get_json(SUB_URL.format(cik=int(cik)))
    if js is None:
        return {}, pd.DataFrame(columns=list(_COLS.values()) + ["cik"])
    frames = [_frame(js.get("filings", {}).get("recent", {}))]
    for f in js.get("filings", {}).get("files", []) or []:
        page = client.get_json(SUB_PAGE_URL.format(name=f["name"]))
        if page:
            frames.append(_frame(page))
    df = pd.concat(frames, ignore_index=True)
    df["cik"] = int(cik)
    df = df.drop_duplicates("accession").reset_index(drop=True)
    meta = {k: js.get(k) for k in ("cik", "name", "tickers", "exchanges", "sic", "sicDescription", "formerNames",
                                   "fiscalYearEnd", "stateOfIncorporation", "entityType", "category", "ein")}
    meta["cik"] = int(cik)
    return meta, df


def fetch_ticker_map(client) -> pd.DataFrame:
    js = client.get_json(TICKERS_URL)
    if not js:
        return pd.DataFrame(columns=["cik", "name", "ticker", "exchange"])
    df = pd.DataFrame(js["data"], columns=js["fields"])
    df["ticker"] = df["ticker"].astype(str).str.upper()
    return df


def norm_ticker(t: str) -> str:
    """Our tickers use '.' for share classes (BRK.B); SEC maps use '-' (BRK-B)."""
    return str(t).upper().replace(".", "-").strip()


In [ ]:
%%writefile $REPO/src/secprice/splits/__init__.py
"""secprice.splits"""


In [ ]:
%%writefile $REPO/src/secprice/splits/leakage.py
"""Leakage checklist (24 items, SPEC.md §6.3) -> leakage_audit.csv, plus the four planted-leak self-tests
(future-return feature, same-event EAR, later-cluster SUE, future-document retrieval). All must be caught."""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..features.registry import availability_violations, ear_rule_violations, sue_rule_violations


def _row(i, item, status, detail):
    return {"item": i, "check": item, "status": status, "detail": detail}


def planted_leaks(feat: pd.DataFrame, events: pd.DataFrame, targets: pd.DataFrame, sue_src: pd.DataFrame,
                  retrieval_guard) -> list[dict]:
    """Inject each leak into a copy and confirm the corresponding guard catches it."""
    res = []
    ev = events.copy()
    # 1. future-return feature: availability = exit of T_trade h=5 (after decision)
    f1 = feat[["event_id"]].copy()
    tt = targets[(targets.target_def == "T_trade") & (targets.horizon == targets.horizon.max())][["event_id", "exit_date"]]
    f1 = f1.merge(tt, on="event_id", how="left")
    f1["px_planted_future_ret"] = 0.0
    f1["_avail_price"] = (pd.to_datetime(f1["exit_date"]) + pd.Timedelta(hours=20)).dt.tz_localize("America/New_York")
    f1 = f1.drop(columns="exit_date")
    dec = ev.set_index("event_id").loc[f1.event_id, "decision_time"]
    v = availability_violations(f1, dec.reset_index(drop=True))
    res.append({"leak": "future_return_feature", "caught": bool(len(v) > 0), "n_flagged": int(len(v))})
    # 2. same-event EAR
    f2 = ev[["event_id", "cluster_id", "decision_time"]].copy()
    if f2.cluster_id.notna().sum() == 0:
        f2.loc[f2.index[:1], "cluster_id"] = "C_planted"
        ev.loc[ev.index[:1], "cluster_id"] = "C_planted"
    f2["ear_src_cluster"] = f2["cluster_id"]
    f2["_avail_ear"] = pd.to_datetime(f2["decision_time"], utc=True) - pd.Timedelta(minutes=1)
    v2 = ear_rule_violations(f2.drop(columns=["cluster_id", "decision_time"]), ev)
    res.append({"leak": "same_event_EAR", "caught": bool(len(v2) > 0), "n_flagged": int(len(v2))})
    # 3. later-cluster SUE: a release event fed by its own cluster's later periodic filing
    pairs = ev[ev.cluster_role == "periodic_after_release"][["cluster_id", "accessions", "public_available_time"]]
    rel = ev[ev.earnings_type == "earnings_announcement_verified"][["event_id", "cluster_id"]]
    planted = rel.merge(pairs, on="cluster_id")
    if planted.empty:          # synthesize a pair when the sample has none (smoke)
        e0 = ev.iloc[[0]].copy()
        e1 = ev.iloc[[0]].copy()
        e0["earnings_type"], e0["cluster_id"] = "earnings_announcement_verified", "C_syn"
        e1["event_id"], e1["accessions"], e1["cluster_id"] = "syn_later", [["syn-later-acc"]], "C_syn"
        e1["cluster_role"] = "periodic_after_release"
        e1["decision_time"] = pd.to_datetime(e0["decision_time"]) + pd.Timedelta(days=30)
        ev2 = pd.concat([ev, e1], ignore_index=True)
        ev2.loc[ev2.event_id == e0.event_id.iloc[0], ["earnings_type", "cluster_id"]] = ["earnings_announcement_verified", "C_syn"]
        src = pd.DataFrame({"event_id": [e0.event_id.iloc[0]], "source_accession": ["syn-later-acc"],
                            "source_public_time": pd.to_datetime(e0["decision_time"]).values - np.timedelta64(1, "h")})
        v3 = sue_rule_violations(src, ev2)
    else:
        p = planted.iloc[0]
        src = pd.DataFrame({"event_id": [p.event_id], "source_accession": [p.accessions[0]],
                            "source_public_time": [p.public_available_time]})
        v3 = sue_rule_violations(src, ev)
    res.append({"leak": "later_cluster_SUE", "caught": bool(len(v3) > 0), "n_flagged": int(len(v3))})
    # 4. future-document retrieval
    caught = retrieval_guard()
    res.append({"leak": "future_document_retrieval", "caught": bool(caught), "n_flagged": int(bool(caught))})
    return res


def audit(events, feat, targets, splits, sue_src, universe, cfg, extra: dict) -> pd.DataFrame:
    rows = []
    wf = splits[splits.scheme == "walk_forward"]
    ev_dates = events.set_index("event_id")["decision_date"]
    tmax = targets[targets.horizon == targets.horizon.max()].groupby("event_id")["exit_date"].max()
    bad_purge, date_overlap = 0, 0
    for f, g in wf.groupby("fold"):
        tr = g[g.role == "train"].event_id
        va = g[g.role == "val"].event_id
        if len(va) == 0:
            continue
        vstart = pd.to_datetime(ev_dates.loc[va]).min()
        bad_purge += int((pd.to_datetime(tmax.reindex(tr)) >= vstart).sum())
        date_overlap += len(set(pd.to_datetime(ev_dates.loc[tr]).dt.date) & set(pd.to_datetime(ev_dates.loc[va]).dt.date))
    rows.append(_row(1, "overlapping windows across blocks -> purge + embargo", "pass" if bad_purge == 0 else "fail",
                     f"training events whose window reaches the next block: {bad_purge}"))
    rows.append(_row(2, "same-date shocks -> date-disjoint splits", "pass" if date_overlap == 0 else "fail",
                     f"dates shared by train and val across folds: {date_overlap}"))
    acc = pd.Series([a for x in events.accessions for a in x])
    rows.append(_row(3, "duplicate / near-duplicate filings -> split by accession; hashes",
                     "pass" if acc.is_unique else "fail", f"accessions in >1 event: {int(acc.duplicated().sum())}; "
                     f"identical-text events across blocks: {extra.get('dup_text_cross_block', 'n/a')}"))
    rows.append(_row(4, "firm memorization -> anonymization; firm-held-out test", "pass",
                     f"masked firm-name hits: {extra.get('mask_firm_total', 'n/a')}; stress_firm folds: "
                     f"{splits[splits.scheme == 'stress_firm'].fold.nunique()}"))
    rows.append(_row(5, "preprocessing fitted on training indices only (scalers, TF-IDF, PCA, imputers, calibration, "
                        "thresholds, clip bounds)", "pass (code-enforced; unit test)", "fold pipelines fit on train rows only"))
    rows.append(_row(6, "test peeking -> prereg hash; G4 freeze; lockbox", "info", "checked at S18/S19 (lockbox log)"))
    intra = int((events.session_bucket.eq("INTRA") & events.tinfo_eligible).sum())
    rows.append(_row(7, "intraday / pre-publication contamination -> T_info eligibility", "pass" if intra == 0 else "fail",
                     f"INTRA events eligible for T_info: {intra}; median pre-publication window (min): "
                     f"{float(np.nanmedian(events.contam_minutes)):.0f}"))
    q = events.ts_quality.value_counts().to_dict()
    rows.append(_row(8, "timestamp errors -> four timestamps; quality flags; sensitivity", "pass",
                     f"ts_quality counts {q}; JSON-rule header mismatch rate {extra.get('json_rule_mismatch', 'n/a')}"))
    rows.append(_row(9, "EAR misuse -> registry rule + planted leak", "pass" if extra.get("ear_violations", 0) == 0 else "fail",
                     f"violations: {extra.get('ear_violations', 0)}"))
    rows.append(_row(10, "earnings-date fabrication -> three earnings types", "pass",
                     f"{events.earnings_type.value_counts(dropna=True).to_dict()}"))
    rows.append(_row(11, "SUE from a later cluster accession -> registry rule + planted leak",
                     "pass" if extra.get("sue_violations", 0) == 0 else "fail", f"violations: {extra.get('sue_violations', 0)}"))
    rows.append(_row(12, "XBRL ordering -> accession / public_available_time ordering", "pass",
                     "quarterly EPS = earliest public accession per period (as first reported)"))
    clean = int((~events[f"overlap_flag_h{int(cfg['contam_window_sessions'])}"]).sum())
    rows.append(_row(13, "same-firm overlaps -> flags + clean cohort", "pass", f"clean events: {clean} of {len(events)}"))
    rows.append(_row(14, "survivorship -> universe table + wording", "info",
                     f"delisted firms in universe: {int(universe.get('delisted_flag', pd.Series(dtype=bool)).sum())}; "
                     "SURVIVOR-ONLY universe (current listings only, survivorship-biased): stated in the report, reported "
                     "with the share of historical S&P 500 members our data covers"))
    cens = int(targets.censored.sum())
    rows.append(_row(15, "right-censoring -> per horizon", "pass", f"censored target rows dropped: {cens}"))
    rows.append(_row(16, "retrieval leakage -> historical index time filter; separate current path",
                     "pass" if extra.get("retrieval_guard_ok") else "fail", "guard unit check on planted future document"))
    rows.append(_row(17, "analogue outcomes (extension)", "na", "ext_analogue disabled"))
    rows.append(_row(18, "pretrained-model look-ahead -> provenance rules", "pass", "per-fold strict/retrospective labels written"))
    rows.append(_row(19, "distillation / continued pretraining -> training-period data only", "info", "asserted at S17"))
    rows.append(_row(20, "stacking -> walk-forward out-of-fold predictions only", "info", "asserted in ext_stacking"))
    price_cols = [c for c in events.columns if c in ("close", "open", "ret", "raw_ret", "ar_mm")]
    rows.append(_row(21, "outcome-based selection -> event construction reads no price fields",
                     "pass" if not price_cols else "fail", f"price fields in events: {price_cols}"))
    rows.append(_row(22, "data snooping -> configuration counter; FDR; MCS / PBO diagnostics", "info",
                     "configuration counter in results_long; FDR families in preregistration"))
    rows.append(_row(23, "covariate shift -> adversarial validation (AUC near 0.5 does not prove a split unbiased)", "info",
                     "computed in S11 (F39)"))
    pl = extra.get("planted", [])
    allc = all(p["caught"] for p in pl) and len(pl) == 4
    rows.append(_row(24, "broken audit -> planted-leak self-tests (4)", "pass" if allc else "fail",
                     "; ".join(f"{p['leak']}: {'caught' if p['caught'] else 'MISSED'}" for p in pl)))
    return pd.DataFrame(rows)


In [ ]:
%%writefile $REPO/src/secprice/splits/provenance.py
"""Model provenance: per-fold strict vs retrospective labels (SPEC.md §6.4)."""
from __future__ import annotations

import pandas as pd


def model_cutoffs(roster: dict) -> dict:
    """{model_key: cutoff Timestamp | 'vintage' | 'folds' | None(unknown)}"""
    out = {}
    for grp in ("encoders", "reranker", "llm", "judge", "student", "ner"):
        for k, v in (roster.get(grp) or {}).items():
            c = v.get("stated_cutoff", "unknown")
            out[k] = "vintage" if c == "vintage" else (None if c in (None, "unknown") else pd.Timestamp(c))
    for k in (roster.get("tabular") or {}):
        out[k] = "folds"
    out["lexical"] = "folds"
    out["tfidf"] = "folds"
    return out


def label(cutoff, block_start: pd.Timestamp) -> str:
    if cutoff in ("vintage", "folds"):
        return "strict"
    if cutoff is None:
        return "retrospective"
    return "strict" if cutoff < block_start else "retrospective"


def ledger_frame(roster: dict) -> pd.DataFrame:
    rows = []
    for grp in ("encoders", "reranker", "llm", "judge", "student", "ner"):
        for k, v in (roster.get(grp) or {}).items():
            rows.append({"key": k, "group": grp, "model_id": v.get("model_id") or v.get("model_id_template"),
                         "revision": v.get("revision") or "per-vintage (see models.yaml)", "card_url": v.get("card_url"),
                         "license": v.get("license"), "training_corpus": v.get("training_corpus"),
                         "stated_cutoff": v.get("stated_cutoff"), "fine_tune_data": v.get("fine_tune_data"),
                         "task": v.get("task"), "labels": str(v.get("labels"))})
    for k in ("lightgbm", "catboost", "enet"):
        rows.append({"key": k, "group": "tabular", "model_id": k, "revision": "library version in environment manifest",
                     "stated_cutoff": "trained on our folds only", "task": "head"})
    return pd.DataFrame(rows)


def fold_labels(roster: dict, blocks: dict) -> pd.DataFrame:
    cut = model_cutoffs(roster)
    rows = []
    allb = [(b.name, b.start) for b in blocks["folds"]] + [("dev", blocks["dev"].start), ("locked", blocks["locked"].start)]
    for m, c in cut.items():
        for name, st in allb:
            rows.append({"model": m, "block": name, "block_start": st, "leaderboard": label(c, st)})
    return pd.DataFrame(rows)


In [ ]:
%%writefile $REPO/src/secprice/splits/walkforward.py
"""S10 splits: expanding walk-forward folds -> development holdout -> locked test, with purge + embargo,
stress folds and the pedagogical random-split demo (SPEC.md §6.1)."""
from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd


@dataclass
class Block:
    name: str
    start: pd.Timestamp
    end: pd.Timestamp        # exclusive


def _add_years(t: pd.Timestamp, years: float) -> pd.Timestamp:
    whole = int(years)
    frac = years - whole
    out = t + pd.DateOffset(years=whole)
    if frac:
        out = out + pd.DateOffset(months=int(round(frac * 12)))
    return out


def make_blocks(cfg: dict, usable_cutoff: pd.Timestamp) -> dict:
    sp = cfg["splits"]
    start = pd.Timestamp(cfg["study"]["start"])
    dev_start, lock_start = pd.Timestamp(sp["dev_start"]), pd.Timestamp(sp["locked_test_start"])
    vals = []
    v0 = _add_years(start, float(sp["min_train_years"]))
    i = 0
    while v0 < dev_start:
        v1 = min(_add_years(v0, float(sp["val_block_years"])), dev_start)
        vals.append(Block(f"wf{i}", v0, v1))
        v0, i = v1, i + 1
    return {"folds": vals, "dev": Block("dev", dev_start, lock_start),
            "locked": Block("locked", lock_start, usable_cutoff + pd.Timedelta(days=1)), "start": start}


def assign_roles(ev: pd.DataFrame, blocks: dict, cfg: dict, exit_dates: pd.Series) -> pd.DataFrame:
    """ev: event_id, decision_date (naive date), cik, ff12_industry. exit_dates: max target exit date at the purge
    horizon (T_info / T_trade), aligned to ev. Returns long table: event_id, scheme, fold, role."""
    emb = pd.Timedelta(days=float(cfg["splits"]["embargo_days"]))
    d = pd.to_datetime(ev["decision_date"])
    ex = pd.to_datetime(exit_dates).fillna(pd.Timestamp.max.normalize())
    out = []

    def scheme(name: str, fold: int, test: Block, role_name: str):
        role = np.full(len(ev), "unused", dtype=object)
        before = d < test.start
        purged = before & (ex >= test.start)
        embargo = before & ~purged & (d >= test.start - emb)
        role[before & ~purged & ~embargo] = "train"
        role[purged] = "purged"
        role[embargo] = "embargo"
        role[(d >= test.start) & (d < test.end)] = role_name
        out.append(pd.DataFrame({"event_id": ev["event_id"].values, "scheme": name, "fold": fold, "role": role}))

    for i, b in enumerate(blocks["folds"]):
        scheme("walk_forward", i, b, "val")
    scheme("dev_holdout", 0, blocks["dev"], "dev")
    scheme("locked_test", 0, blocks["locked"], "locked")
    # stress tests on the pre-holdout period (firm- and sector-held-out); time overlap allowed: stress only
    pre = (d < blocks["dev"].start).values
    k = int(cfg["splits"].get("stress_folds", 5))
    rng = np.random.default_rng(int(cfg["seed"]))
    firms = np.array(sorted(ev["cik"].unique()))
    rng.shuffle(firms)
    fmap = {c: i % k for i, c in enumerate(firms)}
    fold_f = ev["cik"].map(fmap).to_numpy()
    for j in range(k):
        role = np.where(~pre, "unused", np.where(fold_f == j, "test", "train"))
        out.append(pd.DataFrame({"event_id": ev["event_id"].values, "scheme": "stress_firm", "fold": j, "role": role}))
    for j, ind in enumerate(sorted(ev["ff12_industry"].fillna("Other").unique())):
        role = np.where(~pre, "unused", np.where(ev["ff12_industry"].fillna("Other").to_numpy() == ind, "test", "train"))
        out.append(pd.DataFrame({"event_id": ev["event_id"].values, "scheme": "stress_sector", "fold": j, "role": role,
                                 "group": ind}))
    rnd = rng.random(len(ev))
    role = np.where(~pre, "unused", np.where(rnd < 0.2, "test", "train"))
    out.append(pd.DataFrame({"event_id": ev["event_id"].values, "scheme": "random_demo", "fold": 0, "role": role}))
    return pd.concat(out, ignore_index=True)


def blocks_frame(blocks: dict) -> pd.DataFrame:
    rows = [{"block": b.name, "kind": "walk_forward_val", "start": b.start, "end_exclusive": b.end} for b in blocks["folds"]]
    rows += [{"block": "dev", "kind": "development_holdout", "start": blocks["dev"].start, "end_exclusive": blocks["dev"].end},
             {"block": "locked", "kind": "locked_test", "start": blocks["locked"].start, "end_exclusive": blocks["locked"].end}]
    return pd.DataFrame(rows)


In [ ]:
%%writefile $REPO/src/secprice/stages_data.py
"""CPU data stages S00-S10 (SPEC.md §17.1)."""
from __future__ import annotations

import json
import os
import time
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd

from . import common as C
from .config import config_hash, file_sha256, get, text_sha256
from .events.timestamps import acceptance_from_sources, derive_times, header_to_et, json_to_et, timing_policy_label
from .logging_utils import ErrorRateExceeded, ErrorTracker, StageIncomplete
from .runner import stage
from .storage import write_json

PERIODIC = {"10-K", "10-K405", "10-KT", "10-Q"}
WANTED = PERIODIC | {"8-K"}


def _mp_ctx():
    """spawn: CUDA may be initialised in the parent (Colab); children import secprice via PYTHONPATH."""
    import multiprocessing as mp
    src = str(Path(__file__).resolve().parents[1])
    pp = os.environ.get("PYTHONPATH", "")
    if src not in pp.split(os.pathsep):
        os.environ["PYTHONPATH"] = src + (os.pathsep + pp if pp else "")
    return mp.get_context("spawn")


def _ts(s) -> pd.Timestamp:
    return pd.Timestamp(s)


# =====================================================================================================  S00
@stage("S00", "env / config / prereg check", "CPU", [], ["run_profile", "seed"],
       outputs=["environment_manifest.json", "config_resolved.json"],
       assertions=["config valid", "preregistration present", "full profile requires G2 (pilot done + owner opt-in)"],
       card="Environment and resolved configuration. The config hash stamps every artifact.")
def s00(ctx):
    from .config import repo_root
    cfg = ctx.cfg
    env = C.env_manifest()
    write_json(ctx.out / "environment_manifest.json", env)
    write_json(ctx.storage.manifests / "environment_manifest.json", env)
    write_json(ctx.out / "config_resolved.json", cfg)
    root = Path(cfg["_meta"]["repo_root"])
    prereg = root / "configs" / "preregistration.yaml"
    assert prereg.exists(), "configs/preregistration.yaml missing"
    if cfg["run_profile"] == "full":
        s18 = ctx.storage.data_root / "pilot" / "S18"          # pilot complete = development holdout scored (needs the GPU part)
        pilot_done = s18.exists() and any((p / "_SUCCESS").exists() for p in s18.glob("*"))
        if not bool(get(cfg, "full.owner_opt_in_full", False)):
            raise RuntimeError("Gate G2: set full.owner_opt_in_full: true in the owner settings to start a full run.")
        if get(cfg, "full.require_pilot_success", True) and not pilot_done:
            raise RuntimeError("Gate G2: a full run needs a completed pilot (pilot/S18 _SUCCESS: development holdout scored). "
                               "Run the pilot first, or set full.require_pilot_success: false to record an explicit owner override.")
        if not pilot_done:
            ctx.warn("G2 override: full run without a completed pilot (owner decision, logged)")
    if not os.environ.get(cfg["sec"]["user_agent_env"]) and not ctx.services.get("fixture_root"):
        ctx.warn(f"{cfg['sec']['user_agent_env']} not set: network stages S02-S06 will refuse to run")
    return {"config_hash": config_hash(cfg), "prereg_sha256": text_sha256(prereg), "owner_overrides": cfg["_meta"]["owner_overrides"],
            "gpu": env["gpu"]}


# =====================================================================================================  S01
@stage("S01", "Hugging Face dataset audit", "CPU", ["S00"], ["hf.repo", "hf.revision"],
       outputs=["hf_census.parquet", "hf_dataset_audit.md"], assertions=["no labels used"],
       card="Census of the HF 10-K corpus (metadata only). Defines the firm overlap with our OHLCV data; never a text source.")
def s01(ctx):
    from .audit.hf_audit import FORBIDDEN_KEYS, audit_markdown, build_census
    h = ctx.cfg["hf"]
    cached = ctx.storage.inputs / f"hf_census_{h['revision'][:12]}.parquet"
    if cached.exists():
        census = pd.read_parquet(cached)
        src = f"cached census {cached.name} (built by secprice.audit.hf_audit.build_census)"
    elif ctx.cfg["run_profile"] == "smoke" and not ctx.services.get("allow_hf_stream_in_smoke"):
        ctx.warn("smoke: no cached HF census in inputs; HF overlap requirement is skipped for smoke")
        census = pd.DataFrame(columns=["cik", "tickers", "form", "filing_date"])
        src = "none (smoke)"
    else:
        census = build_census(h["repo"], h["revision"], ctx.storage.hot_root / "hf_census_parts", ctx.log)
        src = "streamed raw JSONL shards"
    assert not (set(census.columns) & FORBIDDEN_KEYS), "labels must never be used"
    ctx.write(census, "hf_census.parquet")
    md = audit_markdown(census, h["repo"], h["revision"], f"Source of this census: {src}.") if len(census) else \
        "# Hugging Face SEC dataset audit (S01)\n\nNo census available in this smoke run."
    (ctx.out / "hf_dataset_audit.md").write_text(md, encoding="utf-8")
    (ctx.storage.reports / "hf_dataset_audit.md").write_text(md, encoding="utf-8")
    return {"firms": int(census.cik.nunique()) if len(census) else 0, "filings": int(len(census)), "source": src}


# =====================================================================================================  S02
@stage("S02", "Universe", "CPU", ["S01"], ["universe", "study", "hf.revision", "market.local_ohlcv_subdir", "seed"],
       outputs=["universe.parquet", "firms_per_year.csv", "submissions/"],
       assertions=[">= n_stocks unique firms (full) or failure report", "selection reads no price returns"],
       card="Firms common to our OHLCV files and the HF 10-K corpus, mapped to CIK; S&P-500 members first, then a seeded "
            "sector-stratified fill of non-members; every firm carries sp_bucket (SP500 / NonSP500). SURVIVOR-ONLY: the OHLCV "
            "set holds current listings only, so firms that delisted before 2026 are absent (survivorship-biased).")
def s02(ctx):
    from .market.prices import LocalTriframeProvider
    from .sec.submissions import fetch_submissions, fetch_ticker_map
    from .universe.build import firms_per_year, load_ff12, load_sp500, map_tickers, select_universe, sic_to_ff12, sp500_intervals
    cfg, U = ctx.cfg, ctx.cfg["universe"]
    st, en = _ts(cfg["study"]["start"]), _ts(cfg["study"]["end"])
    prov = LocalTriframeProvider(ctx.storage.inputs / cfg["market"]["local_ohlcv_subdir"])
    cov = prov.coverage()
    if cov.empty:
        raise RuntimeError(f"no OHLCV parquet files in {prov.folder}; copy final/1day/*.parquet there (see README)")
    client = C.sec_client(ctx)
    tmap = fetch_ticker_map(client)
    census = C.read(ctx, "S01", "hf_census.parquet")
    cand = map_tickers(cov, tmap, census if len(census) else pd.DataFrame({"cik": [], "tickers": []}))
    sp = load_sp500(U["sp500_commit"], ctx.storage.inputs / f"sp500_ticker_start_end_{U['sp500_commit'][:8]}.csv")
    ff = load_ff12(U["ff12_url"], ctx.storage.inputs / "Siccodes12.zip")
    hf_ciks = census.groupby(census.cik.astype(int)).size() if len(census) else pd.Series(dtype=int)
    allc = cand.ciks_all.map(json.loads)
    cand["in_hf"] = allc.map(lambda cs: any(c in hf_ciks.index for c in cs))
    cand["hf_10k_count"] = allc.map(lambda cs: int(sum(hf_ciks.get(c, 0) for c in cs)))
    ivs = [sp500_intervals(sp, t) for t in cand.ticker]
    cand["sp500_intervals"] = [json.dumps([[str(a.date()), None if pd.isna(b) else str(b.date())] for a, b in iv]) for iv in ivs]
    cand["ever_sp500"] = [any(a <= en and (pd.isna(b) or b >= st) for a, b in iv) for iv in ivs]
    cand["index_start"] = [min((a for a, b in iv), default=pd.NaT) for iv in ivs]
    cand["index_end"] = [max(((b if pd.notna(b) else pd.Timestamp("2099-12-31")) for a, b in iv), default=pd.NaT) for iv in ivs]
    need_hf = bool(U["require_hf_overlap"]) and len(census) > 0
    pre = cand[cand.cik.notna() & (cand.in_hf | (not need_hf))].copy()
    if U.get("fixed_tickers"):
        pre = pre[pre.ticker.isin([t.upper() for t in U["fixed_tickers"]])]
    sub_dir = ctx.out / "submissions"
    sub_dir.mkdir(exist_ok=True)
    metas = {}
    for cik in sorted({c for cs in pre.ciks_all.map(json.loads) for c in cs}):
        p = sub_dir / f"{cik:010d}.parquet"
        mp = sub_dir / f"{cik:010d}.meta.json"
        if p.exists() and mp.exists():
            metas[cik] = json.loads(mp.read_text())
            continue
        meta, f = fetch_submissions(client, cik)
        f.to_parquet(p, index=False)
        mp.write_text(json.dumps(meta, default=str))
        metas[cik] = meta
    # firm id = the CIK with the most periodic filings in the window; filer CIKs = every CIK with filings
    n_per_c = {}
    for c in metas:
        f = pd.read_parquet(sub_dir / f"{c:010d}.parquet", columns=["form", "filing_date"])
        w = f[(pd.to_datetime(f.filing_date) >= st - pd.DateOffset(years=4)) & (pd.to_datetime(f.filing_date) <= en)]
        n_per_c[c] = int(w.form.isin(WANTED).sum())
    firm, filers = [], []
    for cs, c0 in zip(cand.ciks_all.map(json.loads), cand.cik):
        have = [c for c in cs if n_per_c.get(c, 0) > 0]
        firm.append(max(have, key=lambda c: (n_per_c[c], c == c0)) if have else c0)
        filers.append(json.dumps(sorted(have)))
    cand["cik"] = pd.array(firm, dtype="Int64")
    cand["filer_ciks"] = filers
    rows = []
    for r in cand.itertuples(index=False):
        m = metas.get(int(r.cik)) if pd.notna(r.cik) else None
        others = [metas.get(c) or {} for c in json.loads(r.ciks_all) if c != (int(r.cik) if pd.notna(r.cik) else None)]
        fn = [x.get("name") for x in ((m or {}).get("formerNames") or [])] + [o.get("name") for o in others] + \
             [x.get("name") for o in others for x in (o.get("formerNames") or [])]
        rows.append({"sec_name": (m or {}).get("name"), "sic": (m or {}).get("sic"), "sic_desc": (m or {}).get("sicDescription"),
                     "former_names": json.dumps(sorted({x for x in fn if x})),
                     "sec_tickers": json.dumps((m or {}).get("tickers") or []), "fiscal_year_end": (m or {}).get("fiscalYearEnd"),
                     "entity_type": (m or {}).get("entityType"), "filer_category": (m or {}).get("category")})
    cand = pd.concat([cand.reset_index(drop=True), pd.DataFrame(rows)], axis=1)
    cand["ff12_industry"] = cand.sic.map(lambda s: sic_to_ff12(s, ff))
    n_per = {}
    for cik in metas:
        f = pd.read_parquet(sub_dir / f"{cik:010d}.parquet", columns=["form", "filing_date"])
        w = f[(pd.to_datetime(f.filing_date) >= st) & (pd.to_datetime(f.filing_date) <= en)]
        n_per[cik] = (int(w.form.isin(PERIODIC).sum()), int(w.form.isin({"20-F", "40-F", "6-K"}).sum()))
    fl = cand.filer_ciks.map(json.loads)
    cand["n_periodic_in_window"] = fl.map(lambda cs: sum(n_per.get(c, (0, 0))[0] for c in cs))
    cand["n_foreign_forms"] = fl.map(lambda cs: sum(n_per.get(c, (0, 0))[1] for c in cs))
    cand["delisted_flag"] = False
    reason = np.full(len(cand), None, dtype=object)
    reason[cand.cik.isna()] = "no_cik_mapping"
    if U.get("fixed_tickers"):
        reason[(reason == None) & ~cand.ticker.isin([t.upper() for t in U["fixed_tickers"]])] = "not_in_profile_ticker_list"  # noqa: E711
    reason[(reason == None) & need_hf & ~cand.in_hf] = "not_in_hf_10k_corpus"  # noqa: E711
    reason[(reason == None) & (cand.n_periodic_in_window == 0) & (cand.n_foreign_forms > 0)] = "foreign_filer_no_10k_10q"  # noqa: E711
    reason[(reason == None) & (cand.n_periodic_in_window == 0)] = "no_periodic_filings_in_window"  # noqa: E711
    reason[(reason == None) & ((cand.price_end < st) | (cand.price_start > en))] = "no_prices_in_window"  # noqa: E711
    cand["exclusion_reason"] = reason
    # one row per CIK: keep the ticker with the longest price history (share classes)
    cand["_len"] = (cand.price_end - cand.price_start).dt.days
    cand = cand.sort_values(["cik", "_len"], ascending=[True, False])
    dup = cand.cik.notna() & cand.duplicated("cik", keep="first")
    cand.loc[dup & cand.exclusion_reason.isna(), "exclusion_reason"] = "duplicate_share_class"
    cand["eligible"] = cand.exclusion_reason.isna()
    n = int(U["n_stocks"])
    sel = select_universe(cand.reset_index(drop=True), n, int(cfg["seed"]), U.get("fixed_tickers") or None,
                          balanced=cfg["run_profile"] == "pilot")
    sel["tickers_history"] = sel.apply(lambda r: json.dumps(sorted(set([r.ticker] + json.loads(r.sec_tickers or "[]")))), axis=1)
    sel = sel.drop(columns=["_len"])
    from .evaluation.buckets import firm_bucket
    sel["sp_bucket"] = firm_bucket(sel.ever_sp500)
    ns = int(sel.selected.sum())
    # survivorship, quantified: historical S&P 500 members in the window vs those our survivor-only data can cover
    in_win = sp[(sp.start_date <= en) & (sp.end_date.isna() | (sp.end_date >= st))]
    hist = set(in_win.ticker.str.upper())
    covered = set(sel[sel.eligible & sel.ever_sp500].ticker.str.upper())
    still = set(sp[sp.end_date.isna()].ticker.str.upper())
    cov = {"sp500_tickers_in_window": len(hist), "covered_by_eligible_universe": len(hist & covered),
           "coverage_share": round(len(hist & covered) / max(1, len(hist)), 3),
           "window_members_no_longer_in_index": len(hist - still),
           "selected_sp500": int((sel.selected & sel.ever_sp500).sum()), "selected_non_sp500": int((sel.selected & ~sel.ever_sp500).sum()),
           "eligible_sp500": int((sel.eligible & sel.ever_sp500).sum()), "eligible_non_sp500": int((sel.eligible & ~sel.ever_sp500).sum()),
           "note": "ticker-level match against fja05680 (ticker changes and the list's early-year gaps make this approximate); "
                   "members that delisted, merged or were acquired are missing from our survivor-only OHLCV data"}
    write_json(ctx.out / "sp500_coverage.json", cov)
    if ns < n:
        msg = (f"universe: {ns} firms selected < target {n} (eligible {int(sel.eligible.sum())}); reasons: "
               f"{sel.exclusion_reason.value_counts().to_dict()}")
        ctx.warn(msg)
        (ctx.out / "universe_failure_report.md").write_text(msg, encoding="utf-8")
    ctx.write(sel, "universe.parquet")
    fpy = firms_per_year(sel, cfg["study"]["start"], cfg["study"]["end"])
    fpy.to_csv(ctx.out / "firms_per_year.csv", index=False)
    return {"candidates": int(len(sel)), "eligible": int(sel.eligible.sum()), "selected": ns,
            "selected_sp500": cov["selected_sp500"], "selected_non_sp500": cov["selected_non_sp500"],
            "sp500_history_coverage": cov["coverage_share"], "sec_requests": client.stats}


# =====================================================================================================  S03
@stage("S03", "Calendar, prices, actions, factors", "CPU", ["S02"], ["market", "study"],
       outputs=["calendar.parquet", "prices.parquet", "corporate_actions.parquet", "factors.parquet", "price_coverage.csv"],
       assertions=["split/dividend unit tests pass", "explicit downloader parameters recorded"],
       card="Firm x session panel: split-adjusted OHLC from our local files (+ yfinance warm-up), raw close, total-return index.")
def s03(ctx):
    from .market.calendar import TradingCalendar
    from .market.factors import fetch_factors
    from .market.prices import LocalTriframeProvider, YFinanceProvider, build_series, splice
    cfg, M = ctx.cfg, ctx.cfg["market"]
    u = C.universe(ctx)
    end_cal = (_ts(cfg["study"]["end"]) + pd.Timedelta(days=200)).strftime("%Y-%m-%d")
    cal = ctx.services.get("calendar") or TradingCalendar.nyse(M["warmup_start"], end_cal)
    ctx.write(cal.to_frame(), "calendar.parquet")
    local = LocalTriframeProvider(ctx.storage.inputs / M["local_ohlcv_subdir"])
    yfp = ctx.services.get("yf_provider") or YFinanceProvider(M["yfinance"], ctx.storage.inputs / "yf_frozen.parquet", ctx.log)
    frames, acts, covrows = [], [], []
    for r in u.itertuples(index=False):
        lb = local.load(r.ticker)
        first = lb.date.min() if len(lb) else _ts(end_cal)
        wb, wa = yfp.history(r.ticker, M["warmup_start"], (first + pd.Timedelta(days=60)).strftime("%Y-%m-%d"))
        _, all_acts = yfp.history(r.ticker, M["warmup_start"], end_cal)
        bars, seam = splice(lb, wb[wb.date < first + pd.Timedelta(days=60)] if len(wb) else wb, int(M["seam_check_days"]))
        s = build_series(bars, all_acts)
        s["ticker"], s["cik"] = r.ticker, int(r.cik)
        frames.append(s)
        if len(all_acts):
            acts.append(all_acts.assign(ticker=r.ticker, cik=int(r.cik)))
        sess = cal.sessions[(cal.sessions >= s.date.min()) & (cal.sessions <= s.date.max())] if len(s) else []
        covrows.append({"ticker": r.ticker, "cik": int(r.cik), "first": s.date.min() if len(s) else None,
                        "last": s.date.max() if len(s) else None, "n_bars": len(s),
                        "missing_sessions_in_range": int(len(set(sess) - set(s.date))) if len(s) else None,
                        "delisted_history": "not covered (survivors only)", **seam})
    sb, _ = yfp.history("SPY", M["warmup_start"], end_cal)
    _, sa = yfp.history("SPY", M["warmup_start"], end_cal)
    spy = build_series(sb, sa)
    spy["ticker"], spy["cik"] = "SPY", 0
    frames.append(spy)
    px = pd.concat(frames, ignore_index=True)
    px["source_version"] = np.where(px.source.astype(str).str.startswith("local"), local.version, yfp.version)
    px = px[["cik", "ticker", "date", "open", "high", "low", "close_raw", "close_split_adj", "total_return_index",
             "volume", "dividend", "source", "source_version"]]
    ctx.write(px, "prices.parquet")
    ca = pd.concat(acts, ignore_index=True) if acts else pd.DataFrame(columns=["date", "type", "value", "ticker", "cik"])
    ctx.write(ca, "corporate_actions.parquet")
    fac, fprov = (ctx.services["factors"], {"source": "fixture"}) if "factors" in ctx.services else \
        fetch_factors(M["factors"]["ff5_url"], M["factors"]["mom_url"], ctx.storage.inputs)
    ctx.write(fac, "factors.parquet")
    cov = pd.DataFrame(covrows)
    cov.to_csv(ctx.out / "price_coverage.csv", index=False)
    write_json(ctx.out / "price_params.json", {"yfinance_params": M["yfinance"], "yfinance_version": yfp.version,
                                               "local_version": local.version, "factors": fprov, "calendar": cal.source})
    return {"tickers": int(u.shape[0]), "price_rows": int(len(px)), "spy_rows": int(len(spy)),
            "factor_last_date": str(pd.to_datetime(fac.date).max().date()) if len(fac) else None}


# =====================================================================================================  S04
def _manifest(ctx, u) -> pd.DataFrame:
    cfg = ctx.cfg
    st, en = _ts(cfg["study"]["start"]), _ts(cfg["study"]["end"])
    t0 = st - pd.DateOffset(years=float(cfg["text"]["predecessor_lookback_years"]))
    a0 = st - pd.DateOffset(years=float(cfg["accounting"]["predecessor_lookback_years"]))
    sub_dir = ctx.dep("S02") / "submissions"
    frames = []
    for r in u.itertuples(index=False):
        for fc in (json.loads(r.filer_ciks) if isinstance(r.filer_ciks, str) else [int(r.cik)]) or [int(r.cik)]:
            p = sub_dir / f"{int(fc):010d}.parquet"
            if p.exists():
                f = pd.read_parquet(p)
                f["filer_cik"] = int(fc)
                f["cik"] = int(r.cik)
                frames.append(f)
    m = pd.concat(frames, ignore_index=True).drop_duplicates("accession")
    m["base_form"] = m.form.str.replace("/A", "", regex=False)
    m = m[m.base_form.isin(WANTED)].copy()
    m["is_amendment"] = m.form.str.endswith("/A")
    fd = pd.to_datetime(m.filing_date)
    m["in_study"] = (fd >= st) & (fd <= en)
    per = m.base_form.isin(PERIODIC)
    m["in_window"] = np.where(per, (fd >= t0) & (fd <= en), (fd >= a0) & (fd <= en))
    m = m[m.in_window].copy()
    m["items_list"] = m.items_raw.map(lambda s: [x.strip() for x in str(s).split(",") if x.strip()] if pd.notna(s) and str(s) else [])
    acc = m.acceptance_raw.map(lambda s: json_to_et(s) if pd.notna(s) and str(s) else pd.NaT)
    m["acceptance_json_et"] = acc
    m["json_time_is_placeholder"] = acc.map(lambda t: bool(pd.notna(t) and t.hour == 0 and t.minute == 0 and t.second == 0))
    tax = C.taxonomy(cfg)
    pre_8k = (~per) & ~m.in_study
    earn_like = m.items_list.map(lambda it: (not it) or any(tax.map_item(i, "2000-01-01") == "earnings" or str(i) in tax.legacy_earn
                                                            or tax.map_item(i, "2010-01-01") == "earnings" for i in it))
    m["download_task"] = ~m.is_amendment & (m.in_study | per | (pre_8k & earn_like))
    reason = np.where(m.is_amendment, "AMENDMENT_AUDIT_ONLY", np.where(m.download_task, "OK_TASK", "PRESTUDY_8K_NOT_EARNINGS"))
    m["audit_reason_code"] = reason
    if cfg["run_profile"] == "smoke":
        cap = int(get(cfg, "smoke.max_filings_per_firm_form", 10))
        m = m.sort_values("filing_date")
        spread = lambda g: g.iloc[np.unique(np.linspace(0, len(g) - 1, min(cap, len(g))).round().astype(int))]  # noqa: E731
        keep = pd.concat([spread(g) for _, g in m[m.in_study].groupby(["cik", "base_form"])]).index.union(
            m[~m.in_study].groupby(["cik", "base_form"]).tail(4).index)
        m = m.loc[keep]
    return m.sort_values(["cik", "filing_date", "accession"]).reset_index(drop=True)


def periodic_expectations(per: pd.DataFrame) -> pd.DataFrame:
    """Expected 1 x 10-K and 3 x 10-Q per firm x report year while the firm files; reason codes. 8-Ks are event-driven
    and never expected (a company-year without an 8-K is not missing data)."""
    rows = []
    for cik, g in per.groupby("cik"):
        y0, y1 = int(g.ry.min()), int(g.ry.max())
        for y in range(y0, y1 + 1):
            gy = g[g.ry == y]
            k, q = int(gy.base_form.str.startswith("10-K").sum()), int((gy.base_form == "10-Q").sum())
            code = "OK" if (k == 1 and q == 3) else ("MISSING_10K" if k == 0 else ("EXTRA_10K_POSSIBLE_TRANSITION" if k > 1 else
                                                                                    f"10Q_COUNT_{q}"))
            rows.append({"cik": cik, "year": y, "expected_10k": 1, "observed_10k": k, "expected_10q": 3, "observed_10q": q,
                         "reason_code": code})
    return pd.DataFrame(rows, columns=["cik", "year", "expected_10k", "observed_10k", "expected_10q", "observed_10q", "reason_code"])


@stage("S04", "EDGAR index + coverage audit", "CPU", ["S02"],
       ["study", "text.predecessor_lookback_years", "accounting.predecessor_lookback_years", "sec.header_check_sample",
        "timing", "smoke", "taxonomy_version"],
       outputs=["filing_manifest.parquet", "coverage_firm_year.csv", "ts_quality_by_year.csv", "header_check.csv",
                "data_coverage_summary.md"],
       assertions=["no 8-K expected", "timestamp-quality table"],
       card="Observed filings per firm with corrected acceptance times, periodic-filing expectations and reason codes.")
def s04(ctx):
    from .sec.archives import index_headers_url
    from .events.timestamps import parse_header_acceptance
    cfg = ctx.cfg
    u = C.universe(ctx)
    m = _manifest(ctx, u)
    # periodic expectations by firm x calendar year of report date (8-Ks are event-driven: no expectation)
    per = m[m.base_form.isin(PERIODIC) & ~m.is_amendment & m.in_study].copy()
    per["ry"] = pd.to_datetime(per.report_date.replace("", np.nan)).dt.year.fillna(pd.to_datetime(per.filing_date).dt.year)
    cov = periodic_expectations(per)
    assert "8-K" not in set(cov.columns), "8-Ks carry no expectation"
    cov.to_csv(ctx.out / "coverage_firm_year.csv", index=False)
    # header sample check of the JSON timestamp rule (periodic filings with index-headers pages)
    client = C.sec_client(ctx)
    rng = np.random.default_rng(int(cfg["seed"]))
    pool = m[m.base_form.isin(PERIODIC) & ~m.json_time_is_placeholder & m.acceptance_json_et.notna()]
    pool = pool[pd.to_datetime(pool.filing_date) >= "2016-01-01"]
    k = min(int(cfg["sec"]["header_check_sample"]), len(pool))
    chk = []
    for r in pool.iloc[rng.choice(len(pool), size=k, replace=False)].itertuples(index=False) if k else []:
        b = client.get(index_headers_url(r.filer_cik, r.accession))
        raw = parse_header_acceptance(b.decode("latin-1")) if b else None
        if raw:
            ht = header_to_et(raw)
            chk.append({"accession": r.accession, "json_et": r.acceptance_json_et, "header_et": ht,
                        "match": bool(abs((r.acceptance_json_et - ht).total_seconds()) < 1)})
    hc = pd.DataFrame(chk, columns=["accession", "json_et", "header_et", "match"])
    hc.to_csv(ctx.out / "header_check.csv", index=False)
    m["ts_quality_json"] = np.where(m.acceptance_json_et.isna(), "C", np.where(m.json_time_is_placeholder, "B", "A"))
    tq = m.assign(year=pd.to_datetime(m.filing_date).dt.year).groupby(["year", "ts_quality_json"]).size().unstack(fill_value=0)
    tq.to_csv(ctx.out / "ts_quality_by_year.csv")
    m["url"] = None
    m["parse_status"] = None
    m["checksum"] = None
    ctx.write(m, "filing_manifest.parquet")
    mm = (hc.match.mean() if len(hc) else None)
    md = ["# Data coverage summary (S04)", "",
          f"- Firms: {m.cik.nunique()}; filings in window: {len(m):,}; download tasks: {int(m.download_task.sum()):,}",
          f"- By form: {m.form.value_counts().to_dict()}",
          f"- Periodic firm-years: {len(cov):,}; reason codes: {cov.reason_code.value_counts().to_dict() if len(cov) else {}}",
          "- 8-K filings are event-driven: a company-year without an 8-K is not missing data (no expectation).",
          f"- JSON acceptance-time rule checked against {len(hc)} SGML headers: match rate {mm if mm is None else round(mm, 4)}",
          f"- Timestamp quality (JSON-derived) by year: see ts_quality_by_year.csv"]
    (ctx.out / "data_coverage_summary.md").write_text("\n".join(md), encoding="utf-8")
    (ctx.storage.reports / "data_coverage_summary.md").write_text("\n".join(md), encoding="utf-8")
    return {"filings": int(len(m)), "tasks": int(m.download_task.sum()), "header_checks": int(len(hc)),
            "header_match_rate": None if mm is None else float(mm)}


def teacher_extraction_will_run(cfg: dict, inputs: Path) -> bool:
    """The teacher extraction (budget.llm_teacher_filings) runs only after gold-val labels exist (G3b) or with the owner's
    exploratory override (D-037); otherwise only the gold-sample LLM prompts run."""
    g = Path(inputs) / "gold"
    has_gold = g.exists() and any(g.glob("gold_*_filings_template*.csv"))
    return bool(has_gold or get(cfg, "rag.run_extraction_without_gold", False))


def release_cluster_exclusions(filings: pd.DataFrame, release_accessions: set, window_days: int = 120) -> dict:
    """{release accession: periodic accessions of the same firm filed on or after the release within the cluster window}.
    Hard rule 7 / D-072: none of them may feed the release's SUE, whatever its acceptance time."""
    per = filings[filings.form.isin(PERIODIC)][["cik", "accession", "filing_date"]].copy()
    rel = filings[filings.accession.isin(release_accessions)][["cik", "accession", "filing_date"]].copy()
    if per.empty or rel.empty:
        return {}
    per["fd"], rel["fd"] = pd.to_datetime(per.filing_date), pd.to_datetime(rel.filing_date)
    j = rel.merge(per, on="cik", suffixes=("", "_p"))
    j = j[(j.fd_p >= j.fd) & (j.fd_p <= j.fd + pd.Timedelta(days=window_days))]
    return j.groupby("accession").accession_p.apply(set).to_dict()


def preflight_projection(cfg: dict, manifest: pd.DataFrame, data_root, hot_root) -> dict:
    """CPU-side preflight (hard rule 4, §13.3, G2): projected events, chunks, storage and GPU hours from the filing index,
    before the long download. Constants come from the pilot (configs/default.yaml budget.preflight_*)."""
    B = cfg["budget"]
    t = manifest[manifest.download_task]
    by = t.base_form.map(lambda f: "10-K" if str(f).startswith("10-K") else "10-Q" if str(f).startswith("10-Q") else "8-K")
    cpf = {k: float(v) for k, v in B["preflight_chunks_per_filing"].items()}
    chunks = float(sum(cpf.get(k, 10.0) * n for k, n in by.value_counts().items()))
    n_enc = len(cfg["models"]["encoders"])
    events = int(round(len(t) * 0.89))                       # events per filing on the pilot (same-day 8-Ks merge)
    emb_gb = events * 768 * 2 * 2 * n_enc / 1e9 * 1.1          # mean + max pooled float16 vectors, +10% index
    text_gb = len(t) * float(B["preflight_bytes_per_filing"]) / 1e9
    gpu_b = chunks * n_enc / float(B["preflight_assumed_chunks_per_s"]) / 3600
    teacher = teacher_extraction_will_run(cfg, Path(data_root) / cfg["paths"].get("inputs_dirname", "inputs"))
    prompts = (int(B["llm_teacher_filings"]) if teacher else 0) + int(B["llm_gold_filings"])
    gpu_c = prompts * float(B["preflight_llm_s_per_prompt"]) / 3600
    rep = {"profile": cfg["run_profile"], "firms": int(t.cik.nunique()), "filings_to_download": int(len(t)),
           "filings_by_form": by.value_counts().to_dict(), "est_events": events, "est_chunks": int(chunks),
           "est_storage_gb": round(emb_gb + text_gb, 2), "est_embedding_gb": round(emb_gb, 2), "est_text_gb": round(text_gb, 2),
           "est_gpu_hours_encoders": round(gpu_b, 2), "est_gpu_hours_llm": round(gpu_c, 2), "llm_prompts": prompts,
           "teacher_extraction_runs": teacher,
           "est_sec_download_hours": round(len(t) * 1.6 / float(cfg["sec"]["max_rps"]) / 3600, 1),
           "disk_free_gb": {"data_root": C.disk_free_gb(data_root), "hot_root": C.disk_free_gb(hot_root)},
           "budget": {k: B[k] for k in ("max_storage_gb", "max_gpu_hours_b", "max_gpu_hours_c")}}
    over = []
    if rep["est_storage_gb"] > float(B["max_storage_gb"]):
        over.append(f"storage {rep['est_storage_gb']} GB > max_storage_gb {B['max_storage_gb']}")
    if gpu_b > float(B["max_gpu_hours_b"]):
        over.append(f"encoder GPU hours {gpu_b:.1f} > max_gpu_hours_b {B['max_gpu_hours_b']}")
    if gpu_c > float(B["max_gpu_hours_c"]):
        over.append(f"LLM GPU hours {gpu_c:.1f} > max_gpu_hours_c {B['max_gpu_hours_c']}")
    rep["over_budget"] = over
    rep["passed"] = not over
    return rep


# =====================================================================================================  S05
@stage("S05", "Download + parse + taxonomy + anonymize", "CPU", ["S04"],
       ["sec.max_8k_bytes", "sec.max_doc_bytes", "text.max_chars_per_section", "text.sections", "smoke", "taxonomy_version",
        "budget.max_error_rate", "hf.qa_enabled", "hf.qa_sample_filings"],
       outputs=["sections/<cik>.parquet", "exhibits/<cik>.parquet", "ledger/<cik>.parquet", "sections_index.parquet",
                "parse_qa.csv"],
       assertions=["taxonomy version recorded", "OTHER_OR_UNMAPPED share reported", "error rate within budget"],
       card="Primary documents (10-K/10-Q) and full 8-K submissions downloaded under the fair-access rate limit, parsed into "
            "target sections with Exhibit-13 fallback, anonymized (firm names/tickers masked).")
def s05(ctx):
    from .parse.worker import run_firm, run_firm_safe
    from .sec.client import resolve_user_agent
    cfg, S = ctx.cfg, ctx.cfg["sec"]
    m = C.manifest(ctx)
    pf = preflight_projection(cfg, m, ctx.storage.data_root, ctx.storage.hot_root)
    write_json(ctx.out / "preflight_projection.json", pf)
    write_json(ctx.storage.manifests / "preflight_projection.json", pf)
    if cfg["run_profile"] == "full" and not pf["passed"]:
        raise RuntimeError(f"preflight (G2) refuses the full run before the download: {pf['over_budget']}; raise the budget "
                           "in configs/default.yaml only as an explicit owner decision")
    tasks = m[m.download_task]
    u = C.universe(ctx).set_index("cik")
    fixture = ctx.services.get("fixture_root")
    nproc = int(S.get("parse_workers") or 0) or max(1, (os.cpu_count() or 2) - 1)
    # one process per firm, each limited to max_rps / nproc (the total never exceeds sec.max_rps); downloads are
    # latency-bound, so more processes use more of the fair-access budget (RAM ~100 MB per worker)
    nproc = 1 if fixture else min(nproc, int(S.get("max_parse_workers") or 8), max(1, tasks.cik.nunique()))
    ua = "fixture" if fixture else resolve_user_agent(S["user_agent_env"])
    jobs = []
    for cik, g in tasks.groupby("cik"):
        ur = u.loc[cik] if cik in u.index else None
        names = [ur.sec_name] + json.loads(ur.former_names or "[]") if ur is not None else []
        tickers = json.loads(ur.tickers_history) if ur is not None else []
        jobs.append({"cik": int(cik), "out_dir": str(ctx.out), "fixture_root": fixture, "user_agent": ua,
                     "rps_per_proc": float(S["max_rps"]) / nproc, "cache_dir": str(ctx.storage.hot_root / "sec_cache"),
                     "cache_days": S["cache_days"], "cache_max_bytes": int(S.get("cache_max_bytes") or 0),
                     "retry_errors": bool(S.get("retry_failed_filings", True)),
                     "timeout_s": S["timeout_s"], "max_retries": S["max_retries"],
                     "backoff_base_s": S["backoff_base_s"], "blocked_403_limit": S.get("blocked_403_limit", 20),
                     "names": [n for n in names if n], "tickers": tickers, "max_chars": int(cfg["text"]["max_chars_per_section"]),
                     "max_8k_bytes": int(S["max_8k_bytes"]), "max_doc_bytes": int(S["max_doc_bytes"]),
                     "filings": g[["cik", "filer_cik", "accession", "form", "primary_doc", "filing_date"]].to_dict("records")})
    t0 = time.time()
    done, crashes = 0, []
    wall = float(S.get("s05_max_wall_hours") or 0) * 3600

    def _over():
        return wall > 0 and time.time() - t0 > wall

    if nproc == 1:
        for j in jobs:
            if _over():
                break
            r = run_firm_safe(j)
            done += 1
            if r.get("blocked"):
                raise RuntimeError(r["blocked"])
            if r.get("crash"):
                crashes.append(r)
    else:
        with ProcessPoolExecutor(max_workers=nproc, mp_context=_mp_ctx()) as ex:
            futs = {ex.submit(run_firm_safe, j): j["cik"] for j in jobs}
            for f in as_completed(futs):
                if f.cancelled():
                    continue
                r = f.result()
                done += 1
                if r.get("blocked"):
                    for ff in futs:
                        ff.cancel()
                    raise RuntimeError(r["blocked"])
                if r.get("crash"):
                    crashes.append(r)
                if _over():
                    for ff in futs:
                        ff.cancel()
                if done % 10 == 0 or done == len(jobs):
                    ctx.log.info(f"S05 firms {done}/{len(jobs)} in {time.time() - t0:.0f}s")
    n_ledger = len(list((ctx.out / "ledger").glob("*.parquet"))) if (ctx.out / "ledger").exists() else 0
    if n_ledger < len(jobs) and _over():
        raise StageIncomplete(f"S05 reached its {wall / 3600:.1f} h wall-clock budget with {n_ledger}/{len(jobs)} firms done; "
                              "run the same cell again to continue (finished firms are skipped).")
    if crashes:
        ctx.warn(f"{len(crashes)} firm jobs crashed: {[c['crash'][:120] for c in crashes[:5]]}")
    led = pd.concat([pd.read_parquet(p) for p in sorted((ctx.out / "ledger").glob("*.parquet"))], ignore_index=True)
    sec_idx = []
    for p in sorted((ctx.out / "sections").glob("*.parquet")):
        s = pd.read_parquet(p, columns=["accession", "cik", "form", "section_id", "n_chars", "n_tokens_est", "parse_method",
                                        "by_reference", "truncated", "mask_firm", "mask_ticker"])
        s["part"] = p.name
        sec_idx.append(s)
    sec_idx = pd.concat(sec_idx, ignore_index=True) if sec_idx else pd.DataFrame()
    ctx.write(led, "parse_ledger.parquet")
    ctx.write(sec_idx, "sections_index.parquet")
    # error-rate guard (download/parse errors only; 'no_sections' is reported, not an error)
    err = led.status.isin(["error", "http_404"]).mean() if len(led) else 0.0
    mm = m.merge(led[["accession", "status", "n_sections"]], on="accession", how="left")
    mm["year"] = pd.to_datetime(mm.filing_date).dt.year
    qa = mm[mm.download_task].groupby(["base_form", "year"]).agg(
        observed=("accession", "size"), downloaded_ok=("status", lambda s: int((s == "ok").sum())),
        with_sections=("n_sections", lambda s: int((s.fillna(0) > 0).sum()))).reset_index()
    if len(sec_idx):
        found = sec_idx.groupby(["form", "section_id"]).size().rename("n").reset_index()
        found.to_csv(ctx.out / "section_found_counts.csv", index=False)
    qa.to_csv(ctx.out / "parse_qa.csv", index=False)
    tax = C.taxonomy(cfg)
    k8 = m[(m.base_form == "8-K") & m.download_task]
    cats = [c for r in k8.itertuples(index=False) for c in tax.categories(r.items_list, r.filing_date)]
    other_share = (np.mean([c == tax.catch_all for c in cats]) if cats else None)
    hfqa = _hf_parser_qa(ctx, m, sec_idx)
    if err > float(cfg["budget"]["max_error_rate"]) and len(led) >= 50:
        raise ErrorRateExceeded(f"S05 error rate {err:.3f} > budget; see parse_ledger.parquet")
    return {"firms": len(jobs), "filings": int(len(led)), "status": led.status.value_counts().to_dict(), "hf_parser_qa": hfqa,
            "sections": int(len(sec_idx)), "error_rate": round(float(err), 4), "taxonomy_version": tax.version,
            "other_or_unmapped_share": other_share, "wall_s": round(time.time() - t0, 1)}


def _one_text_per_section(df: pd.DataFrame, col: str) -> pd.Series:
    """section_id -> the longest text (duplicates collapse to one string, missing text to '')."""
    if df.empty:
        return pd.Series(dtype=object)
    d = df.assign(_n=df[col].fillna("").astype(str).str.len()).sort_values("_n", ascending=False)
    return d.drop_duplicates("section_id").set_index("section_id")[col].fillna("").astype(str)


def _hf_parser_qa(ctx, m: pd.DataFrame, sec_idx: pd.DataFrame) -> dict:
    """Optional QA of our 10-K section parser against the HF corpus sections (same CIK + filing date), streaming at
    most hf.qa_max_shards shards. Agreement is reported as length ratio and token-set Jaccard per section."""
    from .audit.hf_audit import qa_sections_for
    h = ctx.cfg["hf"]
    if not h.get("qa_enabled", True) or ctx.cfg["run_profile"] == "smoke" or ctx.services.get("fixture_root"):
        return {"ran": False, "reason": "disabled for smoke/offline runs"}
    census = C.read(ctx, "S01", "hf_census.parquet")
    if census.empty or sec_idx.empty:
        return {"ran": False, "reason": "no census or sections"}
    k = m[(m.base_form == "10-K") & m.download_task][["cik", "accession", "filing_date"]]
    c = census.assign(cik_i=census.cik.astype(int))
    j = k.merge(c[["cik_i", "filing_date", "shard"]], left_on=["cik", "filing_date"], right_on=["cik_i", "filing_date"])
    if j.empty:
        return {"ran": False, "reason": "no matched 10-Ks"}
    top = j.shard.value_counts().index[: int(h.get("qa_max_shards", 2))]
    j = j[j.shard.isin(top)]
    j = j.sample(min(len(j), int(h["qa_sample_filings"])), random_state=0)
    try:
        hf = qa_sections_for(h["repo"], h["revision"], census[census.shard.isin(top)], j[["cik", "filing_date"]])
    except Exception as e:
        return {"ran": False, "reason": f"stream failed: {e}"}
    rows = []
    for r in j.itertuples(index=False):
        p = ctx.out / "sections" / f"{int(r.cik):010d}.parquet"
        if not p.exists():
            continue
        ours = pd.read_parquet(p, columns=["accession", "section_id", "text_raw"])
        ours = _one_text_per_section(ours[ours.accession == r.accession], "text_raw")
        # the HF corpus can hold two filings for one firm and date (e.g. a 10-K and its amendment): keep the longest
        theirs = _one_text_per_section(hf[(hf.cik == int(r.cik)) & (hf.filing_date == r.filing_date)], "hf_text")
        for sid in ("10K_1", "10K_1A", "10K_7", "10K_7A"):
            a, b = ours.get(sid, ""), theirs.get(sid, "")
            if not a and not b:
                continue
            ta, tb = set(a.lower().split()), set(b.lower().split())
            rows.append({"cik": int(r.cik), "accession": r.accession, "section_id": sid, "ours_chars": len(a), "hf_chars": len(b),
                         "len_ratio": len(a) / len(b) if b else None,
                         "token_jaccard": len(ta & tb) / len(ta | tb) if (ta | tb) else None})
    qa = pd.DataFrame(rows)
    qa.to_csv(ctx.out / "hf_parser_qa.csv", index=False)
    return {"ran": True, "filings": int(j.shape[0]), "median_token_jaccard_by_section":
            qa.groupby("section_id").token_jaccard.median().round(3).to_dict() if len(qa) else {}}


# =====================================================================================================  S06
@stage("S06", "XBRL + release EPS + SUE", "CPU", ["S05"], ["accounting", "timing", "sec.max_rps"],
       outputs=["xbrl_facts.parquet", "eps_release.parquet", "sue_by_accession.parquet", "sue_sources.parquet", "sue_coverage.csv"],
       assertions=["no later-cluster source", "ordering by public time"],
       card="Point-in-time accounting: XBRL facts with public times, as-first-reported quarterly EPS, release-exhibit EPS and SUE.")
def s06(ctx):
    from .features.accounting import attach_public_time, compute_sue_table, quarterly_eps
    from .parse.eps_release import USABLE_CONFIDENCE, infer_quarter_end, is_results_release, parse_release_eps
    from .sec.companyfacts import facts_frame, fetch_companyfacts
    cfg, A, T = ctx.cfg, ctx.cfg["accounting"], ctx.cfg["timing"]
    m = C.manifest(ctx)
    led = C.read(ctx, "S05", "parse_ledger.parquet")
    u = C.universe(ctx).set_index("cik")
    acc_t = _final_acceptance(m, led, cfg, ctx)
    times = derive_times(acc_t["acceptance_time"], T["public_lag_minutes"], T["decision_latency_minutes"])
    ft = pd.concat([acc_t[["accession", "cik", "form", "filing_date", "report_date"]].reset_index(drop=True), times], axis=1)
    client = C.sec_client(ctx)
    facts = []
    tags = list(A["eps_tags"]) + list(A["fact_tags"])
    for (cik, fc) in sorted(set(zip(m.cik.astype(int), m.filer_cik.astype(int)))):
        js = fetch_companyfacts(client, int(fc))
        ff = facts_frame(js, tags, [A["shares_tag"]])
        ff["cik"] = int(cik)                       # firm id (a predecessor/successor filer maps to the same firm)
        ff["filer_cik"] = int(fc)
        facts.append(ff)
    facts = pd.concat(facts, ignore_index=True) if facts else pd.DataFrame()
    facts = attach_public_time(facts, ft, T["public_lag_minutes"], T["date_only_assumed_time"]) if len(facts) else facts
    ctx.write(facts, "xbrl_facts.parquet")
    qe = quarterly_eps(facts) if len(facts) else pd.DataFrame(columns=["cik", "kind", "period_end", "qkey", "value",
                                                                       "source_accession", "public_available_time"])
    ctx.write(qe, "eps_quarterly_xbrl.parquet")
    # release EPS from EX-99 exhibits of earnings candidates
    tax = C.taxonomy(cfg)
    rel_rows = []
    k8 = m[(m.base_form == "8-K") & m.download_task]
    for cik, g in k8.groupby("cik"):
        p = ctx.dep("S05") / "exhibits" / f"{int(cik):010d}.parquet"
        if not p.exists():
            continue
        ex = pd.read_parquet(p)
        ex = ex[ex.exhibit_type.str.startswith("EX-99")]
        fye = u.loc[cik, "fiscal_year_end"] if cik in u.index else None
        for r in g.itertuples(index=False):
            modern, legacy = tax.earnings_candidate(r.items_list, r.filing_date)
            if not (modern or legacy or not r.items_list):
                continue
            exs = ex[ex.accession == r.accession]
            best, flag = None, False
            for e in exs.itertuples(index=False):
                if is_results_release(e.text):
                    flag = True
                    pr = parse_release_eps(e.text)
                    if pr["eps_diluted"] is not None or pr["eps_basic"] is not None:
                        best = pr
                        break
                    best = best or pr
            inf, inf_src = infer_quarter_end(r.filing_date, fye)
            pe, ps = (best or {}).get("period_end"), (best or {}).get("period_source")
            if pe and inf and abs((pd.Timestamp(pe) - pd.Timestamp(inf)).days) > 45:
                pe, ps = inf, "inferred_override_text_far"
            if not pe:
                pe, ps = inf, inf_src
            rel_rows.append({"accession": r.accession, "cik": int(cik), "is_results_release": flag,
                             "eps_diluted": (best or {}).get("eps_diluted"), "eps_basic": (best or {}).get("eps_basic"),
                             "parse_confidence": (best or {}).get("parse_confidence", "none"),
                             "source_line": (best or {}).get("source_line"), "period_end": pe, "period_source": ps,
                             "candidate_modern": modern, "candidate_legacy_equivalent": legacy})
    rel = pd.DataFrame(rel_rows, columns=["accession", "cik", "is_results_release", "eps_diluted", "eps_basic", "parse_confidence",
                                          "source_line", "period_end", "period_source", "candidate_modern",
                                          "candidate_legacy_equivalent"])
    rel = rel.merge(ft[["accession", "public_available_time", "decision_time"]], on="accession", how="left")
    from .parse.eps_release import flag_suspect_annual
    fye_map = {int(c): u.loc[c, "fiscal_year_end"] for c in u.index if "fiscal_year_end" in u.columns}
    rel = flag_suspect_annual(rel, fye_map)
    if len(rel) and rel.get("suspect_annual", pd.Series(dtype=bool)).any():
        ctx.warn(f"{int(rel.suspect_annual.sum())} release EPS values withheld as probable full-year figures (suspect_annual)")
    # QA: release EPS vs first-reported XBRL for the same quarter (QA only, never an input)
    from .features.accounting import _qkey
    rel["qkey"] = _qkey(pd.to_datetime(rel["period_end"])) if len(rel) else pd.Series(dtype="Int64")
    xd = qe[qe.kind == "diluted"].sort_values("public_available_time")[["cik", "qkey", "value"]].rename(columns={"value": "xbrl_eps"}) if len(qe) else \
        pd.DataFrame(columns=["cik", "qkey", "xbrl_eps"])
    rel = rel.merge(xd.drop_duplicates(["cik", "qkey"]), on=["cik", "qkey"], how="left").drop_duplicates("accession")
    rel["xbrl_match_flag"] = np.where(rel.xbrl_eps.notna() & rel.eps_diluted.notna(),
                                      (rel.eps_diluted - rel.xbrl_eps).abs() <= 0.015, None)
    ctx.write(rel, "eps_release.parquet")
    usable = rel[rel.is_results_release & rel.parse_confidence.isin(USABLE_CONFIDENCE) & rel.eps_diluted.notna()]
    rel_hist = pd.DataFrame({"cik": usable.cik, "qkey": usable.qkey, "value": usable.eps_diluted,
                             "source_accession": usable.accession, "public_available_time": usable.public_available_time})
    xh = qe[qe.kind == "diluted"][["cik", "qkey", "value", "source_accession", "public_available_time", "period_end"]] if len(qe) else rel_hist.iloc[:0]
    # SUE per candidate accession: verified-release candidates and periodic filings
    evs = []
    for r in usable.itertuples(index=False):
        evs.append({"accession": r.accession, "cik": r.cik, "kind": "release", "decision_time": r.decision_time,
                    "eps_q": r.eps_diluted, "qkey": r.qkey})
    perf = ft[ft.form.isin(PERIODIC)]
    for r in perf.itertuples(index=False):
        evs.append({"accession": r.accession, "cik": r.cik, "kind": "periodic", "decision_time": r.decision_time,
                    "eps_q": np.nan, "qkey": None})
    evdf = pd.DataFrame(evs)
    if len(evdf):
        evdf = evdf.drop_duplicates("accession")
        evdf["decision_time"] = pd.to_datetime(evdf["decision_time"], utc=True).dt.tz_convert("America/New_York")
        # hard rule 7: a release's own cluster (its next 10-Q/10-K, D-017) can never feed its SUE. Clusters are built in S07,
        # so exclude conservatively here: every periodic filing of the firm filed on or after the release, within the
        # 120-day cluster window (matters when such a filing is public before the release event's decision time, D-072)
        cluster_later = release_cluster_exclusions(ft, set(usable.accession))
        sue, src = compute_sue_table(evdf, xh, rel_hist, int(A["sue_min_diffs"]), int(A["sue_max_diffs"]), cluster_later)
    else:
        sue, src = pd.DataFrame(), pd.DataFrame(columns=["accession", "source_accession", "source_public_time", "source_kind"])
    if len(src) and len(evdf):
        dmap = evdf.set_index("accession")["decision_time"]
        bad = pd.to_datetime(src.source_public_time, utc=True) >= pd.to_datetime(src.accession.map(dmap), utc=True)
        assert not bad.any(), "SUE history must be public strictly before decision_time"
    ctx.write(sue, "sue_by_accession.parquet")
    ctx.write(src, "sue_sources.parquet")
    if len(sue):
        sc = sue.merge(ft[["accession", "filing_date"]], on="accession", how="left")
        sc["year"] = pd.to_datetime(sc.filing_date).dt.year
        sc["kind"] = np.where(sc.eps_q_source == "release_exhibit", "8-K release", "10-Q/10-K")
        cov = sc.groupby(["year", "kind"]).agg(n=("accession", "size"), sue_available=("sue", lambda s: int(s.notna().sum()))).reset_index()
        cov.to_csv(ctx.out / "sue_coverage.csv", index=False)
    match = rel.xbrl_match_flag.dropna()
    return {"facts": int(len(facts)), "xbrl_quarters": int(len(qe)), "release_candidates": int(len(rel)),
            "results_releases": int(rel.is_results_release.sum()) if len(rel) else 0,
            "release_eps_parsed_usable": int(len(usable)), "release_vs_xbrl_match_rate": float(match.mean()) if len(match) else None,
            "sue_available": int(sue.sue.notna().sum()) if len(sue) else 0}


def _final_acceptance(m: pd.DataFrame, led: pd.DataFrame, cfg: dict, ctx=None) -> pd.DataFrame:
    """acceptance_time per accession: SGML header when the download carried one, else the JSON rule (quality A when the
    rule's header checks pass, otherwise C), else filing date (quality B)."""
    hdr = led.set_index("accession")["header_acceptance_raw"].to_dict() if len(led) else {}
    # rule verification: every 8-K full-submission header vs the corrected JSON value (+ S04 sample)
    comp = []
    for r in m.itertuples(index=False):
        h = hdr.get(r.accession)
        if h and pd.notna(r.acceptance_json_et) and not r.json_time_is_placeholder:
            comp.append(abs((r.acceptance_json_et - header_to_et(h)).total_seconds()) < 1)
    s04 = []
    if ctx is not None:
        p = ctx.dep("S04") / "header_check.csv"
        if p.exists():
            hc = pd.read_csv(p)
            s04 = hc["match"].astype(bool).tolist() if len(hc) else []
    allc = comp + s04
    mismatch = 1 - float(np.mean(allc)) if allc else None
    ok = mismatch is None or mismatch <= float(cfg["timing"]["json_rule_max_mismatch"])
    rows = []
    for r in m.itertuples(index=False):
        t, q, src, _ = acceptance_from_sources(r.acceptance_raw, hdr.get(r.accession), r.filing_date,
                                               cfg["timing"]["date_only_assumed_time"], ok)
        rows.append((r.accession, r.cik, r.base_form, r.form, r.filing_date, r.report_date, t, q, src))
    out = pd.DataFrame(rows, columns=["accession", "cik", "form", "raw_form", "filing_date", "report_date", "acceptance_time",
                                      "ts_quality", "ts_source"])
    out.attrs["json_rule_mismatch"] = mismatch
    out.attrs["json_rule_n"] = len(allc)
    return out


# =====================================================================================================  S07
@stage("S07", "Events + clocks", "CPU", ["S06", "S03"], ["timing", "study", "targets.horizons", "contam_window_sessions",
                                                          "taxonomy_version"],
       outputs=["events.parquet", "usable_cutoff.json", "event_counts.csv", "filing_manifest_timed.parquet"],
       assertions=["four timestamps", "buckets", "earnings types", "overlaps", "event construction reads no price fields"],
       card="One row per event (periodic filing or same-day 8-K group) with acceptance / public / feature-ready / decision times.")
def s07(ctx):
    from .events.build import build_events, events_to_parquet_safe
    cfg = ctx.cfg
    m = C.manifest(ctx)
    led = C.read(ctx, "S05", "parse_ledger.parquet")
    acc = _final_acceptance(m, led, cfg, ctx)
    mm = m.drop(columns=[c for c in ("acceptance_time",) if c in m.columns]).merge(
        acc[["accession", "acceptance_time", "ts_quality", "ts_source"]], on="accession", how="left")
    mm["form"] = mm["base_form"]
    titles = led.set_index("accession")["header_item_titles"].to_dict() if len(led) else {}
    hitems = led.set_index("accession")["header_items"].to_dict() if len(led) else {}
    mm["item_titles"] = mm.accession.map(lambda a: [t for t in str(titles.get(a) or "").split(" || ") if t])
    mm["items_list"] = [it if len(it) else [x for x in str(hitems.get(a) or "").split(",") if x]
                        for it, a in zip(mm.items_list, mm.accession)]
    mm = mm[mm.download_task]
    rel = C.read(ctx, "S06", "eps_release.parquet")
    rel_info = rel[["accession", "is_results_release"]].assign(release_period_end=rel.period_end) if len(rel) else \
        pd.DataFrame(columns=["accession", "is_results_release", "release_period_end"])
    si = C.read(ctx, "S05", "sections_index.parquet")
    sav = si.groupby("accession")["section_id"].apply(list).rename("sections").reset_index() if len(si) else \
        pd.DataFrame(columns=["accession", "sections"])
    cal = C.calendar(ctx)
    ev = build_events(mm, cal, cfg, C.taxonomy(cfg), rel_info, sav)
    u = C.universe(ctx)
    ev = ev.merge(u[["cik", "ticker", "ff12_industry"]], on="cik", how="left")
    price_cols = [c for c in ev.columns if c in ("close", "open", "ret", "raw_ret")]
    assert not price_cols, "event construction must not read price fields"
    # point in time (D-071): every filing in an event is accepted no later than the event's acceptance time, so all of an
    # event's content is public by its decision time
    acc_t = pd.to_datetime(mm.set_index("accession")["acceptance_time"], utc=True)
    late = [(e, a) for e, accs, t in zip(ev.event_id, ev.accessions, pd.to_datetime(ev.acceptance_time, utc=True))
            for a in accs if a in acc_t.index and acc_t[a] > t]
    assert not late, f"{len(late)} filings accepted after their event's acceptance time, e.g. {late[:3]}"
    px = C.read(ctx, "S03", "prices.parquet", columns=["ticker", "date"])
    last_px = pd.to_datetime(px[px.ticker != "SPY"].date).max()
    last_doc = pd.to_datetime(mm[mm.in_study].filing_date).max()
    cutoff = min(last_px, last_doc, pd.Timestamp(cfg["study"]["end"]))
    partial = cutoff < pd.Timestamp(cfg["study"]["end"]) or pd.Timestamp.today() < pd.Timestamp(cfg["study"]["end"]) + pd.offsets.MonthEnd(0)
    ci = int(cal.sessions.searchsorted(cutoff, side="right") - 1)
    write_json(ctx.out / "usable_cutoff.json", {"usable_cutoff": str(cutoff.date()), "usable_cutoff_idx": ci,
                                                "last_price_date": str(last_px.date()), "last_document_date": str(last_doc.date()),
                                                "final_month_partial": bool(partial),
                                                "json_rule_mismatch": acc.attrs.get("json_rule_mismatch"),
                                                "json_rule_n": acc.attrs.get("json_rule_n"),
                                                "timing_policy": timing_policy_label(cfg)})
    ctx.write(events_to_parquet_safe(ev), "events.parquet")
    # SPEC.md §16 filing_manifest contract: acceptance time, timestamp quality and period of report are only known after the
    # SGML headers are read (S05, D-047), so the complete per-filing manifest is written here
    fm = mm.drop(columns=[c for c in ("items_list", "item_titles") if c in mm.columns]).merge(
        led[["accession", "url", "status", "checksum", "header_period", "header_acceptance_raw"]].rename(
            columns={"status": "parse_status_s05", "header_period": "period_of_report"}), on="accession", how="left",
        suffixes=("", "_s05"))
    fm["items_raw"] = mm["items_list"].map(lambda x: ",".join(map(str, x))).values
    fm["items_source"] = np.where(mm.accession.map(lambda a: bool(str(hitems.get(a) or ""))), "sgml_header_or_json", "json")
    fm["acceptance_raw"] = fm.get("header_acceptance_raw")
    for c in ("url", "checksum"):
        if f"{c}_s05" in fm:
            fm[c] = fm[f"{c}_s05"].where(fm[f"{c}_s05"].notna(), fm[c])
            fm = fm.drop(columns=[f"{c}_s05"])
    fm["parse_status"] = fm.pop("parse_status_s05")
    ctx.write(fm, "filing_manifest_timed.parquet")
    cnt = ev.groupby(["event_type", "session_bucket"]).size().unstack(fill_value=0)
    cnt.to_csv(ctx.out / "event_counts.csv")
    return {"events": int(len(ev)), "by_type": ev.event_type.value_counts().to_dict(),
            "earnings_types": ev.earnings_type.value_counts().to_dict(), "usable_cutoff": str(cutoff.date()),
            "json_rule_mismatch": acc.attrs.get("json_rule_mismatch")}


# =====================================================================================================  S08
@stage("S08", "Targets + labels", "CPU", ["S07"], ["targets", "labels", "timing"],
       outputs=["targets.parquet", "target_summary.csv"],
       assertions=["INTRA not T_info-eligible", "per-horizon censoring", "T_trade entry >= decision_time"],
       card="T_info (information response) and T_trade (tradable, primary) abnormal returns, labels, vol ratio, |AR|.")
def s08(ctx):
    from .targets.build import build_targets
    cfg = ctx.cfg
    ev = C.events(ctx)
    cal = C.calendar(ctx)
    panel, spy = C.panel_and_spy(ctx, cal)
    uc = C.usable_cutoff(ctx)
    tg = build_targets(ev, panel, spy, C.factors(ctx), cfg, int(uc["usable_cutoff_idx"]))
    # assertions
    intra = ev.loc[ev.session_bucket == "INTRA", "event_id"]
    ti = tg[(tg.target_def == "T_info") & tg.event_id.isin(intra)]
    assert ti.label3.isna().all(), "INTRA events must not carry a T_info label"
    e = ev.set_index("event_id")
    ent = e["ttrade_entry_idx"].astype(int)
    ok = ent < len(cal)
    open_t = pd.Series([cal.open_time(i) for i in ent[ok]], index=ent[ok].index)
    assert (open_t >= pd.to_datetime(e.loc[ok[ok].index, "decision_time"])).all(), "T_trade entry must be at/after decision"
    ctx.write(tg, "targets.parquet")
    sm = tg.groupby(["target_def", "horizon"]).agg(rows=("event_id", "size"), labelled=("label3", lambda s: int(s.notna().sum())),
                                                   censored=("censored", "sum"), up=("label3", lambda s: float((s == 2).mean())),
                                                   side=("label3", lambda s: float((s == 1).mean())),
                                                   down=("label3", lambda s: float((s == 0).mean())),
                                                   ar_mean=("ar_mm", "mean"), ar_std=("ar_mm", "std")).reset_index()
    sm.to_csv(ctx.out / "target_summary.csv", index=False)
    return {"rows": int(len(tg)), "primary_labelled": int(tg[(tg.target_def == cfg["targets"]["primary"]["target_def"])
                                                             & (tg.horizon == cfg["targets"]["primary"]["horizon"])].label3.notna().sum())}


# =====================================================================================================  S09
def _text_job(args):
    from .features.text import firm_text_features, load_lm, model_text
    cik, part, evs, lm_path, max_chars = args
    lm = load_lm(lm_path)
    if not Path(part).exists():
        return pd.DataFrame({"event_id": [e["event_id"] for e in evs]}), pd.DataFrame(columns=["event_id", "text"])
    s = pd.read_parquet(part, columns=["accession", "section_id", "text_raw", "text_masked"])
    raw, msk = {}, {}
    for r in s.itertuples(index=False):
        raw.setdefault(r.accession, {})[r.section_id] = r.text_raw
        msk.setdefault(r.accession, {})[r.section_id] = r.text_masked
    ev = pd.DataFrame(evs)
    f = firm_text_features(ev, raw, lm)
    txt = []
    for e in evs:
        secs = {}
        for a in e["accessions"]:
            for sid, t in (msk.get(a) or {}).items():
                secs[sid] = (secs[sid] + "\n\n" + t) if sid in secs else t
        txt.append((e["event_id"], model_text(secs, e["event_type"], max_chars)))
    return f, pd.DataFrame(txt, columns=["event_id", "text"])


def lm_path(ctx) -> Path:
    p = ctx.storage.inputs / "lexicons" / "Loughran-McDonald_MasterDictionary_1993-2025.csv"
    if not p.exists():
        import requests
        rost = C.roster(ctx.cfg)["lexical"]["loughran_mcdonald"]
        r = requests.get(rost["sheet_export_url"], timeout=120)
        r.raise_for_status()
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_bytes(r.content)
    pin = C.roster(ctx.cfg)["lexical"]["loughran_mcdonald"].get("sha256_verified_2026_10_23")
    got = file_sha256(p)
    if pin and got != pin and not ctx.services.get("fixture_root"):   # hard rule 12 (offline fixtures use a tiny list)
        raise RuntimeError(f"Loughran-McDonald dictionary sha256 {got} != pinned {pin} (configs/models.yaml); "
                           "verify the new file and update the pin as a logged decision")
    return p


@stage("S09", "Structured + lexical features", "CPU", ["S08"], ["text", "accounting", "labels"],
       outputs=["features.parquet", "feature_registry.csv", "event_text.parquet"],
       assertions=["availability <= decision_time for all"],
       card="Price, metadata, accounting (SUE, XBRL), lexical (LM), change and novelty features at decision time.")
def s09(ctx):
    from .features.accounting import pit_facts
    from .features.price import price_features
    from .features.registry import availability_violations, registry_frame, sue_rule_violations
    from .track_a.event_study import window_cars
    cfg = ctx.cfg
    ev = C.events(ctx)
    cal = C.calendar(ctx)
    panel, spy = C.panel_and_spy(ctx, cal)
    feats = [ev[["event_id"]].copy()]
    pf = price_features(ev, panel, spy)
    feats.append(pf)
    # ---- metadata
    md = pd.DataFrame({"event_id": ev.event_id})
    for t in ("periodic_10K", "periodic_10Q", "8K"):
        md[f"meta_type_{t}"] = (ev.event_type == t).astype(float)
    tax = C.taxonomy(cfg)
    cats = sorted(set(tax.modern.values()) | set(tax.legacy.values()) | {tax.catch_all})
    for c in cats:
        md[f"meta_cat_{c}"] = ev.categories.map(lambda x: float(c in x))
    md["meta_n_items"] = ev["items"].map(len).astype(float)
    md["meta_n_accessions"] = ev.accessions.map(len).astype(float)
    for sid in [s for v in cfg["text"]["sections"].values() for s in v]:
        md[f"meta_has_{sid}"] = ev.section_ids.map(lambda x: float(sid in x))
    for ind in sorted(ev.ff12_industry.fillna("Other").unique()):
        md[f"meta_ff12_{ind}"] = (ev.ff12_industry.fillna("Other") == ind).astype(float)
    md["meta_year"] = ev.year.astype(float)
    md["meta_month"] = pd.to_datetime(ev.decision_date).dt.month.astype(float)
    md["meta_dow"] = pd.to_datetime(ev.acceptance_time).dt.dayofweek.astype(float)
    for b in ("PRE", "INTRA", "POST", "NONTRADING"):
        md[f"meta_bucket_{b}"] = (ev.session_bucket == b).astype(float)
    for q in ("A", "B", "C"):
        md[f"meta_tsq_{q}"] = (ev.ts_quality == q).astype(float)
    for et in ("earnings_announcement_verified", "earnings_date_ambiguous", "periodic_filing_only"):
        md[f"meta_earn_{et}"] = (ev.earnings_type == et).astype(float)
    lag = (pd.to_datetime(ev.filing_date) - pd.to_datetime(ev.report_date.replace("", np.nan), errors="coerce")).dt.days
    md["meta_filing_lag_days"] = np.where(ev.event_type.str.startswith("periodic"), lag, np.nan)
    ev_sorted = ev.sort_values("decision_time")
    prev = ev_sorted.groupby(["cik", "event_type"])["decision_time"].shift(1)
    md["meta_days_since_prev_same"] = ((ev_sorted.decision_time - prev).dt.total_seconds() / 86400).reindex(ev.index)
    md["_avail_metadata"] = pd.to_datetime(ev.acceptance_time, utc=True)
    feats.append(md)
    # ---- accounting
    facts = C.read(ctx, "S06", "xbrl_facts.parquet")
    tags = [("Assets", "inst"), ("Liabilities", "inst"), ("StockholdersEquity", "inst"), ("NetIncomeLoss", "fy"),
            ("Revenues", "fy"), ("RevenueFromContractWithCustomerExcludingAssessedTax", "fy"), ("SalesRevenueNet", "fy"),
            ("EarningsPerShareDiluted", "q"), (cfg["accounting"]["shares_tag"], "inst")]
    if len(facts):
        pit = pit_facts(ev, facts, tags)
    else:
        pit = pd.DataFrame({"event_id": ev.event_id, "_avail_accounting": pd.NaT})
    ac = pd.DataFrame({"event_id": pit.event_id})
    g = lambda c: pit[c] if c in pit else np.nan  # noqa: E731
    with np.errstate(divide="ignore", invalid="ignore"):
        ac["acc_log_assets"] = np.log(g("Assets_inst"))
        ac["acc_leverage"] = g("Liabilities_inst") / g("Assets_inst")
        ac["acc_roa"] = g("NetIncomeLoss_fy") / g("Assets_inst")
        rev = pd.concat([pd.Series(g("Revenues_fy")), pd.Series(g("RevenueFromContractWithCustomerExcludingAssessedTax_fy")),
                         pd.Series(g("SalesRevenueNet_fy"))], axis=1).bfill(axis=1).iloc[:, 0] if len(pit) else np.nan
        ac["acc_log_revenue"] = np.log(rev)
        ac["acc_eps_q_latest"] = g("EarningsPerShareDiluted_q")
        sh = g(f"{cfg['accounting']['shares_tag']}_inst")
        cidx = cal.v_last_close_at_or_before(pd.DatetimeIndex(pd.to_datetime(ev.set_index("event_id").loc[pit.event_id, "decision_time"])).tz_convert("UTC").asi8)
        tick = ev.set_index("event_id").loc[pit.event_id, "ticker"].to_numpy()
        price = np.array([panel[t]["close_split_adj"][i] if panel.has(t) and 0 <= i < len(cal) else np.nan for t, i in zip(tick, cidx)])
        mcap = price * np.asarray(sh, float)
        ac["acc_log_mcap"] = np.log(mcap)
        ac["acc_book_to_market"] = np.asarray(g("StockholdersEquity_inst"), float) / mcap
    ac = ac.replace([np.inf, -np.inf], np.nan)
    ac["_avail_accounting"] = pit["_avail_accounting"].values
    sue = C.read(ctx, "S06", "sue_by_accession.parquet")
    srcs = C.read(ctx, "S06", "sue_sources.parquet")
    a2e = {a: e for e, accs in zip(ev.event_id, ev.accessions) for a in accs}
    if len(sue):
        sue["event_id"] = sue.accession.map(a2e)
        own = sue.dropna(subset=["event_id"]).sort_values("sue_reason").drop_duplicates("event_id")
        own = own[["event_id", "sue", "n_prior_diffs", "hist_max_public_time"]].rename(columns={"sue": "sue_event", "n_prior_diffs": "sue_n_diffs"})
        ac = ac.merge(own, on="event_id", how="left")
        hm = pd.to_datetime(ac.pop("hist_max_public_time"), utc=True)
        aa = pd.to_datetime(ac["_avail_accounting"], utc=True)
        ac["_avail_accounting"] = pd.concat([aa, hm], axis=1).max(axis=1)
        # last available SUE (from an earlier event) for every event
        evs = ev[["event_id", "cik", "decision_time"]].merge(own[["event_id", "sue_event"]], on="event_id", how="left")
        evs = evs.sort_values("decision_time")
        evs["sue_last"] = evs.groupby("cik")["sue_event"].transform(lambda s: s.shift(1).ffill())
        ac = ac.merge(evs[["event_id", "sue_last"]], on="event_id", how="left")
        sue_src = srcs.assign(event_id=srcs.accession.map(a2e)).dropna(subset=["event_id"])
    else:
        ac["sue_event"] = ac["sue_n_diffs"] = ac["sue_last"] = np.nan
        sue_src = pd.DataFrame(columns=["event_id", "source_accession", "source_public_time"])
    feats.append(ac)
    # ---- EAR of the most recent different-cluster verified release whose [-1,+1] window closed before decision
    rels = ev[ev.earnings_type == "earnings_announcement_verified"]
    ear = pd.DataFrame({"event_id": ev.event_id, "ear_last": np.nan, "ear_src_cluster": None, "_avail_ear": pd.NaT})
    if len(rels):
        uc = C.usable_cutoff(ctx)
        wc = window_cars(rels, panel, spy, None, cfg, int(uc["usable_cutoff_idx"]))
        rels = rels.merge(wc[["event_id", "car_m1_1"]], on="event_id", how="left")
        rels["win_close"] = [cal.close_time(min(i + 1, len(cal) - 1)) for i in rels.trackA_day0_idx]
        out = []
        for cik, g2 in ev.groupby("cik"):
            rr = rels[rels.cik == cik].sort_values("win_close")
            for r in g2.itertuples(index=False):
                c = rr[(rr.win_close < r.decision_time) & (rr.cluster_id != r.cluster_id)]
                if len(c):
                    x = c.iloc[-1]
                    out.append((r.event_id, x.car_m1_1, x.cluster_id, x.win_close))
        if out:
            e2 = pd.DataFrame(out, columns=["event_id", "ear_last", "ear_src_cluster", "_avail_ear"])
            ear = ear[["event_id"]].merge(e2, on="event_id", how="left")
    ear["_avail_ear"] = pd.to_datetime(ear["_avail_ear"], utc=True)
    feats.append(ear)
    # ---- lexical / change / novelty (per firm, parallel)
    lmp = str(lm_path(ctx))
    jobs = []
    sec_dir = ctx.dep("S05") / "sections"
    for cik, g3 in ev.groupby("cik"):
        evs = g3[["event_id", "event_type", "decision_time", "accessions", "public_available_time"]].to_dict("records")
        jobs.append((int(cik), str(sec_dir / f"{int(cik):010d}.parquet"), evs, lmp, int(cfg["text"]["model_text_chars"])))
    tf, tt = [], []
    nproc = 1 if ctx.services.get("fixture_root") else min(8, max(1, (os.cpu_count() or 2) - 1))
    if nproc == 1:
        res = [_text_job(j) for j in jobs]
    else:
        with ProcessPoolExecutor(max_workers=nproc, mp_context=_mp_ctx()) as ex:
            res = list(ex.map(_text_job, jobs, chunksize=2))
    for f, t in res:
        tf.append(f)
        tt.append(t)
    txf = pd.concat(tf, ignore_index=True)
    txf["_avail_lexical"] = txf.event_id.map(ev.set_index("event_id")["public_available_time"]).pipe(pd.to_datetime, utc=True)
    feats.append(txf)
    F = feats[0]
    for f in feats[1:]:
        F = F.merge(f, on="event_id", how="left")
    # ---- registry + availability assertions (EAR source cluster check handled by the registry)
    dec = ev.set_index("event_id").loc[F.event_id, "decision_time"].reset_index(drop=True)
    viol = availability_violations(F, dec)
    assert len(viol) == 0, f"availability violations: {viol.head().to_dict('records')}"
    sv = sue_rule_violations(sue_src.rename(columns={"source_public_time": "source_public_time"}), ev)
    assert len(sv) == 0, f"SUE rule violations: {sv.head().to_dict('records')}"
    cols = [c for c in F.columns if c != "event_id" and not c.startswith("_") and c != "ear_src_cluster"]
    reg = registry_frame(cols)
    reg.to_csv(ctx.out / "feature_registry.csv", index=False)
    F["txt_has_text"] = F["txt_has_text"].fillna(False).astype(bool)
    ctx.write(F, "features.parquet")
    ctx.write(pd.concat(tt, ignore_index=True), "event_text.parquet")
    return {"features": len(cols), "events": int(len(F)), "families": reg.family.value_counts().to_dict(),
            "availability_violations": 0, "sue_rule_violations": 0}


# =====================================================================================================  S10
@stage("S10", "Splits + cohorts + leakage audit", "CPU", ["S09"], ["splits", "seed", "contam_window_sessions"],
       outputs=["splits.parquet", "blocks.csv", "fold_provenance.csv", "leakage_audit.csv", "planted_leaks.json"],
       assertions=["all assertions pass", "4 planted leaks caught"],
       card="Walk-forward folds (expanding), development holdout, locked test; purge + embargo; stress folds; leakage audit.")
def s10(ctx):
    from .rag.core import retrieval_guard
    from .splits.leakage import audit, planted_leaks
    from .splits.provenance import fold_labels, ledger_frame
    from .splits.walkforward import assign_roles, blocks_frame, make_blocks
    cfg = ctx.cfg
    ev = C.events(ctx)
    tg = C.read(ctx, "S08", "targets.parquet")
    F = C.read(ctx, "S09", "features.parquet")
    uc = C.usable_cutoff(ctx)
    blocks = make_blocks(cfg, pd.Timestamp(uc["usable_cutoff"]))
    H = int(cfg["splits"]["purge_horizon_sessions"])
    hz = max([h for h in cfg["targets"]["horizons"] if h <= H] or [max(cfg["targets"]["horizons"])])
    ex = tg[tg.target_def.isin(["T_info", "T_trade"]) & (tg.horizon == hz)].groupby("event_id")["exit_date"].max()
    sp = assign_roles(ev, blocks, cfg, ev.event_id.map(ex))
    ctx.write(sp, "splits.parquet")
    bf = blocks_frame(blocks)
    bf.to_csv(ctx.out / "blocks.csv", index=False)
    rost = C.roster(cfg)
    fl = fold_labels(rost, blocks)
    fl.to_csv(ctx.out / "fold_provenance.csv", index=False)
    ledger_frame(rost).to_csv(ctx.out / "model_provenance_ledger.csv", index=False)
    ledger_frame(rost).to_csv(ctx.storage.reports / "model_provenance_ledger.csv", index=False)
    sue_src = C.read(ctx, "S06", "sue_sources.parquet")
    a2e = {a: e for e, accs in zip(ev.event_id, ev.accessions) for a in accs}
    sue_src = sue_src.assign(event_id=sue_src.accession.map(a2e)).dropna(subset=["event_id"])
    pl = planted_leaks(F, ev, tg, sue_src, retrieval_guard)
    write_json(ctx.out / "planted_leaks.json", pl)
    si = C.read(ctx, "S05", "sections_index.parquet")
    extra = {"planted": pl, "retrieval_guard_ok": retrieval_guard(), "mask_firm_total": int(si.mask_firm.sum()) if len(si) else 0,
             "json_rule_mismatch": uc.get("json_rule_mismatch"), "ear_violations": 0, "sue_violations": 0}
    univ = C.universe(ctx)
    la = audit(ev, F, tg, sp, sue_src, univ, cfg, extra)
    la.to_csv(ctx.out / "leakage_audit.csv", index=False)
    la.to_csv(ctx.storage.reports / "leakage_audit.csv", index=False)
    fails = la[la.status == "fail"]
    assert all(p["caught"] for p in pl), f"planted leaks missed: {[p for p in pl if not p['caught']]}"
    assert fails.empty, f"leakage audit failures: {fails.to_dict('records')}"
    return {"folds": len(blocks["folds"]), "blocks": bf.assign(start=bf.start.astype(str), end_exclusive=bf.end_exclusive.astype(str)).to_dict("records"),
            "planted_leaks_caught": sum(p["caught"] for p in pl), "audit": la.status.value_counts().to_dict()}


In [ ]:
%%writefile $REPO/src/secprice/stages_model.py
"""Track B stages: S11 baselines, S13 preflight, S14 embeddings, S15 heads/G3/fine-tune, S16 RAG, S17 distillation,
S18 dev holdout + G4, S19 locked test, S22 backtest (SPEC.md §9, §11, §13.3, §17.1)."""
from __future__ import annotations

import hashlib
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd

from . import common as C
from .config import canonical_json, get
from .evaluation.bootstrap import macro_f1_ci, paired_macro_f1_diff
from .evaluation.metrics import classification_metrics, decide, power_label
from .models.ladder import RUNGS, Ladder, fit_verdict, summarize
from .runner import stage
from .storage import write_json

PCOLS_ = ["p_down", "p_side", "p_up"]
PRIMARY_METRICS = ["macro_f1", "accuracy", "balanced_accuracy", "weighted_f1", "auc_macro_ovr", "prauc_macro", "kappa", "mcc",
                   "logloss", "logloss_prior", "brier", "ece", "acc_at_50", "macro_f1_at_50", "acc_at_25", "macro_f1_at_25",
                   "acc_at_10", "macro_f1_at_10", "f1_up", "f1_down", "f1_sideways", "auc_up", "auc_down",
                   "guardrail_logloss_ok", "guardrail_ece_ok"]


class Results:
    """results_long rows (SPEC.md §16)."""

    def __init__(self):
        self.rows: list[dict] = []

    def add(self, run_id, rung, model_id, revision, leaderboard, cohort, target_def, horizon, block, event_set, metrics: dict,
            ci: dict | None, n_events, n_firms, n_dates, n_overlap, tuned: str, extra: dict | None = None):
        for k in PRIMARY_METRICS + ["r2", "oos_r2_vs_hist_mean", "mae", "rmse", "spearman", "directional_accuracy",
                                    "rank_ic_mean", "rank_ic_t", "rank_ic_icir", "qlike"]:
            if k in metrics and metrics[k] is not None and not isinstance(metrics[k], (list, dict, str)):
                row = {"run_id": run_id, "track": "B", "rung": rung, "model_id": model_id, "revision": revision,
                       "leaderboard": leaderboard, "cohort": cohort, "target_def": target_def, "horizon": horizon,
                       "block": block, "event_set": event_set, "coverage": 1.0, "metric": k, "value": float(metrics[k]),
                       "ci_low": None, "ci_high": None, "n_events": n_events, "n_firms": n_firms, "n_dates": n_dates,
                       "n_overlap": n_overlap, "tuned": tuned, **(extra or {})}
                if k == "macro_f1" and ci:
                    row["ci_low"], row["ci_high"] = ci.get("ci_low"), ci.get("ci_high")
                self.rows.append(row)

    COLS = ["run_id", "track", "rung", "model_id", "revision", "leaderboard", "cohort", "target_def", "horizon", "block",
            "event_set", "coverage", "metric", "value", "ci_low", "ci_high", "n_events", "n_firms", "n_dates", "n_overlap", "tuned"]

    def frame(self) -> pd.DataFrame:
        df = pd.DataFrame(self.rows)
        for c in self.COLS:
            if c not in df.columns:
                df[c] = pd.Series(dtype=object)
        return df


def _counts(ev_idx: pd.DataFrame, ids, clean_col) -> tuple:
    e = ev_idx.loc[ids]
    return int(len(e)), int(e.cik.nunique()), int(e.decision_date.nunique()), int(e[clean_col].sum())


def _leaderboard(rung: str, enc: str | None, prov: pd.DataFrame, block: str) -> str:
    if enc is None and rung not in ("L8", "L8_ft"):
        return "strict"
    if rung == "L8":
        return "retrospective"      # extraction teacher (Qwen) has no documented cutoff
    key = enc
    r = prov[(prov.model == key) & (prov.block == block)]
    return r.leaderboard.iloc[0] if len(r) else "retrospective"


def build_ladder(ctx, with_emb: bool = False, with_ext: bool = False) -> Ladder:
    ev = C.events(ctx)
    F = C.read(ctx, "S09", "features.parquet")
    # only the columns the ladder reads: the full targets table is ~7M rows at full scale (memory)
    tg = C.read(ctx, "S08", "targets.parquet", columns=["event_id", "target_def", "horizon", "label3", "weight_uniq",
                                                         "ar_mm", "vol_ratio", "abs_std"])
    sp = C.read(ctx, "S10", "splits.parquet")
    tx = C.read(ctx, "S09", "event_text.parquet").set_index("event_id")["text"]
    prov = pd.read_csv(ctx.dep("S10") / "fold_provenance.csv")
    embs = {}
    if with_emb:
        embs = load_embeddings(ctx)
        for enc, E in embs.items():
            extra = E.attrs.get("extra")
            if extra is not None and len(extra):
                ex = extra.set_index("event_id")
                for c in [c for c in ex.columns if c.startswith("tone_")]:
                    F = F.merge(ex[[c]].rename(columns={c: f"emb_{enc}_{c}"}), left_on="event_id", right_index=True, how="left")
    if with_ext:
        x = ctx.dep_or_none("S17")
        if x is not None and (x / "ext_features.parquet").exists():
            F = F.merge(pd.read_parquet(x / "ext_features.parquet"), on="event_id", how="left")
    return Ladder(ctx.cfg, ev, F, tg, sp, tx, embs, ctx.out, ctx.log, prov)


def load_embeddings(ctx) -> dict:
    d = ctx.dep_or_none("S14")
    out = {}
    if d is None:
        return out
    for enc_dir in sorted((d / "embeddings").glob("*")):
        f = enc_dir / "vectors.npy"
        if not f.exists():
            continue
        V = np.load(f).astype(np.float32)
        idx = pd.read_parquet(enc_dir / "index.parquet")
        dim = V.shape[1] // 2
        E = pd.DataFrame(V[:, :dim], index=idx.event_id.values)   # mean-pooled half (max half kept on disk)
        ex = pd.read_parquet(enc_dir / "extra.parquet") if (enc_dir / "extra.parquet").exists() else None
        E.attrs["extra"] = ex
        out[enc_dir.name] = E
    return out


def _run_and_record(lad: Ladder, res: Results, preds: list, rung, head, params, lab, ids, cohort, tdef, h, tuned, enc=None,
                    curves=False, task="cls", n_boot=1000, seed=0):
    r = lad.run(rung, head, params, lab, ids, "walk_forward", task, enc, keep=True, curves=curves)
    s = summarize(r, lab, n_boot, seed, task)
    run_id = f"{cohort}|{rung}|{head}|{enc or '-'}|{tdef}|h{h}|{tuned}"
    evx = lad.ev
    if len(r["preds"]):
        n = _counts(evx, r["preds"].event_id, lad.clean_col)
        lb = _leaderboard(rung, enc, lad.prov, "wf0") if lad.prov is not None else "strict"
        res.add(run_id, rung, head, enc or "n/a", lb, cohort, tdef, h, "walk_forward_pooled", "all", s["pooled"], s.get("ci"),
                *n, tuned, {"mean_fold_score": s["mean_fold"], "std_fold_score": s["std_fold"]})
        clean = r["preds"][~r["preds"].event_id.map(evx[lad.clean_col]).astype(bool)]
        if task == "cls" and len(clean) > 30:
            pri = np.bincount(clean.y.astype(int), minlength=3) / len(clean)
            mc = classification_metrics(clean.y.values, clean[["p_down", "p_side", "p_up"]].values, pri, clean.yhat.values)
            res.add(run_id, rung, head, enc or "n/a", lb, cohort, tdef, h, "walk_forward_pooled", "clean", mc, None,
                    *_counts(evx, clean.event_id, lad.clean_col), tuned)
        for fm in r["fold_metrics"]:
            if "skipped" in fm:
                continue
            res.add(run_id, rung, head, enc or "n/a", _leaderboard(rung, enc, lad.prov, f"wf{fm['fold']}") if lad.prov is not None else "strict",
                    cohort, tdef, h, f"wf{fm['fold']}", "all", fm, None, fm.get("n"), None, None, None, tuned)
        p = r["preds"].assign(run_id=run_id, model_id=head, rung=rung, encoder=enc, cohort=cohort, target_def=tdef,
                              horizon=h, tuned=tuned)
        preds.append(p)
    return r, s, run_id


# =====================================================================================================  S11
@stage("S11", "Baselines L0-L4, L6", "CPU", ["S10"], ["tuning", "targets", "labels", "bootstrap", "audit", "seed", "models.pca_dims"],
       outputs=["predictions.parquet", "results_long.parquet", "ladder_summary.csv", "optuna_trials.csv", "curves.json",
                "fit_diagnosis.csv", "best_params.json"],
       assertions=["common cohort recorded", "preprocessing fitted on training rows only"],
       card="Majority, price, metadata, accounting and lexical baselines and the best structured set (L6), default and tuned.")
def s11(ctx):
    cfg = ctx.cfg
    T = cfg["targets"]
    lad = build_ladder(ctx)
    prim = T["primary"]
    lab = lad.labelled(prim["target_def"], prim["horizon"])
    ids = lab.index
    res, preds = Results(), []
    rungs = ["L1", "L2", "L3", "L4", "L4_levels", "L4_change", "L6", "L6_nochange"]
    common = ids[lad.cohort(["L1", "L2", "L3", "L4", "L6"], [None], ids)]
    cohorts = {"common": common}
    nb, seed = int(cfg["bootstrap"]["n"]), int(cfg["seed"])
    summ, fitd, best_params, trials_all = [], [], {}, []
    _run_and_record(lad, res, preds, "L0", "prior", None, lab, common, "common", prim["target_def"], prim["horizon"], "default",
                    n_boot=nb, seed=seed)
    maj = None
    rr = [r for r in res.rows if r["rung"] == "L0" and r["metric"] == "macro_f1" and r["block"] == "walk_forward_pooled"]
    if rr:
        maj = rr[0]["value"]
    study_db = ctx.storage.hot_root / f"optuna_S11_{ctx.stage_hash}.sqlite3"   # SQLite is unsafe on Drive FUSE
    tuned_rungs = set(cfg["tuning"].get("tuned_rungs") or rungs)
    parent = {"L4_levels": "L4", "L4_change": "L4", "L6_nochange": "L6"}
    for rung in sorted(rungs, key=lambda r: r in parent):        # tuned rungs first; ablation variants inherit
        for head in cfg["tuning"]["heads"]:
            r0, s0, _ = _run_and_record(lad, res, preds, rung, head, None, lab, common, "common", prim["target_def"],
                                        prim["horizon"], "default", curves=(rung == "L6"), n_boot=nb, seed=seed)
            if rung in tuned_rungs:
                n_tr = int(cfg["tuning"]["trials"].get(head, 10))
                bp, tr = lad.tune(rung, head, lab, common, n_tr, int(cfg["tuning"]["timeout_s_per_study"]), study_db,
                                  f"common|{rung}|{head}|{prim['target_def']}|h{prim['horizon']}")
                tr["rung"], tr["head"] = rung, head
                trials_all.append(tr)
                label = "tuned"
            else:
                bp = best_params.get(f"{parent.get(rung, rung)}|{head}", {})
                label = "tuned"                                  # parameters transferred from the parent rung (D-021)
            best_params[f"{rung}|{head}"] = bp
            r1, s1, rid = _run_and_record(lad, res, preds, rung, head, bp, lab, common, "common", prim["target_def"],
                                          prim["horizon"], label, curves=(rung == "L6"), n_boot=nb, seed=seed)
            summ.append({"rung": rung, "head": head, "default_mean_f1": s0["mean_fold"], "tuned_mean_f1": s1["mean_fold"],
                         "tuned_std": s1["std_fold"], "pooled_f1": (s1["pooled"] or {}).get("macro_f1"),
                         "ci_low": (s1.get("ci") or {}).get("ci_low"), "ci_high": (s1.get("ci") or {}).get("ci_high"),
                         "n": s1["n"], "run_id": rid})
            fitd.append({"run_id": rid, "rung": rung, "head": head, "mean_val": s1["mean_fold"], "mean_train": s1["mean_train"],
                         "std": s1["std_fold"], "verdict": fit_verdict(s1["mean_fold"], s1["mean_train"], s1["std_fold"], maj, cfg["audit"])})
            # maximum-available cohort with the tuned parameters
            mx = ids[lad.requirement_mask(rung, None, ids)]
            _run_and_record(lad, res, preds, rung, head, bp, lab, mx, "max_available", prim["target_def"], prim["horizon"],
                            "tuned", n_boot=nb, seed=seed)
    sm = pd.DataFrame(summ)
    sm.to_csv(ctx.out / "ladder_summary.csv", index=False)
    l6 = sm[sm.rung == "L6"].sort_values("tuned_mean_f1", ascending=False)
    best_l6 = l6.iloc[0].to_dict() if len(l6) else None
    write_json(ctx.out / "best_params.json", {"params": best_params, "best_L6": best_l6})
    # complementary T_info model (separate model; never the T_trade backtest signal) + regression heads
    comp = T["complementary"]
    lab_i = lad.labelled(comp["target_def"], comp["horizon"])
    if best_l6:
        for rung in ("L1", "L4", "L6"):
            ci_ids = lab_i.index[lad.cohort(["L1", "L2", "L3", "L4", "L6"], [None], lab_i.index)]
            _run_and_record(lad, res, preds, rung, best_l6["head"], best_params.get(f"{rung}|{best_l6['head']}"), lab_i, ci_ids,
                            "common", comp["target_def"], comp["horizon"], "tuned_transferred", n_boot=nb, seed=seed)
    for tdef, h, col in (("T_trade", 1, "ar_mm"), ("T_trade", 5, "ar_mm"), ("T_trade", 20, "vol_ratio"), ("T_trade", 1, "abs_std")):
        if h not in T["horizons"]:
            continue
        lr = lad.labelled(tdef, h, col)
        rid_ids = lr.index[lad.cohort(["L1", "L2", "L3", "L4", "L6"], [None], lr.index)]
        for rung in ("L0", "L1", "L6"):
            r, s, rid = _run_and_record(lad, res, preds, rung, "lightgbm", None, lr, rid_ids, "common", f"{tdef}:{col}", h,
                                        "default", task="reg", n_boot=nb, seed=seed)
    # learning curve + adversarial validation + permutation null + random-split demo (L6 best)
    diag = {}
    if best_l6:
        head, bp = best_l6["head"], best_params.get(f"L6|{best_l6['head']}")
        folds = lad.folds("walk_forward")
        lc = []
        if folds:
            fl = folds[-1]
            tr, te = lad.fold_ids("walk_forward", fl, common)
            trs = lab.loc[tr].sort_values("date")
            for frac in (0.2, 0.4, 0.6, 0.8, 1.0):
                keep = trs.index[int(len(trs) * (1 - frac)):]
                sub = keep.append(te)
                r = lad.run("L6", head, bp, lab, sub, "walk_forward", keep=False, folds=[fl])
                lc.append({"train_fraction": frac, "n_train": len(keep), "val_macro_f1": r["mean_score"],
                           "train_macro_f1": r["mean_train_score"]})
        pd.DataFrame(lc).to_csv(ctx.out / "learning_curves.csv", index=False)
        diag["adversarial"] = adversarial(lad, common, lab)
        pd.DataFrame(diag["adversarial"]).to_csv(ctx.out / "adversarial.csv", index=False)
        nperm = int(get(cfg, "tuning.permutations", 10))
        pm = permutation_null(lad, head, bp, lab, common, nperm, seed)
        pd.DataFrame(pm).to_csv(ctx.out / "permutation.csv", index=False)
        rd = lad.run("L6", head, bp, lab, common, "random_demo", keep=False)
        write_json(ctx.out / "random_demo.json", {"random_split_macro_f1": rd["mean_score"], "walk_forward_macro_f1": best_l6["tuned_mean_f1"],
                                                  "note": "pedagogical only"})
        if folds:
            tr, te = lad.fold_ids("walk_forward", folds[-1], common)
            if len(tr) > 100:
                from .models.heads import enet_certificate
                Xtr, _, _ = lad.matrices("L6", tr, [te[:1]])
                n = min(len(tr), 20000)
                cert = enet_certificate(Xtr[-n:], lab.loc[tr, "y"].to_numpy()[-n:], lab.loc[tr, "w"].to_numpy()[-n:],
                                        best_params.get("L6|enet") or {"C": 0.1, "l1_ratio": 0.5})
                write_json(ctx.out / "enet_certificate.json", cert)
    if best_l6:
        bucket_analysis(ctx, lad, res, preds, lab, common, best_l6["head"], best_params, prim, nb, seed)
    rl = res.frame()
    rl["config_counter"] = lad.config_counter
    ctx.write(rl, "results_long.parquet")
    ctx.write(pd.concat(preds, ignore_index=True) if preds else pd.DataFrame(), "predictions.parquet")
    pd.concat(trials_all, ignore_index=True).to_csv(ctx.out / "optuna_trials.csv", index=False) if trials_all else None
    (ctx.out / "curves.json").write_text(json.dumps(lad.curves[:200], default=float), encoding="utf-8")
    pd.DataFrame(fitd).to_csv(ctx.out / "fit_diagnosis.csv", index=False)
    if study_db.exists():
        import shutil
        shutil.copy2(study_db, ctx.out / "optuna.sqlite3")
    write_json(ctx.out / "cohorts.json", {"common_primary_n": int(len(common)), "labelled_primary_n": int(len(ids))})
    return {"common_cohort": int(len(common)), "labelled": int(len(ids)), "best_L6": best_l6, "configs_tried": lad.config_counter}


def bucket_analysis(ctx, lad: Ladder, res: Results, preds: list, lab, common, head, best_params, prim, nb, seed) -> None:
    """S&P 500 vs non-S&P 500 vs combined (owner request, D-056).
    (1) composition of each bucket; (2) the combined (pooled) walk-forward predictions scored per bucket;
    (3) bucket-only models (same rung, head and tuned parameters, same folds) and a paired test, on identical rows,
    of pooled minus bucket-only macro-F1: does training on both groups help each group?"""
    from .evaluation.buckets import composition, pooled_vs_bucket, score_predictions
    bk = C.event_buckets(ctx)
    bmap = bk.set_index("event_id").sp_bucket
    allp = pd.concat(preds, ignore_index=True) if preds else pd.DataFrame()
    tdef, h = prim["target_def"], prim["horizon"]
    key = [f"common|L0|prior|-|{tdef}|h{h}|default"] + [f"common|{r}|{head}|-|{tdef}|h{h}|tuned" for r in ("L1", "L2", "L3", "L4", "L6")]
    sc = score_predictions(allp[allp.run_id.isin(key)] if len(allp) else allp, bk, nb, seed)
    if len(sc):
        sc["rung"] = sc.run_id.str.split("|").str[1]
    sc.to_csv(ctx.out / "bucket_scores.csv", index=False)
    labs = pd.DataFrame({"event_id": lab.index, "label3": lab["y"].values})
    feat = C.read(ctx, "S09", "features.parquet")
    composition(C.events(ctx), bk, C.universe(ctx), labs, feat).to_csv(ctx.out / "bucket_composition.csv", index=False)
    rows = []
    min_rows = max(100, int(ctx.cfg["tuning"].get("min_train_rows", 50)))
    for rung in ("L2", "L6"):
        bp = best_params.get(f"{rung}|{head}", {})
        rid = f"common|{rung}|{head}|-|{tdef}|h{h}|tuned"
        pooled = allp[allp.run_id == rid] if len(allp) else allp
        for b in ("SP500", "NonSP500"):
            ids_b = common[common.map(bmap).fillna("NonSP500") == b]
            row = {"rung": rung, "head": head, "sp_bucket": b, "bucket_events": int(len(ids_b))}
            if len(ids_b) < min_rows:
                rows.append({**row, "note": f"fewer than {min_rows} events in the bucket"})
                continue
            r, s, brid = _run_and_record(lad, res, preds, rung, head, bp, lab, ids_b, f"bucket_{b}", tdef, h,
                                         "tuned_transferred", n_boot=nb, seed=seed)
            if r["preds"].empty:
                rows.append({**row, "note": "no walk-forward predictions"})
                continue
            cmp = pooled_vs_bucket(pooled[pooled.event_id.isin(ids_b)], r["preds"], nb, seed)
            rows.append({**row, "bucket_only_macro_f1": (s.get("pooled") or {}).get("macro_f1"),
                         "bucket_only_ci_low": (s.get("ci") or {}).get("ci_low"), "bucket_only_ci_high": (s.get("ci") or {}).get("ci_high"),
                         **cmp})
    pd.DataFrame(rows).to_csv(ctx.out / "bucket_pooled_vs_own.csv", index=False)


def adversarial(lad: Ladder, ids, lab) -> list[dict]:
    """Per fold: can a classifier tell training rows from validation rows? (AUC near 0.5 does not prove no shift.)"""
    import lightgbm as lgb
    from sklearn.metrics import roc_auc_score
    from sklearn.model_selection import StratifiedKFold
    out = []
    for f in lad.folds("walk_forward"):
        tr, te = lad.fold_ids("walk_forward", f, ids)
        if len(tr) < 100 or len(te) < 50:
            continue
        X, _, names = lad.matrices("L6", tr, [te[:1]])
        Xt, _, _ = lad.matrices("L6", te, [te[:1]])
        Xa = np.vstack([X, Xt])
        ya = np.r_[np.zeros(len(X)), np.ones(len(Xt))]
        from sklearn.metrics import roc_curve
        aucs = {}
        for tag, drop in (("all_features", lambda n: n == "meta_year"),
                          ("firm_level_only", lambda n: n.startswith("mkt_") or n in ("meta_year", "meta_month", "meta_dow"))):
            keep = [i for i, n in enumerate(names) if not drop(n)]
            Xk = Xa[:, keep]
            pr = np.zeros(len(ya))
            for a, b in StratifiedKFold(3, shuffle=True, random_state=0).split(Xk, ya):
                m = lgb.LGBMClassifier(n_estimators=150, learning_rate=0.1, num_leaves=31, verbose=-1).fit(Xk[a], ya[a])
                pr[b] = m.predict_proba(Xk[b])[:, 1]
            aucs[tag] = (float(roc_auc_score(ya, pr)), pr)
            if tag == "firm_level_only":
                # which features separate the periods (gain share of a fit on all rows of this fold)
                g = lgb.LGBMClassifier(n_estimators=150, learning_rate=0.1, num_leaves=31, verbose=-1).fit(Xk, ya)
                gain = g.booster_.feature_importance("gain")
                top = np.argsort(-gain)[:5]
                drivers = [[names[keep[i]], round(float(gain[i] / max(gain.sum(), 1e-12)), 3)] for i in top]
        pr = aucs["firm_level_only"][1]
        fpr, tpr, _ = roc_curve(ya, pr)
        out.append({"fold": f, "auc": aucs["firm_level_only"][0], "auc_all_features": aucs["all_features"][0],
                    "top_shift_features": json.dumps(drivers), "n_train": len(X), "n_val": len(Xt),
                    "fpr": json.dumps(fpr[:: max(1, len(fpr) // 50)].round(4).tolist()),
                    "tpr": json.dumps(tpr[:: max(1, len(tpr) // 50)].round(4).tolist())})
    return out


def permutation_null(lad: Ladder, head, params, lab, ids, n_perm, seed) -> list[dict]:
    """Labels permuted within month blocks (class counts per month preserved), refit, score: null of macro-F1."""
    rng = np.random.default_rng(seed)
    out = []
    obs = lad.run("L6", head, params, lab, ids, "walk_forward", keep=False)["mean_score"]
    out.append({"perm": -1, "macro_f1": obs, "kind": "observed"})
    month = lab["date"].dt.to_period("M")
    for k in range(n_perm):
        lp = lab.copy()
        for _, g in lp.groupby(month):
            lp.loc[g.index, "y"] = rng.permutation(g["y"].to_numpy())
        out.append({"perm": k, "macro_f1": lad.run("L6", head, params, lp, ids, "walk_forward", keep=False)["mean_score"],
                    "kind": "permuted"})
    return out


# =====================================================================================================  S13
@stage("S13", "Preflight (GPU)", "GPU", ["S10"], ["models", "text", "budget", "rag", "distill", "tuning.finetune_grid"],
       outputs=["preflight.json"], assertions=["budgets OK (full refuses when exceeded)"],
       card="Workload, storage and GPU-time projection from a timed micro-benchmark.")
def s13(ctx):
    from .models.embeddings import benchmark, chunk_words, device_and_dtype, load_encoder, select_chunks, unload
    cfg = ctx.cfg
    ev = C.events(ctx)
    si = C.read(ctx, "S05", "sections_index.parquet")
    est_chunks = int(np.minimum(np.ceil(si.n_tokens_est / 1.3 / max(1, cfg["text"]["chunk_words"] - cfg["text"]["chunk_overlap_words"])),
                                cfg["text"]["max_chunks_per_section"]).sum()) if len(si) else 0
    encs = list(cfg["models"]["encoders"])
    dim, nb = 768, 2
    storage_gb = len(ev) * dim * 2 * nb * len(encs) / 1e9 * 1.1
    from huggingface_hub import HfApi
    api = HfApi()
    rost = C.roster(cfg)
    sizes = {}
    for k, v in {**rost["encoders"], **rost["reranker"], **rost["llm"], **rost["student"]}.items():
        mid = v.get("model_id") or v.get("model_id_template", "").format(year=2010)
        try:
            info = api.model_info(mid, files_metadata=True)
            sizes[k] = round(sum((s.size or 0) for s in info.siblings if s.rfilename.endswith((".safetensors", ".bin"))) / 1e9, 2)
        except Exception as e:
            sizes[k] = f"unknown ({type(e).__name__})"
    dev, _ = device_and_dtype()
    rate = None
    try:
        b = rost["encoders"]["bge_base"]
        tok, mdl, d = load_encoder(b["model_id"], b["revision"], "base")
        rate = benchmark(tok, mdl, d, 256 if dev == "cuda" else 32)
        unload(mdl)
    except Exception as e:
        ctx.warn(f"micro-benchmark failed: {e}")
    n_encode = est_chunks * (len(encs))
    gpu_h_b = (n_encode / rate / 3600) if rate else None
    from .stages_data import teacher_extraction_will_run
    teacher = teacher_extraction_will_run(cfg, ctx.storage.inputs)
    llm_prompts = (int(cfg["budget"]["llm_teacher_filings"]) if teacher else 0) + int(cfg["budget"]["llm_gold_filings"])
    gpu_h_c = llm_prompts * 20 / 3600 if dev == "cuda" else None     # ~20 s per filing prompt (estimate, logged)
    rep = {"device": dev, "firms": int(ev.cik.nunique()), "events": int(len(ev)), "sections": int(len(si)),
           "est_chunks": est_chunks, "est_tokens": int(si.n_tokens_est.sum()) if len(si) else 0,
           "embedding_storage_gb": round(storage_gb, 2), "model_sizes_gb": sizes, "llm_prompts": llm_prompts,
           "disk_free_gb": {"data_root": C.disk_free_gb(ctx.storage.data_root), "hot_root": C.disk_free_gb(ctx.storage.hot_root)},
           "bench_chunks_per_s": rate, "projected_gpu_hours_encoders": gpu_h_b, "projected_gpu_hours_llm": gpu_h_c,
           "teacher_extraction_runs": teacher,
           "budget": cfg["budget"]}
    write_json(ctx.out / "preflight.json", rep)
    if cfg["run_profile"] == "full":
        if storage_gb > float(cfg["budget"]["max_storage_gb"]):
            raise RuntimeError(f"preflight: embedding storage {storage_gb:.1f} GB > budget")
        if gpu_h_b is not None and gpu_h_b > float(cfg["budget"]["max_gpu_hours_b"]):
            raise RuntimeError(f"preflight: projected encoder GPU hours {gpu_h_b:.1f} > budget.max_gpu_hours_b")
        if gpu_h_c is not None and gpu_h_c > float(cfg["budget"]["max_gpu_hours_c"]):
            raise RuntimeError(f"preflight: projected LLM GPU hours {gpu_h_c:.1f} > budget.max_gpu_hours_c")
    return rep


# =====================================================================================================  S14
@stage("S14", "Core embeddings + vintage ChronoBERT", "GPU", ["S10"], ["models", "text", "seed"],
       outputs=["embeddings/<encoder>/vectors.npy", "index.parquet", "extra.parquet"],
       assertions=["pooling + change vectors", "revisions pinned", "one model in memory at a time"],
       card="Frozen encoder vectors per event (mean and max pooled), FinBERT native tone, change cosine to the prior same-type event.")
def s14(ctx):
    from .models.embeddings import change_cosine, chrono_vintage, encode, event_chunks, load_encoder, pool_events, unload
    cfg = ctx.cfg
    rost = C.roster(cfg)
    ev = C.events(ctx)
    ev = ev[pd.to_datetime(ev.decision_date) >= pd.Timestamp(cfg["study"]["start"])]
    store = _chunk_store(ctx, ev)                    # one pass over the firm files -> chunks partitioned by event year
    out_root = ctx.out / "embeddings"
    stats = {"chunk_store": str(store)}
    years = sorted(int(p.name.split("=")[1]) for p in store.glob("year=*"))
    for enc in cfg["models"]["encoders"]:
        spec = rost["encoders"][enc]
        d = out_root / enc
        if (d / "vectors.npy").exists():
            stats[enc] = "exists"
            continue
        d.mkdir(parents=True, exist_ok=True)
        ids_all, V_all, X_all = [], [], []
        t0 = time.time()
        loaded, mdl = None, None
        for y in years:                               # years in order: ChronoBERT loads each vintage once
            vint = chrono_vintage(y, spec["vintages"]) if enc == "chrono_bert" else None
            mid = spec["model_id_template"].format(year=vint) if vint else spec["model_id"]
            rev = (spec.get("revisions") or {}).get(str(vint)) if vint else spec.get("revision")
            if not rev:
                raise RuntimeError(f"{mid}: no pinned revision in configs/models.yaml (hard rule 12)")
            if loaded != mid:
                if mdl is not None:
                    unload(mdl)
                tok, mdl, dev = load_encoder(mid, rev, "classifier" if spec.get("native_outputs") else "base")
                loaded = mid
            ch = pd.read_parquet(store / f"year={y}")
            if ch.empty:
                continue
            sizes = ch.groupby("event_id", sort=False).size()
            slice_of = (sizes.cumsum() - sizes) // 40000          # whole events per slice (bounded memory)
            for _, part in ch.groupby(ch.event_id.map(slice_of), sort=True):
                V, P = encode(tok, mdl, dev, part.text.tolist(), int(spec.get("max_len", 512)), int(cfg["models"]["embed_batch"]),
                              spec.get("pooling", "mean"), classifier=bool(spec.get("native_outputs")), log=ctx.log)
                ids, M, extra = pool_events(part, V, P)
                extra["vintage"], extra["model_id"] = vint, mid
                ids_all.append(ids)
                V_all.append(M)
                X_all.append(extra)
            ctx.log.info(f"S14 {enc} year {y}: {len(ch):,} chunks ({time.time() - t0:.0f}s)")
        if mdl is not None:
            unload(mdl)
        if not V_all:
            continue
        ids = pd.Index(np.concatenate([np.asarray(i) for i in ids_all]), name="event_id")
        M = np.vstack(V_all)
        ex = pd.concat(X_all, ignore_index=True)
        # an event split across slices is pooled per slice: keep the first occurrence only (rare; logged)
        dup = ids.duplicated()
        if dup.any():
            ctx.warn(f"{enc}: {int(dup.sum())} events split across slices; first slice kept")
            ids, M, ex = ids[~dup], M[~dup], ex[~ex.event_id.duplicated()]
        dim = M.shape[1] // 2
        cc = change_cosine(ev, ids, M, dim)
        ex["chg_emb_cosine"] = ex.event_id.map(cc)
        np.save(d / "vectors.npy", M.astype(np.float16))
        pd.DataFrame({"event_id": ids}).to_parquet(d / "index.parquet", index=False)
        ex.to_parquet(d / "extra.parquet", index=False)
        stats[enc] = {"events": int(len(ids)), "dim": int(M.shape[1]), "seconds": round(time.time() - t0, 1)}
        ctx.log.info(f"S14 {enc}: {stats[enc]}")
    return stats


def _chunk_store(ctx, ev: pd.DataFrame) -> Path:
    """Masked, section-aware chunks for every event, written once per firm into year partitions on fast local disk."""
    from .models.embeddings import event_chunks
    store = ctx.storage.hot_root / f"s14_chunks_{ctx.stage_hash}"
    done = store / "_DONE"
    if done.exists():
        return store
    si_dir = ctx.dep("S05") / "sections"
    for cik, g in ev.groupby("cik"):
        p = si_dir / f"{int(cik):010d}.parquet"
        if not p.exists():
            continue
        ch = event_chunks(g, pd.read_parquet(p, columns=["accession", "section_id", "text_masked"]), ctx.cfg)
        if ch.empty:
            continue
        ch["year"] = ch.event_id.map(g.set_index("event_id")["year"]).astype(int)
        for y, gy in ch.groupby("year"):
            q = store / f"year={int(y)}"
            q.mkdir(parents=True, exist_ok=True)
            gy.drop(columns="year").to_parquet(q / f"{int(cik):010d}.parquet", index=False)
    store.mkdir(parents=True, exist_ok=True)
    done.write_text("ok")
    return store


# =====================================================================================================  S15
@stage("S15", "L5 / L7 heads + tuning; G3; fine-tune top 2", "GPU", ["S11", "S14"],
       ["tuning", "gates", "models", "bootstrap", "seed"],
       outputs=["results_long.parquet", "predictions.parquet", "g3.json", "h4.csv", "ft_curves.json"],
       assertions=["finalist caps", "default and tuned results", "fine-tune only if G3 passes (or labelled owner override)"],
       card="Embedding-only (L5) and structured + embedding (L7) heads per encoder; G3 decision; fine-tuning curves.")
def s15(ctx):
    cfg = ctx.cfg
    lad = build_ladder(ctx, with_emb=True)
    prim = cfg["targets"]["primary"]
    lab = lad.labelled(prim["target_def"], prim["horizon"])
    encs = [e for e in cfg["models"]["encoders"] if e in lad.embs]
    ids = lab.index
    common = ids[lad.cohort(["L1", "L2", "L3", "L4", "L6", "L5", "L7"], encs, ids)]
    res, preds = Results(), []
    nb, seed = int(cfg["bootstrap"]["n"]), int(cfg["seed"])
    s11 = json.loads((ctx.dep("S11") / "best_params.json").read_text())
    bp6 = s11["params"]
    summ = []
    db = ctx.storage.hot_root / f"optuna_S15_{ctx.stage_hash}.sqlite3"
    # re-score L6 on this (embedding-inclusive) common cohort with its tuned parameters: identical rows for the lift
    l6runs = {}
    for head in cfg["tuning"]["heads"]:
        r, s, rid = _run_and_record(lad, res, preds, "L6", head, bp6.get(f"L6|{head}"), lab, common, "common_emb", prim["target_def"],
                                    prim["horizon"], "tuned", n_boot=nb, seed=seed)
        l6runs[head] = (r, s, rid)
    for enc in encs:
        for rung in ("L5", "L7"):
            for head in cfg["tuning"]["heads"]:
                r0, s0, _ = _run_and_record(lad, res, preds, rung, head, None, lab, common, "common_emb", prim["target_def"],
                                            prim["horizon"], "default", enc=enc, curves=(rung == "L7"), n_boot=nb, seed=seed)
                bp, _ = lad.tune(rung, head, lab, common, int(cfg["tuning"]["trials"].get(head, 10)),
                                 int(cfg["tuning"]["timeout_s_per_study"]), db, f"{rung}|{head}|{enc}", enc=enc)
                r1, s1, rid = _run_and_record(lad, res, preds, rung, head, bp, lab, common, "common_emb", prim["target_def"],
                                              prim["horizon"], "tuned", enc=enc, n_boot=nb, seed=seed)
                summ.append({"rung": rung, "encoder": enc, "head": head, "params": json.dumps(bp), "default_mean_f1": s0["mean_fold"],
                             "tuned_mean_f1": s1["mean_fold"], "tuned_std": s1["std_fold"], "run_id": rid,
                             "pooled_f1": (s1["pooled"] or {}).get("macro_f1")})
    sm = pd.DataFrame(summ)
    sm.to_csv(ctx.out / "emb_ladder_summary.csv", index=False)
    if db.exists():
        import shutil
        shutil.copy2(db, ctx.out / "optuna.sqlite3")
    pr = pd.concat(preds, ignore_index=True) if preds else pd.DataFrame()
    # G3: best L7 vs L6 (same head, identical rows), paired month-block bootstrap
    g3 = {"passed": False, "reason": "no L7 runs"}
    if len(sm):
        b7 = sm[sm.rung == "L7"].sort_values("tuned_mean_f1", ascending=False).iloc[0]
        p7 = pr[pr.run_id == b7.run_id].set_index("event_id")
        p6 = pr[pr.run_id == l6runs[b7["head"]][2]].set_index("event_id")
        j = p7.index.intersection(p6.index)
        d = paired_macro_f1_diff(p7.loc[j, "y"].values, p7.loc[j, "yhat"].values, p6.loc[j, "yhat"].values,
                                 p7.loc[j, "date"].values, nb, seed)
        passed = d["value"] >= float(cfg["gates"]["text_lift"]) and d["ci_low"] > 0
        g3 = {"passed": bool(passed), "lift": d, "best_L7": b7.to_dict(), "n_events": int(len(j)),
              "rule": f"lift >= {cfg['gates']['text_lift']} macro-F1 and CI excludes 0"}
    write_json(ctx.out / "g3.json", g3)
    # H4: ChronoBERT vintage vs each static encoder, identical rows and heads
    h4 = []
    if "chrono_bert" in encs and len(sm):
        for rung in ("L5", "L7"):
            for head in cfg["tuning"]["heads"]:
                a = sm[(sm.rung == rung) & (sm.encoder == "chrono_bert") & (sm["head"] == head)]
                if a.empty:
                    continue
                pa = pr[pr.run_id == a.run_id.iloc[0]].set_index("event_id")
                for enc in encs:
                    if enc == "chrono_bert":
                        continue
                    b = sm[(sm.rung == rung) & (sm.encoder == enc) & (sm["head"] == head)]
                    if b.empty:
                        continue
                    pb = pr[pr.run_id == b.run_id.iloc[0]].set_index("event_id")
                    j = pa.index.intersection(pb.index)
                    d = paired_macro_f1_diff(pa.loc[j, "y"].values, pa.loc[j, "yhat"].values, pb.loc[j, "yhat"].values,
                                             pa.loc[j, "date"].values, nb, seed)
                    h4.append({"rung": rung, "head": head, "vintage": "chrono_bert", "static": enc, "diff_vintage_minus_static": d["value"],
                               "ci_low": d["ci_low"], "ci_high": d["ci_high"], "p_boot": d["p_boot"], "n": int(len(j))})
    pd.DataFrame(h4).to_csv(ctx.out / "h4.csv", index=False)
    # fine-tuning: top-2 encoders by L5 walk-forward F1, only if G3 passes (or owner override, labelled exploratory)
    ft_info = {"ran": False}
    override = bool(cfg["gates"].get("owner_override_g3_finetune"))
    if (g3.get("passed") or override) and len(sm):
        top2 = (sm[sm.rung == "L5"].groupby("encoder").tuned_mean_f1.max().sort_values(ascending=False).index[:2].tolist())
        ft_info = finetune_encoders(ctx, lad, lab, common, top2, res, preds, label="G3 passed" if g3.get("passed") else
                                    "EXPLORATORY: owner override, G3 not passed")
    write_json(ctx.out / "finetune_status.json", ft_info)
    rl = res.frame()
    rl["config_counter"] = lad.config_counter
    ctx.write(rl, "results_long.parquet")
    ctx.write(pd.concat(preds, ignore_index=True) if preds else pd.DataFrame(), "predictions.parquet")
    (ctx.out / "curves.json").write_text(json.dumps(lad.curves[:200], default=float), encoding="utf-8")
    return {"common_cohort": int(len(common)), "encoders": encs, "g3": {k: v for k, v in g3.items() if k != "best_L7"},
            "finetune": ft_info.get("ran")}


def finetune_encoders(ctx, lad: Ladder, lab, ids, encs, res: Results, preds: list, label: str) -> dict:
    from .models.embeddings import chrono_vintage
    from .models.finetune import grid, train_eval
    cfg = ctx.cfg
    rost = C.roster(cfg)
    folds = lad.folds("walk_forward")
    texts = lad.texts
    cap = int(cfg["tuning"]["finetune_max_train_events"])
    ml = int(cfg["models"]["max_seq_len"])
    seed = int(cfg["seed"])
    curves, info = [], {"ran": True, "label": label, "encoders": encs, "grid": {}}
    blocks = pd.read_csv(ctx.dep("S10") / "blocks.csv", parse_dates=["start"])

    def mid_for(enc, block_start):
        sp = rost["encoders"][enc]
        if enc == "chrono_bert":
            v = chrono_vintage(pd.Timestamp(block_start).year, sp["vintages"])
            return sp["model_id_template"].format(year=v), (sp.get("revisions") or {}).get(str(v))
        return sp["model_id"], sp.get("revision")

    def one(enc, scheme, fold, hp, keep=True):
        tr, te = lad.fold_ids(scheme, fold, ids)
        trs = lab.loc[tr].sort_values("date").iloc[-cap:]
        cut = int(len(trs) * 0.9)
        itr, iva = trs.index[:cut], trs.index[cut:]
        bname = {"walk_forward": f"wf{fold}", "dev_holdout": "dev", "locked_test": "locked"}[scheme]
        bs = blocks.loc[blocks.block == bname, "start"]
        mid, rev = mid_for(enc, bs.iloc[0] if len(bs) else lab.date.max())
        out = train_eval(mid, rev, texts.reindex(itr).fillna("").tolist(), lab.loc[itr, "y"].astype(int).to_numpy(),
                         lab.loc[itr, "w"].to_numpy(), texts.reindex(iva).fillna("").tolist(),
                         lab.loc[iva, "y"].astype(int).to_numpy(), [texts.reindex(te).fillna("").tolist()], hp, ml, seed, ctx.log)
        prior = np.bincount(lab.loc[itr, "y"].astype(int), minlength=3) / max(1, len(itr))
        from .models.heads import prior_correct
        P = prior_correct(out["preds"][0], prior)       # class-weighted CE -> true-frequency probabilities
        yhat = decide(P, prior)
        m = classification_metrics(lab.loc[te, "y"].to_numpy(), P, prior, yhat)
        return out, m, te, P, yhat

    for enc in encs:
        if not folds:
            break
        g = grid(cfg)
        best = None
        for hp in g:
            out, m, te, P, yhat = one(enc, "walk_forward", folds[-1], hp)
            curves.append({"label": f"{enc} lr{hp['lr']} b{hp['batch']} wd{hp['weight_decay']}", "curve": out["curve"],
                           "val_macro_f1": m.get("macro_f1"), "best_step": out["best_step"]})
            if best is None or (m.get("macro_f1") or 0) > (best[1].get("macro_f1") or 0):
                best = (hp, m)
        info["grid"][enc] = {"trials": len(g), "best_hp": best[0], "best_val_macro_f1": best[1].get("macro_f1")}
        rid = f"common_emb|L8_ft|ft|{enc}|{cfg['targets']['primary']['target_def']}|h{cfg['targets']['primary']['horizon']}|tuned"
        allp = []
        for f in folds:
            out, m, te, P, yhat = one(enc, "walk_forward", f, best[0])
            allp.append(pd.DataFrame({"event_id": te, "fold": f, "y": lab.loc[te, "y"].values, "date": lab.loc[te, "date"].values,
                                      "p_down": P[:, 0], "p_side": P[:, 1], "p_up": P[:, 2], "yhat": yhat}))
            res.add(rid, "L8_ft", "ft", enc, _leaderboard("L8_ft", enc, lad.prov, f"wf{f}"), "common_emb",
                    cfg["targets"]["primary"]["target_def"], cfg["targets"]["primary"]["horizon"], f"wf{f}", "all", m, None,
                    m.get("n"), None, None, None, "tuned", {"label": label})
        p = pd.concat(allp, ignore_index=True)
        prior = np.bincount(p.y.astype(int), minlength=3) / len(p)
        pm = classification_metrics(p.y.values, p[["p_down", "p_side", "p_up"]].values, prior, p.yhat.values)
        ci = macro_f1_ci(p.y.values, p.yhat.values, p.date.values, int(cfg["bootstrap"]["n"]), seed)
        res.add(rid, "L8_ft", "ft", enc, "retrospective" if enc != "chrono_bert" else "strict", "common_emb",
                cfg["targets"]["primary"]["target_def"], cfg["targets"]["primary"]["horizon"], "walk_forward_pooled", "all", pm, ci,
                len(p), None, None, None, "tuned", {"label": label})
        preds.append(p.assign(run_id=rid, model_id="ft", rung="L8_ft", encoder=enc, cohort="common_emb",
                              target_def=cfg["targets"]["primary"]["target_def"], horizon=cfg["targets"]["primary"]["horizon"], tuned="tuned"))
        # development-holdout predictions: trained on all pre-holdout data with the selected grid point (used by S18)
        out, m, te, P, yhat = one(enc, "dev_holdout", 0, best[0])
        if len(te):
            dp = pd.DataFrame({"event_id": te, "fold": 0, "y": lab.loc[te, "y"].values, "date": lab.loc[te, "date"].values,
                               "p_down": P[:, 0], "p_side": P[:, 1], "p_up": P[:, 2], "yhat": yhat})
            preds.append(dp.assign(run_id=f"dev|L8_ft|{enc}", model_id="ft", rung="L8_ft", encoder=enc, cohort="common_emb",
                                   target_def=cfg["targets"]["primary"]["target_def"],
                                   horizon=cfg["targets"]["primary"]["horizon"], tuned="tuned"))
            info["grid"][enc]["dev_macro_f1"] = m.get("macro_f1")
            info["grid"][enc]["dev_curve_best_step"] = out["best_step"]
    (ctx.out / "ft_curves.json").write_text(json.dumps(curves, default=float), encoding="utf-8")
    return info


# =====================================================================================================  S16
@stage("S16", "Core RAG: retrieval eval, extraction (gold-val), freeze", "GPU", ["S12", "S14"], ["rag", "budget", "seed"],
       outputs=["rag_tournament.csv", "g3b.json", "rag_frozen.json"],
       assertions=["historical path time-filtered", "current path separate"],
       card="BM25 vs hybrid vs hybrid+rerank on gold-val qrels; Qwen extraction scored on gold-val; G3b decision.")
def s16(ctx):
    from .rag.core import Retriever, historical_candidates, ir_metrics, retrieval_guard
    from .rag.extract import MockGenerator, ResponseCache, build_prompt, cache_key, field_schema, guard
    cfg = ctx.cfg
    assert retrieval_guard(), "historical retrieval filter failed its self-test"
    qb = C.question_bank(cfg)
    gold_dir = ctx.dep("S12")
    sample = pd.read_parquet(gold_dir / "gold_sample.parquet")
    labels = pd.read_parquet(gold_dir / "gold_labels.parquet") if (gold_dir / "gold_labels.parquet").exists() else pd.DataFrame()
    have_val = len(labels) and (labels.gold_split == "gold_val").any()
    gen, gen_name = _generator(ctx, qb)
    cache = ResponseCache(ctx.out / "llm_cache.jsonl")
    ev = C.events(ctx).set_index("event_id")
    chunks = _gold_chunks(ctx, sample)
    rows, status = [], {}
    setups = ["bm25", "hybrid", "hybrid_rerank"]
    dense, rerank = _dense_and_reranker(ctx) if gen_name != "mock" else (None, None)
    prefill = {}
    ret = Retriever("hybrid_rerank" if rerank else "bm25", dense, rerank, cfg["rag"]["bm25_k1"], cfg["rag"]["bm25_b"],
                    cfg["rag"]["rrf_k"], cfg["rag"]["rerank_top_n"])
    q = " ".join(f["question"] for f in qb["fields"].values())
    todo, pmap = [], {}
    for split in ("gold_dev", "gold_val"):                  # LLM pre-fill: gold-dev / gold-val only (never gold-test)
        for eid in sample[sample.gold_split == split].event_id:
            c = chunks[chunks.event_id == eid]
            if c.empty:
                continue
            top = ret.rank(q, c, int(cfg["rag"]["top_k"]))
            passages = [{"chunk_id": r.chunk_id, "text": r.text} for r in top.itertuples(index=False)]
            k = cache_key(getattr(gen, "revision", gen_name), build_prompt(qb, passages), passages)
            pmap[eid] = (k, passages)
            if cache.get(k) is None:
                todo.append(eid)
    t_gen = time.time()
    for b0 in range(0, len(todo), 64):
        ids_b = todo[b0:b0 + 64]
        for e, raw in zip(ids_b, gen.batch(qb, [pmap[e][1] for e in ids_b])):
            cache.put(pmap[e][0], raw)
    for eid, (k, passages) in pmap.items():
        clean, _flags = guard(cache.get(k) or {}, qb, passages, field_schema(qb))
        prefill[eid] = clean
    gen_seconds = time.time() - t_gen
    if cfg["rag"].get("prefill_dev_val", True):
        from .gold.tools import export_templates
        export_templates(sample, qb, None, ctx.out / "prefilled_templates", prefill=prefill)
    if have_val:
        # retrieval tournament on gold-val qrels (evidence chunk ids) + extraction F1 on gold-val
        val = labels[labels.gold_split == "gold_val"]
        for setup in setups:
            if setup != "bm25" and dense is None:
                rows.append({"setup": setup, "status": "not run (dense model unavailable on CPU)"})
                continue
            ret = Retriever(setup, dense, rerank)
            ms = []
            for (eid, field), g in val[val.evidence_chunk_id != ""].groupby(["event_id", "field"]):
                c = chunks[chunks.event_id == eid]
                top = ret.rank(qb["fields"][field]["question"], c, 10)
                ms.append(ir_metrics(top.chunk_id.tolist(), set(g.evidence_chunk_id), 10))
            d = pd.DataFrame(ms)
            rows.append({"setup": setup, "n_queries": len(d), **(d.mean(numeric_only=True).to_dict() if len(d) else {})})
        ext = []
        for (eid, field), g in val.groupby(["event_id", "field"]):
            pred = (prefill.get(eid) or {}).get(field) or {}
            ext.append({"field": field, "y": bool(g.present_b.iloc[0]), "p": bool(pred.get("present") or False)})
        e = pd.DataFrame(ext)
        from sklearn.metrics import f1_score
        f1 = float(f1_score(e.y, e.p)) if len(e) and e.y.any() else None
        status = {"gold_val_extraction_f1": f1, "passed": bool(f1 is not None and f1 >= float(cfg["gates"]["extraction_min_f1"])),
                  "generator": gen_name}
    else:
        rows.append({"setup": "all", "status": "not evaluable: no gold-val labels imported (annotate and re-run S12/S16)"})
        status = {"passed": False, "reason": "no gold-val labels", "generator": gen_name}
    pd.DataFrame(rows).to_csv(ctx.out / "rag_tournament.csv", index=False)
    write_json(ctx.out / "g3b.json", status)
    write_json(ctx.out / "rag_frozen.json", {"setup": "hybrid_rerank" if rerank else "bm25", "generator": gen_name,
                                             "top_k": cfg["rag"]["top_k"], "frozen": bool(status.get("passed"))})
    return {"g3b": status, "prefilled_events": len(prefill), "generation_seconds": round(gen_seconds, 1),
            "seconds_per_filing": round(gen_seconds / max(1, len(todo)), 2)}


def _generator(ctx, qb):
    from .rag.extract import MockGenerator, QwenGenerator, field_schema
    import torch
    if ctx.services.get("mock_llm") or not torch.cuda.is_available():
        return MockGenerator(), "mock"
    q = C.roster(ctx.cfg)["llm"]["qwen"]
    return QwenGenerator(q["model_id"], q["revision"], field_schema(qb), int(ctx.cfg["rag"]["max_new_tokens"]), ctx.log), "qwen2.5-7b"


def _dense_and_reranker(ctx):
    import torch
    from .models.embeddings import encode, load_encoder
    rost = C.roster(ctx.cfg)
    b = rost["encoders"]["bge_base"]
    tok, mdl, dev = load_encoder(b["model_id"], b["revision"], "base")
    inst = "Represent this sentence for searching relevant passages: "

    def dense(q, texts):
        V, _ = encode(tok, mdl, dev, [inst + q] + texts, 512, 64, "cls")
        V = V.astype(np.float32)
        return V[1:] @ V[0]

    rr = rost["reranker"]["bge_reranker"]
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    rt = AutoTokenizer.from_pretrained(rr["model_id"], revision=rr["revision"])
    rm = AutoModelForSequenceClassification.from_pretrained(rr["model_id"], revision=rr["revision"]).to(dev).eval()

    def rerank(q, texts):
        with torch.no_grad():
            e = rt([q] * len(texts), texts, truncation=True, max_length=512, padding=True, return_tensors="pt").to(dev)
            return rm(**e).logits.view(-1).float().cpu().numpy()

    return dense, rerank


def _gold_chunks(ctx, sample: pd.DataFrame) -> pd.DataFrame:
    from .models.embeddings import event_chunks
    ev = C.events(ctx)
    g = ev[ev.event_id.isin(sample.event_id)]
    secs = []
    for cik in g.cik.unique():
        p = ctx.dep("S05") / "sections" / f"{int(cik):010d}.parquet"
        if p.exists():
            secs.append(pd.read_parquet(p, columns=["accession", "section_id", "text_masked"]))
    if not secs:
        return pd.DataFrame(columns=["event_id", "section_id", "chunk_no", "text", "chunk_id"])
    ch = event_chunks(g, pd.concat(secs, ignore_index=True), ctx.cfg)
    ch["chunk_id"] = ch.event_id + ":" + ch.section_id + ":" + ch.chunk_no.astype(str)
    return ch


# =====================================================================================================  S17
@stage("S17", "Teacher sample + student distillation -> L8", "GPU", ["S16", "S15"], ["distill", "rag", "budget", "seed"],
       outputs=["extractions.parquet", "ext_features.parquet", "student_report.json"],
       assertions=["training-period only"],
       card="Teacher (frozen RAG + Qwen) on a training-period sample; ModernBERT student applied to all events; L8 = L7 + extraction.")
def s17(ctx):
    cfg = ctx.cfg
    g3b = json.loads((ctx.dep("S16") / "g3b.json").read_text())
    if not (g3b.get("passed") or cfg["rag"].get("run_extraction_without_gold")):
        write_json(ctx.out / "student_report.json", {"ran": False, "reason": "G3b not passed (no gold-val evidence); set "
                                                     "rag.run_extraction_without_gold: true to run as exploratory"})
        ctx.write(pd.DataFrame(columns=["event_id"]), "ext_features.parquet")
        return {"ran": False}
    from .rag.distill import run_distillation
    info = run_distillation(ctx, exploratory=not g3b.get("passed"))
    return info


# =====================================================================================================  S18
@stage("S18", "Development holdout + diagnostics (MCS / PBO) + G4 freeze", "CPU", ["S15"],
       ["tuning", "bootstrap", "audit", "extensions", "seed"],
       outputs=["dev_results.parquet", "frozen_architecture.json", "mcs_pbo.json", "review_triggers.json"],
       assertions=["no test access", "prereg hash present"],
       card="Finalists retrained on all pre-holdout data and scored on 2020-2022; MCS/PBO diagnostics; G4 manifest freeze.")
def s18(ctx):
    from .evaluation.mcs_pbo import model_confidence_set, pbo_cscv
    cfg = ctx.cfg
    ph = ctx.storage.manifests / "prereg_hash.txt"
    if not ph.exists():
        raise RuntimeError("pre-registration not frozen: run secprice prereg-freeze (notebook cell 'Freeze pre-registration') first")
    lad = build_ladder(ctx, with_emb=True, with_ext=True)
    prim = cfg["targets"]["primary"]
    lab = lad.labelled(prim["target_def"], prim["horizon"])
    encs = list(lad.embs)
    common = lab.index[lad.cohort(["L1", "L2", "L3", "L4", "L6", "L5", "L7"], encs, lab.index)] if encs else \
        lab.index[lad.cohort(["L1", "L2", "L3", "L4", "L6"], [None], lab.index)]
    s11 = json.loads((ctx.dep("S11") / "best_params.json").read_text())
    s15sum = pd.read_csv(ctx.dep("S15") / "emb_ladder_summary.csv") if (ctx.dep("S15") / "emb_ladder_summary.csv").exists() else pd.DataFrame()
    fin = []
    b6 = s11.get("best_L6")
    if b6:
        fin.append({"name": "L6", "rung": "L6", "head": b6["head"], "enc": None, "params": s11["params"].get(f"L6|{b6['head']}"),
                    "wf_score": b6["tuned_mean_f1"]})
        fin.append({"name": "L6_nochange", "rung": "L6_nochange", "head": b6["head"], "enc": None,
                    "params": s11["params"].get(f"L6_nochange|{b6['head']}"), "wf_score": None})
        fin.append({"name": "L1", "rung": "L1", "head": b6["head"], "enc": None, "params": s11["params"].get(f"L1|{b6['head']}"),
                    "wf_score": None})
    if len(s15sum):
        b7 = s15sum[s15sum.rung == "L7"].sort_values("tuned_mean_f1", ascending=False).iloc[0]
        fin.append({"name": "L7", "rung": "L7", "head": b7["head"], "enc": b7["encoder"], "params": json.loads(b7.params),
                    "wf_score": float(b7.tuned_mean_f1)})
        if any(c.startswith("ext_") for c in lad.F.columns):
            fin.append({"name": "L8", "rung": "L8", "head": b7["head"], "enc": b7["encoder"], "params": json.loads(b7.params),
                        "wf_score": None})
    res, preds = Results(), []
    nb, seed = int(cfg["bootstrap"]["n"]), int(cfg["seed"])
    for f in fin:
        r = lad.run(f["rung"], f["head"], f["params"], lab, common, "dev_holdout", "cls", f["enc"], keep=True)
        if r["preds"].empty:
            continue
        p = r["preds"]
        prior = np.bincount(p.y.astype(int), minlength=3) / len(p)
        m = classification_metrics(p.y.values, p[["p_down", "p_side", "p_up"]].values, prior, p.yhat.values)
        ci = macro_f1_ci(p.y.values, p.yhat.values, p.date.values, nb, seed)
        lb = _leaderboard(f["rung"], f["enc"], lad.prov, "dev")
        res.add(f"dev|{f['name']}", f["rung"], f["head"], f["enc"] or "n/a", lb, "common_emb", prim["target_def"], prim["horizon"],
                "dev", "all", m, ci, *_counts(lad.ev, p.event_id, lad.clean_col), "tuned")
        preds.append(p.assign(run_id=f"dev|{f['name']}", name=f["name"]))
    pr = pd.concat(preds, ignore_index=True) if preds else pd.DataFrame()
    # fine-tuned finalists: development predictions were produced in S15 (trained on all pre-holdout data)
    s15p = ctx.dep("S15") / "predictions.parquet"
    if s15p.exists():
        fp = pd.read_parquet(s15p)
        fp = fp[fp.run_id.astype(str).str.startswith("dev|L8_ft")]
        for rid, g in fp.groupby("run_id"):
            enc = g.encoder.iloc[0]
            prior = np.bincount(g.y.astype(int), minlength=3) / len(g)
            m = classification_metrics(g.y.values, g[PCOLS_].values, prior, g.yhat.values)
            ci = macro_f1_ci(g.y.values, g.yhat.values, g.date.values, nb, seed)
            res.add(rid, "L8_ft", "ft", enc, _leaderboard("L8_ft", enc, lad.prov, "dev"), "common_emb", prim["target_def"],
                    prim["horizon"], "dev", "all", m, ci, *_counts(lad.ev, g.event_id, lad.clean_col), "tuned")
            pr = pd.concat([pr, g.assign(name=f"L8_ft|{enc}")[["event_id", "fold", "y", "date", *PCOLS_, "yhat", "run_id", "name"]]],
                           ignore_index=True)
            wfs = pd.read_parquet(s15p)
            wfs = wfs[(wfs.rung == "L8_ft") & (wfs.encoder == enc) & ~wfs.run_id.astype(str).str.startswith("dev|")]
            if len(wfs):
                from .evaluation.metrics import confusion, f1_from_cm
                sc = [f1_from_cm(confusion(x.y.values, x.yhat.values))["macro_f1"] for _, x in wfs.groupby("fold")]
                fin.append({"name": f"L8_ft|{enc}", "rung": "L8_ft", "head": "ft", "enc": enc, "params": None,
                            "wf_score": float(np.mean(sc)), "dev_from": "S15"})
    # stacking extension (H5), walk-forward OOF only (SPEC.md §6.3 #20)
    stack = None
    if cfg["extensions"].get("ext_stacking"):
        from .extensions.stacking import run_stacking
        stack = run_stacking(ctx, lad, lab, common, res, pr)
        if stack and stack.get("dev_preds") is not None:
            pr = pd.concat([pr, stack["dev_preds"]], ignore_index=True)
    # lifts on the development holdout (identical rows)
    lifts = []
    names = set(pr.name) if len(pr) else set()
    for a, b, hyp in (("L7", "L6", "H1"), ("L6", "L6_nochange", "H2"), ("L8", "L7", "H3"), ("STACK", "L7", "H5")):
        if a in names and b in names:
            pa, pb = pr[pr.name == a].set_index("event_id"), pr[pr.name == b].set_index("event_id")
            j = pa.index.intersection(pb.index)
            d = paired_macro_f1_diff(pa.loc[j, "y"].values, pa.loc[j, "yhat"].values, pb.loc[j, "yhat"].values, pa.loc[j, "date"].values, nb, seed)
            lifts.append({"hypothesis": hyp, "a": a, "b": b, "block": "dev", **d, "n": int(len(j)), "power": power_label(len(j))})
    pd.DataFrame(lifts).to_csv(ctx.out / "dev_lifts.csv", index=False)
    # MCS over finalists (monthly mean log-loss on dev) and PBO over walk-forward configurations
    mcs, pbo = {}, {}
    if len(pr):
        ll = []
        for nm, g in pr.groupby("name"):
            P = np.clip(g[["p_down", "p_side", "p_up"]].values, 1e-9, 1)
            g = g.assign(ll=-np.log(P[np.arange(len(g)), g.y.astype(int).values]), m=pd.to_datetime(g.date).dt.to_period("M"))
            ll.append(g.groupby("m").ll.mean().rename(nm))
        mcs = model_confidence_set(pd.concat(ll, axis=1), reps=500)
    wf = pd.concat([pd.read_parquet(ctx.dep(s) / "predictions.parquet") for s in ("S11", "S15")
                    if (ctx.dep(s) / "predictions.parquet").exists()], ignore_index=True)
    wf = wf[(wf.target_def == prim["target_def"]) & (wf.horizon == prim["horizon"]) & wf.yhat.notna()
            & ~wf.run_id.astype(str).str.startswith("dev|")] if len(wf) else wf
    if len(wf):
        from .evaluation.metrics import confusion, f1_from_cm
        wf["m"] = pd.to_datetime(wf.date).dt.to_period("M")
        perf = wf.groupby(["m", "run_id"]).apply(lambda g: f1_from_cm(confusion(g.y.values, g.yhat.values))["macro_f1"],
                                                 include_groups=False).unstack()
        pbo = pbo_cscv(perf)
        pbo["streams"] = "monthly walk-forward macro-F1 of every stored configuration (S11, S15)"
    mcs["streams"] = "monthly mean log-loss on the development holdout, finalists only"
    write_json(ctx.out / "mcs_pbo.json", {"mcs": mcs, "pbo": pbo})
    # review triggers (audit, never a stop)
    rl = res.frame()
    trig = []
    for _, r in rl[(rl.metric == "auc_macro_ovr")].iterrows():
        if r.value > float(cfg["audit"]["auc_trigger"]):
            trig.append(f"{r.run_id}: dev AUC {r.value:.3f} > {cfg['audit']['auc_trigger']} -> open the audit checklist")
    for l in lifts:
        if l["value"] > float(cfg["audit"]["f1_lift_trigger"]):
            trig.append(f"{l['hypothesis']}: dev lift {l['value']:.3f} > {cfg['audit']['f1_lift_trigger']} -> audit")
    write_json(ctx.out / "review_triggers.json", {"triggers": trig, "note": "review triggers open an audit; they never terminate"})
    # G4: select the primary model by the pre-registered rule (walk-forward mean macro-F1), freeze and hash
    cand = [f for f in fin if f.get("wf_score") is not None]
    if stack and stack.get("wf_score") is not None:
        cand.append({"name": "STACK", "rung": "STACK", "head": "stack", "enc": None, "params": stack.get("spec"),
                     "wf_score": stack["wf_score"]})
    chosen = max(cand, key=lambda f: f["wf_score"]) if cand else None
    frozen = {"primary_model": chosen, "finalists": fin, "selection_rule": "walk-forward mean macro-F1 (validation-selected)",
              "prereg_hash": ph.read_text().strip(), "config_hash": ctx.storage.global_config_hash,
              "common_cohort_n": int(len(common)), "frozen_at": pd.Timestamp.utcnow().isoformat()}
    h = hashlib.sha256(canonical_json(frozen).encode()).hexdigest()
    frozen["g4_hash"] = h
    write_json(ctx.out / "frozen_architecture.json", frozen)
    write_json(ctx.storage.manifests / "frozen_architecture.json", frozen)
    (ctx.storage.manifests / "g4_hash.txt").write_text(h)
    rl["config_counter"] = lad.config_counter
    ctx.write(rl, "dev_results.parquet")
    ctx.write(pr, "dev_predictions.parquet")
    return {"finalists": [f["name"] for f in fin], "chosen": chosen and chosen["name"], "g4_hash": h, "triggers": trig,
            "pbo": pbo.get("pbo")}


# =====================================================================================================  S19
@stage("S19", "unlock-test: retrain + locked test + clean test", "CPU-GPU", ["S18"], ["lockbox", "bootstrap", "seed"],
       outputs=["locked_results.parquet", "lockbox_log.json"], assertions=["single use", "logged"],
       card="The locked test (2023-01-01 to the usable cutoff), scored once after the G4 freeze.")
def s19(ctx):
    cfg = ctx.cfg
    frozen = json.loads((ctx.dep("S18") / "frozen_architecture.json").read_text()) if cfg["lockbox"].get("unlock_locked_test") else {}
    relabel = lockbox_guard(cfg, ctx.storage.manifests, frozen)
    log_p = ctx.storage.manifests / "lockbox_log.json"
    lad = build_ladder(ctx, with_emb=True, with_ext=True)
    prim = cfg["targets"]["primary"]
    lab = lad.labelled(prim["target_def"], prim["horizon"])
    encs = list(lad.embs)
    common = lab.index[lad.cohort(["L1", "L2", "L3", "L4", "L6", "L5", "L7"], encs, lab.index)] if encs else \
        lab.index[lad.cohort(["L1", "L2", "L3", "L4", "L6"], [None], lab.index)]
    rost = C.roster(cfg)
    from .splits.provenance import model_cutoffs
    cut = model_cutoffs(rost)
    res, preds = Results(), []
    nb, seed = int(cfg["bootstrap"]["n"]), int(cfg["seed"])
    block = "locked_reused_as_development" if relabel else "locked"
    notes = []
    for f in frozen["finalists"]:
        if f["rung"] == "L8_ft":
            p = _ft_locked(ctx, lad, lab, common, f)
            if p is None:
                notes.append(f"{f['name']}: not scored on the locked test (GPU required for fine-tuning)")
                continue
        else:
            r = lad.run(f["rung"], f["head"], f["params"], lab, common, "locked_test", "cls", f["enc"], keep=True)
            if r["preds"].empty:
                continue
            p = r["preds"]
        prior = np.bincount(p.y.astype(int), minlength=3) / len(p)
        m = classification_metrics(p.y.values, p[["p_down", "p_side", "p_up"]].values, prior, p.yhat.values)
        ci = macro_f1_ci(p.y.values, p.yhat.values, p.date.values, nb, seed)
        lb = _leaderboard(f["rung"], f["enc"], lad.prov, "locked")
        res.add(f"{block}|{f['name']}", f["rung"], f["head"], f["enc"] or "n/a", lb, "common_emb", prim["target_def"],
                prim["horizon"], block, "all", m, ci, *_counts(lad.ev, p.event_id, lad.clean_col), "tuned")
        c = cut.get(f["enc"]) if f["enc"] else "folds"
        if c not in (None,):
            cs = p if c in ("vintage", "folds") else p[pd.to_datetime(p.date) > c]
            if len(cs) > 30:
                pri = np.bincount(cs.y.astype(int), minlength=3) / len(cs)
                mc = classification_metrics(cs.y.values, cs[["p_down", "p_side", "p_up"]].values, pri, cs.yhat.values)
                res.add(f"{block}|{f['name']}|clean_test", f["rung"], f["head"], f["enc"] or "n/a", lb, "common_emb",
                        prim["target_def"], prim["horizon"], f"{block}_clean_after_cutoff", "all", mc, None, len(cs), None, None, None, "tuned")
        preds.append(p.assign(run_id=f"{block}|{f['name']}", name=f["name"]))
    write_json(log_p, {"scored_at": pd.Timestamp.utcnow().isoformat(), "g4_hash": frozen["g4_hash"], "reused": relabel,
                       "previous": json.loads(log_p.read_text()) if log_p.exists() else None})
    rl = res.frame()
    ctx.write(rl, "locked_results.parquet")
    ctx.write(pd.concat(preds, ignore_index=True) if preds else pd.DataFrame(), "locked_predictions.parquet")
    return {"block": block, "rows": int(len(rl)), "notes": notes}


def lockbox_guard(cfg: dict, manifests: Path, frozen: dict) -> bool:
    """Single-use lockbox (SPEC.md §6.1, §17.2): sealed unless unlocked; needs the G4 hash of the frozen architecture;
    a second scoring is refused unless acknowledge_reuse, which relabels the block as development. Returns relabel."""
    if not cfg["lockbox"].get("unlock_locked_test"):
        raise RuntimeError("lockbox sealed: set lockbox.unlock_locked_test: true in the owner settings to score the locked test once")
    g4 = Path(manifests) / "g4_hash.txt"
    if not g4.exists() or g4.read_text().strip() != frozen.get("g4_hash"):
        raise RuntimeError("G4 hash missing or does not match the frozen architecture")
    log_p = Path(manifests) / "lockbox_log.json"
    if log_p.exists():
        if not cfg["lockbox"].get("acknowledge_reuse"):
            raise RuntimeError(f"locked test already scored ({log_p.read_text()}); set lockbox.acknowledge_reuse: true "
                               "to rescore it as a DEVELOPMENT block")
        return True
    return False


def _ft_locked(ctx, lad, lab, ids, f) -> pd.DataFrame | None:
    """Retrain a fine-tuned finalist on all pre-test data with its selected grid point; None without a GPU."""
    import torch
    if not torch.cuda.is_available():
        return None
    from .models.embeddings import chrono_vintage
    from .models.finetune import train_eval
    cfg = ctx.cfg
    st = json.loads((ctx.dep("S15") / "finetune_status.json").read_text())
    hp = st["grid"][f["enc"]]["best_hp"]
    sp = C.roster(cfg)["encoders"][f["enc"]]
    tr, te = lad.fold_ids("locked_test", 0, ids)
    if f["enc"] == "chrono_bert":
        v = chrono_vintage(pd.Timestamp(cfg["splits"]["locked_test_start"]).year, sp["vintages"])
        mid, rev = sp["model_id_template"].format(year=v), (sp.get("revisions") or {}).get(str(v))
    else:
        mid, rev = sp["model_id"], sp.get("revision")
    trs = lab.loc[tr].sort_values("date").iloc[-int(cfg["tuning"]["finetune_max_train_events"]):]
    cut = int(len(trs) * 0.9)
    itr, iva = trs.index[:cut], trs.index[cut:]
    tx = lad.texts
    out = train_eval(mid, rev, tx.reindex(itr).fillna("").tolist(), lab.loc[itr, "y"].astype(int).to_numpy(),
                     lab.loc[itr, "w"].to_numpy(), tx.reindex(iva).fillna("").tolist(), lab.loc[iva, "y"].astype(int).to_numpy(),
                     [tx.reindex(te).fillna("").tolist()], hp, int(cfg["models"]["max_seq_len"]), int(cfg["seed"]), ctx.log)
    prior = np.bincount(lab.loc[itr, "y"].astype(int), minlength=3) / max(1, len(itr))
    from .models.heads import prior_correct
    P = prior_correct(out["preds"][0], prior)
    return pd.DataFrame({"event_id": te, "fold": 0, "y": lab.loc[te, "y"].values, "date": lab.loc[te, "date"].values,
                         "p_down": P[:, 0], "p_side": P[:, 1], "p_up": P[:, 2], "yhat": decide(P, prior)})


# =====================================================================================================  S22
@stage("S22", "Backtest", "CPU", ["S18"], ["backtest", "seed"], outputs=["backtest.csv", "backtest_daily.parquet"],
       assertions=["entry-rule assertions"],
       card="Economic-relevance check of the frozen T_trade model (signal P(Up)-P(Down), horizon h=5).")
def s22(ctx):
    from .backtest.engine import carhart_alpha, daily_book, exposure_stats, perf, positions, thresholds
    cfg, B = ctx.cfg, ctx.cfg["backtest"]
    frozen = json.loads((ctx.dep("S18") / "frozen_architecture.json").read_text())
    ch = frozen.get("primary_model")
    note = None
    if not ch or ch["rung"] in ("STACK", "L8_ft"):
        note = (f"primary model {ch and ch['name']} needs a horizon-{B['horizon']} refit that is not tabular; the backtest "
                "uses the best tabular finalist (L7, else L6) with the frozen hyperparameters")
        ch = max([f for f in frozen["finalists"] if f["name"] in ("L7", "L6") and f.get("wf_score") is not None],
                 key=lambda f: f["wf_score"], default=None)
        if ch is None:
            ch = next((f for f in frozen["finalists"] if f["name"] in ("L7", "L6")), None)
        ctx.warn(note)
    lad = build_ladder(ctx, with_emb=True, with_ext=True)
    h = int(B["horizon"])
    lab = lad.labelled("T_trade", h)
    encs = list(lad.embs)
    ids = lab.index[lad.requirement_mask(ch["rung"], ch["enc"], lab.index)]
    wf = lad.run(ch["rung"], ch["head"], ch["params"], lab, ids, "walk_forward", "cls", ch["enc"], keep=True)["preds"]
    dv = lad.run(ch["rung"], ch["head"], ch["params"], lab, ids, "dev_holdout", "cls", ch["enc"], keep=True)["preds"]
    blocks = {"dev": dv}
    lpath = ctx.storage.manifests / "lockbox_log.json"
    if lpath.exists():
        blocks["locked"] = lad.run(ch["rung"], ch["head"], ch["params"], lab, ids, "locked_test", "cls", ch["enc"], keep=True)["preds"]
    cal = C.calendar(ctx)
    panel, spy = C.panel_and_spy(ctx, cal)
    fac = C.factors(ctx)
    ev = lad.ev
    lo, hi = thresholds((wf.p_up - wf.p_down).values, float(B["short_pct"]), float(B["long_pct"]))
    rows, daily = [], []
    bmap = C.event_buckets(ctx).set_index("event_id").sp_bucket
    n_trials = int(sum(pd.read_parquet(ctx.dep(s) / "results_long.parquet").config_counter.max() for s in ("S11", "S15")))
    min_names = int(B["min_names_per_side"])
    for bname, p in blocks.items():
        if p.empty:
            continue
        sig_all = pd.DataFrame({"event_id": p.event_id, "ticker": ev.loc[p.event_id, "ticker"].values,
                                "entry_idx": ev.loc[p.event_id, "ttrade_entry_idx"].values.astype(int),
                                "score": (p.p_up - p.p_down).values})
        sig_all["sp_bucket"] = sig_all.event_id.map(bmap).fillna("NonSP500")
        # same frozen model and the same training-period thresholds; only the tradable names differ by bucket (D-056)
        for bucket in ("Combined", "SP500", "NonSP500"):
            sig = sig_all if bucket == "Combined" else sig_all[sig_all.sp_bucket == bucket]
            if sig.empty:
                continue
            for lag in (0, int(B["lag_sessions_sensitivity"])):
                pos = positions(sig, lo, hi, h, lag)
                # entry-rule assertion: the open of the entry session is at or after decision time
                if lag == 0 and len(pos):
                    ent_open = pd.Series([cal.open_time(int(i)) for i in pos.entry], index=pos.index)
                    dec = pd.to_datetime(ev.loc[pos.event_id, "decision_time"]).values
                    assert (ent_open.values >= dec).all(), "backtest entry before decision_time"
                for cost in B["costs_bps"]:
                    d = daily_book(pos, panel, spy, len(cal), float(B["max_weight"]), min_names, float(cost),
                                   float(B["borrow_bps_per_year"]))
                    d["date"] = cal.sessions
                    act = d[(d.n_long > 0) | (d.n_short > 0)]
                    if act.empty:
                        continue
                    a0, a1 = act.index.min(), act.index.max()
                    d = d.loc[a0:a1]
                    base = {"block": bname, "sp_bucket": bucket, "lag": lag, "cost_bps": cost}
                    ex = exposure_stats(d, min_names)
                    for leg, col in (("long_only", "long_ret"), ("long_minus_spy_matched", "long_excess_spy"),
                                     ("long_short_hypothetical", "ls_ret")):
                        rows.append({**base, "leg": leg, **perf(d[col].values, n_trials), **ex})
                    for leg, series in (("long_hold_component_carhart", d.long_hold.values),
                                        ("long_short_hold_component_carhart", (d.long_hold - d.short_hold).values)):
                        ca = carhart_alpha(series, pd.DatetimeIndex(d.date), fac)
                        rows.append({**base, "leg": leg, "ann_return": ca.get("alpha_ann"), "carhart_alpha_ann": ca.get("alpha_ann"),
                                     "carhart_alpha_t": ca.get("alpha_t"), "sharpe": None, "note": json.dumps(ca)})
                    ent = d.long_entry - d.spy_long_entry
                    rows.append({**base, "leg": "long_entry_day_vs_spy_open_close", "ann_return": float(ent.mean() * 252),
                                 "n_days": int((d.n_long > 0).sum())})
                    daily.append(d.assign(block=bname, sp_bucket=bucket, lag=lag, cost_bps=cost))
    bt = pd.DataFrame(rows)
    bt["wording"] = ("economic-relevance check, not an investable strategy: shorts are hypothetical (no claim of borrow "
                     f"availability; assumed borrow cost {B['borrow_bps_per_year']} bps/yr), capacity is not modelled, and the "
                     "survivor-only universe has no delisting returns")
    bt.to_csv(ctx.out / "backtest.csv", index=False)
    if daily:
        ctx.write(pd.concat(daily, ignore_index=True), "backtest_daily.parquet")
    write_json(ctx.out / "backtest_spec.json", {"signal_model": ch, "thresholds": {"short_le": lo, "long_ge": hi},
                                                "threshold_source": "walk-forward OOF signals (training period)", "horizon": h})
    return {"rows": int(len(bt)), "blocks": list(blocks)}


In [ ]:
%%writefile $REPO/src/secprice/stages_report.py
"""S12 gold tooling, S20 Track A, S21 drift tests, S23 Track C final, S24 figures + report (SPEC.md §10, §12, §20, §21)."""
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

from . import common as C
from .config import get
from .evaluation.fdr import apply_families
from .runner import stage
from .storage import write_json


# =====================================================================================================  S12
@stage("S12", "Gold export / import", "CPU", ["S10"], ["seed", "splits.dev_start"], outputs=["gold_sample.parquet", "templates/"],
       assertions=["gold-test blind", "split integrity"],
       card="Stratified gold sample from training-period filings, CSV/Excel annotation templates, validated import.")
def s12(ctx):
    from .gold.tools import cohen_kappa, export_templates, import_gold, sample_gold
    cfg = ctx.cfg
    ev = C.events(ctx)
    qb = C.question_bank(cfg)
    prof = cfg["run_profile"]
    sizes = {"gold_dev": 50, "gold_val": 100, "gold_test": 100} if prof != "smoke" else {"gold_dev": 3, "gold_val": 4, "gold_test": 4}
    smp = sample_gold(ev, pd.Timestamp(cfg["splits"]["dev_start"]), sizes, int(cfg["seed"]))
    ctx.write(smp.drop(columns=[c for c in ("accessions", "forms", "categories", "items", "section_ids") if c in smp]).assign(
        accessions=smp.accessions.map(json.dumps), forms=smp.forms.map(json.dumps)), "gold_sample.parquet")
    from .models.embeddings import event_chunks
    secs = []
    for cik in smp.cik.unique():
        p = ctx.dep("S05") / "sections" / f"{int(cik):010d}.parquet"
        if p.exists():
            secs.append(pd.read_parquet(p, columns=["accession", "section_id", "text_masked"]))
    ch = event_chunks(smp, pd.concat(secs, ignore_index=True), cfg) if secs else None
    paths = export_templates(smp, qb, ch, ctx.out / "templates", prefill=None, n_passages=200 if prof != "smoke" else 10,
                             seed=int(cfg["seed"]))
    gdir = ctx.storage.inputs / "gold"
    status = {"templates": paths, "labels_imported": False, "second_rater": False}
    if gdir.exists():
        g, errs = import_gold(gdir, smp, qb)
        status["import_errors"] = errs
        if len(g):
            ctx.write(g, "gold_labels.parquet")
            status["labels_imported"] = True
            status["n_rows"] = int(len(g))
            ann = g.groupby(["item_id"]).annotator.nunique()
            if (ann > 1).any():
                gg = g[g.item_id.isin(ann[ann > 1].index)]
                a, b = gg.annotator.unique()[:2]
                x = gg[gg.annotator == a].set_index("item_id").present_b
                y = gg[gg.annotator == b].set_index("item_id").present_b
                j = x.index.intersection(y.index)
                status["second_rater"] = True
                status["kappa_present"] = cohen_kappa(x.loc[j], y.loc[j])
    status["rater_status"] = "double-rater" if status.get("second_rater") else "single-rater, provisional"
    write_json(ctx.out / "gold_status.json", status)
    return {"sample": int(len(smp)), "labels_imported": status["labels_imported"], "rater": status["rater_status"]}


# =====================================================================================================  S20
@stage("S20", "Track A event study + correlations + topics", "CPU", ["S10"], ["targets", "contam_window_sessions", "bootstrap",
                                                                            "splits.dev_start", "seed"],
       outputs=["track_a_cars.parquet", "track_a_groups.csv", "car_paths.csv", "correlations.csv", "topics.csv"],
       assertions=["clean / all", "pre-trends"], card="CARs, abnormal volume and volatility around disclosures; association only.")
def s20(ctx):
    from .track_a.event_study import WINDOWS, car_paths, group_table, wname, window_cars
    cfg = ctx.cfg
    ev = C.events(ctx)
    cal = C.calendar(ctx)
    panel, spy = C.panel_and_spy(ctx, cal)
    uc = C.usable_cutoff(ctx)
    wc = window_cars(ev, panel, spy, C.factors(ctx), cfg, int(uc["usable_cutoff_idx"]))
    clean_col = f"overlap_flag_h{int(cfg['contam_window_sessions'])}"
    d = ev[["event_id", "cik", "event_type", "session_bucket", "earnings_type", "categories", "trackA_day0", clean_col,
            "decision_date"]].merge(wc, on="event_id", how="inner")
    d["intra_mixed_day0"] = d.session_bucket == "INTRA"
    ctx.write(d.drop(columns=["categories"]), "track_a_cars.parquet")
    cols = [wname(a, b) for a, b in WINDOWS] + [c for c in d.columns if c.startswith("car4_")] + ["abn_log_volume_0_1", "vol_response_0_1"]
    tabs = [group_table(d, "event_type", cols, clean_col), group_table(d, "session_bucket", [wname(0, 1), wname(-1, 1)], clean_col),
            group_table(d.assign(earnings_type=d.earnings_type.fillna("none")), "earnings_type", [wname(-1, 1), wname(2, 60)], clean_col)]
    ex = d.explode("categories")
    ex = ex[ex.event_type == "8K"]
    if len(ex):
        tabs.append(group_table(ex.rename(columns={"categories": "category"}), "category", [wname(-1, 1), wname(0, 1), wname(2, 20)], clean_col))
    gt = pd.concat(tabs, ignore_index=True)
    gt["fdr_family"] = "track_a_car"
    # S&P 500 vs non-S&P 500 vs combined (D-056): its own FDR family, so the original family's adjustment is unchanged
    from .evaluation.buckets import expand
    db = expand(d, C.event_buckets(ctx))
    db["event_type_x_sp_bucket"] = db.event_type + " | " + db.sp_bucket
    bcols = [wname(-1, 1), wname(0, 1), wname(0, 5), wname(2, 20), wname(2, 60), wname(-20, -2), "abn_log_volume_0_1",
             "vol_response_0_1"]
    btab = pd.concat([group_table(db, "sp_bucket", bcols, clean_col),
                      group_table(db, "event_type_x_sp_bucket", [wname(-1, 1), wname(0, 1), wname(2, 60)], clean_col)],
                     ignore_index=True)
    btab["fdr_family"] = "track_a_car_sp_bucket"
    gt = pd.concat([gt, btab], ignore_index=True)
    gt["p_value"] = gt["p"]
    gt = apply_families(gt[gt.p_value.notna()], q=float(cfg["fdr"]["q"]))
    gt.to_csv(ctx.out / "track_a_groups.csv", index=False)
    cp = car_paths(ev, panel, spy, cfg)
    cp_clean = car_paths(ev[~ev[clean_col]], panel, spy, cfg)
    pd.concat([cp.assign(event_set="all"), cp_clean.assign(event_set="clean")]).to_csv(ctx.out / "car_paths.csv", index=False)
    car_paths(expand(ev, C.event_buckets(ctx)[["event_id", "sp_bucket"]]), panel, spy, cfg,
              group_col="sp_bucket").to_csv(ctx.out / "car_paths_sp_bucket.csv", index=False)
    # correlations: tone persistence, release vs later periodic filing, filing tone vs CAR (Spearman + partial)
    F = C.read(ctx, "S09", "features.parquet")
    m = d.merge(F[["event_id"] + [c for c in F.columns if c in ("lm_net_tone", "chg_lm_net_tone", "chg_cosine", "novelty_share",
                                                                 "lm_uncertainty", "lm_negative", "acc_log_mcap", "px_mom_12_2")]],
                on="event_id", how="left")
    from scipy.stats import spearmanr
    corr = []
    for f in ("lm_net_tone", "chg_lm_net_tone", "chg_cosine", "novelty_share", "lm_uncertainty", "lm_negative"):
        for c in (wname(-1, 1), wname(0, 1), wname(2, 60)):
            if f in m and c in m:
                x = m[[f, c]].dropna()
                if len(x) > 30:
                    rho, p = spearmanr(x[f], x[c])
                    corr.append({"feature": f, "car": c, "spearman": float(rho), "p": float(p), "n": len(x)})
                    pc = _partial(m, f, c, ["acc_log_mcap", "px_mom_12_2"])
                    if pc is not None:
                        corr.append({"feature": f, "car": c, "partial_spearman_size_mom": pc[0], "p": pc[1], "n": pc[2]})
    evs = ev.merge(F[["event_id", "lm_net_tone"]], on="event_id", how="left").sort_values("decision_time")
    evs["prev_tone"] = evs.groupby(["cik", "event_type"]).lm_net_tone.shift(1)
    x = evs[["lm_net_tone", "prev_tone"]].dropna()
    if len(x) > 30:
        corr.append({"feature": "tone_persistence(prior same type)", "car": "-", "spearman": float(spearmanr(x.iloc[:, 0], x.iloc[:, 1])[0]),
                     "n": len(x)})
    cl = evs[evs.cluster_id.notna()]
    rr = cl[cl.cluster_role == "release"][["cluster_id", "lm_net_tone"]].merge(
        cl[cl.cluster_role == "periodic_after_release"][["cluster_id", "lm_net_tone"]], on="cluster_id", suffixes=("_rel", "_10q"))
    rr = rr.dropna()
    if len(rr) > 30:
        corr.append({"feature": "release_tone vs later 10-Q/10-K tone", "car": "-",
                     "spearman": float(spearmanr(rr.lm_net_tone_rel, rr.lm_net_tone_10q)[0]), "n": len(rr)})
    pd.DataFrame(corr).to_csv(ctx.out / "correlations.csv", index=False)
    # topics fitted on training years only
    topics = _topics(ctx, ev, d, pd.Timestamp(cfg["splits"]["dev_start"]))
    topics.to_csv(ctx.out / "topics.csv", index=False)
    return {"events_with_cars": int(len(d)), "group_rows": int(len(gt))}


def _partial(m, f, c, ctrls):
    from scipy.stats import spearmanr
    import statsmodels.api as sm
    cols = [f, c] + [x for x in ctrls if x in m]
    x = m[cols].dropna()
    if len(x) < 50 or len(cols) < 3:
        return None
    r = x.rank()
    Z = sm.add_constant(r[cols[2:]])
    e1 = sm.OLS(r[f], Z).fit().resid
    e2 = sm.OLS(r[c], Z).fit().resid
    rho, p = spearmanr(e1, e2)
    return float(rho), float(p), int(len(x))


def _topics(ctx, ev, cars, train_end, k=12):
    from sklearn.decomposition import NMF
    from sklearn.feature_extraction.text import TfidfVectorizer
    tx = C.read(ctx, "S09", "event_text.parquet").set_index("event_id")["text"]
    tr = ev[pd.to_datetime(ev.decision_date) < train_end].event_id
    tr = [e for e in tr if e in tx.index][:40000]
    if len(tr) < 50:
        return pd.DataFrame({"note": ["too few training documents for topics"]})
    vec = TfidfVectorizer(max_features=8000, min_df=3, stop_words="english", token_pattern=r"(?u)\b[a-zA-Z]{3,}\b", sublinear_tf=True)
    M = vec.fit_transform(tx.loc[tr].fillna(""))
    kk = min(k, max(2, M.shape[0] // 10))
    nmf = NMF(n_components=kk, random_state=0, init="nndsvda", max_iter=300).fit(M)
    W = nmf.transform(vec.transform(tx.reindex(cars.event_id).fillna("")))
    words = np.array(vec.get_feature_names_out())
    top = {i: ", ".join(words[np.argsort(-nmf.components_[i])[:8]]) for i in range(kk)}
    dom = W.argmax(1)
    c = cars.assign(topic=dom)
    rows = []
    for t, g in c.groupby("topic"):
        rows.append({"topic": int(t), "top_words": top[t], "n": len(g), "car_m1_1": g.car_m1_1.mean(), "car_2_60": g.car_2_60.mean(),
                     "fit_on": "training years only (before development holdout)"})
    return pd.DataFrame(rows)


# =====================================================================================================  S21
@stage("S21", "Drift tests", "CPU", ["S20"], ["trivial_effect_bps", "fdr", "bootstrap", "seed", "backtest.costs_bps"],
       outputs=["hypothesis_tests.csv"], assertions=["FDR families", "verdict rubric"],
       card="SUE-conditioned post-earnings drift and other drift tests with pre-registered verdict rules.")
def s21(ctx):
    from .track_a.drift import (detect_dividend, momentum_benchmark, newey_west_t, prior_quarter_quintiles, robust_same_sign,
                                spread_test, subperiod_signs, trivial_bps, verdict)
    from .track_a.event_study import window_cars
    cfg = ctx.cfg
    tb = cfg["trivial_effect_bps"]
    nb, seed = int(cfg["bootstrap"]["n"]), int(cfg["seed"])
    d = C.read(ctx, "S20", "track_a_cars.parquet")
    ev = C.events(ctx)
    F = C.read(ctx, "S09", "features.parquet")
    clean_col = f"overlap_flag_h{int(cfg['contam_window_sessions'])}"
    rows = []
    car = "car4_2_60" if "car4_2_60" in d and d["car4_2_60"].notna().sum() > 100 else "car_2_60"
    rel = d[d.earnings_type == "earnings_announcement_verified"].merge(F[["event_id", "sue_event"]], on="event_id", how="left")
    cal = C.calendar(ctx)
    panel, spy = C.panel_and_spy(ctx, cal)
    uc = C.usable_cutoff(ctx)
    def add(test, design, res, trivial, signs=None, clean_ok=None, cost_ok=None, lag_ok=None, note=""):
        r = {"test": test, "fdr_family": "drift", "design": design, "trivial_bps": trivial, "note": note,
             "subperiod_signs": json.dumps(signs) if signs else None, "clean_robust": clean_ok, "cost_robust": cost_ok,
             "lag_robust": lag_ok}
        if res:
            r.update({k: res.get(k) for k in ("estimate", "ci_low", "ci_high", "p_value", "n", "t")})
        rows.append(r)

    if rel.sue_event.notna().sum() >= 50:
        rel["q_sue"] = prior_quarter_quintiles(rel, "sue_event")
        res = spread_test(rel, "q_sue", car, "cik", "trackA_day0", nb, seed)
        signs = subperiod_signs(rel.dropna(subset=["q_sue"]), "q_sue", car) if res else None
        cres = spread_test(rel[~rel[clean_col]], "q_sue", car, "cik", "trackA_day0", nb, seed)
        clean_ok = robust_same_sign(res, cres)
        cost = 4 * max(cfg["backtest"]["costs_bps"]) / 1e4
        cost_ok = bool(res and abs(res["estimate"]) - cost > trivial_bps(60, tb) / 1e4)
        lag_col = "car4_3_61" if car.startswith("car4") and "car4_3_61" in rel else "car_3_61"
        lres = spread_test(rel, "q_sue", lag_col, "cik", "trackA_day0", nb, seed)
        lag_ok = robust_same_sign(res, lres)
        add("SUE-conditioned post-earnings drift (Q5-Q1, SUE_release)", f"verified releases; {car}; prior-quarter breakpoints",
            res, trivial_bps(60, tb), signs, clean_ok, cost_ok, lag_ok,
            f"one-session lag = {lag_col}; costs = 4 x {max(cfg['backtest']['costs_bps'])} bps round trip on the spread; "
            "never a consensus-based PEAD replication; the verdict says nothing about direction: a negative Q5-Q1 is a "
            "reversal, and the survivor-only universe (firms listed in 2026) biases low-SUE drift upwards (D-057, D-073)")
        x_sue = _xbrl_sue_for_releases(ctx, rel)
        if x_sue is not None and x_sue.notna().sum() >= 50:
            rx = rel.assign(sue_x=x_sue.values)
            rx["q_x"] = prior_quarter_quintiles(rx, "sue_x")
            add("drift sorted on SUE_xbrl (EX-POST sensitivity, not tradable)", f"verified releases; {car}",
                spread_test(rx, "q_x", car, "cik", "trackA_day0", nb, seed), trivial_bps(60, tb),
                note="XBRL EPS comes from the LATER 10-Q/10-K of the same cluster: labelled ex-post, never a feature")
        ear = rel.assign(q_ear=prior_quarter_quintiles(rel, "car_m1_1"))
        add("drift sorted on EAR (observed before +2)", f"verified releases; {car}", spread_test(ear, "q_ear", car, "cik", "trackA_day0", nb, seed),
            trivial_bps(60, tb), note="sensitivity")
    else:
        add("SUE-conditioned post-earnings drift", "verified releases", None, trivial_bps(60, tb),
            note=f"only {int(rel.sue_event.notna().sum())} verified releases with SUE_release")
    # the same SUE drift test within each index bucket (D-056); own FDR family, exploratory decomposition
    bmap = C.event_buckets(ctx).set_index("event_id").sp_bucket
    for b, lab_b in (("SP500", "S&P 500 members"), ("NonSP500", "non-S&P 500 firms")):
        rb = rel[rel.event_id.map(bmap).fillna("NonSP500") == b].copy()
        nsue = int(rb.sue_event.notna().sum())
        if nsue >= 50:
            rb["q_sue"] = prior_quarter_quintiles(rb, "sue_event")
            res_b = spread_test(rb, "q_sue", car, "cik", "trackA_day0", nb, seed)
            signs_b = subperiod_signs(rb.dropna(subset=["q_sue"]), "q_sue", car) if res_b else None
            add(f"SUE-conditioned post-earnings drift (Q5-Q1) [{lab_b}]", f"verified releases of {lab_b}; {car}; "
                "prior-quarter breakpoints within the bucket", res_b, trivial_bps(60, tb), signs_b,
                note="bucket decomposition of the combined test (exploratory; robustness checks only on the combined test; "
                     "survivorship bias is likely larger for non-S&P firms)")
            rows[-1]["fdr_family"] = "drift_sp_bucket"
        else:
            rows.append({"test": f"SUE-conditioned post-earnings drift (Q5-Q1) [{lab_b}]", "fdr_family": "drift_sp_bucket",
                         "n": nsue, "verdict": "inconclusive/low power", "note": f"only {nsue} verified releases with SUE_release"})
    # momentum benchmarks (never attributed to filings)
    u = C.universe(ctx)
    mb = momentum_benchmark(panel, u.ticker.tolist(), cfg["study"]["start"], cfg["study"]["end"])
    for sig, g in mb.groupby("signal") if len(mb) else []:
        x = g.ls_ret.to_numpy()
        t = newey_west_t(x)
        rows.append({"test": f"momentum benchmark: {sig} (monthly decile long-short)", "fdr_family": "drift_benchmark",
                     "design": "equal-weight deciles within our universe", "estimate": float(np.mean(x)), "t": t,
                     "p_value": None if t is None else float(2 * (1 - __import__("scipy.stats").stats.norm.cdf(abs(t)))),
                     "n": len(x), "trivial_bps": trivial_bps(20, tb), "note": "benchmark only; never attributed to filings"})
    # does text add value beyond momentum? CAR[+2,+60] on text features with momentum controls (two-way clustered)
    import statsmodels.api as sm
    mm = d.merge(F[["event_id", "lm_net_tone", "chg_cosine", "novelty_share", "px_mom_12_2", "px_rev_1m", "acc_log_mcap"]],
                 on="event_id", how="left")
    cols = ["lm_net_tone", "chg_cosine", "novelty_share", "px_mom_12_2", "px_rev_1m"]
    x = mm[cols + ["car_2_60", "cik", "trackA_day0"]].dropna()
    if len(x) > 200:
        from .track_a.event_study import cluster_fit
        sd = x[cols].std().replace(0, np.nan)
        Z = sm.add_constant(((x[cols] - x[cols].mean()) / sd).fillna(0.0))
        fit, _how = cluster_fit(x.car_2_60.to_numpy(float), Z, x.cik.to_numpy(), x.trackA_day0.astype(str).to_numpy())
        fit.params.index = Z.columns
        fit.tvalues.index = Z.columns
        fit.pvalues.index = Z.columns
        for c in ("lm_net_tone", "chg_cosine", "novelty_share"):
            rows.append({"test": f"text beyond momentum: {c} -> CAR[+2,+60]", "fdr_family": "drift", "design": "OLS with momentum "
                         "controls, standardized, two-way clustered", "estimate": float(fit.params[c]), "t": float(fit.tvalues[c]),
                         "p_value": float(fit.pvalues[c]), "n": int(len(x)), "trivial_bps": trivial_bps(60, tb)})
    # post-dividend announcement drift (detector provisional until validated on gold)
    k8 = d[d.event_type == "8K"]
    tx = C.read(ctx, "S09", "event_text.parquet").set_index("event_id")["text"]
    k8 = k8.assign(div=k8.event_id.map(lambda e: detect_dividend(tx.get(e, "") or "")))
    for kind in ("increase", "decrease", "initiation"):
        g = k8[k8["div"] == kind]
        if len(g) >= 100:
            from .track_a.event_study import clustered_mean
            st = clustered_mean(g["car_2_60"].to_numpy(float), g.cik.to_numpy(), g.trackA_day0.astype(str).to_numpy())
            rows.append({"test": f"post-dividend-announcement drift: {kind}", "fdr_family": "drift", "design": "8-K text detector; CAR[+2,+60]",
                         "estimate": st["mean"], "t": st["t"], "p_value": st["p"], "n": st["n"], "trivial_bps": trivial_bps(60, tb),
                         "note": "detector provisional (precision not validated on gold); announcement != ex-date; total return"})
        else:
            rows.append({"test": f"post-dividend-announcement drift: {kind}", "fdr_family": "drift", "n": int(len(g)),
                         "verdict": "inconclusive/low power", "note": "fewer than 100 detected announcements"})
    # spin-off / restructuring drift (parent only)
    ex = d[d.event_type == "8K"].merge(ev[["event_id", "categories"]], on="event_id")
    for cat in ("restructuring_exit", "impairment", "acquisition_disposition"):
        g = ex[ex.categories.map(lambda c: cat in c)]
        if len(g) >= 100:
            from .track_a.event_study import clustered_mean
            st = clustered_mean(g["car_2_60"].to_numpy(float), g.cik.to_numpy(), g.trackA_day0.astype(str).to_numpy())
            rows.append({"test": f"restructuring / spin-off drift: 8-K {cat}", "fdr_family": "drift", "design": "parent only; CAR[+2,+60]",
                         "estimate": st["mean"], "t": st["t"], "p_value": st["p"], "n": st["n"], "trivial_bps": trivial_bps(60, tb)})
    for t, why in (("analyst revisions drift", "no consensus data; a management-guidance proxy is a different construct"),
                   ("insider trading drift", "Form 4 is an extension (ext_data_modules)"),
                   ("institutional ownership drift", "13F is an extension (ext_data_modules)")):
        rows.append({"test": t, "fdr_family": "not_testable", "verdict": "not testable in scope", "note": why})
    ht = pd.DataFrame(rows)
    for c in ("p_value", "estimate", "ci_low", "ci_high", "n", "verdict", "subperiod_signs", "clean_robust", "cost_robust",
              "lag_robust", "trivial_bps"):
        if c not in ht:
            ht[c] = None
    ht["p_adj_bh"] = None
    tst = ht[ht.p_value.notna()]
    if len(tst):
        adj = apply_families(tst, q=float(cfg["fdr"]["q"]))
        ht.loc[adj.index, "p_adj_bh"] = adj.p_adj_bh
    for i, r in ht.iterrows():
        if pd.notna(r.get("verdict")) and r.get("verdict"):
            continue
        if r.get("fdr_family") == "drift_benchmark":
            ht.at[i, "verdict"] = "benchmark (not a filing effect)"
            continue
        res = None if pd.isna(r.get("estimate")) else {"estimate": r.estimate, "ci_low": r.get("ci_low"), "ci_high": r.get("ci_high"),
                                                       "p_value": r.get("p_value"), "n": int(r.n) if pd.notna(r.n) else 0}
        signs = json.loads(r.subperiod_signs) if isinstance(r.get("subperiod_signs"), str) else []
        ht.at[i, "verdict"] = verdict(res, float(r.trivial_bps or 50), r.get("p_adj_bh"), signs,
                                      r.get("clean_robust"), r.get("cost_robust"), r.get("lag_robust"))
    ht["rule"] = "supported needs FDR p<0.05, CI excluding the trivial threshold, >=2/3 subperiod signs, clean, cost and lag robustness"
    ht.to_csv(ctx.out / "hypothesis_tests.csv", index=False)
    ht.to_csv(ctx.storage.reports / "hypothesis_tests.csv", index=False)
    return {"tests": int(len(ht)), "verdicts": ht.verdict.value_counts().to_dict()}


def _xbrl_sue_for_releases(ctx, rel: pd.DataFrame):
    """SUE of each release computed with the later same-cluster periodic filing's own XBRL (ex-post sensitivity)."""
    ev = C.events(ctx)
    sue = C.read(ctx, "S06", "sue_by_accession.parquet")
    if sue.empty:
        return None
    a2s = dict(zip(sue.accession, sue.sue))
    per = ev[ev.cluster_role == "periodic_after_release"].drop_duplicates("cluster_id").set_index("cluster_id")
    cl = ev.set_index("event_id")["cluster_id"]
    out = []
    for eid in rel.event_id:
        c = cl.get(eid)
        v = np.nan
        if c is not None and c in per.index:
            v = a2s.get(list(per.loc[c, "accessions"])[0], np.nan)
        out.append(v)
    return pd.Series(out, index=rel.index)


# =====================================================================================================  S23
@stage("S23", "Track C final (gold-test)", "CPU", ["S12"], ["gates"], outputs=["content_validity.csv"],
       assertions=["gold-test touched once"], card="Content validity on held-out human labels (never against prices).")
def s23(ctx):
    from .gold.tools import GoldTestLock
    gd = ctx.dep("S12")
    s16 = ctx.dep_or_none("S16")
    lab = pd.read_parquet(gd / "gold_labels.parquet") if (gd / "gold_labels.parquet").exists() else pd.DataFrame()
    test = lab[lab.gold_split == "gold_test"] if len(lab) else lab
    if test.empty or s16 is None or not json.loads((s16 / "rag_frozen.json").read_text()).get("frozen"):
        pd.DataFrame([{"metric": "all", "status": "not evaluated: gold-test labels and a frozen RAG pipeline are both required"}]).to_csv(
            ctx.out / "content_validity.csv", index=False)
        return {"evaluated": False}
    lock = GoldTestLock(ctx.storage.manifests / "gold_test_lock.json")
    frozen = json.loads((ctx.storage.manifests / "frozen_architecture.json").read_text()) if (ctx.storage.manifests / "frozen_architecture.json").exists() else {}
    lock.consume(frozen.get("g4_hash", "no-g4"))
    ext = pd.read_parquet(ctx.dep("S17") / "extractions.parquet") if ctx.dep_or_none("S17") and (ctx.dep("S17") / "extractions.parquet").exists() else pd.DataFrame()
    rows = []
    from sklearn.metrics import f1_score, precision_score, recall_score
    for src in ("teacher", "student"):
        e = ext[ext.source == src] if len(ext) else ext
        if e.empty:
            continue
        m = test.merge(e[["event_id", "field", "present"]].rename(columns={"present": "pred"}), on=["event_id", "field"], how="inner")
        for f, g in m.groupby("field"):
            y, p = g.present_b.astype(bool), g.pred.fillna(False).astype(bool)
            rows.append({"source": src, "field": f, "n": len(g), "precision": precision_score(y, p, zero_division=0),
                         "recall": recall_score(y, p, zero_division=0), "f1": f1_score(y, p, zero_division=0)})
    cv = pd.DataFrame(rows)
    st = json.loads((gd / "gold_status.json").read_text())
    cv["rater_status"] = st.get("rater_status")
    cv.to_csv(ctx.out / "content_validity.csv", index=False)
    return {"evaluated": True, "rows": len(cv)}


# =====================================================================================================  S24
@stage("S24", "Figures + report", "CPU", ["S10", "S11", "S20", "S21"], ["run_profile"],
       outputs=["reports/results_summary.md", "reports/results_report.html", "reports/results_report.pdf", "figures/"],
       assertions=["computed values only"], card="All figures and the full extracted report, built only from artifacts.")
def s24(ctx):
    from .report.build import build_html, build_markdown, build_pdf
    from .viz.figures import Figs
    facts, figs = collect(ctx)
    md = build_markdown(facts)
    rep = ctx.storage.reports
    (rep / "results_summary.md").write_text(md, encoding="utf-8")
    (ctx.out / "results_summary.md").write_text(md, encoding="utf-8")
    html = build_html(md, figs)
    (rep / "results_report.html").write_text(html, encoding="utf-8")
    err = build_pdf(md, figs, rep / "results_report.pdf")
    if err:
        ctx.warn(err)
    return {"figures_ok": sum(f["status"] == "ok" for f in figs), "figures_skipped": sum(f["status"] != "ok" for f in figs),
            "report": str(rep / "results_summary.md")}


def _csv(path, **kw):
    """read_csv that returns an empty frame for an empty file (tiny smoke folds can write empty tables)."""
    try:
        return pd.read_csv(path, **kw)
    except (pd.errors.EmptyDataError, FileNotFoundError):
        return pd.DataFrame()


def _opt(ctx, stage_id, name, kind="parquet"):
    d = ctx.dep_or_none(stage_id)
    if d is None or not (d / name).exists():
        return None
    p = d / name
    if kind == "parquet":
        return pd.read_parquet(p)
    if kind == "csv":
        x = _csv(p)
        return x if len(x.columns) else None
    return json.loads(p.read_text(encoding="utf-8"))


def collect(ctx):
    """Assemble report facts + draw all figures from artifacts (missing inputs -> 'not produced: reason')."""
    from .evaluation.metrics import classification_metrics
    from .events.timestamps import timing_policy_label
    from .viz.figures import Figs
    cfg = ctx.cfg
    F = Figs(ctx.storage.figures)
    ev = C.events(ctx)
    u = C.universe(ctx, selected_only=False)
    usel = u[u.selected]
    man = C.manifest(ctx)
    led = C.read(ctx, "S05", "parse_ledger.parquet")
    tg = C.read(ctx, "S08", "targets.parquet")
    fpy = _csv(ctx.dep("S02") / "firms_per_year.csv")
    uc = C.usable_cutoff(ctx)
    facts = {"profile": cfg["run_profile"], "generated": pd.Timestamp.utcnow().strftime("%Y-%m-%d %H:%M UTC"),
             "config_hash": ctx.storage.global_config_hash, "timing_policy": timing_policy_label(cfg)}
    from .storage import git_commit
    facts["code"] = git_commit()
    ph = ctx.storage.manifests / "prereg_hash.txt"
    facts["prereg_hash"] = ph.read_text().strip() if ph.exists() else "not frozen"
    # ---------------- data integrity
    spc = _opt(ctx, "S02", "sp500_coverage.json", "json") or {}
    facts["universe_text"] = (
        f"Universe: **{len(usel)} unique firms** (target {cfg['universe']['n_stocks']}) common to our OHLCV files and the HF 10-K corpus: "
        f"**{int(usel.ever_sp500.sum())} S&P 500 members** (at some point in {cfg['study']['start']} to {cfg['study']['end']}, "
        f"fja05680/sp500 @ {cfg['universe']['sp500_commit'][:8]}) and **{int((~usel.ever_sp500.astype(bool)).sum())} non-members** "
        "(seeded, sector-stratified fill); results are reported for each bucket and combined. "
        "**Survivor-only:** every firm is a current listing, so firms that delisted, merged or were acquired before 2026 are absent "
        "(survivorship-biased; drift and backtest results are biased towards survivors)"
        + (f"; our data covers about {spc['covered_by_eligible_universe']} of the {spc['sp500_tickers_in_window']} tickers that were "
           f"S&P 500 members in the window ({spc['coverage_share']:.0%}, ticker-level, approximate). " if spc else ". ")
        + f"Candidates: {len(u)}; eligible: {int(u.eligible.sum())}. "
        f"Exclusions: {u.exclusion_reason.value_counts().to_dict()}. Usable cutoff {uc['usable_cutoff']} "
        f"(last price {uc['last_price_date']}, last document {uc['last_document_date']}); final month partial: {uc['final_month_partial']}. "
        "Firms per year are reported separately below; 500 firms per year is never claimed.")
    pc = _opt(ctx, "S03", "price_coverage.csv", "csv")
    if pc is not None and len(fpy) and {"first", "last"} <= set(pc.columns):
        # S02 counts only the local OHLCV span; S03 knows the spliced series (yfinance warm-up before the local start)
        f0, f1 = pd.to_datetime(pc["first"]), pd.to_datetime(pc["last"])
        fpy = fpy.copy()
        fpy["firms_with_prices"] = [int(((f0 <= pd.Timestamp(f"{y}-12-31")) & (f1 >= pd.Timestamp(f"{y}-01-01"))).sum())
                                    for y in fpy.year]
    facts["firms_per_year"] = fpy
    mm = man.merge(led[["accession", "status", "n_sections"]], on="accession", how="left")
    mm["year"] = pd.to_datetime(mm.filing_date).dt.year
    cov = mm[mm.download_task].groupby("base_form").agg(observed=("accession", "size"), parsed=("n_sections", lambda s: int((s.fillna(0) > 0).sum())))
    cov["parsed_share"] = cov.parsed / cov.observed
    facts["coverage_by_form"] = cov.reset_index()
    cfy = _csv(ctx.dep("S04") / "coverage_firm_year.csv")
    facts["reason_codes"] = cfy.reason_code.value_counts().rename_axis("reason_code").reset_index(name="firm_years") if len(cfy) else None
    k8 = ev[ev.event_type == "8K"].explode("categories")
    cby = k8.groupby(["year", "categories"]).size().unstack(fill_value=0) if len(k8) else pd.DataFrame()
    facts["category_by_year"] = cby.T.reset_index().rename(columns={"categories": "category"}) if len(cby) else None
    tq = ev.groupby(["year", "ts_quality"]).size().unstack(fill_value=0)
    facts["ts_quality"] = tq.reset_index()
    facts["timestamp_text"] = (f"Acceptance times: SGML header when the download carried one; otherwise the submissions-JSON value under the "
                               f"verified double-conversion rule (header disagreement rate {uc.get('json_rule_mismatch')} over "
                               f"{uc.get('json_rule_n')} checks); pre-2002 filings have midnight placeholders (quality B, assumed 17:30 ET). "
                               "Public availability is ASSUMED (acceptance + lag), not observed.")
    hfa = ctx.dep("S01") / "hf_dataset_audit.md"
    facts["hf_text"] = hfa.read_text(encoding="utf-8").split("## Filings by year")[0] if hfa.exists() else "n/a"
    facts["data_limitations"] = ["SURVIVOR-ONLY universe: all firms are current listings (no delisted, merged or acquired firms), so the "
                                 "sample is survivorship-biased, not merely survivorship-aware; drift and backtest results overstate what "
                                 "a point-in-time universe would show. Daily bars only.",
                                 "Free prices are weak on delistings; local OHLCV is split-adjusted and dividends come from Yahoo actions.",
                                 "Selected sections only (10-K Items 1/1A/7/7A, 10-Q MD&A/market risk/risk factors, 8-K items + EX-99).",
                                 "SUE coverage is limited before the XBRL era (2009); release-EPS parsing is heuristic (QA vs XBRL reported).",
                                 "The HF corpus contains 10-Ks only (1994-2021); it defines the firm overlap and QA, never the study text."]
    # ---------------- design
    facts["design_text"] = ("Four timestamps per event; T_trade (primary, prospective: entry at the first open at or after decision time, "
                            "SPY on the identical interval) and T_info (information response; INTRA excluded); 3-class labels with "
                            f"k={cfg['labels']['k']} x sigma x sqrt(h); per-horizon censoring at the usable cutoff; overlap flags "
                            f"(clean events = no same-firm event within {cfg['contam_window_sessions']} sessions).")
    facts["event_counts"] = ev.groupby(["event_type", "session_bucket"]).size().unstack(fill_value=0).reset_index()
    # ---------------- leakage / provenance
    facts["leakage_audit"] = _csv(ctx.dep("S10") / "leakage_audit.csv")
    facts["ledger"] = _csv(ctx.dep("S10") / "model_provenance_ledger.csv")[["key", "model_id", "revision", "license", "stated_cutoff"]]
    sanity = []
    perm = _opt(ctx, "S11", "permutation.csv", "csv")
    if perm is not None and len(perm):
        obs = perm[perm.kind == "observed"].macro_f1.iloc[0]
        null = perm[perm.kind == "permuted"].macro_f1
        sanity.append({"check": "permutation test (within-month, class-stratified)", "observed": obs, "null_mean": null.mean(),
                       "p_value": float((null >= obs).mean()) if len(null) else None, "n_perm": len(null)})
    rd = _opt(ctx, "S11", "random_demo.json", "json")
    if rd:
        sanity.append({"check": "random-split demo (pedagogical)", "observed": rd["random_split_macro_f1"], "null_mean": rd["walk_forward_macro_f1"]})
    adv = _opt(ctx, "S11", "adversarial.csv", "csv")
    if adv is not None and len(adv):
        note = ("AUC near 1: training and validation periods are separable (covariate shift); this is not by itself evidence "
                "of leakage, which the audit above tests directly")
        if "top_shift_features" in adv:
            top = json.loads(adv.top_shift_features.iloc[-1])
            note += "; main drivers (last fold): " + ", ".join(n for n, _ in top[:4])
        sanity.append({"check": "adversarial validation AUC, firm-level features (mean over folds)", "observed": adv.auc.mean(),
                       "null_mean": 0.5, "note": note if adv.auc.mean() > 0.8 else "AUC near 0.5 does not prove the split is unbiased"})
        if "auc_all_features" in adv:
            sanity.append({"check": "adversarial validation AUC, all features incl. market-wide / calendar (mean)",
                           "observed": adv.auc_all_features.mean(), "null_mean": 0.5,
                           "note": "date-level features fingerprint each date, so this AUC is ~1 by construction"})
    trig = _opt(ctx, "S18", "review_triggers.json", "json")
    if trig:
        sanity.append({"check": "review triggers", "observed": len(trig["triggers"]), "null_mean": None, "p_value": None,
                       "n_perm": None})
    facts["sanity"] = pd.DataFrame(sanity)
    # ---------------- Track A
    ta = _opt(ctx, "S20", "track_a_groups.csv", "csv")
    if ta is not None and len(ta):
        main = ta[(ta.group_by == "event_type") & ta.measure.isin(["car_m1_1", "car_0_1", "car_2_60", "car4_m1_1", "car4_2_60"])]
        facts["track_a_main"] = main[["group", "event_set", "measure", "mean", "t", "p_adj_bh", "n"]]
    facts["hypothesis_tests"] = _opt(ctx, "S21", "hypothesis_tests.csv", "csv")
    ht0 = facts["hypothesis_tests"]
    if ht0 is not None and len(ht0):
        sue = ht0[ht0.test.str.startswith("SUE-conditioned post-earnings drift (Q5-Q1, SUE_release)")]
        if len(sue) and pd.notna(sue.estimate.iloc[0]):
            r = sue.iloc[0]
            direction = ("a REVERSAL (high-SUE firms underperform low-SUE firms after the announcement), the opposite of classic "
                         "post-earnings drift" if r.estimate < 0 else "a continuation in the direction of the surprise")
            facts["drift_text"] = (f"SUE-conditioned drift (Q5-Q1, CAR[+2,+60]) = {r.estimate:+.2%} [{r.ci_low:+.2%}, {r.ci_high:+.2%}], "
                                   f"n={int(r.n):,}, verdict **{r.verdict}**: {direction}. The verdict rubric tests strength and "
                                   "robustness, not direction. The universe is survivor-only: firms that delisted after bad news "
                                   "are absent, which pushes low-SUE drift upwards; these data cannot separate that bias from a "
                                   "genuine reversal. Never a consensus-based PEAD replication (decision log D-073).")
    if facts["hypothesis_tests"] is not None:
        facts["hypothesis_tests"] = facts["hypothesis_tests"][[c for c in ("test", "estimate", "ci_low", "ci_high", "p_value", "p_adj_bh", "n",
                                                                           "verdict") if c in facts["hypothesis_tests"]]]
    # ---------------- Track B
    rl = pd.concat([x for x in (_opt(ctx, "S11", "results_long.parquet"), _opt(ctx, "S15", "results_long.parquet")) if x is not None],
                   ignore_index=True)
    prim = cfg["targets"]["primary"]
    lad = rl[(rl.metric == "macro_f1") & (rl.block == "walk_forward_pooled") & (rl.event_set == "all")
             & (rl.target_def == prim["target_def"]) & (rl.horizon == prim["horizon"])]
    facts["ladder"] = lad[["rung", "model_id", "revision", "cohort", "tuned", "leaderboard", "mean_fold_score", "value", "ci_low",
                           "ci_high", "n_events"]].rename(columns={"value": "pooled_macro_f1", "revision": "encoder"}).sort_values(["cohort", "rung"])
    gr = rl[(rl.block == "walk_forward_pooled") & (rl.event_set == "all") & (rl.target_def == prim["target_def"])
            & (rl.horizon == prim["horizon"]) & (rl.cohort == "common") & rl.tuned.isin(["tuned", "default"])
            & rl.metric.isin(["logloss", "logloss_prior", "guardrail_logloss_ok", "ece", "guardrail_ece_ok"])]
    facts["guardrails"] = gr.pivot_table(index=["rung", "model_id", "revision", "tuned"], columns="metric", values="value").reset_index() \
        if len(gr) else None
    tb = _csv(ctx.dep("S11") / "ladder_summary.csv")
    t15 = _opt(ctx, "S15", "emb_ladder_summary.csv", "csv")
    tba = pd.concat([tb.assign(encoder="-"), t15] if t15 is not None else [tb.assign(encoder="-")], ignore_index=True)
    facts["tuning_before_after"] = tba[["rung", "encoder", "head", "default_mean_f1", "tuned_mean_f1"]]
    hyp = []
    g3 = _opt(ctx, "S15", "g3.json", "json")
    if g3 and g3.get("lift"):
        d = g3["lift"]
        bl0 = _csv(ctx.dep("S10") / "blocks.csv")
        wfb = bl0[bl0.kind == "walk_forward_val"] if len(bl0) else bl0
        span = (f"walk-forward validation blocks {str(wfb.start.min())[:10]} to {str(wfb.end_exclusive.max())[:10]}"
                if len(wfb) else "walk-forward validation")
        facts["h1"] = {"value": d["value"], "ci_low": d["ci_low"], "ci_high": d["ci_high"], "n_events": g3.get("n_events"),
                       "block": span}
        hyp.append({"hypothesis": "H1 (walk-forward): best L7 - L6", **{k: d[k] for k in ("value", "ci_low", "ci_high", "p_boot")},
                    "n": g3.get("n_events"), "G3": g3["passed"]})
    dl = _opt(ctx, "S18", "dev_lifts.csv", "csv")
    if dl is not None:
        for r in dl.itertuples(index=False):
            hyp.append({"hypothesis": f"{r.hypothesis} (development holdout): {r.a} - {r.b}", "value": r.value, "ci_low": r.ci_low,
                        "ci_high": r.ci_high, "p_boot": r.p_boot, "n": r.n})
    h4 = _opt(ctx, "S15", "h4.csv", "csv")
    if h4 is not None and len(h4):
        for r in h4.itertuples(index=False):
            hyp.append({"hypothesis": f"H4 {r.rung}/{r.head}: ChronoBERT vintage - {r.static}", "value": r.diff_vintage_minus_static,
                        "ci_low": r.ci_low, "ci_high": r.ci_high, "p_boot": r.p_boot, "n": r.n})
    h2 = lad[(lad.rung.isin(["L6", "L6_nochange"])) & (lad.cohort == "common") & (lad.tuned == "tuned")]
    facts["hypotheses_core"] = pd.DataFrame(hyp)
    facts["trackb_text"] = (f"Primary target: {prim['target_def']} h={prim['horizon']} 3-class; primary metric: mean walk-forward macro-F1 "
                            "(guardrails: log-loss no worse than the class prior; ECE). Results are validation-selected. Strict leaderboard = "
                            "models whose documented cutoff precedes every evaluated event (ChronoBERT vintages, models trained on our folds, "
                            "lexical baselines; SEC-BERT only after 2019); retrospective = all models.")
    facts["dev_results"] = _filter_main(_opt(ctx, "S18", "dev_results.parquet"))
    lr = _opt(ctx, "S19", "locked_results.parquet")
    lock = ctx.storage.manifests / "lockbox_log.json"
    facts["locked_text"] = ("Locked test scored once: " + lock.read_text()) if lock.exists() else \
        "The locked test (2023-01-01 to the usable cutoff) is **sealed**: it has not been scored. Set lockbox.unlock_locked_test: true after reviewing the development results."
    facts["locked_results"] = _filter_main(lr)
    prs = _opt(ctx, "S11", "predictions.parquet")
    sel_rows = rl[(rl.block == "walk_forward_pooled") & (rl.event_set == "all") & (rl.target_def == prim["target_def"])
                  & (rl.horizon == prim["horizon"]) & rl.metric.str.contains("_at_")]
    facts["selective"] = sel_rows.pivot_table(index=["rung", "model_id", "revision", "tuned"], columns="metric", values="value").reset_index().head(20) \
        if len(sel_rows) else None
    ti = rl[(rl.metric == "macro_f1") & (rl.block == "walk_forward_pooled") & (rl.event_set == "all") & rl.target_def.isin(["T_info", "T_trade"])
            & (rl.horizon == 1) & (rl.cohort == "common")]
    facts["tinfo_vs_ttrade"] = ti[["rung", "model_id", "target_def", "tuned", "value", "n_events"]] if len(ti) else None
    bt_all = _opt(ctx, "S22", "backtest.csv", "csv")
    facts["backtest"] = None
    if bt_all is not None:
        btc = bt_all[bt_all.sp_bucket == "Combined"] if "sp_bucket" in bt_all else bt_all
        facts["backtest"] = btc[[c for c in ("block", "lag", "cost_bps", "leg", "ann_return", "ann_vol", "sharpe", "deflated_sharpe_prob",
                                             "max_drawdown", "carhart_alpha_ann", "carhart_alpha_t", "turnover_long_daily",
                                             "avg_names_long", "avg_names_short", "share_days_short_in_cash", "n_days") if c in btc]]
        facts["backtest_text"] = (bt_all.wording.iloc[0] if "wording" in bt_all and len(bt_all) else "") + \
            ". Turnover is the mean daily one-way turnover; 'short in cash' = days the short leg had fewer than " \
            f"{cfg['backtest']['min_names_per_side']} names and held cash."
    _bucket_facts(ctx, facts, cfg, prim, bt_all)
    # ---------------- Track C, fit, extensions, limitations
    gs = _opt(ctx, "S12", "gold_status.json", "json") or {}
    cv = _opt(ctx, "S23", "content_validity.csv", "csv")
    g3b = _opt(ctx, "S16", "g3b.json", "json")
    facts["trackc_text"] = (f"Gold set: {gs.get('rater_status', 'n/a')}; labels imported: {gs.get('labels_imported')}. "
                            f"G3b: {g3b}. Price outcomes never validate semantic labels.")
    facts["trackc"] = cv
    fd = _csv(ctx.dep("S11") / "fit_diagnosis.csv")
    facts["fit_diag"] = fd[["rung", "head", "mean_val", "mean_train", "std", "verdict"]]
    cert = _opt(ctx, "S11", "enet_certificate.json", "json")
    facts["optim_text"] = ("\n**Optimisation diagnostics.** " + (
        f"Elastic-net certificate: objective spread across random starts {cert['spread']:.2e} -> global optimum of the convex problem "
        f"{'certified' if cert['certified_global_min'] else 'NOT certified'}. " if cert else "") +
        "Tree and transformer heads are non-convex: their optima are validation-selected (best Optuna trial / early-stopped checkpoint), "
        "not global minima.")
    ext = [{"extension": k, "enabled": v} for k, v in cfg["extensions"].items() if v]
    mp = _opt(ctx, "S18", "mcs_pbo.json", "json")
    facts["extensions"] = pd.DataFrame(ext) if ext else None
    facts["limitations"] = ["Association is not causation; prediction is not profitability; the backtest is an economic-relevance check only.",
                            "Public availability times are assumed (acceptance + 5 min), not observed; results carry that timing policy.",
                            "Pretrained encoders without documented cutoffs (FinBERT-tone, BGE, Qwen, ModernBERT) appear only on the "
                            "retrospective leaderboard and support no clean-test claim.",
                            f"MCS / PBO are diagnostics only: {json.dumps(mp)[:400] if mp else 'not run'}"]
    man_run = json.loads((ctx.storage.manifests / "run_manifest.json").read_text()) if (ctx.storage.manifests / "run_manifest.json").exists() else {}
    facts["resources"] = pd.DataFrame([{"stage": k, "wall_s": v.get("wall_s"), "status": v.get("status")} for k, v in man_run.get("stages", {}).items()])
    facts["decision_log"] = "docs/decision_log.md"
    # executive bullets (computed values only)
    eb = [f"{len(usel)} firms, {len(ev):,} events ({ev.event_type.value_counts().to_dict()}), {cfg['study']['start']} to {uc['usable_cutoff']}."]
    if len(lad):
        best = lad[lad.cohort.str.startswith("common")].sort_values("mean_fold_score", ascending=False).head(1)
        if len(best):
            b = best.iloc[0]
            eb.append(f"Best walk-forward model: {b.rung}/{b.model_id}{'/' + str(b.revision) if b.revision != 'n/a' else ''} "
                      f"mean macro-F1 {b.mean_fold_score:.3f} (pooled {b.value:.3f}, CI {b.ci_low if b.ci_low is None else round(b.ci_low, 3)} to "
                      f"{b.ci_high if b.ci_high is None else round(b.ci_high, 3)}).")
        l0 = lad[lad.rung == "L0"]
        if len(l0):
            eb.append(f"Majority-class baseline macro-F1: {l0.value.iloc[0]:.3f}.")
    if g3:
        eb.append(f"G3 (text lift >= {cfg['gates']['text_lift']} with CI excluding 0): {'passed' if g3.get('passed') else 'not passed'}.")
    sc = facts.get("sp_scores_wf")
    if sc is not None and len(sc) and "rung" in sc:
        l6 = sc[sc.rung == "L6"].set_index("sp_bucket")
        comp = facts.get("sp_composition")
        nfirms = comp.set_index("sp_bucket").firms.to_dict() if comp is not None and len(comp) else {}
        if len(l6):
            eb.append("S&P 500 vs non-S&P 500 (combined L6 model, walk-forward macro-F1): " + "; ".join(
                f"{bb} {l6.loc[bb, 'macro_f1']:.3f} [{l6.loc[bb, 'ci_low']:.3f}, {l6.loc[bb, 'ci_high']:.3f}]"
                + (f" ({int(nfirms[bb])} firms)" if bb in nfirms else "") for bb in ("SP500", "NonSP500", "Combined") if bb in l6.index) + ".")
    ht = facts.get("hypothesis_tests")
    if ht is not None and len(ht):
        eb.append("Drift verdicts: " + "; ".join(f"{r.test}: {r.verdict}" for r in ht.head(3).itertuples(index=False)))
    facts["exec_bullets"] = eb
    figures(ctx, F, facts, ev, u, usel, man, mm, led, tg, rl, fpy)
    facts["figures"] = F.write_index()
    return facts, facts["figures"]


def _bucket_facts(ctx, facts, cfg, prim, bt_all) -> None:
    """S&P 500 vs non-S&P 500 vs combined (owner request, D-056): every number comes from stage artifacts."""
    from .evaluation.buckets import LABELS, score_predictions
    nb, seed = int(cfg["bootstrap"]["n"]), int(cfg["seed"])
    comp = _opt(ctx, "S11", "bucket_composition.csv", "csv")
    facts["sp_composition"] = comp
    sc = _opt(ctx, "S11", "bucket_scores.csv", "csv")
    facts["sp_scores_wf"] = sc[[c for c in ("rung", "sp_bucket", "macro_f1", "ci_low", "ci_high", "n_events", "n_firms",
                                            "auc_macro_ovr", "logloss", "logloss_prior", "power") if c in sc]] if sc is not None else None
    facts["sp_pooled_vs_own"] = _opt(ctx, "S11", "bucket_pooled_vs_own.csv", "csv")
    bk = C.event_buckets(ctx)
    tag = f"|{prim['target_def']}|h{prim['horizon']}|"
    later = []
    for stage_id, name, block in (("S18", "dev_predictions.parquet", "development holdout"),
                                  ("S19", "locked_predictions.parquet", "locked test")):
        p = _opt(ctx, stage_id, name)
        if p is not None and len(p) and {"run_id", "y", "yhat", "date"} <= set(p.columns):
            p = p[p.run_id.astype(str).str.contains(tag, regex=False)]
            s = score_predictions(p, bk, nb, seed)
            if len(s):
                later.append(s.assign(block=block))
    facts["sp_scores_later"] = pd.concat(later, ignore_index=True)[["block", "run_id", "sp_bucket", "macro_f1", "ci_low", "ci_high",
                                                                    "n_events", "power"]] if later else None
    ta = _opt(ctx, "S20", "track_a_groups.csv", "csv")
    if ta is not None and "group_by" in ta:
        x = ta[(ta.group_by == "sp_bucket") & ta.measure.isin(["car_m1_1", "car_0_1", "car_2_60", "car_m20_m2", "abn_log_volume_0_1"])]
        facts["sp_track_a"] = x[[c for c in ("group", "event_set", "measure", "mean", "t", "p_adj_bh", "n") if c in x]]
    ht = _opt(ctx, "S21", "hypothesis_tests.csv", "csv")
    if ht is not None and "fdr_family" in ht:
        facts["sp_drift"] = ht[ht.test.str.startswith("SUE-conditioned")][[c for c in ("test", "estimate", "ci_low", "ci_high", "p_value",
                                                                                     "p_adj_bh", "n", "verdict") if c in ht]]
    if bt_all is not None and "sp_bucket" in bt_all:
        b = bt_all[(bt_all.lag == 0) & (bt_all.cost_bps == min(cfg["backtest"]["costs_bps"]))
                   & bt_all.leg.isin(["long_only", "long_minus_spy_matched", "long_short_hypothetical"])]
        facts["sp_backtest"] = b[[c for c in ("block", "sp_bucket", "leg", "ann_return", "sharpe", "deflated_sharpe_prob", "max_drawdown",
                                              "avg_names_long", "avg_names_short", "share_days_short_in_cash", "n_days") if c in b]]
    # computed-only reading of the decomposition
    txt = ["Buckets: " + "; ".join(f"**{k}** = {v}" for k, v in LABELS.items()) + ". The bucket is a firm attribute from the pinned "
           "fja05680 list; models are trained on the combined data unless stated, and every bucket uses the same folds."]
    if comp is not None and len(comp):
        c = comp.set_index("sp_bucket")
        for bb in ("SP500", "NonSP500"):
            if bb in c.index:
                r = c.loc[bb]
                txt.append(f"{LABELS[bb]}: {int(r.firms)} firms, {int(r.events):,} events ({r.events_per_firm:.0f} per firm), "
                           f"8-K share {r.share_8k:.0%}"
                           + (f", median log market cap {r.median_log_mcap:.2f}" if "median_log_mcap" in r and pd.notna(r.median_log_mcap) else "")
                           + (f", SUE available for {r.sue_available_share:.0%} of events" if "sue_available_share" in r and pd.notna(r.sue_available_share) else "")
                           + ".")
    if sc is not None and len(sc):
        l6 = sc[sc.rung == "L6"].set_index("sp_bucket")
        if len(l6):
            txt.append("Combined L6 model, walk-forward macro-F1 by bucket: " + "; ".join(
                f"{bb} {l6.loc[bb, 'macro_f1']:.3f} [{l6.loc[bb, 'ci_low']:.3f}, {l6.loc[bb, 'ci_high']:.3f}] (n={int(l6.loc[bb, 'n_events'])})"
                for bb in ("Combined", "SP500", "NonSP500") if bb in l6.index) + ". Bucket CIs that overlap do not show a difference.")
    pv = facts.get("sp_pooled_vs_own")
    if pv is not None and len(pv) and "pooled_minus_bucket_macro_f1" in pv:
        for r in pv[pv.pooled_minus_bucket_macro_f1.notna()].itertuples(index=False):
            txt.append(f"{r.rung}, {r.sp_bucket} rows: pooled model minus bucket-only model = {r.pooled_minus_bucket_macro_f1:+.3f} "
                       f"[{r.ci_low:+.3f}, {r.ci_high:+.3f}], bootstrap p {r.p_boot:.2f} (n={int(r.n_events)}): "
                       + ("pooling helps" if r.ci_low > 0 else "pooling hurts" if r.ci_high < 0 else "no detectable difference") + ".")
    facts["sp_text"] = " ".join(txt)


def _filter_main(df):
    if df is None or not len(df):
        return None
    d = df[(df.metric.isin(["macro_f1", "auc_macro_ovr", "logloss", "ece", "accuracy"])) & (df.event_set == "all")]
    return d.pivot_table(index=["run_id", "leaderboard", "n_events"], columns="metric", values="value").reset_index()


def figures(ctx, F, facts, ev, u, usel, man, mm, led, tg, rl, fpy):
    cfg = ctx.cfg
    prim = cfg["targets"]["primary"]
    # F01-F06 data and coverage
    piv = mm[mm.download_task].assign(parsed=lambda d: d.n_sections.fillna(0) > 0).groupby(["year", "base_form"]).parsed.mean().unstack()
    F.safe("F01", "Coverage heatmap (parsed share)", lambda i, t: F.heat(i, t, piv.T, vmin=0, vmax=1))
    cfy = _csv(ctx.dep("S04") / "coverage_firm_year.csv")
    F.safe("F02", "Audit reason codes over time", lambda i, t: F.stacked(i, t, cfy.groupby(["year", "reason_code"]).size().unstack(fill_value=0)))
    k8 = ev[ev.event_type == "8K"].explode("categories")
    F.safe("F03", "8-K category availability by year", lambda i, t: F.heat(i, t, k8.groupby(["categories", "year"]).size().unstack(fill_value=0),
                                                                             fmt="{:.0f}", cmap="magma"))
    F.safe("F04", "Universe: unique firms vs firms per year", lambda i, t: F.lines(i, t, {
        "unique firms": (fpy.year, fpy.unique_firms), "firms with prices": (fpy.year, fpy.firms_with_prices),
        "firms in S&P 500": (fpy.year, fpy.firms_in_sp500)}, "year", "firms"))
    F.safe("F05", "Timestamp quality and session buckets by year", lambda i, t: _f05(F, i, t, ev))
    F.safe("F06", "Pre-publication window and lag sensitivity", lambda i, t: _f06(F, i, t, ev, tg))
    F.safe("F07", "Event-clock diagram", lambda i, t: F.f07_clock(i, t))
    F.safe("F08", "Earnings types over time", lambda i, t: F.stacked(i, t, ev.groupby(["year", ev.earnings_type.fillna("none")]).size().unstack(fill_value=0)))
    oc = {h: float(ev[f"overlap_flag_h{h}"].mean()) for h in cfg["targets"]["horizons"] if f"overlap_flag_h{h}" in ev}
    F.safe("F09", "Overlap counts and clean share", lambda i, t: F.bars_ci(i, t, pd.DataFrame({"label": [f"h={h}" for h in oc],
                                                                                              "value": [1 - v for v in oc.values()],
                                                                                              "ci_low": [1 - v for v in oc.values()],
                                                                                              "ci_high": [1 - v for v in oc.values()]})))
    F.safe("F10", "Target distributions and class balance", lambda i, t: _f10(F, i, t, tg))
    # F11-F16 Track A
    cp = _opt(ctx, "S20", "car_paths.csv", "csv")
    F.safe("F11", "CAR paths by event type (all vs clean)", lambda i, t: F.lines(i, t, {f"{g} ({es})": (x.rel_day, x.caar * 100)
                                                                                      for (g, es), x in cp.groupby(["group", "event_set"])},
                                                                              "session relative to day 0", "CAAR (%)") if cp is not None and len(cp)
           else F.skip(i, t, "no CAR paths"))
    ta = _opt(ctx, "S20", "track_a_groups.csv", "csv")
    if ta is not None and len(ta):
        b = ta[(ta.group_by == "session_bucket") & (ta.measure == "car_0_1") & (ta.event_set == "all")]
        F.safe("F12", "CAR[0,+1] by session bucket", lambda i, t: F.bars_ci(i, t, b.assign(label=b.group, value=b["mean"] * 100,
                                                                                            ci_low=(b["mean"] - 1.96 * b.se) * 100,
                                                                                            ci_high=(b["mean"] + 1.96 * b.se) * 100), ref=0))
        pt = ta[(ta.group_by == "event_type") & ta.measure.isin(["car_m20_m2", "car_m5_m1"]) & (ta.event_set == "all")]
        F.safe("F13", "Pre-trend diagnostics", lambda i, t: F.bars_ci(i, t, pt.assign(label=pt.group, value=pt["mean"] * 100,
                                                                                       ci_low=(pt["mean"] - 1.96 * pt.se) * 100,
                                                                                       ci_high=(pt["mean"] + 1.96 * pt.se) * 100), hue="measure", ref=0))
        av = ta[(ta.group_by == "event_type") & ta.measure.isin(["abn_log_volume_0_1", "vol_response_0_1"]) & (ta.event_set == "all")]
        F.safe("F14", "Abnormal volume and volatility response", lambda i, t: F.bars_ci(i, t, av.assign(label=av.group, value=av["mean"],
                                                                                                         ci_low=av["mean"] - 1.96 * av.se,
                                                                                                         ci_high=av["mean"] + 1.96 * av.se), hue="measure"))
    else:
        for i in ("F12", "F13", "F14"):
            F.skip(i, "", "Track A not run")
    corr = _opt(ctx, "S20", "correlations.csv", "csv")
    if corr is not None and len(corr) and "spearman" in corr:
        cm = corr.dropna(subset=["spearman"]).pivot_table(index="feature", columns="car", values="spearman")
        F.safe("F15", "Correlation heatmap", lambda i, t: F.heat(i, t, cm, cmap="coolwarm", vmin=-0.2, vmax=0.2))
    else:
        F.skip("F15", "Correlation heatmap", "no correlations")
    F.safe("F16", "SUE-drift quintile CAR[+2,+60]", lambda i, t: _f16(ctx, F, i, t))
    # F17-F31 Track B
    bl = _csv(ctx.dep("S10") / "blocks.csv", parse_dates=["start", "end_exclusive"])
    prov = _csv(ctx.dep("S10") / "fold_provenance.csv")
    F.safe("F17", "Split timeline", lambda i, t: F.f17_timeline(i, t, bl, prov, cfg["study"]["start"], float(cfg["splits"]["embargo_days"])))
    lad = facts["ladder"]
    if lad is not None and len(lad):
        L = lad[lad.tuned.isin(["tuned", "default"]) & lad.cohort.isin(["common", "common_emb", "max_available"])].copy()
        L = L.sort_values("pooled_macro_f1", ascending=False).drop_duplicates(["rung", "cohort", "encoder"])
        L["label"] = L.rung + np.where(L.encoder.astype(str) != "n/a", "/" + L.encoder.astype(str), "") + " [" + L.leaderboard.str[0].str.upper() + "]"
        F.safe("F18", "Ladder lift with CIs", lambda i, t: F.bars_ci(i, t, L.rename(columns={"pooled_macro_f1": "value"}), hue="cohort",
                                                                       ref=float(L[L.rung == "L0"].value.iloc[0]) if (L.rung == "L0").any() and "value" in L else None))
    else:
        F.skip("F18", "Ladder lift with CIs", "no ladder results")
    h4 = _opt(ctx, "S15", "h4.csv", "csv")
    F.safe("F19", "Vintage vs static (H4)", lambda i, t: F.bars_ci(i, t, h4.assign(label=h4.rung + "/" + h4["head"] + " vs " + h4.static,
                                                                                  value=h4.diff_vintage_minus_static), ref=0)
           if h4 is not None and len(h4) else F.skip(i, t, "ChronoBERT or embeddings not run"))
    curves = json.loads((ctx.dep("S11") / "curves.json").read_text())
    c15 = _opt(ctx, "S15", "curves.json", "json") or []
    ft = _opt(ctx, "S15", "ft_curves.json", "json") or []
    F.safe("F20", "Loss curves", lambda i, t: F.f20_loss(i, t, (curves + c15)[:6], ft))
    tri = _opt(ctx, "S11", "optuna_trials.csv", "csv")
    F.safe("F21", "Validation curves", lambda i, t: _f21(F, i, t, tri) if tri is not None and len(tri) else F.skip(i, t, "no trials"))
    lc = _opt(ctx, "S11", "learning_curves.csv", "csv")
    F.safe("F22", "Learning curves", lambda i, t: F.lines(i, t, {"validation": (lc.train_fraction, lc.val_macro_f1),
                                                                 "train": (lc.train_fraction, lc.train_macro_f1)}, "training fraction", "macro-F1")
           if lc is not None and len(lc) else F.skip(i, t, "no learning curve"))
    tba = facts["tuning_before_after"]
    F.safe("F23", "Before / after tuning", lambda i, t: _f23(F, i, t, tba))
    preds = pd.concat([x for x in (_opt(ctx, "S11", "predictions.parquet"), _opt(ctx, "S15", "predictions.parquet")) if x is not None],
                      ignore_index=True)
    pp = preds[(preds.target_def == prim["target_def"]) & (preds.horizon == prim["horizon"]) & preds.yhat.notna()
               & ~preds.run_id.astype(str).str.startswith("dev|")] if len(preds) else preds
    cms, best_run = {}, None
    if len(pp) and lad is not None and len(lad):
        from .evaluation.metrics import confusion
        for rung in ("L0", "L1", "L6", "L7", "L8_ft"):
            cand = lad[(lad.rung == rung) & lad.cohort.str.startswith("common")].sort_values("mean_fold_score", ascending=False)
            if len(cand):
                rid_rows = rl[(rl.rung == rung) & (rl.metric == "macro_f1") & (rl.block == "walk_forward_pooled")
                              & (rl.target_def == prim["target_def"]) & (rl.horizon == prim["horizon"])
                              & rl.cohort.astype(str).str.startswith("common") & (rl.event_set == "all")]
                rid_rows = rid_rows.sort_values("mean_fold_score", ascending=False)
                if len(rid_rows):
                    rid = rid_rows.run_id.iloc[0]
                    g = pp[pp.run_id == rid]
                    if len(g):
                        cms[f"{rung}"] = confusion(g.y.values.astype(int), g.yhat.values.astype(int))
                        if rung in ("L6", "L7", "L8_ft"):
                            best_run = g
    F.safe("F24", "Confusion matrices", lambda i, t: F.f24_cm(i, t, cms))
    if best_run is not None and len(best_run):
        y = best_run.y.values.astype(int)
        P = best_run[["p_down", "p_side", "p_up"]].values
        from .evaluation.metrics import ece_top
        F.safe("F25", "ROC / PR curves", lambda i, t: F.f25_roc(i, t, y, P))
        F.safe("F26", "Calibration", lambda i, t: F.f26_calib(i, t, y, P, ece_top(P, y)))
        F.safe("F27", "Coverage-accuracy curve", lambda i, t: F.f27_coverage(i, t, y, P, best_run.yhat.values.astype(int)))
        F.safe("F31", "Performance by slice", lambda i, t: _f31(F, i, t, best_run, ctx))
    else:
        for i in ("F25", "F26", "F27", "F31"):
            F.skip(i, "", "no primary predictions")
    reg = preds[preds.target_def.astype(str).str.contains(":ar_mm") & preds.yhat_reg.notna()] if len(preds) and "yhat_reg" in preds else pd.DataFrame()
    F.safe("F28", "Rolling IC", lambda i, t: _f28(F, i, t, reg) if len(reg) else F.skip(i, t, "no regression predictions"))
    ti = facts.get("tinfo_vs_ttrade")
    if ti is not None and len(ti):
        ti = ti.sort_values("value", ascending=False).drop_duplicates(["rung", "model_id", "target_def"])
    F.safe("F29", "T_info vs T_trade", lambda i, t: F.bars_ci(i, t, ti.assign(label=ti.rung + "/" + ti.model_id, ci_low=ti.value, ci_high=ti.value),
                                                                hue="target_def") if ti is not None and len(ti) else F.skip(i, t, "no rows"))
    bd = _opt(ctx, "S22", "backtest_daily.parquet")
    F.safe("F30", "Backtest", lambda i, t: _f30(F, i, t, bd) if bd is not None and len(bd) else F.skip(i, t, "backtest not run"))
    rt = _opt(ctx, "S16", "rag_tournament.csv", "csv")
    if rt is not None and "recall" in rt:
        F.safe("F32", "Retrieval metrics by setup", lambda i, t: F.bars_ci(i, t, rt.assign(label=rt.setup, value=rt.recall, ci_low=rt.recall,
                                                                                           ci_high=rt.recall)))
    else:
        F.skip("F32", "Retrieval metrics by setup", "not evaluable without gold-val qrels")
    cv = _opt(ctx, "S23", "content_validity.csv", "csv")
    if cv is not None and "f1" in cv:
        F.safe("F33", "Extraction F1 (gold-test)", lambda i, t: F.bars_ci(i, t, cv.assign(label=cv.field, value=cv.f1, ci_low=cv.f1, ci_high=cv.f1),
                                                                           hue="source"))
    else:
        F.skip("F33", "Extraction F1 / faithfulness (gold-test)", "gold-test not evaluated")
    for i, t in (("F34", "Numeric fidelity"), ("F35", "Judge vs human"), ("F36", "Student vs teacher vs human")):
        F.skip(i, t, "needs gold labels (Track C)")
    perm = _opt(ctx, "S11", "permutation.csv", "csv")
    F.safe("F37", "Permutation null", lambda i, t: _f37(F, i, t, perm) if perm is not None and len(perm) else F.skip(i, t, "not run"))
    rd = _opt(ctx, "S11", "random_demo.json", "json")
    F.safe("F38", "Random-split demo", lambda i, t: F.bars_ci(i, t, pd.DataFrame({"label": ["random split (demo)", "walk-forward"],
                                                                                  "value": [rd["random_split_macro_f1"], rd["walk_forward_macro_f1"]],
                                                                                  "ci_low": [rd["random_split_macro_f1"], rd["walk_forward_macro_f1"]],
                                                                                  "ci_high": [rd["random_split_macro_f1"], rd["walk_forward_macro_f1"]]}))
           if rd else F.skip(i, t, "not run"))
    adv = _opt(ctx, "S11", "adversarial.csv", "csv")
    F.safe("F39", "Adversarial-validation ROC", lambda i, t: F.lines(i, t, {f"fold {r.fold} AUC {r.auc:.2f}": (json.loads(r.fpr), json.loads(r.tpr))
                                                                            for r in adv.itertuples(index=False)}, "FPR", "TPR")
           if adv is not None and len(adv) else F.skip(i, t, "not run"))
    mp = _opt(ctx, "S18", "mcs_pbo.json", "json")
    F.safe("F40", "MCS / PBO diagnostics", lambda i, t: _f40(F, i, t, mp) if mp else F.skip(i, t, "S18 not run"))
    F.safe("FX1", "Optimisation diagnostics", lambda i, t: _fx1(ctx, F, i, t, tri))
    sc = facts.get("sp_scores_wf")
    F.safe("FX2", "Macro-F1 by index bucket", lambda i, t: F.bars_ci(
        i, t, sc[sc.rung.isin(["L0", "L2", "L6"])].rename(columns={"rung": "label", "macro_f1": "value"}), hue="sp_bucket")
        if sc is not None and len(sc) and "rung" in sc else F.skip(i, t, "bucket analysis not run"))
    cpb = _opt(ctx, "S20", "car_paths_sp_bucket.csv", "csv")
    F.safe("FX3", "CAR paths by index bucket", lambda i, t: F.lines(i, t, {g: (x.rel_day, x.caar * 100) for g, x in cpb.groupby("group")},
                                                                     "session relative to day 0", "CAAR (%)")
           if cpb is not None and len(cpb) else F.skip(i, t, "not run"))


def _f05(F, i, t, ev):
    fig, (a, b) = F.plt.subplots(1, 2, figsize=(12, 4.2))
    for ax, col in ((a, "ts_quality"), (b, "session_bucket")):
        d = ev.groupby(["year", col]).size().unstack(fill_value=0)
        bottom = np.zeros(len(d))
        for c in d.columns:
            ax.bar(d.index.astype(str), d[c], bottom=bottom, label=c)
            bottom += d[c].values
        ax.tick_params(axis="x", rotation=90)
        ax.legend(fontsize=7)
        ax.set_title(col, fontsize=8)
    F.save(fig, i, t)


def _f06(F, i, t, ev, tg):
    fig, (a, b) = F.plt.subplots(1, 2, figsize=(12, 4.2))
    x = ev.contam_minutes.dropna() / 60
    a.hist(x.clip(upper=96), bins=60)
    a.set_xlabel("hours from last close to acceptance (clipped at 96)")
    rows = []
    base = tg[(tg.target_def == "T_info") & (tg.horizon == 1)].set_index("event_id").exit_idx
    for lag in (0, 15, 60):
        s = tg[(tg.target_def == f"T_info@pub{lag}") & (tg.horizon == 1)].set_index("event_id").exit_idx
        j = base.index.intersection(s.index)
        rows.append((f"lag {lag} min", float((base.loc[j] != s.loc[j]).mean()) if len(j) else np.nan))
    b.bar([r[0] for r in rows], [r[1] for r in rows])
    b.set_ylabel("share of events whose T_info exit changes")
    F.save(fig, i, t)


def _f10(F, i, t, tg):
    fig, (a, b) = F.plt.subplots(1, 2, figsize=(12, 4.2))
    for td in ("T_trade", "T_info"):
        x = tg[(tg.target_def == td) & (tg.horizon == 1)].ar_mm.dropna()
        a.hist(x.clip(-0.2, 0.2), bins=80, alpha=0.5, label=td)
    a.legend(fontsize=7)
    a.set_xlabel("abnormal return, h=1 (clipped +-20%)")
    cb = tg[tg.target_def.isin(["T_trade", "T_info"])].groupby(["target_def", "horizon"]).label3.value_counts(normalize=True).unstack()
    cb.plot(kind="bar", stacked=True, ax=b)
    b.set_ylabel("class share (0=down, 1=side, 2=up)")
    F.save(fig, i, t)


def _f16(ctx, F, i, t):
    from .track_a.drift import prior_quarter_quintiles
    d = C.read(ctx, "S20", "track_a_cars.parquet")
    Fe = C.read(ctx, "S09", "features.parquet")
    r = d[d.earnings_type == "earnings_announcement_verified"].merge(Fe[["event_id", "sue_event"]], on="event_id")
    if r.sue_event.notna().sum() < 50:
        return F.skip(i, t, "too few verified releases with SUE")
    r["q"] = prior_quarter_quintiles(r, "sue_event")
    g = r.dropna(subset=["q"]).groupby("q").car_2_60.agg(["mean", "sem", "size"]).reset_index()
    F.bars_ci(i, t, g.assign(label="Q" + g.q.astype(int).astype(str), value=g["mean"] * 100, ci_low=(g["mean"] - 1.96 * g["sem"]) * 100,
                             ci_high=(g["mean"] + 1.96 * g["sem"]) * 100), ref=0)


def _f21(F, i, t, tri):
    pcols = [c for c in tri.columns if c.startswith("params_")]
    k = min(6, len(pcols))
    if k == 0:
        return F.skip(i, t, "no tuned parameters")
    fig, axes = F.plt.subplots(1, k, figsize=(3.2 * k, 3.4), squeeze=False)
    for ax, c in zip(axes[0], pcols[:k]):
        d = tri[[c, "value", "head"]].dropna()
        for h, g in d.groupby("head"):
            ax.scatter(g[c], g.value, s=8, label=h)
        ax.set_xlabel(c.replace("params_", ""), fontsize=7)
        ax.set_ylabel("walk-forward macro-F1", fontsize=7)
    axes[0][0].legend(fontsize=6)
    F.save(fig, i, t)


def _f23(F, i, t, tba):
    d = tba.dropna(subset=["tuned_mean_f1"]).copy()
    d["label"] = d.rung + "/" + d["head"] + np.where(d.encoder.astype(str) != "-", "/" + d.encoder.astype(str), "")
    fig, ax = F.plt.subplots(figsize=(11, 4.5))
    x = np.arange(len(d))
    ax.bar(x - 0.2, d.default_mean_f1, 0.4, label="default")
    ax.bar(x + 0.2, d.tuned_mean_f1, 0.4, label="tuned")
    ax.set_xticks(x, d.label, rotation=90, fontsize=6)
    ax.legend(fontsize=7)
    ax.set_ylabel("mean walk-forward macro-F1")
    F.save(fig, i, t)


def _f28(F, i, t, reg):
    from scipy.stats import spearmanr
    g = reg[reg.rung == reg.rung.iloc[-1]]
    g = g.assign(m=pd.to_datetime(g.date).dt.to_period("M"))
    ic = g.groupby("m").apply(lambda x: spearmanr(x.y, x.yhat_reg)[0] if len(x) > 10 else np.nan, include_groups=False).dropna()
    roll = ic.rolling(12, min_periods=6).mean()
    F.lines(i, t, {"monthly rank IC": (ic.index.to_timestamp(), ic.values), "12-month mean": (roll.index.to_timestamp(), roll.values)},
            "month", "rank IC")


def _f30(F, i, t, bd):
    fig, ax = F.plt.subplots(figsize=(11, 4.6))
    for (blk, lag, cost), g in bd.groupby(["block", "lag", "cost_bps"]):
        ax.plot(pd.to_datetime(g.date), np.cumprod(1 + g.long_ret.fillna(0)) - 1, lw=1, label=f"{blk} long lag{lag} {cost}bps")
        ax.plot(pd.to_datetime(g.date), np.cumprod(1 + g.ls_ret.fillna(0)) - 1, lw=1, ls="--", label=f"{blk} L/S lag{lag} {cost}bps")
        ax.plot(pd.to_datetime(g.date), np.cumprod(1 + g.spy_matched.fillna(0)) - 1, lw=0.8, ls=":", color="k")
    ax.set_ylabel("cumulative return")
    ax.legend(fontsize=6, ncol=2)
    F.save(fig, i, t)


def _f31(F, i, t, best, ctx):
    from .evaluation.metrics import confusion, f1_from_cm
    ev = C.events(ctx).set_index("event_id")
    b = best.join(ev[["event_type", "session_bucket", "ts_quality", "ff12_industry", "year"]], on="event_id")
    fig, axes = F.plt.subplots(1, 4, figsize=(14, 3.8))
    for ax, col in zip(axes, ("event_type", "session_bucket", "ff12_industry", "year")):
        s = b.groupby(col).apply(lambda g: f1_from_cm(confusion(g.y.values.astype(int), g.yhat.values.astype(int)))["macro_f1"],
                                 include_groups=False)
        ax.bar(s.index.astype(str), s.values)
        ax.tick_params(axis="x", rotation=90, labelsize=6)
        ax.set_title(col, fontsize=8)
    F.save(fig, i, t)


def _f37(F, i, t, perm):
    fig, ax = F.plt.subplots(figsize=(6, 4))
    null = perm[perm.kind == "permuted"].macro_f1.dropna()
    obs = perm[perm.kind == "observed"].macro_f1.iloc[0]
    ax.hist(null, bins=max(5, len(null) // 2), alpha=0.7, label="permuted labels")
    ax.axvline(obs, color="C3", lw=2, label=f"observed {obs:.3f}")
    ax.legend(fontsize=7)
    ax.set_xlabel("walk-forward macro-F1")
    F.save(fig, i, t)


def _f40(F, i, t, mp):
    fig, (a, b) = F.plt.subplots(1, 2, figsize=(11, 4))
    pv = (mp.get("mcs") or {}).get("pvalues") or {}
    if pv:
        a.barh(list(pv), list(pv.values()))
        a.axvline(0.10, color="k", ls="--", lw=0.8)
        a.set_xlabel("MCS p-value (in set if > 0.10)")
    else:
        a.text(0.1, 0.5, (mp.get("mcs") or {}).get("note", "MCS unavailable"), fontsize=8)
    pbo = (mp.get("pbo") or {}).get("pbo")
    b.bar(["PBO"], [pbo if pbo is not None else 0])
    b.set_ylim(0, 1)
    b.set_title(f"PBO = {pbo}" if pbo is not None else (mp.get("pbo") or {}).get("note", "n/a"), fontsize=8)
    F.save(fig, i, t)


def _fx1(ctx, F, i, t, tri):
    fig, (a, b) = F.plt.subplots(1, 2, figsize=(12, 4.2))
    if tri is not None and len(tri):
        for (rung, head), g in tri.groupby(["rung", "head"]):
            if rung not in ("L6", "L4", "L1"):
                continue
            v = g.sort_values("number").value.cummax()
            a.plot(np.arange(len(v)), v.values, label=f"{rung}/{head}")
        a.set_xlabel("trial")
        a.set_ylabel("best-so-far walk-forward macro-F1")
        a.legend(fontsize=7)
    cert = _opt(ctx, "S11", "enet_certificate.json", "json")
    if cert:
        b.bar([f"start {k}" for k in range(len(cert["objective_by_start"]))], cert["objective_by_start"])
        lo = min(cert["objective_by_start"])
        b.set_ylim(lo - max(1e-4, abs(lo) * 1e-3), lo + max(1e-4, abs(lo) * 1e-3))
        b.set_title(f"elastic-net objective by random start (spread {cert['spread']:.1e}; "
                    f"{'global optimum certified' if cert['certified_global_min'] else 'not certified'})", fontsize=8)
    F.save(fig, i, t)


In [ ]:
%%writefile $REPO/src/secprice/storage.py
"""Artifact storage: DATA_ROOT/<profile>/<stage>/<stage_hash>/, manifests, _SUCCESS markers, atomic writes
(SPEC.md §16, §17.3)."""
from __future__ import annotations

import hashlib
import json
import os
import shutil
import subprocess
import tempfile
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd

from . import __version__
from .config import canonical_json, config_hash, file_sha256, get

_CODE_HASH: str | None = None


def utcnow() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def git_commit() -> str:
    """Git commit of the code that produced an artifact, with a '-dirty' suffix when the package, configs or tests have
    uncommitted changes. In notebook-embedded mode (no .git) the notebook's build stamp (the commit of the embedded code,
    itself '-dirty' if the build had uncommitted code) plus the source hash stand in."""
    env = os.environ.get("SECPRICE_BUILD_COMMIT")
    here = Path(__file__).resolve().parent
    try:
        out = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, timeout=10, cwd=here)
        if out.returncode == 0 and out.stdout.strip():
            st = subprocess.run(["git", "status", "--porcelain", "--", "src", "configs", "tests"], capture_output=True,
                                text=True, timeout=10, cwd=here.parents[1])
            dirty = st.returncode == 0 and bool(st.stdout.strip())
            return out.stdout.strip() + ("-dirty" if dirty else "")
    except Exception:
        pass
    return f"embedded:{env or 'unknown'}:src-{code_hash()}"


def code_hash() -> str:
    """Hash of the package source with line endings normalised (identical code hashes identically on Windows and Colab)."""
    global _CODE_HASH
    if _CODE_HASH is None:
        h = hashlib.sha256()
        base = Path(__file__).resolve().parent
        for p in sorted(base.rglob("*.py")):
            h.update(p.relative_to(base).as_posix().encode())
            h.update(p.read_bytes().replace(b"\r\n", b"\n"))
        _CODE_HASH = h.hexdigest()[:12]
    return _CODE_HASH


def atomic_write_bytes(path: Path, data: bytes) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(dir=path.parent, prefix=".tmp_", suffix=path.suffix)
    try:
        with os.fdopen(fd, "wb") as f:
            f.write(data)
        os.replace(tmp, path)
    finally:
        if os.path.exists(tmp):
            os.remove(tmp)


def atomic_write_text(path: Path, text: str) -> None:
    atomic_write_bytes(path, text.encode("utf-8"))


def write_json(path: Path, obj: Any) -> None:
    atomic_write_text(Path(path), json.dumps(obj, indent=2, default=str, sort_keys=False))


def read_json(path: Path, default: Any = None) -> Any:
    p = Path(path)
    if not p.exists():
        return default
    with open(p, encoding="utf-8") as f:
        return json.load(f)


def _to_parquet_safe(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in out.columns:
        if out[c].dtype == object:
            sample = out[c].dropna().head(50)
            if len(sample) and sample.map(lambda v: isinstance(v, (list, dict, tuple, set))).any():
                out[c] = out[c].map(lambda v: None if v is None else json.dumps(v, default=str)
                                    if isinstance(v, (list, dict, tuple, set)) else v)
    return out


class Storage:
    def __init__(self, cfg: dict, data_root: str | os.PathLike | None = None, hot_root: str | os.PathLike | None = None):
        self.cfg = cfg
        dr = data_root or os.environ.get(get(cfg, "paths.data_root_env", "DATA_ROOT"))
        if not dr:
            dr = Path.cwd() / "data"
        self.data_root = Path(dr)
        hr = hot_root or os.environ.get(get(cfg, "paths.hot_root_env", "HOT_ROOT")) or (self.data_root / "_hot")
        self.hot_root = Path(hr)
        self.profile = cfg["run_profile"]
        self.profile_root = self.data_root / self.profile
        self.inputs = self.data_root / get(cfg, "paths.inputs_dirname", "inputs")
        for p in (self.profile_root, self.hot_root, self.inputs, self.profile_root / "logs",
                  self.profile_root / "manifests", self.profile_root / "reports"):
            p.mkdir(parents=True, exist_ok=True)
        self.global_config_hash = config_hash(cfg)

    # ---------------- paths
    def stage_dir(self, stage_id: str, stage_hash: str) -> Path:
        p = self.profile_root / stage_id / stage_hash
        p.mkdir(parents=True, exist_ok=True)
        return p

    @property
    def reports(self) -> Path:
        return self.profile_root / "reports"

    @property
    def manifests(self) -> Path:
        return self.profile_root / "manifests"

    @property
    def figures(self) -> Path:
        p = self.profile_root / "reports" / "figures"
        p.mkdir(parents=True, exist_ok=True)
        return p

    # ---------------- artifacts
    def manifest_payload(self, stage_id: str, stage_hash: str, rows: int | None, started: str,
                         inputs: dict | None = None, warnings: list | None = None, extra: dict | None = None) -> dict:
        return {
            "stage": stage_id, "stage_hash": stage_hash, "config_hash": self.global_config_hash,
            "git_commit": git_commit(), "code_hash": code_hash(), "package_version": __version__,
            "input_artifact_hashes": inputs or {}, "row_count": rows, "start_time": started,
            "end_time": utcnow(), "warnings": warnings or [], **(extra or {}),
        }

    def write_parquet(self, df: pd.DataFrame, path: Path, stage_id: str, stage_hash: str, started: str,
                      inputs: dict | None = None, warnings: list | None = None) -> Path:
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        fd, tmp = tempfile.mkstemp(dir=path.parent, prefix=".tmp_", suffix=".parquet")
        os.close(fd)
        try:
            _to_parquet_safe(df).to_parquet(tmp, index=False, compression="zstd")
            os.replace(tmp, path)
        finally:
            if os.path.exists(tmp):
                os.remove(tmp)
        man = self.manifest_payload(stage_id, stage_hash, len(df), started, inputs, warnings)
        man["sha256"] = file_sha256(path)
        write_json(path.with_suffix(path.suffix + ".manifest.json"), man)
        return path

    @staticmethod
    def read_parquet(path: Path, columns: list[str] | None = None) -> pd.DataFrame:
        return pd.read_parquet(path, columns=columns)

    def success_path(self, stage_id: str, stage_hash: str) -> Path:
        return self.profile_root / stage_id / stage_hash / "_SUCCESS"

    def mark_success(self, stage_id: str, stage_hash: str, info: dict) -> None:
        d = self.stage_dir(stage_id, stage_hash)
        outputs = {}
        for p in sorted(d.rglob("*")):
            if p.is_file() and p.name != "_SUCCESS" and not p.name.endswith(".manifest.json") and p.stat().st_size < 4e9:
                outputs[p.relative_to(d).as_posix()] = p.stat().st_size
        write_json(d / "_SUCCESS", {"stage": stage_id, "stage_hash": stage_hash, "finished": utcnow(),
                                    "config_hash": self.global_config_hash, "outputs": outputs, **info})

    def is_success(self, stage_id: str, stage_hash: str) -> bool:
        return self.success_path(stage_id, stage_hash).exists()

    def other_successes(self, stage_id: str, stage_hash: str) -> list[str]:
        base = self.profile_root / stage_id
        if not base.exists():
            return []
        return [p.name for p in base.iterdir() if p.is_dir() and p.name != stage_hash and (p / "_SUCCESS").exists()]

    def clear_stage(self, stage_id: str, stage_hash: str) -> None:
        d = self.profile_root / stage_id / stage_hash
        if d.exists():
            shutil.rmtree(d)


def df_hash(df: pd.DataFrame) -> str:
    h = hashlib.sha256(pd.util.hash_pandas_object(df.reset_index(drop=True), index=False).values.tobytes())
    return h.hexdigest()[:16]


def json_hash(obj: Any) -> str:
    return hashlib.sha256(canonical_json(obj).encode()).hexdigest()[:16]


class Timer:
    def __init__(self):
        self.t0 = time.time()

    def elapsed(self) -> float:
        return time.time() - self.t0


In [ ]:
%%writefile $REPO/src/secprice/targets/__init__.py
"""secprice.targets"""


In [ ]:
%%writefile $REPO/src/secprice/targets/build.py
"""S08 targets and labels (SPEC.md §5.4-5.5; addendum P0.1, P0.2).

T_info  (Track A information response; complementary Track B target): base = last regular close <= acceptance;
        exit = close of the h-th session whose close is after public time; INTRA excluded; close->close total
        return; abnormal = CAR of daily market-model residuals (alpha, beta from sessions -250..-30 before base).
T_trade (primary Track B target): entry = open of the first session whose open is at or after decision_time
        (all buckets); exit = close of session entry + h - 1; executable open->close total return; abnormal =
        R_stock - beta x R_SPY on the identical interval (alpha omitted on partial-day intervals; decision log).
        No close-to-close factor return is ever subtracted from an open-to-close return: Carhart is null here.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..market.returns import v_cc_return, v_oc_return

NULL_CARHART_TTRADE = "carhart_not_interval_matched_for_open_entry"


class Estimator:
    """O(1) market-model OLS over any session window via prefix sums of valid (stock, market) return pairs."""

    def __init__(self, r: np.ndarray, rm: np.ndarray):
        ok = np.isfinite(r) & np.isfinite(rm)
        x, y = np.where(ok, rm, 0.0), np.where(ok, r, 0.0)
        self.c = [np.concatenate([[0.0], np.cumsum(a)]) for a in (ok.astype(float), x, y, x * y, x * x, y * y)]

    def window(self, lo: np.ndarray, hi: np.ndarray):
        """Sums over sessions [lo, hi] inclusive (clipped)."""
        n = len(self.c[0]) - 1
        lo_c, hi_c = np.clip(lo, 0, n), np.clip(hi + 1, 0, n)
        s = [c[hi_c] - c[lo_c] for c in self.c]
        cnt, sx, sy, sxy, sxx, syy = s
        with np.errstate(divide="ignore", invalid="ignore"):
            den = cnt * sxx - sx * sx
            beta = (cnt * sxy - sx * sy) / den
            alpha = (sy - beta * sx) / cnt
            ssr = syy - 2 * alpha * sy - 2 * beta * sxy + cnt * alpha ** 2 + 2 * alpha * beta * sx + beta ** 2 * sxx
            sigma = np.sqrt(np.maximum(ssr, 0) / (cnt - 2))
        bad = (cnt < 3) | ~np.isfinite(beta)
        beta[bad], alpha[bad], sigma[bad] = np.nan, np.nan, np.nan
        return cnt, alpha, beta, sigma


def _carhart_car(r, f, rf, base, x, lo, hi, min_obs):
    """CAR of Carhart residuals over (base, x] with loadings estimated on [lo, hi]; f = [mkt, smb, hml, mom]."""
    out = np.full(len(base), np.nan)
    for i in range(len(base)):
        if lo[i] < 1 or x[i] >= len(r) or base[i] < 0:
            continue
        yy = r[lo[i]:hi[i] + 1] - rf[lo[i]:hi[i] + 1]
        XX = f[lo[i]:hi[i] + 1]
        ok = np.isfinite(yy) & np.isfinite(XX).all(1)
        if ok.sum() < min_obs:
            continue
        A = np.column_stack([np.ones(ok.sum()), XX[ok]])
        coef, *_ = np.linalg.lstsq(A, yy[ok], rcond=None)
        seg = slice(base[i] + 1, x[i] + 1)
        ex = r[seg] - rf[seg]
        Xs = f[seg]
        if not (np.isfinite(ex).all() and np.isfinite(Xs).all()):
            continue
        out[i] = float(np.sum(ex - coef[0] - Xs @ coef[1:]))
    return out


def label3(ar: np.ndarray, sigma: np.ndarray, h: int, k: float) -> np.ndarray:
    th = k * sigma * np.sqrt(h)
    lab = np.full(len(ar), np.nan)
    ok = np.isfinite(ar) & np.isfinite(th)
    lab[ok] = 1.0                                   # Sideways
    lab[ok & (ar > th)] = 2.0                       # Up
    lab[ok & (ar < -th)] = 0.0                      # Down
    return lab


def _vol_ratio(r: np.ndarray, d1: np.ndarray, post: tuple, pre: tuple) -> np.ndarray:
    out = np.full(len(d1), np.nan)
    n = len(r)
    for i, s in enumerate(d1):
        a0, a1 = s + post[0] - 1, s + post[1] - 1
        b0, b1 = s - 1 + pre[0], s - 1 + pre[1]
        if b0 < 0 or a1 >= n:
            continue
        pa, pb = r[a0:a1 + 1], r[b0:b1 + 1]
        pa, pb = pa[np.isfinite(pa)], pb[np.isfinite(pb)]
        if len(pa) >= 10 and len(pb) >= 20:
            sa, sb = np.std(pa, ddof=1), np.std(pb, ddof=1)
            if sa > 0 and sb > 0:
                out[i] = np.log(sa / sb)
    return out


def uniqueness_weights(start: np.ndarray, stop: np.ndarray) -> np.ndarray:
    """Average uniqueness of each event window within one firm (label-uniqueness weights)."""
    w = np.full(len(start), np.nan)
    ok = (start >= 0) & (stop >= start)
    if not ok.any():
        return w
    lo, hi = int(start[ok].min()), int(stop[ok].max())
    conc = np.zeros(hi - lo + 2)
    for a, b in zip(start[ok], stop[ok]):
        conc[a - lo] += 1
        conc[b - lo + 1] -= 1
    conc = np.cumsum(conc)[:-1]
    for i in np.where(ok)[0]:
        seg = conc[start[i] - lo: stop[i] - lo + 1]
        w[i] = float(np.mean(1.0 / seg)) if len(seg) else np.nan
    return w


def build_targets(ev: pd.DataFrame, panel, spy: dict, factors: pd.DataFrame | None, cfg: dict,
                  usable_cutoff_idx: int) -> pd.DataFrame:
    """ev needs: event_id, ticker, tinfo_base_idx, trackA_day0_idx, ttrade_entry_idx, tinfo_eligible,
    acceptance_time, public_available_time; plus sensitivity entry indexes (computed here from cal)."""
    hs = cfg["targets"]["horizons"]
    k = float(cfg["labels"]["k"])
    k_sens = [float(x) for x in cfg["labels"]["k_sensitivity"]]
    w0, w1 = cfg["targets"]["est_window"]
    min_obs = int(cfg["targets"]["est_min_obs"])
    post, pre = tuple(cfg["targets"]["vol_post"]), tuple(cfg["targets"]["vol_pre"])
    cal = panel.cal
    n = len(cal)
    rm = spy["ret"]
    fac = None
    if factors is not None and len(factors):
        fidx = cal.sessions.searchsorted(pd.to_datetime(factors["date"]).values)
        okf = (fidx < n) & (cal.sessions[np.clip(fidx, 0, n - 1)] == pd.to_datetime(factors["date"]).values)
        fac = np.full((n, 4), np.nan)
        rf = np.full(n, np.nan)
        fac[fidx[okf]] = factors.loc[okf, ["mkt_rf", "smb", "hml", "mom"]].to_numpy(float)
        rf[fidx[okf]] = factors.loc[okf, "rf"].to_numpy(float)
    out = []
    t = cfg["timing"]
    sens_h = [h for h in cfg["targets"].get("sensitivity_horizons", [1, 5]) if h in hs]
    for tk, g in ev.groupby("ticker", sort=False):
        if not panel.has(tk):
            continue
        P = panel[tk]
        est = Estimator(P["ret"], rm)
        defs = []
        base = g["tinfo_base_idx"].to_numpy(int)
        pub_first = g["trackA_day0_idx"].to_numpy(int)
        entry = g["ttrade_entry_idx"].to_numpy(int)
        elig = g["tinfo_eligible"].to_numpy(bool)
        for h in hs:
            defs.append(("T_info", h, base, pub_first + h - 1, elig, "cc"))
            defs.append(("T_trade", h, entry, entry + h - 1, np.ones(len(g), bool), "oc"))
        for h in sens_h:
            defs.append(("T_trade_lag1", h, entry + 1, entry + h, np.ones(len(g), bool), "oc"))
            for lat in t.get("decision_latency_sensitivity", []):
                dec = (pd.to_datetime(g["public_available_time"]) + pd.Timedelta(minutes=float(lat)))
                e2 = cal.v_first_open_at_or_after(pd.DatetimeIndex(dec).tz_convert("UTC").asi8)
                defs.append((f"T_trade@lat{lat}", h, e2, e2 + h - 1, np.ones(len(g), bool), "oc"))
            for lag in t.get("public_lag_sensitivity", []):
                pub = pd.to_datetime(g["acceptance_time"]) + pd.Timedelta(minutes=float(lag))
                d0 = cal.v_first_close_after(pd.DatetimeIndex(pub).tz_convert("UTC").asi8)
                defs.append((f"T_info@pub{lag}", h, base, d0 + h - 1, elig, "cc"))
        for name, h, b, x, eligible, kind in defs:
            b = np.asarray(b, dtype=int)
            x = np.asarray(x, dtype=int)
            anchor = b if kind == "cc" else b          # estimation window relative to the base / entry session
            cnt, alpha, beta, sigma = est.window(anchor + w0, anchor + w1)
            lowobs = cnt < min_obs
            alpha[lowobs], beta[lowobs], sigma[lowobs] = np.nan, np.nan, np.nan
            valid_idx = (b >= 0) & (x < n) & (x >= b)
            if kind == "cc":
                raw = v_cc_return(P["total_return_index"], b, x)
                bench = v_cc_return(spy["total_return_index"], b, x)
                # CAR of daily residuals over (b, x]
                fin_r, fin_m = np.isfinite(P["ret"]), np.isfinite(rm)
                cs_r = np.concatenate([[0.0], np.cumsum(np.where(fin_r, P["ret"], 0.0))])
                cs_m = np.concatenate([[0.0], np.cumsum(np.where(fin_m, rm, 0.0))])
                cs_bad = np.concatenate([[0], np.cumsum(~(fin_r & fin_m))])
                bb, xx = np.clip(b, 0, n - 1), np.clip(x, 0, n - 1)
                sum_r = cs_r[xx + 1] - cs_r[bb + 1]
                sum_m = cs_m[xx + 1] - cs_m[bb + 1]
                gaps = (cs_bad[xx + 1] - cs_bad[bb + 1]) > 0     # any missing day in (b, x] -> CAR undefined
                ar = sum_r - (alpha * (x - b) + beta * sum_m)
                ar[gaps] = np.nan
                ar_c = _carhart_car(P["ret"], fac, rf, b, x, anchor + w0, anchor + w1, min_obs) if fac is not None \
                    else np.full(len(b), np.nan)
                d1 = b + 1
                car_reason = None
            else:
                raw = v_oc_return(P["open"], P["close_split_adj"], P["total_return_index"], b, x)
                bench = v_oc_return(spy["open"], spy["close_split_adj"], spy["total_return_index"], b, x)
                ar = raw - beta * bench
                ar_c = np.full(len(b), np.nan)
                d1 = b
                car_reason = NULL_CARHART_TTRADE
            ar[~valid_idx] = np.nan
            raw[~valid_idx] = np.nan
            censored = (x > usable_cutoff_idx) | (x >= n)
            ok = eligible & ~censored
            for arr in (raw, ar, ar_c):
                arr[~ok] = np.nan
            vr = _vol_ratio(P["ret"], d1, post, pre)
            vr[~ok] = np.nan
            with np.errstate(divide="ignore", invalid="ignore"):
                abs_std = np.abs(ar) / (sigma * np.sqrt(h))
            rec = pd.DataFrame({
                "event_id": g["event_id"].to_numpy(), "ticker": tk, "target_def": name, "horizon": h,
                "start_idx": b, "exit_idx": x, "raw_ret": raw, "bench_ret": bench, "ar_mm": ar, "ar_carhart": ar_c,
                "alpha": alpha, "beta": beta, "sigma": sigma, "n_est": cnt, "label3": label3(ar, sigma, h, k),
                **{f"label3_k{str(kk).replace('.', '')}": label3(ar, sigma, h, kk) for kk in k_sens},
                "vol_ratio": vr, "abs_std": abs_std, "eligible": eligible, "censored": censored,
                "carhart_null_reason": car_reason,
            })
            rec["null_reason"] = np.select(
                [~eligible, censored, ~valid_idx, lowobs, ~np.isfinite(raw)],
                ["not_eligible_intra_or_C", "censored_after_usable_cutoff", "index_out_of_range",
                 f"est_obs_lt_{min_obs}", "missing_price"], default=None)
            rec["weight_uniq"] = uniqueness_weights(np.where(ok, b, -1), np.where(ok, x, -1))
            out.append(rec)
    res = pd.concat(out, ignore_index=True) if out else pd.DataFrame()
    if len(res):
        res["start_date"] = [cal.date(i) if 0 <= i < n else pd.NaT for i in res["start_idx"]]
        res["exit_date"] = [cal.date(i) if 0 <= i < n else pd.NaT for i in res["exit_idx"]]
    return res


In [ ]:
%%writefile $REPO/src/secprice/testing/__init__.py
"""Test helpers (synthetic fixtures)."""


In [ ]:
%%writefile $REPO/src/secprice/testing/fixtures.py
"""Deterministic synthetic world for the offline integration smoke (SPEC.md §18.2; addendum P1.5).

Three firms (AAA, BBB, CCC), 2019-06 to 2024-06: prices (with a split and dividends), SPY, factors, and SEC
responses in the real formats (submissions JSON with the double-shifted acceptance times observed on EDGAR, 10-K /
10-Q HTML primary documents, 8-K full submissions with SGML headers and EX-99 releases, XBRL company facts).
"""
from __future__ import annotations

import io
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

FIRMS = [("AAA", 1001, "AAA TECHNOLOGIES INC", "3571", 0.0004, 0.018),
         ("BBB", 1002, "BBB PHARMACEUTICAL CORP", "2834", 0.0002, 0.014),
         ("CCC", 1003, "CCC BANCORP", "6021", 0.0001, 0.016)]
POS = ["strong", "growth", "improved", "record", "gains", "achieved", "success", "favorable"]
NEG = ["loss", "decline", "adverse", "impairment", "litigation", "weak", "deterioration", "uncertain"]
UNC = ["uncertain", "may", "approximately", "possible", "risk", "unpredictable"]
ET = "America/New_York"


def _json_time(t_et: pd.Timestamp) -> str:
    """Encode an ET time the way the submissions JSON does (ET shifted by the ET->UTC offset twice, 'Z')."""
    off = -t_et.utcoffset().total_seconds()
    return (t_et.tz_localize(None) + pd.Timedelta(seconds=2 * off)).strftime("%Y-%m-%dT%H:%M:%S.000Z")


def _sentences(rng, tone: float, n: int, firm: str) -> str:
    out = []
    for _ in range(n):
        words = list(rng.choice(POS if rng.random() < 0.5 + tone / 2 else NEG, 2)) + list(rng.choice(UNC, 1))
        out.append(f"{firm} reported {words[0]} results with {words[1]} trends and {words[2]} conditions in the period "
                   f"across segment {int(rng.integers(1, 9))} operations.")
    return " ".join(out)


def make_world(root: Path, seed: int = 7) -> dict:
    import pandas_market_calendars as mcal
    rng = np.random.default_rng(seed)
    root = Path(root)
    inputs = root / "inputs"
    sec = root / "sec_fixtures"
    (inputs / "ohlcv_1day").mkdir(parents=True, exist_ok=True)
    (inputs / "lexicons").mkdir(parents=True, exist_ok=True)
    sch = mcal.get_calendar("NYSE").schedule(start_date="2019-06-01", end_date="2024-06-30")
    days = pd.DatetimeIndex(sch.index).tz_localize(None)
    n = len(days)
    mkt = rng.normal(0.0003, 0.01, n)
    # SPY + factors
    spy_close = 300 * np.exp(np.cumsum(mkt))
    spy_open = spy_close * np.exp(rng.normal(0, 0.002, n))
    yf_rows = []
    for d, o, c in zip(days, spy_open, spy_close):
        yf_rows.append(("SPY", d, o, max(o, c) * 1.003, min(o, c) * 0.997, c, 5e7, 0.0, 0.0))
    for k, d in enumerate(days):
        if d.month in (3, 6, 9, 12) and d.day >= 15 and (k == 0 or days[k - 1].month != d.month or days[k - 1].day < 15):
            yf_rows[k] = yf_rows[k][:7] + (1.5, 0.0)
    fac = pd.DataFrame({"date": days, "mkt_rf": mkt - 0.0001, "smb": rng.normal(0, 0.004, n), "hml": rng.normal(0, 0.004, n),
                        "rmw": rng.normal(0, 0.003, n), "cma": rng.normal(0, 0.003, n), "mom": rng.normal(0, 0.005, n),
                        "rf": np.full(n, 0.0001)})
    tickers_ex, hf_rows, sp_rows, facts_all = [], [], [], {}
    for tk, cik, name, sic, mu, vol in FIRMS:
        beta = 0.8 + 0.4 * rng.random()
        r = mu + beta * mkt + rng.normal(0, vol, n)
        close = 50 * np.exp(np.cumsum(r))
        opn = close * np.exp(rng.normal(0, 0.004, n))
        vol_sh = rng.integers(1_000_000, 5_000_000, n).astype(float)
        local = pd.DataFrame({"date": days, "open": opn, "high": np.maximum(opn, close) * 1.01, "low": np.minimum(opn, close) * 0.99,
                              "close": close, "volume": vol_sh, "minute_bars": 390, "source": "1min"})
        local[local.date >= "2020-01-02"].to_parquet(inputs / "ohlcv_1day" / f"{tk}.parquet", index=False)
        for d, o, c, v in zip(days, opn, close, vol_sh):
            if d < pd.Timestamp("2020-03-01"):
                yf_rows.append((tk, d, o, max(o, c) * 1.01, min(o, c) * 0.99, c, v, 0.0, 0.0))
        if tk == "AAA":
            yf_rows.append((tk, pd.Timestamp("2022-06-06"), np.nan, np.nan, np.nan, np.nan, np.nan, 0.0, 2.0))
        if tk == "BBB":
            for d in days[(days.month.isin([2, 5, 8, 11])) & (days.day <= 7)][::5]:
                yf_rows.append((tk, d, np.nan, np.nan, np.nan, np.nan, np.nan, 0.45, 0.0))
        tickers_ex.append([cik, name, tk, "NYSE"])
        if tk != "CCC":                               # CCC is never an S&P 500 member: both index buckets are exercised (D-056)
            sp_rows.append((tk, "2010-01-01", "" if tk != "BBB" else "2022-12-31"))   # BBB leaves the index (point-in-time flag)
        # ---------------- filings
        filings = []
        sub_docs = {}
        eps_hist = []
        eps_q = 1.0 + rng.normal(0, 0.1)
        for y in range(2018, 2024):
            for q, (m10q, mrel) in enumerate([(5, 4), (8, 7), (11, 10), (2, 1)]):
                fy = y if q < 3 else y + 1
                pe = pd.Timestamp(year=y, month=[3, 6, 9, 12][q], day=1) + pd.offsets.MonthEnd(0)
                eps_q = eps_q * (1 + rng.normal(0.01, 0.08))
                eps_hist.append((pe, round(eps_q, 2), q))
                rel_day = pd.Timestamp(year=fy, month=mrel, day=1) + pd.offsets.BDay(int(rng.integers(15, 20)))
                hour = [7, 16, 12][int(rng.integers(0, 3))]
                rel_t = (rel_day + pd.Timedelta(hours=hour, minutes=int(rng.integers(1, 55)), seconds=int(rng.integers(0, 59)))).tz_localize(ET)
                filings.append(("8-K", rel_t, "2.02,9.01", pe, {"eps": round(eps_q, 2), "pe": pe}))
                per_day = pd.Timestamp(year=fy, month=m10q, day=1) + pd.offsets.BDay(int(rng.integers(2, 8)))
                per_t = (per_day + pd.Timedelta(hours=int(rng.choice([6, 16, 17])), minutes=int(rng.integers(0, 59)))).tz_localize(ET)
                filings.append(("10-K" if q == 3 else "10-Q", per_t, "", pe, {"eps": round(eps_q, 2), "pe": pe, "q": q}))
            mg = (pd.Timestamp(year=y, month=9, day=10) + pd.offsets.BDay(int(rng.integers(0, 10)))) + pd.Timedelta(hours=8, minutes=30)
            filings.append(("8-K", mg.tz_localize(ET), "5.02", None, {}))
        filings.sort(key=lambda x: x[1])
        recent = {k: [] for k in ("accessionNumber", "filingDate", "reportDate", "acceptanceDateTime", "act", "form", "fileNumber",
                                  "filmNumber", "items", "core_type", "size", "isXBRL", "isInlineXBRL", "isXBRLNumeric",
                                  "primaryDocument", "primaryDocDescription")}
        cf_eps = []
        for j, (form, t, items, pe, info) in enumerate(filings):
            acc = f"{cik:010d}-{t.year % 100:02d}-{j:06d}"
            nd = acc.replace("-", "")
            fdate = (t.tz_localize(None) + (pd.Timedelta(days=1) if t.hour >= 22 else pd.Timedelta(0))).strftime("%Y-%m-%d")
            doc = f"{tk.lower()}-{nd}.htm"
            recent["accessionNumber"].append(acc)
            recent["filingDate"].append(fdate)
            recent["reportDate"].append(pe.strftime("%Y-%m-%d") if pe is not None else fdate)
            recent["acceptanceDateTime"].append(_json_time(t))
            recent["act"].append("34")
            recent["form"].append(form)
            recent["fileNumber"].append("001-0000")
            recent["filmNumber"].append("0")
            recent["items"].append(items)
            recent["core_type"].append(form)
            recent["size"].append(10000)
            recent["isXBRL"].append(1)
            recent["isInlineXBRL"].append(1)
            recent["isXBRLNumeric"].append(1)
            recent["primaryDocument"].append(doc)
            recent["primaryDocDescription"].append(form)
            base = sec / "www.sec.gov" / "Archives" / "edgar" / "data" / str(cik) / nd
            base.mkdir(parents=True, exist_ok=True)
            tone = float(rng.normal(0, 0.4))
            hdr = (f"<SEC-HEADER>\nACCESSION NUMBER: {acc}\nCONFORMED SUBMISSION TYPE: {form}\n<ACCEPTANCE-DATETIME>{t.strftime('%Y%m%d%H%M%S')}\n"
                   f"FILED AS OF DATE: {fdate.replace('-', '')}\n")
            if form == "8-K":
                titles = ["Results of Operations and Financial Condition", "Financial Statements and Exhibits"] if "2.02" in items else \
                    ["Departure of Directors or Certain Officers; Election of Directors"]
                hdr += "".join(f"ITEM INFORMATION: {x}\n" for x in titles) + "</SEC-HEADER>\n"
                body8k = (f"<html><body><p>UNITED STATES SECURITIES AND EXCHANGE COMMISSION</p><p>FORM 8-K</p>"
                          f"<p>{name}</p>" + "".join(f"<p>Item {it}. {tt}</p><p>{_sentences(rng, tone, 3, name)}</p>"
                                                     for it, tt in zip(items.split(","), titles)) +
                          "<p>SIGNATURES</p><p>Pursuant to the requirements of the Securities Exchange Act of 1934, the registrant has duly caused</p></body></html>")
                txt = hdr + f"<DOCUMENT>\n<TYPE>8-K\n<SEQUENCE>1\n<FILENAME>{doc}\n<TEXT>\n{body8k}\n</TEXT>\n</DOCUMENT>\n"
                if "2.02" in items:
                    e = info["eps"]
                    pe_s = info["pe"].strftime("%B %d, %Y").replace(" 0", " ")
                    ex = (f"<html><body><p>{name} Reports Results for the Quarter Ended {pe_s}</p><p>{_sentences(rng, tone, 4, name)}</p>"
                          f"<p>Net income for the three months ended {pe_s} increased. Revenue grew.</p>"
                          f"<table><tr><td>Earnings per share:</td></tr><tr><td>Diluted earnings per share</td><td>$</td><td>{e:.2f}</td>"
                          f"<td>$</td><td>{e * 0.9:.2f}</td></tr></table></body></html>")
                    txt += f"<DOCUMENT>\n<TYPE>EX-99.1\n<SEQUENCE>2\n<FILENAME>ex991.htm\n<TEXT>\n{ex}\n</TEXT>\n</DOCUMENT>\n"
                (base / f"{acc}.txt").write_text(txt, encoding="utf-8")
            else:
                hdr += "</SEC-HEADER>\n"
                if form == "10-K":
                    html = ("<html><body><p>FORM 10-K</p><p>TABLE OF CONTENTS</p><p>Item 1. Business</p><p>Item 1A. Risk Factors</p>"
                            "<p>Item 7. Management's Discussion and Analysis</p>"
                            f"<p>PART I</p><p>Item 1. Business</p><p>{_sentences(rng, tone, 25, name)}</p>"
                            f"<p>Item 1A. Risk Factors</p><p>{_sentences(rng, tone - 0.3, 30, name)}</p>"
                            "<p>Item 2. Properties</p><p>We own offices.</p>"
                            f"<p>PART II</p><p>Item 7. Management's Discussion and Analysis of Financial Condition and Results of Operations</p>"
                            f"<p>{_sentences(rng, tone, 40, name)}</p>"
                            f"<p>Item 7A. Quantitative and Qualitative Disclosures About Market Risk</p><p>{_sentences(rng, tone, 8, name)}</p>"
                            "<p>Item 8. Financial Statements and Supplementary Data</p><table><tr><td>Total assets</td><td>1,000</td></tr></table>"
                            "</body></html>")
                else:
                    html = ("<html><body><p>FORM 10-Q</p><p>PART I - FINANCIAL INFORMATION</p><p>Item 1. Financial Statements</p>"
                            "<table><tr><td>Revenue</td><td>100</td></tr></table>"
                            f"<p>Item 2. Management's Discussion and Analysis of Financial Condition and Results of Operations</p>"
                            f"<p>{_sentences(rng, tone, 25, name)}</p>"
                            f"<p>Item 3. Quantitative and Qualitative Disclosures About Market Risk</p><p>{_sentences(rng, tone, 5, name)}</p>"
                            "<p>Item 4. Controls and Procedures</p><p>Effective.</p><p>PART II - OTHER INFORMATION</p>"
                            f"<p>Item 1A. Risk Factors</p><p>{_sentences(rng, tone - 0.3, 6, name)}</p><p>Item 6. Exhibits</p></body></html>")
                (base / doc).write_text(html, encoding="utf-8")
                (base / f"{acc}.txt").write_text(hdr + f"<DOCUMENT>\n<TYPE>{form}\n<TEXT>\n{html}\n</TEXT>\n</DOCUMENT>\n", encoding="utf-8")
                (base / f"{acc}-index-headers.html").write_text(hdr, encoding="utf-8")
                # XBRL EPS facts reported in this filing (quarterly for 10-Q; FY + 9M for 10-K)
                qn = info["q"]
                pe = info["pe"]
                st = (pe - pd.offsets.MonthEnd(3) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
                if form == "10-Q":
                    cf_eps.append({"start": st, "end": pe.strftime("%Y-%m-%d"), "val": info["eps"], "accn": acc, "fy": pe.year,
                                   "fp": f"Q{qn + 1}", "form": form, "filed": fdate})
                else:
                    fy_start = (pe - pd.offsets.MonthEnd(12) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
                    prev3 = [e for (p_, e, q_) in eps_hist if p_ < pe][-3:]
                    cf_eps.append({"start": fy_start, "end": pe.strftime("%Y-%m-%d"), "val": round(sum(prev3) + info["eps"], 2), "accn": acc,
                                   "fy": pe.year, "fp": "FY", "form": form, "filed": fdate})
                    cf_eps.append({"start": fy_start, "end": (pe - pd.offsets.MonthEnd(3)).strftime("%Y-%m-%d"), "val": round(sum(prev3), 2),
                                   "accn": acc, "fy": pe.year, "fp": "FY", "form": form, "filed": fdate})
        subj = {"cik": str(cik), "entityType": "operating", "sic": sic, "sicDescription": "synthetic", "name": name, "tickers": [tk],
                "exchanges": ["NYSE"], "fiscalYearEnd": "1231", "stateOfIncorporation": "DE", "category": "Large accelerated filer",
                "formerNames": [], "filings": {"recent": recent, "files": []}}
        p = sec / "data.sec.gov" / "submissions"
        p.mkdir(parents=True, exist_ok=True)
        (p / f"CIK{cik:010d}.json").write_text(json.dumps(subj), encoding="utf-8")
        assets = [{"end": e["end"], "val": 1e9 * (1 + 0.1 * k), "accn": e["accn"], "fy": e["fy"], "fp": e["fp"], "form": e["form"],
                   "filed": e["filed"]} for k, e in enumerate(cf_eps)]
        cf = {"cik": cik, "entityName": name, "facts": {
            "us-gaap": {"EarningsPerShareDiluted": {"units": {"USD/shares": cf_eps}},
                        "Assets": {"units": {"USD": assets}},
                        "Liabilities": {"units": {"USD": [{**a, "val": a["val"] * 0.6} for a in assets]}},
                        "StockholdersEquity": {"units": {"USD": [{**a, "val": a["val"] * 0.4} for a in assets]}}},
            "dei": {"EntityCommonStockSharesOutstanding": {"units": {"shares": [{**a, "val": 1e8} for a in assets]}}}}}
        p = sec / "data.sec.gov" / "api" / "xbrl" / "companyfacts"
        p.mkdir(parents=True, exist_ok=True)
        (p / f"CIK{cik:010d}.json").write_text(json.dumps(cf), encoding="utf-8")
        for (form, t, *_r) in filings:
            if form == "10-K" and t.year <= 2021:
                hf_rows.append({"cik": f"{cik:010d}", "name": name, "tickers": tk, "exchanges": "NYSE", "sic": sic,
                                "entity_type": "operating", "form": "10-K", "filing_date": t.strftime("%Y-%m-%d"),
                                "acceptance_raw": _json_time(t), "report_date": None, "doc_id": f"{cik:010d}_10-K_{t.year}",
                                "n_sections_nonempty": 4, "n_sentences": 100, "split": "train", "shard": "synthetic"})
    p = sec / "www.sec.gov" / "files"
    p.mkdir(parents=True, exist_ok=True)
    (p / "company_tickers_exchange.json").write_text(json.dumps({"fields": ["cik", "name", "ticker", "exchange"], "data": tickers_ex}))
    yf = pd.DataFrame(yf_rows, columns=["ticker", "date", "open", "high", "low", "close", "volume", "dividends", "splits"])
    yf.to_parquet(inputs / "yf_frozen.parquet", index=False)
    pd.DataFrame(hf_rows).to_parquet(inputs / "hf_census_5da7e3c8b920.parquet", index=False)
    sp = pd.DataFrame(sp_rows, columns=["ticker", "start_date", "end_date"])
    sp.to_csv(inputs / "sp500_ticker_start_end_a2430f2a.csv", index=False)
    txt = (" 1 NoDur  Consumer Nondurables\n          0100-0999\n\n 5 Chems  Chemicals\n          2800-2829\n\n"
           " 6 BusEq  Business Equipment\n          3570-3579\n\n10 Hlth   Healthcare\n          2830-2839\n\n11 Money  Finance\n          6000-6999\n")
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, "w") as z:
        z.writestr("Siccodes12.txt", txt)
    (inputs / "Siccodes12.zip").write_bytes(buf.getvalue())
    lm = pd.DataFrame({"Word": [w.upper() for w in POS + NEG + UNC]})
    for c in ["Negative", "Positive", "Uncertainty", "Litigious", "Strong_Modal", "Weak_Modal", "Constraining"]:
        lm[c] = 0
    lm.loc[lm.Word.isin([w.upper() for w in NEG]), "Negative"] = 2009
    lm.loc[lm.Word.isin([w.upper() for w in POS]), "Positive"] = 2009
    lm.loc[lm.Word.isin([w.upper() for w in UNC]), "Uncertainty"] = 2009
    lm.loc[lm.Word == "LITIGATION", "Litigious"] = 2009
    lm.to_csv(inputs / "lexicons" / "Loughran-McDonald_MasterDictionary_1993-2025.csv", index=False)
    return {"inputs": str(inputs), "sec": str(sec), "factors": fac, "tickers": [f[0] for f in FIRMS]}


def offline_overrides() -> dict:
    return {"universe": {"n_stocks": 3, "fixed_tickers": ["AAA", "BBB", "CCC"]},
            "study": {"start": "2021-01-01", "end": "2023-12-31"},
            "market": {"warmup_start": "2019-06-03"},
            "smoke": {"max_filings_per_firm_form": 20},
            "tuning": {"trials": {"lightgbm": 2, "catboost": 1, "enet": 2}, "permutations": 2},
            "bootstrap": {"n": 50}}


In [ ]:
%%writefile $REPO/src/secprice/track_a/__init__.py
"""secprice.track_a"""


In [ ]:
%%writefile $REPO/src/secprice/track_a/drift.py
"""Drift tests and verdict rubric (SPEC.md §10.2). 'Supported' needs: consistent definitions, adequate coverage, an
FDR-adjusted p < 0.05 AND a CI excluding the pre-registered trivial threshold, the same sign in >= 2 of 3
subperiods, clean-event robustness, and survival under 10/20 bps costs and a one-session lag. Failing one test
never proves an anomaly is absent."""
from __future__ import annotations

import re

import numpy as np
import pandas as pd

from ..evaluation.bootstrap import mean_ci
from .event_study import clustered_mean

VERDICTS = ["supported", "weak/mixed", "not supported", "inconclusive/low power", "not testable in scope"]


def trivial_bps(h: int, tb: dict) -> float:
    return float(tb["h_le_2"] if h <= 2 else tb["h_5"] if h <= 5 else tb["h_ge_20"])


def prior_quarter_quintiles(df: pd.DataFrame, col: str) -> pd.Series:
    """Quintile using breakpoints from the previous calendar quarter's distribution (tradable; no look-ahead)."""
    q = pd.to_datetime(df["trackA_day0"]).dt.to_period("Q")
    out = pd.Series(np.nan, index=df.index)
    for per, g in df.groupby(q):
        prev = df[(q < per) & (q >= per - 4)][col].dropna()
        if len(prev) < 25:
            continue
        bp = np.quantile(prev, [0.2, 0.4, 0.6, 0.8])
        out[g.index] = np.searchsorted(bp, g[col].to_numpy(), side="right") + 1
    out[df[col].isna()] = np.nan
    return out


def spread_test(df, qcol, car_col, firm, date, n_boot, seed):
    """Q5 - Q1 spread: OLS of CAR on a Q5 dummy within Q1 u Q5 with two-way (firm, date) clustered SE;
    CI from a month-block bootstrap of mean(Q5) - mean(Q1)."""
    import statsmodels.api as sm
    from scipy.stats import norm
    sub = df[df[qcol].isin([1, 5]) & df[car_col].notna()]
    hi, lo = sub[sub[qcol] == 5], sub[sub[qcol] == 1]
    if len(hi) < 10 or len(lo) < 10:
        return None
    from .event_study import cluster_fit
    y = sub[car_col].to_numpy(float)
    X = np.column_stack([np.ones(len(sub)), (sub[qcol] == 5).to_numpy(float)])
    res, _how = cluster_fit(y, X, sub[firm].to_numpy(), sub[date].astype(str).to_numpy())
    est, se = float(res.params[1]), float(res.bse[1])
    t = est / se if se > 0 else None
    months = pd.to_datetime(sub[date]).dt.to_period("M").astype(str).to_numpy()
    um, inv = np.unique(months, return_inverse=True)
    is_hi = (sub[qcol] == 5).to_numpy()
    s_hi = np.bincount(inv, weights=np.where(is_hi, y, 0), minlength=len(um))
    n_hi = np.bincount(inv, weights=is_hi.astype(float), minlength=len(um))
    s_lo = np.bincount(inv, weights=np.where(~is_hi, y, 0), minlength=len(um))
    n_lo = np.bincount(inv, weights=(~is_hi).astype(float), minlength=len(um))
    rng = np.random.default_rng(seed)
    bs = []
    for _ in range(n_boot):
        s = rng.integers(0, len(um), len(um))
        a, b = n_hi[s].sum(), n_lo[s].sum()
        if a > 0 and b > 0:
            bs.append(s_hi[s].sum() / a - s_lo[s].sum() / b)
    lo_ci, hi_ci = (np.quantile(bs, [0.025, 0.975]) if bs else (np.nan, np.nan))
    return {"estimate": est, "t": t, "p_value": float(2 * (1 - norm.cdf(abs(t)))) if t is not None else None,
            "n": int(len(sub)), "ci_low": float(lo_ci), "ci_high": float(hi_ci),
            "q5_mean": float(hi[car_col].mean()), "q1_mean": float(lo[car_col].mean())}


def subperiod_signs(df, qcol, car_col, n_sub=3) -> tuple[int, list]:
    d = pd.to_datetime(df["trackA_day0"])
    edges = np.quantile(d.astype("int64"), np.linspace(0, 1, n_sub + 1))
    signs = []
    for a, b in zip(edges[:-1], edges[1:]):
        g = df[(d.astype("int64") >= a) & (d.astype("int64") <= b)]
        hi, lo = g[g[qcol] == 5][car_col].mean(), g[g[qcol] == 1][car_col].mean()
        signs.append(float(np.sign(hi - lo)) if np.isfinite(hi - lo) else 0.0)
    return signs


def robust_same_sign(main: dict | None, alt: dict | None, alpha: float = 0.10) -> bool | None:
    """Robustness of a drift estimate (clean events, one-session lag): same sign and p < alpha. None when the alternative
    could not be estimated (too few events) - unknown, not 'failed'. A p-value of exactly 0 counts as significant (the
    earlier `p or 1` idiom treated 0.0 as missing; D-073)."""
    if not main or not alt or alt.get("estimate") is None or main.get("estimate") is None:
        return None
    p = alt.get("p_value")
    return bool(np.sign(alt["estimate"]) == np.sign(main["estimate"]) and p is not None and p < alpha)


def verdict(res: dict | None, trivial: float, p_adj: float | None, signs: list, clean_ok: bool | None,
            cost_ok: bool | None, lag_ok: bool | None, min_n: int = 100) -> str:
    if res is None:
        return "inconclusive/low power"
    if res["n"] < min_n:
        return "inconclusive/low power"
    sgn = np.sign(res["estimate"])
    ci_excl = (res["ci_low"] is not None and res["ci_high"] is not None and
               ((res["ci_low"] > trivial / 1e4) or (res["ci_high"] < -trivial / 1e4)))
    same = sum(1 for s in signs if s == sgn) >= 2
    if p_adj is not None and p_adj < 0.05 and ci_excl and same and clean_ok and cost_ok and lag_ok:
        return "supported"
    if p_adj is not None and p_adj < 0.10 or (same and res.get("p_value") is not None and res["p_value"] < 0.10):
        return "weak/mixed"
    return "not supported"


DIV_UP = re.compile(r"(increas|rais)\w*\s+(its\s+|the\s+)?(regular\s+|quarterly\s+)*(cash\s+)?dividend|dividend\s+increase", re.I)
DIV_DN = re.compile(r"(reduc|cut|suspend|eliminat|lower)\w*\s+(its\s+|the\s+)?(regular\s+|quarterly\s+)*(cash\s+)?dividend", re.I)
DIV_INIT = re.compile(r"(initiat|first[- ]ever|inaugural)\w*\s+(a\s+|its\s+)?(quarterly\s+)?(cash\s+)?dividend", re.I)
SPIN = re.compile(r"spin-?off|spinoff|separation of .{0,60}(into|as) (an )?independent", re.I)


def detect_dividend(text: str) -> str | None:
    t = text[:20000]
    if DIV_DN.search(t):
        return "decrease"
    if DIV_INIT.search(t):
        return "initiation"
    if DIV_UP.search(t):
        return "increase"
    return None


def momentum_benchmark(panel, universe_tickers: list[str], start, end) -> pd.DataFrame:
    """Monthly equal-weight decile long-short returns: 1-month reversal, 2-12 (12-2), 12-1. Benchmark only."""
    cal = panel.cal
    me = pd.Series(np.arange(len(cal)), index=cal.sessions).groupby(cal.sessions.to_period("M")).max()
    me = me[(me.index >= pd.Period(start, "M")) & (me.index <= pd.Period(end, "M"))]
    tri = np.vstack([panel[t]["total_return_index"] for t in universe_tickers if panel.has(t)])
    rows = []
    idx = me.to_numpy()
    for k in range(13, len(idx) - 1):
        c, n1 = idx[k], idx[k + 1]
        fwd = tri[:, n1] / tri[:, c] - 1
        sig = {"rev_1m": -(tri[:, c] / tri[:, idx[k - 1]] - 1),
               "mom_12_2": tri[:, idx[k - 1]] / tri[:, idx[k - 12]] - 1,
               "mom_12_1": tri[:, c] / tri[:, idx[k - 12]] - 1}
        for name, s in sig.items():
            ok = np.isfinite(s) & np.isfinite(fwd)
            if ok.sum() < 50:
                continue
            q = pd.qcut(s[ok], 10, labels=False, duplicates="drop")
            ls = fwd[ok][q == q.max()].mean() - fwd[ok][q == 0].mean()
            rows.append({"month": str(me.index[k + 1]), "signal": name, "ls_ret": float(ls), "n": int(ok.sum())})
    return pd.DataFrame(rows)


def newey_west_t(x: np.ndarray, lags: int = 3) -> float | None:
    import statsmodels.api as sm
    x = x[np.isfinite(x)]
    if len(x) < 12:
        return None
    r = sm.OLS(x, np.ones((len(x), 1))).fit(cov_type="HAC", cov_kwds={"maxlags": lags})
    return float(r.tvalues[0])


In [ ]:
%%writefile $REPO/src/secprice/track_a/event_study.py
"""Track A event study (SPEC.md §5.3, §10.1): CARs around day 0, abnormal volume and volatility response, by form /
8-K category / bucket, all-event and clean-event, pre-trend windows; errors clustered by firm and date.
Association only. INTRA day-0 returns are a mixed pre/post interval (labelled)."""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..targets.build import Estimator

WINDOWS = [(-1, 1), (0, 1), (0, 5), (2, 20), (2, 60), (-20, -2), (-5, -1), (3, 61)]   # (3, 61): one-session-lag drift


def wname(a, b) -> str:
    return f"car_{a}_{b}".replace("-", "m")


def window_cars(ev: pd.DataFrame, panel, spy: dict, factors: pd.DataFrame | None, cfg: dict,
                usable_cutoff_idx: int) -> pd.DataFrame:
    cal = panel.cal
    n = len(cal)
    w0, w1 = cfg["targets"]["est_window"]
    min_obs = int(cfg["targets"]["est_min_obs"])
    fac = rf = None
    if factors is not None and len(factors):
        fidx = cal.sessions.searchsorted(pd.to_datetime(factors["date"]).values)
        ok = (fidx < n) & (cal.sessions[np.clip(fidx, 0, n - 1)] == pd.to_datetime(factors["date"]).values)
        fac = np.full((n, 4), np.nan)
        rf = np.full(n, np.nan)
        fac[fidx[ok]] = factors.loc[ok, ["mkt_rf", "smb", "hml", "mom"]].to_numpy(float)
        rf[fidx[ok]] = factors.loc[ok, "rf"].to_numpy(float)
    rm = spy["ret"]
    out = []
    for tk, g in ev.groupby("ticker", sort=False):
        if not panel.has(tk):
            continue
        P = panel[tk]
        r = P["ret"]
        d0 = g["trackA_day0_idx"].to_numpy(int)
        est = Estimator(r, rm)
        cnt, al, be, sig = est.window(d0 + w0, d0 + w1)
        low = cnt < min_obs
        al[low], be[low], sig[low] = np.nan, np.nan, np.nan
        rec = {"event_id": g["event_id"].to_numpy(), "sigma": sig, "beta": be, "n_est": cnt}
        fin = np.isfinite(r) & np.isfinite(rm)
        cs = np.concatenate([[0.0], np.cumsum(np.where(fin, r - 0.0, 0.0))])
        csm = np.concatenate([[0.0], np.cumsum(np.where(fin, rm, 0.0))])
        bad = np.concatenate([[0], np.cumsum(~fin)])
        for a, b in WINDOWS:
            lo, hi = d0 + a, d0 + b
            okw = (lo >= 1) & (hi < n) & (hi <= usable_cutoff_idx)
            loc, hic = np.clip(lo, 0, n - 1), np.clip(hi, 0, n - 1)
            sr = cs[hic + 1] - cs[loc]
            sm_ = csm[hic + 1] - csm[loc]
            gaps = (bad[hic + 1] - bad[loc]) > 0
            car = sr - (al * (b - a + 1) + be * sm_)
            car[~okw | gaps] = np.nan
            rec[wname(a, b)] = car
            if fac is not None and (a, b) in ((-1, 1), (0, 1), (2, 60), (2, 20), (3, 61)):
                c4 = np.full(len(d0), np.nan)
                for i in range(len(d0)):
                    if not okw[i] or gaps[i] or d0[i] + w0 < 1:
                        continue
                    seg = slice(d0[i] + w0, d0[i] + w1 + 1)
                    y = r[seg] - rf[seg]
                    X = fac[seg]
                    m = np.isfinite(y) & np.isfinite(X).all(1)
                    if m.sum() < min_obs:
                        continue
                    A = np.column_stack([np.ones(m.sum()), X[m]])
                    coef, *_ = np.linalg.lstsq(A, y[m], rcond=None)
                    s2 = slice(lo[i], hi[i] + 1)
                    yy, XX = r[s2] - rf[s2], fac[s2]
                    if np.isfinite(yy).all() and np.isfinite(XX).all():
                        c4[i] = float(np.sum(yy - coef[0] - XX @ coef[1:]))
                rec[wname(a, b).replace("car_", "car4_")] = c4
        vol = P["volume"]
        lv = np.log(np.where(vol > 0, vol, np.nan))
        csv = np.concatenate([[0.0], np.nancumsum(np.where(np.isfinite(lv), lv, 0.0))])
        cnv = np.concatenate([[0], np.cumsum(np.isfinite(lv))])
        lo, hi = np.clip(d0 + w0, 0, n - 1), np.clip(d0 + w1, 0, n - 1)
        with np.errstate(divide="ignore", invalid="ignore"):
            base_lv = (csv[hi + 1] - csv[lo]) / (cnv[hi + 1] - cnv[lo])
            e0, e1 = np.clip(d0, 0, n - 1), np.clip(d0 + 1, 0, n - 1)
            ev_lv = (csv[e1 + 1] - csv[e0]) / (cnv[e1 + 1] - cnv[e0])
        rec["abn_log_volume_0_1"] = ev_lv - base_lv
        with np.errstate(divide="ignore", invalid="ignore"):
            rec["vol_response_0_1"] = np.abs(rec[wname(0, 1)]) / (sig * np.sqrt(2))
        out.append(pd.DataFrame(rec))
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()


def cluster_fit(y: np.ndarray, X: np.ndarray, firm, date):
    """OLS with two-way (firm, date) clustered errors; steps down to one-way and then to HC1 when a dimension has fewer
    than two clusters. Returns (result, method)."""
    import statsmodels.api as sm
    gf, gd = pd.factorize(pd.Series(firm).astype(str))[0], pd.factorize(pd.Series(date).astype(str))[0]
    nf, nd = len(np.unique(gf)), len(np.unique(gd))
    tries = []
    if nf >= 2 and nd >= 2:
        tries.append(("two-way firm x date", {"cov_type": "cluster", "cov_kwds": {"groups": np.column_stack([gf, gd])}}))
    if nf >= 2:
        tries.append(("firm", {"cov_type": "cluster", "cov_kwds": {"groups": gf}}))
    if nd >= 2:
        tries.append(("date", {"cov_type": "cluster", "cov_kwds": {"groups": gd}}))
    tries.append(("HC1 (fewer than 2 clusters)", {"cov_type": "HC1"}))
    for name, kw in tries:
        try:
            r = sm.OLS(y, X).fit(**kw)
            if np.all(np.isfinite(r.bse)):
                return r, name
        except Exception:
            continue
    return sm.OLS(y, X).fit(), "iid"


def clustered_mean(y: np.ndarray, firm: np.ndarray, date: np.ndarray) -> dict:
    """Mean with two-way (firm, date) clustered standard error (intercept-only OLS)."""
    from scipy.stats import norm
    ok = np.isfinite(y)
    y, firm, date = y[ok], np.asarray(firm)[ok], np.asarray(date)[ok]
    n = len(y)
    if n < 3:
        return {"mean": float(np.mean(y)) if n else None, "se": None, "t": None, "p": None, "n": n, "se_method": None}
    res, how = cluster_fit(y, np.ones((n, 1)), firm, date)
    se = float(res.bse[0])
    t = float(res.params[0] / se) if se > 0 else None
    return {"mean": float(res.params[0]), "se": se, "t": t, "p": float(2 * (1 - norm.cdf(abs(t)))) if t is not None else None,
            "n": int(n), "se_method": how}


def group_table(df: pd.DataFrame, group_col: str, cols: list[str], clean_col: str) -> pd.DataFrame:
    rows = []
    for gname, g in df.groupby(group_col):
        for es, gg in (("all", g), ("clean", g[~g[clean_col]])):
            for c in cols:
                if c not in gg:
                    continue
                st = clustered_mean(gg[c].to_numpy(float), gg["cik"].to_numpy(), gg["trackA_day0"].astype(str).to_numpy())
                rows.append({"group_by": group_col, "group": gname, "event_set": es, "measure": c, **st})
    return pd.DataFrame(rows)


def car_paths(ev: pd.DataFrame, panel, spy, cfg, rel=range(-20, 61), group_col: str = "event_type") -> pd.DataFrame:
    """Average cumulative market-model abnormal return path from day -20 to +60 by group (F11)."""
    cal = panel.cal
    n = len(cal)
    w0, w1 = cfg["targets"]["est_window"]
    rows = []
    rel = list(rel)
    for tk, g in ev.groupby("ticker", sort=False):
        if not panel.has(tk):
            continue
        r = panel[tk]["ret"]
        est = Estimator(r, spy["ret"])
        d0 = g["trackA_day0_idx"].to_numpy(int)
        cnt, al, be, _ = est.window(d0 + w0, d0 + w1)
        for i, (eid, grp) in enumerate(zip(g.event_id, g[group_col])):
            if cnt[i] < int(cfg["targets"]["est_min_obs"]) or d0[i] + rel[0] < 1 or d0[i] + rel[-1] >= n:
                continue
            idx = d0[i] + np.array(rel)
            ar = r[idx] - (al[i] + be[i] * spy["ret"][idx])
            if not np.isfinite(ar).all():
                continue
            rows.append(pd.DataFrame({"group": grp, "rel_day": rel, "ar": ar}))
    if not rows:
        return pd.DataFrame(columns=["group", "rel_day", "caar", "n"])
    d = pd.concat(rows)
    m = d.groupby(["group", "rel_day"])["ar"].agg(["mean", "count"]).reset_index()
    m["caar"] = m.groupby("group")["mean"].cumsum()
    return m.rename(columns={"count": "n"})[["group", "rel_day", "caar", "n"]]


In [ ]:
%%writefile $REPO/src/secprice/universe/__init__.py
"""secprice.universe"""


In [ ]:
%%writefile $REPO/src/secprice/universe/build.py
"""S02 Universe: firms common to our OHLCV files and the HF 10-K corpus, mapped ticker->CIK, S&P-500-history
aware, outcome-free selection (SPEC.md §3 universe.*, hard rule 9)."""
from __future__ import annotations

import io
import json
import re
import zipfile

import numpy as np
import pandas as pd
import requests

from ..sec.submissions import norm_ticker

SP500_URL = "https://raw.githubusercontent.com/fja05680/sp500/{commit}/sp500_ticker_start_end.csv"
FF12_NAMES = {1: "NoDur", 2: "Durbl", 3: "Manuf", 4: "Enrgy", 5: "Chems", 6: "BusEq", 7: "Telcm", 8: "Utils",
              9: "Shops", 10: "Hlth", 11: "Money", 12: "Other"}


def load_sp500(commit: str, cache_path=None) -> pd.DataFrame:
    if cache_path is not None and cache_path.exists():
        df = pd.read_csv(cache_path)
    else:
        r = requests.get(SP500_URL.format(commit=commit), timeout=60)
        r.raise_for_status()
        df = pd.read_csv(io.StringIO(r.text))
        if cache_path is not None:
            cache_path.write_text(r.text, encoding="utf-8")
    df["ticker"] = df["ticker"].astype(str).str.upper()
    df["start_date"] = pd.to_datetime(df["start_date"])
    df["end_date"] = pd.to_datetime(df["end_date"])
    return df


def load_ff12(url: str, cache_path=None) -> list[tuple[int, int, int]]:
    if cache_path is not None and cache_path.exists():
        raw = cache_path.read_bytes()
    else:
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        raw = r.content
        if cache_path is not None:
            cache_path.write_bytes(raw)
    z = zipfile.ZipFile(io.BytesIO(raw))
    text = z.read(z.namelist()[0]).decode("latin1")
    ranges, cur = [], None
    for line in text.splitlines():
        m = re.match(r"\s*(\d+)\s+(\w+)\s", line)
        if m and not re.match(r"\s*\d{4}-\d{4}", line):
            cur = int(m.group(1))
            continue
        m2 = re.match(r"\s*(\d{4})-(\d{4})", line)
        if m2 and cur:
            ranges.append((int(m2.group(1)), int(m2.group(2)), cur))
    return ranges


def sic_to_ff12(sic, ranges) -> str:
    try:
        s = int(sic)
    except (TypeError, ValueError):
        return "Other"
    for lo, hi, k in ranges:
        if lo <= s <= hi:
            return FF12_NAMES[k]
    return "Other"


def map_tickers(ohlcv: pd.DataFrame, sec_map: pd.DataFrame, hf_census: pd.DataFrame) -> pd.DataFrame:
    """ticker -> CIK via SEC's current ticker map, then the HF corpus tickers as a fallback."""
    m = dict(zip(sec_map["ticker"], sec_map["cik"].astype(int)))
    hf = {}
    for c, ts in zip(hf_census["cik"].astype(int), hf_census["tickers"].fillna("")):
        for t in str(ts).split("|"):
            if t:
                hf.setdefault(t.upper(), c)
    out = ohlcv.copy()
    nt = out["ticker"].map(norm_ticker)
    out["cik_sec"] = nt.map(m).astype("Int64")
    out["cik_hf"] = nt.map(hf).astype("Int64")
    out["cik"] = out["cik_sec"].fillna(out["cik_hf"]).astype("Int64")
    out["mapping_source"] = np.where(out["cik_sec"].notna(), "sec_company_tickers_exchange",
                                     np.where(out["cik_hf"].notna(), "hf_corpus_tickers", None))
    # a reorganisation (new holding-company CIK) leaves the history under the predecessor CIK: keep both
    out["ciks_all"] = [json.dumps(sorted({int(x) for x in (a, b) if pd.notna(x)})) for a, b in zip(out.cik_sec, out.cik_hf)]
    return out


def sp500_intervals(sp: pd.DataFrame, ticker: str) -> list[tuple]:
    rows = sp[sp.ticker.isin({ticker.upper(), norm_ticker(ticker), ticker.upper().replace("-", ".")})]
    return [(a, b) for a, b in zip(rows.start_date, rows.end_date)]


def select_universe(cand: pd.DataFrame, n: int, seed: int, fixed: list[str] | None = None,
                    balanced: bool = False) -> pd.DataFrame:
    """Deterministic, outcome-free selection: S&P-500 members first, then seeded sector-stratified fill.
    Never reads price returns (hard rule 9)."""
    cand = cand.copy()
    cand["selected"] = False
    cand["selection_rank"] = np.nan
    if fixed:
        sel = cand["ticker"].isin([t.upper() for t in fixed]) & cand["eligible"]
        cand.loc[sel, "selected"] = True
        cand.loc[sel, "selection_rank"] = np.arange(sel.sum())
        return cand
    elig = cand[cand["eligible"]].copy()
    rng = np.random.default_rng(seed)
    elig["_u"] = rng.random(len(elig))
    picked: list[int] = []

    def take(pool: pd.DataFrame, k: int) -> list[int]:
        if k <= 0 or pool.empty:
            return []
        if k >= len(pool):
            return pool.sort_values("_u").index.tolist()
        counts = pool.groupby("ff12_industry").size()
        if balanced:
            quota = pd.Series(k // len(counts), index=counts.index).clip(upper=counts)
        else:
            raw = counts / counts.sum() * k
            quota = np.floor(raw).astype(int)
            rem = k - quota.sum()
            for ind in (raw - quota).sort_values(ascending=False).index[:rem]:
                quota[ind] += 1
            quota = quota.clip(upper=counts)
        out = []
        for ind, q in quota.items():
            out += pool[pool.ff12_industry == ind].sort_values("_u").index[:q].tolist()
        short = k - len(out)
        if short > 0:
            rest = pool.drop(index=out).sort_values("_u")
            out += rest.index[:short].tolist()
        return out

    a = elig[elig["ever_sp500"]]
    b = elig[~elig["ever_sp500"]]
    picked += take(a, min(n, len(a)))
    picked += take(b, n - len(picked))
    cand.loc[picked, "selected"] = True
    order = elig.loc[picked].sort_values(["ever_sp500", "_u"], ascending=[False, True]).index
    cand.loc[order, "selection_rank"] = np.arange(len(order))
    return cand


def firms_per_year(univ: pd.DataFrame, start: str, end: str) -> pd.DataFrame:
    sel = univ[univ.selected]
    rows = []
    for y in range(pd.Timestamp(start).year, pd.Timestamp(end).year + 1):
        y0, y1 = pd.Timestamp(f"{y}-01-01"), pd.Timestamp(f"{y}-12-31")
        has_px = ((pd.to_datetime(sel.price_start) <= y1) & (pd.to_datetime(sel.price_end) >= y0)).sum()
        in_idx = 0
        for iv in sel["sp500_intervals"]:
            ivs = json.loads(iv) if isinstance(iv, str) else (iv or [])
            if any(pd.Timestamp(a) <= y1 and (pd.isna(pd.Timestamp(b)) or pd.Timestamp(b) >= y0) for a, b in ivs):
                in_idx += 1
        rows.append({"year": y, "firms_with_prices": int(has_px), "firms_in_sp500": int(in_idx), "unique_firms": len(sel)})
    return pd.DataFrame(rows)


In [ ]:
%%writefile $REPO/src/secprice/viz/__init__.py
"""secprice.viz"""


In [ ]:
%%writefile $REPO/src/secprice/viz/figures.py
"""Figures F01-F40 (SPEC.md §20.1), one function per ID. Each figure is saved with its 'how to read' caption;
a figure whose inputs do not exist is recorded with the reason instead of being drawn empty."""
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

CAPTIONS = {
    "F01": "Coverage heatmap: share of observed filings that parsed into at least one section, by year x form. Pale cells = parser or download gaps.",
    "F02": "Audit reason codes by year (periodic-filing expectations only; 8-Ks have no expectation).",
    "F03": "8-K category availability by year under taxonomy v1 (legacy items before 2004-08-23).",
    "F04": "Universe: unique firms (constant) vs firms with prices and firms in the S&P 500 each year. Survivor-only universe (current listings; survivorship-biased).",
    "F05": "Timestamp quality (A exact / B date-only / C uncertain) and session buckets by year.",
    "F06": "Pre-publication window (last close -> acceptance) and the share of events whose clock changes under public-lag sensitivity.",
    "F07": "Event-clock diagram: acceptance, assumed public time, decision time; T_info base/exit; T_trade entry/exit.",
    "F08": "Earnings event types over time (verified releases, periodic-only, ambiguous).",
    "F09": "Same-firm overlap counts by horizon and the clean-event share.",
    "F10": "Target distributions (abnormal returns) and 3-class balance by target definition.",
    "F11": "Average cumulative abnormal return path (market model) around day 0 by event type, all events vs clean events.",
    "F12": "CAR[0,+1] by session bucket (INTRA day 0 is a mixed pre/post interval).",
    "F13": "Pre-trend diagnostics: CAR[-20,-2] and CAR[-5,-1] by event type (interpreted, not forced to zero).",
    "F14": "Abnormal log volume and volatility response on days [0,+1] by event type.",
    "F15": "Spearman correlations between text features and event-window CARs (association only).",
    "F16": "SUE-conditioned post-earnings drift: CAR[+2,+60] by SUE_release quintile (prior-quarter breakpoints).",
    "F17": "Split timeline: walk-forward validation blocks, purge/embargo gap, development holdout, locked test; per-fold strict/retrospective labels.",
    "F18": "Ablation ladder: walk-forward macro-F1 by rung with month-block bootstrap CIs (common and maximum-available cohorts).",
    "F19": "H4: point-in-time ChronoBERT vintages vs static encoders on identical rows and heads.",
    "F20": "Loss curves: train vs inner-validation loss per boosting round (GBDT) and per step/epoch (fine-tuning).",
    "F21": "Validation curves: walk-forward objective vs each tuned hyperparameter (Optuna trials).",
    "F22": "Learning curves: validation macro-F1 vs training-set fraction (most recent rows kept).",
    "F23": "Before vs after tuning: default vs tuned walk-forward macro-F1 per rung and head.",
    "F24": "Confusion matrices (row-normalised) for the selected models.",
    "F25": "One-vs-rest ROC and precision-recall curves for the selected primary model.",
    "F26": "Calibration: reliability curves and expected calibration error.",
    "F27": "Coverage-accuracy curve: accuracy and macro-F1 when only the most confident predictions are kept.",
    "F28": "Rolling monthly rank IC of the abnormal-return regression head.",
    "F29": "T_info vs T_trade: the same rung scored on the information-response and the tradable target.",
    "F30": "Backtest: cumulative long-only (vs interval-matched SPY) and hypothetical long-short returns, by cost level and with a one-session lag.",
    "F31": "Primary-model macro-F1 by form, category, bucket, timestamp quality, size, sector and year.",
    "F32": "Retrieval metrics by setup on gold-val qrels (BM25, hybrid, hybrid + rerank).",
    "F33": "Extraction field F1 and faithfulness on gold-test.",
    "F34": "Numeric fidelity: exact and tolerance matches against source/XBRL.",
    "F35": "Faithfulness judge vs human ratings (kappa).",
    "F36": "Student vs teacher vs human agreement.",
    "F37": "Permutation null: distribution of macro-F1 under within-block class-stratified label permutation vs the observed value.",
    "F38": "Random-split demo (pedagogical only): random split vs walk-forward macro-F1.",
    "F39": "Adversarial validation ROC per fold (an AUC near 0.5 does not prove a split is unbiased).",
    "F40": "Selection diagnostics: Model Confidence Set membership and PBO (diagnostics only).",
    "FX1": "Supplementary - optimisation diagnostics: Optuna best-so-far objective per trial and the elastic-net convexity certificate (equal objective from several random starts = global optimum of that convex problem; tree and transformer optima are validation-selected, not global).",
    "FX2": "Supplementary - S&P 500 vs non-S&P 500 vs combined: walk-forward macro-F1 of the combined L0/L2/L6 models scored on each "
           "bucket, with month-block bootstrap CIs (overlapping CIs do not show a difference).",
    "FX3": "Supplementary - average cumulative abnormal return path (market model) by index bucket, all event types "
           "(association only; survivor-only universe).",
}


class Figs:
    def __init__(self, out_dir: Path, dpi: int = 130):
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        self.plt = plt
        self.out = Path(out_dir)
        self.out.mkdir(parents=True, exist_ok=True)
        self.dpi = dpi
        self.index: list[dict] = []
        plt.rcParams.update({"figure.figsize": (9, 4.8), "axes.grid": True, "grid.alpha": 0.3, "font.size": 9,
                             "axes.spines.top": False, "axes.spines.right": False})

    def save(self, fig, fid: str, title: str):
        p = self.out / f"{fid}.png"
        fig.suptitle(f"{fid} - {title}", fontsize=10, x=0.01, ha="left")
        fig.tight_layout()
        fig.savefig(p, dpi=self.dpi)
        self.plt.close(fig)
        self.index.append({"id": fid, "title": title, "path": str(p), "caption": CAPTIONS.get(fid, ""), "status": "ok"})

    def skip(self, fid: str, title: str, reason: str):
        self.index.append({"id": fid, "title": title, "path": None, "caption": CAPTIONS.get(fid, ""),
                           "status": f"not produced: {reason}"})

    def safe(self, fid, title, fn, *a, **k):
        try:
            fn(fid, title, *a, **k)
        except Exception as e:
            self.plt.close("all")
            self.skip(fid, title, f"error {type(e).__name__}: {str(e)[:150]}")

    # ------------------------------------------------------------------ generic helpers
    def heat(self, fid, title, mat: pd.DataFrame, fmt="{:.2f}", cmap="viridis", vmin=None, vmax=None):
        fig, ax = self.plt.subplots(figsize=(min(16, 2 + 0.45 * mat.shape[1]), min(10, 1.5 + 0.35 * mat.shape[0])))
        im = ax.imshow(mat.values.astype(float), aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_xticks(range(mat.shape[1]), [str(c) for c in mat.columns], rotation=90)
        ax.set_yticks(range(mat.shape[0]), [str(i) for i in mat.index])
        if mat.size <= 400:
            for i in range(mat.shape[0]):
                for j in range(mat.shape[1]):
                    v = mat.values[i, j]
                    if np.isfinite(v):
                        ax.text(j, i, fmt.format(v), ha="center", va="center", fontsize=6, color="w")
        fig.colorbar(im, ax=ax)
        ax.grid(False)
        self.save(fig, fid, title)

    def stacked(self, fid, title, df: pd.DataFrame, xlabel="year", ylabel="count", share=False):
        d = df.div(df.sum(axis=1), axis=0) if share else df
        fig, ax = self.plt.subplots()
        bottom = np.zeros(len(d))
        for c in d.columns:
            ax.bar(d.index.astype(str), d[c].values, bottom=bottom, label=str(c))
            bottom += d[c].fillna(0).values
        ax.set_xlabel(xlabel)
        ax.set_ylabel("share" if share else ylabel)
        ax.tick_params(axis="x", rotation=90)
        ax.legend(fontsize=7, ncol=3)
        self.save(fig, fid, title)

    def bars_ci(self, fid, title, df: pd.DataFrame, x="label", y="value", lo="ci_low", hi="ci_high", hue=None, ref=None):
        fig, ax = self.plt.subplots(figsize=(10, 4.8))
        groups = [None] if hue is None else list(df[hue].unique())
        width = 0.8 / len(groups)
        labels = list(dict.fromkeys(df[x]))
        for k, gname in enumerate(groups):
            g = df if gname is None else df[df[hue] == gname]
            g = g.set_index(x).reindex(labels)
            xs = np.arange(len(labels)) + k * width - 0.4 + width / 2
            err = np.vstack([(g[y] - g[lo]).clip(lower=0).fillna(0), (g[hi] - g[y]).clip(lower=0).fillna(0)]) if lo in g else None
            ax.bar(xs, g[y].values, width, yerr=err, capsize=3, label=str(gname) if gname is not None else None)
        if ref is not None:
            ax.axhline(ref, color="k", lw=0.8, ls="--")
        ax.set_xticks(np.arange(len(labels)), labels, rotation=45, ha="right")
        if hue is not None:
            ax.legend(fontsize=7)
        self.save(fig, fid, title)

    def lines(self, fid, title, series: dict, xlabel="", ylabel="", logy=False):
        fig, ax = self.plt.subplots()
        for name, (x, y) in series.items():
            ax.plot(x, y, label=name, lw=1.2)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        if logy:
            ax.set_yscale("log")
        ax.legend(fontsize=7)
        self.save(fig, fid, title)

    # ------------------------------------------------------------------ specific figures
    def f07_clock(self, fid, title):
        plt = self.plt
        fig, ax = plt.subplots(figsize=(10, 3.4))
        ax.set_xlim(0, 10)
        ax.set_ylim(0, 3.2)
        ax.axis("off")
        for x0, x1, lab in ((0.5, 3.0, "session D-1"), (4.5, 7.0, "session D (next)")):
            ax.add_patch(plt.Rectangle((x0, 2.2), x1 - x0, 0.4, color="#ddd"))
            ax.text((x0 + x1) / 2, 2.72, lab, ha="center", fontsize=8)
        marks = [(3.0, "T_info base\n(last close <= acceptance)"), (3.4, "acceptance"), (3.55, "public\n(+lag, assumed)"),
                 (3.85, "decision\n(+latency)"), (4.5, "T_trade entry\n(first open >= decision)"),
                 (7.0, "T_info exit h=1 /\nT_trade exit h=1")]
        for i, (x, lab) in enumerate(marks):
            ax.axvline(x, ymin=0.25, ymax=0.85, color=["k", "C3", "C1", "C2", "C0", "C4"][i], lw=1.2)
            ax.text(x, 0.25 + 0.42 * (i % 2), lab, ha="center", fontsize=7)
        ax.annotate("", xy=(7.0, 1.75), xytext=(3.0, 1.75), arrowprops=dict(arrowstyle="<->", color="k"))
        ax.text(5.0, 1.82, "T_info: close -> close (INTRA excluded)", ha="center", fontsize=7)
        ax.annotate("", xy=(7.0, 1.45), xytext=(4.5, 1.45), arrowprops=dict(arrowstyle="<->", color="C0"))
        ax.text(5.75, 1.52, "T_trade: open -> close, SPY on the same interval", ha="center", fontsize=7, color="C0")
        self.save(fig, fid, title)

    def f17_timeline(self, fid, title, blocks: pd.DataFrame, prov: pd.DataFrame | None, start, embargo_days):
        fig, ax = self.plt.subplots(figsize=(11, 4.5))
        y = 0
        for _, b in blocks.iterrows():
            colr = {"walk_forward_val": "C0", "development_holdout": "C1", "locked_test": "C3"}[b.kind]
            ax.barh(y, (b.end_exclusive - b.start).days, left=b.start, color=colr, alpha=0.8)
            if b.kind == "walk_forward_val":
                ax.barh(y, (b.start - pd.Timestamp(start)).days, left=pd.Timestamp(start), color="C2", alpha=0.35)
                ax.barh(y, embargo_days, left=b.start - pd.Timedelta(days=embargo_days), color="k", alpha=0.5)
            lab = b.block
            if prov is not None and len(prov):
                p = prov[prov.block == b.block]
                st = ", ".join(f"{m}:{'S' if l == 'strict' else 'R'}" for m, l in zip(p.model, p.leaderboard)
                               if m in ("sec_bert", "finbert_tone", "chrono_bert", "bge_base", "qwen"))
                lab += f"  [{st}]"
            ax.text(b.start, y + 0.38, lab, fontsize=6)
            y += 1
        ax.set_yticks([])
        ax.set_xlabel("decision date")
        from matplotlib.patches import Patch
        ax.legend(handles=[Patch(color="C2", alpha=0.35, label="train (expanding)"), Patch(color="k", alpha=0.5, label="purge/embargo"),
                           Patch(color="C0", label="validation block"), Patch(color="C1", label="development holdout"),
                           Patch(color="C3", label="locked test")], fontsize=7, loc="lower right")
        self.save(fig, fid, title)

    def f20_loss(self, fid, title, curves: list[dict], ft_curves: list[dict]):
        n = (1 if curves else 0) + (1 if ft_curves else 0)
        if n == 0:
            return self.skip(fid, title, "no recorded curves")
        fig, axes = self.plt.subplots(1, n, figsize=(6 * n, 4.2), squeeze=False)
        k = 0
        if curves:
            ax = axes[0, k]
            for c in curves[:6]:
                lab = f"{c['rung']}/{c['head']}{'/' + c['enc'] if c.get('enc') else ''} f{c['fold']}"
                ax.plot(c["train"], lw=1, label=f"{lab} train")
                if c.get("val"):
                    ax.plot(c["val"], lw=1, ls="--", label=f"{lab} val")
            ax.set_xlabel(curves[0].get("unit", "iteration"))
            ax.set_ylabel(curves[0].get("metric", "loss"))
            ax.legend(fontsize=6)
            k += 1
        if ft_curves:
            ax = axes[0, k]
            for c in ft_curves[:4]:
                cu = c["curve"]
                st = np.array(cu["step"])
                tl = pd.Series(cu["train_loss"]).rolling(20, min_periods=1).mean()
                ax.plot(st, tl, lw=1, label=f"{c['label']} train (smoothed)")
                ax.plot(cu["eval_step"], cu["inner_val_loss"], "o--", ms=3, lw=1, label=f"{c['label']} inner-val")
            ax.set_xlabel("step")
            ax.set_ylabel("cross-entropy")
            ax.legend(fontsize=6)
        self.save(fig, fid, title)

    def f24_cm(self, fid, title, cms: dict):
        if not cms:
            return self.skip(fid, title, "no predictions")
        k = len(cms)
        fig, axes = self.plt.subplots(1, k, figsize=(3.4 * k, 3.4), squeeze=False)
        for ax, (name, cm) in zip(axes[0], cms.items()):
            cm = np.asarray(cm, float)
            cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
            ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
            for i in range(3):
                for j in range(3):
                    ax.text(j, i, f"{cmn[i, j]:.2f}\n({int(cm[i, j])})", ha="center", va="center", fontsize=7)
            ax.set_xticks(range(3), ["down", "side", "up"])
            ax.set_yticks(range(3), ["down", "side", "up"])
            ax.set_xlabel("predicted")
            ax.set_ylabel("true")
            ax.set_title(name, fontsize=8)
            ax.grid(False)
        self.save(fig, fid, title)

    def f25_roc(self, fid, title, y, proba):
        from sklearn.metrics import precision_recall_curve, roc_curve
        fig, (a1, a2) = self.plt.subplots(1, 2, figsize=(10, 4.2))
        for c, nm in enumerate(["down", "side", "up"]):
            pos = (y == c)
            if pos.any() and (~pos).any():
                fpr, tpr, _ = roc_curve(pos, proba[:, c])
                a1.plot(fpr, tpr, label=nm)
                pr, rc, _ = precision_recall_curve(pos, proba[:, c])
                a2.plot(rc, pr, label=f"{nm} (base {pos.mean():.2f})")
        a1.plot([0, 1], [0, 1], "k--", lw=0.7)
        a1.set_xlabel("FPR")
        a1.set_ylabel("TPR")
        a2.set_xlabel("recall")
        a2.set_ylabel("precision")
        a1.legend(fontsize=7)
        a2.legend(fontsize=7)
        self.save(fig, fid, title)

    def f26_calib(self, fid, title, y, proba, ece):
        from ..evaluation.metrics import reliability
        fig, ax = self.plt.subplots(figsize=(5.5, 4.5))
        for c, nm in enumerate(["down", "side", "up"]):
            r = reliability(proba, y, c)
            if len(r):
                ax.plot(r.mean_pred, r.frac_pos, "o-", ms=3, label=nm)
        ax.plot([0, 1], [0, 1], "k--", lw=0.7)
        ax.set_xlabel("mean predicted probability")
        ax.set_ylabel("observed frequency")
        ax.set_title(f"top-label ECE = {ece:.3f}" if ece is not None else "", fontsize=8)
        ax.legend(fontsize=7)
        self.save(fig, fid, title)

    def f27_coverage(self, fid, title, y, proba, yhat):
        from ..evaluation.metrics import confusion, f1_from_cm
        conf = proba.max(1)
        order = np.argsort(-conf)
        covs = np.linspace(0.05, 1, 20)
        acc, f1 = [], []
        for c in covs:
            m = order[: max(1, int(c * len(y)))]
            f = f1_from_cm(confusion(y[m], yhat[m]))
            acc.append(f["accuracy"])
            f1.append(f["macro_f1"])
        self.lines(fid, title, {"accuracy": (covs, acc), "macro-F1": (covs, f1)}, "coverage", "metric")

    def write_index(self):
        (self.out / "figure_index.json").write_text(json.dumps(self.index, indent=2), encoding="utf-8")
        return self.index


In [ ]:
%%writefile $REPO/tests/fixtures/aapl_8k_2002_full.txt
-----BEGIN PRIVACY-ENHANCED MESSAGE-----
Proc-Type: 2001,MIC-CLEAR
Originator-Name: webmaster@www.sec.gov
Originator-Key-Asymmetric:
 MFgwCgYEVQgBAQICAf8DSgAwRwJAW2sNKK9AVtBzYZmr6aGjlWyK3XmZv3dTINen
 TWSM7vrzLADbmYQaionwg5sDW3P6oaM5D3tdezXMm7z1T+B+twIDAQAB
MIC-Info: RSA-MD5,RSA,
 LVGztq9tcVtkr2g8/n6odgdlVo0f4zbPP4IliaGNbvTBsnx3X9mncmcTTA5yFP6Q
 ZuuC5GIN6TVbQQ4hHSykaw==

<SEC-DOCUMENT>0000912057-02-031595.txt : 20020813
<SEC-HEADER>0000912057-02-031595.hdr.sgml : 20020813
<ACCEPTANCE-DATETIME>20020813164730
ACCESSION NUMBER:		0000912057-02-031595
CONFORMED SUBMISSION TYPE:	8-K
PUBLIC DOCUMENT COUNT:		3
CONFORMED PERIOD OF REPORT:	20020808
ITEM INFORMATION:		Other events
ITEM INFORMATION:		Financial statements and exhibits
FILED AS OF DATE:		20020813

FILER:

	COMPANY DATA:	
		COMPANY CONFORMED NAME:			APPLE COMPUTER INC
		CENTRAL INDEX KEY:			0000320193
		STANDARD INDUSTRIAL CLASSIFICATION:	ELECTRONIC COMPUTERS [3571]
		IRS NUMBER:				942404110
		STATE OF INCORPORATION:			CA
		FISCAL YEAR END:			0930

	FILING VALUES:
		FORM TYPE:		8-K
		SEC ACT:		1934 Act
		SEC FILE NUMBER:	000-10030
		FILM NUMBER:		02730272

	BUSINESS ADDRESS:	
		STREET 1:		1 INFINITE LOOP
		CITY:			CUPERTINO
		STATE:			CA
		ZIP:			95014
		BUSINESS PHONE:		4089961010

	MAIL ADDRESS:	
		STREET 1:		ONE INFINITE LOOP
		CITY:			CUPERTINO
		STATE:			CA
		ZIP:			95014
</SEC-HEADER>
<DOCUMENT>
<TYPE>8-K
<SEQUENCE>1
<FILENAME>a2086495z8-k.htm
<DESCRIPTION>8-K
<TEXT>
<HTML>
<HEAD>
<TITLE>
</TITLE>
</HEAD>
<BODY BGCOLOR="#FFFFFF" LINK=BLUE  VLINK=PURPLE>
<BR>
<FONT SIZE=3 ><A HREF="#02PAL2033_1">QuickLinks</A></FONT>
<font size=3> -- Click here to rapidly navigate through this document</font>

<!-- TOC_END -->
<HR NOSHADE>
<HR NOSHADE>
<P ALIGN="CENTER"><FONT SIZE=5><B>SECURITIES AND EXCHANGE COMMISSION<BR>  </B></FONT><FONT SIZE=2><B>Washington, D. C. 20549  </B></FONT></P>

<HR NOSHADE ALIGN="CENTER" WIDTH="120">
<P ALIGN="CENTER"><FONT SIZE=5><B>FORM 8-K  </B></FONT></P>

<P ALIGN="CENTER"><FONT SIZE=5><B>CURRENT REPORT  </B></FONT></P>

<P ALIGN="CENTER"><FONT SIZE=2>Pursuant to Section&nbsp;13 or 15(d) of the Securities Exchange Act of 1934 </FONT></P>

<P ALIGN="CENTER"><FONT SIZE=2>Date
of Report (date of earliest event reported): </FONT><FONT SIZE=2><B>August&nbsp;8, 2002  </B></FONT></P>

<P ALIGN="CENTER"><FONT SIZE=5><B>Apple Computer,&nbsp;Inc.<BR>  </B></FONT><FONT SIZE=2>(Exact name of registrant as specified in its charter) </FONT></P>

<!-- User-specified TAGGED TABLE -->
<TABLE WIDTH="100%" BORDER=0 CELLSPACING=0 CELLPADDING=0>
<TR VALIGN="TOP">
<TD WIDTH="32%" ALIGN="CENTER"><FONT SIZE=2><B>California</B></FONT><FONT SIZE=2><BR>
(State or other jurisdiction of<BR>
incorporation or organization)</FONT></TD>
<TD WIDTH="3%"><FONT SIZE=2>&nbsp;</FONT></TD>
<TD WIDTH="32%" ALIGN="CENTER"><FONT SIZE=2><B>0-10030</B></FONT><FONT SIZE=2><BR>
(Commission File<BR>
Number)</FONT></TD>
<TD WIDTH="3%"><FONT SIZE=2>&nbsp;</FONT></TD>
<TD WIDTH="32%" ALIGN="CENTER"><FONT SIZE=2><B>94-2404110</B></FONT><FONT SIZE=2><BR>
(I.R.S. Employer<BR>
Identification No.)</FONT></TD>
</TR>
</TABLE>
<!-- end of user-specified TAGGED TABLE -->

<P ALIGN="CENTER"><FONT SIZE=2><B>1 Infinite Loop, Cupertino, California 95014<BR>  </B></FONT><FONT SIZE=2>(Address of principal executive offices) </FONT></P>

<P ALIGN="CENTER"><FONT SIZE=2>Registrant's
telephone number, including area code: </FONT><FONT SIZE=2><B>(408)&nbsp;996-1010</B></FONT></P>

<P ALIGN="CENTER"><FONT SIZE=2><B>Not Applicable<BR>  </B></FONT><FONT SIZE=2>(Former name or former address, if changed since last report.) </FONT></P>

<HR NOSHADE>
<HR NOSHADE>
<HR NOSHADE>
<!-- ZEQ.=1,SEQ=1,EFW="2086495",CP="APPLE COMPUTER, INC.",DN="1",CHK=89112,FOLIO='blank',FILE='DISK016:[02PAL3.02PAL2033]BA2033A.;3',USER='DPHAMA',CD='13-AUG-2002;13:59' -->
<!-- THIS IS THE END OF A COMPOSITION COMPONENT -->
<!-- TOC_END -->
<P ALIGN="CENTER"><FONT SIZE=2><A
NAME="de2033_information_included_in_this_report"> </A>
<A NAME="toc_de2033_1"> </A>
<BR></FONT><FONT SIZE=2><B>Information Included in this Report    <BR>  </B></FONT></P>

<P><FONT SIZE=2>Items
1 through 4, 6, 8 and 9 Not Applicable. </FONT></P>

<P><FONT SIZE=2><A
NAME="de2033_item_5._other_events."> </A>
<A NAME="toc_de2033_2"> </A>
<BR></FONT><FONT SIZE=2><B>Item 5.&nbsp;&nbsp;&nbsp;&nbsp;</B></FONT><FONT SIZE=2><B><I>Other Events.</I></B></FONT></P>

<P><FONT SIZE=2>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;On
August&nbsp;9, 2002, Apple Computer,&nbsp;Inc. submitted to the Securities and Exchange Commission the Statements under Oath of Principal Executive Officer and Principal Financial
Officer in accordance with the SEC's June&nbsp;27, 2002 Order requiring the filing of sworn statements pursuant to Section&nbsp;21(a)(1) of the Securities and Exchange Act of 1934. </FONT></P>

<P><FONT SIZE=2><A
NAME="de2033_item_7._financial_statements,___ite02795"> </A>
<A NAME="toc_de2033_3"> </A>
<BR></FONT><FONT SIZE=2><B>Item 7.&nbsp;&nbsp;&nbsp;&nbsp;</B></FONT><FONT SIZE=2><B><I>Financial Statements, Pro Forma Financial Information and Exhibits.</I></B></FONT></P>

<!-- User-specified TAGGED TABLE -->
<TABLE WIDTH="100%" BORDER=0 CELLSPACING=0 CELLPADDING=0>
<TR VALIGN="TOP">
<TD WIDTH="5%"><FONT SIZE=2>&nbsp;</FONT></TD>
<TD WIDTH="3%"><FONT SIZE=2>&nbsp;</FONT></TD>
<TD WIDTH="6%"><FONT SIZE=2>(c)</FONT></TD>
<TD WIDTH="3%"><FONT SIZE=2>&nbsp;</FONT></TD>
<TD WIDTH="83%"><FONT SIZE=2>Exhibits</FONT></TD>
</TR>
<TR VALIGN="TOP">
<TD WIDTH="5%"><FONT SIZE=2><BR>
&nbsp;</FONT></TD>
<TD WIDTH="3%"><FONT SIZE=2><BR>&nbsp;</FONT></TD>
<TD WIDTH="6%"><FONT SIZE=2><BR>
99.1.</FONT></TD>
<TD WIDTH="3%"><FONT SIZE=2><BR>&nbsp;</FONT></TD>
<TD WIDTH="83%"><FONT SIZE=2><BR>
Statement Under Oath of Steven&nbsp;P. Jobs, Principal Executive Officer of Apple Computer,&nbsp;Inc., Regarding Facts and Circumstances Relating to Exchange Act Filings.</FONT></TD>
</TR>
<TR VALIGN="TOP">
<TD WIDTH="5%"><FONT SIZE=2><BR>
&nbsp;</FONT></TD>
<TD WIDTH="3%"><FONT SIZE=2><BR>&nbsp;</FONT></TD>
<TD WIDTH="6%"><FONT SIZE=2><BR>
99.2.</FONT></TD>
<TD WIDTH="3%"><FONT SIZE=2><BR>&nbsp;</FONT></TD>
<TD WIDTH="83%"><FONT SIZE=2><BR>
Statement Under Oath of Fred&nbsp;D. Anderson, Principal Financial Officer of Apple Computer,&nbsp;Inc., Regarding Facts and Circumstances Relating to Exchange Act Filings.</FONT></TD>
</TR>
</TABLE>
<!-- end of user-specified TAGGED TABLE -->

<HR NOSHADE>
<!-- ZEQ.=1,SEQ=2,EFW="2086495",CP="APPLE COMPUTER, INC.",DN="1",CHK=1048186,FOLIO='blank',FILE='DISK016:[02PAL3.02PAL2033]DE2033A.;18',USER='DPHAMA',CD='13-AUG-2002;14:01' -->
<!-- THIS IS THE END OF A COMPOSITION COMPONENT -->
<!-- TOC_END -->
<P ALIGN="CENTER"><FONT SIZE=2><A
NAME="jc2033_signature"> </A>
<A NAME="toc_jc2033_1"> </A>
<BR></FONT><FONT SIZE=2><B>SIGNATURE    <BR>  </B></FONT></P>


<P><FONT SIZE=2>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Pursuant to the requirements of the Securities Exchange Act of 1934, the registrant has duly caused this report to be signed on its behalf by the undersigned,
thereunto duly authorized. </FONT></P>

<!-- User-specified TAGGED TABLE -->
<TABLE WIDTH="100%" BORDER=0 CELLSPACING=0 CELLPADDING=0>
<TR VALIGN="TOP">
<TD WIDTH="48%"><FONT SIZE=2>&nbsp;</FONT></TD>
<TD COLSPAN=2><FONT SIZE=2>APPLE COMPUTER,&nbsp;INC.</FONT></TD>
</TR>
<TR VALIGN="TOP">
<TD WIDTH="48%"><FONT SIZE=2><BR>
&nbsp;</FONT></TD>
<TD WIDTH="5%"><FONT SIZE=2><BR>
By:</FONT></TD>
<TD WIDTH="48%"><FONT SIZE=2><BR>
/s/&nbsp;&nbsp;</FONT><FONT SIZE=2>FRED ANDERSON</FONT><FONT SIZE=2>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;</FONT><HR NOSHADE><FONT SIZE=2> Executive Vice President and Chief Financial Officer</FONT></TD>
</TR>
</TABLE>
<!-- end of user-specified TAGGED TABLE -->


<P><FONT SIZE=2>Date:
August&nbsp;9, 2002 </FONT></P>

<HR NOSHADE>
<!-- ZEQ.=1,SEQ=3,EFW="2086495",CP="APPLE COMPUTER, INC.",DN="1",CHK=690683,FOLIO='blank',FILE='DISK016:[02PAL3.02PAL2033]JC2033A.;5',USER='DPHAMA',CD='13-AUG-2002;15:26' -->
<BR>
<!-- THIS IS THE END OF A COMPOSITION COMPONENT -->
<!-- TOC_END -->
<P ALIGN="CENTER"><FONT SIZE=2><A
NAME="ka2033_index_to_exhibits"> </A>
<A NAME="toc_ka2033_1"> </A>
<BR></FONT><FONT SIZE=2><B>INDEX TO EXHIBITS    <BR>  </B></FONT></P>

<!-- User-specified TAGGED TABLE -->
<TABLE WIDTH="100%" BORDER=0 CELLSPACING=0 CELLPADDING=0>
<TR VALIGN="BOTTOM">
<TH WIDTH="8%" ALIGN="LEFT"><FONT SIZE=1><B>Exhibit<BR> </B></FONT><HR NOSHADE></TH>
<TH WIDTH="3%"><FONT SIZE=1>&nbsp;</FONT></TH>
<TH WIDTH="81%" ALIGN="LEFT"><FONT SIZE=1><B>Document<BR> </B></FONT><HR NOSHADE></TH>
<TH WIDTH="3%"><FONT SIZE=1>&nbsp;</FONT></TH>
<TH WIDTH="5%" ALIGN="CENTER"><FONT SIZE=1><B>Page</B></FONT><HR NOSHADE></TH>
</TR>
<TR BGCOLOR="#CCEEFF" VALIGN="TOP">
<TD WIDTH="8%"><FONT SIZE=2>99.1</FONT></TD>
<TD WIDTH="3%"><FONT SIZE=2>&nbsp;</FONT></TD>
<TD WIDTH="81%"><FONT SIZE=2>Statement Under Oath of Steven&nbsp;P. Jobs, Principal Executive Officer of Apple Computer, Inc., Regarding Facts and Circumstances Relating to Exchange Act Filings.</FONT></TD>
<TD WIDTH="3%"><FONT SIZE=2>&nbsp;</FONT></TD>
<TD WIDTH="5%" VALIGN="BOTTOM"><FONT SIZE=2>&nbsp;</FONT></TD>
</TR>
<TR BGCOLOR="White" VALIGN="TOP">
<TD WIDTH="8%"><FONT SIZE=2><BR>
99.2</FONT></TD>
<TD WIDTH="3%"><FONT SIZE=2><BR>&nbsp;</FONT></TD>
<TD WIDTH="81%"><FONT SIZE=2><BR>
Statement Under Oath of Fred D. Anderson, Principal Financial Officer of Apple Computer, Inc., Regarding Facts and Circumstances Relating to Exchange Act Filings.</FONT></TD>
<TD WIDTH="3%"><FONT SIZE=2><BR>&nbsp;</FONT></TD>
<TD WIDTH="5%" VALIGN="BOTTOM"><FONT SIZE=2><BR>
&nbsp;</FONT></TD>
</TR>
</TABLE>
<!-- end of user-specified TAGGED TABLE -->

<HR NOSHADE>
<!-- ZEQ.=1,SEQ=4,EFW="2086495",CP="APPLE COMPUTER, INC.",DN="1",CHK=664657,FOLIO='blank',FILE='DISK016:[02PAL3.02PAL2033]KA2033A.;6',USER='DPHAMA',CD='13-AUG-2002;13:59' -->
<!-- THIS IS THE END OF A COMPOSITION COMPONENT -->
<BR>
<P><br><A NAME="02PAL2033_1">QuickLinks</A><br></P><!-- TOC_BEGIN -->
<FONT SIZE=2><A HREF="#toc_de2033_1">Information Included in this Report</A></FONT><BR>
<UL>
<FONT SIZE=2><A HREF="#toc_de2033_2">Item 5. Other Events.</A></FONT><BR>
<FONT SIZE=2><A HREF="#toc_de2033_3">Item 7. Financial Statements, Pro Forma Financial Information and Exhibits.</A></FONT><BR>
</UL>
<!-- TOC_BEGIN -->
<FONT SIZE=2><A HREF="#toc_jc2033_1">SIGNATURE</A></FONT><BR>
<!-- TOC_BEGIN -->
<FONT SIZE=2><A HREF="#toc_ka2033_1">INDEX TO EXHIBITS</A></FONT><BR>
<!-- SEQ=,FILE='QUICKLINK',USER=DNICHOL,SEQ=,EFW="2086495",CP="APPLE COMPUTER, INC.",DN="1" -->
<!-- TOCEXISTFLAG -->
</BODY>
</HTML>

</TEXT>
</DOCUMENT>
<DOCUMENT>
<TYPE>EX-99.1
<SEQUENCE>3
<FILENAME>a2086495zex-99_1.htm
<DESCRIPTION>EX-99.1
<TEXT>
<HTML>
<HEAD>
<TITLE>
</TITLE>
</HEAD>
<BODY BGCOLOR="#FFFFFF" LINK=BLUE  VLINK=PURPLE>
<BR>
<FONT SIZE=3 ><A HREF="#02PAL2033_2">QuickLinks</A></FONT>
<font size=3> -- Click here to rapidly navigate through this document</font>
<!-- TOC_END -->
<P ALIGN="RIGHT"><FONT SIZE=2><A
NAME="kc2033_exhibit_99.1"> </A>
<A NAME="toc_kc2033_1"> </A>
<BR></FONT><FONT SIZE=2><B>Exhibit&nbsp;99.1    <BR>  </B></FONT></P>

<P><FONT SIZE=2>Statement
Under Oath of Principal Executive Officer and Principal Financial Officer Regarding Facts and Circumstances Relating to Exchange Act Filings </FONT></P>

<P><FONT SIZE=2>I,
Steven P. Jobs, state and attest that: </FONT></P>

<P><FONT SIZE=2>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;(1)&nbsp;&nbsp;To
the best of my knowledge, based upon a review of the covered reports of Apple Computer,&nbsp;Inc., and, except as corrected or supplemented in a subsequent covered
report: </FONT></P>

<UL>
<DL compact>
<DT style='margin-bottom:-11pt;'><FONT SIZE=2>&#149;</FONT></DT><DD><FONT SIZE=2>no
covered report contained an untrue statement of a material fact as of the end of the period covered by such report (or in the case of a report on
Form&nbsp;8-K or definitive proxy materials, as of the date on which it was filed); and
<BR><BR></FONT></DD><DT style='margin-bottom:-11pt;'><FONT SIZE=2>&#149;</FONT></DT><DD><FONT SIZE=2>no
covered report omitted to state a material fact necessary to make the statements in the covered report, in light of the circumstances under which they
were made, not misleading as of the end of the period covered by such report (or in the case of a report on Form&nbsp;8-K or definitive proxy materials, as of the date on which it was
filed). </FONT></DD></DL>
</UL>

<P><FONT SIZE=2>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;(2)&nbsp;&nbsp;I
have reviewed the contents of this statement with the Company's audit committee. </FONT></P>


<P><FONT SIZE=2>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;(3)&nbsp;&nbsp;In
this statement under oath, each of the following, if filed on or before the date of this statement, is a "covered report": </FONT></P>

<UL>
<DL compact>
<DT style='margin-bottom:-11pt;'><FONT SIZE=2>&#149;</FONT></DT><DD><FONT SIZE=2>Apple
Computer,&nbsp;Inc. Annual Report on Form&nbsp;10-K for the fiscal year ended September&nbsp;29, 2001;
<BR><BR></FONT></DD><DT style='margin-bottom:-11pt;'><FONT SIZE=2>&#149;</FONT></DT><DD><FONT SIZE=2>all
reports on Form&nbsp;10-Q, all reports on Form&nbsp;8-K and all definitive proxy materials of Apple Computer,&nbsp;Inc.
filed with the Commission subsequent to the filing of the Form&nbsp;10-K identified above; and
<BR><BR></FONT></DD><DT style='margin-bottom:-11pt;'><FONT SIZE=2>&#149;</FONT></DT><DD><FONT SIZE=2>any
amendments to any of the foregoing. </FONT></DD></DL>
</UL>
<BR>

<!-- User-specified TAGGED TABLE -->
<TABLE WIDTH="76%" BORDER=0 CELLSPACING=0 CELLPADDING=0>
<TR VALIGN="BOTTOM">
<TD WIDTH="49%" VALIGN="TOP"><FONT SIZE=2>/s/&nbsp;&nbsp;</FONT><FONT SIZE=2>STEVEN P. JOBS</FONT><FONT SIZE=2>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;</FONT><HR NOSHADE><FONT SIZE=2> Steven P. Jobs<BR>
Chief Executive Officer<BR>
August&nbsp;8, 2002</FONT></TD>
<TD WIDTH="3%" VALIGN="TOP"><FONT SIZE=2>&nbsp;</FONT></TD>
<TD WIDTH="49%" VALIGN="TOP"><FONT SIZE=2>&nbsp;</FONT></TD>
</TR>
<TR VALIGN="BOTTOM">
<TD WIDTH="49%" VALIGN="TOP"><FONT SIZE=2>&nbsp;</FONT></TD>
<TD WIDTH="3%" VALIGN="TOP"><FONT SIZE=2>&nbsp;</FONT></TD>
<TD WIDTH="49%" VALIGN="TOP"><FONT SIZE=2>&nbsp;</FONT></TD>
</TR>
<TR VALIGN="BOTTOM">
<TD WIDTH="49%" VALIGN="TOP"><FONT SIZE=2>Subscribed and sworn to<BR>
before me this 8<SUP>th</SUP> day of August, 2002.</FONT></TD>
<TD WIDTH="3%" VALIGN="TOP"><FONT SIZE=2>&nbsp;</FONT></TD>
<TD WIDTH="49%" VALIGN="TOP"><FONT SIZE=2>&nbsp;</FONT></TD>
</TR>
<TR VALIGN="BOTTOM">
<TD WIDTH="49%" VALIGN="TOP"><FONT SIZE=2>&nbsp;</FONT></TD>
<TD WIDTH="3%" VALIGN="TOP"><FONT SIZE=2>&nbsp;</FONT></TD>
<TD WIDTH="49%" VALIGN="TOP"><FONT SIZE=2>&nbsp;</FONT></TD>
</TR>
<TR VALIGN="BOTTOM">
<TD WIDTH="49%" VALIGN="TOP"><FONT SIZE=2>/s/&nbsp;&nbsp;</FONT><FONT SIZE=2>MARCIA M. STUBENDORFF</FONT><FONT SIZE=2>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;</FONT><HR NOSHADE><FONT SIZE=2> Notary Public<BR>
My Commission Expires: May&nbsp;21, 2006</FONT></TD>
<TD WIDTH="3%" VALIGN="TOP"><FONT SIZE=2>&nbsp;</FONT></TD>
<TD WIDTH="49%" VALIGN="TOP"><FONT SIZE=2>&nbsp;</FONT></TD>
</TR>
</TABLE>
<!-- end of user-specified TAGGED TABLE -->

<HR NOSHADE>
<!-- ZEQ.=1,SEQ=1,EFW="2086495",CP="APPLE COMPUTER, INC.",DN="2",CHK=923045,FOLIO='blank',FILE='DISK016:[02PAL3.02PAL2033]KC2033A.;6',USER='DPHAMA',CD='13-AUG-2002;13:59' -->
<!-- THIS IS THE END OF A COMPOSITION COMPONENT -->
<BR>
<P><br><A NAME="02PAL2033_2">QuickLinks</A><br></P><!-- TOC_BEGIN -->
<UL>
<FONT SIZE=2><A HREF="#toc_kc2033_1">Exhibit 99.1</A></FONT><BR>
</UL>
<!-- SEQ=,FILE='QUICKLINK',USER=DNICHOL,SEQ=,EFW="2086495",CP="APPLE COMPUTER, INC.",DN="2" -->
<!-- TOCEXISTFLAG -->
</BODY>
</HTML>

</TEXT>
</DOCUMENT>
<DOCUMENT>
<TYPE>EX-99.2
<SEQUENCE>4
<FILENAME>a2086495zex-99_2.htm
<DESCRIPTION>EX-99.2
<TEXT>
<HTML>
<HEAD>
<TITLE>
</TITLE>
</HEAD>
<BODY BGCOLOR="#FFFFFF" LINK=BLUE  VLINK=PURPLE>
<BR>
<FONT SIZE=3 ><A HREF="#02PAL2033_3">QuickLinks</A></FONT>
<font size=3> -- Click here to rapidly navigate through this document</font>
<!-- TOC_END -->
<P ALIGN="RIGHT"><FONT SIZE=2><A
NAME="ke2033_exhibit_99.2"> </A>
<A NAME="toc_ke2033_1"> </A>
<BR></FONT><FONT SIZE=2><B>Exhibit&nbsp;99.2    <BR>  </B></FONT></P>

<P><FONT SIZE=2>Statement
Under Oath of Principal Executive Officer and Principal Financial Officer Regarding Facts and Circumstances Relating to Exchange Act Filings </FONT></P>

<P><FONT SIZE=2>I,
Fred D. Anderson, state and attest that: </FONT></P>

<P><FONT SIZE=2>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;(1)&nbsp;&nbsp;To
the best of my knowledge, based upon a review of the covered reports of Apple Computer,&nbsp;Inc., and, except as corrected or supplemented in a subsequent covered
report: </FONT></P>

<UL>
<DL compact>
<DT style='margin-bottom:-11pt;'><FONT SIZE=2>&#149;</FONT></DT><DD><FONT SIZE=2>no
covered report contained an untrue statement of a material fact as of the end of the period covered by such report (or in the case of a report on
Form&nbsp;8-K or definitive proxy materials, as of the date on which it was filed); and
<BR><BR></FONT></DD><DT style='margin-bottom:-11pt;'><FONT SIZE=2>&#149;</FONT></DT><DD><FONT SIZE=2>no
covered report omitted to state a material fact necessary to make the statements in the covered report, in light of the circumstances under which they
were made, not misleading as of the end of the period covered by such report (or in the case of a report on Form&nbsp;8-K or definitive proxy materials, as of the date on which it was
filed). </FONT></DD></DL>
</UL>

<P><FONT SIZE=2>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;(2)&nbsp;&nbsp;I
have reviewed the contents of this statement with the Company's audit committee. </FONT></P>


<P><FONT SIZE=2>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;(3)&nbsp;&nbsp;In
this statement under oath, each of the following, if filed on or before the date of this statement, is a "covered report": </FONT></P>

<UL>
<DL compact>
<DT style='margin-bottom:-11pt;'><FONT SIZE=2>&#149;</FONT></DT><DD><FONT SIZE=2>Apple
Computer,&nbsp;Inc. Annual Report on Form&nbsp;10-K for the fiscal year ended September&nbsp;29, 2001;
<BR><BR></FONT></DD><DT style='margin-bottom:-11pt;'><FONT SIZE=2>&#149;</FONT></DT><DD><FONT SIZE=2>all
reports on Form&nbsp;10-Q, all reports on Form&nbsp;8-K and all definitive proxy materials of Apple Computer,&nbsp;Inc.
filed with the Commission subsequent to the filing of the Form&nbsp;10-K identified above; and
<BR><BR></FONT></DD><DT style='margin-bottom:-11pt;'><FONT SIZE=2>&#149;</FONT></DT><DD><FONT SIZE=2>any
amendments to any of the foregoing. </FONT></DD></DL>
</UL>
<BR>

<!-- User-specified TAGGED TABLE -->
<TABLE WIDTH="76%" BORDER=0 CELLSPACING=0 CELLPADDING=0>
<TR VALIGN="BOTTOM">
<TD WIDTH="49%" VALIGN="TOP"><FONT SIZE=2>/s/&nbsp;&nbsp;</FONT><FONT SIZE=2>FRED D. ANDERSON</FONT><FONT SIZE=2>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;</FONT><HR NOSHADE><FONT SIZE=2> Fred D. Anderson<BR>
Chief Financial Officer<BR>
August&nbsp;8, 2002</FONT></TD>
<TD WIDTH="3%" VALIGN="TOP"><FONT SIZE=2>&nbsp;</FONT></TD>
<TD WIDTH="49%" VALIGN="TOP"><FONT SIZE=2>&nbsp;</FONT></TD>
</TR>
<TR VALIGN="BOTTOM">
<TD WIDTH="49%" VALIGN="TOP"><FONT SIZE=2>&nbsp;</FONT></TD>
<TD WIDTH="3%" VALIGN="TOP"><FONT SIZE=2>&nbsp;</FONT></TD>
<TD WIDTH="49%" VALIGN="TOP"><FONT SIZE=2>&nbsp;</FONT></TD>
</TR>
<TR VALIGN="BOTTOM">
<TD WIDTH="49%" VALIGN="TOP"><FONT SIZE=2>Subscribed and sworn to<BR>
before me this 8<SUP>th</SUP> day of August, 2002.</FONT></TD>
<TD WIDTH="3%" VALIGN="TOP"><FONT SIZE=2>&nbsp;</FONT></TD>
<TD WIDTH="49%" VALIGN="TOP"><FONT SIZE=2>&nbsp;</FONT></TD>
</TR>
<TR VALIGN="BOTTOM">
<TD WIDTH="49%" VALIGN="TOP"><FONT SIZE=2>&nbsp;</FONT></TD>
<TD WIDTH="3%" VALIGN="TOP"><FONT SIZE=2>&nbsp;</FONT></TD>
<TD WIDTH="49%" VALIGN="TOP"><FONT SIZE=2>&nbsp;</FONT></TD>
</TR>
<TR VALIGN="BOTTOM">
<TD WIDTH="49%" VALIGN="TOP"><FONT SIZE=2>/s/&nbsp;&nbsp;</FONT><FONT SIZE=2>MARCIA M. STUBENDORFF</FONT><FONT SIZE=2>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;</FONT><HR NOSHADE><FONT SIZE=2> Notary Public<BR>
My Commission Expires: May&nbsp;21, 2006</FONT></TD>
<TD WIDTH="3%" VALIGN="TOP"><FONT SIZE=2>&nbsp;</FONT></TD>
<TD WIDTH="49%" VALIGN="TOP"><FONT SIZE=2>&nbsp;</FONT></TD>
</TR>
</TABLE>
<!-- end of user-specified TAGGED TABLE -->

<HR NOSHADE>
<!-- ZEQ.=1,SEQ=1,EFW="2086495",CP="APPLE COMPUTER, INC.",DN="3",CHK=361411,FOLIO='blank',FILE='DISK016:[02PAL3.02PAL2033]KE2033A.;5',USER='DPHAMA',CD='13-AUG-2002;13:59' -->
<!-- THIS IS THE END OF A COMPOSITION COMPONENT -->
<BR>
<P><br><A NAME="02PAL2033_3">QuickLinks</A><br></P><!-- TOC_BEGIN -->
<UL>
<FONT SIZE=2><A HREF="#toc_ke2033_1">Exhibit 99.2</A></FONT><BR>
</UL>
<!-- SEQ=,FILE='QUICKLINK',USER=DNICHOL,SEQ=,EFW="2086495",CP="APPLE COMPUTER, INC.",DN="3" -->
<!-- TOCEXISTFLAG -->
</BODY>
</HTML>

</TEXT>
</DOCUMENT>
</SEC-DOCUMENT>
-----END PRIVACY-ENHANCED MESSAGE-----


In [ ]:
%%writefile $REPO/tests/fixtures/aapl_8k_20260730_header.html
<HTML><HEAD><TITLE>SEC EDGAR Submission 0000320193-26-000018</TITLE>
<!--
<SEC-HEADER>0000320193-26-000018.hdr.sgml : 20260730
<ACCEPTANCE-DATETIME>20260730163028
<ACCESSION-NUMBER>0000320193-26-000018
<TYPE>8-K
<PUBLIC-DOCUMENT-COUNT>14
<PERIOD>20260730
<ITEMS>2.02
<ITEMS>9.01
<FILING-DATE>20260730
<DATE-OF-FILING-DATE-CHANGE>20260730
<FILER>
<COMPANY-DATA>
<CONFORMED-NAME>Apple Inc.
<CIK>0000320193
<ASSIGNED-SIC>3571
<ORGANIZATION-NAME>06 Technology
<IRS-NUMBER>942404110
<STATE-OF-INCORPORATION>CA
<FISCAL-YEAR-END>0926
</COMPANY-DATA>
<FILING-VALUES>
<FORM-TYPE>8-K
<ACT>34
<FILE-NUMBER>001-36743
<FILM-NUMBER>261223821
</FILING-VALUES>
<BUSINESS-ADDRESS>
<STREET1>ONE APPLE PARK WAY
<CITY>CUPERTINO
<STATE>CA
<ZIP>95014
<PHONE>(408) 996-1010
</BUSINESS-ADDRESS>
<MAIL-ADDRESS>
<STREET1>ONE APPLE PARK WAY
<CITY>CUPERTINO
<STATE>CA
<ZIP>95014
</MAIL-ADDRESS>
<FORMER-COMPANY>
<FORMER-CONFORMED-NAME>APPLE INC
<DATE-CHANGED>20070109
</FORMER-COMPANY>
<FORMER-COMPANY>
<FORMER-CONFORMED-NAME>APPLE COMPUTER INC
<DATE-CHANGED>19970808
</FORMER-COMPANY>
</FILER>
</SEC-HEADER>
-->
</HEAD><BODY>
<PRE>&lt;SEC-DOCUMENT&gt;0000320193-26-000018-index.html : 20260730
&lt;SEC-HEADER&gt;0000320193-26-000018.hdr.sgml : 20260730
&lt;ACCEPTANCE-DATETIME&gt;20260730163028
ACCESSION NUMBER:		0000320193-26-000018
CONFORMED SUBMISSION TYPE:	8-K
PUBLIC DOCUMENT COUNT:		14
CONFORMED PERIOD OF REPORT:	20260730
ITEM INFORMATION:		Results of Operations and Financial Condition
ITEM INFORMATION:		Financial Statements and Exhibits
FILED AS OF DATE:		20260730
DATE AS OF CHANGE:		20260730

FILER:

	COMPANY DATA:	
		COMPANY CONFORMED NAME:			Apple Inc.
		CENTRAL INDEX KEY:			0000320193
		STANDARD INDUSTRIAL CLASSIFICATION:	ELECTRONIC COMPUTERS [3571]
		ORGANIZATION NAME:           	06 Technology
		EIN:				942404110
		STATE OF INCORPORATION:			CA
		FISCAL YEAR END:			0926

	FILING VALUES:
		FORM TYPE:		8-K
		SEC ACT:		1934 Act
		SEC FILE NUMBER:	001-36743
		FILM NUMBER:		261223821

	BUSINESS ADDRESS:	
		STREET 1:		ONE APPLE PARK WAY
		CITY:			CUPERTINO
		STATE:			CA
		ZIP:			95014
		BUSINESS PHONE:		(408) 996-1010

	MAIL ADDRESS:	
		STREET 1:		ONE APPLE PARK WAY
		CITY:			CUPERTINO
		STATE:			CA
		ZIP:			95014

	FORMER COMPANY:	
		FORMER CONFORMED NAME:	APPLE INC
		DATE OF NAME CHANGE:	20070109

	FORMER COMPANY:	
		FORMER CONFORMED NAME:	APPLE COMPUTER INC
		DATE OF NAME CHANGE:	19970808
&lt;/SEC-HEADER&gt;
&lt;DOCUMENT&gt;
&lt;TYPE&gt;8-K
&lt;SEQUENCE&gt;1
&lt;FILENAME&gt;aapl-20260730.htm
&lt;DESCRIPTION&gt;8-K
&lt;TEXT&gt;
<a href="aapl-20260730.htm">Document 1 - file: aapl-20260730.htm</a><br>
&lt;/DOCUMENT&gt;
&lt;DOCUMENT&gt;
&lt;TYPE&gt;EX-99.1
&lt;SEQUENCE&gt;2
&lt;FILENAME&gt;a8-kex991q3202606272026.htm
&lt;DESCRIPTION&gt;EX-99.1
&lt;TEXT&gt;
<a href="a8-kex991q3202606272026.htm">Document 2 - file: a8-kex991q3202606272026.htm</a><br>
&lt;/DOCUMENT&gt;
&lt;DOCUMENT&gt;
&lt;TYPE&gt;EX-101.SCH
&lt;SEQUENCE&gt;3
&lt;FILENAME&gt;aapl-20260730.xsd
&lt;DESCRIPTION&gt;XBRL TAXONOMY EXTENSION SCHEMA DOCUMENT
&lt;TEXT&gt;
<a href="aapl-20260730.xsd">Document 3 - file: aapl-20260730.xsd</a><br>
&lt;/DOCUMENT&gt;
&lt;DOCUMENT&gt;
&lt;TYPE&gt;EX-101.DEF
&lt;SEQUENCE&gt;4
&lt;FILENAME&gt;aapl-20260730_def.xml
&lt;DESCRIPTION&gt;XBRL TAXONOMY EXTENSION DEFINITION LINKBASE DOCUMENT
&lt;TEXT&gt;
<a href="aapl-20260730_def.xml">Document 4 - file: aapl-20260730_def.xml</a><br>
&lt;/DOCUMENT&gt;
&lt;DOCUMENT&gt;
&lt;TYPE&gt;EX-101.LAB
&lt;SEQUENCE&gt;5
&lt;FILENAME&gt;aapl-20260730_lab.xml
&lt;DESCRIPTION&gt;XBRL TAXONOMY EXTENSION LABEL LINKBASE DOCUMENT
&lt;TEXT&gt;
<a href="aapl-20260730_lab.xml">Document 5 - file: aapl-20260730_lab.xml</a><br>
&lt;/DOCUMENT&gt;
&lt;DOCUMENT&gt;
&lt;TYPE&gt;EX-101.PRE
&lt;SEQUENCE&gt;6
&lt;FILENAME&gt;aapl-20260730_pre.xml
&lt;DESCRIPTION&gt;XBRL TAXONOMY EXTENSION PRESENTATION LINKBASE DOCUMENT
&lt;TEXT&gt;
<a href="aapl-20260730_pre.xml">Document 6 - file: aapl-20260730_pre.xml</a><br>
&lt;/DOCUMENT&gt;
&lt;DOCUMENT&gt;
&lt;TYPE&gt;GRAPHIC
&lt;SEQUENCE&gt;7
&lt;FILENAME&gt;aapl-20260730_g1.jpg
&lt;TEXT&gt;
<a href="aapl-20260730_g1.jpg">Document 7 - file: aapl-20260730_g1.jpg</a><br>
&lt;/DOCUMENT&gt;
&lt;DOCUMENT&gt;
&lt;TYPE&gt;XML
&lt;SEQUENCE&gt;9
&lt;FILENAME&gt;R1.htm
&lt;DESCRIPTION&gt;IDEA: XBRL DOCUMENT
&lt;TEXT&gt;
<a href="R1.htm">Document 9 - file: R1.htm</a><br>
&lt;/DOCUMENT&gt;
&lt;DOCUMENT&gt;
&lt;TYPE&gt;XML
&lt;SEQUENCE&gt;10
&lt;FILENAME&gt;report.css
&lt;DESCRIPTION&gt;IDEA: XBRL DOCUMENT
&lt;TEXT&gt;
<a href="report.css">Document 10 - file: report.css</a><br>
&lt;/DOCUMENT&gt;
&lt;DOCUMENT&gt;
&lt;TYPE&gt;XML
&lt;SEQUENCE&gt;11
&lt;FILENAME&gt;Show.js
&lt;DESCRIPTION&gt;IDEA: XBRL DOCUMENT
&lt;TEXT&gt;
<a href="Show.js">Document 11 - file: Show.js</a><br>
&lt;/DOCUMENT&gt;
&lt;DOCUMENT&gt;
&lt;TYPE&gt;XML
&lt;SEQUENCE&gt;13
&lt;FILENAME&gt;FilingSummary.xml
&lt;DESCRIPTION&gt;IDEA: XBRL DOCUMENT
&lt;TEXT&gt;
<a href="xsl/FilingSummary.xml">Document 13 - file: FilingSummary.html</a><br>
<a href="FilingSummary.xml">Document 13 - RAW XML: FilingSummary.xml</a><br>
&lt;/DOCUMENT&gt;
&lt;DOCUMENT&gt;
&lt;TYPE&gt;JSON
&lt;SEQUENCE&gt;15
&lt;FILENAME&gt;MetaLinks.json
&lt;DESCRIPTION&gt;IDEA: XBRL DOCUMENT
&lt;TEXT&gt;
<a href="MetaLinks.json">Document 15 - file: MetaLinks.json</a><br>
&lt;/DOCUMENT&gt;
&lt;DOCUMENT&gt;
&lt;TYPE&gt;ZIP
&lt;SEQUENCE&gt;16
&lt;FILENAME&gt;0000320193-26-000018-xbrl.zip
&lt;DESCRIPTION&gt;IDEA: XBRL DOCUMENT
&lt;TEXT&gt;
<a href="0000320193-26-000018-xbrl.zip">Document 16 - file: 0000320193-26-000018-xbrl.zip</a><br>
&lt;/DOCUMENT&gt;
&lt;DOCUMENT&gt;
&lt;TYPE&gt;XML
&lt;SEQUENCE&gt;17
&lt;FILENAME&gt;aapl-20260730_htm.xml
&lt;DESCRIPTION&gt;IDEA: XBRL DOCUMENT
&lt;TEXT&gt;
<a href="xsl/aapl-20260730_htm.xml">Document 17 - file: aapl-20260730_htm.html</a><br>
<a href="aapl-20260730_htm.xml">Document 17 - RAW XML: aapl-20260730_htm.xml</a><br>
&lt;/DOCUMENT&gt;
&lt;/SEC-DOCUMENT&gt;</PRE><script type="text/javascript"  src="/Qp9qZ/J_/s/d/a6FX2ZB7lIP/uEV36Gh

In [ ]:
%%writefile $REPO/tests/integration/test_offline_smoke.py
"""Offline integration smoke (SPEC.md §18.2, gate G1): S00 -> S12 and S20 -> S24 on synthetic fixtures, CPU only,
no network, no 7B model (addendum P1.5). TF-IDF stands in for embeddings."""
import json
from pathlib import Path

import pandas as pd
import pytest

from secprice import pipeline as P
from secprice.testing.fixtures import make_world, offline_overrides


@pytest.fixture(scope="module")
def world(tmp_path_factory):
    root = tmp_path_factory.mktemp("offline")
    w = make_world(root)
    return root, w


def test_offline_smoke_end_to_end(world):
    root, w = world
    services = {"fixture_root": w["sec"], "factors": w["factors"]}
    stages = ["S00", "S01", "S02", "S03", "S04", "S05", "S06", "S07", "S08", "S09", "S10", "S11", "S12", "S20", "S21", "S24"]
    res = P.run(stages, "smoke", offline_overrides(), data_root=root, hot_root=root / "_hot", services=services)
    assert all(v in ("done", "skipped") for v in res.values()), res
    rep = root / "smoke" / "reports" / "results_summary.md"
    assert rep.exists() and "Smallest defensible claim" in rep.read_text(encoding="utf-8")
    assert (root / "smoke" / "reports" / "results_report.html").exists()
    la = pd.read_csv(root / "smoke" / "reports" / "leakage_audit.csv")
    assert (la.status != "fail").all()
    st = P.status("smoke", offline_overrides(), data_root=root)
    assert set(st[st.stage.isin(stages)].status) == {"done"}


def test_per_horizon_censoring(world):
    """SPEC.md §18.1 #7: a target is censored exactly when its exit session is after the usable cutoff; censored rows carry
    no label; longer horizons are censored at least as often as shorter ones."""
    root, w = world
    services = {"fixture_root": w["sec"], "factors": w["factors"]}
    P.run(["S08"], "smoke", offline_overrides(), data_root=root, hot_root=root / "_hot", services=services, with_deps=True)
    r = P.make_runner("smoke", offline_overrides(), root, root / "_hot")
    tg = pd.read_parquet(root / "smoke" / "S08" / r.stage_hash("S08") / "targets.parquet")
    uc = json.loads((root / "smoke" / "S07" / r.stage_hash("S07") / "usable_cutoff.json").read_text())
    cut = int(uc["usable_cutoff_idx"])
    assert (tg.censored == (tg.exit_idx > cut) | tg.censored).all()
    assert (tg[tg.exit_idx > cut].censored).all()
    assert tg[tg.censored].label3.isna().all() and tg[tg.censored].ar_mm.isna().all()
    share = tg[tg.target_def == "T_trade"].groupby("horizon").censored.mean().sort_index()
    assert share.is_monotonic_increasing, share.to_dict()
    assert share.iloc[-1] > 0, "the fixture should censor the longest horizon near the usable cutoff"


In [ ]:
%%writefile $REPO/tests/integration/test_online_smoke.py
"""Online smoke (SPEC.md §18.2): the smoke chain S00 -> S08 against the live SEC and our price inputs for one firm.
Marked `network` and skipped unless SECPRICE_RUN_NETWORK=1, SEC_USER_AGENT and SECPRICE_NETWORK_INPUTS (folder with the
owner's inputs: ohlcv_1day/, yf_frozen.parquet, hf_census_*.parquet, sp500/Siccodes/factor caches) are set."""
import os
import shutil
from pathlib import Path

import pandas as pd
import pytest

from secprice import pipeline as P

pytestmark = pytest.mark.network
NEED = os.environ.get("SECPRICE_RUN_NETWORK") == "1" and os.environ.get("SEC_USER_AGENT") and os.environ.get("SECPRICE_NETWORK_INPUTS")


@pytest.mark.skipif(not NEED, reason="set SECPRICE_RUN_NETWORK=1, SEC_USER_AGENT and SECPRICE_NETWORK_INPUTS to run")
def test_online_smoke_one_firm(tmp_path):
    src = Path(os.environ["SECPRICE_NETWORK_INPUTS"])
    inp = tmp_path / "inputs"
    (inp / "ohlcv_1day").mkdir(parents=True)
    for f in src.glob("*.*"):
        if f.is_file():
            shutil.copy2(f, inp / f.name)
    shutil.copy2(src / "ohlcv_1day" / "AAPL.parquet", inp / "ohlcv_1day" / "AAPL.parquet")
    if (src / "lexicons").exists():
        shutil.copytree(src / "lexicons", inp / "lexicons")
    ov = {"universe": {"n_stocks": 1, "fixed_tickers": ["AAPL"]}, "study": {"start": "2022-07-01", "end": "2023-06-30"},
          "splits": {"dev_start": "2023-03-01", "locked_test_start": "2023-05-01"},
          "market": {"warmup_start": "2021-06-01"}, "smoke": {"max_filings_per_firm_form": 4}, "hf": {"qa_enabled": False}}
    res = P.run(["S00", "S01", "S02", "S03", "S04", "S05", "S06", "S07", "S08"], "smoke", ov, data_root=tmp_path,
                hot_root=tmp_path / "_hot")
    assert all(v == "done" for v in res.values()), res
    r = P.make_runner("smoke", ov, tmp_path, tmp_path / "_hot")
    led = pd.read_parquet(tmp_path / "smoke" / "S05" / r.stage_hash("S05") / "parse_ledger.parquet")
    assert len(led) > 0 and (led.status == "ok").mean() > 0.9
    assert led.header_acceptance_raw.notna().all(), "every filing's SGML acceptance time must be read (D-047)"
    ev = pd.read_parquet(tmp_path / "smoke" / "S07" / r.stage_hash("S07") / "events.parquet")
    assert len(ev) > 0 and (ev.ts_quality == "A").all()


In [ ]:
%%writefile $REPO/tests/integration/test_s05_budget.py
"""S05 stops cleanly at its wall-clock budget (StageIncomplete) and a second run resumes and completes."""
import pytest

from secprice import pipeline as P
from secprice.logging_utils import StageIncomplete
from secprice.testing.fixtures import make_world, offline_overrides


def test_s05_wall_budget_then_resume(tmp_path):
    w = make_world(tmp_path)
    services = {"fixture_root": w["sec"], "factors": w["factors"]}
    P.run(["S00", "S01", "S02", "S03", "S04"], "smoke", offline_overrides(), data_root=tmp_path, hot_root=tmp_path / "_hot",
          services=services)
    ov = offline_overrides()
    ov["sec"] = {**ov.get("sec", {}), "s05_max_wall_hours": 1e-9}
    with pytest.raises(StageIncomplete):
        P.run(["S05"], "smoke", ov, data_root=tmp_path, hot_root=tmp_path / "_hot", services=services)
    res = P.run(["S05"], "smoke", offline_overrides(), data_root=tmp_path, hot_root=tmp_path / "_hot", services=services)
    assert res["S05"] == "done"


def test_s05_rerun_retries_only_failed_filings(tmp_path):
    """A re-run re-fetches only filings whose first attempt ended in a transient error; other rows are untouched."""
    import pandas as pd
    w = make_world(tmp_path)
    services = {"fixture_root": w["sec"], "factors": w["factors"]}
    P.run(["S00", "S01", "S02", "S03", "S04", "S05"], "smoke", offline_overrides(), data_root=tmp_path, hot_root=tmp_path / "_hot",
          services=services)
    r = P.make_runner("smoke", offline_overrides(), tmp_path, tmp_path / "_hot")
    led_p = sorted((tmp_path / "smoke" / "S05" / r.stage_hash("S05") / "ledger").glob("*.parquet"))[0]
    led = pd.read_parquet(led_p)
    victim = led.accession.iloc[0]
    before_other = led[led.accession != victim].reset_index(drop=True)
    led.loc[led.accession == victim, ["status", "error"]] = ["error", "RuntimeError: HTTP 503 after 6 attempts"]
    led.to_parquet(led_p, index=False)
    P.run(["S05"], "smoke", offline_overrides(), data_root=tmp_path, hot_root=tmp_path / "_hot", services=services, force=True)
    after = pd.read_parquet(led_p)
    assert after.loc[after.accession == victim, "status"].iloc[0] == "ok"
    assert len(after) == len(led) and after.accession.is_unique
    pd.testing.assert_frame_equal(after[after.accession != victim].reset_index(drop=True)[before_other.columns], before_other)


In [ ]:
%%writefile $REPO/tests/unit/test_accounting.py
"""SPEC.md §18.1 #8 (SUE) and #9 (EAR); addendum P1.4 acceptance test (3 / 4 / 8 seasonal differences)."""
import numpy as np
import pandas as pd
import pytest

from secprice.features.accounting import _qkey, compute_sue_table, quarterly_eps, sue_for
from secprice.features.registry import ear_rule_violations, sue_rule_violations

ET = "America/New_York"


def _hist(n_quarters, start_q=2000 * 4):
    rng = np.random.default_rng(3)
    return pd.DataFrame({"qkey": np.arange(start_q, start_q + n_quarters), "value": 1 + rng.normal(0, 0.1, n_quarters)})


@pytest.mark.parametrize("n_prior,expect_ok,expect_n", [(4 + 3, False, 3), (4 + 4, True, 4), (12, True, 8)])
def test_sue_seasonal_difference_counts(n_prior, expect_ok, expect_n):
    h = _hist(n_prior)
    qk = int(h.qkey.max()) + 1
    r = sue_for(1.5, qk, h, 4, 8)
    assert r["n_prior_diffs"] == expect_n                 # the number of prior observations is always recorded
    assert np.isfinite(r["sue"]) == expect_ok
    if not expect_ok:
        assert r["sue_reason"] == "only_3_prior_diffs"     # unavailable, never fabricated or silently shortened


def test_ytd_to_quarterly_and_first_reported():
    pub = lambda s: pd.Timestamp(s).tz_localize(ET)  # noqa: E731
    f = pd.DataFrame([
        # Q1-Q3 direct quarterly facts, FY + 9M YTD in the 10-K -> Q4 derived
        ("EarningsPerShareDiluted", "USD/shares", 1.0, "2020-01-01", "2020-03-31", "a1", pub("2020-05-01 16:00")),
        ("EarningsPerShareDiluted", "USD/shares", 1.1, "2020-04-01", "2020-06-30", "a2", pub("2020-08-01 16:00")),
        ("EarningsPerShareDiluted", "USD/shares", 1.2, "2020-07-01", "2020-09-30", "a3", pub("2020-11-01 16:00")),
        ("EarningsPerShareDiluted", "USD/shares", 3.3, "2020-01-01", "2020-09-30", "a3", pub("2020-11-01 16:00")),
        ("EarningsPerShareDiluted", "USD/shares", 4.6, "2020-01-01", "2020-12-31", "a4", pub("2021-02-20 16:00")),
        # a later restatement of Q1 must not replace the first-reported value
        ("EarningsPerShareDiluted", "USD/shares", 0.7, "2020-01-01", "2020-03-31", "a9", pub("2021-05-01 16:00")),
    ], columns=["tag", "unit", "value", "period_start", "period_end", "accession", "public_available_time"])
    f["cik"], f["form"] = 1, "10-Q"
    q = quarterly_eps(f)
    q1 = q[q.period_end == pd.Timestamp("2020-03-31")]
    assert q1.value.iloc[0] == pytest.approx(1.0) and q1.source_accession.iloc[0] == "a1"
    q4 = q[q.period_end == pd.Timestamp("2020-12-31")]
    assert q4.value.iloc[0] == pytest.approx(1.3) and q4.derivation.iloc[0] == "fy_minus_9m"


def _xbrl_hist(cik=1):
    rows = []
    for k in range(12):
        pe = pd.Timestamp("2017-03-31") + pd.offsets.QuarterEnd(k)
        rows.append({"cik": cik, "qkey": int(_qkey(pd.Series([pe])).iloc[0]), "value": 1 + 0.05 * k,
                     "source_accession": f"x{k}", "public_available_time": (pe + pd.Timedelta(days=40)).tz_localize(ET)})
    return pd.DataFrame(rows)


def test_release_eps_path_and_later_cluster_rejection():
    xh = _xbrl_hist()
    pe = pd.Timestamp("2020-03-31")
    qk = int(_qkey(pd.Series([pe])).iloc[0])
    dec = pd.Timestamp("2020-04-20 17:00").tz_localize(ET)
    # a later 10-Q of the same cluster reports the same quarter AFTER the release: must not enter
    later = pd.DataFrame([{"cik": 1, "qkey": qk, "value": 9.9, "source_accession": "later10q",
                           "public_available_time": pd.Timestamp("2020-05-05 16:00").tz_localize(ET)}])
    ev = pd.DataFrame([{"accession": "rel", "cik": 1, "kind": "release", "decision_time": dec, "eps_q": 1.9, "qkey": qk}])
    sue, src = compute_sue_table(ev, pd.concat([xh, later]), xh.iloc[:0], 4, 8)
    assert sue.eps_q_source.iloc[0] == "release_exhibit" and sue.eps_q.iloc[0] == pytest.approx(1.9)
    assert "later10q" not in set(src.source_accession)
    assert (pd.to_datetime(src.source_public_time) < dec).all()


def test_registry_rejects_later_cluster_sue_source():
    events = pd.DataFrame([
        {"event_id": "e_rel", "accessions": ["rel"], "cluster_id": "C1", "decision_time": pd.Timestamp("2020-04-20 17:00", tz=ET),
         "earnings_type": "earnings_announcement_verified"},
        {"event_id": "e_q", "accessions": ["q10"], "cluster_id": "C1", "decision_time": pd.Timestamp("2020-05-05 17:00", tz=ET),
         "earnings_type": None}])
    bad = pd.DataFrame([{"event_id": "e_rel", "source_accession": "q10", "source_public_time": pd.Timestamp("2020-05-05 16:35", tz=ET)}])
    assert len(sue_rule_violations(bad, events)) == 1


def test_ear_same_cluster_rejected_later_cluster_accepted_after_window():
    events = pd.DataFrame([{"event_id": "e1", "cluster_id": "C1", "decision_time": pd.Timestamp("2020-05-01 09:00", tz=ET)}])
    same = pd.DataFrame([{"event_id": "e1", "ear_src_cluster": "C1", "_avail_ear": pd.Timestamp("2020-04-01", tz="UTC")}])
    assert len(ear_rule_violations(same, events)) == 1
    open_window = pd.DataFrame([{"event_id": "e1", "ear_src_cluster": "C0", "_avail_ear": pd.Timestamp("2020-05-02", tz="UTC")}])
    assert len(ear_rule_violations(open_window, events)) == 1
    ok = pd.DataFrame([{"event_id": "e1", "ear_src_cluster": "C0", "_avail_ear": pd.Timestamp("2020-04-02", tz="UTC")}])
    assert len(ear_rule_violations(ok, events)) == 0


def test_release_eps_parser_formats():
    from secprice.parse.eps_release import parse_release_eps
    assert parse_release_eps("Net income of $106 million, or $.26 per diluted share.")["eps_diluted"] == pytest.approx(0.26)
    assert parse_release_eps("Earnings per share (assuming dilution) were $2.09, a decrease")["eps_diluted"] == pytest.approx(2.09)
    t = "Earnings per share:\nDiluted earnings per share | $ | 0.97 | $ | 0.67"
    r = parse_release_eps(t)
    assert r["eps_diluted"] == pytest.approx(0.97) and r["parse_confidence"] == "high"
    assert parse_release_eps("Diluted earnings per share, excluding specified items, were $1.30")["eps_diluted"] is None


def test_full_year_eps_in_q4_release_is_withheld():
    """Found on real 1999-2003 releases (AXP $5.42 and XOM $3.23 are full-year figures; the quarters were ~$1.4 and $1.01)."""
    from secprice.parse.eps_release import flag_suspect_annual
    t = pd.to_datetime(["1999-04-20", "1999-07-20", "1999-10-20", "2000-01-25", "2000-04-20"], utc=True)
    rel = pd.DataFrame({"accession": list("abcde"), "cik": 1, "is_results_release": True, "eps_diluted": [1.30, 1.41, 1.42, 5.42, 1.44],
                        "parse_confidence": "medium", "period_end": ["1999-03-31", "1999-06-30", "1999-09-30", "1999-12-31", "2000-03-31"],
                        "public_available_time": t})
    out = flag_suspect_annual(rel, {1: "1231"})
    assert out.set_index("accession").suspect_annual.to_dict() == {"a": False, "b": False, "c": False, "d": True, "e": False}
    assert out.set_index("accession").loc["d", "parse_confidence"] == "suspect_annual"
    # a non-Q4 quarter with a big jump is kept (the check applies at the fiscal year end only); too little history is kept
    rel2 = rel.copy(); rel2.loc[3, "period_end"] = "2000-02-15"
    assert not flag_suspect_annual(rel2, {1: "1231"}).suspect_annual.any()
    assert not flag_suspect_annual(rel.iloc[3:], {1: "1231"}).suspect_annual.any()


def test_release_eps_losses_text_tables_undifferentiated_and_multiyear():
    """Real 2004-2011 release formats (early-era run, D-066): loss wording, plain-text tables, EPS without 'diluted',
    five-year summary tables."""
    from secprice.parse.eps_release import USABLE_CONFIDENCE, _text_table_row, parse_release_eps as p
    r = p("For the second quarter the company reported a net loss of $25.3 million, or $0.80 per diluted share, compared "
          "with net income of $0.40 per diluted share a year ago.")
    assert r["eps_diluted"] == -0.80                                  # sign from 'net loss'; '25.3' is not a sentence end
    assert p("Diluted earnings per share were $0.20, compared with a loss of $0.05 per share.")["eps_diluted"] == 0.20
    assert p("Second quarter net loss per diluted share was $0.31.")["eps_diluted"] == -0.31
    assert _text_table_row("Diluted $ 0.08 $ (0.05)") == "Diluted | 0.08 | (0.05)"
    assert _text_table_row("Diluted (0.04) 0.27") == "Diluted | (0.04) | 0.27"   # parenthesis stays with the value
    assert _text_table_row("Diluted 16,983 15,198") is None                      # share counts are not EPS
    t = "Three months ended September 30, 2004\n(In thousands, except per share data)\nEarnings per share\nDiluted $ 0.08 $ (0.05)\n" \
        "Weighted average shares\nDiluted 16,953 15,182"
    assert p(t)["eps_diluted"] == 0.08 and p(t)["parse_confidence"] == "medium_table"
    u = p("Third quarter earnings per share were $0.39, up from $0.33.")
    assert u["eps_diluted"] == 0.39 and u["parse_confidence"] == "medium_undiff" and "medium_undiff" in USABLE_CONFIDENCE
    assert p("We expect earnings per share for the third quarter to come in between $0.55 to $0.59.")["eps_diluted"] is None
    five = "\n".join(["Selected financial data", "2010", "", "2009", "", "2008", "", "2007", "", "2006", "",
                      "Net income per common share - assuming dilution (dollars) | 6.22 | 3.98 | 8.69"])
    assert p(five)["eps_diluted"] is None                             # a multi-year summary, not the quarter


def test_suspect_annual_handles_february_fiscal_year_end_in_leap_years():
    """Regression (full run): FYE '0229' / period 2008-02-29 crashed the year shift."""
    from secprice.parse.eps_release import flag_suspect_annual
    t = pd.to_datetime(["2007-06-10", "2007-09-10", "2007-12-10", "2008-04-01"], utc=True)
    rel = pd.DataFrame({"accession": list("abcd"), "cik": 7, "is_results_release": True, "eps_diluted": [0.5, 0.52, 0.55, 2.1],
                        "parse_confidence": "medium", "period_end": ["2007-05-31", "2007-08-31", "2007-11-30", "2008-02-29"],
                        "public_available_time": t})
    out = flag_suspect_annual(rel, {7: "0229"}).set_index("accession")
    assert out.loc["d", "suspect_annual"] and not out.loc["a":"c", "suspect_annual"].any()


def test_release_cluster_exclusions_cover_same_day_and_later_periodic_filings():
    """D-072 (full run): a 10-Q filed minutes after the earnings 8-K on the same day was public before the release's
    decision time and entered its SUE history; every periodic filing on/after the release within 120 days is excluded."""
    import pandas as pd
    from secprice.stages_data import release_cluster_exclusions
    f = pd.DataFrame({"cik": [1, 1, 1, 1, 2], "accession": ["r1", "q_same_day", "q_later", "q_prior", "q_other_firm"],
                      "form": ["8-K", "10-Q", "10-K", "10-Q", "10-Q"],
                      "filing_date": ["2012-04-20", "2012-04-20", "2012-09-30", "2012-01-30", "2012-04-20"]})
    ex = release_cluster_exclusions(f, {"r1"})
    assert ex == {"r1": {"q_same_day"}}                    # q_later is beyond 120 days; q_prior is earlier; other firm excluded
    assert release_cluster_exclusions(f, {"r1"}, window_days=180) == {"r1": {"q_same_day", "q_later"}}


In [ ]:
%%writefile $REPO/tests/unit/test_lockbox_buckets_provenance.py
"""Lockbox single use (§6.1, M12), index buckets (D-056), provenance stamps (hard rule 15), backtest exposure (§11.7)."""
import json

import numpy as np
import pandas as pd
import pytest

from secprice.backtest.engine import exposure_stats
from secprice.evaluation.buckets import composition, event_buckets, expand, pooled_vs_bucket, score_predictions
from secprice.stages_model import lockbox_guard


def _cfg(unlock=True, reuse=False):
    return {"lockbox": {"unlock_locked_test": unlock, "acknowledge_reuse": reuse}}


def test_lockbox_single_use(tmp_path):
    frozen = {"g4_hash": "abc"}
    with pytest.raises(RuntimeError, match="sealed"):
        lockbox_guard(_cfg(unlock=False), tmp_path, frozen)
    with pytest.raises(RuntimeError, match="G4 hash"):
        lockbox_guard(_cfg(), tmp_path, frozen)                     # no G4 freeze yet
    (tmp_path / "g4_hash.txt").write_text("other")
    with pytest.raises(RuntimeError, match="G4 hash"):
        lockbox_guard(_cfg(), tmp_path, frozen)                     # frozen architecture changed after G4
    (tmp_path / "g4_hash.txt").write_text("abc")
    assert lockbox_guard(_cfg(), tmp_path, frozen) is False         # first and only scoring
    (tmp_path / "lockbox_log.json").write_text(json.dumps({"scored_at": "2026-10-23"}))
    with pytest.raises(RuntimeError, match="already scored"):
        lockbox_guard(_cfg(), tmp_path, frozen)                     # second scoring refused
    assert lockbox_guard(_cfg(reuse=True), tmp_path, frozen) is True   # explicit reuse -> relabelled as development


def _universe():
    return pd.DataFrame({"cik": [1, 2, 3], "ever_sp500": [True, True, False], "sp_bucket": ["SP500", "SP500", "NonSP500"],
                         "sp500_intervals": [json.dumps([["2010-01-01", None]]), json.dumps([["2010-01-01", "2022-12-31"]]),
                                             json.dumps([])]})


def _events():
    d = pd.to_datetime(["2022-06-01", "2023-06-01", "2023-06-01", "2022-06-01"])
    return pd.DataFrame({"event_id": ["a", "b", "c", "d"], "cik": [1, 2, 3, 2], "decision_date": d,
                         "event_type": ["8K", "8K", "periodic_10Q", "periodic_10K"], "earnings_type": [None] * 4,
                         "ts_quality": ["A"] * 4})


def test_event_buckets_firm_level_and_point_in_time():
    bk = event_buckets(_events(), _universe()).set_index("event_id")
    assert bk.sp_bucket.to_dict() == {"a": "SP500", "b": "SP500", "c": "NonSP500", "d": "SP500"}
    # firm 2 left the index on 2022-12-31: its 2023 event is in the S&P bucket (firm level) but not a member at the event
    assert bk.sp500_member_at_event.to_dict() == {"a": True, "b": False, "c": False, "d": True}
    x = expand(_events(), bk.reset_index())
    assert len(x) == 8 and (x.sp_bucket == "Combined").sum() == 4
    comp = composition(_events(), bk.reset_index(), _universe()).set_index("sp_bucket")
    assert comp.loc["Combined", "events"] == 4 and comp.loc["SP500", "firms"] == 2 and comp.loc["NonSP500", "events"] == 1
    assert comp.loc["SP500", "member_at_event_share"] == pytest.approx(2 / 3)


def test_bucket_scores_and_pooled_comparison():
    rng = np.random.default_rng(0)
    n = 300
    ev = pd.DataFrame({"event_id": [f"e{i}" for i in range(n)], "cik": rng.integers(1, 4, n)})
    bk = pd.DataFrame({"event_id": ev.event_id, "cik": ev.cik, "sp_bucket": np.where(ev.cik == 3, "NonSP500", "SP500")})
    y = rng.integers(0, 3, n)
    p = rng.dirichlet([1, 1, 1], n)
    dates = pd.date_range("2015-01-01", periods=n, freq="W")
    preds = pd.DataFrame({"run_id": "r", "event_id": ev.event_id, "y": y, "yhat": p.argmax(1), "date": dates,
                          "p_down": p[:, 0], "p_side": p[:, 1], "p_up": p[:, 2]})
    sc = score_predictions(preds, bk, 50, 0)
    assert list(sc.sp_bucket) == ["Combined", "SP500", "NonSP500"]
    assert sc.set_index("sp_bucket").n_events.to_dict() == {"Combined": n, "SP500": int((ev.cik != 3).sum()),
                                                            "NonSP500": int((ev.cik == 3).sum())}
    same = pooled_vs_bucket(preds, preds, 50, 0)                    # identical predictions -> zero difference
    assert same["pooled_minus_bucket_macro_f1"] == 0 and same["n_events"] == n


def test_exposure_stats_counts_cash_days_and_turnover():
    d = pd.DataFrame({"n_long": [0, 2, 12, 12], "n_short": [0, 3, 0, 11], "turnover_long": [0, 0.5, 0.2, 0.0],
                      "turnover_short": [0, 0, 0, 0.4]})
    e = exposure_stats(d, 10)
    assert e["share_days_long_invested"] == 0.5 and e["share_days_short_in_cash"] == 0.25
    assert e["avg_names_long"] == pytest.approx(26 / 3) and e["turnover_short_daily"] == pytest.approx(0.1)


def test_git_commit_and_code_hash_are_stamped():
    from secprice.storage import code_hash, git_commit
    c = git_commit()
    assert c and (len(c.split("-")[0]) == 40 or c.startswith("embedded:"))
    assert len(code_hash()) == 12


In [ ]:
%%writefile $REPO/tests/unit/test_parse_taxonomy.py
"""Parsing, sections, 8-K taxonomy (SPEC.md §18.1 #11) and coverage reason codes (#12)."""
import pandas as pd
import pytest

from secprice.config import load_yaml, repo_root
from secprice.parse.html import drop_numeric_lines, to_text
from secprice.parse.sections import eightk_items_text, extract_sections
from secprice.parse.taxonomy import Taxonomy8K
from secprice.sec.archives import parse_sgml


@pytest.fixture(scope="module")
def tax():
    return Taxonomy8K(load_yaml(repo_root() / "configs" / "taxonomy_8k_v1.yaml"))


def test_taxonomy_legacy_modern_and_catch_all(tax):
    assert tax.map_item("2.02", "2010-01-01") == "earnings"
    assert tax.map_item("2.2", "2010-01-01") == "earnings"                 # zero-padding
    assert tax.map_item("12", "2003-06-01") == "earnings"                  # legacy item 12 after 2003-03-28
    assert tax.map_item("12", "2002-06-01") == tax.catch_all               # item 12 did not exist yet
    assert tax.map_item("5", "2001-01-01") == "other_events"
    assert tax.map_item("5", "2010-01-01") == tax.catch_all                # legacy codes are not valid after 2004-08-23
    assert tax.map_item("9.99", "2010-01-01") == tax.catch_all
    assert tax.categories([], "2001-01-01", ["Results of Operations and Financial Condition"]) == ["earnings"]
    modern, legacy = tax.earnings_candidate(["5", "7"], "2001-05-01")
    assert not modern and legacy


def test_html_tables_and_hidden_xbrl():
    html = ("<html><body><div style='display:none'><ix:header>hidden 123</ix:header></div><p>Item 7. MD&amp;A</p>"
            "<table><tr><td>Diluted</td><td>$</td><td>1.23</td></tr></table><p>Results improved &#146;s</p></body></html>")
    t, n, meth = to_text(html)
    assert "hidden" not in t and "Diluted | $ | 1.23" in t and n == 1 and meth == "html"
    assert "’" in t                                                      # C1 code points mapped via cp1252
    assert "1.23" not in drop_numeric_lines(t)


def test_10k_sections_skip_toc():
    body = ("TABLE OF CONTENTS\nItem 1. Business\nItem 1A. Risk Factors\nItem 7. Management's Discussion and Analysis\n"
            "PART I\nItem 1. Business\n" + "We make widgets. " * 100 + "\nItem 1A. Risk Factors\n" + "Risks exist. " * 200 +
            "\nItem 2. Properties\nOffices.\nItem 7. Management's Discussion and Analysis of Financial Condition\n" +
            "Revenue grew. " * 300 + "\nItem 7A. Quantitative and Qualitative Disclosures About Market Risk\n" + "Rates. " * 50 +
            "\nItem 8. Financial Statements\nTables.")
    s = extract_sections(body, "10-K")
    assert set(s) == {"10K_1", "10K_1A", "10K_7", "10K_7A"}
    assert s["10K_7"]["text"].count("Revenue grew.") == 300 and "Risks exist" not in s["10K_7"]["text"]


def test_10q_parts():
    body = ("PART I - FINANCIAL INFORMATION\nItem 1. Financial Statements\nTables\nItem 2. Management's Discussion and Analysis\n" +
            "Sales rose. " * 100 + "\nItem 3. Quantitative and Qualitative Disclosures About Market Risk\n" + "FX. " * 30 +
            "\nItem 4. Controls\nOK\nPART II - OTHER INFORMATION\nItem 1A. Risk Factors\n" + "New risk. " * 40 + "\nItem 6. Exhibits\n")
    s = extract_sections(body, "10-Q")
    assert {"10Q_MDA", "10Q_MR", "10Q_RF"} <= set(s)


def test_sgml_parse_real_fixture():
    from pathlib import Path
    raw = (Path(__file__).resolve().parents[1] / "fixtures" / "aapl_8k_2002_full.txt").read_bytes()
    sub = parse_sgml(raw)
    assert sub.acceptance_raw == "20020813164730"
    assert any(d.type.upper().startswith("8-K") for d in sub.documents)
    assert "Other events" in " ".join(sub.item_titles)
    main = [d for d in sub.documents if d.type.upper().startswith("8-K")][0]
    t, _, _ = to_text(main.text)
    assert eightk_items_text(t)["text"]


def test_coverage_reason_codes_no_8k_expectation():
    from secprice.stages_data import periodic_expectations
    per = pd.DataFrame({"cik": [1] * 5, "ry": [2020, 2020, 2020, 2020, 2021],
                        "base_form": ["10-K", "10-Q", "10-Q", "10-Q", "10-Q"]})
    cov = periodic_expectations(per)
    assert cov.set_index("year").loc[2020, "reason_code"] == "OK"
    assert cov.set_index("year").loc[2021, "reason_code"] == "MISSING_10K"
    assert not any("8-K" in c for c in cov.columns)


def test_ex13_mdna_found_under_varied_annual_report_titles():
    """Real EX-13s head the MD&A 'FINANCIAL REVIEW' (AXP, PFE) or 'Management Discussion' (IBM); found on 1999-2004 filings."""
    from secprice.parse.sections import mdna_from_ex13
    body = "Revenues rose on higher volume and pricing across all segments of the business. " * 60
    for title in ("FINANCIAL REVIEW", "Management Discussion", "MANAGEMENT'S DISCUSSION AND ANALYSIS OF FINANCIAL CONDITION AND RESULTS"):
        text = f"Letter to shareholders\n\n{title}\n\n{body}\n\nConsolidated Statements of Income\n\nnumbers"
        out = mdna_from_ex13(text)
        assert out is not None and out["method"] == "ex13_title" and len(out["text"]) > 3000, title
    assert mdna_from_ex13("Letter\n\nFinancial review of the year was positive.\n\n" + body) is None   # not a heading line


def test_sec_cache_is_compressed_and_size_capped(tmp_path):
    """Disk budget for the full run: cached bodies are zlib-compressed; bodies above cache_max_bytes are not cached."""
    from secprice.sec.client import _ZMAGIC, SECClient

    class _Resp:
        def __init__(self, body):
            self.status_code, self._b = 200, body

        def iter_content(self, n):
            for i in range(0, len(self._b), n):
                yield self._b[i:i + n]

        def close(self):
            pass

    calls = []

    class _Sess:
        def get(self, url, timeout=None, stream=None):
            calls.append(url)
            return _Resp(b"<html>" + b"filing text " * (10 if "small" in url else 5000) + b"</html>")

    c = SECClient("Test User test@example.com", max_rps=100, cache_dir=tmp_path, cache_max_bytes=1000)
    c._session = lambda: _Sess()
    small = c.get("https://www.sec.gov/small")
    big = c.get("https://www.sec.gov/big")
    files = list(tmp_path.rglob("*.bin"))
    assert len(files) == 1 and files[0].read_bytes().startswith(_ZMAGIC)        # only the small body, compressed
    assert c.get("https://www.sec.gov/small") == small and len(calls) == 2         # served from the cache
    assert c.get("https://www.sec.gov/big") == big and len(calls) == 3             # large body re-fetched, never cached


def test_s05_worker_client_honours_cache_cap(tmp_path):
    """Regression (found in the full run): the per-firm worker built its SEC client without the profile's cache cap."""
    from secprice.parse.worker import _client
    job = {"fixture_root": None, "user_agent": "Test User test@example.com", "rps_per_proc": 1.0, "cache_dir": str(tmp_path),
           "cache_days": 30, "timeout_s": 10, "max_retries": 2, "backoff_base_s": 2.0, "blocked_403_limit": 20,
           "cache_max_bytes": 400000}
    assert _client(job).cache_max_bytes == 400000


def test_hf_qa_collapses_duplicate_sections():
    """Regression (full run): two HF rows for one firm-date made the parser-QA lookup return a Series."""
    import pandas as pd
    from secprice.stages_data import _one_text_per_section
    d = pd.DataFrame({"section_id": ["10K_7", "10K_7", "10K_1"], "hf_text": ["short", "a much longer MD&A text", None]})
    s = _one_text_per_section(d, "hf_text")
    assert s["10K_7"] == "a much longer MD&A text" and s["10K_1"] == "" and s.index.is_unique
    assert _one_text_per_section(d.iloc[:0], "hf_text").empty


In [ ]:
%%writefile $REPO/tests/unit/test_regex_backtracking.py
"""No catastrophic regex backtracking on adversarial filing lines (found in the full run: html.normalize spent hours on a
wide table row of empty cells). Every module-level compiled pattern in the parser modules, plus the normalisation path,
must finish fast on long pathological inputs."""
import re
import time

import pytest

from secprice.parse import eps_release, html, sections, taxonomy

ADVERSARIAL = [
    "a" + " | " * 4000 + "x",                                   # many empty cells, then text (trailing-pipe pattern fails)
    " | " * 4000,
    "Diluted" + " 1" * 4000 + " x",                             # long numeric row that fails the end anchor
    "Diluted $" + " $ 0" * 3000 + " end",
    "management " * 3000,
    "(" * 3000 + "0.12",
    " " * 20000 + "x",
    "1," * 5000 + "x",
    "per share " * 2000 + "$0.1",
]


def _patterns():
    for mod in (html, eps_release, sections, taxonomy):
        for name, v in vars(mod).items():
            if isinstance(v, re.Pattern):
                yield f"{mod.__name__}.{name}", v
            elif isinstance(v, (list, tuple)):
                for i, x in enumerate(v):
                    rx = x[0] if isinstance(x, tuple) and x and isinstance(x[0], re.Pattern) else x
                    if isinstance(rx, re.Pattern):
                        yield f"{mod.__name__}.{name}[{i}]", rx


@pytest.mark.parametrize("name,rx", list(_patterns()))
def test_pattern_is_linear_enough(name, rx):
    for s in ADVERSARIAL:
        t0 = time.perf_counter()
        rx.search(s)
        list(rx.finditer(s[:5000]))
        assert time.perf_counter() - t0 < 2.0, f"{name} is slow on {s[:40]!r}..."


def test_normalize_and_eps_parser_fast_on_wide_tables():
    row = "<tr>" + "<td></td>" * 3000 + "<td>Net income</td></tr>"
    doc = "<html><body><table>" + row * 3 + "</table></body></html>"
    t0 = time.perf_counter()
    txt, _, _ = html.to_text(doc)
    eps_release.parse_release_eps(txt + "\n" + "Diluted" + " | " * 3000 + "0.12")
    assert time.perf_counter() - t0 < 5.0


In [ ]:
%%writefile $REPO/tests/unit/test_reproducibility.py
"""Hard rule 12 / 15: every model revision is pinned; the Colab install pins equal requirements.lock."""
import re
from pathlib import Path

import pytest
import yaml

ROOT = Path(__file__).resolve().parents[2]


def test_all_model_revisions_pinned():
    m = yaml.safe_load((ROOT / "configs" / "models.yaml").read_text(encoding="utf-8"))
    chrono = m["encoders"]["chrono_bert"]
    assert set(map(str, chrono["vintages"])) == set(chrono["revisions"]), "every ChronoBERT vintage needs a revision"
    sha = re.compile(r"^[0-9a-f]{40}$")
    assert all(sha.match(str(r)) for r in chrono["revisions"].values())
    for sec in ("encoders", "reranker", "llm", "judge", "student", "ner"):
        for name, spec in m[sec].items():
            if "model_id" in spec:
                assert sha.match(str(spec.get("revision", ""))), f"{name} lacks a pinned revision"


def test_notebook_pins_match_lock_file():
    # repository-only check: the notebook builder and the lock file are not part of the notebook-embedded package
    if not (ROOT / "tools" / "build_notebook.py").exists() or not (ROOT / "requirements.lock").exists():
        pytest.skip("repository-only check (builder / lock file not embedded in the notebook)")
    lock = {}
    for line in (ROOT / "requirements.lock").read_text(encoding="utf-8").splitlines():
        if "==" in line and not line.startswith("#"):
            k, v = line.strip().split("==")
            lock[k.lower().replace("_", "-")] = v
    src = (ROOT / "tools" / "build_notebook.py").read_text(encoding="utf-8")
    block = src[src.index("PINS = {"):src.index("}", src.index("PINS = {"))]
    pins = dict(re.findall(r'"([^"]+)":\s*"([^"]+)"', block))
    assert pins, "PINS block not found"
    for k, v in pins.items():
        assert lock.get(k.lower().replace("_", "-")) == v, f"{k}: notebook {v} vs lock {lock.get(k)}"


In [ ]:
%%writefile $REPO/tests/unit/test_returns_targets.py
"""SPEC.md §18.1 #5-#7, #10 and addendum P0.2 acceptance tests (exact returns, interval-matched benchmark)."""
import numpy as np
import pandas as pd
import pytest

from secprice.market.prices import build_series
from secprice.market.returns import cc_return, oc_return
from secprice.targets.build import Estimator, label3, uniqueness_weights


def _bars(closes, opens=None, dates=None):
    n = len(closes)
    d = pd.date_range("2024-01-01", periods=n, freq="B") if dates is None else dates
    o = closes if opens is None else opens
    return pd.DataFrame({"date": d, "open": o, "high": np.maximum(o, closes), "low": np.minimum(o, closes),
                         "close": closes, "volume": 1e6, "source": "t"})


def test_split_raw_vs_adjusted_and_tri():
    # raw prices 100 -> 102 -> (2:1 split) 51.5 -> 52 ; split-adjusted basis = 50, 51, 51.5, 52
    adj = np.array([50.0, 51.0, 51.5, 52.0])
    d = pd.date_range("2024-01-01", periods=4, freq="B")
    acts = pd.DataFrame({"date": [d[2]], "type": ["split"], "value": [2.0]})
    s = build_series(_bars(adj, dates=d), acts)
    np.testing.assert_allclose(s.close_raw.values, [100, 102, 51.5, 52])
    np.testing.assert_allclose(s.total_return_index.values[-1] / s.total_return_index.values[0], 52 / 50)


def test_dividend_total_return_and_ex_date_rule():
    c = np.array([100.0, 101.0, 99.0, 100.0])        # ex-date at index 2 with a 2.0 dividend
    o = np.array([100.0, 100.5, 99.5, 99.0])
    d = pd.date_range("2024-01-01", periods=4, freq="B")
    s = build_series(_bars(c, o, d), pd.DataFrame({"date": [d[2]], "type": ["dividend"], "value": [2.0]}))
    tri = s.total_return_index.values
    assert cc_return(tri, 0, 3) == pytest.approx((101 / 100) * ((99 + 2) / 101) * (100 / 99) - 1)
    # bought at the open of the day before the ex-date: entitled to the dividend
    assert oc_return(o, c, tri, 1, 3) == pytest.approx((101 / 100.5) * ((99 + 2) / 101) * (100 / 99) - 1)
    # bought at the open OF the ex-date: not entitled
    assert oc_return(o, c, tri, 2, 3) == pytest.approx((99 / 99.5) * (100 / 99) - 1)
    # overnight gap before entry is excluded from an open->close holding return
    assert oc_return(o, c, tri, 3, 3) == pytest.approx(100 / 99 - 1)


def test_benchmark_interval_matches_stock_interval():
    """Stock and benchmark legs use the same formula on the same (entry open -> exit close) interval."""
    c = np.array([10.0, 11.0, 12.0, 13.0])
    o = np.array([9.5, 10.5, 11.5, 12.5])
    tri = c / c[0]
    for e, x in ((1, 1), (1, 3), (2, 3)):
        assert oc_return(o, c, tri, e, x) == pytest.approx(c[x] / o[e] - 1)


def test_market_model_known_beta():
    rng = np.random.default_rng(0)
    rm = rng.normal(0, 0.01, 400)
    r = 0.0002 + 1.5 * rm + rng.normal(0, 0.002, 400)
    cnt, a, b, s = Estimator(r, rm).window(np.array([50]), np.array([349]))
    assert cnt[0] == 300 and b[0] == pytest.approx(1.5, abs=0.05) and s[0] == pytest.approx(0.002, rel=0.2)


def test_carhart_on_synthetic():
    from secprice.targets.build import _carhart_car
    rng = np.random.default_rng(1)
    n = 400
    f = rng.normal(0, 0.01, (n, 4))
    rf = np.full(n, 0.0001)
    r = rf + f @ np.array([1.1, 0.3, -0.2, 0.1]) + rng.normal(0, 0.001, n)
    car = _carhart_car(r, f, rf, np.array([360]), np.array([365]), np.array([50]), np.array([330]), 120)
    assert abs(car[0]) < 0.01


def test_k_band_labels():
    ar = np.array([0.03, -0.03, 0.001, np.nan])
    sig = np.array([0.02, 0.02, 0.02, 0.02])
    lab = label3(ar, sig, 1, 0.5)                      # threshold 0.01
    assert lab[0] == 2 and lab[1] == 0 and lab[2] == 1 and np.isnan(lab[3])
    lab4 = label3(np.array([0.015]), np.array([0.02]), 4, 0.5)   # threshold 0.5*0.02*2 = 0.02 -> sideways
    assert lab4[0] == 1


def test_uniqueness_weights_overlaps():
    w = uniqueness_weights(np.array([0, 0, 10]), np.array([4, 4, 12]))
    assert w[0] == pytest.approx(0.5) and w[2] == pytest.approx(1.0)


def test_overlap_flags_and_clean_cohort():
    from secprice.events.build import _overlaps
    ev = pd.DataFrame({"cik": [1, 1, 1, 2], "trackA_day0_idx": [10, 12, 40, 11]})
    out = _overlaps(ev, {"targets": {"horizons": [1, 5]}, "contam_window_sessions": 5})
    assert out.overlap_flag_h5.tolist() == [True, False, False, False]
    assert out.overlap_flag_h1.tolist() == [False, False, False, False]


def test_robust_same_sign_treats_zero_p_as_significant_and_missing_as_unknown():
    """D-073 (full run): `(p or 1) < 0.10` turned p = 0.0 into 1 and marked a t = -9.8 lag check as failed."""
    from secprice.track_a.drift import robust_same_sign
    main = {"estimate": -0.030, "p_value": 0.0}
    assert robust_same_sign(main, {"estimate": -0.031, "p_value": 0.0}) is True
    assert robust_same_sign(main, {"estimate": 0.010, "p_value": 0.0}) is False       # opposite sign
    assert robust_same_sign(main, {"estimate": -0.010, "p_value": 0.4}) is False      # not significant
    assert robust_same_sign(main, {}) is None and robust_same_sign(main, None) is None  # too few events: unknown


def test_overlap_flags_ignore_own_cluster_members():
    """D-073: a verified release and its own 10-Q are one information event (§5.2) and do not contaminate each other."""
    from secprice.events.build import _overlaps
    ev = pd.DataFrame({"cik": [1, 1, 1, 2, 2], "trackA_day0_idx": [100, 130, 300, 100, 120],
                       "cluster_id": ["c1", "c1", None, None, None]})
    cfg = {"targets": {"horizons": [60]}, "contam_window_sessions": 60}
    out = _overlaps(ev.copy(), cfg)
    assert out.overlap_flag_h60.tolist() == [False, False, False, True, False]


In [ ]:
%%writefile $REPO/tests/unit/test_splits_metrics.py
"""Retrieval (#13), splits (#14), planted leaks (#15), cohorts (#16), metrics (#17), bootstrap (#18), resilience (#19),
config (#20); gold-set integrity (addendum P1.6); backtest entry rules."""
import json

import numpy as np
import pandas as pd
import pytest

from secprice.config import config_hash, load_config
from secprice.evaluation.bootstrap import macro_f1_ci, paired_macro_f1_diff
from secprice.evaluation.metrics import classification_metrics, f1_from_cm, regression_metrics
from secprice.rag.core import historical_candidates, retrieval_guard
from secprice.splits.walkforward import assign_roles, make_blocks


def test_historical_retrieval_never_current_or_future():
    assert retrieval_guard()
    now = pd.Timestamp("2015-06-01 12:00", tz="America/New_York")
    ch = pd.DataFrame({"chunk_id": ["a", "b"], "cik": [1, 1], "accession": ["x", "y"],
                       "public_time": [now, now - pd.Timedelta(seconds=1)], "text": ["", ""]})
    assert historical_candidates(ch, 1, now, []).chunk_id.tolist() == ["b"]       # strict '<'


def _cfg():
    return load_config("smoke")


def test_walk_forward_purge_embargo_and_date_disjoint():
    cfg = _cfg()
    cfg["splits"].update({"min_train_years": 1, "val_block_years": 1, "dev_start": "2023-01-01", "locked_test_start": "2023-07-01",
                          "embargo_days": 30})
    cfg["study"]["start"] = "2021-01-01"
    b = make_blocks(cfg, pd.Timestamp("2023-12-31"))
    assert [x.start for x in b["folds"]] == [pd.Timestamp("2022-01-01")]
    d = pd.to_datetime(["2021-03-01", "2021-12-10", "2021-12-28", "2022-02-01", "2023-02-01"])
    ev = pd.DataFrame({"event_id": list("abcde"), "decision_date": d, "cik": [1, 2, 1, 2, 1], "ff12_industry": "Other"})
    exits = pd.Series(pd.to_datetime(["2021-04-01", "2021-12-20", "2022-01-20", "2022-03-01", "2023-03-01"]))
    sp = assign_roles(ev, b, cfg, exits)
    wf = sp[sp.scheme == "walk_forward"].set_index("event_id").role
    assert wf["a"] == "train" and wf["b"] == "embargo" and wf["c"] == "purged" and wf["d"] == "val"
    tr_dates = set(ev[ev.event_id.isin(wf[wf == "train"].index)].decision_date.dt.date)
    va_dates = set(ev[ev.event_id.isin(wf[wf == "val"].index)].decision_date.dt.date)
    assert not tr_dates & va_dates


def test_planted_leaks_all_caught():
    from secprice.splits.leakage import planted_leaks
    et = "America/New_York"
    ev = pd.DataFrame({"event_id": ["e1", "e2"], "cik": [1, 1], "accessions": [["a1"], ["a2"]],
                       "decision_time": [pd.Timestamp("2020-01-10 17:00", tz=et), pd.Timestamp("2020-02-10 17:00", tz=et)],
                       "public_available_time": [pd.Timestamp("2020-01-10 16:35", tz=et), pd.Timestamp("2020-02-10 16:35", tz=et)],
                       "cluster_id": ["C1", "C1"], "cluster_role": ["release", "periodic_after_release"],
                       "earnings_type": ["earnings_announcement_verified", None]})
    tg = pd.DataFrame({"event_id": ["e1", "e2"], "target_def": "T_trade", "horizon": 5,
                       "exit_date": pd.to_datetime(["2020-01-17", "2020-02-18"])})
    F = pd.DataFrame({"event_id": ["e1", "e2"]})
    res = planted_leaks(F, ev, tg, pd.DataFrame(columns=["event_id", "source_accession", "source_public_time"]), retrieval_guard)
    assert len(res) == 4 and all(r["caught"] for r in res), res


def test_common_cohort_identical_across_models():
    from secprice.models.ladder import Ladder
    cfg = _cfg()
    ev = pd.DataFrame({"event_id": list("abcd"), "decision_date": pd.date_range("2021-01-01", periods=4), "cik": 1})
    F = pd.DataFrame({"event_id": list("abcd"), "px_ret_20": [0.1, np.nan, 0.2, 0.3], "px_vol_20": [0.1, 0.1, 0.2, 0.3],
                      "acc_log_assets": [1, 1, np.nan, 1], "txt_has_text": [True, True, True, False], "overlap_flag_h20": False})
    ev["overlap_flag_h20"] = False
    lad = Ladder(cfg, ev, F, pd.DataFrame(columns=["event_id", "target_def", "horizon", "label3", "weight_uniq"]),
                 pd.DataFrame(columns=["event_id", "scheme", "fold", "role"]))
    ids = pd.Index(list("abcd"))
    m1 = lad.cohort(["L1", "L3", "L4", "L6"], [None], ids)
    m2 = lad.cohort(["L6", "L1", "L4", "L3"], [None], ids)
    assert m1.tolist() == m2.tolist() == [True, False, False, False]


def test_metrics_hand_computed_and_nulls():
    y = np.array([0, 0, 1, 1, 2, 2])
    yhat = np.array([0, 1, 1, 1, 2, 0])
    cm = np.zeros((3, 3), int)
    np.add.at(cm, (y, yhat), 1)
    f = f1_from_cm(cm)
    # class0 P=1/2 R=1/2 F=.5 ; class1 P=2/3 R=1 F=.8 ; class2 P=1 R=1/2 F=2/3
    assert f["macro_f1"] == pytest.approx((0.5 + 0.8 + 2 / 3) / 3)
    assert f["accuracy"] == pytest.approx(4 / 6)
    P = np.eye(3)[yhat] * 0.8 + 0.2 / 3
    m = classification_metrics(y, P, np.ones(3) / 3, yhat)
    assert m["macro_f1"] == pytest.approx(f["macro_f1"])
    m2 = classification_metrics(np.array([0, 0, 1]), np.tile([0.5, 0.3, 0.2], (3, 1)), np.ones(3) / 3)
    assert m2["auc_up"] is None and m2["auc_macro_ovr"] is None            # class absent -> null, never 0
    r = regression_metrics(np.array([1.0, 2, 3, 4]), np.array([1.0, 2, 3, 4]), hist_mean=np.array([2.5] * 4))
    assert r["r2"] == pytest.approx(1.0) and r["oos_r2_vs_hist_mean"] == pytest.approx(1.0)


def test_block_bootstrap_resamples_months():
    d = pd.to_datetime(["2020-01-05"] * 50 + ["2020-02-05"] * 50)
    y = np.r_[np.zeros(50), np.ones(50)].astype(int)
    ci = macro_f1_ci(y, y, d, n_boot=200, seed=0)
    assert ci["n_months"] == 2 and ci["value"] == pytest.approx(1.0)
    diff = paired_macro_f1_diff(y, y, y, d, n_boot=100)
    assert diff["value"] == 0 and diff["ci_low"] == diff["ci_high"] == 0


def test_resume_does_not_duplicate(tmp_path):
    from secprice.parse.worker import run_firm
    (tmp_path / "ledger").mkdir()
    pd.DataFrame({"accession": ["x"]}).to_parquet(tmp_path / "ledger" / "0000000001.parquet")
    out = run_firm({"cik": 1, "out_dir": str(tmp_path), "filings": []})
    assert out["skipped"] is True


def test_checkpoint_reload_reproduces_predictions(tmp_path):
    import pickle
    from secprice.models.heads import Head
    rng = np.random.default_rng(0)
    X = rng.normal(size=(300, 5))
    y = (X[:, 0] > 0).astype(int) + (X[:, 1] > 1).astype(int)
    h = Head("lightgbm", {"num_leaves": 8}, seed=1).fit(X[:200], y[:200], np.ones(200), X[200:250], y[200:250], np.ones(50))
    p1 = h.predict_proba(X[250:])
    pickle.dump(h, open(tmp_path / "h.pkl", "wb"))
    h2 = pickle.load(open(tmp_path / "h.pkl", "rb"))
    np.testing.assert_allclose(p1, h2.predict_proba(X[250:]))


def test_config_validation_and_hash_stability():
    a = load_config("smoke")
    b = load_config("smoke")
    assert config_hash(a) == config_hash(b)
    c = load_config("smoke", {"labels": {"k": 0.25}})
    assert config_hash(c) != config_hash(a)
    assert c["_meta"]["owner_overrides"][0]["key"] == "labels.k"
    with pytest.raises(ValueError):
        load_config("smoke", {"sec": {"max_rps": 20}})
    with pytest.raises(ValueError):
        load_config("smoke", {"study": {"start": "2024-01-01", "end": "2023-01-01"}})


def test_gold_partitions_disjoint_and_test_blind(tmp_path):
    from secprice.config import load_yaml, repo_root
    from secprice.gold.tools import GoldTestLock, export_templates, sample_gold
    ev = pd.DataFrame({"event_id": [f"e{i}" for i in range(60)], "decision_date": pd.date_range("2010-01-01", periods=60, freq="W"),
                       "year": 2010, "event_type": "8K", "categories": [["earnings"]] * 60, "ff12_industry": "Other",
                       "accessions": [[f"a{i}"] for i in range(60)], "forms": [["8-K"]] * 60, "filing_date": "2010-01-01"})
    s = sample_gold(ev, pd.Timestamp("2020-01-01"), {"gold_dev": 5, "gold_val": 10, "gold_test": 10}, 0)
    assert not s.event_id.duplicated().any() and set(s.gold_split) == {"gold_dev", "gold_val", "gold_test"}
    qb = load_yaml(repo_root() / "configs" / "question_bank_v1.yaml")
    pre = {e: {f: {"present": True} for f in qb["fields"]} for e in s.event_id}
    export_templates(s, qb, None, tmp_path, prefill=pre)
    t = pd.read_csv(tmp_path / "gold_gold_test_filings_template.csv").fillna("")
    assert (t.llm_prefill_json == "").all()                                     # gold-test labelled blind
    v = pd.read_csv(tmp_path / "gold_gold_val_filings_template.csv").fillna("")
    assert (v.llm_prefill_json != "").all()
    lock = GoldTestLock(tmp_path / "lock.json")
    lock.consume("h")
    with pytest.raises(RuntimeError):
        lock.consume("h")


def test_backtest_entry_and_one_position_per_firm():
    from secprice.backtest.engine import positions
    sig = pd.DataFrame({"event_id": ["a", "b", "c"], "ticker": ["X", "X", "Y"], "entry_idx": [10, 12, 10], "score": [0.9, 0.95, -0.9]})
    p = positions(sig, -0.5, 0.5, 5).set_index("event_id")
    assert p.loc["a", "exit"] == 12 and bool(p.loc["a", "exit_at_open"])           # replaced at the new entry open
    assert p.loc["b", "entry"] == 12 and p.loc["c", "side"] == -1
    p1 = positions(sig, -0.5, 0.5, 5, lag=1).set_index("event_id")
    assert p1.loc["c", "entry"] == 11


In [ ]:
%%writefile $REPO/tests/unit/test_timing.py
"""SPEC.md §18.1 #1-#4 and addendum P0.1 / P0.3 acceptance tests."""
from pathlib import Path

import numpy as np
import pandas as pd
import pytest

from secprice.events.timestamps import (acceptance_from_sources, derive_times, header_to_et, json_to_et,
                                        parse_header_acceptance)
from secprice.market.calendar import TradingCalendar

FIX = Path(__file__).resolve().parents[1] / "fixtures"
ET = "America/New_York"


@pytest.fixture(scope="module")
def cal():
    return TradingCalendar.nyse("2019-01-01", "2026-12-31")


def T(s):
    return pd.Timestamp(s).tz_localize(ET)


# ---------------------------------------------------------------- #1 session buckets
def test_buckets_boundaries(cal):
    assert cal.bucket(T("2024-03-05 09:29:59")) == "PRE"
    assert cal.bucket(T("2024-03-05 09:30:00")) == "INTRA"
    assert cal.bucket(T("2024-03-05 15:59:59")) == "INTRA"
    assert cal.bucket(T("2024-03-05 16:00:00")) == "POST"
    assert cal.bucket(T("2024-03-09 12:00:00")) == "NONTRADING"           # Saturday
    assert cal.bucket(T("2024-07-04 10:00:00")) == "NONTRADING"           # holiday


def test_buckets_early_close_and_dst(cal):
    assert cal.is_early_close(cal.index_of_date("2024-11-29"))            # day after Thanksgiving, 13:00 close
    assert cal.bucket(T("2024-11-29 13:30:00")) == "POST"
    assert cal.bucket(T("2024-11-29 12:59:00")) == "INTRA"
    # DST changes: 09:30 ET open on both sides of the switch
    assert cal.bucket(T("2024-03-08 09:31:00")) == "INTRA" and cal.bucket(T("2024-03-11 09:31:00")) == "INTRA"
    assert cal.bucket(T("2024-11-01 09:29:00")) == "PRE" and cal.bucket(T("2024-11-04 09:29:00")) == "PRE"


# ---------------------------------------------------------------- #2 timestamps
def test_sgml_header_fixture_and_json_rule():
    hdr = (FIX / "aapl_8k_20260730_header.html").read_text(encoding="utf-8")
    raw = parse_header_acceptance(hdr)
    assert raw == "20260730163028"
    # the submissions JSON reported this filing as 2026-07-31T00:30:28.000Z (verified 2026-10-23)
    assert json_to_et("2026-07-31T00:30:28.000Z") == header_to_et(raw)
    assert json_to_et("2020-02-27T16:14:21.000Z") == T("2020-02-27 06:14:21")   # EST case
    full = (FIX / "aapl_8k_2002_full.txt").read_text(encoding="utf-8", errors="replace")
    assert header_to_et(parse_header_acceptance(full)) == json_to_et("2002-08-14T00:47:30.000Z")


def test_public_decision_and_quality_b():
    t, q, src, date_only = acceptance_from_sources("2001-02-12T10:00:00.000Z", None, "2001-02-12")
    assert q == "B" and date_only and t == T("2001-02-12 17:30:00")         # midnight placeholder -> assumed after close
    t2, q2, _, _ = acceptance_from_sources("2026-07-31T00:30:28.000Z", None, "2026-07-30")
    assert q2 == "A" and t2 == T("2026-07-30 16:30:28")
    t3, q3, _, _ = acceptance_from_sources("2026-07-31T00:30:28.000Z", None, "2026-07-30", json_rule_ok=False)
    assert q3 == "C"
    d = derive_times(pd.Series([t2]), 5, 30)
    assert d.public_available_time.iloc[0] == T("2026-07-30 16:35:28")
    assert d.decision_time.iloc[0] == T("2026-07-30 17:05:28")
    assert d.public_time_source.iloc[0] == "assumed_lag"                       # addendum P0.3: labelled as an assumption


# ---------------------------------------------------------------- #3 / #4 T_info and T_trade clocks
CASES = {  # acceptance, bucket, expected T_info base date, T_info exit date (h=1), T_trade entry date
    "PRE": ("2024-03-05 07:00:00", "2024-03-04", "2024-03-05", "2024-03-05"),
    "LATE_PRE": ("2024-03-05 09:10:00", "2024-03-04", "2024-03-05", "2024-03-06"),   # decision 09:45 > open -> next day
    "INTRA": ("2024-03-05 11:00:00", None, None, "2024-03-06"),
    "POST": ("2024-03-05 16:30:00", "2024-03-05", "2024-03-06", "2024-03-06"),
    "NONTRADING": ("2024-03-09 12:00:00", "2024-03-08", "2024-03-11", "2024-03-11"),
}


@pytest.mark.parametrize("case", list(CASES))
def test_clocks(cal, case):
    acc, base_d, exit_d, entry_d = CASES[case]
    a = T(acc)
    tt = derive_times(pd.Series([a]), 5, 30).iloc[0]
    b = cal.last_close_at_or_before(a)
    d0 = cal.first_close_after(tt.public_available_time)
    e = cal.first_open_at_or_after(tt.decision_time)
    assert str(cal.date(e).date()) == entry_d
    assert cal.open_time(e) >= tt.decision_time                                  # T_trade entry never before decision
    if base_d:
        assert str(cal.date(b).date()) == base_d and str(cal.date(d0).date()) == exit_d
        assert cal.close_time(b) <= a                                              # base close precedes acceptance


def test_ttrade_no_component_before_decision(cal):
    """Addendum P0.1 acceptance test: with synthetic PRE, POST, NONTRADING and INTRA events no return component used
    as a Track B (T_trade) target occurs before decision time: the first price used is the entry open."""
    for acc, *_ in CASES.values():
        tt = derive_times(pd.Series([T(acc)]), 5, 30).iloc[0]
        e = cal.first_open_at_or_after(tt.decision_time)
        first_price_time = cal.open_time(e)
        assert first_price_time >= tt.decision_time


def test_same_day_8k_group_is_clocked_at_its_last_filing(cal):
    """Point-in-time (D-071, found by the full run's availability assertion): a same-day 8-K group uses every filing's
    content, so its clock must start at the last acceptance; the event id still comes from the earliest accession."""
    from secprice import common as C
    from secprice.config import load_config
    from secprice.events.build import build_events, event_id
    cfg = load_config("smoke")
    tz = "America/New_York"
    man = pd.DataFrame({"accession": ["0001-12-000001", "0001-12-000002"], "cik": [7, 7], "form": ["8-K", "8-K"],
                        "filing_date": ["2012-02-29", "2012-02-29"],
                        "acceptance_time": [pd.Timestamp("2012-02-29 11:00", tz=tz), pd.Timestamp("2012-02-29 16:50", tz=tz)],
                        "ts_quality": ["A", "A"], "ts_source": ["sgml_header", "sgml_header"], "report_date": [None, None],
                        "items_list": [["8.01"], ["2.02", "9.01"]], "item_titles": [[], []], "in_study": [True, True],
                        "is_amendment": [False, False]})
    tax = C.taxonomy(cfg)
    ev = build_events(man, cal, cfg, tax, pd.DataFrame(), pd.DataFrame())
    assert len(ev) == 1
    e = ev.iloc[0]
    assert pd.Timestamp(e.acceptance_time) == pd.Timestamp("2012-02-29 16:50", tz=tz)
    assert pd.Timestamp(e.first_acceptance_time) == pd.Timestamp("2012-02-29 11:00", tz=tz)
    assert pd.Timestamp(e.decision_time) >= pd.Timestamp("2012-02-29 16:50", tz=tz)
    assert e.event_id == event_id(7, "0001-12-000001")


In [ ]:
sys.path.insert(0, str(REPO / "src"))
for m in [m for m in list(sys.modules) if m.startswith("secprice")]:
    del sys.modules[m]
import secprice, pandas as pd
from secprice import pipeline as P
from secprice.storage import code_hash
P.quiet()
pd.set_option("display.max_columns", 40, "display.width", 220, "display.max_colwidth", 80)
print("secprice", secprice.__version__, "| code hash", code_hash())

def run(stages, force=False):
    t0 = time.time()
    r = P.run(stages, RUN_PROFILE, OWNER_OVERRIDES, force=force, data_root=DATA_ROOT, hot_root=HOT_ROOT)
    print(f"{r}  ({time.time() - t0:.0f}s)")
    return r

def art(stage, name):
    rr = P.make_runner(RUN_PROFILE, OWNER_OVERRIDES, DATA_ROOT, HOT_ROOT)
    return rr.storage.profile_root / stage / rr.stage_hash(stage) / name

def show(stage, name, n=20):
    p = art(stage, name)
    if not p.exists():
        print("(missing)", p); return None
    if p.suffix == ".parquet":
        d = pd.read_parquet(p)
    elif p.suffix == ".csv":
        try:
            d = pd.read_csv(p)
        except pd.errors.EmptyDataError:
            print("(empty table)", p.name); return None
    elif p.suffix == ".json":
        print(json.dumps(json.loads(p.read_text()), indent=1, default=str)[:4000]); return None
    else:
        print(p.read_text()[:4000]); return None
    display(d.head(n)); return d

def card(stage):
    p = DATA_ROOT / RUN_PROFILE / "stage_cards" / f"{stage}.md"
    print(p.read_text()[:3000] if p.exists() else "(no stage card yet)")

## 3. Gate G1 — unit tests + offline integration smoke (must pass before any GPU stage)
43 unit tests cover the spec's §18.1 list and every addendum acceptance test (timing, prospective target, interval-matched
benchmark, SUE history counts, gold-test lock). The offline smoke runs S00→S12 and S20→S24 on a synthetic world with no network.
A pass is recorded per code hash, so it is not repeated after a runtime restart.

In [ ]:
g1 = DATA_ROOT / "_g1" / f"g1_pass_{code_hash()}.json"
if g1.exists():
    print("G1 already passed for this code:", g1.read_text())
else:
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", str(REPO / "tests")], cwd=str(REPO),
                       capture_output=True, text=True, env={**os.environ, "PYTHONPATH": str(REPO / "src")})
    print(r.stdout[-3000:], r.stderr[-2000:])
    assert r.returncode == 0, "Gate G1 failed: fix before running GPU stages"
    g1.parent.mkdir(parents=True, exist_ok=True)
    g1.write_text(json.dumps({"passed_at": time.strftime("%Y-%m-%d %H:%M:%S"), "summary": r.stdout.strip().splitlines()[-1]}))
    print("G1 passed")

## 4. Inputs (`DATA_ROOT/inputs`)
| File | Content | Source |
|---|---|---|
| `ohlcv_1day/<TICKER>.parquet` | our 979 verified daily OHLCV series (split-adjusted) | TriFrame `final/1day` |
| `yf_frozen.parquet` | yfinance warm-up bars 1998–2001, all dividends/splits, SPY | frozen 2026-10-22 (explicit parameters) |
| `lexicons/Loughran-McDonald_MasterDictionary_1993-2025.csv` | LM dictionary (academic use) | SRAF Notre Dame |
| `hf_census_5da7e3c8b920.parquet` | metadata census of the HF 10-K corpus (no labels) | built by `secprice.audit.hf_audit` |

All four were prepared and copied to `MyDrive/sec-price-study/inputs`. Missing optional files are rebuilt automatically
(the census by streaming 13 GB of HF shards; the LM dictionary from SRAF).

In [ ]:
inp = DATA_ROOT / "inputs"
chk = {"ohlcv_1day files": len(list((inp / "ohlcv_1day").glob("*.parquet"))) if (inp / "ohlcv_1day").exists() else 0,
       "yf_frozen.parquet": (inp / "yf_frozen.parquet").exists(),
       "LM dictionary": (inp / "lexicons" / "Loughran-McDonald_MasterDictionary_1993-2025.csv").exists(),
       "HF census": (inp / "hf_census_5da7e3c8b920.parquet").exists()}
print(json.dumps(chk, indent=1))
status = P.status(RUN_PROFILE, OWNER_OVERRIDES, DATA_ROOT)
part1_stages = [f"S{i:02d}" for i in range(0, 13)] + ["S20", "S21"]
part1_done = bool((status[status.stage.isin(part1_stages)].status == "done").all())
print("Part 1 stages (S00-S12, S20-S21) already done under DATA_ROOT/%s: %s" % (RUN_PROFILE, part1_done))

def _where_is_my_data():
    # what Colab can actually see, so a wrong folder / account / unfinished Drive upload is obvious
    lines = ["DATA_ROOT = %s (exists: %s)" % (DATA_ROOT, DATA_ROOT.exists())]
    if DATA_ROOT.exists():
        lines.append("  contents: %s" % sorted(p.name for p in DATA_ROOT.iterdir())[:30])
        for sub in ("inputs", "inputs/ohlcv_1day", RUN_PROFILE):
            q = DATA_ROOT / sub
            if q.exists():
                n_files = sum(1 for x in q.rglob("*") if x.is_file())
                n_ok = sum(1 for _ in q.rglob("_SUCCESS"))
                lines.append("  %-18s %6d files%s" % (sub + "/", n_files, ("  (%d finished stages; expected 16 for Part 1 of full)" % n_ok) if sub == RUN_PROFILE else ""))
            else:
                lines.append("  %-18s MISSING" % (sub + "/"))
    if IN_COLAB:
        top = Path("/content/drive/MyDrive")
        lines.append("  MyDrive folders that look relevant: %s" % sorted(p.name for p in top.iterdir() if "sec" in p.name.lower() or "price" in p.name.lower())[:20])
    return "\n".join(lines)

if not part1_done and chk["ohlcv_1day files"] == 0:
    if PART in ("gpu", "cpu2"):
        raise RuntimeError("Part 1 results are not under DATA_ROOT/%s, and re-running Part 1 takes ~15 h. Copy the folder "
                           "'%s' from your computer's Google Drive (sec-price-study/%s, ~4.2 GB) into MyDrive/%s/ and wait until "
                           "Drive shows it as fully uploaded, then re-run.\n%s" % (RUN_PROFILE, RUN_PROFILE, RUN_PROFILE, DRIVE_FOLDER, _where_is_my_data()))
    raise RuntimeError("The input files are not visible to Colab: expected MyDrive/%s/inputs/ohlcv_1day/*.parquet (979 files).\n"
                       "Fix: on your computer open Google Drive, check sec-price-study/inputs has finished uploading (no pending sync), "
                       "that you are signed in to the SAME Google account as Colab, and that DRIVE_FOLDER above matches the folder name.\n%s"
                       % (DRIVE_FOLDER, _where_is_my_data()))
if not part1_done and PART in ("gpu", "cpu2"):
    raise RuntimeError("Part 1 is incomplete under DATA_ROOT/%s; run PART='cpu1' first (or copy the finished '%s' folder from Drive).\n%s"
                       % (RUN_PROFILE, RUN_PROFILE, _where_is_my_data()))
# fail fast, before any long stage
if PART in ("all", "cpu1") and not part1_done and len(os.environ.get("SEC_USER_AGENT", "").split()) < 2:
    raise RuntimeError("Add the Colab secret SEC_USER_AGENT = 'Your Name your.email@example.com' (SEC fair-access policy), then re-run.")
if PART == "gpu":
    import torch
    assert torch.cuda.is_available(), "PART='gpu' needs a GPU runtime: Runtime > Change runtime type > A100 or H100, then restart and re-run."
free_gb = shutil.disk_usage(HOT_ROOT).free / 1e9
print(f"free disk at HOT_ROOT: {free_gb:.0f} GB")
if RUN_PROFILE == "full" and free_gb < 60 and not part1_done:
    print("WARNING: the full profile caches filings on local disk; free at least ~60 GB (or use a High-RAM/large-disk runtime).")
display(status)

# Part 1 — data, events, targets, features, baselines, Track A (CPU)

### S00 environment · S01 Hugging Face audit · S02 universe
S02 maps our tickers to SEC CIKs, keeps firms that are also in the HF 10-K corpus and file 10-K/10-Q, then selects S&P-500 members first and fills the rest with a seeded, sector-stratified draw (no price returns are read).

In [ ]:
if PART in ('all', 'cpu1'):
    run(['S00', 'S01', 'S02'])
    u = show('S02', 'universe.parquet', 5)
    show('S02', 'firms_per_year.csv', 30)
    print(u.selected.sum() if u is not None else None, 'firms selected')
    if u is not None: print(u[u.selected].sp_bucket.value_counts().to_dict(), '(S&P 500 / non-S&P 500)')
    show('S02', 'sp500_coverage.json')

### S03 calendar, prices, corporate actions, factors
NYSE sessions (holidays, early closes), our split-adjusted bars + yfinance warm-up, raw / split-adjusted / total-return series, SPY, Kenneth French FF5 + momentum.

In [ ]:
if PART in ('all', 'cpu1'):
    run(['S03'])
    show('S03', 'price_coverage.csv', 10)

### S04 EDGAR index + coverage audit
Every 10-K / 10-Q / 8-K in the window with corrected acceptance times (verified rule; SGML header sample), periodic expectations and reason codes. 8-Ks carry no expectation.

In [ ]:
if PART in ('all', 'cpu1'):
    run(['S04'])
    show('S04', 'data_coverage_summary.md')
    show('S04', 'ts_quality_by_year.csv', 30)
    pf = P.preflight(RUN_PROFILE, OWNER_OVERRIDES, DATA_ROOT, HOT_ROOT)   # G2 budget check before the long download
    print(json.dumps(pf, indent=1, default=str))
    if RUN_PROFILE == 'full' and not pf['passed']: raise RuntimeError(f"preflight refuses the full run: {pf['over_budget']}")

### S05 download + parse + taxonomy + anonymize (long, resumable)
Downloads primary documents / full 8-K submissions at ≤ 6 requests/s with your User-Agent, extracts the target sections (Exhibit-13 fallback for MD&A by reference), masks firm names and tickers. If the runtime disconnects, re-run this cell: finished firms are skipped.

In [ ]:
if PART in ('all', 'cpu1'):
    run(['S05'])
    show('S05', 'parse_qa.csv', 40)
    card('S05')

### S06 XBRL + release EPS + SUE
Company facts with public-availability times, as-first-reported quarterly EPS, EPS parsed from earnings-release exhibits, cluster-aware SUE (history strictly before decision time).

In [ ]:
if PART in ('all', 'cpu1'):
    run(['S06'])
    show('S06', 'sue_coverage.csv', 40)
    card('S06')

### S07 events + four timestamps · S08 targets and labels
Events (periodic filings, same-day 8-K groups), session buckets, earnings types, clusters, overlaps; T_trade (primary, entry at the first open at/after decision time) and T_info targets.

In [ ]:
if PART in ('all', 'cpu1'):
    run(['S07', 'S08'])
    show('S07', 'event_counts.csv')
    show('S07', 'usable_cutoff.json')
    show('S08', 'target_summary.csv', 40)

### S09 features · S10 splits, cohorts and the leakage audit
Price, metadata, accounting, Loughran-McDonald, filing-change and novelty features (availability ≤ decision time). Walk-forward folds → development holdout → locked test with purge and embargo; 24-item leakage audit and four planted leaks.

In [ ]:
if PART in ('all', 'cpu1'):
    run(['S09', 'S10'])
    show('S10', 'blocks.csv')
    show('S10', 'leakage_audit.csv', 30)
    show('S10', 'planted_leaks.json')

### S11 baselines L0–L4, L6 (default and tuned)
Majority, price, metadata, accounting and lexical baselines and the best structured set, with Optuna tuning on walk-forward macro-F1, loss curves, learning curves, adversarial validation, permutation null and the elastic-net convexity certificate.

In [ ]:
if PART in ('all', 'cpu1'):
    run(['S11'])
    show('S11', 'ladder_summary.csv', 40)
    show('S11', 'fit_diagnosis.csv', 40)
    show('S11', 'enet_certificate.json')
    show('S11', 'bucket_composition.csv')       # S&P 500 vs non-S&P 500 vs combined (D-056)
    show('S11', 'bucket_scores.csv', 30)
    show('S11', 'bucket_pooled_vs_own.csv')

### S12 gold-set export / import (Track C)
Stratified gold sample from training-period filings and annotation templates in Drive. Annotate them (gold-test blind), put the completed CSVs in `inputs/gold/`, then re-run S12 with force=True.

In [ ]:
if PART in ('all', 'cpu1'):
    run(['S12'])
    show('S12', 'gold_status.json')

### S20 Track A event study · S21 drift tests
CARs, abnormal volume and volatility by form / 8-K category / bucket, all vs clean events, pre-trends, correlations, topics; SUE-conditioned drift and other tests with the pre-registered verdict rubric.

In [ ]:
if PART in ('all', 'cpu1'):
    run(['S20', 'S21'])
    show('S21', 'hypothesis_tests.csv', 30)

# Part 2 — embeddings, tuned heads, G3, fine-tuning, RAG (GPU, A100)
Switch the runtime to **A100** (Runtime → Change runtime type), restart, set `PART = "gpu"` and *Run all* (stages already done
are skipped). On CPU these stages run only for the smoke profile.

### S13 preflight
Workload, storage and GPU-hour projection from a timed micro-benchmark; refuses `full` over budget.

In [ ]:
if PART in ('all', 'gpu'):
    run(['S13'])
    show('S13', 'preflight.json')

### S14 frozen encoder embeddings
SEC-BERT, FinBERT-tone (with native tone), ChronoBERT (vintage Y−1 for year-Y events) and BGE-base; one model in memory at a time; mean/max pooled section vectors; change cosine.

In [ ]:
if PART in ('all', 'gpu'):
    run(['S14'])
    card('S14')

### S15 L5 / L7 heads, G3 and fine-tuning
Embedding-only and structured+embedding heads per encoder (same rows, heads, budget), H4 vintage-vs-static, G3 text-lift gate, and — only if G3 passes or the owner override is set — fine-tuning of the top-2 encoders with the fixed grid (loss curves saved).

In [ ]:
if PART in ('all', 'gpu'):
    run(['S15'])
    show('S15', 'g3.json')
    show('S15', 'emb_ladder_summary.csv', 40)
    show('S15', 'h4.csv', 20)
    show('S15', 'finetune_status.json')

### S16 core RAG · S17 teacher/student distillation
BM25 / hybrid / hybrid+rerank retrieval and Qwen2.5-7B extraction scored on gold-val; G3b; LLM pre-fill for gold-dev/val templates. S17 runs only after G3b passes (or the exploratory owner flag).

In [ ]:
if PART in ('all', 'gpu'):
    run(['S16', 'S17'])
    show('S16', 'rag_tournament.csv')
    show('S16', 'g3b.json')
    show('S17', 'student_report.json')

# Part 3 — development holdout, G4 freeze, locked test, backtest, Track C, report (CPU)

### Freeze the pre-registration (required before the development holdout is scored)

In [ ]:
if PART in ('all', 'cpu2'):
    print('prereg hash:', P.prereg_freeze(RUN_PROFILE, OWNER_OVERRIDES, DATA_ROOT))

### S18 development holdout, MCS / PBO, stacking (H5) and the G4 freeze
Finalists retrained on all pre-holdout data and scored on 2020–2022; Model Confidence Set and PBO as diagnostics only; the stacking extension; the frozen architecture is hashed.

In [ ]:
if PART in ('all', 'cpu2'):
    run(['S18'])
    show('S18', 'dev_lifts.csv')
    show('S18', 'frozen_architecture.json')
    show('S18', 'review_triggers.json')

### S19 locked test (single use)
Runs only when `lockbox.unlock_locked_test` is True. A second scoring is refused unless `acknowledge_reuse` is True, which relabels the block as development.

In [ ]:
if PART in ('all', 'cpu2'):
    if OWNER_OVERRIDES['lockbox'].get('unlock_locked_test'):
        run(['S19'])
        show('S19', 'locked_results.parquet', 40)
    else:
        print('Locked test sealed (set lockbox.unlock_locked_test = True after reviewing the development results).')

### S22 backtest (economic-relevance check)
Signal P(Up) − P(Down) of the frozen T_trade model; training-period 80/20 thresholds; entry at the first open after decision time; 10/20 bps costs; long-only vs interval-matched SPY and hypothetical long-short.

In [ ]:
if PART in ('all', 'cpu2'):
    run(['S22'])
    show('S22', 'backtest.csv', 30)

### S23 Track C final (gold-test, once) · S24 figures and the full report
Content validity on human gold-test labels (when present), then all figures F01–F40 and the full extracted report (Markdown, HTML with embedded figures, PDF), built from computed artifacts only.

In [ ]:
if PART in ('all', 'cpu2'):
    run(['S23'])
    run(['S24'], force=True)

## Results — executive summary, smallest defensible claim, and every figure

In [ ]:
from IPython.display import Markdown, Image
rep = DATA_ROOT / RUN_PROFILE / "reports"
md_ = (rep / "results_summary.md").read_text(encoding="utf-8") if (rep / "results_summary.md").exists() else "(report not built yet)"
display(Markdown(md_.split("## 1. Data integrity")[0]))
idx = json.loads((rep / "figures" / "figure_index.json").read_text()) if (rep / "figures" / "figure_index.json").exists() else []
for f in idx:
    if f["status"] == "ok":
        display(Markdown(f"**{f['id']} — {f['title']}.** {f['caption']}"))
        display(Image(filename=f["path"]))
    else:
        print(f"{f['id']} {f['title']}: {f['status']}")
print("Full report:", rep / "results_report.html", "|", rep / "results_report.pdf", "|", rep / "results_summary.md")

In [ ]:
display(Markdown(md_))